# 🎻 Expressive Techniques in Solo Violin Performance and Their Effect on Perceived Emotion — Full Pipeline
**Diploma Thesis · School of Electrical and Computer Engineering, NTUA · supervisor G. Stamou**

### Framing
The thesis follows one chain — **expressive technique → acoustic result → perceived emotion** — and
delivers two categorisations: of the **tools** that detect a technique in the signal (signal processing,
pitch tracking, alignment, a self-supervised representation) and of the **modalities** that measure its
effect on the audience (valence–arousal ratings, emotion tags). It is an **interpretable-attribution**
study, not a leaderboard: MERT enters as one tool category — the opaque (SSL) reference — against which
the **cost of explainability** of the named, interpretable features is measured.

**The notebook order is the story order (Phase C2).** Each part answers the question the previous one
opens: the data (Parts 0–3) → *did the manipulation work?* (Part 4) → *what is chance for this pipeline?*
(Part 5) → *can the acoustics predict the audience on unseen pieces?* (Part 6) → *what does MERT encode?*
(Part 7) → *does technique change predict rating change within a piece?* (Part 8) → *which technique?*
(Part 9) → *what does explainability cost, and where does the signal live?* (Part 10) → *the defensible
claims* (Part 11).

### Questions
1. Which techniques does the performer escalate across **MEC → EXP → EXG**, and which of them reach the
   listeners as a change in **valence** or **arousal**?
2. Can perceived emotion be **attributed interpretably** to technique, and what does that explainability
   **cost** relative to the opaque representation?
3. Where do the interpretable features and the SSL representation **agree**, and where do they
   **complement** each other?

### Design in one paragraph
60 excerpts = 20 pieces × 3 renditions (MEC mechanical / EXP expressive / EXG exaggerated), all played
by the author. Listening study: 118 responses over 5 linked (ring) forms, **N = 116 after participant
screening** (§1.1; screening v3 runs outside the notebook). Ground truth = **5 classes** — the 4
circumplex quadrants + Neutral — from rater-normalised valence/arousal means (§1.1b–§1.2). Every
cross-validated number uses **leave-one-piece-out as the PRIMARY protocol**; LOOCV is reported next to it
as the **optimistic secondary** (95 % / 91 % of the V / A variance is between pieces).

### The spine — one hierarchical numbering, twelve parts
| Part | Theme | Question | Hands on |
|---|---|---|---|
| **0** | Setup | Is the environment reproducible and every input reachable? | pinned runtime, config, `bh_fdr`, `audio_files` |
| **1** | Audience ground truth | What did listeners perceive (normalised, labelled)? | `df_expr`, `Y`, `mlb` |
| **2** | Feature extraction | What can we measure in the signal? | `emb_mert`, `df_crepe`, `df_essentia`, `df_dtw`, `X_A`–`X_F` |
| **3** | Tool validation | Is each tool's output sane, and the data fit to model at N = 60? | go/no-go on the matrices |
| **4** | Manipulation check | Did the performer escalate the techniques, and did the listeners hear it? | `df_interp`, `INTERP_PRED_COLS` / `INTERP_DESC_COLS`, `df_dose`, `MANIP_CHECK` |
| **5** | Evaluation yardstick | How is every number cross-validated, and what is chance for the deployed pipeline? | CV engine, Dummy floors, `EMP_NULL_F1`, `EMP_NULL_PRIMARY` |
| **6** | Prediction | Can the acoustics predict the audience on unseen pieces? | `all_recs`, `REG_RES`, the lattice, `df_emp_null_readout` |
| **7** | MERT diagnostics — what the embedding encodes | Piece identity, layers, named techniques, rating shift, what each tool adds | `MG_VERDICT`, layer study, `PROBE_R2`, `df_sh`, `df_block` |
| **8** | Within-piece prediction | Does technique change predict rating change beyond the condition? | `df_within_piece` |
| **9** | Which techniques | Which technique moves which rating / class? | `df_delta_tests`, read-out betas, SHAP |
| **10** | Cost of explainability | What does explainability cost, and where does the signal live? | `COE`, `df_abl_subfamily`, overlap verdicts |
| **11** | Synthesis | What is the defensible summary? | `claims.csv`, `final_summary.csv`, `all_predictions.csv` |

### Data flow
```
Questionnaire CSVs (Greek, wide-format)      →  Part 1
  parse → drop the 2 screened participants (N = 116) → per-rater normalization
  → aggregate → VA-derived labels (5 circumplex classes)
  Neutral by proximity to the VA centre (all 60 excerpts, any condition)

60 audio files (20 pieces × MEC / EXP / EXG) →  Part 2
  ├─ MERT-v1-95M   (frozen, layers 5-7)  → 768-dim   ── Set A (MERT-PCA25, PCA-forwarded)
  │    §2.1c all 13 hidden states cached (label-free; read only by the diagnostic §7.2)
  ├─ CREPE        (tracks cached once)  → 10 descriptors: vibrato · portamento · F0  ── Set B
  ├─ Essentia DSP  (dynamics/timbre/tonal) → forwarded subset
  └─ timing + DTW  (timing_* from note onsets + tempogram · dtw_* deviation vs MEC)
                                           → forwarded subset  (§2.4)
  Set C = MERT + CREPE + Essentia + DTW · Set D = MERT + Essentia · Set F = MERT + DTW
  regression heads: MERT-full 768 instead of MERT-PCA25 (B-MERTFULL)
  stubs: madmom · MusiCNN                  (§2.4)

Part 3   tool validation / EDA (class floors · coverage · CREPE / DTW sanity · MERT scree)
Part 4   §4.1 interpretable layer: PREDICTIVE (27: CREPE + Essentia + DTW ref-free) /
         DESCRIPTIVE (+ 3 DTW-deviation, ⚑) · §4.2 Page trend on the techniques ·
         §4.3 Friedman / Page on the ratings, lift, class share  (one BH family, ⚑ flags)
Part 5   §5.1 CV engine (leave-one-piece-out PRIMARY + LOOCV) · §5.2 Dummy floors ·
         §5.3 empirical null of NestedSelect on noise: 100-draw map (d = 10 / 25) +
         EXACT-WIDTH null of the pre-declared primary family (C3: 1000 draws at d = 7 / 17, 600 elsewhere;
         cached, resumable from a Kaggle input) — the ONE reference of every F1 (C3 rule)
Part 6   §6.1 classification (NestedSelect default + 8-classifier sensitivity) · §6.2 nested-ridge VA ·
         §6.3 lattice (+ tempogram sensitivity) · §6.4 configuration · §6.5 target side ·
         §6.6 per-class + α · §6.7 per condition · §6.8 quadrants · §6.9 every F1 vs its null (BH)
Part 7   §7.1 train vs test / NN identity · §7.2 layer study (DIAGNOSTIC; MERT_LAYERS stays [5, 6, 7]) ·
         §7.3 correspondence + probe · §7.4 embedding shift · §7.5 what each tool adds on top of MERT
Part 8   §8.1 within-piece prediction + Freedman–Lane incremental test + noise ceiling
Part 9   §9.1 condition deltas · §9.2 linear read-out · §9.3 SHAP (in-sample cross-check)
Part 10  §10.1 cost of explainability · §10.2 sub-families · §10.3 family overlap
Part 11  §11.1 claims table + all outputs
```

> **Status (Phase C2, 2026-11-20 — applied, NOT RUN).** The notebook was restructured so that its order is
> the story order (old → new section map in `docs/thesis.md`, "Phase C2"). Content changes: the interpretable
> layer is split into a **predictive** (27) and a **descriptive** (30, DTW-deviation ⚑) version (replaces
> `DTW_DEV_IN_INTERP`); the manipulation check is one part (Part 4) with **BH over the Page tests**; the
> empirical null is drawn **before** any F1 is printed and every ✓ reads it, with an **exact-width null** for a
> pre-declared primary family and BH within it (Phase C3: 1000 draws at d = 7 / 17, 600 at the other widths; one
> reference rule for every cell — exact width, else the next larger primary width); the tempogram features are flagged ⚑ and get a
> sensitivity row; §6.2 / §6.8 / Fig 5 use the deployed nested ridge; §7.4 / §7.5 / §9.3 / §11.1 are
> rewritten (signed shifts with BH, the CREPE-unique probe rule, the SHAP-vs-β agreement, the claims
> table); the manual audit (§2.6), the 8-classifier Wilcoxon (§4.4) and the §6.2b cross-check are removed.
> `MERT_LAYERS = [5, 6, 7]`, `NEUTRAL_RADIUS = 0.75`, `SEED = 42`, the exclusions and `REART_ENABLED = False`
> are unchanged.
>
> **Earlier status (Phase C1, run 2026-11-16).** C1 added the empirical chance null of the deployed
> classifier (the B3 run had shown that 25 noise columns through NestedSelect score piece F1 0.354, above the
> Dummy strict P95 0.323) and a diagnostic MERT layer study. Phase B3 (run 2026-11-11) disabled the B2
> re-articulation detector (`REART_ENABLED = False` — its splits followed the vibrato amplitude modulation; it
> stays as a report-only negative result in §2.2), added the merge range check and the vibrato band-edge rule;
> the descriptor caches carry `_v4` names and §2.5 asserts that no retired name is read.


## Part 0 · Setup — §0.1 Installation
**Question of this part:** is the environment reproducible and is every input reachable?  
**Hands to the next part:** a pinned runtime (torch 2.3.1), the config constants, and the `audio_files` manifest.

torch is pinned to **2.3.1 first** so nothing later pulls a newer build. The pin was chosen for the
Kaggle **P100** (compute capability sm_60 — PyTorch ≥ 2.4 dropped sm_60); the Phase A and Phase B runs
actually used a **Tesla T4** (sm_75), which 2.3.1 supports as well, so the pin is kept and the notebook
runs on either GPU. Restart the kernel after this cell, then run top-to-bottom.

In [ ]:
# ── §0.1 Installation ───────────────────────────────────────────────
# P100 = compute capability sm_60 (Pascal). PyTorch ≥ 2.4 dropped sm_60 support, so torch is
# pinned to 2.3.1 FIRST — before other installs can accidentally pull a newer torch. The Phase A /
# Phase B runs used a Tesla T4 (sm_75), which 2.3.1 also supports; the pin is kept so the notebook
# runs on either Kaggle GPU.

import subprocess, sys

def pip_install(packages, label):
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + packages,
        capture_output=True, text=True
    )
    ok = r.returncode == 0
    print(f'{"✅" if ok else "❌"} {label}')
    if not ok:
        print(f'   ↳ {r.stderr.strip()[-400:]}')
    return ok

# ── PyTorch 2.3.1 — last release supporting P100 (sm_60); also runs on the T4 ──
pip_install([
    'torch==2.3.1',
    'torchvision==0.18.1',
    'torchaudio==2.3.1',
    '--index-url', 'https://download.pytorch.org/whl/cu121'
], 'PyTorch 2.3.1 (sm_60 / P100 compatible)')

# ── Everything else ───────────────────────────────────────────────────────────
pip_install(['librosa', 'soundfile'],                          'librosa + soundfile')
pip_install(['statsmodels', 'xgboost'],                        'statsmodels + xgboost')
pip_install(['transformers>=4.38,<4.42', 'accelerate'],        'transformers')
pip_install(['crepe', 'resampy'],                              'crepe (pitch tracker)')
pip_install(['shap'],                                              'shap (feature importance)')
# Plain DSP-only `essentia`. NOT `essentia-tensorflow`: that build bundles a
# TensorFlow that conflicts with the pinned torch 2.3.1. TF is only needed for
# MusiCNN (§2.4), which remains a skip stub.
pip_install(['essentia'],                                          'essentia (DSP: loudness / timbre / tonal)')

print('\n✅ Installation complete — restart kernel, then run all cells top-to-bottom')

## Part 0 · Setup — §0.2 Imports & GPU check


In [ ]:
import os, re, ast, warnings, urllib.request
import numpy as np
import pandas as pd
import torch
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from scipy.stats import pearsonr, wilcoxon, spearmanr
import shap
from scipy.signal import butter, filtfilt, welch
from sklearn.svm import SVC, SVR
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import (
    hamming_loss, f1_score, accuracy_score,
    mean_squared_error, confusion_matrix
)
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from transformers import AutoModel, AutoProcessor
warnings.filterwarnings('ignore')

# ── Required packages (hard imports — no fallbacks) ──────────────────────────
import crepe
print('✅ crepe')


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\n{"✅" if DEVICE=="cuda" else "⚠️"}  Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Part 0 · Setup — §0.3 Configuration
**Edit this cell before running anything else.**


In [ ]:
# ─── PATHS ────────────────────────────────────────────────────────────────────
AUDIO_DIR    = '/kaggle/input/datasets/george031218732003/violin-thesis-data/ThesisDataset'
OUTPUT_DIR   = '/kaggle/working/pipeline_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
# Optional input (Phase B, §2.3c): excerpt_id → recording session. Absent → the session check
# is skipped and the cell says so; it is never substituted.
SESSION_MAP_CSV = '/kaggle/input/datasets/george031218732003/violin-thesis-data/session_map.csv'
# Retired cache names — §2.5 asserts that no cell reads any of them (B-cache). Pre-Phase-B names,
# plus the Phase B descriptor caches retired by Phase B2 (the B2 note segmentation changes every
# CREPE descriptor and the timing note onsets, so a cached v2 file would silently keep the old
# segmentation), plus the B2 caches retired by Phase B3 (no re-articulation splits, the merge range
# check and the vibrato band-edge rule change the descriptors and the note onsets again — a cached
# dtw_features_v3.csv would silently keep the B2 onsets). crepe_tracks_v2.npz, rms10_v3.npz and the
# MERT / Essentia caches are unchanged.
OLD_CACHE_NAMES = ['emb_mert.npz', 'feat_crepe.csv', 'essentia_features.csv',
                   'essentia_features.npz', 'dtw_features.csv',
                   'feat_crepe_v2.csv', 'dtw_features_v2.csv',
                   'feat_crepe_v3.csv', 'dtw_features_v3.csv']

# ─── QUESTIONNAIRE CSV FILES ──────────────────────────────────────────────────
# Upload your form CSVs as a separate Kaggle dataset (e.g. violin-thesis-forms)
# They will be mounted at /kaggle/input/violin-thesis-forms/
QUESTIONNAIRE_CSV_PATHS = [
    '/kaggle/input/datasets/george031218732003/violin-thesis-forms/Forms/questionnaire_1.csv',
    '/kaggle/input/datasets/george031218732003/violin-thesis-forms/Forms/questionnaire_2.csv',
    '/kaggle/input/datasets/george031218732003/violin-thesis-forms/Forms/questionnaire_3.csv',
    '/kaggle/input/datasets/george031218732003/violin-thesis-forms/Forms/questionnaire_4.csv',
    '/kaggle/input/datasets/george031218732003/violin-thesis-forms/Forms/questionnaire_5.csv',
]

# ─── EXCERPT ORDER PER QUESTIONNAIRE ─────────────────────────────────────────
QUESTIONNAIRE_EXCERPT_MAP = {
    '/kaggle/input/datasets/george031218732003/violin-thesis-forms/Forms/questionnaire_1.csv': [
        'Bach_Adagio_S1_MEC',           # Απόσπασμα 1
        'Bruch_VC1_EXG',                # Απόσπασμα 2
        'Cinema_Paradiso_EXP',          # Απόσπασμα 3
        'Don_Juan_EXG',                 # Απόσπασμα 4
        'Godfather_MEC',                # Απόσπασμα 5
        'La_Vita_E_Bella_MEC',          # Απόσπασμα 6
        'Meditation_Thais_EXP',         # Απόσπασμα 7
        'Mendelssohn_VC_Mvt2_MEC',      # Απόσπασμα 8
        'Mozart_VC3_EXG',               # Απόσπασμα 9
        'Mozart_VC4_MEC',               # Απόσπασμα 10
        'Schindlers_List_MEC',          # Απόσπασμα 11
        'Secret_Garden_EXP',            # Απόσπασμα 12
        'Shostakovich_SQ8_exrpt1_EXP',  # Απόσπασμα 13
        'Shostakovich_SQ8_exrpt2_MEC',  # Απόσπασμα 14
        'Shostakovich_SQ8_exrpt3_EXG',  # Απόσπασμα 15
        'Shostakovich_Sym5_Mvt1_MEC',   # Απόσπασμα 16
        'Shostakovich_Sym5_Mvt3_EXP',   # Απόσπασμα 17
        'Shostakovich_VC1_EXP',         # Απόσπασμα 18
        'Tanzil_Serenade_EXP',          # Απόσπασμα 19
        'Vitali_Chaconne_EXP',          # Απόσπασμα 20
    ],
     '/kaggle/input/datasets/george031218732003/violin-thesis-forms/Forms/questionnaire_2.csv': [
        'Bach_Adagio_S1_EXG',           # Απόσπασμα 1
        'Bruch_VC1_EXP',                # Απόσπασμα 2
        'Cinema_Paradiso_EXP',          # Απόσπασμα 3
        'Don_Juan_EXP',                 # Απόσπασμα 4
        'Godfather_EXG',                # Απόσπασμα 5
        'La_Vita_E_Bella_EXG',          # Απόσπασμα 6
        'Meditation_Thais_MEC',         # Απόσπασμα 7
        'Mendelssohn_VC_Mvt2_EXG',      # Απόσπασμα 8
        'Mozart_VC3_EXP',               # Απόσπασμα 9
        'Mozart_VC4_MEC',               # Απόσπασμα 10
        'Schindlers_List_MEC',          # Απόσπασμα 11
        'Secret_Garden_EXP',            # Απόσπασμα 12
        'Shostakovich_SQ8_exrpt1_EXP',  # Απόσπασμα 13
        'Shostakovich_SQ8_exrpt2_MEC',  # Απόσπασμα 14
        'Shostakovich_SQ8_exrpt3_EXP',  # Απόσπασμα 15
        'Shostakovich_Sym5_Mvt1_MEC',   # Απόσπασμα 16
        'Shostakovich_Sym5_Mvt3_MEC',   # Απόσπασμα 17
        'Shostakovich_VC1_MEC',         # Απόσπασμα 18
        'Tanzil_Serenade_EXP',          # Απόσπασμα 19
        'Vitali_Chaconne_MEC',          # Απόσπασμα 20
    ],
    '/kaggle/input/datasets/george031218732003/violin-thesis-forms/Forms/questionnaire_3.csv': [
        'Bach_Adagio_S1_EXP',           # Απόσπασμα 1
        'Bruch_VC1_EXP',                # Απόσπασμα 2
        'Cinema_Paradiso_MEC',          # Απόσπασμα 3
        'Don_Juan_EXP',                 # Απόσπασμα 4
        'Godfather_EXP',                # Απόσπασμα 5
        'La_Vita_E_Bella_EXP',          # Απόσπασμα 6
        'Meditation_Thais_MEC',         # Απόσπασμα 7
        'Mendelssohn_VC_Mvt2_EXP',      # Απόσπασμα 8
        'Mozart_VC3_EXP',               # Απόσπασμα 9
        'Mozart_VC4_EXG',               # Απόσπασμα 10
        'Schindlers_List_EXG',          # Απόσπασμα 11
        'Secret_Garden_MEC',            # Απόσπασμα 12
        'Shostakovich_SQ8_exrpt1_MEC',  # Απόσπασμα 13
        'Shostakovich_SQ8_exrpt2_EXG',  # Απόσπασμα 14
        'Shostakovich_SQ8_exrpt3_EXP',  # Απόσπασμα 15
        'Shostakovich_Sym5_Mvt1_EXG',   # Απόσπασμα 16
        'Shostakovich_Sym5_Mvt3_MEC',   # Απόσπασμα 17
        'Shostakovich_VC1_MEC',         # Απόσπασμα 18
        'Tanzil_Serenade_MEC',          # Απόσπασμα 19
        'Vitali_Chaconne_MEC',          # Απόσπασμα 20
    ],
    '/kaggle/input/datasets/george031218732003/violin-thesis-forms/Forms/questionnaire_4.csv': [
        'Bach_Adagio_S1_EXP',           # Απόσπασμα 1
        'Bruch_VC1_MEC',                # Απόσπασμα 2
        'Cinema_Paradiso_MEC',          # Απόσπασμα 3
        'Don_Juan_MEC',                 # Απόσπασμα 4
        'Godfather_EXP',                # Απόσπασμα 5
        'La_Vita_E_Bella_EXP',          # Απόσπασμα 6
        'Meditation_Thais_EXG',         # Απόσπασμα 7
        'Mendelssohn_VC_Mvt2_EXP',      # Απόσπασμα 8
        'Mozart_VC3_MEC',               # Απόσπασμα 9
        'Mozart_VC4_EXP',               # Απόσπασμα 10
        'Schindlers_List_EXP',          # Απόσπασμα 11
        'Secret_Garden_MEC',            # Απόσπασμα 12
        'Shostakovich_SQ8_exrpt1_MEC',  # Απόσπασμα 13
        'Shostakovich_SQ8_exrpt2_EXP',  # Απόσπασμα 14
        'Shostakovich_SQ8_exrpt3_MEC',  # Απόσπασμα 15
        'Shostakovich_Sym5_Mvt1_EXP',   # Απόσπασμα 16
        'Shostakovich_Sym5_Mvt3_EXG',   # Απόσπασμα 17
        'Shostakovich_VC1_EXG',         # Απόσπασμα 18
        'Tanzil_Serenade_MEC',          # Απόσπασμα 19
        'Vitali_Chaconne_EXG',          # Απόσπασμα 20
    ],
    '/kaggle/input/datasets/george031218732003/violin-thesis-forms/Forms/questionnaire_5.csv': [
        'Bach_Adagio_S1_MEC',           # Απόσπασμα 1
        'Bruch_VC1_MEC',                # Απόσπασμα 2
        'Cinema_Paradiso_EXG',          # Απόσπασμα 3
        'Don_Juan_MEC',                 # Απόσπασμα 4
        'Godfather_MEC',                # Απόσπασμα 5
        'La_Vita_E_Bella_MEC',          # Απόσπασμα 6
        'Meditation_Thais_EXP',         # Απόσπασμα 7
        'Mendelssohn_VC_Mvt2_MEC',      # Απόσπασμα 8
        'Mozart_VC3_MEC',               # Απόσπασμα 9
        'Mozart_VC4_EXP',               # Απόσπασμα 10
        'Schindlers_List_EXP',          # Απόσπασμα 11
        'Secret_Garden_EXG',            # Απόσπασμα 12
        'Shostakovich_SQ8_exrpt1_EXG',  # Απόσπασμα 13
        'Shostakovich_SQ8_exrpt2_EXP',  # Απόσπασμα 14
        'Shostakovich_SQ8_exrpt3_MEC',  # Απόσπασμα 15
        'Shostakovich_Sym5_Mvt1_EXP',   # Απόσπασμα 16
        'Shostakovich_Sym5_Mvt3_EXP',   # Απόσπασμα 17
        'Shostakovich_VC1_EXP',         # Απόσπασμα 18
        'Tanzil_Serenade_EXG',          # Απόσπασμα 19
        'Vitali_Chaconne_EXP',          # Απόσπασμα 20
    ],
}

# ─── PARTICIPANT SCREENING (applied in §1.1, before the §1.1b normalization) ───
# Screening v3 (questionnaire_cleaning_v3.py — runs OUTSIDE the notebook, on the five form
# CSVs) flags a participant with five detectors and excludes at ≥ 2 flags:
#   D1 within-form leave-one-out person-total r < 0.10 · D2 MCD distance on form-centred ratings
#   above the simulated 99.9th pct · D3 scale reversal · D4 mean IRV outside [0.25, 2.20] ·
#   D5 LongString ≥ 8 in answer order.
# questionnaire_5_P012 = D1 + D2 · questionnaire_5_P020 = D1 + D3 (reversed valence scale).
EXCLUDED_PARTICIPANTS = ['questionnaire_5_P012', 'questionnaire_5_P020']

# ─── CONSTANTS ────────────────────────────────────────────────────────────────
# ⚠️ EMOTION_LABELS now holds circumplex CLASSES, not emotion names.
# QUADRANT_EMOTIONS maps each quadrant to a *set* of emotions, so derive_labels
# could only ever emit 5 distinct outputs; the MultiLabelBinarizer then built 8
# columns of which 3 pairs were byte-identical (Tenderness≡Peacefulness,
# Power≡Joyful Activation, Sadness≡Nostalgia). The variable NAME is kept (too many
# downstream references) but its CONTENTS are the 4 quadrant codes + Neutral —
# the 5 genuine classes.
EMOTION_LABELS = ['HA_HV', 'HA_LV', 'LA_HV', 'LA_LV', 'Neutral']

# Human-readable names for figure axes / titles ONLY — never used as data keys.
QUADRANT_DISPLAY = {
    'HA_HV'  : 'High Arousal / High Valence',
    'HA_LV'  : 'High Arousal / Low Valence',
    'LA_HV'  : 'Low Arousal / High Valence',
    'LA_LV'  : 'Low Arousal / Low Valence',
    'Neutral': 'Neutral',
}

CONDITIONS = {'MEC':'Mechanical','EXP':'Expressive','EXG':'Exaggerated'}
SAMPLE_RATE   = 24000
MERT_LAYERS   = [5, 6, 7]
MERT_MODEL_ID = 'm-a-p/MERT-v1-95M'
VA_MID        = 3.0
NEUTRAL_RADIUS = 0.75   # excerpts within this VA distance of the centre are Neutral

# ─── REPRODUCIBILITY / RESAMPLING (Phase A audit) ─────────────────────────────
SEED            = 42    # the single seed: torch at MERT load (§2.1), CV engine (§5.1), ablation (§6.3)
N_DUMMY_REPEATS = 200   # §5.1 / §5.2: repeats of the random Dummy floors (stratified, uniform)
WP_N_PERM       = 500   # §8.1: within-piece permutations per (feature set, target)
NC_N_SPLITS     = 200   # §8.1 (B-NC): random rater split-halves for the noise ceiling

# ─── MODEL SELECTION / REPRESENTATION (Phase B) ───────────────────────────────
# B-NMS: the default classifier is a NESTED selection over these 13 configs (§5.1). The list
# ORDER is the tie-break order: NB first, then by C ascending (strongest regularisation first),
# SVM before LogReg at equal C. Factories are functools.partial objects (picklable for joblib).
from functools import partial as _partial
SELECT_C_GRID  = np.logspace(-3, 2, 6)
SELECT_CONFIGS = [('NB', _partial(GaussianNB))]
for _c in SELECT_C_GRID:
    SELECT_CONFIGS += [(f'SVM-Linear C={_c:g}', _partial(SVC, kernel='linear', class_weight='balanced',
                                                         C=float(_c))),
                       (f'LogReg C={_c:g}', _partial(LogisticRegression, max_iter=2000,
                                                     class_weight='balanced', C=float(_c)))]
assert len(SELECT_CONFIGS) == 13
# B-MERTFULL: the regression heads see MERT at full dimensionality ('full' = X_MERT_FULL, 768-d)
# while the classification heads keep the PCA-forwarded X_A; 'pca' restores the Phase A behaviour.
MERT_REG_REPR   = 'full'
# Block weighting of the MULTI-BLOCK MERT-full regression heads (§5.1 block_weights / BlockScaler,
# §6.3–§6.5). After StandardScaler the 768 MERT columns carry 768 units of variance against
# ~10 per interpretable block, so one ridge α lets MERT dominate. 'equal' gives every tool block
# an equal a-priori share of the penalty (w_b = sqrt(P / (B·p_b)), group ridge with
# α_b ∝ B·p_b / P); the weights depend only on column counts — never on data or labels — and are
# applied inside the fold, after the scaler. 'none' restores the unweighted Phase B heads.
# Scope: only regression matrices where MERT-full is concatenated with ≥ 1 other tool block.
REG_BLOCK_WEIGHTING = 'equal'
if REG_BLOCK_WEIGHTING not in ('equal', 'none'):
    raise ValueError(f"REG_BLOCK_WEIGHTING must be 'equal' or 'none', got {REG_BLOCK_WEIGHTING!r}")

# ─── INTERPRETABLE LAYER · FLAGS · PRIMARY NULL (Phase C2) ────────────────────
# DTW-deviation (the dtw_* family of §2.4) is DESCRIPTIVE ONLY: MEC is its alignment reference, so its
# values are 0 / 1 at MEC by construction (it encodes the condition). It never enters a model:
# DTW_DEV_IN_MODELS = False (§2.4b) keeps it out of X_C / X_F, and §4.1 splits the interpretable layer
# into INTERP_PRED_COLS (every cross-validated / model-based analysis — no DTW-deviation) and
# INTERP_DESC_COLS (= INTERP_PRED_COLS + the DTW-deviation features; the descriptive §4.2 and §9.1 only,
# flagged ⚑). This replaces the former DTW_DEV_IN_INTERP switch.
# Tempogram features that failed the manual audit (the §2.6 audit of the B2 / B3 runs, since removed):
# they STAY in the models, are flagged ⚑ in §4.2 / §9.1 / §9.2 / §9.3, are never reported as a "top
# feature" on their own, and §6.3 scores the timing block without them (sensitivity rows).
TEMPOGRAM_UNRELIABLE = ['timing_local_tempo_bpm', 'timing_tempo_logsd', 'timing_pulse_clarity']
# §5.3: draws of the exact-width empirical null of each PRE-DECLARED primary set (same seeding scheme as
# the 100-draw map: draw i = np.random.default_rng(NULL_SEED + i)); cached and resumable.
# Phase C3: PER-WIDTH draw targets (the C2 run stopped at its time budget with 600 draws per width, 561 at
# d = 52). d = 7 (DTW) and d = 17 (CREPE + DTW, Essentia + DTW) are drawn to 1000 — the DTW set passed BH with
# p = 4/601 = 0.0067 against a BH limit of 0.0071, so a single draw decided it; every other primary width stays
# at NULL_N_DRAWS_PRIMARY_DEFAULT. The targets do NOT enter the settings hash of the §5.3 cache.
NULL_N_DRAWS_PRIMARY = {7: 1000, 17: 1000}   # every other primary width → NULL_N_DRAWS_PRIMARY_DEFAULT
NULL_N_DRAWS_PRIMARY_DEFAULT = 600
# Phase C3: read-only Kaggle input holding the C2 null caches (null_f1_draws.csv, null_f1_draws_primary.csv);
# §5.3 copies them into OUTPUT_DIR (the cell appends to the cache) before any cache is read.
NULL_CACHE_INPUT_DIR = '/kaggle/input/datasets/george031218732003/violin-thesis-null-cache'
# §7.5: CREPE-unique = the CREPE features whose §7.3b MERT-probe R² is below this (MERT does not encode them)
CREPE_UNIQUE_R2_MAX = 0.2

# Still the source of truth for the tag → quadrant map (kept, not used as labels).
QUADRANT_EMOTIONS = {
    'HA_HV': ['Power','Joyful Activation'],
    'HA_LV': ['Tension'],
    'LA_HV': ['Tenderness','Peacefulness'],
    'LA_LV': ['Sadness','Nostalgia'],
}
GREEK_TO_ENGLISH = {
    'Χαρά / Ενέργεια (Joyful Activation)' : 'Joyful Activation',
    'Γαλήνη (Peacefulness)'                : 'Peacefulness',
    'Νοσταλγία (Nostalgia)'                : 'Nostalgia',
    'Θλίψη (Sadness)'                      : 'Sadness',
    'Τρυφερότητα (Tenderness)'             : 'Tenderness',
    'Δύναμη (Power)'                       : 'Power',
    'Ένταση (Tension)'                     : 'Tension',
    'Ουδέτερο (Neutral)'                   : 'Neutral',
}

# The RAW participant vocabulary (the 8 names printed on the questionnaire).
# Kept so §1.1 parsing and §4.3a / §4.3c can still speak the original tag language and so
# the genuine human-tag counts are never destroyed by the bucketing.
RAW_EMOTION_TAGS = list(GREEK_TO_ENGLISH.values())

# raw tag → circumplex class. A FIXED CONCEPTUAL grouping (Tension is always
# HA_LV), independent of any excerpt's measured VA position — so bucketing the raw
# tags with it in §4.3a / §4.3c does NOT reintroduce circularity.
TAG_TO_QUADRANT = {emo: q for q, emos in QUADRANT_EMOTIONS.items() for emo in emos}
TAG_TO_QUADRANT['Neutral'] = 'Neutral'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✅ Config loaded — {len(EMOTION_LABELS)} circumplex classes: {EMOTION_LABELS}')


# ── Benjamini–Hochberg (Phase C2: moved here from the §5.1 engine, because the Part 4 manipulation
#    check (§4.2) runs before the engine; same function, same behaviour; §5.1 and later read it) ──
def bh_fdr(p):
    """Benjamini-Hochberg q-values. NaNs pass through as NaN (never counted)."""
    p  = np.asarray(p, dtype=float)
    ok = np.isfinite(p)
    q  = np.full(p.shape, np.nan)
    ps = p[ok]
    n  = ps.size
    if n == 0:
        return q
    o = np.argsort(ps)
    r = np.empty(n)
    r[o] = np.minimum.accumulate((ps[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    q[ok] = np.clip(r, 0.0, 1.0)
    return q

# ═══════════════════════════════════════════════════════════════════════════════
# Shared UNSUPERVISED forwarding gate — used by §2.1b (MERT alt), §2.2b (CREPE)
# and §2.3b (Essentia) so all three tools share ONE code path.
# ═══════════════════════════════════════════════════════════════════════════════
import inspect as _inspect

def unsup_forward_select(df, corr_max=0.95, name='block', eps=1e-8, out_dir=OUTPUT_DIR,
                         order=None, verbose=True):
    """UNSUPERVISED feature-forwarding gate. READS ONLY `df` — NEVER the labels.

    df       : DataFrame, columns = features, index = excerpt_id
    order    : optional column order expressing a-priori preference; when two columns
               are redundant the EARLIER one is kept. Defaults to df.column order.
    corr_max : |Pearson r| above which a column is considered a duplicate.

    (1) drop near-constant columns (std <= eps)
    (2) correlation prune in `order`: keep a column unless it correlates |r| > corr_max
        with an already-kept column (then drop it, citing the kept partner)

    Returns the kept column names; writes <name>_forwarding.csv with the full audit.
    """
    cols = list(order) if order is not None else list(df.columns)
    absent = [c for c in cols if c not in df.columns]
    if absent:
        raise KeyError(f'{name}: columns absent from the extracted block: {absent}')

    Xf = df[cols].apply(pd.to_numeric, errors='coerce')
    Xf = Xf.fillna(Xf.mean(numeric_only=True))
    dropped = {}

    for c in cols:                                   # (1) near-constant
        s = float(Xf[c].std(ddof=0))
        if (not np.isfinite(s)) or s <= eps:
            dropped[c] = f'near-constant across the {len(Xf)} excerpts (std={s:.2e})'

    R, kept = Xf.corr(method='pearson'), []          # (2) redundancy prune
    for c in cols:
        if c in dropped:
            continue
        dup = next((k for k in kept
                    if np.isfinite(R.loc[c, k]) and abs(R.loc[c, k]) > corr_max), None)
        if dup is None:
            kept.append(c)
        else:
            dropped[c] = f'|r|={abs(R.loc[c, dup]):.2f} > {corr_max} with kept "{dup}"'

    if verbose:
        print(f'── {name} forwarding (UNSUPERVISED — no labels read) ──')
        print(f'   FULL block : {len(cols)} columns')
        print(f'   FORWARDED  : {len(kept)}')
        for c in kept:
            print(f'     ✅ {c}')
        print(f'   dropped    : {len(dropped)}')
        for c, why in dropped.items():
            print(f'     ❌ {c:<44} {why}')

    pd.DataFrame({'feature'  : cols,
                  'forwarded': [c in kept for c in cols],
                  'reason'   : [dropped.get(c, 'kept') for c in cols]}
                 ).to_csv(os.path.join(out_dir, f'{name}_forwarding.csv'), index=False)
    return kept

# Runtime guard on the leakage invariant: the gate's own source must not reference
# a label symbol. Cheap, and it fails loudly if someone later "improves" it.
# (Only the source LOOKUP is guarded — some kernels cannot retrieve cell source.
#  The invariant itself is also verified statically outside the notebook.)
try:
    _gate_src = _inspect.getsource(unsup_forward_select)
except (OSError, TypeError):
    _gate_src = ''
    print('ℹ️  gate source not retrievable in this kernel — static check still applies')
assert not re.search(r'\bY\b|\by_[a-z]', _gate_src), \
    'unsup_forward_select must never read labels — unsupervised invariant violated'
print('✅ unsup_forward_select ready (unsupervised invariant asserted)')

## Part 0 · Setup — §0.4 Data discovery / audio manifest

Kaggle datasets are auto-mounted — no Drive mount is needed. This cell only builds the `audio_files` manifest (excerpt_id → wav path) and checks that every questionnaire excerpt ID resolves.


In [ ]:
# ─── NO DRIVE MOUNT NEEDED — Kaggle datasets are auto-mounted ────────────────
# Audio files live at: /kaggle/input/violin-thesis-data/ThesisDataset/{piece}/{excerpt}_1.wav
# The _1 suffix is stripped below so excerpt IDs match the rest of the pipeline.

audio_files = {}
for piece_dir in sorted(Path(AUDIO_DIR).iterdir()):
    if not piece_dir.is_dir():
        continue
    for wav_path in sorted(piece_dir.glob('*.wav')):
        # Strip trailing _1 (or _2, etc.) version suffix from stem
        # e.g. Bach_Adagio_S1_EXG_1  →  Bach_Adagio_S1_EXG
        eid = re.sub(r'_\d+$', '', wav_path.stem)
        audio_files[eid] = str(wav_path)

print(f'✅ {len(audio_files)} audio files indexed')
missing = [e for e in QUESTIONNAIRE_EXCERPT_MAP[QUESTIONNAIRE_CSV_PATHS[0]] if e not in audio_files]
if missing:
    print(f'⚠️  Not found in audio_files: {missing}')
else:
    print('✅ All Form 1 excerpt IDs resolve correctly')

## Part 1 · Audience ground truth — §1.1 Parse questionnaire CSVs · apply the participant exclusions
**Question of this part:** what did the audience perceive, and does the MEC → EXP → EXG manipulation move it?  
**Hands to the next part:** `df_agg` / `df_expr` (per-excerpt VA + quadrant), the label matrix `Y`, and `mlb`.

Wide-format Google Forms export (Greek, 1 row/participant, 3 cols/excerpt).
Parsed by **column position**, not column name, since pandas auto-suffixes duplicates.

**Participant screening.** The screening itself (v3, five detectors, exclusion at ≥ 2 flags) runs outside
the notebook (`questionnaire_cleaning_v3.py`). Its result, `EXCLUDED_PARTICIPANTS` (§0.3:
`questionnaire_5_P012` D1 + D2, `questionnaire_5_P020` D1 + D3 — reversed valence), is dropped from
`df_long` **right after parsing and before §1.1b**, so the normalization constants and everything
downstream are derived on **N = 116** (2,320 ratings). The cell asserts both ids exist and prints the
counts before / after (118 → 116 participants, 2,360 → 2,320 rows).

In [ ]:
# ── §1.1 Parse questionnaire CSVs · apply the participant exclusions ─
# NEEDS : CONDITIONS, EXCLUDED_PARTICIPANTS, GREEK_TO_ENGLISH, QUESTIONNAIRE_CSV_PATHS,
#          QUESTIONNAIRE_EXCERPT_MAP, RAW_EMOTION_TAGS
# MAKES : df_long (screened, N = 116), N_PARTICIPANTS_RAW, parse_emotion_cell, parse_questionnaire_csv
# FEEDS : §1.1b, §1.2, §4.3a
def parse_emotion_cell(cell):
    if pd.isna(cell) or str(cell).strip() == '': return []
    result = []
    for part in [p.strip() for p in str(cell).split(',')]:
        if part in GREEK_TO_ENGLISH:
            result.append(GREEK_TO_ENGLISH[part])
        else:
            m = re.search(r'\(([^)]+)\)', part)
            if m and m.group(1).strip() in RAW_EMOTION_TAGS:
                result.append(m.group(1).strip())
    return result

def parse_questionnaire_csv(csv_path, excerpt_order):
    df = pd.read_csv(csv_path, header=0)
    DATA_START, COLS_PER = 3, 3   # col layout: [emotion_tags, valence, arousal] × 20
    records = []
    for p_idx, row in df.iterrows():
        pid = f'P{p_idx+1:03d}'
        for i in range(20):
            col = DATA_START + i * COLS_PER
            if col + 2 >= len(row): break
            try:
                val = float(str(row.iloc[col+1]).strip())
                aro = float(str(row.iloc[col+2]).strip())
            except ValueError:
                val = aro = np.nan
            eid  = excerpt_order[i]
            cond = next((c for c in CONDITIONS if eid.endswith('_'+c)), 'UNKNOWN')
            records.append({
                'participant_id': pid,
                'excerpt_id'    : eid,
                'condition'     : cond,
                'valence'       : val,
                'arousal'       : aro,
                'emotion_tags'  : parse_emotion_cell(row.iloc[col]),
            })
    return pd.DataFrame(records)

all_responses = []
for csv_path in QUESTIONNAIRE_CSV_PATHS:
    if csv_path not in QUESTIONNAIRE_EXCERPT_MAP:
        print(f'⚠️  No excerpt order for {Path(csv_path).name} — skipping'); continue
    df_q = parse_questionnaire_csv(csv_path, QUESTIONNAIRE_EXCERPT_MAP[csv_path])
    tag  = Path(csv_path).stem
    df_q['participant_id'] = tag + '_' + df_q['participant_id']
    all_responses.append(df_q)
    print(f'  {Path(csv_path).name}: {df_q["participant_id"].nunique()} participants')

df_long = pd.concat(all_responses, ignore_index=True)
print(f'\n   parsed: {len(df_long)} rows | {df_long["participant_id"].nunique()} participants | '
      f'{df_long["excerpt_id"].nunique()} excerpts')

# ── Participant exclusions (screening v3, §0.3) — BEFORE §1.1b, so the normalization
#    constants and every downstream number are derived on the screened panel ─────────────
_form_of = lambda s: s.str.rsplit('_P', n=1).str[0]           # 'questionnaire_5_P012' → 'questionnaire_5'
_ids_all = set(df_long['participant_id'])
_absent  = [p for p in EXCLUDED_PARTICIPANTS if p not in _ids_all]
assert not _absent, f'§1.1: excluded participant id(s) not found in the parsed forms: {_absent}'
N_PARTICIPANTS_RAW = int(df_long['participant_id'].nunique())
_rows_raw = len(df_long)
_per_form_raw = df_long.groupby(_form_of(df_long['participant_id']))['participant_id'].nunique()
df_long = df_long[~df_long['participant_id'].isin(EXCLUDED_PARTICIPANTS)].reset_index(drop=True)
_per_form = df_long.groupby(_form_of(df_long['participant_id']))['participant_id'].nunique()
print(f'\n── Participant screening v3 (exclude at ≥ 2 flags): {len(EXCLUDED_PARTICIPANTS)} excluded — '
      f'{", ".join(EXCLUDED_PARTICIPANTS)}')
print(f'   participants {N_PARTICIPANTS_RAW} → {df_long["participant_id"].nunique()} · '
      f'rows {_rows_raw} → {len(df_long)}')
print('   per form (before → after): '
      + ' · '.join(f'{f.replace("questionnaire_", "F")} {int(_per_form_raw[f])}→{int(_per_form.get(f, 0))}'
                   for f in _per_form_raw.index))
# expected (reproduced offline): 118 → 116 participants, 2360 → 2320 rows,
# per form after = 20 / 21 / 26 / 19 / 30
assert df_long['participant_id'].nunique() == 116, \
    f'§1.1: expected 116 participants after screening, got {df_long["participant_id"].nunique()}'
assert len(df_long) == 2320, f'§1.1: expected 2320 rating rows after screening, got {len(df_long)}'
assert df_long['excerpt_id'].nunique() == 60, \
    f'§1.1: expected 60 excerpts, got {df_long["excerpt_id"].nunique()}'
print(f'\n✅ {len(df_long)} rows | {df_long["participant_id"].nunique()} participants | '
      f'{df_long["excerpt_id"].nunique()} excerpts (screened)')

## Part 1 · Audience ground truth — §1.1b Per-rater normalization (new ground truth)

Raters do not share a scale. One participant's 5 is another's 3, and some compress every
judgement into two adjacent points while others use the full 0–6 range. Because the five
questionnaire forms were rated by **disjoint panels**, that per-rater offset and gain lands
directly on the excerpt means as **panel bias** — a difference between excerpts that is really
a difference between the people who happened to hear them.

**What this cell does.** Per rater, per dimension *separately*, ratings are z-scored (σ floored
at 0.5 so a near-constant rater cannot be amplified into noise), then mapped back with a single
**affine rescale per dimension** whose constants are derived from the data at runtime — nothing
is hardcoded. The rescale is chosen so the normalized excerpt means reproduce the **raw grand
mean** and the **raw between-excerpt SD** exactly.

That last point matters: the grand mean is **preserved, not forced to 3.0**. Arousal legitimately
sits above the midpoint, and pinning it to 3.0 would invent a shift. Preserving location and
spread is what keeps `VA_MID = 3.0`, `NEUTRAL_RADIUS` and the quadrant geometry meaningful.

**Raw values are never destroyed** — they continue downstream as `valence_mean_raw` /
`arousal_mean_raw`, and §1.2b reports exactly what changed.

A second, independent estimate is computed as a robustness check: a **crossed additive**
(rater + excerpt) decomposition, which removes rater offsets without the ipsative distortion
z-scoring introduces. §1.2b reports whether the two methods agree.

> Everything is derived from `df_long` at runtime — nothing is hardcoded. The participant exclusions
> are applied **upstream in §1.1**, so these constants are computed on the screened panel (N = 116).
> **No exclusion logic lives here.**

In [ ]:
# ── §1.1b Per-rater normalization ───────────────────────────────────
# NEEDS : df_long
# MAKES : df_long (+ valence_z, arousal_z, valence_norm, arousal_norm), df_mm,
#          NORM_CONST (dict of per-dim affine constants), n_sigma_floor
# FEEDS : §1.2 (normalized means), §1.2b (validation)
# ═══════════════════════════════════════════════════════════════════════════════
# §1.1b · Per-rater normalization — remove rater offset/gain, preserve the scale
# ═══════════════════════════════════════════════════════════════════════════════
# EVERY constant below is derived from df_long at runtime. Nothing is hardcoded. The
# participant exclusions are applied UPSTREAM (§1.1), so the constants are those of the
# screened panel (N = 116). No exclusion logic is applied here.
DIMS          = ('valence', 'arousal')
Z_SIGMA_FLOOR = 0.5      # a rater flatter than this is NOT amplified into noise

# ── 1. PER-RATER Z, per dimension, over that rater's non-NaN ratings ─────────
n_sigma_floor = 0
_low_n        = []
for dim in DIMS:
    zcol = f'{dim}_z'
    df_long[zcol] = np.nan
    for pid, idx in df_long.groupby('participant_id').groups.items():
        x = df_long.loc[idx, dim]
        v = x.dropna()
        if len(v) < 2:                       # cannot estimate a spread → neutral z
            _low_n.append((pid, dim, int(len(v))))
            df_long.loc[idx, zcol] = np.where(x.notna(), 0.0, np.nan)
            continue
        m = float(v.mean())
        s = float(v.std(ddof=0))
        if s < Z_SIGMA_FLOOR:
            n_sigma_floor += 1
        df_long.loc[idx, zcol] = (x - m) / max(s, Z_SIGMA_FLOOR)   # NaN stays NaN

print(f'σ-floor ({Z_SIGMA_FLOOR}) engaged for {n_sigma_floor} (rater, dimension) pairs '
      f'of {df_long["participant_id"].nunique() * len(DIMS)}')
if _low_n:
    print(f'⚠️  {len(_low_n)} (rater, dimension) pairs had <2 valid ratings → z set to 0.0 '
          f'on their valid rows: {_low_n[:8]}{" …" if len(_low_n) > 8 else ""}')

# ── 2. AFFINE RESCALE per dimension — constants derived at runtime ──────────
# Chosen so the NORMALIZED excerpt means reproduce the RAW grand mean (mu0) and the
# RAW between-excerpt SD (sig0). The map is linear, so this holds EXACTLY — that is
# the invariant §1.2b asserts. The grand mean is PRESERVED, not forced to VA_MID.
NORM_CONST = {}
for dim in DIMS:
    raw_j = df_long.groupby('excerpt_id')[dim].mean()
    z_j   = df_long.groupby('excerpt_id')[f'{dim}_z'].mean().reindex(raw_j.index)
    mu0   = float(raw_j.mean())            # unweighted over the excerpts
    sig0  = float(raw_j.std(ddof=0))       # raw between-excerpt SD
    m0z   = float(z_j.mean())
    s0z   = float(z_j.std(ddof=0))
    if not np.isfinite(s0z) or s0z <= 0:
        raise ValueError(f'{dim}: z-scored excerpt means have no spread — cannot rescale')
    b = sig0 / s0z
    a = mu0 - b * m0z
    df_long[f'{dim}_norm'] = a + b * df_long[f'{dim}_z']
    NORM_CONST[dim] = {'a': a, 'b': b, 'mu0': mu0, 'sig0': sig0, 'm0z': m0z, 's0z': s0z}

# expected at N = 116 (reproduced offline, not asserted): valence ≈ 2.94 + 1.57·z,
# arousal ≈ 3.39 + 1.53·z  (N = 118 run: 2.95 + 1.58·z / 3.39 + 1.52·z)
print('\n── Affine rescale constants (all derived from df_long) ──')
for dim in DIMS:
    k = NORM_CONST[dim]
    print(f'  {dim:<8} norm = {k["a"]:+.4f} + {k["b"]:.4f}·z   '
          f'| raw grand mean {k["mu0"]:.4f}, raw between-excerpt SD {k["sig0"]:.4f}')

# ── 3. CROSSED ADDITIVE (rater + excerpt) — independent robustness estimate ──
# Alternating least squares on x ≈ grand + rater_i + excerpt_j. No ipsative
# distortion (nothing is divided by a rater's own spread). The ring design makes the
# rater×excerpt graph connected, so the effects are identified up to the constant.
def _additive_excerpt_effects(dim, n_iter=50, tol=1e-6):
    """Return a Series (index=excerpt_id) of grand + excerpt_effect for `dim`."""
    sub  = df_long[['participant_id', 'excerpt_id', dim]].dropna(subset=[dim])
    x    = sub[dim].to_numpy(dtype=float)
    rcat = pd.Categorical(sub['participant_id']); ecat = pd.Categorical(sub['excerpt_id'])
    ri, ei = rcat.codes, ecat.codes
    nr, ne = len(rcat.categories), len(ecat.categories)
    rcnt = np.maximum(np.bincount(ri, minlength=nr), 1)
    ecnt = np.maximum(np.bincount(ei, minlength=ne), 1)
    grand = float(x.mean())
    r_eff, e_eff = np.zeros(nr), np.zeros(ne)
    for it in range(n_iter):
        pr, pe = r_eff.copy(), e_eff.copy()
        r_eff = np.bincount(ri, weights=x - grand - e_eff[ei], minlength=nr) / rcnt
        e_eff = np.bincount(ei, weights=x - grand - r_eff[ri], minlength=ne) / ecnt
        if max(np.abs(r_eff - pr).max(), np.abs(e_eff - pe).max()) < tol:
            break
    print(f'  {dim:<8} converged in {it + 1} iterations (grand={grand:.4f})')
    return pd.Series(grand + e_eff, index=pd.Index(ecat.categories, name='excerpt_id'))

print('\n── Crossed additive (rater + excerpt) estimate ──')
df_mm = pd.DataFrame({f'{dim}_mm': _additive_excerpt_effects(dim) for dim in DIMS})
df_mm.index.name = 'excerpt_id'
df_mm = df_mm.reset_index()
MM_METHOD = 'crossed additive (alternating least squares)'

# ── 3b. OPTIONAL: statsmodels crossed random effects for BLUP shrinkage ─────
# Off by default: a crossed variance-component fit with ~(n_raters + n_excerpts)
# levels inside a single group can be very slow, and the additive estimate above
# already supplies the robustness check. Set True to attempt it.
TRY_MIXEDLM = False
if TRY_MIXEDLM:
    try:
        import statsmodels.formula.api as smf
        _blup = {}
        for dim in DIMS:
            sub = df_long[['participant_id', 'excerpt_id', dim]].dropna(subset=[dim]).copy()
            sub['_grp'] = 1
            mf = smf.mixedlm(f'{dim} ~ 1', sub, groups='_grp',
                             vc_formula={'rater'  : '0 + C(participant_id)',
                                         'excerpt': '0 + C(excerpt_id)'}).fit(reml=True)
            re = mf.random_effects[1]
            eff = {k.split('[', 1)[1].rstrip(']'): float(v)
                   for k, v in re.items() if k.startswith('excerpt')}
            _blup[f'{dim}_mm'] = pd.Series(eff) + float(mf.fe_params['Intercept'])
        cand = pd.DataFrame(_blup)
        if set(cand.index) == set(df_long['excerpt_id'].unique()) and not cand.isna().any().any():
            df_mm = cand.rename_axis('excerpt_id').reset_index()
            MM_METHOD = 'statsmodels MixedLM crossed RE (BLUP shrinkage)'
            print(f'  ✅ {MM_METHOD}')
        else:
            print('  ℹ️  MixedLM returned an incomplete BLUP set — keeping the additive estimate')
    except Exception as e:
        print(f'  ℹ️  MixedLM unavailable or failed ({type(e).__name__}: {e}) — '
              f'keeping the additive estimate')
print(f'  robustness estimate in use: {MM_METHOD}')

# ── 4. Show one rater: raw vs z vs norm ─────────────────────────────────────
_pid = df_long['participant_id'].iloc[0]
print(f'\n── Example rater {_pid}: raw → z → norm (first 3 rated excerpts) ──')
print(df_long.loc[df_long['participant_id'] == _pid,
                  ['excerpt_id', 'valence', 'valence_z', 'valence_norm',
                   'arousal', 'arousal_z', 'arousal_norm']].head(3).to_string(index=False))
print(f'\n✅ §1.1b — df_long now carries valence_z/arousal_z and valence_norm/arousal_norm '
      f'({len(df_long)} rows, unchanged)')

## Part 1 · Audience ground truth — §1.2 Aggregate per excerpt · derive labels from VA
Ground-truth labels are **derived from participant mean VA via Circumplex mapping** —
not from raw Q3 emotion-tag votes. This is more theoretically grounded and robust.

> The derived label is the **circumplex quadrant code itself** (`HA_HV` / `HA_LV` /
> `LA_HV` / `LA_LV`), or `Neutral` when the excerpt sits within `NEUTRAL_RADIUS` of the
> VA centre. Expanding each quadrant into its emotion *set* (the old behaviour) could
> only ever produce 5 distinct outputs and made 3 pairs of `Y` columns byte-identical.


In [ ]:
# ── §1.2 Aggregate per excerpt · derive labels from VA ──────────────
# NEEDS : CONDITIONS, df_long (incl. valence_norm / arousal_norm from §1.1b), df_mm,
#          NEUTRAL_RADIUS, VA_MID
# MAKES : agg_excerpt, derive_labels, df_agg, get_piece, top_tags, va_to_quadrant
#          — df_agg carries valence_mean/arousal_mean (NORMALIZED = ground truth),
#          valence_mean_raw/arousal_mean_raw, *_std (raw) and *_std_norm,
#          quadrant/emotion_labels (normalized) + quadrant_raw/emotion_labels_raw
#          + valence_mm/arousal_mm/quadrant_mm (crossed-additive robustness)
# FEEDS : §1.2b, §1.3, §4.3a, §6.8
def va_to_quadrant(v, a, mid=VA_MID):
    return f'{"HA" if a>=mid else "LA"}_{"HV" if v>=mid else "LV"}'

def top_tags(series):
    flat = [t for tags in series for t in (tags if isinstance(tags,list) else [])]
    return Counter(flat).most_common(3)

def agg_excerpt(g):
    return pd.Series({
        # ── GROUND TRUTH: rater-normalized means (§1.1b). The affine rescale preserves
        #    the raw grand mean and raw between-excerpt SD, so VA_MID=3.0 and the
        #    quadrant geometry are unchanged. ────────────────────────────────────────
        'valence_mean'     : g['valence_norm'].mean(),
        'arousal_mean'     : g['arousal_norm'].mean(),
        # ── raw parallels, kept for the §1.2b stability check ──────────────────────
        'valence_mean_raw' : g['valence'].mean(),
        'arousal_mean_raw' : g['arousal'].mean(),
        'valence_std'      : g['valence'].std(),        # RAW dispersion (unchanged)
        'arousal_std'      : g['arousal'].std(),
        'valence_std_norm' : g['valence_norm'].std(),   # for the variance-reduction check
        'arousal_std_norm' : g['arousal_norm'].std(),
        'n_responses'      : len(g),
        'top_tags'         : top_tags(g['emotion_tags']),
        'condition'        : g['condition'].iloc[0],
    })

df_agg = df_long.groupby('excerpt_id').apply(agg_excerpt).reset_index()
df_agg['condition_name'] = df_agg['condition'].map(CONDITIONS)
df_agg['quadrant']       = df_agg.apply(lambda r: va_to_quadrant(r['valence_mean'],r['arousal_mean']),axis=1)

def derive_labels(row):
    # The quadrant CODE is the class now — no expansion into an emotion set
    # (that expansion is exactly what produced the 3 byte-identical column pairs).
    d = np.hypot(row['valence_mean'] - VA_MID, row['arousal_mean'] - VA_MID)
    if d <= NEUTRAL_RADIUS:
        return ['Neutral']
    return [row['quadrant']]

df_agg['emotion_labels'] = df_agg.apply(derive_labels, axis=1)

# ── RAW-based parallels for the §1.2b stability check ────────────────────────
# Same va_to_quadrant / derive_labels logic, applied to the *_raw means by viewing
# them under the canonical column names — the label logic itself is not duplicated.
df_agg['quadrant_raw'] = df_agg.apply(
    lambda r: va_to_quadrant(r['valence_mean_raw'], r['arousal_mean_raw']), axis=1)
_raw_view = df_agg[['valence_mean_raw', 'arousal_mean_raw', 'quadrant_raw']].rename(
    columns={'valence_mean_raw': 'valence_mean', 'arousal_mean_raw': 'arousal_mean',
             'quadrant_raw': 'quadrant'})
df_agg['emotion_labels_raw'] = _raw_view.apply(derive_labels, axis=1)

# ── Crossed-additive (rater + excerpt) robustness estimate from §1.1b ────────
df_agg = df_agg.merge(df_mm, on='excerpt_id', how='left')
df_agg['quadrant_mm'] = df_agg.apply(
    lambda r: va_to_quadrant(r['valence_mm'], r['arousal_mm']), axis=1)

def get_piece(eid):
    for c in CONDITIONS:
        if eid.endswith('_'+c): return eid[:-(len(c)+1)]
    return eid
df_agg['piece'] = df_agg['excerpt_id'].apply(get_piece)

print(f'✅ {len(df_agg)} excerpts aggregated')
print(df_agg[['excerpt_id','condition','valence_mean','arousal_mean',
              'valence_mean_raw','arousal_mean_raw','quadrant','emotion_labels']].to_string())

# expected at N = 116 (reproduced offline, not asserted): HA_LV 20, LA_LV 17, HA_HV 11,
# Neutral 7, LA_HV 5 — vs the N = 118 run, Schindlers_List_EXP moves HA_LV → LA_LV
print('\nClass distribution (circumplex quadrant + Neutral):')
print(df_agg['emotion_labels'].apply(lambda L: L[0]).value_counts().to_string())

print('\nNeutral excerpts by condition (assigned by VA proximity, not by condition):')
print(df_agg[df_agg['emotion_labels'].apply(lambda L: 'Neutral' in L)]['condition'].value_counts())

## Part 1 · Audience ground truth — §1.2b Normalization validation (diagnostic)

Everything here is a **check on §1.1b**, not a step in the pipeline.

The headline is the **anchor cross-form agreement**: 40 of the 60 excerpts were rated in more
than one questionnaire form, so the *same stimulus* was judged by *disjoint rater panels*. Any
disagreement between forms on those anchors is, by construction, rater-panel bias rather than
a property of the excerpt. If per-rater standardization is doing its job, the mean absolute
between-form difference must **shrink**.

The **invariant check** is the safety rail: because the rescale is affine, the grand mean and
the between-excerpt SD of the normalized excerpt means must equal the raw ones *exactly*. That
is what keeps `VA_MID = 3.0` and the quadrant geometry meaningful after normalization — the
grand mean is preserved, **not** forced to 3.0 (arousal legitimately sits above it).

The rest is stability reporting: how many of the 60 change class, how tightly raw and
normalized means still agree, whether within-excerpt dispersion fell, and whether the
z-primary answer survives the independent crossed-additive estimate.


In [ ]:
# ── §1.2b Normalization validation ──────────────────────────────────
# NEEDS : df_long, df_agg, NEUTRAL_RADIUS, NORM_CONST, VA_MID
# MAKES : normalization_report.csv, 1 figure
# FEEDS : — (diagnostic only)
# ═══════════════════════════════════════════════════════════════════════════════
# §1.2b · Did the per-rater normalization do what it claims?
# ═══════════════════════════════════════════════════════════════════════════════
DIMS = ('valence', 'arousal')

# ── 1. INVARIANT — the affine map must preserve grand mean and between-excerpt SD ──
print('═' * 78)
print('1 · INVARIANT CHECK (affine rescale preserves the raw location and spread)')
print('═' * 78)
_inv_ok = True
for dim in DIMS:
    got_mu  = float(df_agg[f'{dim}_mean'].mean())
    got_sig = float(df_agg[f'{dim}_mean'].std(ddof=0))
    exp_mu, exp_sig = NORM_CONST[dim]['mu0'], NORM_CONST[dim]['sig0']
    ok = abs(got_mu - exp_mu) < 1e-6 and abs(got_sig - exp_sig) < 1e-6
    _inv_ok &= ok
    print(f'  {dim:<8} grand mean {got_mu:.6f} vs raw {exp_mu:.6f}   |   '
          f'between-excerpt SD {got_sig:.6f} vs raw {exp_sig:.6f}   {"✅" if ok else "❌"}')
assert _inv_ok, 'Affine invariant violated — normalized means no longer share the raw scale'
print(f'  → VA_MID={VA_MID} stays on the same scale; the grand mean is PRESERVED, not forced.')

# ── 2. HEADLINE — anchor cross-form agreement (raw vs normalized) ────────────
print('\n' + '═' * 78)
print('2 · ANCHOR CROSS-FORM AGREEMENT — the headline test')
print('═' * 78)
_lf = df_long.copy()
_lf['form'] = _lf['participant_id'].str.rsplit('_P', n=1).str[0]
_forms_per_excerpt = _lf.groupby('excerpt_id')['form'].nunique()
ANCHORS = sorted(_forms_per_excerpt[_forms_per_excerpt >= 2].index)
print(f'  Forms: {_lf["form"].nunique()}  ·  anchor excerpts rated in ≥2 forms: '
      f'{len(ANCHORS)} of {df_long["excerpt_id"].nunique()}')

def _mean_abs_pairwise(vals):
    """Mean |difference| over all pairs of form-means (reduces to |a−b| for two forms)."""
    v = np.asarray([x for x in vals if np.isfinite(x)], dtype=float)
    if len(v) < 2:
        return np.nan
    return float(np.mean([abs(v[i] - v[j]) for i in range(len(v)) for j in range(i + 1, len(v))]))

anchor_rows = []
for eid in ANCHORS:
    sub = _lf[_lf['excerpt_id'] == eid]
    rec = {'excerpt_id': eid, 'n_forms': int(sub['form'].nunique())}
    for dim in DIMS:
        rec[f'{dim}_raw_spread']  = _mean_abs_pairwise(sub.groupby('form')[dim].mean().values)
        rec[f'{dim}_norm_spread'] = _mean_abs_pairwise(sub.groupby('form')[f'{dim}_norm'].mean().values)
    anchor_rows.append(rec)
df_anchor = pd.DataFrame(anchor_rows)

print(f'\n  {"dimension":<10}{"BEFORE (raw)":>15}{"AFTER (norm)":>15}{"change":>12}{"":>4}')
for dim in DIMS:
    before = float(df_anchor[f'{dim}_raw_spread'].mean())
    after  = float(df_anchor[f'{dim}_norm_spread'].mean())
    d      = after - before
    mark   = '✅ shrank' if after < before else '❌ grew'
    pct    = f'({100 * d / before:+.1f}%)' if before else '(n/a)'
    print(f'  {dim:<10}{before:>15.4f}{after:>15.4f}{d:>+12.4f}   {mark}  {pct}')
# expected at N = 116 (reproduced offline, not asserted): valence 0.342 → 0.338,
# arousal 0.276 → 0.253  (N = 118 run: 0.334 → 0.320 / 0.289 → 0.267)
print('\n  Mean |between-form difference| on the anchors. Disjoint panels judged the SAME')
print('  stimulus, so any gap here is rater-panel bias — normalization should shrink it.')

# ── 3. QUADRANT / CLASS STABILITY raw → normalized ──────────────────────────
print('\n' + '═' * 78)
print('3 · CLASS STABILITY (raw-derived label → normalized-derived label)')
print('═' * 78)
_lab_norm = df_agg['emotion_labels'].apply(lambda L: L[0])
_lab_raw  = df_agg['emotion_labels_raw'].apply(lambda L: L[0])
_chg = df_agg.loc[_lab_norm.values != _lab_raw.values, ['excerpt_id', 'condition']].copy()
_chg['from_raw'] = _lab_raw[_lab_norm.values != _lab_raw.values].values
_chg['to_norm']  = _lab_norm[_lab_norm.values != _lab_raw.values].values
# expected at N = 116 (reproduced offline, not asserted): 8 of 60 change class raw → norm
# (N = 118 run: 10 of 60)
print(f'  changed class: {len(_chg)} of {len(df_agg)}')
if len(_chg):
    for t in _chg.itertuples():
        print(f'    {t.excerpt_id:<32} {t.condition}  {t.from_raw:>8} → {t.to_norm}')
_qchg = int((df_agg['quadrant'] != df_agg['quadrant_raw']).sum())
print(f'  (quadrant-only changes, ignoring the Neutral radius: {_qchg} of {len(df_agg)})')

# ── 4. Raw ↔ normalized agreement of the excerpt means ──────────────────────
print('\n' + '═' * 78)
print('4 · RAW ↔ NORMALIZED excerpt-mean agreement')
print('═' * 78)
for dim in DIMS:
    r, _  = pearsonr(df_agg[f'{dim}_mean_raw'], df_agg[f'{dim}_mean'])
    rs, _ = spearmanr(df_agg[f'{dim}_mean_raw'], df_agg[f'{dim}_mean'])
    print(f'  {dim:<10} Pearson r = {r:+.4f}   Spearman ρ = {rs:+.4f}')
print('  Near-1 is expected and healthy: normalization removes rater offset/spread,')
print('  it does not re-order the excerpts.')

# ── 5. Within-excerpt dispersion: did rater noise fall? ─────────────────────
print('\n' + '═' * 78)
print('5 · WITHIN-EXCERPT DISPERSION (mean over the 60 excerpts)')
print('═' * 78)
for dim in DIMS:
    sr = float(df_agg[f'{dim}_std'].mean())
    sn = float(df_agg[f'{dim}_std_norm'].mean())
    print(f'  {dim:<10} raw SD {sr:.4f} → normalized SD {sn:.4f}   '
          f'{"✅ reduced" if sn <= sr else "⚠️ increased"} ({100*(sn-sr)/sr:+.1f}%)')

# ── 6. Neutral count under each estimate ────────────────────────────────────
print('\n' + '═' * 78)
print('6 · NEUTRAL COUNT under each estimate')
print('═' * 78)
def _n_neutral(vcol, acol):
    d = np.hypot(df_agg[vcol] - VA_MID, df_agg[acol] - VA_MID)
    return int((d <= NEUTRAL_RADIUS).sum())
print(f'  raw        : {_n_neutral("valence_mean_raw", "arousal_mean_raw")}')
print(f'  normalized : {_n_neutral("valence_mean", "arousal_mean")}   ← ground truth')
print(f'  crossed-mm : {_n_neutral("valence_mm", "arousal_mm")}')

# ── 7. Robustness: does the z-primary answer survive the crossed model? ─────
print('\n' + '═' * 78)
print('7 · ROBUSTNESS — z-primary vs crossed-additive (rater+excerpt) estimate')
print('═' * 78)
for dim in DIMS:
    rs, _ = spearmanr(df_agg[f'{dim}_mean'], df_agg[f'{dim}_mm'])
    print(f'  {dim:<10} Spearman ρ(normalized, crossed) = {rs:+.4f}')
_agree = int((df_agg['quadrant'] == df_agg['quadrant_mm']).sum())
print(f'  quadrant agreement normalized-vs-crossed: {_agree}/{len(df_agg)} identical')

# ── Per-excerpt report ──────────────────────────────────────────────────────
REPORT_COLS = ['excerpt_id', 'condition',
               'valence_mean_raw', 'arousal_mean_raw',
               'valence_mean', 'arousal_mean',
               'valence_mm', 'arousal_mm',
               'quadrant_raw', 'quadrant', 'quadrant_mm']
df_norm_report = df_agg[REPORT_COLS].copy()
df_norm_report['class_raw']  = _lab_raw.values
df_norm_report['class_norm'] = _lab_norm.values
df_norm_report.to_csv(os.path.join(OUTPUT_DIR, 'normalization_report.csv'), index=False)

# ── Figure: raw vs normalized excerpt means, VA_MID grid ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, dim in zip(axes, DIMS):
    ax.scatter(df_agg[f'{dim}_mean_raw'], df_agg[f'{dim}_mean'],
               s=45, alpha=0.8, color='#2a6dd9', edgecolor='#333', linewidth=0.4)
    lo = float(min(df_agg[f'{dim}_mean_raw'].min(), df_agg[f'{dim}_mean'].min())) - 0.2
    hi = float(max(df_agg[f'{dim}_mean_raw'].max(), df_agg[f'{dim}_mean'].max())) + 0.2
    ax.plot([lo, hi], [lo, hi], '--', color='#bbb', lw=1, label='identity')
    ax.axvline(VA_MID, color='#c44', lw=1, alpha=0.6)
    ax.axhline(VA_MID, color='#c44', lw=1, alpha=0.6)
    r, _ = pearsonr(df_agg[f'{dim}_mean_raw'], df_agg[f'{dim}_mean'])
    ax.set_title(f'{dim}  ·  r = {r:+.4f}', fontsize=11)
    ax.set_xlabel(f'raw excerpt mean ({dim})'); ax.set_ylabel(f'normalized excerpt mean ({dim})')
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.grid(alpha=0.2); ax.legend(fontsize=8)
fig.suptitle('§1.2b  Raw vs rater-normalized excerpt means '
             f'(red lines = VA_MID={VA_MID}; points crossing them changed quadrant)', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_normalization_check.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ §1.2b — normalization_report.csv, fig_normalization_check.png')

## Part 1 · Audience ground truth — §1.3 Label matrix construction
All 60 excerpts are included in classification. `Y` is **(60 × 5)**: the four circumplex
quadrants plus **Neutral**, assigned by proximity to the VA centre — any excerpt whose
mean valence/arousal lies within `NEUTRAL_RADIUS` of the centre becomes Neutral,
independent of condition. This applies uniformly to all 60 excerpts (MEC/EXP/EXG).

The cell asserts that **no two `Y` columns are byte-identical**, which is the check that
the old "8 emotions" fiction (3 duplicated column pairs) is really gone.


In [ ]:
# ── §1.3 Label matrix construction ──────────────────────────────────
# NEEDS : df_agg, EMOTION_LABELS, QUADRANT_DISPLAY
# MAKES : df_all, df_expr, mlb, Y
# FEEDS : §2.3, §2.3b, §2.5, §3.1, §3.2
df_expr = df_agg.reset_index(drop=True)   # ALL 60 excerpts — no neutral gate
df_all  = df_agg.copy()

# classes=EMOTION_LABELS keeps the Y column order fixed and reproducible.
mlb = MultiLabelBinarizer(classes=EMOTION_LABELS)
Y   = mlb.fit_transform(df_expr['emotion_labels'])

print(f'Total excerpts (all conditions): {len(df_expr)}')
print(f'  MEC : {(df_expr["condition"]=="MEC").sum()}')
print(f'  EXP : {(df_expr["condition"]=="EXP").sum()}')
print(f'  EXG : {(df_expr["condition"]=="EXG").sum()}')
print(f'Y shape: {Y.shape}  |  classes: {list(mlb.classes_)}')
print('\nClass frequency:')
for lbl, n in zip(mlb.classes_, Y.sum(axis=0)):
    print(f'  {lbl:<8} {QUADRANT_DISPLAY[lbl]:<30} {int(n):>3}  ({100*n/len(Y):.0f}%)')

# ── Verify the 8→5 collapse actually removed the duplicated columns ───────────
dupes = [(mlb.classes_[i], mlb.classes_[j])
         for i in range(Y.shape[1]) for j in range(i + 1, Y.shape[1])
         if np.array_equal(Y[:, i], Y[:, j])]
print(f'\nByte-identical label columns: {dupes if dupes else "none ✅ (8→5 collapse verified)"}')


## Part 2 · Feature extraction — §2.1 MERT (SSL embedding, 768-dim)
**Question of this part:** what can we measure in the signal itself?  
**Hands to the next part:** `emb_mert`, `df_crepe`, `df_essentia` / `ESSENTIA_FORWARD`, `df_dtw` / `DTW_FORWARD`, and the feature matrices `X_A`–`X_D`, `X_F`.

Music-specific self-supervised transformer. CQT teacher captures violin pitch micro-variations.  
Layers 5–7 are best for emotion-related tasks (per MERT paper probing experiments).

> **Phase A fix (A3).** The remote `modeling_MERT.py` names the positional-convolution weight-norm
> parameters `…parametrizations.weight.original0/1`, the checkpoint stores `…weight_g/_v`, and nothing
> renamed them — so in every earlier run the positional convolution had **random** weights. The cell
> now restores both tensors from the checkpoint and asserts they equal it; the cache is
> `emb_mert_v2.npz`, so pre-fix embeddings can never be reused. Every earlier MERT-based number is
> superseded.


In [ ]:
# ── §2.1 MERT SSL embedding ─────────────────────────────────────────
# NEEDS : audio_files, DEVICE, MERT_LAYERS, MERT_MODEL_ID, OUTPUT_DIR, SAMPLE_RATE, SEED
# MAKES : CACHE_MERT, emb_mert, extract_mert, mert_model, mert_proc, MERT_POSCONV_KEYS
#          · emb_mert_v2.npz
# FEEDS : §2.1b, §7.4
import huggingface_hub

print('Loading MERT-v1-95M...')
torch.manual_seed(SEED)                       # reproducibility of anything initialised at load
mert_proc  = AutoProcessor.from_pretrained(MERT_MODEL_ID, trust_remote_code=True)
mert_model, _mert_load_info = AutoModel.from_pretrained(
    MERT_MODEL_ID, trust_remote_code=True, output_hidden_states=True,
    output_loading_info=True)

# ── Positional-conv weight-norm restore (Phase A · A3) ──────────────────────
# The remote modeling_MERT.py builds pos_conv_embed with the NEW torch weight-norm
# parametrization (…parametrizations.weight.original0 / original1), but the checkpoint
# stores the OLD names (…weight_g / …weight_v) and nothing renames them. In the
# 2026-08-16 run both checkpoint tensors were "not used" and both model parameters were
# "newly initialized" — MERT's positional convolution ran with RANDOM weights. Every
# MERT-based number from before this fix is superseded. Restore them from the checkpoint
# and assert they now equal it exactly. Any OTHER missing/unexpected key raises.
_PC = 'encoder.pos_conv_embed.conv.'
MERT_POSCONV_KEYS = {_PC + 'weight_g': _PC + 'parametrizations.weight.original0',
                     _PC + 'weight_v': _PC + 'parametrizations.weight.original1'}
_missing    = set(_mert_load_info.get('missing_keys', []))
_unexpected = set(_mert_load_info.get('unexpected_keys', []))
_bad_miss   = _missing - set(MERT_POSCONV_KEYS.values())
_bad_unexp  = _unexpected - set(MERT_POSCONV_KEYS)
if _bad_miss or _bad_unexp or _mert_load_info.get('mismatched_keys') or _mert_load_info.get('error_msgs'):
    raise RuntimeError(
        f'§2.1: unexpected MERT loading state — a different problem than the pos-conv rename.\n'
        f'   other missing    : {sorted(_bad_miss)}\n   other unexpected : {sorted(_bad_unexp)}\n'
        f'   mismatched       : {_mert_load_info.get("mismatched_keys")}\n'
        f'   error_msgs       : {_mert_load_info.get("error_msgs")}')
print(f'   loading info: missing={sorted(_missing)}  unexpected={sorted(_unexpected)}')

_ckpt_path = huggingface_hub.hf_hub_download(MERT_MODEL_ID, 'pytorch_model.bin')   # cache hit
_ckpt = torch.load(_ckpt_path, map_location='cpu', weights_only=True)
_absent = [k for k in MERT_POSCONV_KEYS if k not in _ckpt]
if _absent:
    raise KeyError(f'§2.1: checkpoint lacks {_absent}; pos_conv keys present: '
                   f'{[k for k in _ckpt if "pos_conv_embed" in k]}')

_params, _restored = dict(mert_model.named_parameters()), []
with torch.no_grad():                         # BEFORE .to(DEVICE) / .eval()
    for _ck, _mk in MERT_POSCONV_KEYS.items():
        if _mk not in _missing:
            continue                          # loaded correctly → checked below, not copied
        if _mk not in _params:
            raise KeyError(f'§2.1: {_mk} reported missing but is not a model parameter')
        _p, _t = _params[_mk], _ckpt[_ck]
        assert tuple(_p.shape) == tuple(_t.shape), \
            f'§2.1: shape mismatch {_mk} {tuple(_p.shape)} vs checkpoint {_ck} {tuple(_t.shape)}'
        _p.copy_(_t.to(_p.dtype))
        _restored.append(_mk)

# Check (not a fallback): whatever path loaded them, the weights must equal the checkpoint.
_sd = mert_model.state_dict()
for _ck, _mk in MERT_POSCONV_KEYS.items():
    _key = _mk if _mk in _sd else (_ck if _ck in _sd else None)
    if _key is None:
        raise KeyError(f'§2.1: neither {_mk} nor {_ck} is in the model state_dict')
    assert torch.equal(_sd[_key].cpu(), _ckpt[_ck].to(_sd[_key].dtype)), \
        f'§2.1: {_key} does not equal checkpoint {_ck} after loading'
if _restored:
    print('✅ MERT pos_conv weight-norm parameters restored from checkpoint '
          f'({", ".join(k.split(".")[-1] for k in _restored)})')
else:
    print('✅ MERT pos_conv weights were loaded by transformers itself — verified equal to the '
          'checkpoint (no copy needed)')
del _ckpt, _sd, _params

mert_model = mert_model.to(DEVICE)
mert_model.eval()
for p in mert_model.parameters(): p.requires_grad = False
print(f'✅ MERT loaded ({sum(p.numel() for p in mert_model.parameters())/1e6:.1f}M params)')

def extract_mert(audio_path):
    y, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    inp   = mert_proc(y, sampling_rate=SAMPLE_RATE, return_tensors='pt', padding=True)
    inp   = {k: v.to(DEVICE) for k,v in inp.items()}
    with torch.no_grad():
        out = mert_model(**inp)
    return np.mean([out.hidden_states[l].mean(1).squeeze(0).cpu().numpy() for l in MERT_LAYERS], axis=0)

# v2 = embeddings extracted AFTER the pos-conv restore. The pre-fix cache (emb_mert.npz)
# was computed with a randomly initialised positional conv and must never be reused.
CACHE_MERT = os.path.join(OUTPUT_DIR, 'emb_mert_v2.npz')
if os.path.exists(CACHE_MERT):
    print('📂 Loading cached MERT embeddings (v2, post pos-conv fix)')
    emb_mert = {str(k): v for k,v in np.load(CACHE_MERT, allow_pickle=True).items()}
else:
    print('🔄 Extracting MERT embeddings (~10 min on T4)...')
    emb_mert = {}
    for eid, fp in tqdm(audio_files.items(), desc='MERT'):
        try: emb_mert[eid] = extract_mert(fp)
        except Exception as e: print(f'  ⚠️  {eid}: {e}')
    np.savez(CACHE_MERT, **emb_mert)
    print(f'✅ Saved: {CACHE_MERT}')
print(f'MERT: {len(emb_mert)} excerpts  dim={list(emb_mert.values())[0].shape[0]}')


In [ ]:
# ── §2.1b Controlled MERT embedding (UNSUPERVISED) ──────────────────
# NEEDS : emb_mert, OUTPUT_DIR
# MAKES : df_mert, emb_mert_fwd, MERT_FORWARD, MERT_PCA_VAR · mert_forwarding.csv
# FEEDS : §2.5
# ─── UNSUPERVISED INVARIANT ──────────────────────────────────────────────────
# This cell reads ONLY the feature matrix. It never touches Y, y_*, or any label.
# That is precisely what makes a fixed, once-up-front selection safe to feed into
# the downstream LOOCV models: a supervised selection here would leak the held-out
# fold into the choice of features. StandardScaler stays INSIDE every CV pipeline.
# ═══════════════════════════════════════════════════════════════════════════════
# §7B · Controlled MERT embedding — reduce 768 dims to the necessary subset
# ═══════════════════════════════════════════════════════════════════════════════
# No tool is dumped raw into the models. MERT gets the same controlled forwarding
# gate as CREPE (§2.2b) and Essentia (§2.3b) — only the METHOD differs, because at
# 768 dims a pairwise-correlation prune barely reduces anything.
df_mert = pd.DataFrame(emb_mert).T                      # 60 × 768
df_mert.columns = [f'mert_{d:03d}' for d in range(df_mert.shape[1])]
df_mert.index.name = 'excerpt_id'

# 768 >> N=60 → unsupervised PCA retaining ~95% variance (leakage-free dimensionality
# reduction). NOTE: PCA is fit once on all 60 excerpts; this is a mild, standard
# approximation for LOOCV. A stricter per-fold PCA option is noted below.
from sklearn.decomposition import PCA
MERT_PCA_VAR = 0.95
_pca = PCA(n_components=MERT_PCA_VAR, svd_solver='full').fit(df_mert.values)
Z = _pca.transform(df_mert.values)
MERT_FORWARD = [f'mertpc_{i:02d}' for i in range(Z.shape[1])]
df_mert_fwd  = pd.DataFrame(Z, index=df_mert.index, columns=MERT_FORWARD)
emb_mert_fwd = {e: df_mert_fwd.loc[e].values for e in df_mert_fwd.index}
print(f'MERT: 768 → {Z.shape[1]} PCA comps (≥{MERT_PCA_VAR:.0%} var, '
      f'{_pca.explained_variance_ratio_.sum():.3f} explained)')
df_mert_fwd.to_csv(os.path.join(OUTPUT_DIR, 'mert_forwarding.csv'))

# ALTERNATIVE (commented): keep raw dims instead of PCA components —
#   MERT_FORWARD = unsup_forward_select(df_mert, corr_max=0.90, name='mert')
#   df_mert_fwd  = df_mert[MERT_FORWARD]
#   → same gate as the other tools, but a pairwise prune barely reduces 768 dims.

# emb_mert (the FULL dict) is deliberately left intact: §7.4 embedding distance and
# §7.3/§7.3b correspondence measure the REPRESENTATION, not a model input, so they
# must see the full space.


## Part 2 · Feature extraction — §2.1c Per-layer MERT embeddings (Phase C1 · label-free cache)

**Why.** The supervisors asked for a MERT per-layer study. The deployed embedding (§2.1) is the time-mean of
`hidden_states[5]`, `[6]` and `[7]` averaged over the three layers, and only that average is cached
(`emb_mert_v2.npz`), so no other layer could be read. This cell runs **one more forward pass per excerpt,
exactly as `extract_mert`** (same `librosa` loading at `SAMPLE_RATE`, the same processor, device and the
pos-conv-restored model) and keeps the time-mean of **every** hidden state: MERT-v1-95M exposes 13
(`0` = the input to the transformer, `1`–`12` = the transformer layers) → a `(13, 768)` float32 array per
excerpt, cached in `emb_mert_layers_v3.npz`.

**Nothing deployed changes.** `extract_mert`, `emb_mert`, `emb_mert_v2.npz`, §2.1b (`X_A`), `X_MERT_FULL`
and `MERT_REG` are untouched; only the **diagnostic** layer study (§7.2) reads the new cache, and
`MERT_LAYERS` stays `[5, 6, 7]` until the author decides by hand.

**Consistency check.** For every excerpt the mean of the `MERT_LAYERS` rows must equal `emb_mert[eid]`
(the max |diff| is printed; `np.allclose(rtol=1e-3, atol=1e-4)` is asserted). It is computed with the same
per-layer expression as `extract_mert`, so it is exactly 0 when both come from the same session's forward
passes; if either cache came from an earlier session, GPU float32 kernels can differ slightly, which the
tolerance covers. **Label-free** (Part 2 rule): audio only — no rating, `Y` or label is read.
**Runtime:** one forward pass per excerpt (the B3 run extracted `emb_mert` in ≈ 30 s on a T4); cached
afterwards.

In [ ]:
# ── §2.1c Per-layer MERT embeddings (Phase C1 · label-free cache for the §7.2 layer study) ──
# NEEDS : audio_files, CACHE_MERT, DEVICE, emb_mert, librosa, MERT_LAYERS, MERT_MODEL_ID, mert_model,
#          mert_proc, OLD_CACHE_NAMES, OUTPUT_DIR, SAMPLE_RATE, torch, tqdm
# MAKES : CACHE_MERT_LAYERS, emb_mert_layers, extract_mert_layers, MERT_N_HIDDEN, _mlc_* ·
#          emb_mert_layers_v3.npz
# FEEDS : §7.2 (the DIAGNOSTIC MERT layer study) — nothing else reads it. extract_mert, emb_mert,
#          emb_mert_v2.npz, §2.1b (X_A), X_MERT_FULL and MERT_REG are unchanged.
# ─── LABEL-FREE (Part 2 rule): audio only — never a rating, Y or any label. ───
MERT_N_HIDDEN = 13     # MERT-v1-95M: hidden_states[0] = input to the transformer, [1]..[12] = the 12 layers


def extract_mert_layers(audio_path):
    """ONE forward pass exactly as extract_mert (§2.1 — same librosa loading at SAMPLE_RATE, processor,
    device and pos-conv-restored model) → (MERT_N_HIDDEN, 768) float32: the time-mean of EVERY hidden
    state, each row computed with extract_mert's own expression, so the mean of the MERT_LAYERS rows is
    what extract_mert returns."""
    y, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    inp   = mert_proc(y, sampling_rate=SAMPLE_RATE, return_tensors='pt', padding=True)
    inp   = {k: v.to(DEVICE) for k, v in inp.items()}
    with torch.no_grad():
        out = mert_model(**inp)
    assert len(out.hidden_states) == MERT_N_HIDDEN, \
        f'§2.1c: expected {MERT_N_HIDDEN} hidden states from {MERT_MODEL_ID}, got {len(out.hidden_states)}'
    return np.stack([out.hidden_states[l].mean(1).squeeze(0).cpu().numpy()
                     for l in range(MERT_N_HIDDEN)]).astype(np.float32, copy=False)


# v3 = a NEW cache name (the deployed emb_mert_v2.npz is neither replaced nor invalidated)
CACHE_MERT_LAYERS = os.path.join(OUTPUT_DIR, 'emb_mert_layers_v3.npz')
assert os.path.basename(CACHE_MERT_LAYERS) not in OLD_CACHE_NAMES, '§2.1c: retired cache name'
assert os.path.basename(CACHE_MERT_LAYERS) != os.path.basename(CACHE_MERT), \
    '§2.1c: the per-layer cache must never overwrite the deployed emb_mert_v2.npz'
if os.path.exists(CACHE_MERT_LAYERS):
    print(f'📂 Loading cached per-layer MERT embeddings ({os.path.basename(CACHE_MERT_LAYERS)})')
    emb_mert_layers = {str(k): v for k, v in np.load(CACHE_MERT_LAYERS, allow_pickle=True).items()}
    _mlc_src = f'loaded from {os.path.basename(CACHE_MERT_LAYERS)} (an earlier extraction)'
else:
    print(f'🔄 Extracting per-layer MERT embeddings — {len(emb_mert)} excerpts, one forward pass each '
          '(the B3 run extracted emb_mert in ≈ 30 s on a T4)...')
    emb_mert_layers = {}
    for _mlc_e in tqdm(list(emb_mert), desc='MERT layers'):      # same excerpts, same order as emb_mert
        emb_mert_layers[_mlc_e] = extract_mert_layers(audio_files[_mlc_e])   # a failure raises (no skip)
    np.savez(CACHE_MERT_LAYERS, **emb_mert_layers)
    print(f'✅ Saved: {CACHE_MERT_LAYERS}')
    _mlc_src = 'extracted in this session'

# ── Shape / coverage checks ─────────────────────────────────────────────────────
assert set(emb_mert_layers) == set(emb_mert), \
    (f'§2.1c: per-layer cache and emb_mert cover different excerpts — only in layers: '
     f'{sorted(set(emb_mert_layers) - set(emb_mert))}, only in emb_mert: {sorted(set(emb_mert) - set(emb_mert_layers))}')
_mlc_bad = {e: tuple(v.shape) for e, v in emb_mert_layers.items()
            if tuple(v.shape) != (MERT_N_HIDDEN, int(np.shape(emb_mert[e])[0]))}
assert not _mlc_bad, f'§2.1c: per-layer arrays must be ({MERT_N_HIDDEN}, 768): {_mlc_bad}'

# ── Consistency: mean of the MERT_LAYERS rows == the deployed emb_mert (extract_mert's own np.mean) ──
_mlc_diff = {e: float(np.max(np.abs(np.mean([emb_mert_layers[e][l] for l in MERT_LAYERS], axis=0)
                                    - np.asarray(emb_mert[e]))))
             for e in emb_mert}
_mlc_worst = max(_mlc_diff, key=_mlc_diff.get)
print(f'   per-layer cache: {_mlc_src}; {len(emb_mert_layers)} excerpts × {MERT_N_HIDDEN} hidden states × '
      f'{int(np.shape(emb_mert[_mlc_worst])[0])} dims')
print(f'   consistency: mean of hidden states {list(MERT_LAYERS)} vs emb_mert (§2.1, '
      f'{os.path.basename(CACHE_MERT)}): max |diff| = {_mlc_diff[_mlc_worst]:.3e} ({_mlc_worst}); '
      f'{sum(v == 0.0 for v in _mlc_diff.values())}/{len(_mlc_diff)} excerpts bit-identical')
print('   ℹ️  0 is expected when both caches come from forward passes of THIS session; when either was written by an '
      'earlier session (cached file), GPU float32 kernels can differ slightly — covered by rtol=1e-3, atol=1e-4')
assert all(np.allclose(np.mean([emb_mert_layers[e][l] for l in MERT_LAYERS], axis=0), np.asarray(emb_mert[e]),
                       rtol=1e-3, atol=1e-4) for e in emb_mert), \
    (f'§2.1c: the mean of hidden states {list(MERT_LAYERS)} differs from emb_mert beyond rtol=1e-3 / atol=1e-4 '
     f'(max |diff| {_mlc_diff[_mlc_worst]:.3e} at {_mlc_worst}; per-layer cache {_mlc_src}) — the two '
     'extractions are not the same forward pass')
print(f'✅ §2.1c — emb_mert_layers ready (label-free; read only by the diagnostic §7.2); '
      f'deployed MERT_LAYERS = {list(MERT_LAYERS)} unchanged')

## Part 2 · Feature extraction — §2.2 CREPE · pitch contour / vibrato / portamento

**Phase B (B-shared, B8–B10) + Phase B2 (descriptor repairs after the manual audit of 2026-11-05) +
Phase B3 (after the full B2 run: re-articulation detector disabled, merge range check, vibrato band
edge).** CREPE runs once per excerpt with the unchanged settings (10 ms step, `viterbi=True`, voiced =
confidence > 0.5) and its tracks (`time, f0_hz, confidence, voiced`) are cached to
`crepe_tracks_v2.npz`; a 10 ms RMS envelope of the same 16 kHz mono audio (25 ms rectangular window
centred on each CREPE frame) is cached to `rms10_v3.npz` (since B3 it only feeds the report-only
evaluation of the re-articulation detector); every pitch descriptor is recomputed from the tracks on
each run (`feat_crepe_v4.csv`). The helpers live in the `# >>> HELPERS:descriptors` block:

- **octave correction** — a voiced frame 1000–1400 cents from its ±100 ms running median is shifted
  by ∓1200 (corrections printed per excerpt);
- **note segmentation (B2 / B3)** — per voiced run (never across an unvoiced gap): **plateaus** of the
  150 ms median trend (every frame within ±`PLATEAU_TOL_CENTS` = 40 c of the plateau's running median,
  range ≤ 80 c, ≥ 60 ms); adjacent plateaus < 50 c apart and ≤ 50 ms apart are merged **only if the
  merged note, gap frames included, keeps a trend range ≤ 80 c** (B3 — B2 merged Méditation EXG C♯6
  into a 128 c note). Frames between plateaus are transitions. **No re-articulation split**
  (`REART_ENABLED = False`, B3): the B2 detector split a note at ≥ 6 dB dips of the 10 ms envelope, but
  on the violin vibrato also modulates the amplitude, so the splits followed the vibrato (MEC 24 / EXP 51
  / EXG 68 splits for the same scores; Méditation EXG F♯5 cut into 7 pieces of 1–2 vibrato periods) and
  inflated the note-based timing features with expressiveness. The known cost — a repeated same-pitch
  note without a silence is one note — is set by the score and ≈ equal in the three renditions of a
  piece. The detector is kept as a **report-only negative result**: §2.2 prints the note totals and
  split counts per condition with the detector off (deployed) vs on at 6 dB, the per-excerpt counts and
  the `REART_DIP_DB` ∈ {3, 6, 9} sensitivity with Mozart VC4 (19 notes in the score). The Phase B
  boundary rule (> 70 c trend move over 5 frames) is kept as `_segment_notes_v2`, report only;
- **condition-independence check (B3, label-free, report only)** — the same score is played in every
  condition, so the deployed note count should not depend on it: per-condition totals, the per-piece
  range (max − min over MEC / EXP / EXG) and a Friedman test on the 20 × 3 count table (deployed, and the
  detector-on counts for comparison); ✅ if the deployed p ≥ 0.05, ⚠️ otherwise (printed, never raised);
- **vibrato (B8b + B3)** — per note ≥ `VIB_MIN_NOTE_MS` = 500 ms (sensitivity 500 / 650 / 800 printed):
  500 ms windows every 250 ms (the last window shifted to end at the note end), degree-2 detrend,
  least-squares sinusoid a·sin + b·cos over 4–8 Hz in 0.05 Hz steps, R² of the fit, rate at the R²
  maximum (parabolic refinement clamped to the grid), half-extent = √(a² + b²). A window is vibrato
  iff its best rate (argmax of R² before refinement) is **not the first or last grid point** (B3 — the
  best fit then lies outside the band: a residual slow trend, not vibrato; the B2 run had an excerpt at
  exactly 4.00 Hz), R² ≥ 0.50 and half-extent ≥ 3 c; every window records its `reason` (band edge /
  low R² / small extent / vibrato) and §2.2 prints how many windows the edge rule rejected (all
  excerpts and the 20 MEC). `crepe_vibrato_coverage` = vibrato-window time (union) /
  analysed-note time; `_rate_hz` / `_extent_cents` = duration-weighted medians over the vibrato
  windows. Declared median substitutions (no vibrato window → rate undefined; no qualifying note →
  rate / extent / coverage undefined) are counted and printed; ⚠️ if more than 10 excerpts have no
  qualifying note. The Phase B FFT detector stays as `vibrato_note` (legacy, report only — pending decision: keep or remove, Phase C2);
- **portamento (B9b)** — transitions between consecutive notes (≥ 60 ms) of one voiced run. crepe's
  Viterbi decoder moves at most 11 bins × 20 c = 220 c per frame (`CREPE_VITERBI_MAX_CENTS_PER_FRAME`,
  checked against the installed `crepe.core.to_viterbi_cents`, printed), so a pure jump of interval I
  already shows a ramp of t_min = 0.8·|I| / 220 · 10 ms × 1.25. A glide needs a 10–90 % rise (× 1.25)
  of 80–500 ms, an **excess** rise − t_min ≥ 50 ms, |I| 80–2000 c and ≥ 80 % of its raw-cent steps in
  the interval's sign → `crepe_portamento_rate` / `_share` / `_mean_ext_cents`. Verdicts per transition:
  interval out of range · no 90 % crossing · too short · viterbi-limited jump · too slow · not monotone ·
  across an unvoiced gap · glide;
- **F0 (B10)** — on the corrected track: range = P95 − P5 of voiced cents, CV, mean Hz;
- **calibration (report only)** — MEC is the **negative control** (played with no vibrato and no
  portamento by instruction): the false-positive rate of vibrato windows over R² ∈ {0.3 … 0.7} ×
  half-extent ∈ {2, 3, 4, 5} c (every cell applies the B3 band-edge rule) and of glides over
  `GLIDE_MIN_EXCESS_MS` ∈ {25, 50, 75, 100} ms is printed, ⚠️ if the deployed setting exceeds 5 %.
  Audio-derived detections only — never a rating, never `Y`; nothing is chosen automatically.

`crepe_vibrato_n_notes` is diagnostic only (`CREPE_DIAGNOSTIC`) and is never forwarded.

In [ ]:
# ── §2.2 CREPE pitch features ───────────────────────────────────────
# NEEDS : audio_files, crepe, df_expr, OUTPUT_DIR
# MAKES : CACHE_CREPE, CACHE_CREPE_TRACKS, CACHE_CREPE_ENV, CREPE_TRACKS, CREPE_ENV_DB, CREPE_CENTS,
#          CREPE_VOICED, CREPE_DIAG, CREPE_CANDIDATES, CREPE_DIAGNOSTIC, CREPE_LABELS, df_crepe,
#          VIB_N_SUBST_NONOTES, VIB_N_SUBST_NOVIB, CREPE_VITERBI_SOURCE_OK, CREPE_REART_REPORT,
#          CREPE_NOTE_COUNTS, CREPE_NOTECOUNT_FRIEDMAN, + the HELPERS:descriptors functions/constants
#          (REART_ENABLED, crepe_descriptors, segment_notes, segment_notes_detail, _segment_notes_v2,
#          _reart_splits, env_db_on_grid, vibrato_window_fit, vibrato_features, vibrato_note (legacy),
#          portamento_features, note_onset_times, timing_note_features, local_tempo_from_tempogram,
#          l1_normalize_spectrum, dr_gate, frame_power_spectra, spectral_flatness, …)
#          · crepe_tracks_v2.npz, rms10_v3.npz, feat_crepe_v4.csv
# FEEDS : §2.2b, §2.3 (flux / gate helpers), §2.3c (flatness), §2.4 (tracks, env, onsets, tempogram
#          read-out), §3.3, §4.1, §7.3, §7.3b, §7.5 (VIB_N_SUBST_*), §10.2 (the §2.6 audit that also read
#          these helpers was removed in Phase C2)
# ═══════════════════════════════════════════════════════════════════════════════
# §2.2 · CREPE — tracks cached once, every pitch descriptor computed from them (Phase B / B2 / B3)
# ═══════════════════════════════════════════════════════════════════════════════
# CREPE settings are UNCHANGED (10 ms step, viterbi=True, voiced = confidence > 0.5); the tracks
# (time, f0_hz, confidence, voiced) stay cached in crepe_tracks_v2.npz. Phase B2 (after the
# 2026-11-05 manual audit, the former §2.6) repaired the note segmentation (trend plateaus), the portamento rule
# (Viterbi-aware) and the vibrato detector (windowed sinusoid fit). Phase B3 (after the full B2 run)
# DISABLES the B2 re-articulation splits for the features (REART_ENABLED = False — they followed the
# vibrato amplitude modulation, not bow changes; the detector stays as a report-only negative result
# on the cached 10 ms envelope rms10_v3.npz), re-checks the plateau range after a merge, and rejects
# vibrato windows whose best rate sits on the edge of the 4–8 Hz grid. The vibrato, portamento,
# F0-range and timing descriptors all come from the same corrected tracks.
# >>> HELPERS:descriptors
# Every CREPE-derived descriptor (vibrato B8/B8b, portamento B9/B9b, F0 B10) and the note onsets of
# the timing family (B13) are computed from the cached CREPE tracks with these functions (the 10 ms RMS
# envelope is still accepted everywhere, but only used when the re-articulation detector is enabled);
# §2.3 / §2.3c reuse the spectrum / gate / flatness helpers. Self-contained (numpy + scipy + the
# standard-library bisect) so it can be extracted and tested outside the notebook.
# RULE: frames are NEVER concatenated across unvoiced gaps — every per-note computation works inside
# one voiced run (the core bug of the pre-Phase-B descriptors).
import bisect
import numpy as np
from scipy.ndimage import median_filter
from scipy.signal import butter, filtfilt, hilbert

CREPE_STEP_MS      = 10          # CREPE hop (unchanged) → 100 frames / s
CREPE_FS           = 1000.0 / CREPE_STEP_MS
CREPE_CONF_MIN     = 0.5         # voiced = confidence > 0.5 (unchanged)
OCT_WIN            = 10          # octave fix: running median over ±10 frames (±100 ms) of voiced frames
OCT_DEV_RANGE      = (1000.0, 1400.0)   # |cents − running median| in this range → octave error → ∓1200
TREND_WIN          = 15          # vibrato-free trend = rolling median over 15 frames (150 ms)
# ── B2 note segmentation: plateaus of the trend (+ the re-articulation detector, DISABLED in B3) ──
PLATEAU_TOL_CENTS  = 40.0        # a plateau grows while |trend[t] − median(trend[plateau])| ≤ 40 c …
PLATEAU_MAX_RANGE_CENTS = 2 * PLATEAU_TOL_CENTS   # … and its trend range stays ≤ 80 c (stops a slow
                                 # ramp from passing as a plateau — the running median follows a ramp);
                                 # B3: a MERGED note (incl. its gap frames) must also stay ≤ 80 c
PLATEAU_MERGE_CENTS = 50.0       # adjacent plateaus < 50 c apart …
PLATEAU_MERGE_GAP_MS = 50.0      # … and ≤ 50 ms apart are ONE note (vibrato / intonation wobble)
# Re-articulation detector (B2) — DISABLED for every feature since Phase B3 (REART_ENABLED = False).
# Evidence of the full B2 run: on the violin, vibrato also modulates the amplitude (the frequency sweeps
# through the body resonances), the 25 ms envelope resolves that modulation, and the 6 dB dip rule took
# it for a bow re-attack. Splits per condition MEC 24 / EXP 51 / EXG 68 for the same scores;
# Meditation_Thais 0 / 3 / 11 (the EXG F♯5 cut into 7 pieces of 140–370 ms = 1–2 vibrato periods);
# Shostakovich_Sym5_Mvt3_EXG 11; note totals per condition 616 / 596 / 592 without the splits vs
# 640 / 647 / 660 with them; no REART_DIP_DB fixes Mozart_VC4 (score 19: 26/23/28 at 3 dB, 22/21/23 at
# 6 dB, 19/17/22 at 9 dB). The split count grows with expressiveness, so it fed the note-based timing
# features (§4.2 note rate / IOI turned significant, §10.2 Timing alone F1 0.417 → 0.542) and cut long
# EXG vibrato notes below the 500 ms analysis minimum. Without the detector a repeated same-pitch note
# with no silence between is one note — a score-set bias, ≈ equal in the three renditions of a piece,
# so it cancels within piece; the vibrato artefact did not. The detector (_reart_splits, env_db_on_grid,
# the constants below) is KEPT so §2.2 can still report it (segment_notes_detail(…, reart_enabled=True)).
REART_ENABLED      = False
REART_DIP_DB       = 6.0         # re-articulation: an RMS dip of ≥ 6 dB inside a plateau …
REART_PRE_MS       = 150.0       # … below the max of the preceding 150 ms …
REART_POST_MS      = 60.0        # … and below the max of the following 60 ms splits the note
REART_SENS_DB      = (3.0, 6.0, 9.0)   # report-only sensitivity of REART_DIP_DB (§2.2)
ENV_WIN_MS         = 25.0        # env_db: RECTANGULAR 25 ms window, centred on the CREPE frame times
NOTE_MIN_MS_TIMING = 60          # notes for timing onsets (B13) and portamento transitions (B9);
                                 # also the minimum plateau / re-articulation part length
BOUNDARY_LAG       = 5           # LEGACY (_segment_notes_v2, report only): boundary where
BOUNDARY_CENTS     = 70.0        # |trend[t] − trend[t−5]| > 70 c; still the n_close_splits threshold
# ── B8b vibrato: windowed least-squares sinusoid fit ──
VIB_MIN_NOTE_MS    = 500         # notes analysed for vibrato (Phase B: 800; B2 audit: narrow vibrato
                                 # on 0.5–0.8 s notes was invisible)
VIB_NOTE_SENS_MS   = (500, 650, 800)    # report-only sensitivity of the vibrato note length
VIB_WIN_MS         = 500         # analysis window …
VIB_HOP_MS         = 250         # … and hop; the last window is shifted to end at the note end
VIB_RATE_BAND      = (4.0, 8.0)  # rate grid of the sinusoid fit (Hz) …
VIB_RATE_STEP      = 0.05        # … step; the parabolic refinement is CLAMPED to the grid
VIB_FIT_R2_MIN     = 0.50        # vibrato window: R² of the sinusoid fit ≥ 0.50 …
VIB_MIN_EXTENT     = 3.0         # … and half-extent ≥ 3 c (Phase B: 5 c rejected the author-confirmed
                                 # narrow vibrato of Meditation_Thais_EXP D5 / B4, ≈ 4.2–4.4 c)
# The FFT PRESENCE test of Phase B (power within ±0.5 Hz of the peak / power in [1, 15] Hz ≥ 0.30 on
# a Hann-windowed, zero-padded note spectrum) is REPLACED for the descriptors: the Hann main lobe is
# ±2/T Hz wide, so at T = 0.5 s (±4 Hz) a ±0.5 Hz ratio cannot separate vibrato from noise (clean
# vibrato ≈ 0.49 at 0.8 s; noise ≈ 0.15–0.17, P95 ≈ 0.27, measured offline); the time-domain R² of a
# fitted sinusoid works on 0.5 s windows. Its constants are kept ONLY for the report-only legacy
# helper vibrato_note (whose parabolic peak could also step outside the band at the band edge — the
# out-of-band rates 1.5 / 2.4 / 2.5 Hz the audit found; the new fit clamps to the grid).
VIB_BAND           = (3.0, 10.0)        # legacy: rate search band and band-pass (Hz)
VIB_DENOM_BAND     = (1.0, 15.0)        # legacy: presence denominator band (Hz)
VIB_NFFT           = 4096               # legacy: zero-padded FFT at 100 Hz → 0.024 Hz bins
VIB_PEAK_HALFWIDTH = 0.5                # legacy: presence numerator ±0.5 Hz around the peak
VIB_PRESENCE_MIN   = 0.30               # legacy presence threshold
VIB_LEGACY_MIN_EXTENT = 5.0             # legacy half-extent threshold (Phase B value)
# ── B9b portamento: Viterbi-aware glide rule ──
# crepe's Viterbi decoder (crepe/core.py, to_viterbi_cents) uses the transition weight
# max(12 − |Δbin|, 0) over 360 bins of 20 cents: the decoded pitch moves at most 11 bins = 220 c per
# 10 ms frame, so ANY jump is forced into a ramp of ≥ |interval| / 220 frames (an octave ≥ ~55 ms).
# A glide must therefore last clearly longer than that forced ramp (§2.2 checks the source at runtime).
CREPE_VITERBI_MAX_CENTS_PER_FRAME = 220.0
GLIDE_DUR_MS       = (80.0, 500.0)      # glide = 10–90 % rise × 1.25 of 80–500 ms (Phase B: 40–400) …
GLIDE_INTERVAL     = (80.0, 2000.0)     # … over an interval of 80–2000 cents …
GLIDE_SIGN_MIN     = 0.80               # … with ≥ 80 % of frame diffs carrying the interval's sign …
GLIDE_RISE_SCALE   = 1.25               # (transition duration = 10–90 % rise time of the trend × 1.25)
GLIDE_MIN_EXCESS_MS = 50.0              # … and ≥ 50 ms longer than the Viterbi-forced ramp t_min
TEMPO_PRIOR_CENTER = 100.0              # BPM — log-normal tempo prior exp(−½·log2(bpm/100)²)
TEMPO_BPM_RANGE    = (40.0, 200.0)
OCT_JUMP_TOL       = 0.1                # consecutive local tempi with |log2 ratio| ∈ 1 ± 0.1 = octave jump
TEMPO_INTERIOR_ONLY = True              # statistics over frames whose whole window lies inside the
                                        # excerpt: a centred tempogram pads the onset envelope with a
                                        # ramp to zero, and the edge frames (half a window at each end)
                                        # pick spurious tempi — a steady 100-BPM click gives log2-SD
                                        # ≈ 0.09 with edges vs 0.0 on interior frames (offline test)
TEMPO_MIN_INTERIOR = 20                 # interior frames required (the shortest excerpt, 5.4 s, has ≈ 80)
DR_ABS_GATE_DB     = -60.0              # B12: frames below −60 dBFS are not "dynamics" …
DR_REL_GATE_DB     = 40.0               # … nor are frames > 40 dB below the excerpt's P95 frame
FLAT_BAND_HZ       = (50.0, 8000.0)     # B-NF spectral-flatness band


def hz_to_cents(f_hz):
    return 1200.0 * np.log2(np.clip(np.asarray(f_hz, dtype=float), 1e-6, None) / 440.0)


def cents_to_hz(cents):
    return 440.0 * 2.0 ** (np.asarray(cents, dtype=float) / 1200.0)


def voiced_runs(mask):
    """Maximal runs of True → list of (start, end) index pairs, end exclusive."""
    m = np.asarray(mask, dtype=bool).astype(np.int8)
    d = np.diff(np.concatenate([[0], m, [0]]))
    return list(zip(np.flatnonzero(d == 1).tolist(), np.flatnonzero(d == -1).tolist()))


def fix_octave_errors(cents, voiced, win=OCT_WIN):
    """Each voiced frame vs the median of the voiced frames within ±win frames (computed on the
    ORIGINAL track, one pass); |deviation| ∈ [1000, 1400] cents → shift by ∓1200.
    → (corrected cents, number of corrected frames)."""
    c = np.asarray(cents, dtype=float)
    v = np.asarray(voiced, dtype=bool)
    out, n_fixed = c.copy(), 0
    for i in np.flatnonzero(v):
        lo, hi = max(0, i - win), min(len(c), i + win + 1)
        d = c[i] - np.median(c[lo:hi][v[lo:hi]])
        if OCT_DEV_RANGE[0] <= abs(d) <= OCT_DEV_RANGE[1]:
            out[i] = c[i] - np.sign(d) * 1200.0
            n_fixed += 1
    return out, n_fixed


def note_trend(cents, voiced):
    """Vibrato-free pitch trend: rolling median (TREND_WIN frames) INSIDE each voiced run; NaN
    elsewhere. A median filter preserves monotone ramps and steps exactly."""
    c = np.asarray(cents, dtype=float)
    tr = np.full(len(c), np.nan)
    for a, b in voiced_runs(voiced):
        tr[a:b] = median_filter(c[a:b], size=TREND_WIN, mode='nearest')
    return tr


def env_db_on_grid(y, sr, n_frames, step_ms=CREPE_STEP_MS, win_ms=ENV_WIN_MS):
    """B2 RMS envelope on the CREPE grid: frame i is centred on t_i = i·step_ms (CREPE's frame times,
    center=True) and covers a RECTANGULAR window of win_ms (25 ms) of the same 16 kHz mono audio CREPE
    reads; samples outside the signal count as zeros. → (n_frames,) dB re full scale, floored at
    −120 dB (digital silence)."""
    y = np.asarray(y, dtype=float)
    half = int(round(sr * win_ms / 2000.0))
    cs = np.concatenate([[0.0], np.cumsum(y ** 2)])
    ctr = np.round(np.arange(int(n_frames)) * sr * step_ms / 1000.0).astype(int)
    lo, hi = np.clip(ctr - half, 0, len(y)), np.clip(ctr + half, 0, len(y))
    ms = (cs[hi] - cs[lo]) / float(2 * half)
    return 10.0 * np.log10(np.maximum(ms, 1e-12))


def _plateaus_in_run(tr, min_len, tol=PLATEAU_TOL_CENTS, max_range=PLATEAU_MAX_RANGE_CENTS):
    """Greedy plateaus of one voiced run's vibrato-free trend: from frame i, grow while
    |tr[j] − median(tr[plateau])| ≤ tol and the plateau's range stays ≤ max_range. A plateau of
    ≥ min_len frames is a note candidate and the scan resumes after it; otherwise frame i is a
    transition frame and the scan resumes at i + 1. → [(start, end_excl)] run-relative."""
    out, i, n = [], 0, len(tr)
    while i < n:
        srt, lo, hi, j = [float(tr[i])], float(tr[i]), float(tr[i]), i + 1
        while j < n:
            m = len(srt)
            med = srt[m // 2] if m % 2 else 0.5 * (srt[m // 2 - 1] + srt[m // 2])
            x = float(tr[j])
            if abs(x - med) > tol or max(hi, x) - min(lo, x) > max_range:
                break
            bisect.insort(srt, x)
            lo, hi, j = min(lo, x), max(hi, x), j + 1
        if j - i >= min_len:
            out.append((i, j))
            i = j
        else:
            i += 1
    return out


def _merge_plateaus(pl, tr, merge_cents=PLATEAU_MERGE_CENTS, max_gap=None,
                    max_range=PLATEAU_MAX_RANGE_CENTS):
    """Merge adjacent plateaus whose trend medians differ < merge_cents and that are ≤ max_gap
    frames apart (a vibrato / intonation wobble, not a new note); the gap frames join the note.
    B3: a merge is accepted only if the trend range of the MERGED note — both plateaus AND the gap
    frames between them — stays ≤ max_range (PLATEAU_MAX_RANGE_CENTS), so a merge can never break the
    plateau range invariant (B2 merged Meditation_Thais_EXG C♯6 into a 128 c note) nor absorb the
    transition frames of a real pitch change."""
    max_gap = int(round(PLATEAU_MERGE_GAP_MS / CREPE_STEP_MS)) if max_gap is None else int(max_gap)
    out = []
    for s, e in pl:
        if out:
            ps, pe = out[-1]
            if (s - pe <= max_gap and abs(np.median(tr[s:e]) - np.median(tr[ps:pe])) < merge_cents
                    and float(np.max(tr[ps:e]) - np.min(tr[ps:e])) <= max_range):
                out[-1] = (ps, e)
                continue
        out.append((s, e))
    return out


def _reart_splits(env, s, e, min_len, dip_db, pre, post):
    """Re-articulation split frames (absolute) inside the note [s, e): a local minimum of env with
    max(env[t−pre : t]) − env[t] ≥ dip_db and max(env[t : t+post]) − env[t] ≥ dip_db, accepted
    deepest first only if every resulting part keeps ≥ min_len frames."""
    cand = []
    for t in range(s + min_len, e - min_len + 1):
        if not (env[t] < env[t - 1] and env[t] <= env[t + 1]):
            continue
        rise_pre = float(env[max(0, t - pre):t].max()) - env[t]
        rise_post = float(env[t:min(len(env), t + post + 1)].max()) - env[t]
        if min(rise_pre, rise_post) >= dip_db:
            cand.append((min(rise_pre, rise_post), t))
    acc = []
    for _, t in sorted(cand, key=lambda z: (-z[0], z[1])):
        edges = [s] + sorted(acc + [t]) + [e]
        if all(b - a >= min_len for a, b in zip(edges[:-1], edges[1:])):
            acc.append(t)
    return sorted(acc)


def segment_notes_detail(cents, voiced, min_note_ms, step_ms=CREPE_STEP_MS, env_db=None,
                         reart_dip_db=REART_DIP_DB, reart_enabled=None):
    """B2 plateau segmentation with its provenance. Per voiced run (never across an unvoiced gap):
    plateaus of the TREND_WIN median trend (≥ NOTE_MIN_MS_TIMING, tolerance PLATEAU_TOL_CENTS,
    range ≤ PLATEAU_MAX_RANGE_CENTS) → merge plateaus < PLATEAU_MERGE_CENTS / ≤ PLATEAU_MERGE_GAP_MS
    apart whose merged trend range stays ≤ PLATEAU_MAX_RANGE_CENTS (B3) → [re-articulation splits]
    → drop notes < min_note_ms. Frames outside plateaus are transitions.
    Re-articulation splits at env_db dips (parts ≥ NOTE_MIN_MS_TIMING) happen ONLY when the detector
    is enabled AND env_db is given: reart_enabled=None → the module switch REART_ENABLED (False since
    B3, so every feature path ignores env_db); reart_enabled=True forces the detector on (report only —
    §2.2 evidence table / sensitivity). env_db is still length-checked whenever it is given.
    → dict(notes = int n×3 [start, end_excl, run_id], meds = note median cents, kinds = how each
    note starts: 'unvoiced gap' (first plateau of a voiced run) / 'plateau' (pitch change) /
    're-articulation split' (detector on only), splits = absolute split frames (empty when off))."""
    c = np.asarray(cents, dtype=float)
    use_reart = REART_ENABLED if reart_enabled is None else bool(reart_enabled)
    env = None
    if env_db is not None:
        env_chk = np.asarray(env_db, dtype=float)
        if len(env_chk) != len(c):
            raise ValueError(f'env_db has {len(env_chk)} frames, the track {len(c)}')
        env = env_chk if use_reart else None
    elif reart_enabled:
        raise ValueError('segment_notes_detail: reart_enabled=True needs env_db')
    min_len = int(np.ceil(min_note_ms / step_ms))
    cand_len = int(np.ceil(NOTE_MIN_MS_TIMING / step_ms))
    gap = int(round(PLATEAU_MERGE_GAP_MS / step_ms))
    pre, post = int(round(REART_PRE_MS / step_ms)), int(round(REART_POST_MS / step_ms))
    notes, meds, kinds, splits = [], [], [], []
    for r, (a, b) in enumerate(voiced_runs(voiced)):
        tr = median_filter(c[a:b], size=TREND_WIN, mode='nearest')
        first = True
        for s, e in _merge_plateaus(_plateaus_in_run(tr, cand_len), tr, max_gap=gap):
            cuts = ([] if env is None
                    else _reart_splits(env, a + s, a + e, cand_len, reart_dip_db, pre, post))
            splits.extend(cuts)
            edges = [a + s] + cuts + [a + e]
            for k, (p0, p1) in enumerate(zip(edges[:-1], edges[1:])):
                kind = ('re-articulation split' if k > 0 else 'unvoiced gap' if first else 'plateau')
                if p1 - p0 >= min_len:
                    notes.append((p0, p1, r))
                    meds.append(float(np.median(c[p0:p1])))
                    kinds.append(kind)
            first = False
    return dict(notes=np.asarray(notes, dtype=int).reshape(-1, 3), meds=np.asarray(meds, dtype=float),
                kinds=kinds, splits=np.asarray(sorted(splits), dtype=int))


def segment_notes(cents, voiced, min_note_ms, step_ms=CREPE_STEP_MS, env_db=None,
                  reart_dip_db=REART_DIP_DB, reart_enabled=None):
    """Notes = trend plateaus inside voiced runs (B2/B3; see segment_notes_detail — re-articulation
    splits only when the detector is enabled). → (int array n×3 [start, end_excl, run_id], float
    array of note median pitches in cents) — the Phase B return contract. Notes shorter than
    min_note_ms are dropped."""
    d = segment_notes_detail(cents, voiced, min_note_ms, step_ms, env_db, reart_dip_db, reart_enabled)
    return d['notes'], d['meds']


def _segment_notes_v2(cents, voiced, min_note_ms, step_ms=CREPE_STEP_MS):
    """LEGACY Phase B segmentation (report only — §2.2 prints old vs new): split at unvoiced
    gaps and where |trend[t] − trend[t−5]| > 70 cents; never sees re-articulations."""
    c = np.asarray(cents, dtype=float)
    min_len = int(np.ceil(min_note_ms / step_ms))
    notes, meds = [], []
    for r, (a, b) in enumerate(voiced_runs(voiced)):
        seg = c[a:b]
        tr = median_filter(seg, size=TREND_WIN, mode='nearest')
        flag = np.zeros(len(seg), dtype=bool)
        if len(seg) > BOUNDARY_LAG:
            flag[BOUNDARY_LAG:] = np.abs(tr[BOUNDARY_LAG:] - tr[:-BOUNDARY_LAG]) > BOUNDARY_CENTS
        for s, e in voiced_runs(~flag):
            if e - s >= min_len:
                notes.append((a + s, a + e, r))
                meds.append(float(np.median(seg[s:e])))
    return np.asarray(notes, dtype=int).reshape(-1, 3), np.asarray(meds, dtype=float)


def n_close_splits(notes, meds, kinds=None):
    """Report-only: adjacent notes of the same run whose medians differ by < BOUNDARY_CENTS —
    possible spurious splits; with `kinds`, re-articulation splits (repeated notes by design; only
    present when the detector is enabled) are not counted."""
    return int(sum(1 for i in range(len(notes) - 1)
                   if notes[i][2] == notes[i + 1][2] and abs(meds[i + 1] - meds[i]) < BOUNDARY_CENTS
                   and (kinds is None or kinds[i + 1] != 're-articulation split')))


def weighted_median(x, w):
    x, w = np.asarray(x, dtype=float), np.asarray(w, dtype=float)
    o = np.argsort(x)
    cw = np.cumsum(w[o])
    return float(x[o][np.searchsorted(cw, 0.5 * cw[-1])])


_VIB_BA = butter(2, [VIB_BAND[0] / (CREPE_FS / 2), VIB_BAND[1] / (CREPE_FS / 2)], btype='band')


def vibrato_note(seg, fs=CREPE_FS):
    """LEGACY Phase B detector (report only — the removed §2.6 audit showed it next to the new fit). One note (cents)
    → (rate_hz, half_extent_cents, presence, is_vibrato). 1 degree-2 detrend · 2 Hann + zero-pad to
    VIB_NFFT, peak in [3,10] Hz, parabolic interpolation on the log-magnitude (unclamped — it can
    step outside the band at the band edge) · 3 presence = P(±0.5 Hz)/P([1,15] Hz) ≥ 0.30 AND
    half-extent ≥ VIB_LEGACY_MIN_EXTENT (5 c) · 4 extent = median Hilbert envelope of the 3–10 Hz
    band-pass."""
    seg = np.asarray(seg, dtype=float)
    n = len(seg)
    t = np.arange(n) / fs
    r = seg - np.polyval(np.polyfit(t, seg, 2), t)
    X = np.abs(np.fft.rfft(r * np.hanning(n), VIB_NFFT))
    f = np.fft.rfftfreq(VIB_NFFT, 1.0 / fs)
    band = np.flatnonzero((f >= VIB_BAND[0]) & (f <= VIB_BAND[1]))
    k = int(band[np.argmax(X[band])])
    rate = float(f[k])
    if 0 < k < len(X) - 1 and np.all(X[k - 1:k + 2] > 0):
        la, lb, lc = np.log(X[k - 1:k + 2])
        den = la - 2.0 * lb + lc
        if den < 0:
            rate = float(f[k] + 0.5 * (la - lc) / den * (f[1] - f[0]))
    P = X ** 2
    presence = float(P[np.abs(f - rate) <= VIB_PEAK_HALFWIDTH].sum()
                     / P[(f >= VIB_DENOM_BAND[0]) & (f <= VIB_DENOM_BAND[1])].sum())
    extent = float(np.median(np.abs(hilbert(filtfilt(*_VIB_BA, r)))))
    return rate, extent, presence, bool(presence >= VIB_PRESENCE_MIN and extent >= VIB_LEGACY_MIN_EXTENT)


_VIB_RATES = np.round(np.arange(VIB_RATE_BAND[0], VIB_RATE_BAND[1] + 0.5 * VIB_RATE_STEP,
                                VIB_RATE_STEP), 10)


def _sin_fit(r, t, f):
    """Least-squares a·sin(2πft) + b·cos(2πft) to r (zero-mean) for every rate in f (array).
    → (a, b, explained sum of squares) arrays."""
    w = 2.0 * np.pi * np.asarray(f, dtype=float)[:, None] * t[None, :]
    S, C = np.sin(w), np.cos(w)
    Sr, Cr = S @ r, C @ r
    SS, CC, SC = (S * S).sum(1), (C * C).sum(1), (S * C).sum(1)
    det = SS * CC - SC ** 2
    a = (CC * Sr - SC * Cr) / det
    b = (SS * Cr - SC * Sr) / det
    return a, b, a * Sr + b * Cr


def vibrato_window_fit(x, fs=CREPE_FS):
    """B8b, one analysis window (cents): degree-2 detrend, least-squares sinusoid over the rate grid
    VIB_RATE_BAND / VIB_RATE_STEP, R² = 1 − SSR/SST of the detrended window, rate = argmax R² with
    a parabolic refinement CLAMPED to the grid, half-extent = sqrt(a² + b²) at that rate.
    B3: edge = the argmax (BEFORE refinement) is the first or the last grid point — the best fit then
    lies at or beyond the band limit (a residual slow trend, not vibrato); vibrato_features rejects
    such a window whatever its R² and extent.
    → dict(rate, extent, r2, a, b, edge, trend, fit) (fit = trend + the fitted sinusoid, for plots)."""
    x = np.asarray(x, dtype=float)
    t = np.arange(len(x)) / fs
    trend = np.polyval(np.polyfit(t, x, 2), t)
    r = x - trend
    sst = float(np.sum((r - r.mean()) ** 2))
    if sst <= 1e-12:
        return dict(rate=float(_VIB_RATES[0]), extent=0.0, r2=0.0, a=0.0, b=0.0, edge=False,
                    trend=trend, fit=trend)
    _, _, ess = _sin_fit(r, t, _VIB_RATES)
    r2g = ess / sst
    k = int(np.argmax(r2g))
    edge = bool(k == 0 or k == len(_VIB_RATES) - 1)
    rate = float(_VIB_RATES[k])
    if 0 < k < len(_VIB_RATES) - 1:
        y0, y1, y2 = r2g[k - 1:k + 2]
        den = y0 - 2.0 * y1 + y2
        if den < 0:
            rate += float(np.clip(0.5 * (y0 - y2) / den, -0.5, 0.5)) * VIB_RATE_STEP
    rate = float(np.clip(rate, _VIB_RATES[0], _VIB_RATES[-1]))
    a, b, e1 = _sin_fit(r, t, np.array([rate]))
    a, b = float(a[0]), float(b[0])
    fit = trend + a * np.sin(2 * np.pi * rate * t) + b * np.cos(2 * np.pi * rate * t)
    return dict(rate=rate, extent=float(np.hypot(a, b)), r2=float(min(1.0, e1[0] / sst)), a=a, b=b,
                edge=edge, trend=trend, fit=fit)


def vibrato_window_reason(f):
    """B3 verdict of one window fit (vibrato_window_fit dict), in rule order: 'band edge' (the R²
    argmax is the first / last rate of the grid — rejected whatever its R² and extent) · 'low R²' ·
    'small extent' · 'vibrato'."""
    if f['edge']:
        return 'band edge'
    if f['r2'] < VIB_FIT_R2_MIN:
        return 'low R²'
    if f['extent'] < VIB_MIN_EXTENT:
        return 'small extent'
    return 'vibrato'


def vibrato_window_starts(n, win, hop):
    """Window starts covering a note of n frames: every `hop` frames, the last window shifted to end
    at the note end (a note of win ≤ n < win + hop gets a start and an end window; n == win → one)."""
    if n < win:
        return []
    st = list(range(0, n - win + 1, hop))
    if st[-1] + win < n:
        st.append(n - win)
    return st


def vibrato_features(cents, voiced, min_note_ms=VIB_MIN_NOTE_MS, env_db=None):
    """B8b, per excerpt. Notes ≥ min_note_ms (B2/B3 segmentation; env_db is ignored while the
    re-articulation detector is disabled) are cut into VIB_WIN_MS windows (hop VIB_HOP_MS); a window
    is vibrato iff its best rate is NOT on the edge of the rate grid (B3), R² ≥ VIB_FIT_R2_MIN and
    half-extent ≥ VIB_MIN_EXTENT (each window records `edge` and its `reason`, vibrato_window_reason).
    coverage = vibrato-window time (union, no double counting) / time of all analysed notes;
    rate / extent = duration-weighted medians over vibrato windows. Undefined values are NaN:
    no analysed note → rate, extent, coverage NaN; notes but no vibrato window → rate NaN,
    extent 0, coverage 0 (the caller applies the DECLARED median substitution).
    → dict(rate, extent, coverage, n_notes, n_windows, n_vib_windows, n_vib_notes, n_edge_windows,
    n_edge_only, notes, windows) — n_edge_windows = windows whose rate is on the grid edge,
    n_edge_only = those that would have passed R² and extent (rejected ONLY by the edge rule)."""
    notes, _ = segment_notes(cents, voiced, min_note_ms, env_db=env_db)
    win = int(round(VIB_WIN_MS / CREPE_STEP_MS))
    hop = int(round(VIB_HOP_MS / CREPE_STEP_MS))
    if len(notes) == 0:
        return dict(rate=np.nan, extent=np.nan, coverage=np.nan, n_notes=0, n_windows=0,
                    n_vib_windows=0, n_vib_notes=0, n_edge_windows=0, n_edge_only=0, notes=notes,
                    windows=[])
    c = np.asarray(cents, dtype=float)
    windows, vib_frames, note_frames, n_vib_notes = [], 0, 0, 0
    for k, (s, e, _) in enumerate(notes):
        mask = np.zeros(e - s, dtype=bool)
        for o in vibrato_window_starts(e - s, win, hop):
            f = vibrato_window_fit(c[s + o:s + o + win])
            reason = vibrato_window_reason(f)
            isv = reason == 'vibrato'
            windows.append(dict(note=k, start=int(s + o), end=int(s + o + win), rate=f['rate'],
                                extent=f['extent'], r2=f['r2'], edge=bool(f['edge']),
                                is_vibrato=isv, reason=reason))
            if isv:
                mask[o:o + win] = True
        vib_frames += int(mask.sum())
        note_frames += int(e - s)
        n_vib_notes += int(mask.any())
    vw = [w for w in windows if w['is_vibrato']]
    if vw:
        wd = np.array([w['end'] - w['start'] for w in vw], dtype=float)
        rate = weighted_median([w['rate'] for w in vw], wd)
        extent = weighted_median([w['extent'] for w in vw], wd)
    else:
        rate, extent = np.nan, 0.0
    n_edge = sum(w['edge'] for w in windows)
    n_edge_only = sum(w['edge'] and w['r2'] >= VIB_FIT_R2_MIN and w['extent'] >= VIB_MIN_EXTENT
                      for w in windows)
    return dict(rate=rate, extent=extent, coverage=float(vib_frames / note_frames),
                n_notes=int(len(notes)), n_windows=len(windows), n_vib_windows=len(vw),
                n_vib_notes=int(n_vib_notes), n_edge_windows=int(n_edge), n_edge_only=int(n_edge_only),
                notes=notes, windows=windows)


def portamento_features(cents, voiced, step_ms=CREPE_STEP_MS, env_db=None):
    """B9b. A transition = the region between consecutive notes (B2/B3 plateaus, ≥ 60 ms) of the same
    voiced run (env_db is ignored while the re-articulation detector is disabled; with it enabled,
    re-articulation neighbours have interval ≈ 0 and fail the interval rule). Rise =
    10–90 % time of the vibrato-free trend, searched from the middle of the first note to the
    middle of the second, × GLIDE_RISE_SCALE. t_min = 0.8·|interval| / CREPE_VITERBI_MAX_CENTS_PER_FRAME
    · step × GLIDE_RISE_SCALE is the shortest rise a pure jump can produce through crepe's Viterbi
    decoder; excess = rise − t_min. Glide iff |interval| ∈ GLIDE_INTERVAL, rise ∈ GLIDE_DUR_MS,
    excess ≥ GLIDE_MIN_EXCESS_MS and ≥ GLIDE_SIGN_MIN of the RAW frame diffs in the 10–90 % span
    carry the interval's sign. → rate (glides / voiced s), share (glides / transitions), mean_ext
    (cents), n_trans, n_glides, durs, exts, excess (per glide), transitions (one dict per pair of
    consecutive notes, with the reject reason)."""
    c = np.asarray(cents, dtype=float)
    v = np.asarray(voiced, dtype=bool)
    notes, meds = segment_notes(c, v, NOTE_MIN_MS_TIMING, step_ms, env_db=env_db)
    tr = note_trend(c, v)
    n_trans, glides, trans = 0, [], []
    for i in range(len(notes) - 1):
        (sA, eA, rA), (sB, eB, rB) = notes[i], notes[i + 1]
        interval = float(meds[i + 1] - meds[i])
        rec = dict(from_note=i, to_note=i + 1, boundary=int(eA), interval=interval, counted=bool(rA == rB),
                   rise_ms=np.nan, t_min_ms=np.nan, excess_ms=np.nan, sign_frac=np.nan,
                   span_start=-1, span_end=-1, passes_except_excess=False, glide=False)
        if rA != rB:
            rec['reason'] = 'across an unvoiced gap'          # not a transition
            trans.append(rec)
            continue
        n_trans += 1
        if not (GLIDE_INTERVAL[0] <= abs(interval) <= GLIDE_INTERVAL[1]):
            rec['reason'] = 'interval out of range'
            trans.append(rec)
            continue
        sgn = np.sign(interval)
        lo, hi = eA - (eA - sA) // 2, sB + (eB - sB) // 2 + 1
        prog = sgn * (tr[lo:hi] - meds[i]) / abs(interval)     # 0 at note A → 1 at note B
        hit90 = np.flatnonzero(prog >= 0.9)
        if not len(hit90):
            rec['reason'] = 'no 90 % crossing'
            trans.append(rec)
            continue
        t2 = int(hit90[0])
        low10 = np.flatnonzero(prog[:t2] < 0.1)
        t1 = int(low10[-1]) + 1 if len(low10) else 0
        rise = (t2 - t1) * step_ms * GLIDE_RISE_SCALE
        t_min = 0.8 * abs(interval) / CREPE_VITERBI_MAX_CENTS_PER_FRAME * step_ms * GLIDE_RISE_SCALE
        d = np.diff(c[lo + t1: lo + t2 + 1])
        sf = float(np.mean(np.sign(d) == sgn)) if len(d) else np.nan
        dur_ok = GLIDE_DUR_MS[0] <= rise <= GLIDE_DUR_MS[1]
        mono_ok = len(d) > 0 and sf >= GLIDE_SIGN_MIN
        rec.update(rise_ms=rise, t_min_ms=t_min, excess_ms=rise - t_min, sign_frac=sf,
                   span_start=int(lo + t1), span_end=int(lo + t2 + 1),
                   passes_except_excess=bool(dur_ok and mono_ok))
        if rise < GLIDE_DUR_MS[0]:
            rec['reason'] = 'too short'
        elif rise - t_min < GLIDE_MIN_EXCESS_MS:
            rec['reason'] = 'viterbi-limited jump'
        elif rise > GLIDE_DUR_MS[1]:
            rec['reason'] = 'too slow'
        elif not mono_ok:
            rec['reason'] = 'not monotone'
        else:
            rec['reason'], rec['glide'] = 'glide', True
            glides.append((rise, abs(interval), rise - t_min))
        trans.append(rec)
    voiced_s = v.sum() * step_ms / 1000.0
    return dict(rate=len(glides) / voiced_s if voiced_s > 0 else np.nan,
                share=len(glides) / n_trans if n_trans else 0.0,
                mean_ext=float(np.mean([g[1] for g in glides])) if glides else 0.0,
                n_trans=n_trans, n_glides=len(glides),
                durs=[g[0] for g in glides], exts=[g[1] for g in glides],
                excess=[g[2] for g in glides], transitions=trans)


def f0_features(cents, voiced):
    """B10 — on the octave-corrected track: range = P95 − P5 of voiced cents; mean and CV in Hz."""
    cv = np.asarray(cents, dtype=float)[np.asarray(voiced, dtype=bool)]
    hz = cents_to_hz(cv)
    return dict(range=float(np.percentile(cv, 95) - np.percentile(cv, 5)),
                mean_hz=float(hz.mean()), cv=float(hz.std() / hz.mean()))


def note_onset_times(cents, voiced, step_ms=CREPE_STEP_MS, env_db=None):
    """B13 onsets (s): a note's onset = the start of its voiced run if it is the run's first
    note, otherwise the start of the transition region (= end of the previous note). env_db is
    accepted but ignored while REART_ENABLED is False (B3): a repeated same-pitch note with no
    silence between is ONE note. (With the detector enabled, a re-articulation split point would be
    a note onset — the previous note ends exactly there.)"""
    notes, _ = segment_notes(cents, voiced, NOTE_MIN_MS_TIMING, step_ms, env_db=env_db)
    runs = voiced_runs(voiced)
    on = [runs[r][0] if (i == 0 or notes[i - 1][2] != r) else notes[i - 1][1]
          for i, (s, e, r) in enumerate(notes)]
    return np.asarray(on, dtype=float) * step_ms / 1000.0


def timing_note_features(cents, voiced, duration_s, step_ms=CREPE_STEP_MS, env_db=None):
    """B13 note-based timing: note rate (notes / s of the whole excerpt) and IOI statistics
    (IOIs between consecutive onsets, rests included). ≥ 3 notes required (invariant). Onsets from
    note_onset_times (B3: no re-articulation splits)."""
    on = note_onset_times(cents, voiced, step_ms, env_db=env_db)
    if len(on) < 3:
        raise ValueError(f'only {len(on)} notes ≥ {NOTE_MIN_MS_TIMING} ms — IOIs undefined')
    ioi = np.diff(on)
    return dict(note_rate=float(len(on) / duration_s), ioi_mean_s=float(ioi.mean()),
                ioi_sd_s=float(ioi.std()), ioi_cv=float(ioi.std() / ioi.mean()))


def local_tempo_from_tempogram(tg, tempi, win_length=None):
    """B13 tempogram read-out. Per frame: argmax over 40–200 BPM of tempogram × log-normal prior
    w(bpm) = exp(−½·log2(bpm/100)²). → mean local tempo, SD of log2(local tempo) (octave-invariant
    rubato proxy, in octaves), fraction of octave jumps between consecutive frames (diagnostic),
    pulse clarity = max over the band of the frame-averaged tempogram, number of frames used.
    With TEMPO_INTERIOR_ONLY and win_length given, only frames whose window lies inside the
    excerpt are used (≥ TEMPO_MIN_INTERIOR required)."""
    tempi = np.asarray(tempi, dtype=float)
    band = np.isfinite(tempi) & (tempi >= TEMPO_BPM_RANGE[0]) & (tempi <= TEMPO_BPM_RANGE[1])
    tg = np.asarray(tg, dtype=float)
    if TEMPO_INTERIOR_ONLY and win_length is not None:
        h = int(win_length) // 2
        tg = tg[:, h:tg.shape[1] - h]
        if tg.shape[1] < TEMPO_MIN_INTERIOR:
            raise ValueError(f'only {tg.shape[1]} interior tempogram frames (< {TEMPO_MIN_INTERIOR}) — '
                             'excerpt too short for the tempogram window')
    if not band.any() or tg.shape[1] == 0:
        raise ValueError('tempogram has no frame or no bin inside the 40–200 BPM band')
    T = np.nan_to_num(tg[band], nan=0.0, posinf=0.0, neginf=0.0)
    bpm = tempi[band]
    prior = np.exp(-0.5 * np.log2(bpm / TEMPO_PRIOR_CENTER) ** 2)
    loc = bpm[np.argmax(T * prior[:, None], axis=0)]
    lg = np.log2(loc)
    jumps = np.abs(np.abs(np.diff(lg)) - 1.0) < OCT_JUMP_TOL
    return dict(local_tempo_bpm=float(loc.mean()), tempo_logsd=float(lg.std()),
                octave_jump_frac=float(jumps.mean()) if len(jumps) else 0.0,
                pulse_clarity=float(np.max(T.mean(axis=1))), n_frames=int(T.shape[1]))


def l1_normalize_spectrum(S, eps=1e-12):
    """B11: frame magnitude spectrum → unit L1 mass, so the flux measures spectral SHAPE change
    and not level change."""
    S = np.asarray(S, dtype=np.float32)
    return np.ascontiguousarray(S / (S.sum() + eps), dtype=np.float32)


def dr_gate(rms_db, abs_gate_db=DR_ABS_GATE_DB, rel_gate_db=DR_REL_GATE_DB):
    """B12: keep frames ≥ abs gate AND within rel_gate dB of the excerpt's P95 frame.
    → (P95 − P5 of the kept frames, keep mask)."""
    x = np.asarray(rms_db, dtype=float)
    keep = (x >= abs_gate_db) & (x >= np.percentile(x, 95) - rel_gate_db)
    if keep.sum() < 2:
        return np.nan, keep
    return float(np.percentile(x[keep], 95) - np.percentile(x[keep], 5)), keep


def frame_power_spectra(frames, sr):
    """Hann-windowed power spectrum per frame (rows) → (P, freqs)."""
    frames = np.asarray(frames, dtype=float)
    P = np.abs(np.fft.rfft(frames * np.hanning(frames.shape[1]), axis=1)) ** 2
    return P, np.fft.rfftfreq(frames.shape[1], 1.0 / sr)


def spectral_flatness(P, freqs, band=FLAT_BAND_HZ, eps=1e-20):
    """Geometric / arithmetic mean of the power spectrum within band (upper edge clamped to
    Nyquist): ≈ 0.56 for white noise, ≈ 0 for a tone."""
    m = (freqs >= band[0]) & (freqs <= min(band[1], freqs[-1]))
    Pb = np.asarray(P, dtype=float)[:, m] + eps
    return np.exp(np.mean(np.log(Pb), axis=1)) / np.mean(Pb, axis=1)


def crepe_descriptors(f0_hz, voiced, env_db=None):
    """One CREPE track (+ its 10 ms RMS envelope env_db — length-checked, but used for the
    re-articulation splits only when REART_ENABLED; ignored since B3) →
    (features dict, diagnostics dict, octave-corrected cents, voiced mask)."""
    f0 = np.asarray(f0_hz, dtype=float)
    v = np.asarray(voiced, dtype=bool) & (f0 > 0)
    if v.sum() < 20:
        raise ValueError(f'only {int(v.sum())} voiced frames — no pitch descriptor is defined')
    if env_db is not None and len(env_db) != len(f0):
        raise ValueError(f'env_db has {len(env_db)} frames, the track {len(f0)}')
    cents, n_fix = fix_octave_errors(hz_to_cents(np.where(v, f0, 440.0)), v)
    vib = vibrato_features(cents, v, env_db=env_db)
    por = portamento_features(cents, v, env_db=env_db)
    f0f = f0_features(cents, v)
    seg = segment_notes_detail(cents, v, NOTE_MIN_MS_TIMING, env_db=env_db)
    n60, m60 = seg['notes'], seg['meds']
    feats = {
        'crepe_vibrato_rate_hz'          : vib['rate'],
        'crepe_vibrato_extent_cents'     : vib['extent'],
        'crepe_vibrato_coverage'         : vib['coverage'],
        'crepe_portamento_rate'          : por['rate'],
        'crepe_portamento_share'         : por['share'],
        'crepe_portamento_mean_ext_cents': por['mean_ext'],
        'crepe_f0_range_cents'           : f0f['range'],
        'crepe_f0_cv'                    : f0f['cv'],
        'crepe_f0_mean_hz'               : f0f['mean_hz'],
        'crepe_voiced_fraction'          : float(v.mean()),
        'crepe_vibrato_n_notes'          : float(vib['n_notes']),     # diagnostic only
    }
    diag = dict(n_octave_fixed=int(n_fix), n_vib_notes=vib['n_vib_notes'], n_windows=vib['n_windows'],
                n_vib_windows=vib['n_vib_windows'], n_edge_windows=vib['n_edge_windows'],
                n_edge_only=vib['n_edge_only'], n_notes_60=int(len(n60)),
                n_notes_60_v2=int(len(_segment_notes_v2(cents, v, NOTE_MIN_MS_TIMING)[0])),
                n_reart_splits=int(sum(k == 're-articulation split' for k in seg['kinds'])),
                n_close_splits=n_close_splits(n60, m60, seg['kinds']), n_transitions=por['n_trans'],
                n_glides=por['n_glides'], glide_durs=por['durs'], glide_exts=por['exts'],
                glide_excess=por['excess'], transitions=por['transitions'],
                vib_windows=[(w['r2'], w['extent'], float(w['edge'])) for w in vib['windows']],
                notes_by_min={ms: int(len(segment_notes(cents, v, ms, env_db=env_db)[0]))
                              for ms in VIB_NOTE_SENS_MS})
    return feats, diag, cents, v
# <<< HELPERS:descriptors


CREPE_CANDIDATES = ['crepe_vibrato_rate_hz', 'crepe_vibrato_extent_cents', 'crepe_vibrato_coverage',
                    'crepe_portamento_rate', 'crepe_portamento_share', 'crepe_portamento_mean_ext_cents',
                    'crepe_f0_range_cents', 'crepe_f0_cv', 'crepe_f0_mean_hz', 'crepe_voiced_fraction']
CREPE_DIAGNOSTIC = ['crepe_vibrato_n_notes']                # diagnostic only — never forwarded
CREPE_LABELS = {'crepe_vibrato_rate_hz'          : 'Vibrato Rate',
                'crepe_vibrato_extent_cents'     : 'Vibrato Extent',
                'crepe_vibrato_coverage'         : 'Vibrato Coverage',
                'crepe_portamento_rate'          : 'Portamento Rate',
                'crepe_portamento_share'         : 'Portamento Share',
                'crepe_portamento_mean_ext_cents': 'Portamento Extent',
                'crepe_f0_range_cents'           : 'F0 Range',
                'crepe_f0_cv'                    : 'F0 Coeff. Var.',
                'crepe_f0_mean_hz'               : 'F0 Mean',
                'crepe_voiced_fraction'          : 'Voiced Fraction'}

# ── 1. CREPE tracks — extracted once, cached ──────────────────────────────────
CACHE_CREPE_TRACKS = os.path.join(OUTPUT_DIR, 'crepe_tracks_v2.npz')
CACHE_CREPE        = os.path.join(OUTPUT_DIR, 'feat_crepe_v4.csv')   # B3 descriptors (v2 = Phase B,
                                                                     # v3 = B2 — both retired)
CACHE_CREPE_ENV    = os.path.join(OUTPUT_DIR, 'rms10_v3.npz')        # B2 10 ms RMS envelope (env_db;
                                                                     # unchanged — report-only in B3)
_TRACK_KEYS = ('time', 'f0_hz', 'confidence', 'voiced')
if os.path.exists(CACHE_CREPE_TRACKS):
    _z = np.load(CACHE_CREPE_TRACKS)
    CREPE_TRACKS = {e: {k: _z[f'{e}__{k}'] for k in _TRACK_KEYS} for e in audio_files}
    print(f'📂 Loaded CREPE tracks ({len(CREPE_TRACKS)} excerpts) from {CACHE_CREPE_TRACKS}')
else:
    print('🔄 Running CREPE (10 ms, viterbi) …')
    CREPE_TRACKS = {}
    for _e, _p in tqdm(audio_files.items(), desc='CREPE'):
        _y16, _ = librosa.load(_p, sr=16000, mono=True)
        _tm, _fr, _cf, _ = crepe.predict(_y16, 16000, viterbi=True, step_size=CREPE_STEP_MS, verbose=0)
        _fr, _cf = np.asarray(_fr, dtype=float), np.asarray(_cf, dtype=float)
        CREPE_TRACKS[_e] = {'time': np.asarray(_tm, dtype=float), 'f0_hz': _fr, 'confidence': _cf,
                            'voiced': (_cf > CREPE_CONF_MIN) & (_fr > 0)}
    np.savez_compressed(CACHE_CREPE_TRACKS, **{f'{e}__{k}': v for e, d in CREPE_TRACKS.items()
                                                for k, v in d.items()})
    print(f'✅ Saved: {CACHE_CREPE_TRACKS}')
_miss22 = [e for e in df_expr['excerpt_id'] if e not in CREPE_TRACKS]
assert not _miss22, f'§2.2: no CREPE track for {_miss22}'
assert all(np.allclose(np.diff(np.asarray(d['time'], dtype=float)), CREPE_STEP_MS / 1000.0)
           for d in CREPE_TRACKS.values()), '§2.2: a CREPE track is not on the 10 ms grid'

# ── 1b. crepe's Viterbi transition limit, read from the INSTALLED package (report only) ──────
import inspect
import crepe.core as _crepe_core
try:
    _vit_src = inspect.getsource(_crepe_core.to_viterbi_cents)
except (AttributeError, OSError, TypeError) as _err:
    _vit_src = ''
    print(f'⚠️  §2.2: crepe.core.to_viterbi_cents source not readable ({type(_err).__name__}: {_err})')
CREPE_VITERBI_SOURCE_OK = '12 - abs' in _vit_src
print('── Viterbi limit: crepe.core.to_viterbi_cents '
      + (f'contains the "12 - abs" transition → max 11 bins × 20 c = '
         f'{CREPE_VITERBI_MAX_CENTS_PER_FRAME:g} c per frame (B9b assumption holds)'
         if CREPE_VITERBI_SOURCE_OK else
         '⚠️ does NOT contain the "12 - abs" transition — CREPE_VITERBI_MAX_CENTS_PER_FRAME = '
         f'{CREPE_VITERBI_MAX_CENTS_PER_FRAME:g} may not match the installed crepe; re-check t_min / excess'))

# ── 1c. 10 ms RMS envelope (B2) on the CREPE grid, from the same 16 kHz mono audio — cached ──
if os.path.exists(CACHE_CREPE_ENV):
    _ze = np.load(CACHE_CREPE_ENV)
    CREPE_ENV_DB = {e: np.asarray(_ze[e], dtype=float) for e in CREPE_TRACKS}
    print(f'📂 Loaded the 10 ms RMS envelope ({len(CREPE_ENV_DB)} excerpts) from {CACHE_CREPE_ENV}')
else:
    print(f'🔄 10 ms RMS envelope ({ENV_WIN_MS:g} ms rectangular window centred on the CREPE frames) …')
    CREPE_ENV_DB = {}
    for _e in tqdm(list(CREPE_TRACKS), desc='RMS10'):
        _y16, _ = librosa.load(audio_files[_e], sr=16000, mono=True)
        CREPE_ENV_DB[_e] = env_db_on_grid(_y16, 16000, len(CREPE_TRACKS[_e]['f0_hz']))
    np.savez_compressed(CACHE_CREPE_ENV, **CREPE_ENV_DB)
    print(f'✅ Saved: {CACHE_CREPE_ENV}')
_bad_env = [e for e in CREPE_TRACKS if len(CREPE_ENV_DB[e]) != len(CREPE_TRACKS[e]['f0_hz'])]
assert not _bad_env, f'§2.2: env_db length ≠ CREPE track length for {_bad_env}'

# ── 2. Descriptors from the tracks (recomputed every run — cheap; feat_crepe_v4.csv is the output) ──
print('\n' + '═' * 100)
print('§2.2 · re-articulation detector ' + (
    'DISABLED (REART_ENABLED = False) — see Phase B3: every feature uses the plateau notes only; '
    'the detector is evaluated below as a report-only negative result' if not REART_ENABLED else
    '⚠️ ENABLED (REART_ENABLED = True) — NOT the Phase B3 deployment: the features split notes at '
    'env_db dips'))
print('═' * 100)
_rows, CREPE_DIAG, CREPE_CENTS, CREPE_VOICED = {}, {}, {}, {}
for _e, _tr in CREPE_TRACKS.items():
    try:
        _f, _d, _c, _v = crepe_descriptors(_tr['f0_hz'], _tr['voiced'], CREPE_ENV_DB[_e])
    except ValueError as _err:
        raise ValueError(f'§2.2: {_e}: {_err}') from None
    _rows[_e], CREPE_DIAG[_e], CREPE_CENTS[_e], CREPE_VOICED[_e] = _f, _d, _c, _v
df_crepe = pd.DataFrame(_rows).T.astype(float)
df_crepe.index.name = 'excerpt_id'
_cond22 = pd.Series({e: e.rsplit('_', 1)[1] for e in CREPE_DIAG})

# ── 3. Octave correction, per excerpt ─────────────────────────────────────────
_oct = pd.Series({e: d['n_octave_fixed'] for e, d in CREPE_DIAG.items()})
print('\n── Octave correction (|cents − ±100 ms running median| ∈ [1000, 1400] → ∓1200) ──')
print(f'   corrected frames in total: {int(_oct.sum())} · excerpts with ≥ 1 correction: '
      f'{int((_oct > 0).sum())}/{len(_oct)}')
for _e, _n in _oct[_oct > 0].sort_values(ascending=False).items():
    print(f'     {_e:<32} {int(_n):>5} frames')

# ── 4. Note segmentation: old (Phase B boundary rule) vs DEPLOYED (B2 plateaus + the B3 merge-range
#       check, no re-articulation split) + the re-articulation detector evaluated REPORT-ONLY ──
# The detector is forced on (reart_enabled=True) on the cached tracks + env_db only to document the
# B3 decision; none of these counts reaches a feature. Everything here is audio-derived — never a
# rating, never Y.
_cond_order = ('MEC', 'EXP', 'EXG')
_on = {e: segment_notes_detail(CREPE_CENTS[e], CREPE_VOICED[e], NOTE_MIN_MS_TIMING, env_db=CREPE_ENV_DB[e],
                               reart_dip_db=REART_DIP_DB, reart_enabled=True) for e in CREPE_DIAG}
_seg = pd.DataFrame({'old_v2': {e: d['n_notes_60_v2'] for e, d in CREPE_DIAG.items()},
                     'deployed': {e: d['n_notes_60'] for e, d in CREPE_DIAG.items()},
                     'reart_on_6dB': {e: int(len(_on[e]['notes'])) for e in CREPE_DIAG},
                     'reart_splits_6dB': {e: int(sum(k == 're-articulation split' for k in _on[e]['kinds']))
                                          for e in CREPE_DIAG}})
_seg['condition'] = _cond22
_seg['piece'] = [e.rsplit('_', 1)[0] for e in _seg.index]
if not REART_ENABLED:
    assert (_seg['reart_on_6dB'] == _seg['deployed'] + _seg['reart_splits_6dB']).all(), \
        '§2.2: detector-on notes ≠ deployed notes + splits (every split must add exactly one note)'
    assert all(d['n_reart_splits'] == 0 for d in CREPE_DIAG.values()), \
        '§2.2: REART_ENABLED is False but a deployed note starts at a re-articulation split'
print(f'\n── Note segmentation — notes ≥ {NOTE_MIN_MS_TIMING} ms: plateaus (±{PLATEAU_TOL_CENTS:g} c, '
      f'range ≤ {PLATEAU_MAX_RANGE_CENTS:g} c), merge < {PLATEAU_MERGE_CENTS:g} c / ≤ {PLATEAU_MERGE_GAP_MS:g} ms '
      f'with the merged range ≤ {PLATEAU_MAX_RANGE_CENTS:g} c (B3) · re-articulation splits: '
      + ('OFF (deployed)' if not REART_ENABLED else f'ON at {REART_DIP_DB:g} dB (⚠️ not B3)'))
print(f'   total notes: Phase B rule {int(_seg.old_v2.sum())} → deployed {int(_seg.deployed.sum())} '
      f'(report only — detector on at {REART_DIP_DB:g} dB: {int(_seg.reart_on_6dB.sum())}, '
      f'{int(_seg.reart_splits_6dB.sum())} splits)')
print(f'\n   REPORT-ONLY evidence table — re-articulation detector OFF (deployed) vs ON at {REART_DIP_DB:g} dB')
print('   condition   Phase B rule   detector OFF   detector ON   splits')
for _c in _cond_order:
    _g = _seg[_seg.condition == _c]
    print(f'   {_c:<9} {int(_g.old_v2.sum()):>14} {int(_g.deployed.sum()):>14} {int(_g.reart_on_6dB.sum()):>13} '
          f'{int(_g.reart_splits_6dB.sum()):>8}')
print('   per excerpt (Phase B rule · OFF = deployed · ON · splits):')
print(_seg[['old_v2', 'deployed', 'reart_on_6dB', 'reart_splits_6dB']].astype(int).to_string())
print(f'   REART_DIP_DB sensitivity with the detector FORCED ON (report only; the features use OFF):')
_mz_dep = ' / '.join(str(int(_seg.deployed.get(f'Mozart_VC4_{c}', -1))) for c in _cond_order)
print(f'      OFF  : {int(_seg.deployed.sum()):>5} notes (' + ' · '.join(
    f'{c} {int(_seg[_seg.condition == c].deployed.sum())}' for c in _cond_order)
      + f') · Mozart_VC4 MEC/EXP/EXG = {_mz_dep}   (score: 19 each) ← deployed')
CREPE_REART_REPORT = {}
for _db in REART_SENS_DB:
    _cnt = {e: segment_notes_detail(CREPE_CENTS[e], CREPE_VOICED[e], NOTE_MIN_MS_TIMING,
                                    env_db=CREPE_ENV_DB[e], reart_dip_db=_db, reart_enabled=True)
            for e in CREPE_DIAG}
    _n = pd.Series({e: int(len(d['notes'])) for e, d in _cnt.items()})
    CREPE_REART_REPORT[float(_db)] = _n
    print(f'     {_db:>4g} dB : {int(_n.sum()):>5} notes (' + ' · '.join(
        f'{c} {int(_n[_cond22 == c].sum())}' for c in _cond_order) + ') · Mozart_VC4 MEC/EXP/EXG = '
          + ' / '.join(str(int(_n.get(f'Mozart_VC4_{c}', -1))) for c in _cond_order) + '   (score: 19 each)')
print('   → the split count grows MEC → EXG for the same scores (vibrato amplitude modulation, not bow '
      'changes) and no threshold\n     gives 19 notes in all three Mozart_VC4 renditions — the detector '
      'stays OFF for the features (Phase B3).')

# ── 4b. Condition-independence of the DEPLOYED note counts (label-free; report only) ──────────
# The same score is played in MEC, EXP and EXG, so the number of notes should not depend on the
# condition. Friedman test over the pieces with all three renditions (blocks = pieces, treatments =
# conditions), on the deployed counts and — for comparison — on the detector-on counts. ✅ if the
# deployed p ≥ 0.05, ⚠️ otherwise (printed, never raised).
from scipy.stats import friedmanchisquare


def _friedman_p(tab):
    """(statistic, p) of the Friedman test on a pieces × conditions table; a table whose counts are
    tied within every piece has no rank variance → (0.0, 1.0) (perfectly condition-independent)."""
    if (tab.nunique(axis=1) == 1).all():
        return 0.0, 1.0
    _res = friedmanchisquare(*[tab[c].to_numpy(dtype=float) for c in tab.columns])
    return float(_res.statistic), float(_res.pvalue)


CREPE_NOTE_COUNTS = (_seg.pivot(index='piece', columns='condition', values='deployed')
                     .reindex(columns=list(_cond_order)).dropna().astype(int))
_cnt_on = (_seg.pivot(index='piece', columns='condition', values='reart_on_6dB')
           .reindex(columns=list(_cond_order)).dropna().astype(int))
_fs_dep, _fp_dep = _friedman_p(CREPE_NOTE_COUNTS)
_fs_on, _fp_on = _friedman_p(_cnt_on)
_rng_dep = CREPE_NOTE_COUNTS.max(axis=1) - CREPE_NOTE_COUNTS.min(axis=1)
_rng_on = _cnt_on.max(axis=1) - _cnt_on.min(axis=1)
CREPE_NOTECOUNT_FRIEDMAN = dict(n_pieces=int(len(CREPE_NOTE_COUNTS)), statistic=_fs_dep, p=_fp_dep,
                                ok=bool(_fp_dep >= 0.05), statistic_on=_fs_on, p_on=_fp_on,
                                totals={c: int(CREPE_NOTE_COUNTS[c].sum()) for c in _cond_order},
                                totals_on={c: int(_cnt_on[c].sum()) for c in _cond_order},
                                range_median=float(_rng_dep.median()), range_max=int(_rng_dep.max()))
print(f'\n── Condition-independence of the note counts ({len(CREPE_NOTE_COUNTS)} pieces with MEC / EXP / EXG; '
      'the same score in every condition)')
print('   deployed totals: ' + ' · '.join(f'{c} {CREPE_NOTECOUNT_FRIEDMAN["totals"][c]}' for c in _cond_order)
      + '   |   detector on: ' + ' · '.join(f'{c} {CREPE_NOTECOUNT_FRIEDMAN["totals_on"][c]}' for c in _cond_order))
print(f'   per-piece range (max − min over the 3 renditions): deployed median {_rng_dep.median():.1f}, '
      f'max {int(_rng_dep.max())} · detector on median {_rng_on.median():.1f}, max {int(_rng_on.max())}')
print(pd.concat([CREPE_NOTE_COUNTS, _rng_dep.rename('range'), _cnt_on.add_suffix('_on'),
                 _rng_on.rename('range_on')], axis=1).to_string())
print(f'   Friedman χ²(2) deployed = {_fs_dep:.3f}, p = {_fp_dep:.4f}   |   detector on = {_fs_on:.3f}, '
      f'p = {_fp_on:.4f}')
print(('✅' if CREPE_NOTECOUNT_FRIEDMAN['ok'] else '⚠️ ') + ' deployed note counts '
      + ('do not depend on the condition (p ≥ 0.05)' if CREPE_NOTECOUNT_FRIEDMAN['ok'] else
         'DEPEND on the condition (p < 0.05) — the segmentation reacts to the performance style; '
         'read the note-based timing features with care'))

# ── 5. Vibrato (B8b): windows, declared substitutions, note-length sensitivity ─────
_no_notes = df_crepe.index[df_crepe['crepe_vibrato_n_notes'] == 0].tolist()
_no_vib   = df_crepe.index[(df_crepe['crepe_vibrato_n_notes'] > 0)
                           & df_crepe['crepe_vibrato_rate_hz'].isna()].tolist()
print(f'\n── Vibrato (B8b + B3 band-edge rule) — notes ≥ {VIB_MIN_NOTE_MS} ms, {VIB_WIN_MS} ms windows every '
      f'{VIB_HOP_MS} ms, sinusoid fit {VIB_RATE_BAND[0]:g}–{VIB_RATE_BAND[1]:g} Hz (step {VIB_RATE_STEP:g}), '
      f'R² ≥ {VIB_FIT_R2_MIN:g}, half-extent ≥ {VIB_MIN_EXTENT:g} c, best rate NOT on the grid edge '
      f'({VIB_RATE_BAND[0]:g} / {VIB_RATE_BAND[1]:g} Hz) ──')
print('   note-length sensitivity (report only):')
for _ms in VIB_NOTE_SENS_MS:
    _cnt = pd.Series({e: d['notes_by_min'][_ms] for e, d in CREPE_DIAG.items()})
    print(f'     ≥ {_ms:>4} ms : {int(_cnt.sum()):>4} analysed notes · excerpts with none: '
          f'{int((_cnt == 0).sum())}/{len(_cnt)}{"   ← deployed" if _ms == VIB_MIN_NOTE_MS else ""}')
print(f'   windows: {sum(d["n_windows"] for d in CREPE_DIAG.values())} · vibrato windows: '
      f'{sum(d["n_vib_windows"] for d in CREPE_DIAG.values())} · notes with ≥ 1 vibrato window: '
      f'{sum(d["n_vib_notes"] for d in CREPE_DIAG.values())} of {int(df_crepe["crepe_vibrato_n_notes"].sum())}')
_MEC22e = [e for e in CREPE_DIAG if e.endswith('_MEC')]
for _lab, _es in (('all excerpts', list(CREPE_DIAG)), (f'the {len(_MEC22e)} MEC', _MEC22e)):
    print(f'   band-edge rule (B3), {_lab}: {sum(CREPE_DIAG[e]["n_edge_windows"] for e in _es)} windows with the '
          f'best rate on the grid edge · {sum(CREPE_DIAG[e]["n_edge_only"] for e in _es)} of them rejected ONLY '
          f'by this rule (R² and half-extent would have passed) · of '
          f'{sum(CREPE_DIAG[e]["n_windows"] for e in _es)} windows')
# DECLARED substitution 1 — notes exist but no vibrato window: the rate is UNDEFINED (not 0)
# → median rate of the other excerpts; extent 0 and coverage 0 are meaningful and kept.
_med_rate = float(df_crepe['crepe_vibrato_rate_hz'].median(skipna=True))
df_crepe.loc[_no_vib, 'crepe_vibrato_rate_hz'] = _med_rate
print(f'   declared substitution — no vibrato window (rate undefined) → median rate {_med_rate:.3f} Hz: '
      f'{len(_no_vib)} excerpt(s) {_no_vib if _no_vib else ""}')
# DECLARED substitution 2 — no note ≥ VIB_MIN_NOTE_MS at all: rate, extent and coverage are all
# undefined → medians of the other excerpts (Phase B decision 1).
for _col in ('crepe_vibrato_extent_cents', 'crepe_vibrato_coverage'):
    df_crepe.loc[_no_notes, _col] = float(df_crepe.loc[~df_crepe.index.isin(_no_notes), _col].median())
df_crepe.loc[_no_notes, 'crepe_vibrato_rate_hz'] = _med_rate   # median over excerpts with a defined rate
print(f'   declared substitution — no note ≥ {VIB_MIN_NOTE_MS} ms (rate/extent/coverage undefined) → '
      f'medians: {len(_no_notes)} excerpt(s) {_no_notes if _no_notes else ""}')
VIB_N_SUBST_NOVIB, VIB_N_SUBST_NONOTES = len(_no_vib), len(_no_notes)   # read by §10.2
if len(_no_notes) > 10:
    print(f'⚠️⚠️  {len(_no_notes)} of {len(df_crepe)} excerpts have no note ≥ {VIB_MIN_NOTE_MS} ms — the '
          'vibrato family is UNRELIABLE at this note length (median-substituted for those excerpts)')
assert not df_crepe[CREPE_CANDIDATES].isna().any().any(), \
    f'§2.2: undefined CREPE values remain: {df_crepe[CREPE_CANDIDATES].isna().sum().to_dict()}'
print(f'   rate: {df_crepe["crepe_vibrato_rate_hz"].nunique()} distinct values '
      f'({df_crepe["crepe_vibrato_rate_hz"].min():.2f}–{df_crepe["crepe_vibrato_rate_hz"].max():.2f} Hz, '
      f'must lie in {VIB_RATE_BAND[0]:g}–{VIB_RATE_BAND[1]:g}) · half-extent '
      f'{df_crepe["crepe_vibrato_extent_cents"].min():.1f}–{df_crepe["crepe_vibrato_extent_cents"].max():.1f} c · '
      f'coverage {df_crepe["crepe_vibrato_coverage"].min():.2f}–{df_crepe["crepe_vibrato_coverage"].max():.2f}')
assert df_crepe['crepe_vibrato_rate_hz'].between(*VIB_RATE_BAND).all(), \
    '§2.2: a vibrato rate lies outside VIB_RATE_BAND — the clamp is not working'

# ── 6. Portamento (B9b), F0 range (B10), segmentation diagnostics ──────────────
_ng = sum(d['n_glides'] for d in CREPE_DIAG.values())
_nt = sum(d['n_transitions'] for d in CREPE_DIAG.values())
_gd = np.concatenate([np.asarray(d['glide_durs'], dtype=float) for d in CREPE_DIAG.values()])
_ge = np.concatenate([np.asarray(d['glide_exts'], dtype=float) for d in CREPE_DIAG.values()])
_gx = np.concatenate([np.asarray(d['glide_excess'], dtype=float) for d in CREPE_DIAG.values()])
_rs = pd.Series([t['reason'] for d in CREPE_DIAG.values() for t in d['transitions']]).value_counts()
print(f'\n── Portamento (B9b) — {_ng} glides in {_nt} transitions ({100 * _ng / max(_nt, 1):.1f} %) · rule: '
      f'rise {GLIDE_DUR_MS[0]:g}–{GLIDE_DUR_MS[1]:g} ms, excess over the Viterbi ramp ≥ {GLIDE_MIN_EXCESS_MS:g} ms, '
      f'|interval| {GLIDE_INTERVAL[0]:g}–{GLIDE_INTERVAL[1]:g} c, sign ≥ {GLIDE_SIGN_MIN:g}')
if _ng:
    print(f'   glide rise {_gd.min():.0f}–{_gd.max():.0f} ms (median {np.median(_gd):.0f}) · excess '
          f'{_gx.min():.0f}–{_gx.max():.0f} ms · extent {_ge.min():.0f}–{_ge.max():.0f} c')
print('   transitions by verdict: ' + ' · '.join(f'{k} {int(v)}' for k, v in _rs.items()))
print(f'── F0 range (B10, P95 − P5 on the corrected track): max {df_crepe["crepe_f0_range_cents"].max():.0f} c '
      f'(old max − min descriptor: 5343 c)')
_close = sum(d['n_close_splits'] for d in CREPE_DIAG.values())
_n60 = sum(d['n_notes_60'] for d in CREPE_DIAG.values())
print(f'── Segmentation (report only): {_n60} notes ≥ {NOTE_MIN_MS_TIMING} ms; adjacent pitch-change notes < '
      f'{BOUNDARY_CENTS:.0f} c apart (possible spurious splits; re-articulations excluded): {_close}'
      + ('   ⚠️' if _close > 0.05 * max(_n60, 1) else ''))

# ── 7. CALIBRATION (report only) — MEC as a NEGATIVE CONTROL for the detection thresholds ──
# The performance instructions say MEC is played with NO vibrato and NO portamento, so every vibrato
# window / glide detected in a MEC excerpt counts as a false positive. Only the AUDIO-derived
# detections of the 20 MEC excerpts are used — never a rating, never Y — and nothing is fitted or
# chosen automatically: the deployed constants above stay as declared; this only shows how far
# from them the false-positive rate would move.
_MEC22 = [e for e in CREPE_DIAG if e.endswith('_MEC')]
_wm = np.array([w for e in _MEC22 for w in CREPE_DIAG[e]['vib_windows']], dtype=float).reshape(-1, 3)
print(f'\n── Calibration (MEC = negative control: {len(_MEC22)} excerpts, {len(_wm)} analysis windows) — '
      'false-positive vibrato windows, % (rows R² ≥, columns half-extent ≥; every cell applies the B3 '
      'band-edge rule: windows whose best rate is on the 4 / 8 Hz grid edge never count)')
_fp_vib = np.nan
if len(_wm):
    print('     R² \\ ext ' + ''.join(f'{x:>8g} c' for x in (2, 3, 4, 5)))
    for _r2 in (0.3, 0.4, 0.5, 0.6, 0.7):
        _line = []
        for _x in (2, 3, 4, 5):
            _p = float(np.mean((_wm[:, 0] >= _r2) & (_wm[:, 1] >= _x) & (_wm[:, 2] < 0.5)))
            _dep = np.isclose(_r2, VIB_FIT_R2_MIN) and np.isclose(_x, VIB_MIN_EXTENT)
            if _dep:
                _fp_vib = _p
            _line.append(f'{100 * _p:>8.1f}{"*" if _dep else " "} ')
        print(f'     {_r2:>8g}  ' + ''.join(_line))
    print('     (* = deployed)')
    # the deployed cell must be exactly the MEC vibrato-window share of the descriptors (same rule)
    assert (not np.isfinite(_fp_vib)
            or np.isclose(_fp_vib, sum(CREPE_DIAG[e]['n_vib_windows'] for e in _MEC22) / len(_wm))), \
        '§2.2: the deployed calibration cell does not reproduce the MEC vibrato windows of the descriptors'
_tm22 = [t for e in _MEC22 for t in CREPE_DIAG[e]['transitions'] if t['counted']]
print(f'── Calibration — false-positive glides per MEC transition ({len(_tm22)} transitions), by '
      'GLIDE_MIN_EXCESS_MS (other glide rules as deployed):')
_fp_gl = np.nan
for _x in (25.0, 50.0, 75.0, 100.0):
    _p = (float(np.mean([bool(t['passes_except_excess'] and t['excess_ms'] >= _x) for t in _tm22]))
          if _tm22 else np.nan)
    if np.isclose(_x, GLIDE_MIN_EXCESS_MS):
        _fp_gl = _p
    print(f'     excess ≥ {_x:>5g} ms : {100 * _p:5.1f} %' + ('   ← deployed' if np.isclose(_x, GLIDE_MIN_EXCESS_MS) else ''))
if np.isfinite(_fp_vib) and _fp_vib > 0.05:
    print(f'⚠️  deployed vibrato setting gives {100 * _fp_vib:.1f} % false-positive windows on MEC (> 5 %)')
if np.isfinite(_fp_gl) and _fp_gl > 0.05:
    print(f'⚠️  deployed glide setting gives {100 * _fp_gl:.1f} % false-positive glides per MEC transition (> 5 %)')

df_crepe = df_crepe[CREPE_CANDIDATES + CREPE_DIAGNOSTIC]
df_crepe.to_csv(CACHE_CREPE)
print(f'\n✅ §2.2 — {df_crepe.shape[0]} excerpts × {len(CREPE_CANDIDATES)} candidates '
      f'(+ {len(CREPE_DIAGNOSTIC)} diagnostic) → {CACHE_CREPE}')

In [ ]:
# ── §2.2b Controlled CREPE features (UNSUPERVISED) ──────────────────
# NEEDS : CREPE_CANDIDATES, CREPE_DIAGNOSTIC, df_crepe, df_expr, unsup_forward_select
# MAKES : CREPE_FORWARD, df_crepe_fwd
# FEEDS : §2.5, §9.2, §7.5
# ─── UNSUPERVISED INVARIANT ──────────────────────────────────────────────────
# This cell reads ONLY the feature matrix. It never touches Y, y_*, or any label.
# That is precisely what makes a fixed, once-up-front selection safe to feed into
# the downstream LOOCV models: a supervised selection here would leak the held-out
# fold into the choice of features. StandardScaler stays INSIDE every CV pipeline.
# ═══════════════════════════════════════════════════════════════════════════════
# §8B · Controlled CREPE features — conservative gate, preserves named techniques
# ═══════════════════════════════════════════════════════════════════════════════
# With only 10 interpretable features this will likely keep all/most of them — the
# point is the UNIFORM controlled gate, not aggressive reduction. corr_max=0.95 means
# a feature the attribution analyses name is only dropped if it is a true duplicate.
# Phase B: the gate runs over the explicit CREPE_CANDIDATES (a-priori order) — the diagnostic
# column(s) in CREPE_DIAGNOSTIC are never forwarded.
CREPE_FORWARD = unsup_forward_select(df_crepe.reindex(df_expr['excerpt_id']), corr_max=0.95,
                                     name='crepe', order=CREPE_CANDIDATES)
assert not set(CREPE_FORWARD) & set(CREPE_DIAGNOSTIC), '§2.2b: a diagnostic column was forwarded'
df_crepe_fwd  = df_crepe[CREPE_FORWARD]
print(f'\nCREPE: {len(CREPE_CANDIDATES)} candidates → {len(CREPE_FORWARD)} forwarded features '
      f'(diagnostic, never forwarded: {CREPE_DIAGNOSTIC})')

# df_crepe (FULL) stays intact — §9.1 condition-delta, §9.3 SHAP and §7.3/§7.3b name
# individual techniques and must keep the complete interpretable vocabulary (CREPE_CANDIDATES).


## Part 2 · Feature extraction — §2.3 Essentia DSP · dynamics / timbre / tonal
Covers the expressive dimensions CREPE cannot see. The **full block is extracted and
saved**; only a small curated subset (`ESSENTIA_FORWARD`) is concatenated into the
feature matrices — see the forwarding cell below.

The forwarding-selection cell that follows is **§2.3b**: it decides which of the extracted block actually reaches the models (`ESSENTIA_FORWARD`).

**Phase B (B11, B12).** The spectral flux is computed on **L1-normalised** frame spectra
(`essentia_spectral_flux_norm_mean/_std`; a fresh `es.Flux()` per excerpt), so it measures spectral
*shape* change rather than loudness change; the old raw flux survives only as the diagnostic
`essentia_spectral_flux_raw_mean`. The dynamic range is P95 − P5 of frame RMS dB **after a silence
gate** (drop frames below −60 dBFS or more than 40 dB below the excerpt's P95). The cell prints the
share of gated frames, a 3 × 3 gate-sensitivity sweep (Spearman ρ vs the deployed gate) and
r(flux, LUFS) raw vs normalised. Caches: `essentia_features_v2.csv/.npz`, `rms_frames_v2.npz`.

In [ ]:
# ── §2.3 Essentia DSP extraction (full block) ───────────────────────
# NEEDS : audio_files, df_expr, dr_gate, DR_ABS_GATE_DB, DR_REL_GATE_DB, l1_normalize_spectrum,
#          OUTPUT_DIR
# MAKES : _stats, CACHE_ESSENTIA, CACHE_ESSENTIA_NPZ, CACHE_RMS, df_essentia, ESS_FRAME, ESS_HOP,
#          ESSENTIA_DIAGNOSTIC, ESSENTIA_LOUD_SR, ESSENTIA_SR, ESS_RMS_FRAMES,
#          extract_essentia_features · essentia_features_v2.csv, essentia_features_v2.npz,
#          rms_frames_v2.npz
# FEEDS : §2.3b, §2.3c, §2.5, §9.1, §7.3b
# ═══════════════════════════════════════════════════════════════════════════════
# §11 · Essentia DSP — Dynamics / Timbre / Tonal   (ENABLED)
# ═══════════════════════════════════════════════════════════════════════════════
# CREPE is pitch-only. This block adds the expressive dimensions it cannot see:
#   dynamics (loudness / LUFS / dynamic range) · timbre (flux, centroid, rolloff,
#   log-attack-time, dissonance) · tonal (key strength, HPCP entropy).
# Phase B: B11 the spectral flux is computed on an L1-NORMALISED magnitude spectrum (spectral
# shape change, not level change — the raw flux is kept as a diagnostic-only column); B12 the
# dynamic range is measured on SILENCE-GATED frames (frames < −60 dBFS or > 40 dB below the
# excerpt's P95 frame are not "dynamics"), and the same gate excludes those frames from the flux
# statistics.
#
# Plain DSP-only `essentia` is deliberate — `essentia-tensorflow` bundles a TF build
# that conflicts with the pinned torch 2.3.1. TF is only needed for MusiCNN (§12),
# which stays a skip stub.
#
# HARD import: fail loudly if missing. No try/except → librosa degradation (§3.1 policy).
import essentia
import essentia.standard as es
essentia.log.warningActive = False              # silence per-frame algorithm chatter
print(f'✅ essentia {essentia.__version__}')

ESSENTIA_SR      = 16000    # spectral / tonal analysis rate (matches the CREPE load rate)
ESSENTIA_LOUD_SR = 44100    # EBU R128 K-weighting filters are defined at 44.1 kHz
ESS_FRAME, ESS_HOP = 2048, 1024
ESSENTIA_DIAGNOSTIC = ['essentia_spectral_flux_raw_mean']   # diagnostic only — never a candidate


def _stats(name, arr):
    """mean + std of a per-frame feature (matches the CREPE mean/std convention)."""
    a = np.asarray([v for v in arr if np.isfinite(v)], dtype=float)
    if a.size == 0:
        return {f'essentia_{name}_mean': 0.0, f'essentia_{name}_std': 0.0}
    return {f'essentia_{name}_mean': float(a.mean()),
            f'essentia_{name}_std' : float(a.std())}


def extract_essentia_features(audio_path):
    """Full Essentia DSP block for one excerpt → (features dict, per-frame RMS dB).

    Nothing is dropped here — the whole block is kept and written to disk. The
    curated subset that actually reaches the models is chosen in the next cell.
    Scalar (whole-excerpt) descriptors get a `_mean` column only; per-frame
    descriptors get `_mean` and `_std`.
    """
    feats = {}
    y = es.MonoLoader(filename=audio_path, sampleRate=ESSENTIA_SR)()
    y = np.ascontiguousarray(np.asarray(y, dtype=np.float32))

    # ── DYNAMICS ──────────────────────────────────────────────────────────────
    # EBU R128 expects 44.1 kHz and a 2-channel signal → duplicate the mono channel.
    y44    = np.asarray(es.MonoLoader(filename=audio_path, sampleRate=ESSENTIA_LOUD_SR)(),
                        dtype=np.float32)
    stereo = np.ascontiguousarray(np.column_stack([y44, y44]).astype(np.float32))
    _, short_term, integrated, lrange = es.LoudnessEBUR128(sampleRate=ESSENTIA_LOUD_SR)(stereo)
    feats['essentia_loudness_integrated_lufs_mean'] = float(integrated)   # scalar
    feats['essentia_loudness_range_lu_mean']        = float(lrange)       # scalar
    feats.update(_stats('loudness_short_term', short_term))               # per frame

    rms = np.array([es.RMS()(fr) for fr in es.FrameGenerator(
        y, frameSize=ESS_FRAME, hopSize=ESS_HOP, startFromZero=True)])
    rms_db = 20 * np.log10(np.clip(rms, 1e-9, None))
    # B12 — dynamic range on silence-gated frames (P95 − P5 of the KEPT frames)
    dr, keep = dr_gate(rms_db, DR_ABS_GATE_DB, DR_REL_GATE_DB)
    if not np.isfinite(dr):
        raise ValueError(f'{audio_path}: fewer than 2 frames survive the dynamic-range gate')
    feats['essentia_dynamic_range_db_mean'] = dr                           # scalar

    # ── TIMBRE + TONAL (per frame) ────────────────────────────────────────────
    w, spec = es.Windowing(type='hann'), es.Spectrum()
    flux_norm_a = es.Flux()                              # stateful → a NEW instance per excerpt
    flux_raw_a  = es.Flux()                              # diagnostic: the pre-Phase-B raw flux
    cent_a  = es.Centroid(range=ESSENTIA_SR / 2)
    roll_a  = es.RollOff(sampleRate=ESSENTIA_SR)
    peaks_a = es.SpectralPeaks(sampleRate=ESSENTIA_SR, magnitudeThreshold=1e-5,
                               minFrequency=40, maxFrequency=5000, orderBy='frequency')
    diss_a  = es.Dissonance()
    hpcp_a  = es.HPCP(sampleRate=ESSENTIA_SR)
    ent_a   = es.Entropy()

    flux_n, flux_r, cent, roll, diss, hpcp_ent, hpcps = [], [], [], [], [], [], []
    for fr in es.FrameGenerator(y, frameSize=ESS_FRAME, hopSize=ESS_HOP, startFromZero=True):
        S = spec(w(fr))
        flux_n.append(flux_norm_a(l1_normalize_spectrum(S)))   # B11 — level-invariant flux
        flux_r.append(flux_raw_a(S))
        cent.append(cent_a(S)); roll.append(roll_a(S))
        f, m = peaks_a(S)
        if len(f) >= 2:
            diss.append(diss_a(f, m))
            h   = np.asarray(hpcp_a(f, m), dtype=np.float32)
            tot = float(h.sum())
            hpcps.append(h)
            hpcp_ent.append(float(ent_a(np.ascontiguousarray(h / tot))) if tot > 0 else np.nan)
    if len(flux_n) != len(keep):
        raise RuntimeError(f'{audio_path}: {len(flux_n)} spectral frames vs {len(keep)} RMS frames — '
                           'the B12 gate mask cannot be aligned with the flux frames')
    flux_n, flux_r = np.asarray(flux_n, dtype=float), np.asarray(flux_r, dtype=float)
    feats.update(_stats('spectral_flux_norm', flux_n[keep]))    # gated frames only (B11 + B12)
    feats['essentia_spectral_flux_raw_mean'] = float(np.mean(flux_r[keep]))   # diagnostic only
    feats.update(_stats('spectral_centroid', cent))
    feats.update(_stats('spectral_rolloff',  roll))
    feats.update(_stats('dissonance',        diss))
    feats.update(_stats('hpcp_entropy',      hpcp_ent))

    # tonal strength = Key confidence on the excerpt-averaged HPCP (scalar)
    if hpcps:
        hp_mean = np.ascontiguousarray(np.mean(np.vstack(hpcps), axis=0).astype(np.float32))
        feats['essentia_key_strength_mean'] = float(es.Key()(hp_mean)[2])
    else:
        feats['essentia_key_strength_mean'] = 0.0

    # ── TIMBRE · log-attack-time per detected onset (expressive attack slope) ──
    env   = es.Envelope(sampleRate=ESSENTIA_SR)(y)
    lat_a = es.LogAttackTime(sampleRate=ESSENTIA_SR)
    def _lat(seg):
        out = lat_a(np.ascontiguousarray(np.asarray(seg, dtype=np.float32)))
        return float(out[0]) if isinstance(out, (tuple, list)) else float(out)
    onsets = np.asarray(es.OnsetRate()(y)[0], dtype=float)
    lats = []
    for t in onsets:
        s0 = int(t * ESSENTIA_SR); s1 = s0 + int(0.30 * ESSENTIA_SR)
        seg = env[s0:s1]
        if len(seg) >= 64 and float(np.max(seg)) > 0:
            lats.append(_lat(seg))
    if not lats and len(env) >= 64:                 # degenerate signal → whole-excerpt attack
        lats = [_lat(env)]
    feats.update(_stats('log_attack_time', lats))
    return feats, rms_db


CACHE_ESSENTIA     = os.path.join(OUTPUT_DIR, 'essentia_features_v2.csv')
CACHE_ESSENTIA_NPZ = os.path.join(OUTPUT_DIR, 'essentia_features_v2.npz')
CACHE_RMS          = os.path.join(OUTPUT_DIR, 'rms_frames_v2.npz')
if os.path.exists(CACHE_ESSENTIA) and os.path.exists(CACHE_RMS):
    df_essentia = pd.read_csv(CACHE_ESSENTIA, index_col='excerpt_id')
    _zr = np.load(CACHE_RMS)
    ESS_RMS_FRAMES = {e: _zr[e] for e in _zr.files}
    print(f'📂 Loaded Essentia cache  {df_essentia.shape} + per-frame RMS ({len(ESS_RMS_FRAMES)})')
else:
    print('🔄 Extracting Essentia DSP features...')
    _ess, ESS_RMS_FRAMES = {}, {}
    for e, p in tqdm(audio_files.items(), desc='Essentia'):
        _ess[e], ESS_RMS_FRAMES[e] = extract_essentia_features(p)
    df_essentia = pd.DataFrame(_ess).T
    df_essentia.index.name = 'excerpt_id'
    df_essentia.to_csv(CACHE_ESSENTIA)
    np.savez_compressed(CACHE_RMS, **ESS_RMS_FRAMES)
    print(f'✅ Saved  {df_essentia.shape} + {CACHE_RMS}')

# FULL block also persisted as .npz (nothing from the extraction is lost)
np.savez(CACHE_ESSENTIA_NPZ,
         columns     = np.array(list(df_essentia.columns), dtype=object),
         excerpt_ids = np.array(list(df_essentia.index),   dtype=object),
         values      = df_essentia.values.astype(float))

print('\n── Essentia FULL block sanity ──')
print(f'   excerpts                  : {df_essentia.shape[0]}')
print(f'   columns                   : {df_essentia.shape[1]}')
print(f'   NaN cells                 : {int(df_essentia.isna().sum().sum())}'
      f'  (rows with any NaN: {int(df_essentia.isna().any(axis=1).sum())})')
_miss_ess = [e for e in df_expr['excerpt_id'] if e not in df_essentia.index]
print(f'   excerpts missing from block: {_miss_ess if _miss_ess else "none ✅"}')
print('\n' + df_essentia.describe().T[['mean', 'std', 'min', 'max']].round(4).to_string())

# ── B12 · % frames gated per excerpt + gate sensitivity sweep (report only) ───
_ids23 = list(df_expr['excerpt_id'])
_gated = pd.Series({e: 100.0 * (1 - dr_gate(ESS_RMS_FRAMES[e])[1].mean()) for e in _ids23})
print(f'\n── B12 dynamic-range gate (abs {DR_ABS_GATE_DB:g} dBFS, rel {DR_REL_GATE_DB:g} dB below P95) ──')
print(f'   frames gated per excerpt: median {_gated.median():.1f} %, max {_gated.max():.1f} % · '
      f'DR max {df_essentia["essentia_dynamic_range_db_mean"].max():.1f} dB')
for _e, _g in _gated[_gated > 30].items():
    print(f'   ⚠️  {_e}: {_g:.1f} % of frames gated (> 30 %)')
_dr_dep = pd.Series({e: dr_gate(ESS_RMS_FRAMES[e])[0] for e in _ids23})
print('   sensitivity — Spearman ρ of each gate variant vs the deployed (−60, 40):')
_low_rho = []
for _ab in (-50.0, -60.0, -70.0):
    _line = []
    for _rl in (30.0, 40.0, 50.0):
        _v = pd.Series({e: dr_gate(ESS_RMS_FRAMES[e], _ab, _rl)[0] for e in _ids23})
        _rho = float(spearmanr(_v, _dr_dep)[0])
        _line.append(f'rel {_rl:g}: ρ={_rho:.3f}')
        if _rho < 0.9:
            _low_rho.append((_ab, _rl, _rho))
    print(f'     abs {_ab:>5g}  ' + '   '.join(_line))
for _ab, _rl, _rho in _low_rho:
    print(f'   ⚠️  gate (abs {_ab:g}, rel {_rl:g}) gives ρ = {_rho:.3f} < 0.9 vs the deployed DR')

# ── B11 · does the normalised flux decouple from loudness? (report only) ──────
_E = df_essentia.reindex(_ids23)
_r_raw  = float(pearsonr(_E['essentia_spectral_flux_raw_mean'],  _E['essentia_loudness_integrated_lufs_mean'])[0])
_r_norm = float(pearsonr(_E['essentia_spectral_flux_norm_mean'], _E['essentia_loudness_integrated_lufs_mean'])[0])
print(f'\n── B11 flux vs integrated LUFS across the 60 excerpts: raw r = {_r_raw:+.3f} · '
      f'L1-normalised r = {_r_norm:+.3f}  (expectation to verify: |r| smaller when normalised)')
print('\n✅ Essentia full block → essentia_features_v2.csv , essentia_features_v2.npz , rms_frames_v2.npz')


In [ ]:
# ── §2.3b Essentia forwarding selection (UNSUPERVISED) ──────────────
# NEEDS : df_essentia, df_expr, ESSENTIA_DIAGNOSTIC, unsup_forward_select
# MAKES : _ESS_PRETTY, _pretty_ess, CORR_MAX, ESSENTIA_CANDIDATES, ESSENTIA_FEAT_LABELS, ESSENTIA_FORWARD
#          ESSENTIA_FULL, STD_MIN
# FEEDS : §2.5, §9.1, §7.3b
# ─── UNSUPERVISED INVARIANT ──────────────────────────────────────────────────
# This cell reads ONLY the feature matrix. It never touches Y, y_*, or any label.
# That is precisely what makes a fixed, once-up-front selection safe to feed into
# the downstream LOOCV models: a supervised selection here would leak the held-out
# fold into the choice of features. StandardScaler stays INSIDE every CV pipeline.
# ═══════════════════════════════════════════════════════════════════════════════
# §11B · Essentia forwarding — which of the FULL block actually reaches the models
# ═══════════════════════════════════════════════════════════════════════════════
# Philosophy: extract everything (cell above — nothing lost), but forward only a small
# subset chosen A-PRIORI for complementarity to CREPE (pitch-only): dynamics + timbre
# + tonal. Redundancy pruning is delegated to the SHARED gate `unsup_forward_select`
# (same near-constant + correlation prune as MERT §2.1b and CREPE §2.2b).

ESSENTIA_FULL = list(df_essentia.columns)

# One representative per expressive dimension CREPE misses, ranked by a-priori
# preference (earlier = preferred when two are redundant).
ESSENTIA_CANDIDATES = [
    # dynamics
    'essentia_loudness_integrated_lufs_mean',
    'essentia_loudness_range_lu_mean',
    'essentia_loudness_short_term_std',
    'essentia_dynamic_range_db_mean',
    # timbre  (Phase B B11: flux on the L1-normalised spectrum; the raw flux is diagnostic only)
    'essentia_spectral_flux_norm_mean',
    'essentia_spectral_centroid_mean',
    'essentia_spectral_rolloff_mean',
    'essentia_dissonance_mean',
    'essentia_log_attack_time_mean',
    # tonal
    'essentia_key_strength_mean',
    'essentia_hpcp_entropy_mean',
]
CORR_MAX, STD_MIN = 0.90, 1e-8

# Row-align to the 60 modelled excerpts, then run the shared gate over the curated
# candidates in a-priori order (identical behaviour to the previous inline version).
_E_cand = df_essentia.reindex(df_expr['excerpt_id'])
ESSENTIA_FORWARD = unsup_forward_select(_E_cand, corr_max=CORR_MAX, name='essentia',
                                        eps=STD_MIN, order=ESSENTIA_CANDIDATES)

assert not set(ESSENTIA_FORWARD) & set(ESSENTIA_DIAGNOSTIC), '§2.3b: a diagnostic column was forwarded'

_ESS_PRETTY = {'essentia_spectral_flux_norm_mean': 'Spectral Flux, L1-normalised (mean)',
               'essentia_spectral_flux_norm_std' : 'Spectral Flux, L1-normalised (SD)',
               'essentia_dynamic_range_db_mean'  : 'Dynamic Range, gated (dB)'}

def _pretty_ess(col):
    if col in _ESS_PRETTY:
        return _ESS_PRETTY[col]
    base, suffix = col.replace('essentia_', ''), ''
    if   base.endswith('_std') : base, suffix = base[:-4], ' (SD)'
    elif base.endswith('_mean'): base, suffix = base[:-5], ' (mean)'
    return base.replace('_', ' ').title() + suffix

ESSENTIA_FEAT_LABELS = {c: _pretty_ess(c) for c in ESSENTIA_FORWARD}

print(f'\nEssentia: FULL block {len(ESSENTIA_FULL)} cols (all on disk) · '
      f'{len(ESSENTIA_CANDIDATES)} a-priori candidates · {len(ESSENTIA_FORWARD)} forwarded')
if not (6 <= len(ESSENTIA_FORWARD) <= 10):
    print(f'   ⚠️  forwarded dim count = {len(ESSENTIA_FORWARD)}, outside the intended 6–10 range')
print('✅ Saved: essentia_forwarding.csv')


## Part 2 · Feature extraction — §2.3c Recording-level sanity check (report only)

**Purpose (supervisor's request):** make sure no WAV was recorded at a clearly different *level* —
higher preamp gain, or the violin closer to the microphone — than the others. The stimuli played
to the participants were **not loudness-normalized**, so level differences are part of the stimulus
and **nothing here is normalized, dropped or changed**: this cell only flags.

The key test separates *playing louder* from *recording louder*. Playing louder raises the signal
but not the room-noise floor; more gain or a closer microphone raises **both**. So each file's quiet
frames are compared against the median floor of the other recordings. **Phase B (B-NF):** the floor
is the median RMS dB of *noise-like* 50 ms frames — spectral flatness ≥ `FLAT_MIN` = 0.2 over
50 Hz–8 kHz (white noise ≈ 0.56, sustained violin tone ≈ 0.01–0.05) **and** below the excerpt's
20th-percentile RMS. With fewer than 20 such frames (< 1 s) the file has no measurable silence and the
test is reported as **inconclusive** (the Phase A 5th-percentile floor was unreliable exactly there).
The flatness distribution and the flag counts at `FLAT_MIN` ∈ {0.1, 0.2, 0.3} are printed. An
optional session map (`SESSION_MAP_CSV`, columns `excerpt_id, session`) adds per-session LUFS / floor
and a session × condition crosstab; while it is not uploaded the cell says so and skips.

| Flag | Rule |
|---|---|
| level outlier | robust z of integrated LUFS, `(x − median)/(1.4826·MAD)`, \|z\| > 3 |
| within-piece outlier | \|LUFS − piece median\| > 10 LU (loose on purpose: EXG is ≈ +4 LU louder by design) |
| recording-chain change | \|floor − reference median floor\| > 6 dB (only when the floor is conclusive: ≥ 20 noise-like frames) |
| clipping | sample peak ≥ −0.1 dBFS or ≥ 10 samples with \|x\| ≥ 0.999 |

Per condition, a higher EXG **floor** would point to a recording-chain change; a higher EXG **LUFS**
with an equal floor is the intended performance dynamics.


In [ ]:
# ── §2.3c Recording-level sanity check (REPORT ONLY) ────────────────
# NEEDS : audio_files, df_essentia, df_expr, frame_power_spectra, OUTPUT_DIR, SESSION_MAP_CSV,
#          spectral_flatness
# MAKES : df_reclevel, df_session_check (only if the session map exists), FLAT_* / QC_*
#          thresholds · recording_level_check.csv
# FEEDS : — (report only: nothing is normalised, dropped or changed; flags never raise)
# ═══════════════════════════════════════════════════════════════════════════════
# §2.3c · Was any WAV recorded at a clearly different level (gain / mic distance)?
# ═══════════════════════════════════════════════════════════════════════════════
# Phase B (B-NF): the noise floor is no longer the 5th-percentile frame — which is still music
# when an excerpt has no silence — but the median level of NOISE-LIKE frames: spectrally flat
# (flatness ≥ FLAT_MIN in 50 Hz–8 kHz) AND among the excerpt's quietest 20 % of frames. At least
# FLAT_MIN_FRAMES such frames (≥ 1 s) are required, else the floor test is "inconclusive".
import soundfile as sf

QC_ROBUST_Z_MAX    = 3.0    # |robust z| of LUFS above this = level outlier (standard robust-outlier rule)
QC_PIECE_LU_MAX    = 10.0   # |LUFS − piece median| above this = within-piece outlier (EXG ≈ +4 LU by design → loose)
QC_FLOOR_DEV_DB    = 6.0    # |floor − reference floor| above this = chain change (6 dB = amplitude ×2, beyond room-noise drift)
QC_PEAK_DBFS_MAX   = -0.1   # sample peak at/above this = clipping risk
QC_CLIP_LEVEL      = 0.999  # |x| at/above this counts as a clipped sample
QC_CLIP_N_MAX      = 10     # this many clipped samples = clipping
QC_FRAME_S         = 0.050  # frame RMS: 50 ms frames …
QC_HOP_S           = 0.025  # … 25 ms hop
QC_SILENCE_RMS     = 1e-7   # frame RMS below this = digital silence → excluded, counted separately
FLAT_MIN           = 0.2    # noise-like frame: spectral flatness ≥ 0.2 (white noise ≈ 0.56; room noise is
                            # coloured but typically well above sustained violin tones ≈ 0.01–0.05)
FLAT_SENS          = (0.1, 0.2, 0.3)   # sensitivity of the flag counts to FLAT_MIN (report only)
FLAT_RMS_PCTL      = 20     # … AND RMS below the excerpt's 20th-percentile frame
FLAT_MIN_FRAMES    = 20     # ≥ 20 noise-like frames (≥ 1 s of 50 ms frames) else "inconclusive"
SESSION_CONFINED_FRAC = 0.80   # a condition with ≥ 80 % of its excerpts in one session is "confined"

_LUFS = 'essentia_loudness_integrated_lufs_mean'
_qc_ids = list(df_expr['excerpt_id'])
assert _LUFS in df_essentia.columns, f'§2.3c: {_LUFS} missing from df_essentia'
_qc_lufs = df_essentia.reindex(_qc_ids)[_LUFS].astype(float)
assert _qc_lufs.notna().all(), \
    f'§2.3c: integrated LUFS missing for {list(_qc_lufs[_qc_lufs.isna()].index)}'
_qc_noaudio = [e for e in _qc_ids if e not in audio_files]
assert not _qc_noaudio, f'§2.3c: no WAV for {_qc_noaudio}'
print(f'§2.3c · {len(_qc_ids)} WAVs, LUFS present for all {int(_qc_lufs.notna().sum())}')


def _noise_floor(rms_db, flat, quiet, thr):
    """Median RMS dB of noise-like frames (flatness ≥ thr AND quiet); NaN if < FLAT_MIN_FRAMES."""
    m = (flat >= thr) & quiet
    return (float(np.median(rms_db[m])) if m.sum() >= FLAT_MIN_FRAMES else np.nan), int(m.sum())


_qc_rows, _flat_all = [], []
for _e in _qc_ids:
    _x, _sr = sf.read(audio_files[_e], dtype='float64', always_2d=True)   # original rate, all channels
    _ax   = np.abs(_x)
    _peak = float(_ax.max()) if _ax.size else 0.0
    _mono = _x.mean(axis=1)
    _fl, _hp = int(round(QC_FRAME_S * _sr)), int(round(QC_HOP_S * _sr))
    _frames = (np.lib.stride_tricks.sliding_window_view(_mono, _fl)[::_hp]
               if len(_mono) >= _fl else _mono[None, :])
    _rms   = np.sqrt(np.mean(_frames ** 2, axis=1))
    _sil   = _rms < QC_SILENCE_RMS
    _frames, _rms = _frames[~_sil], _rms[~_sil]
    _db    = 20 * np.log10(_rms) if len(_rms) else np.array([np.nan])
    _P, _f = frame_power_spectra(_frames, _sr)
    _flat  = spectral_flatness(_P, _f)
    _quiet = _db < np.percentile(_db, FLAT_RMS_PCTL)
    _flat_all.append(_flat)
    _row = dict(excerpt_id=_e, sample_rate=int(_sr), n_channels=int(_x.shape[1]),
                duration_s=len(_mono) / _sr, peak_dbfs=20 * np.log10(max(_peak, 1e-12)),
                n_clipped=int((_ax >= QC_CLIP_LEVEL).sum()), n_frames=int(len(_rms) + _sil.sum()),
                n_silent_frames=int(_sil.sum()), active_db=float(np.median(_db)),
                p5_db=float(np.percentile(_db, 5)),
                headroom_db=float(np.median(_db) - np.percentile(_db, 5)),   # info only (Phase A rule)
                flat_median=float(np.median(_flat)),
                flat_median_quiet=float(np.median(_flat[_quiet])) if _quiet.any() else np.nan)
    for _thr in FLAT_SENS:
        _row[f'floor_db_flat{_thr:g}'], _row[f'n_noise_frames_flat{_thr:g}'] = \
            _noise_floor(_db, _flat, _quiet, _thr)
    _qc_rows.append(_row)
df_reclevel = pd.DataFrame(_qc_rows).set_index('excerpt_id')
df_reclevel.insert(0, 'condition', df_expr.set_index('excerpt_id').loc[_qc_ids, 'condition'].values)
df_reclevel.insert(0, 'piece', df_expr.set_index('excerpt_id').loc[_qc_ids, 'piece'].values)
df_reclevel['lufs'] = _qc_lufs.values
df_reclevel['floor_db'] = df_reclevel[f'floor_db_flat{FLAT_MIN:g}']        # the deployed estimate

# ── level outlier: robust z of LUFS ─────────────────────────────────────────
_med = float(df_reclevel['lufs'].median())
_mad = float(np.median(np.abs(df_reclevel['lufs'] - _med)))
df_reclevel['lufs_robust_z'] = ((df_reclevel['lufs'] - _med) / (1.4826 * _mad)
                                if _mad > 0 else np.nan)
if not _mad > 0:
    print('⚠️  MAD of LUFS is 0 — robust z undefined, level-outlier test skipped')
# ── within-piece outlier ──────────────────────────────────────────────────────
df_reclevel['lufs_vs_piece_median'] = (df_reclevel['lufs']
                                       - df_reclevel.groupby('piece')['lufs'].transform('median'))


# ── recording-chain change: noise-like floor vs the reference (conclusive files only) ──
def _chain_flags(floor):
    conc = floor.notna()
    ref = float(floor[conc].median()) if conc.any() else np.nan
    dev = floor - ref
    return conc, ref, dev, dev.abs() > QC_FLOOR_DEV_DB

_conc, _ref_floor, _dev, _chain = _chain_flags(df_reclevel['floor_db'])
df_reclevel['floor_conclusive']      = _conc
df_reclevel['floor_vs_reference_db'] = _dev
df_reclevel['flag_level']    = df_reclevel['lufs_robust_z'].abs() > QC_ROBUST_Z_MAX
df_reclevel['flag_in_piece'] = df_reclevel['lufs_vs_piece_median'].abs() > QC_PIECE_LU_MAX
df_reclevel['flag_chain']    = _chain
df_reclevel['flag_clipping'] = ((df_reclevel['peak_dbfs'] >= QC_PEAK_DBFS_MAX)
                                | (df_reclevel['n_clipped'] >= QC_CLIP_N_MAX))
_FLAGS = ['flag_level', 'flag_in_piece', 'flag_chain', 'flag_clipping']

print(f'   LUFS median {_med:.2f} (MAD {_mad:.2f}) · reference floor (median over '
      f'{int(_conc.sum())} conclusive files, ≥ {FLAT_MIN_FRAMES} noise-like frames) = {_ref_floor:.1f} dBFS')
if int((~_conc).sum()):
    print(f'   noise-floor test INCONCLUSIVE (< {FLAT_MIN_FRAMES} noise-like frames) for '
          f'{int((~_conc).sum())} files: {list(df_reclevel.index[~_conc])}')
print(f'   digital-silence frames excluded: {int(df_reclevel["n_silent_frames"].sum())} '
      f'in {int((df_reclevel["n_silent_frames"] > 0).sum())} files')

_fa = np.concatenate(_flat_all)
print(f'\n── Spectral flatness (50 Hz–8 kHz) over all {len(_fa)} non-silent frames: '
      + ' · '.join(f'P{q} {np.percentile(_fa, q):.3f}' for q in (5, 25, 50, 75, 95))
      + f' · frames ≥ {FLAT_MIN}: {100 * np.mean(_fa >= FLAT_MIN):.1f} %')
print('── FLAT_MIN sensitivity (report only): conclusive files / chain flags')
for _thr in FLAT_SENS:
    _c2, _r2, _d2, _f2 = _chain_flags(df_reclevel[f'floor_db_flat{_thr:g}'])
    print(f'   FLAT_MIN = {_thr:g}: {int(_c2.sum()):>2} conclusive · reference floor {_r2:6.1f} dBFS · '
          f'{int(_f2.sum()):>2} chain flags{"   ← deployed" if _thr == FLAT_MIN else ""}')

print('\n── Per condition (a higher EXG FLOOR → chain change; higher EXG LUFS with an equal floor → '
      'intended dynamics) ──')
_pc = df_reclevel.groupby('condition').agg(
    n=('lufs', 'size'), lufs_mean=('lufs', 'mean'), peak_dbfs_mean=('peak_dbfs', 'mean'),
    n_conclusive=('floor_conclusive', 'sum'), floor_db_mean_conclusive=('floor_db', 'mean'))
print(_pc.reindex([c for c in ('MEC', 'EXP', 'EXG') if c in _pc.index]).round(2).to_string())

# ── Optional session map (an optional INPUT, not a fallback) ─────────────────
if os.path.exists(SESSION_MAP_CSV):
    _sm = pd.read_csv(SESSION_MAP_CSV)
    assert {'excerpt_id', 'session'} <= set(_sm.columns), \
        f'§2.3c: {SESSION_MAP_CSV} needs the columns excerpt_id, session'
    _sm_missing = sorted(set(_qc_ids) - set(_sm['excerpt_id']))
    assert not _sm_missing, f'§2.3c: session map lacks {_sm_missing}'
    df_reclevel['session'] = _sm.set_index('excerpt_id').loc[_qc_ids, 'session'].values
    df_session_check = df_reclevel.groupby('session').agg(
        n=('lufs', 'size'), lufs_mean=('lufs', 'mean'), floor_db_mean=('floor_db', 'mean'),
        n_conclusive=('floor_conclusive', 'sum'))
    print('\n── Session map: per-session level and floor ──')
    print(df_session_check.round(2).to_string())
    _ct = pd.crosstab(df_reclevel['session'], df_reclevel['condition'])
    print('\n── session × condition ──')
    print(_ct.to_string())
    for _cond in _ct.columns:
        _share = _ct[_cond].max() / _ct[_cond].sum()
        if _share >= SESSION_CONFINED_FRAC:
            print(f'⚠️  condition {_cond}: {100 * _share:.0f} % of its excerpts come from session '
                  f'{_ct[_cond].idxmax()} — condition and session are (nearly) confounded')
else:
    print(f'\nℹ️  session map not provided — session check skipped (upload it to enable): {SESSION_MAP_CSV}')

_reason = {'flag_level'   : lambda r: f'level outlier: robust z = {r.lufs_robust_z:+.2f} (LUFS {r.lufs:.1f})',
           'flag_in_piece': lambda r: f'within-piece outlier: {r.lufs_vs_piece_median:+.1f} LU vs piece median',
           'flag_chain'   : lambda r: f'recording-chain change: noise-like floor {r.floor_db:.1f} dBFS = '
                                      f'{r.floor_vs_reference_db:+.1f} dB vs reference',
           'flag_clipping': lambda r: f'clipping: peak {r.peak_dbfs:+.2f} dBFS, {r.n_clipped} samples ≥ {QC_CLIP_LEVEL}'}
_flagged = df_reclevel[df_reclevel[_FLAGS].any(axis=1)]
df_reclevel.to_csv(os.path.join(OUTPUT_DIR, 'recording_level_check.csv'))
if len(_flagged):
    print(f'\n⚠️  RECORDING-LEVEL FLAGS — {len(_flagged)} file(s) (report only, nothing changed):')
    for _e, _r in _flagged.iterrows():
        for _f in _FLAGS:
            if bool(_r[_f]):
                print(f'   ⚠️  {_e:<32} {_reason[_f](_r)}')
else:
    print('\n✅ no recording-level outliers')
print('✅ §2.3c — recording_level_check.csv')


## Part 2 · Feature extraction — §2.4 DTW timing (ACTIVE) · madmom / MusiCNN (skip stubs)
DTW is **enabled** — it closes the *χρονικές παραλλαγές* (timing / rubato) axis named in the brief, which no other block measures. Two complementary families, both forwarded through the same unsupervised gate as every other tool (§2.4b):

| Family | What it measures | Condition-independent? |
|---|---|---|
| **(1) reference-free timing** (`timing_*`, Phase B B13) — tempo spread `timing_tempo_logsd` (SD of log2 local tempo, 151-frame tempogram × log-normal prior centred on 100 BPM, 40–200 BPM), local tempo, pulse clarity, note rate and IOI mean / SD / CV from CREPE note onsets | rubato as *timing variability*, computed per excerpt with no reference | **yes** — all 60 excerpts graded, MEC included |
| **(2) within-piece DTW deviation** (`dtw_*`) — warp-path deviation, local-slope variability, duration/path ratios vs the piece's own MEC rendition | rubato *magnitude relative to the mechanical take* (`librosa.sequence.dtw` on chroma) | **no** — MEC is the reference, so its values are the self-alignment identity (0 / 1) by construction |

Family (2) therefore partially encodes the condition. That is not hidden: §2.4b splits the families, family (2) is kept **out of every predictive model input** (Part 6 onwards) by default (`DTW_DEV_IN_MODELS=False`) and stays only in the descriptive views (§4.2, §9.1) where MEC = 0 is exactly the point, and §4.2 flags it in the dose-response table.

The note onsets come from the cached CREPE tracks (`segment_notes` of §2.2 — B2 plateau segmentation
with the B3 merge range check, notes ≥ 60 ms; a legato onset is the start of the transition region),
not from librosa onset detection, which fires on bow and vibrato modulation. **Phase B3:** the B2
re-articulation splits (10 ms envelope dips) are **disabled** (`REART_ENABLED = False`) — they followed
the vibrato amplitude modulation and grew with expressiveness, which inflated note rate / IOI with the
condition. A repeated same-pitch note with no silence between is therefore one note; that bias is set
by the score and ≈ equal in a piece's three renditions. Only 4 of the 7 timing features use note onsets
(`timing_note_rate`, `timing_ioi_mean_s`, `timing_ioi_sd_s`, `timing_ioi_cv`); tempo variability, local
tempo and pulse clarity come from the tempogram and are unaffected. `timing_octave_jump_frac` is
diagnostic only. Cache: `dtw_features_v4.csv` (B3: the note onsets changed again, so the v2 / v3 caches
are retired).

In [ ]:
# ── §2.4 DTW timing / rubato (χρονικές παραλλαγές) ──────────────────
# NEEDS : audio_files, CONDITIONS, CREPE_CENTS, CREPE_ENV_DB, CREPE_VOICED, CREPE_STEP_MS, df_expr,
#          local_tempo_from_tempogram, OUTPUT_DIR, timing_note_features, TEMPO_BPM_RANGE
# MAKES : CACHE_DTW, df_dtw, DTW_DEV, DTW_DEV_SELF, DTW_HOP, DTW_REFFREE, DTW_SLOPE_WIN_S,
#          DTW_SR, DTW_TG_WIN, TIMING_DIAGNOSTIC, _DTW_PRETTY, _dtw_chroma, _dtw_split,
#          _pretty_dtw, dtw_pair_stats, extract_timing_reffree · dtw_features_v4.csv
# FEEDS : §2.4b, §2.5, §3.3b, §9.1, §4.2, §7.3, §7.3b, §7.5
# ═══════════════════════════════════════════════════════════════════════════════
# §2.4 · DTW / timing — the χρονικές παραλλαγές axis of the brief   (ENABLED)
# ═══════════════════════════════════════════════════════════════════════════════
# CREPE is pitch-only and Essentia is dynamics/timbre/tonal — neither measures WHEN
# notes happen. This block closes the timing gap with TWO complementary families:
#
#   (1) REFERENCE-FREE timing / rubato descriptors (`timing_*`, Phase B B13) — computed
#       independently per excerpt, so ALL 60 excerpts (MEC included) get graded values and the
#       family is condition-independent. They measure what their names say:
#       · note rate and IOIs from NOTE ONSETS of the CREPE tracks (§2.2 segment_notes, ≥ 60 ms;
#         B2 plateau notes + the B3 merge range check; Phase B3 DISABLES the B2 re-articulation
#         splits — REART_ENABLED = False — because they followed the vibrato amplitude modulation,
#         so a repeated note re-bowed without an unvoiced gap is ONE note, a score-set bias that is
#         ≈ equal in the three renditions of a piece) — not librosa onset detection, which fires on
#         bow/vibrato modulation in legato. env_db is still passed (signature unchanged) and ignored;
#       · local tempo from a 151-frame (≈ 3.5 s) tempogram × a log-normal prior (40–200 BPM);
#         timing_tempo_logsd = SD of log2(local tempo) — an octave-invariant rubato proxy.
#       They are NOT DTW — hence `timing_*` (the old `dtw_*` names were misleading).
#
#   (2) WITHIN-PIECE DTW deviation — the actual alignment step. Per piece, the
#       chroma sequences of EXP and EXG are DTW-aligned against the SAME piece's
#       MEC rendition (librosa.sequence.dtw); the warping path yields rubato
#       magnitude relative to the mechanical reference.
#       ⚠️ MEC is the reference, so its own family-(2) values are the SELF-ALIGNMENT
#       identity BY CONSTRUCTION (deviation = 0, ratio = 1) — meaningful for
#       attribution/dose-response (mechanical ⇒ no rubato vs itself) but it means
#       family (2) partially encodes CONDITION. Not hidden: §2.4b splits the two
#       families, prints the property, and keeps family (2) out of every predictive
#       model input (descriptive only since Phase C2, §4.1).
#
# HARD dependency: librosa (already imported in §0.2). No try/except fallback.
DTW_SR          = 22050    # modest analysis rate — timing, not timbre
DTW_HOP         = 512      # ≈ 43 frames/s → ~23 ms resolution
DTW_TG_WIN      = 151      # tempogram window (frames) ≈ 3.5 s of context (Phase B: was 384 ≈ 8.9 s)
DTW_SLOPE_WIN_S = 1.0      # smoothing window (s) for the local warping slope

# Family (1) — reference-free, condition-INDEPENDENT (all 60 excerpts graded)
DTW_REFFREE = [
    'timing_local_tempo_bpm',
    'timing_tempo_logsd',
    'timing_note_rate',
    'timing_ioi_mean_s',
    'timing_ioi_sd_s',
    'timing_ioi_cv',
    'timing_pulse_clarity',
]
TIMING_DIAGNOSTIC = ['timing_octave_jump_frac']        # diagnostic only — never forwarded
# Family (2) — within-piece DTW deviation vs the piece's own MEC rendition
DTW_DEV = [
    'dtw_path_deviation_mean',
    'dtw_path_deviation_max',
    'dtw_local_slope_std',
    'dtw_duration_ratio',
    'dtw_path_length_ratio',
]
# What MEC gets by construction: the value its own self-alignment would produce.
DTW_DEV_SELF = {'dtw_path_deviation_mean': 0.0,
                'dtw_path_deviation_max' : 0.0,
                'dtw_local_slope_std'    : 0.0,
                'dtw_duration_ratio'     : 1.0,
                'dtw_path_length_ratio'  : 1.0}


def _dtw_split(eid):
    """`PieceName_ConditionCode` → (piece, condition). condition None if unparsable."""
    for c in CONDITIONS:
        if eid.endswith('_' + c):
            return eid[:-(len(c) + 1)], c
    return eid, None


def extract_timing_reffree(y, sr, cents, voiced, env_db):
    """Family (1) — timing descriptors from ONE excerpt, no reference (B13).
    Note rate / IOIs from the CREPE note onsets (B2/B3 plateau segmentation; env_db = the §2.2 10 ms
    RMS envelope, accepted but ignored since B3 — no re-articulation split); local tempo / rubato / pulse clarity
    from the tempogram (prior-weighted argmax, interior frames)."""
    oenv  = librosa.onset.onset_strength(y=y, sr=sr, hop_length=DTW_HOP)
    tg    = librosa.feature.tempogram(onset_envelope=oenv, sr=sr,
                                      hop_length=DTW_HOP, win_length=DTW_TG_WIN)
    tempi = librosa.tempo_frequencies(tg.shape[0], hop_length=DTW_HOP, sr=sr)
    tp    = local_tempo_from_tempogram(tg, tempi, win_length=DTW_TG_WIN)
    nt    = timing_note_features(cents, voiced, duration_s=len(y) / sr, step_ms=CREPE_STEP_MS,
                                 env_db=env_db)
    return {'timing_local_tempo_bpm' : tp['local_tempo_bpm'],
            'timing_tempo_logsd'     : tp['tempo_logsd'],
            'timing_note_rate'       : nt['note_rate'],
            'timing_ioi_mean_s'      : nt['ioi_mean_s'],
            'timing_ioi_sd_s'        : nt['ioi_sd_s'],
            'timing_ioi_cv'          : nt['ioi_cv'],
            'timing_pulse_clarity'   : tp['pulse_clarity'],
            'timing_octave_jump_frac': tp['octave_jump_frac']}


def _dtw_chroma(y, sr):
    """L2-normalised CQT chroma — the alignment representation (pitch-class content,
    invariant to the loudness/timbre differences the conditions also carry)."""
    C = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=DTW_HOP)
    C = np.nan_to_num(np.asarray(C, dtype=float), nan=0.0, posinf=0.0, neginf=0.0) + 1e-8
    return C / np.linalg.norm(C, axis=0, keepdims=True)      # no zero-norm frame → cosine safe


def dtw_pair_stats(C_var, C_ref, sr=DTW_SR):
    """Family (2) — align one rendition (C_var) to its piece's MEC reference (C_ref).

    Deviations are LENGTH-NORMALISED: both axes are mapped to [0,1] first, so
    |path − diagonal| is a fraction of the excerpt, comparable across pieces.
    """
    # NB: librosa's `Y=` is the second SEQUENCE, not the label matrix — this cell
    # reads no labels (the unsupervised invariant holds here as in every §2.x cell).
    _, wp = librosa.sequence.dtw(X=C_var, Y=C_ref, metric='cosine',
                                 subseq=False, backtrack=True)
    wp = np.asarray(wp)[::-1]                                # → ascending order
    n_var, n_ref = int(C_var.shape[1]), int(C_ref.shape[1])

    u   = wp[:, 0].astype(float) / max(n_var - 1, 1)         # position within the variant
    v   = wp[:, 1].astype(float) / max(n_ref - 1, 1)         # position within the reference
    dev = np.abs(u - v)                                      # rubato magnitude vs MEC

    # ── single-valued warping function j(i), smoothed, then differentiated ──
    s      = pd.Series(wp[:, 1].astype(float)).groupby(wp[:, 0].astype(int)).mean()
    j_of_i = s.reindex(np.arange(n_var)).interpolate(limit_direction='both').to_numpy()
    k      = max(3, int(round(DTW_SLOPE_WIN_S * sr / DTW_HOP)))
    j_sm   = np.convolve(j_of_i, np.ones(k) / k, mode='valid') if len(j_of_i) > 4 * k else j_of_i
    slope  = np.gradient(j_sm) if len(j_sm) > 1 else np.zeros(1)
    expect = n_ref / max(n_var, 1)                           # slope under uniform tempo scaling
    return {'dtw_path_deviation_mean': float(np.mean(dev)),
            'dtw_path_deviation_max' : float(np.max(dev)),
            'dtw_local_slope_std'    : float(np.std(slope / max(expect, 1e-9))),
            'dtw_duration_ratio'     : float(n_var / max(n_ref, 1)),      # duration vs MEC (>1 = slower)
            'dtw_path_length_ratio'  : float(len(wp) / max(n_var, n_ref))}  # path/diagonal


CACHE_DTW = os.path.join(OUTPUT_DIR, 'dtw_features_v4.csv')   # B3 onsets (v2 = Phase B, v3 = B2 — retired)
if os.path.exists(CACHE_DTW):
    df_dtw = pd.read_csv(CACHE_DTW, index_col='excerpt_id')
    print(f'📂 Loaded DTW cache  {df_dtw.shape}')
else:
    print('🔄 Extracting timing (CREPE note onsets + tempogram) and DTW features...')
    _reffree, _chroma = {}, {}
    for e, p in tqdm(audio_files.items(), desc='DTW timing'):
        _y, _sr    = librosa.load(p, sr=DTW_SR, mono=True)
        try:
            _reffree[e] = extract_timing_reffree(_y, _sr, CREPE_CENTS[e], CREPE_VOICED[e],
                                                 CREPE_ENV_DB[e])
        except ValueError as _err:
            raise ValueError(f'§2.4: {e}: {_err}') from None
        _chroma[e]  = _dtw_chroma(_y, _sr)

    # ── family (2): group by piece, align every rendition to that piece's MEC ──
    _by_piece = {}
    for e in _reffree:
        _p, _c = _dtw_split(e)
        if _c is not None:
            _by_piece.setdefault(_p, {})[_c] = e

    _dev, _no_ref, _no_variant = {}, [], []
    for _piece, _ids in tqdm(sorted(_by_piece.items()), desc='DTW align'):
        if 'MEC' not in _ids:
            _no_ref.append(_piece)                      # cannot align → stays NaN
            continue
        _Cref = _chroma[_ids['MEC']]
        _dev[_ids['MEC']] = dict(DTW_DEV_SELF)          # reference vs itself, by construction
        for _c in ('EXP', 'EXG'):
            if _c not in _ids:
                _no_variant.append(f'{_piece}/{_c}')
                continue
            _dev[_ids[_c]] = dtw_pair_stats(_chroma[_ids[_c]], _Cref)

    if _no_ref:
        print(f'⚠️⚠️  NO MEC REFERENCE for piece(s): {_no_ref}')
        print('      → every rendition of those pieces keeps NaN family-(2) values '
              '(never silent-zero).')
    if _no_variant:
        print(f'⚠️⚠️  rendition(s) absent, nothing to align: {_no_variant}')

    df_dtw  = pd.DataFrame(_reffree).T[DTW_REFFREE + TIMING_DIAGNOSTIC]
    _dev_df = (pd.DataFrame(_dev).T.reindex(df_dtw.index).reindex(columns=DTW_DEV)
               if _dev else pd.DataFrame(np.nan, index=df_dtw.index, columns=DTW_DEV))
    df_dtw = pd.concat([df_dtw, _dev_df], axis=1)
    df_dtw.index.name = 'excerpt_id'
    df_dtw.to_csv(CACHE_DTW)
    print(f'✅ Saved  {df_dtw.shape}')

_vals = df_dtw.to_numpy(dtype=float)
print('\n── DTW / timing FULL block sanity ──')
print(f'   excerpts                     : {df_dtw.shape[0]}')
print(f'   columns                      : {df_dtw.shape[1]}  '
      f'({len(DTW_REFFREE)} reference-free timing + {len(TIMING_DIAGNOSTIC)} diagnostic + '
      f'{len(DTW_DEV)} DTW-deviation)')
print(f'   inf cells                    : {int(np.isinf(_vals).sum())}   (must be 0)')
print(f'   NaN cells in family (1)      : {int(df_dtw[DTW_REFFREE].isna().sum().sum())}   '
      f'(must be 0 — reference-free, every excerpt gets a value)')
print(f'   family-(2) rows with any NaN : {int(df_dtw[DTW_DEV].isna().any(axis=1).sum())}'
      f'   (expected 0 while every piece has all three renditions)')
_miss_dtw = [e for e in df_expr['excerpt_id'] if e not in df_dtw.index]
print(f'   excerpts missing from block  : {_miss_dtw if _miss_dtw else "none ✅"}')
print(f'   tempogram: {DTW_TG_WIN}-frame window, prior-weighted argmax in '
      f'{TEMPO_BPM_RANGE[0]:g}–{TEMPO_BPM_RANGE[1]:g} BPM · octave-jump fraction (diagnostic): '
      f'median {df_dtw["timing_octave_jump_frac"].median():.3f}, max {df_dtw["timing_octave_jump_frac"].max():.3f}')
print('   ⚠️  BY CONSTRUCTION: MEC rows carry the self-alignment identity for family (2) '
      '(deviation = 0, ratio = 1)')
print('      → family (2) partially encodes CONDITION; family (1) does not. '
      'See the split in §2.4b.')
print('\n' + df_dtw.describe().T[['mean', 'std', 'min', 'max']].round(4).to_string())


_DTW_PRETTY = {
    'timing_local_tempo_bpm'   : 'Local Tempo (mean BPM)',
    'timing_tempo_logsd'       : 'Tempo Variability (SD log2)',
    'timing_note_rate'         : 'Note Rate (/s)',
    'timing_ioi_mean_s'        : 'IOI (mean s)',
    'timing_ioi_sd_s'          : 'IOI (SD s)',
    'timing_ioi_cv'            : 'IOI Coeff. Var.',
    'timing_pulse_clarity'     : 'Pulse Clarity',
    'timing_octave_jump_frac'  : 'Tempo Octave Jumps (fraction)',
    'dtw_path_deviation_mean'  : 'DTW Deviation vs MEC (mean)',
    'dtw_path_deviation_max'   : 'DTW Deviation vs MEC (max)',
    'dtw_local_slope_std'      : 'DTW Slope Variability (SD)',
    'dtw_duration_ratio'       : 'Duration ratio vs MEC (>1 = slower)',
    'dtw_path_length_ratio'    : 'DTW Path Elasticity',
}

def _pretty_dtw(col):
    """Readable label for a DTW / timing column — mirrors `_pretty_ess` (§2.3b)."""
    if col in _DTW_PRETTY:
        return _DTW_PRETTY[col]
    base, suffix = col.replace('dtw_', '').replace('timing_', ''), ''
    if   base.endswith('_std') : base, suffix = base[:-4], ' (SD)'
    elif base.endswith('_mean'): base, suffix = base[:-5], ' (mean)'
    return base.replace('_', ' ').title() + suffix

print('\n✅ DTW / timing full block → dtw_features_v4.csv')


In [ ]:
# ── §2.4b DTW forwarding selection (UNSUPERVISED) ───────────────────
# NEEDS : CORR_MAX, df_dtw, df_expr, DTW_DEV, DTW_REFFREE, STD_MIN, TIMING_DIAGNOSTIC,
#          _pretty_dtw, unsup_forward_select
# MAKES : df_dtw_attr, DTW_CANDIDATES, DTW_DEV_IN_MODELS
#          DTW_FEAT_LABELS, DTW_FORWARD, DTW_FORWARD_DEV, DTW_FORWARD_REFFREE
#          DTW_FULL, DTW_MODEL_COLS · dtw_forwarding.csv
# FEEDS : §2.5, §4.1 (df_dtw_attr → the interpretable layer), §4.2, §7.1, §7.3, §7.3b, §7.5, §9.1
# ─── UNSUPERVISED INVARIANT ──────────────────────────────────────────────────
# This cell reads ONLY the feature matrix. It never touches Y, y_*, or any label.
# That is precisely what makes a fixed, once-up-front selection safe to feed into
# the downstream LOOCV models: a supervised selection here would leak the held-out
# fold into the choice of features. StandardScaler stays INSIDE every CV pipeline.
# ═══════════════════════════════════════════════════════════════════════════════
# §2.4b · DTW forwarding — which of the FULL timing block reaches the models
# ═══════════════════════════════════════════════════════════════════════════════
# Same philosophy and the SAME shared gate as MERT (§2.1b), CREPE (§2.2b) and
# Essentia (§2.3b): extract everything (cell above — nothing lost on disk), forward
# a curated subset in a-priori order. Reference-free timing first (it is the honest,
# condition-independent signal), DTW-deviation second.

DTW_FULL = list(df_dtw.columns)

DTW_CANDIDATES = [
    # ── family (1) · reference-free timing / rubato (`timing_*`, B13) — graded for ALL 60 ──
    'timing_tempo_logsd',            # octave-invariant tempo spread — the headline rubato proxy
    'timing_ioi_cv',
    'timing_pulse_clarity',
    'timing_note_rate',
    'timing_local_tempo_bpm',
    'timing_ioi_sd_s',
    'timing_ioi_mean_s',
    # ── family (2) · within-piece DTW deviation vs the piece's MEC rendition ──
    'dtw_path_deviation_mean',
    'dtw_local_slope_std',
    'dtw_path_deviation_max',
    'dtw_duration_ratio',
    'dtw_path_length_ratio',
]
# TIMING_DIAGNOSTIC (timing_octave_jump_frac) is deliberately NOT a candidate.

# Row-align to the 60 modelled excerpts, then run the shared gate over the curated
# candidates in a-priori order (identical call pattern to §2.3b).
_D_cand = df_dtw.reindex(df_expr['excerpt_id'])
DTW_FORWARD = unsup_forward_select(_D_cand, corr_max=CORR_MAX, name='dtw',
                                   eps=STD_MIN, order=DTW_CANDIDATES)
assert not set(DTW_FORWARD) & set(TIMING_DIAGNOSTIC), '§2.4b: a diagnostic column was forwarded'

DTW_FEAT_LABELS = {c: _pretty_dtw(c) for c in DTW_FORWARD}

# ── The family split, carried explicitly downstream ─────────────────────────
DTW_FORWARD_REFFREE = [c for c in DTW_FORWARD if c in DTW_REFFREE]
DTW_FORWARD_DEV     = [c for c in DTW_FORWARD if c in DTW_DEV]

# Family (2) is 0/1 at MEC BY CONSTRUCTION (self-alignment), so it partially encodes
# the condition. It is DESCRIPTIVE ONLY (Phase C2, §0.3):
#   · DTW_DEV_IN_MODELS — may family (2) enter the predictive matrices (X_C / X_F)?  NO: the
#                         models rely on the graded, condition-independent family (1).
#   · the interpretable layer is split in §4.1: INTERP_PRED_COLS (family 1 only — every
#     cross-validated / model-based analysis) and INTERP_DESC_COLS (families 1 + 2 — only the
#     descriptive §4.2 dose-response and §9.1 condition deltas, where MEC = 0 is the point;
#     flagged ⚑). This replaces the former DTW_DEV_IN_INTERP switch.
DTW_DEV_IN_MODELS = False

DTW_MODEL_COLS = [c for c in DTW_FORWARD if DTW_DEV_IN_MODELS or c in DTW_REFFREE]
df_dtw_attr    = df_dtw[list(DTW_FORWARD)]          # both families — the DESCRIPTIVE view (§4.1)

print(f'\nDTW: FULL block {len(DTW_FULL)} cols (all on disk) · '
      f'{len(DTW_CANDIDATES)} a-priori candidates · {len(DTW_FORWARD)} forwarded '
      f'({len(DTW_FORWARD_REFFREE)} reference-free + {len(DTW_FORWARD_DEV)} DTW-deviation)')
if not (3 <= len(DTW_FORWARD) <= 10):
    print(f'   ⚠️  forwarded dim count = {len(DTW_FORWARD)}, outside the intended 3–10 range')
if not DTW_FORWARD_REFFREE:
    raise RuntimeError('§2.4b: no reference-free timing feature survived the gate — '
                       'the models would then depend entirely on condition-encoding columns.')
print(f'   → model inputs (X_C / X_F)      : {len(DTW_MODEL_COLS)} cols '
      f'{"(family 1 + 2)" if DTW_DEV_IN_MODELS else "(family 1 only — condition-independent)"}')
print(f'   → descriptive view (df_dtw_attr): {df_dtw_attr.shape[1]} cols (family 1 + 2; family 2 is '
      'descriptive only — §4.1 keeps it out of every predictive analysis)')
print('   ⚠️  family-(2) columns are 0 (deviation) / 1 (ratio) at MEC BY CONSTRUCTION — a '
      'MEC→EXP→EXG rise\n       in them is partly built in; family (1) is the honest '
      'monotonicity test (§4.2).')
print('✅ Saved: dtw_forwarding.csv')


## Part 2 · Feature extraction — §2.5 Assemble feature matrices
| Set | Sources | Purpose |
|---|---|---|
| **A** | MERT (768 → PCA-forwarded) | SSL embedding only |
| **B** | CREPE (forwarded subset of the 10 Phase B descriptors) | Interpretable pitch features only |
| **C** | MERT + CREPE + Essentia(`ESSENTIA_FORWARD`) + DTW(`DTW_MODEL_COLS`) | Combined — every tool |
| **D** | MERT + Essentia(`ESSENTIA_FORWARD`) | Isolates Essentia's contribution over MERT |
| **F** | MERT + DTW(`DTW_MODEL_COLS`) | Isolates DTW/timing's contribution over MERT (§7.5) |

DTW enters through `DTW_MODEL_COLS` — by default the **reference-free** timing family only, because the DTW-deviation family is 0/1 at MEC by construction and would partly encode the condition (§2.4b). madmom / MusiCNN remain skip stubs.
`bfv()` **refuses to zero-fill** a missing row for blocks listed in `STRICT_BLOCKS`
(handoff §6.7 — a re-enabled extractor must never become a silent zero-vector); DTW is not strict but a missing row still prints a loud warning.

**B-MERTFULL.** The regression heads use `X_C_REG` / `X_D_REG` / `X_F_REG` (and `MERT_REG` alone):
the same matrices with the PCA-forwarded MERT block replaced by `X_MERT_FULL` (768-d) when
`MERT_REG_REPR = 'full'`. The classification heads keep `X_A`–`X_F`. The cell also asserts that no
cache path carries a pre-Phase-B name. The `*_REG` matrices are stored **raw**: the equal-share block
weighting of the multi-block MERT-full heads (`REG_BLOCK_WEIGHTING`, §0.3) is applied later, inside every
CV fold, by the ridge pipeline (§5.1 / §6.3).

In [ ]:
# ── §2.5 Assemble feature matrices ──────────────────────────────────
# NEEDS : CACHE_* (§2.1–§2.4), CREPE_FORWARD, df_crepe_fwd, df_dtw, df_essentia, df_expr, df_mert,
#          DTW_MODEL_COLS, emb_mert_fwd, ESSENTIA_FORWARD, MERT_FORWARD, MERT_REG_REPR, OLD_CACHE_NAMES
#          (CACHE_* incl. CACHE_CREPE_ENV, §2.2)
# MAKES : bfv, df_dtw_fwd, df_ess_fwd, eids, FEAT_A, FEAT_B, FEAT_C, FEAT_D, FEAT_DTW
#          FEAT_E, FEAT_F, MERT_REG, STRICT_BLOCKS, X_A, X_B, X_C, X_D, X_F, X_MERT_FULL,
#          X_C_REG, X_D_REG, X_F_REG
# FEEDS : §3.3, §3.3b, §3.4, §5.1 (engine guards), §5.3 (null widths), §6.1–§6.5, §7.1, §7.2, §8.1
# Blocks whose absence must NEVER become a silent zero-vector (handoff §6.7):
# a re-enabled extractor that quietly failed would look like real, low-valued data.
STRICT_BLOCKS = ('MERT', 'Essentia')

def bfv(eid, feat_sets):
    parts = []
    for name, src, kind in feat_sets:
        if kind == 'emb':
            v = src.get(eid)
            if v is None:
                msg = f'{name}: NO embedding for excerpt {eid!r}'
                if name in STRICT_BLOCKS:
                    raise KeyError(msg + ' — refusing to zero-fill (STRICT_BLOCKS).')
                print(f'⚠️  {msg} → zero-filled')
                v = np.zeros(list(src.values())[0].flatten().shape)
        else:
            if eid in src.index:
                v = src.loc[eid].values.astype(float)
            else:
                msg = f'{name}: NO feature row for excerpt {eid!r}'
                if name in STRICT_BLOCKS:
                    raise KeyError(msg + ' — refusing to zero-fill (STRICT_BLOCKS).')
                print(f'⚠️  {msg} → zero-filled')
                v = np.zeros(src.shape[1])
        parts.append(np.array(v, dtype=np.float32).flatten())
    return np.concatenate(parts)

eids       = df_expr['excerpt_id'].tolist()
df_ess_fwd = df_essentia[ESSENTIA_FORWARD]        # curated subset only — see §2.3b
# DTW: only the columns §2.4b cleared for MODEL use. By default that is the
# reference-free family, which is graded for all 60 excerpts and does NOT encode the
# condition — unlike the DTW-deviation family, whose MEC values are 0/1 by
# construction and which therefore stays in the attribution views (df_dtw_attr).
df_dtw_fwd = df_dtw[DTW_MODEL_COLS]

# Full-space MERT matrix, row-aligned to eids. NOT a model input: the descriptive
# analyses (§3.4 embedding structure, §7.3/§7.3b correspondence) measure the
# REPRESENTATION, so — like §7.4's distances — they must see the full 768 dims.
X_MERT_FULL = df_mert.reindex(eids).values.astype(float)

# ── Every block enters the models FORWARDED — no tool is dumped in raw ──────
#    A = MERT-forward (§2.1b) · B = CREPE-forward (§2.2b) · E = Essentia-forward (§2.3b)
#    DTW = timing-forward (§2.4b)
FEAT_A   = [('MERT',     emb_mert_fwd, 'emb')]
FEAT_B   = [('CREPE',    df_crepe_fwd, 'tab')]
FEAT_E   = [('Essentia', df_ess_fwd,   'tab')]
FEAT_DTW = [('DTW',      df_dtw_fwd,   'tab')]
FEAT_C   = FEAT_A + FEAT_B + FEAT_E + FEAT_DTW      # Combined = every tool, forwarded
FEAT_D   = FEAT_A + FEAT_E
FEAT_F   = FEAT_A + FEAT_DTW                        # isolates DTW's contribution over MERT

# DTW is deliberately NOT in STRICT_BLOCKS — like CREPE it is interpretable and
# non-essential to the other blocks — but a missing row still WARNS LOUDLY in bfv()
# above before it is zero-filled. It is never silently dropped.
X_A = np.array([bfv(e, FEAT_A) for e in eids])
X_B = np.array([bfv(e, FEAT_B) for e in eids])
X_C = np.array([bfv(e, FEAT_C) for e in eids])
X_D = np.array([bfv(e, FEAT_D) for e in eids])
X_F = np.array([bfv(e, FEAT_F) for e in eids])

for nm, X in [('A', X_A), ('B', X_B), ('C', X_C), ('D', X_D), ('F', X_F)]:
    if np.isnan(X).any():
        cm = np.nanmean(X, axis=0); idx = np.where(np.isnan(X)); X[idx] = np.take(cm, idx[1])
        print(f'⚠️  Set {nm}: NaNs imputed')
print(f'Set A (MERT-fwd: {len(MERT_FORWARD)} PCA comps):        {X_A.shape}')
print(f'Set B (CREPE-fwd: {len(CREPE_FORWARD)} features):        {X_B.shape}')
print(f'Set C (MERT+CREPE+Essentia+DTW, all fwd):   {X_C.shape}')
print(f'Set D (MERT+Essentia, forwarded):           {X_D.shape}')
print(f'Set F (MERT+DTW-fwd: {len(DTW_MODEL_COLS)} timing features):  {X_F.shape}')
print(f'   DTW columns entering the models: {DTW_MODEL_COLS}')
print(f'   full-space MERT kept aside for descriptive use: X_MERT_FULL {X_MERT_FULL.shape}')

# ── B-MERTFULL: REGRESSION matrices — MERT enters the ridge heads at full dimensionality ──
# The classification heads keep the PCA-forwarded X_A (§2.1b); the regression heads use
# MERT_REG (X_MERT_FULL when MERT_REG_REPR == 'full'). PCA-95 discarded the arousal-relevant
# MERT variance and kept piece identity in the Phase A run (4A-mert-pca, piece: R²a −0.25 PCA
# vs +0.45 full). The non-MERT columns are exactly those of X_C / X_D / X_F.
if MERT_REG_REPR not in ('full', 'pca'):
    raise ValueError(f"MERT_REG_REPR must be 'full' or 'pca', got {MERT_REG_REPR!r}")
MERT_REG = (np.asarray(X_MERT_FULL, dtype=float) if MERT_REG_REPR == 'full'
            else np.asarray(X_A, dtype=float))
_nA = X_A.shape[1]
X_C_REG = np.hstack([MERT_REG, np.asarray(X_C, dtype=float)[:, _nA:]])
X_D_REG = np.hstack([MERT_REG, np.asarray(X_D, dtype=float)[:, _nA:]])
X_F_REG = np.hstack([MERT_REG, np.asarray(X_F, dtype=float)[:, _nA:]])
print(f'   regression matrices (MERT_REG_REPR={MERT_REG_REPR!r}): X_C_REG {X_C_REG.shape} · '
      f'X_D_REG {X_D_REG.shape} · X_F_REG {X_F_REG.shape}')

# ── B-cache: no cell may read a retired cache (pre-Phase-B names + the Phase B v2 descriptor
#    caches retired by B2 + the B2 v3 descriptor caches retired by B3) ──────────────────────
_CACHES = {'MERT': CACHE_MERT, 'CREPE tracks': CACHE_CREPE_TRACKS, 'CREPE env (10 ms RMS)': CACHE_CREPE_ENV,
           'CREPE': CACHE_CREPE, 'Essentia': CACHE_ESSENTIA, 'Essentia npz': CACHE_ESSENTIA_NPZ,
           'RMS frames': CACHE_RMS, 'DTW': CACHE_DTW}
assert {'feat_crepe_v2.csv', 'dtw_features_v2.csv'} <= set(OLD_CACHE_NAMES), \
    '§2.5: the Phase B v2 descriptor caches must be listed in OLD_CACHE_NAMES'
assert {'feat_crepe_v3.csv', 'dtw_features_v3.csv'} <= set(OLD_CACHE_NAMES), \
    '§2.5: the Phase B2 v3 descriptor caches (retired by B3) must be listed in OLD_CACHE_NAMES'
assert (os.path.basename(CACHE_CREPE), os.path.basename(CACHE_DTW)) == ('feat_crepe_v4.csv',
                                                                         'dtw_features_v4.csv'), \
    '§2.5: the B3 descriptor caches must be feat_crepe_v4.csv / dtw_features_v4.csv'
_old = {k: v for k, v in _CACHES.items() if os.path.basename(v) in OLD_CACHE_NAMES}
assert not _old, f'§2.5: retired cache names in use: {_old}'
print(f'✅ caches: {", ".join(os.path.basename(v) for v in _CACHES.values())} (no retired name)')


## Part 3 · Tool validation — §3.1–3.4 Exploratory data analysis and tool sanity checks
**Question of this part:** is this data fit to model at N=60 — class balance, feature sanity, embedding structure?  
**Hands to the next part:** a documented go/no-go on the very matrices Parts 4–10 use.

Sanity-check the data that feeds classification **before** any model is fit.
Runs here because it needs `df_expr`, `df_crepe`, `emb_mert`, `X_A` and `Y` to all
exist. Nothing below refits a model, changes labels, or touches cleaning — it only
reads existing variables and writes `fig_eda_*.png` / `eda_*.csv` to `OUTPUT_DIR`.

- **Group 1** — label structure & rater coverage (degenerate-label + trivial-floor flags)
- **Group 2** — CREPE feature sanity against physical priors
- **Group 3** — MERT embedding structure (PCA scree)

`§3.1` class structure & trivial floors · `§3.2` VA distribution & rater coverage · `§3.3` CREPE feature sanity · `§3.3b` DTW timing sanity · `§3.4` MERT embedding structure.

> **⚑ Tempogram features (Phase C2).** `timing_local_tempo_bpm`, `timing_tempo_logsd` and `timing_pulse_clarity`
> (§3.3b) failed the manual audit (they react to articulation; the local-tempo direction contradicts the
> note rate). They stay in the models but are flagged ⚑ wherever techniques are named (`TEMPOGRAM_UNRELIABLE`,
> §0.3; §4.2, §9.1–§9.3) and §6.3 scores the timing block without them.


In [ ]:
# ── §3.1 Class structure & trivial floors ───────────────────────────
# NEEDS : mlb, OUTPUT_DIR, Y
# MAKES : COND_COLORS, COND_ORDER, emo_names, RS · eda_label_cooccurrence.csv
#          fig_eda_label_cooccurrence.png, fig_eda_label_counts.png
# FEEDS : §3.2, §3.3, §3.4
# ═══════════════════════════════════════════════════════════════════════════════
# §3.1 · EDA — Group 1a: Label structure & trivial baselines
# ═══════════════════════════════════════════════════════════════════════════════
emo_names   = list(mlb.classes_)          # 5 circumplex classes, not emotion names
COND_ORDER  = ['MEC', 'EXP', 'EXG']
COND_COLORS = {'MEC': '#999999', 'EXP': '#2a6dd9', 'EXG': '#2a9d2a'}
RS = 42

# ── Per-label positive counts ─────────────────────────────────────────────────
label_counts = Y.sum(axis=0).astype(int)
df_lab = pd.DataFrame({'class': emo_names, 'positives': label_counts,
                       'prevalence': label_counts / len(Y)})
degenerate = [e for e, n in zip(emo_names, label_counts) if n < 6]
print('── Class positive counts (N=%d) ──' % len(Y))
print(df_lab.to_string(index=False))
print(f'\n⚠️  LOOCV-degenerate risk (<6 positives): {degenerate if degenerate else "none"}')

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(range(len(emo_names)), label_counts,
       color=['#c44' if n < 6 else '#2a6dd9' for n in label_counts])
ax.axhline(6, ls='--', lw=1, color='#c44', label='degenerate threshold (6)')
ax.set_xticks(range(len(emo_names))); ax.set_xticklabels(emo_names, rotation=35, ha='right')
ax.set_ylabel('positive excerpts'); ax.set_title('Class positive counts (red = <6, LOOCV-degenerate risk)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_eda_label_counts.png'), dpi=150, bbox_inches='tight'); plt.show()

# ── class co-occurrence (diagonal = positives; off-diagonal ≈ 0 — classes are
#    mutually exclusive by construction, each excerpt gets exactly one) ───────
cooc = Y.T @ Y
df_cooc = pd.DataFrame(cooc, index=emo_names, columns=emo_names)
df_cooc.to_csv(os.path.join(OUTPUT_DIR, 'eda_label_cooccurrence.csv'))
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(df_cooc, annot=True, fmt='d', cmap='Blues', square=True,
            cbar_kws={'label': 'co-occurrences'}, ax=ax)
ax.set_title('Class co-occurrence (diagonal = positives)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_eda_label_cooccurrence.png'), dpi=150, bbox_inches='tight'); plt.show()

# ── Trivial reference floor: predict-all-absent Hamming loss ──────────────────
hl_floor = float(Y.mean())          # every cell predicted 0 → error rate = prevalence
print('\n── Trivial baseline (reference floor) ──')
print(f'   Predict-all-absent Hamming loss = {hl_floor:.4f}  (= overall label prevalence)')
print('   Per-class prevalence:')
for e, p in zip(emo_names, Y.mean(axis=0)):
    print(f'     {e:<20} {p:.3f}')
print('✅ Group 1a complete')

In [ ]:
# ── §3.2 VA distribution & rater coverage ───────────────────────────
# NEEDS : COND_COLORS, COND_ORDER, CONDITIONS, df_expr, OUTPUT_DIR, QUADRANT_DISPLAY, VA_MID
# MAKES : eda_quadrant_counts.csv, fig_eda_n_responses.png
# FEEDS : — (terminal: writes files / figures only)
# ═══════════════════════════════════════════════════════════════════════════════
# §3.2 · EDA — Group 1b: VA distribution & rater coverage
# ═══════════════════════════════════════════════════════════════════════════════
# ── Quadrant occupancy relative to VA_MID (the VA-plane scatter itself is the
#    §6.8 circumplex figure, which is its superset) ─────────────────────────
# quadrant occupancy relative to VA_MID
vv, aa = df_expr['valence_mean'], df_expr['arousal_mean']
quad = {
    f'HA_HV ({QUADRANT_DISPLAY["HA_HV"]})': int(((vv >= VA_MID) & (aa >= VA_MID)).sum()),
    f'HA_LV ({QUADRANT_DISPLAY["HA_LV"]})': int(((vv <  VA_MID) & (aa >= VA_MID)).sum()),
    f'LA_HV ({QUADRANT_DISPLAY["LA_HV"]})': int(((vv >= VA_MID) & (aa <  VA_MID)).sum()),
    f'LA_LV ({QUADRANT_DISPLAY["LA_LV"]})': int(((vv <  VA_MID) & (aa <  VA_MID)).sum()),
}
df_quad = pd.DataFrame({'quadrant': list(quad), 'n_excerpts': list(quad.values())})
df_quad.to_csv(os.path.join(OUTPUT_DIR, 'eda_quadrant_counts.csv'), index=False)
print('── Quadrant occupancy (split at VA_MID=%.1f) ──' % VA_MID)
print(df_quad.to_string(index=False))

# ── Rater coverage: n_responses per condition & per excerpt ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
by_cond = df_expr.groupby('condition')['n_responses'].sum().reindex(COND_ORDER)
axes[0].bar([CONDITIONS[c] for c in COND_ORDER], by_cond.values,
            color=[COND_COLORS[c] for c in COND_ORDER])
axes[0].set_title('Total responses per condition'); axes[0].set_ylabel('responses')
for i, val in enumerate(by_cond.values):
    axes[0].text(i, val, str(int(val)), ha='center', va='bottom', fontsize=9)
resp_order = df_expr.sort_values('n_responses')
axes[1].bar(range(len(resp_order)), resp_order['n_responses'].values,
            color=[COND_COLORS[c] for c in resp_order['condition']])
axes[1].set_title('Responses per excerpt (sorted)')
axes[1].set_xlabel('excerpt (sorted by coverage)'); axes[1].set_ylabel('responses')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_eda_n_responses.png'), dpi=150, bbox_inches='tight'); plt.show()
print('   Responses per condition:', dict(zip(COND_ORDER, by_cond.astype(int).values)))
print('✅ Group 1b complete')

In [ ]:
# ── §3.3 CREPE feature sanity ───────────────────────────────────────
# NEEDS : COND_COLORS, COND_ORDER, CREPE_CANDIDATES, df_crepe, df_expr, eids, OUTPUT_DIR
# MAKES : cond_by_eid · eda_crepe_summary.csv, fig_eda_crepe_corr.png
#          fig_eda_crepe_hist.png
# FEEDS : §3.4
# ═══════════════════════════════════════════════════════════════════════════════
# §3.3 · EDA — Group 2: CREPE feature sanity vs physical priors
# ═══════════════════════════════════════════════════════════════════════════════
crepe_cols = list(CREPE_CANDIDATES)        # the Phase B descriptors (B8–B10), §2.2
dfc         = df_crepe.reindex(eids)[crepe_cols].apply(pd.to_numeric, errors='coerce')
cond_by_eid = df_expr.set_index('excerpt_id')['condition'].reindex(eids)

# ── Reliability flags (guard NaNs & degenerate descriptors) ──────────────────
n_nan       = int(dfc.isna().any(axis=1).sum())
n_rates     = int(dfc['crepe_vibrato_rate_hz'].nunique())
n_lowvoiced = int((dfc['crepe_voiced_fraction'].fillna(0) < 0.5).sum())
print('── CREPE reliability ──')
print(f'   rows with any NaN feature         : {n_nan}/60')
print(f'   distinct vibrato-rate values      : {n_rates}/60   '
      f'{"✅ (> 20)" if n_rates > 20 else "⚠️  ≤ 20 — rate still quantised / substituted too often"}')
print(f'   voiced_fraction < 0.5 (unreliable): {n_lowvoiced}/60')
_cov = dfc['crepe_vibrato_coverage']
print(f'   vibrato coverage                  : ' + ' · '.join(
      f'P{q} {np.percentile(_cov, q):.2f}' for q in (0, 25, 50, 75, 100))
      + f'   (excerpts with coverage 0: {int((_cov == 0).sum())})')
print(f'   vibrato half-extent (cents)       : {dfc["crepe_vibrato_extent_cents"].min():.1f}–'
      f'{dfc["crepe_vibrato_extent_cents"].max():.1f} (typical violin vibrato ≈ 10–40)')

crepe_summ = dfc.describe().T[['mean', 'std', 'min', 'max']]
crepe_summ['n_zero'] = (dfc == 0).sum().values
crepe_summ['n_nan']  = dfc.isna().sum().values
crepe_summ.to_csv(os.path.join(OUTPUT_DIR, 'eda_crepe_summary.csv'))
print('\n── CREPE feature summary ──'); print(crepe_summ.round(3).to_string())

# ── Histograms (vibrato rate annotated against expected 4.5–6.5 Hz prior) ─────
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for ax, col in zip(axes.ravel(), crepe_cols):
    ax.hist(dfc[col].dropna().values, bins=15, color='#2a6dd9', alpha=0.8)
    if col == 'crepe_vibrato_rate_hz':
        ax.axvspan(4.5, 6.5, color='#2a9d2a', alpha=0.15, label='expected 4.5–6.5 Hz')
        ax.legend(fontsize=7)
    ax.set_title(col, fontsize=9); ax.tick_params(labelsize=8)
fig.suptitle('CREPE feature distributions (60 excerpts)', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_eda_crepe_hist.png'), dpi=150, bbox_inches='tight'); plt.show()

# ── Correlation heatmap (expose redundant features, e.g. f0_range vs f0_cv) ────
crepe_corr = dfc.corr()
const_cols = [c for c in crepe_cols if dfc[c].nunique(dropna=True) <= 1]
fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(crepe_corr, annot=True, fmt='.2f', vmin=-1, vmax=1, cmap='RdBu_r', square=True,
            mask=crepe_corr.isna(), cbar_kws={'label': 'Pearson r'}, ax=ax)
ax.set_title('CREPE feature correlation (redundancy check)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_eda_crepe_corr.png'), dpi=150, bbox_inches='tight'); plt.show()
if const_cols:
    print(f'⚠️  near-constant CREPE columns (corr undefined, masked): {const_cols}')
print('✅ Group 2 complete')

In [ ]:
# ── §3.3b DTW timing sanity ─────────────────────────────────────────
# NEEDS : COND_COLORS, COND_ORDER, df_dtw, df_expr, DTW_DEV, DTW_REFFREE, eids, OUTPUT_DIR,
#          TIMING_DIAGNOSTIC
# MAKES : dfd, dtw_summ · eda_dtw_summary.csv, fig_eda_dtw_hist.png
#          fig_eda_dtw_by_condition.png
# FEEDS : §3.4
# ═══════════════════════════════════════════════════════════════════════════════
# §3.3b · EDA — Group 2b: DTW timing sanity + manipulation check
# ═══════════════════════════════════════════════════════════════════════════════
# Mirrors the CREPE sanity cell above. Two questions:
#   (a) is the block clean (ranges, NaN, inf, degenerate columns)?
#   (b) is the TIMING manipulation visible at all — does reference-free timing
#       variability rise MEC ≤ EXP ≤ EXG?  (§4.2 tests this properly with Page's
#       trend; here it is only a quick group-mean look.)
dtw_cols = DTW_REFFREE + DTW_DEV
dfd      = df_dtw.reindex(eids)[dtw_cols].apply(pd.to_numeric, errors='coerce')
_cond    = df_expr.set_index('excerpt_id')['condition'].reindex(eids)

# ── Reliability flags ─────────────────────────────────────────────────────────
_arr    = dfd.to_numpy(dtype=float)
n_inf   = int(np.isinf(_arr).sum())
n_nan_1 = int(dfd[DTW_REFFREE].isna().sum().sum())
n_nan_2 = int(dfd[DTW_DEV].isna().any(axis=1).sum())
n_noons = int((dfd['timing_note_rate'].fillna(0) <= 0).sum())
print('── DTW reliability ──')
print(f'   inf cells anywhere                 : {n_inf}   (must be 0)')
print(f'   NaN cells in family (1)            : {n_nan_1}   (must be 0 — reference-free)')
print(f'   rows with NaN in family (2)        : {n_nan_2}/60  (only pieces missing a rendition)')
print(f'   note_rate == 0 (no notes found)    : {n_noons}/60')
_ojf = pd.to_numeric(df_dtw.reindex(eids)[TIMING_DIAGNOSTIC[0]], errors='coerce')
print(f'   {TIMING_DIAGNOSTIC[0]} (diagnostic) : median {_ojf.median():.3f} · max {_ojf.max():.3f}'
      '   (share of interior tempogram frames whose argmax jumps by ~2× — not a model input)')
_const = [c for c in dtw_cols if dfd[c].nunique(dropna=True) <= 1]
print(f'   near-constant columns              : {_const if _const else "none ✅"}')

dtw_summ = dfd.describe().T[['mean', 'std', 'min', 'max']]
dtw_summ['family'] = ['reference-free' if c in DTW_REFFREE else 'DTW-deviation' for c in dtw_cols]
dtw_summ['n_zero'] = (dfd == 0).sum().values
dtw_summ['n_nan']  = dfd.isna().sum().values
dtw_summ.to_csv(os.path.join(OUTPUT_DIR, 'eda_dtw_summary.csv'))
print('\n── DTW feature summary ──'); print(dtw_summ.round(4).to_string())

# ── Manipulation check: family-(1) timing variability by condition ────────────
# NOTE the expected signs: variability measures should RISE with expressivity, while
# pulse clarity (rhythmic regularity) should FALL — rubato is the same phenomenon
# seen from both sides.
VARIABILITY = [c for c in ['timing_tempo_logsd', 'timing_ioi_cv', 'timing_ioi_sd_s']
               if c in dfd.columns]
_g = dfd.groupby(_cond.values).mean().reindex(COND_ORDER)
print('\n── Manipulation check · reference-free timing by condition (group means) ──')
print(f'  {"feature":<28}{"MEC":>10}{"EXP":>10}{"EXG":>10}   monotone MEC ≤ EXP ≤ EXG?')
_n_ok = 0
for c in VARIABILITY:
    m, e, x = (float(_g.loc[k, c]) for k in COND_ORDER)
    ok = (m <= e <= x)
    _n_ok += int(ok)
    print(f'  {c:<28}{m:>10.4f}{e:>10.4f}{x:>10.4f}   {"✅ yes" if ok else "❌ no"}')
if 'timing_pulse_clarity' in _g.columns:
    m, e, x = (float(_g.loc[k, 'timing_pulse_clarity']) for k in COND_ORDER)
    print(f'  {"timing_pulse_clarity":<28}{m:>10.4f}{e:>10.4f}{x:>10.4f}   '
          f'{"✅ falls (expected)" if m >= e >= x else "— not monotone"}')
print(f'\n  {_n_ok}/{len(VARIABILITY)} timing-variability features are non-decreasing across '
      'MEC → EXP → EXG.')
print('  This is a GROUP-MEAN look only; §4.2 runs the within-piece Page trend test.')
print('  ⚠️  The DTW-deviation family is excluded from this check on purpose: it is 0 at MEC '
      'by construction,\n      so it would pass trivially.')
print('  ℹ️  The PRIMARY rubato measure is the DTW-deviation local-slope SD (dtw_local_slope_std): it\n'
      '      reads tempo modulation directly off the warp path against the same piece\'s MEC. It is\n'
      '      used for ATTRIBUTION only (df_interp, §9.1/§4.2) — 0 at MEC by construction and never a\n'
      '      model input while DTW_DEV_IN_MODELS = False. The timing_* rows above are the\n'
      '      condition-independent proxy.')

# ── Histograms ────────────────────────────────────────────────────────────────
_nc = 4
_nr = int(np.ceil(len(dtw_cols) / _nc))
fig, axes = plt.subplots(_nr, _nc, figsize=(15, 3.4 * _nr))
for ax, col in zip(np.ravel(axes), dtw_cols):
    ax.hist(dfd[col].dropna().values, bins=15,
            color='#2a6dd9' if col in DTW_REFFREE else '#d98a2a', alpha=0.85)
    ax.set_title(col, fontsize=8); ax.tick_params(labelsize=7)
for ax in np.ravel(axes)[len(dtw_cols):]:
    ax.axis('off')
fig.suptitle('DTW timing feature distributions (60 excerpts)  ·  '
             'blue = reference-free · orange = DTW-deviation vs MEC', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_eda_dtw_hist.png'), dpi=150, bbox_inches='tight'); plt.show()

# ── Family-(1) variability by condition ───────────────────────────────────────
fig, axes = plt.subplots(1, max(len(VARIABILITY), 1), figsize=(3.6 * max(len(VARIABILITY), 1), 3.8))
for ax, col in zip(np.atleast_1d(axes), VARIABILITY):
    data = [dfd.loc[(_cond == k).values, col].dropna().values for k in COND_ORDER]
    bp   = ax.boxplot(data, patch_artist=True, widths=0.6)   # no `labels=` — renamed in mpl 3.9
    ax.set_xticks(range(1, len(COND_ORDER) + 1)); ax.set_xticklabels(COND_ORDER)
    for patch, k in zip(bp['boxes'], COND_ORDER):
        patch.set_facecolor(COND_COLORS[k]); patch.set_alpha(0.75)
    ax.set_title(col, fontsize=9); ax.tick_params(labelsize=8); ax.grid(axis='y', alpha=0.2)
fig.suptitle('§3.3b  Reference-free timing variability by condition '
             '(condition-independent measurement)', y=1.03, fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_eda_dtw_by_condition.png'), dpi=150, bbox_inches='tight'); plt.show()
print('✅ Group 2b complete — eda_dtw_summary.csv, fig_eda_dtw_hist.png, fig_eda_dtw_by_condition.png')


In [ ]:
# ── §3.4 MERT embedding structure ───────────────────────────────────
# NEEDS : OUTPUT_DIR, RS, X_MERT_FULL
# MAKES : eda_pca_variance.csv, fig_eda_pca_scree.png
# FEEDS : — (terminal: writes files / figures only)
# ═══════════════════════════════════════════════════════════════════════════════
# §3.4 · EDA — Group 3: MERT embedding structure (PCA scree)
# ═══════════════════════════════════════════════════════════════════════════════
from sklearn.decomposition import PCA

# Embedding STRUCTURE is a property of the representation → use the FULL 768-dim
# space (X_MERT_FULL), not the PCA-forwarded model input X_A (§2.1b).
Xs = StandardScaler().fit_transform(np.nan_to_num(X_MERT_FULL, nan=0.0))  # defensive NaN guard

# ── PCA scree / cumulative variance → informs later dimensionality reduction ──
pca = PCA(random_state=RS).fit(Xs)
evr = pca.explained_variance_ratio_; cum = np.cumsum(evr)
n90 = int(np.argmax(cum >= 0.90)) + 1
pd.DataFrame({'component': np.arange(1, len(evr) + 1), 'evr': evr, 'cumulative': cum}
             ).to_csv(os.path.join(OUTPUT_DIR, 'eda_pca_variance.csv'), index=False)
k_scree = min(30, len(evr))
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.bar(np.arange(1, k_scree + 1), evr[:k_scree], color='#2a6dd9', alpha=0.7)
ax1.set_xlabel('component'); ax1.set_ylabel('explained variance ratio', color='#2a6dd9')
ax2 = ax1.twinx()
ax2.plot(np.arange(1, k_scree + 1), cum[:k_scree], color='#c44', marker='o', ms=3)
ax2.axhline(0.90, ls='--', color='#c44', lw=1)
ax2.set_ylabel('cumulative', color='#c44'); ax2.set_ylim(0, 1.02)
ax1.set_title(f'MERT PCA scree — {n90} components explain ≥90% variance')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_eda_pca_scree.png'), dpi=150, bbox_inches='tight'); plt.show()
print(f'── PCA ── {n90} components explain ≥90% variance (of {len(evr)} total)')

print('✅ Group 3 complete')

## Part 4 · Manipulation check — §4.1 Interpretable matrices
**Question of this part:** did the manipulation work — did the performer actually escalate the techniques  
from MEC → EXP → EXG (§4.2), and did the listeners hear a difference (§4.3)?  
**Hands to the next part:** the interpretable layer (`df_interp`, `INTERP_PRED_COLS`, `INTERP_DESC_COLS`,
`X_INTERP`), the trend table `df_dose` and the reading *techniques escalated · arousal changed · valence
did not* — the premise every later prediction and attribution rests on.

**Two versions of the interpretable layer, built once here** (moved up from the head of the former §5.1):

| Name | Columns | Used by |
|---|---|---|
| `INTERP_PRED_COLS` — **predictive** | CREPE (10) + Essentia-forward + DTW **reference-free** timing (`timing_*`) — the lattice set "CREPE + Essentia + DTW" (asserted) | every cross-validated or model-based analysis: §6.2, §6.8, §7.1, §7.2, §8.1, §9.2, §9.3, §10.1–§10.3 |
| `INTERP_DESC_COLS` — **descriptive** | `INTERP_PRED_COLS` + the DTW-deviation features (`DTW_FORWARD_DEV`) | only the descriptive §4.2 (dose-response) and §9.1 (condition deltas), rows flagged ⚑ |

The DTW-deviation features are 0 / 1 at MEC by construction (MEC is their alignment reference, §2.4), so
they encode the condition and never enter a model; this split replaces the former `DTW_DEV_IN_INTERP`
switch. **⚑ flags:** the DTW-deviation rows and the three tempogram features that failed the manual audit
(`TEMPOGRAM_UNRELIABLE`, §0.3 — local tempo, tempo log2-SD, pulse clarity). The tempogram features stay
in the models, are flagged wherever techniques are named (§4.2, §9.1–§9.3), are never quoted alone as a
"top feature", and §6.3 scores the timing block without them.


In [ ]:
# ── §4.1 Interpretable matrices — one PREDICTIVE and one DESCRIPTIVE version ──
# NEEDS : CREPE_CANDIDATES, CREPE_FORWARD, CREPE_LABELS, df_crepe, df_dtw_attr, df_essentia,
#          DTW_FEAT_LABELS, DTW_FORWARD_DEV, DTW_MODEL_COLS, eids, ESSENTIA_FEAT_LABELS, ESSENTIA_FORWARD,
#          TEMPOGRAM_UNRELIABLE
# MAKES : CREPE_FEAT_LABELS, df_interp, DTW_INTERP_LABELS, feat_keys, feat_short, flag_label,
#          INTERP_DESC_COLS, INTERP_FEAT_LABELS, INTERP_FLAG_REASON, INTERP_PRED_COLS, pred_short,
#          top_feature_note, X_INTERP
# FEEDS : §4.2 and §9.1 (INTERP_DESC_COLS — descriptive) · §6.2, §6.8, §7.1, §7.2, §8.1, §9.2, §9.3,
#          §10.1–§10.3 (INTERP_PRED_COLS / X_INTERP — every cross-validated or model-based analysis)
# ═══════════════════════════════════════════════════════════════════════════════
# §4.1 · The interpretable layer, built ONCE (moved up from the head of the former §5.1)
# ═══════════════════════════════════════════════════════════════════════════════
# Interpretable = the named, measured techniques: CREPE pitch (the 10 Phase B descriptors) + the forwarded
# Essentia dynamics / timbre / tonal block + the forwarded timing / DTW block (§2.4b). Two versions:
#   · INTERP_PRED_COLS — PREDICTIVE: CREPE + Essentia + DTW reference-free timing. Every cross-validated
#     or model-based analysis uses it. It must be the lattice set "CREPE + Essentia + DTW" (asserted).
#   · INTERP_DESC_COLS — DESCRIPTIVE: INTERP_PRED_COLS + the DTW-deviation features (DTW_FORWARD_DEV),
#     which are 0 / 1 at MEC by construction (§2.4). Only the descriptive §4.2 (dose-response) and §9.1
#     (condition deltas) read it, and they flag those rows ⚑.
# This replaces the former DTW_DEV_IN_INTERP switch (§0.3).

# The Phase B CREPE descriptors (B8–B10) and their labels come from §2.2 — one source of truth. The
# diagnostic column (crepe_vibrato_n_notes) is not a technique and stays out.
CREPE_FEAT_LABELS  = {c: CREPE_LABELS[c] for c in CREPE_CANDIDATES}
DTW_INTERP_LABELS  = {c: DTW_FEAT_LABELS[c] for c in df_dtw_attr.columns}
INTERP_FEAT_LABELS = {**CREPE_FEAT_LABELS, **ESSENTIA_FEAT_LABELS, **DTW_INTERP_LABELS}   # all DESC columns
df_interp = pd.concat([df_crepe[CREPE_CANDIDATES], df_essentia[ESSENTIA_FORWARD], df_dtw_attr], axis=1)

INTERP_DESC_COLS = [c for c in INTERP_FEAT_LABELS if c in df_interp.columns]
INTERP_PRED_COLS = [c for c in INTERP_DESC_COLS if c not in DTW_FORWARD_DEV]
assert list(df_interp.columns) == INTERP_DESC_COLS, '§4.1: df_interp columns out of sync with the labels'
assert not any('mert' in c.lower() for c in INTERP_DESC_COLS), '§4.1: an embedding dimension leaked in'
# The predictive layer IS the lattice set "CREPE + Essentia + DTW" (§6.3: CREPE_FORWARD + ESSENTIA_FORWARD +
# DTW_MODEL_COLS) — same columns; §6.3 also asserts the values.
_lat_cols = list(CREPE_FORWARD) + list(ESSENTIA_FORWARD) + list(DTW_MODEL_COLS)
assert sorted(INTERP_PRED_COLS) == sorted(_lat_cols) and len(INTERP_PRED_COLS) == len(_lat_cols), \
    (f'§4.1: INTERP_PRED_COLS ({len(INTERP_PRED_COLS)}) ≠ the lattice set CREPE + Essentia + DTW '
     f'({len(_lat_cols)}): only in PRED {sorted(set(INTERP_PRED_COLS) - set(_lat_cols))}, '
     f'only in the lattice {sorted(set(_lat_cols) - set(INTERP_PRED_COLS))}')

# Descriptive matrices use these keys / labels (§4.2, §9.1); the predictive analyses use pred_short.
feat_keys  = list(INTERP_DESC_COLS)
feat_short = [INTERP_FEAT_LABELS[c] for c in feat_keys]
pred_short = [INTERP_FEAT_LABELS[c] for c in INTERP_PRED_COLS]

# ── ⚑ flags: DTW-deviation (descriptive only) and the tempogram features that failed the manual audit ──
INTERP_FLAG_REASON = {}
for _c in INTERP_DESC_COLS:
    if _c in DTW_FORWARD_DEV:
        INTERP_FLAG_REASON[_c] = 'DTW-deviation — 0/1 at MEC by construction (descriptive only)'
    elif _c in TEMPOGRAM_UNRELIABLE:
        INTERP_FLAG_REASON[_c] = 'tempogram — failed the manual audit (kept in the models, flagged)'
_tg_missing = [c for c in TEMPOGRAM_UNRELIABLE if c not in INTERP_PRED_COLS]
if _tg_missing:
    print(f'ℹ️  TEMPOGRAM_UNRELIABLE columns not in the interpretable layer (not forwarded): {_tg_missing}')


def flag_label(col, label=None):
    """Display label of an interpretable column, with ' ⚑' if it is flagged (DTW-deviation / tempogram)."""
    lab = INTERP_FEAT_LABELS.get(col, col) if label is None else label
    return f'{lab} ⚑' if col in INTERP_FLAG_REASON else lab


def top_feature_note(ordered_cols):
    """'<top>' — or, if the top-ranked column is flagged ⚑, '<top> ⚑ (flagged) · top non-flagged: <next>'.
    A flagged feature is never reported as THE top feature on its own (Phase C2)."""
    cols = list(ordered_cols)
    if not cols:
        return 'n/a'
    if cols[0] not in INTERP_FLAG_REASON:
        return INTERP_FEAT_LABELS.get(cols[0], cols[0])
    nxt = next((c for c in cols if c not in INTERP_FLAG_REASON), None)
    return (f'{flag_label(cols[0])} (flagged: {INTERP_FLAG_REASON[cols[0]].split(" — ")[0]}) · top non-flagged: '
            + (INTERP_FEAT_LABELS.get(nxt, nxt) if nxt else 'none'))


# Row-aligned PREDICTIVE matrix (SHAP §9.3 reads it; every other model-based cell builds the same
# columns through its own row-alignment helper and asserts the column list).
X_INTERP = df_interp.reindex(eids)[INTERP_PRED_COLS].apply(pd.to_numeric, errors='coerce')
X_INTERP = X_INTERP.fillna(X_INTERP.mean(numeric_only=True)).values.astype(float)

print(f'Interpretable (PREDICTIVE) — INTERP_PRED_COLS: {len(INTERP_PRED_COLS)} features = '
      f'{len(CREPE_FEAT_LABELS)} CREPE + {len(ESSENTIA_FORWARD)} Essentia-forward + '
      f'{len(INTERP_PRED_COLS) - len(CREPE_FEAT_LABELS) - len(ESSENTIA_FORWARD)} DTW reference-free timing '
      '(= the lattice set "CREPE + Essentia + DTW" ✅)')
print(f'Interpretable (DESCRIPTIVE) — INTERP_DESC_COLS: {len(INTERP_DESC_COLS)} features = the predictive '
      f'{len(INTERP_PRED_COLS)} + {len(INTERP_DESC_COLS) - len(INTERP_PRED_COLS)} DTW-deviation '
      f'({", ".join(INTERP_FEAT_LABELS[c] for c in INTERP_DESC_COLS if c not in INTERP_PRED_COLS) or "none"})')
print('⚑ flagged columns (shown with ⚑ in §4.2 / §9.1 / §9.2 / §9.3; never quoted alone as a top feature):')
for _c, _why in INTERP_FLAG_REASON.items():
    print(f'   ⚑ {INTERP_FEAT_LABELS[_c]:<30} {_why}')
print(f'X_INTERP (predictive, row-aligned to eids): {X_INTERP.shape}')


## Part 4 · Manipulation check — §4.2 Performers escalated — Page's trend test on the techniques

Exploits the graded, **within-piece** design: the three conditions are not three unrelated categories,
they are an **ordered dose** — `MEC < EXP < EXG`.

The question: does each technique change **monotonically** as expressivity increases? Tested with
**Page's trend test** (repeated-measures, ordered alternative), the right instrument for an *a-priori
ordered* hypothesis: it has more power than Friedman precisely because it does not spend degrees of
freedom on orderings we never predicted. The techniques are the **descriptive** interpretable layer
(`INTERP_DESC_COLS`, §4.1). This replaces the former §5.1b (technique half); its rating half is §4.3b.

**Multiple testing (Phase C2).** One Benjamini–Hochberg family = every technique test here **plus** the
two rating tests (valence, arousal) printed in §4.3b; the `q` column carries it and the significance
stars come from `q`, not from `p`. **⚑** marks the DTW-deviation rows (0/1 at MEC by construction — their
rise is partly built in) and the three tempogram features that failed the manual audit.


In [ ]:
# ── §4.2 Performers escalated — Page's trend test on the techniques ──
# NEEDS : bh_fdr (§0.3), df_all (piece, condition, excerpt_id, valence_mean, arousal_mean), df_interp,
#          DTW_FORWARD_DEV, INTERP_DESC_COLS, INTERP_FEAT_LABELS, INTERP_FLAG_REASON, OUTPUT_DIR,
#          TEMPOGRAM_UNRELIABLE
# MAKES : df_dose (var, label, kind, direction, z, p, q, stars_q, pct_monotone, mean_MEC_EXG,
#          builtin_at_MEC, tempogram_unreliable, flag), DOSE_FAMILY_N, EMO_VARS, ORDER, page_trend, stars,
#          TECH_COLS, VAR_LABEL · dose_response.csv
# FEEDS : §4.3b (the two rating rows of the SAME BH family + the figure), the Part 4 conclusion, §11.1
# ═══════════════════════════════════════════════════════════════════════════════
# §4.2 · Dose-response — monotonic trend across ordered expressivity (MEC < EXP < EXG)
# ═══════════════════════════════════════════════════════════════════════════════
# DESCRIPTIVE: reads the interpretable DESCRIPTIVE layer (INTERP_DESC_COLS, §4.1) — the DTW-deviation rows
# are kept and flagged ⚑ (0/1 at MEC by construction), as are the tempogram features that failed the
# manual audit (TEMPOGRAM_UNRELIABLE). One BH-FDR family = every technique test HERE + the two rating
# tests (valence, arousal) printed in §4.3b; significance stars come from q, not p.
from scipy.stats import norm as _norm, rankdata as _rankdata

ORDER    = ['MEC', 'EXP', 'EXG']

# ── 1. Per-(piece, condition) frame: techniques + audience ratings ───────────
_meta = df_all.set_index('excerpt_id')[['piece', 'condition', 'valence_mean', 'arousal_mean']]
TECH_COLS = [c for c in INTERP_DESC_COLS if c in df_interp.columns]
D = _meta.join(df_interp[TECH_COLS], how='inner')

# ── 2. Keep ONLY pieces present in all three conditions (complete blocks) ────
_cov   = D.groupby('piece')['condition'].apply(lambda s: set(ORDER).issubset(set(s)))
PIECES = sorted(_cov[_cov].index)
print(f'Complete pieces (present in all of {ORDER}): {len(PIECES)} of {D["piece"].nunique()}')
D = D[D['piece'].isin(PIECES)]

# ── 3. Variables under test: techniques + the two emotion dimensions (one family) ──
EMO_VARS  = ['valence_mean', 'arousal_mean']
VARS      = TECH_COLS + EMO_VARS
VAR_LABEL = dict(INTERP_FEAT_LABELS)
VAR_LABEL.update({'valence_mean': 'Valence (mean rating)',
                  'arousal_mean': 'Arousal (mean rating)'})

_DTW_TIMING  = [c for c in TECH_COLS if str(c).startswith(('dtw_', 'timing_'))]   # B13 rename
_DTW_BUILTIN = [c for c in _DTW_TIMING if c in DTW_FORWARD_DEV]
_DTW_HONEST  = [c for c in _DTW_TIMING if c not in DTW_FORWARD_DEV]
print(f'Techniques under test: {len(TECH_COLS)} (INTERP_DESC_COLS) · timing: {len(_DTW_TIMING)} '
      f'({len(_DTW_HONEST)} reference-free + {len(_DTW_BUILTIN)} DTW-deviation)')
print('⚑ = flagged: ' + '; '.join(f'{VAR_LABEL.get(c, c)} ({INTERP_FLAG_REASON[c].split(" — ")[0]})'
                                  for c in TECH_COLS if c in INTERP_FLAG_REASON))


# ── 4. Page's L trend test (Page 1963 normal approximation) ─────────────────
def page_trend(M):
    """M = (N pieces x k conditions) in the PREDICTED order MEC, EXP, EXG.
    Ranks within each block ascending (ties average-ranked via rankdata), then
        L = sum_j  j * R_j
        z = (12L - 3Nk(k+1)^2) / (k(k+1)*sqrt(N(k-1)))
    z > 0 => the variable INCREASES along the predicted order; z < 0 => decreases."""
    A = np.asarray(M, dtype=float)
    N, k = A.shape
    Rj = np.zeros(k)
    for row in A:
        Rj += _rankdata(row)                       # average ties
    L = float(np.sum(np.arange(1, k + 1) * Rj))
    z = ((12.0 * L - 3.0 * N * k * (k + 1) ** 2)
         / (k * (k + 1) * np.sqrt(N * (k - 1))))
    return L, float(z), float(2 * _norm.sf(abs(z))), N, A


def stars(p):
    return '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else 'n.s.'


rows = []
for v in VARS:
    M = (D.pivot_table(index='piece', columns='condition', values=v, aggfunc='first')
           .reindex(columns=ORDER).dropna())
    if len(M) < 3:
        print(f'⚠️  {VAR_LABEL.get(v, v)}: only {len(M)} complete pieces — skipped')
        continue
    L, z, p, N, A = page_trend(M)
    mono = (np.mean((A[:, 0] < A[:, 1]) & (A[:, 1] < A[:, 2])) if z > 0
            else np.mean((A[:, 0] > A[:, 1]) & (A[:, 1] > A[:, 2])))
    rows.append({'var'          : v,
                 'label'        : VAR_LABEL.get(v, v),
                 'kind'         : 'emotion' if v in EMO_VARS else 'technique',
                 'direction'    : 'increases' if z > 0 else 'decreases',
                 'L'            : L,
                 'z'            : z,
                 'p'            : p,
                 'pct_monotone' : float(mono),
                 'mean_MEC_EXG' : float(np.mean(A[:, 2] - A[:, 0])),
                 'builtin_at_MEC': bool(v in DTW_FORWARD_DEV),
                 'tempogram_unreliable': bool(v in TEMPOGRAM_UNRELIABLE),
                 'n_pieces'     : int(N)})

df_dose = pd.DataFrame(rows)
# ── 5. ONE BH-FDR family: every technique test + the two rating tests (Phase C2, C2 obligation 2) ──
df_dose['q'] = bh_fdr(df_dose['p'].to_numpy(dtype=float))
df_dose['stars_q'] = [stars(q) if np.isfinite(q) else 'n/a' for q in df_dose['q']]
df_dose['flag'] = ['⚑' if v in INTERP_FLAG_REASON else '' for v in df_dose['var']]
df_dose = df_dose.sort_values('p').reset_index(drop=True)
DOSE_FAMILY_N = int(np.isfinite(df_dose['p']).sum())


def _dose_block(kind, title):
    sub = df_dose[df_dose['kind'] == kind]
    print('\n' + '─' * 112)
    print(title)
    print('─' * 112)
    print(f'  {"variable":<36}{"direction":<11}{"z":>8}{"p":>10}{"q":>10}  {"sig(q)":<7}'
          f'{"%monotone":>11}{"mean EXG−MEC":>14}')
    for t in sub.itertuples():
        _mark = ('  ⚑ 0/1 at MEC by construction' if t.builtin_at_MEC else
                 '  ⚑ tempogram — failed the manual audit' if t.tempogram_unreliable else '')
        print(f'  {t.label:<36}{t.direction:<11}'
              f'{t.z:>+8.2f}{t.p:>10.4f}{t.q:>10.4f}  {t.stars_q:<7}'
              f'{t.pct_monotone*100:>10.0f}%{t.mean_MEC_EXG:>+14.3f}{_mark}')


print('\n' + '═' * 112)
print(f"§4.2 · PAGE'S TREND TEST — ordered alternative MEC < EXP < EXG "
      f'(N={int(df_dose["n_pieces"].max())} complete pieces, k=3) · BH-FDR over ONE family of '
      f'{DOSE_FAMILY_N} tests ({int((df_dose.kind == "technique").sum())} techniques here + the 2 ratings of §4.3b)')
print('═' * 112)
_dose_block('technique', 'A · TECHNIQUES — what the performer escalated (stars from q)')

df_dose.to_csv(os.path.join(OUTPUT_DIR, 'dose_response.csv'), index=False)

_sig_tech = df_dose[(df_dose.kind == 'technique') & (df_dose.q < 0.05)]
_sig_clean = _sig_tech[_sig_tech['flag'] == '']
print('\n── Reading (techniques) ──')
print(f'  {len(_sig_tech)} of {int((df_dose.kind == "technique").sum())} techniques show a monotonic trend with '
      f'q < .05 ({len(_sig_clean)} of them not flagged ⚑; raw p < .05: '
      f'{int(((df_dose.kind == "technique") & (df_dose.p < 0.05)).sum())}).')
if len(_sig_clean):
    print('  Performers escalated (not flagged): ' + ', '.join(
        f'{t.label} ({t.direction}, z={t.z:+.2f}, q={t.q:.3g})' for t in _sig_clean.itertuples()))
print('\n── Caveat ──')
print("  Page's test uses only the ORDINAL order of the three conditions — it does not assume")
print('  equal spacing between MEC, EXP and EXG. Read mean_MEC_EXG for the raw-unit magnitude.')
print('  Ties within a piece are average-ranked.')
if _DTW_BUILTIN:
    print('  ⚑ DTW-DEVIATION rows are 0 (deviation) / 1 (ratio) at MEC BY CONSTRUCTION — MEC is')
    print('    the alignment reference, so those rows cannot fail to start at the floor. Their z')
    print('    quantifies HOW MUCH EXP/EXG drift from the mechanical take, not WHETHER timing')
    print('    escalates. The reference-free timing features answer that question honestly.')
print('  ⚑ TEMPOGRAM rows (local tempo, tempo log2-SD, pulse clarity) failed the manual audit: they react to')
print('    articulation, and the local-tempo direction contradicts the note rate — never cite them alone.')
print('\n✅ §4.2 — dose_response.csv (techniques + ratings, one BH family; the figure follows in §4.3b)')


## Part 4 · Manipulation check — §4.3 Listeners heard it — rating statistics · trend · class binding

Did the audience register the manipulation? Three complementary views of the **ratings** (no model, no
acoustic feature): **§4.3a** Friedman / Wilcoxon on the piece-level condition means (moved here from the
former §1.4), **§4.3b** Page's trend test on valence and arousal (the rating half of the §4.2 BH family),
**§4.3c** the tag-based lift (class–condition binding, the former §1.5) and the ground-truth class share
per condition.


## Part 4 · Manipulation check — §4.3a Rating statistics (Friedman · Wilcoxon)
Ground-truth emotional structure in the **human ratings**, before any modeling.
Repeated-measures design: the same 20 pieces are played under all three conditions,
so condition effects on valence/arousal are tested with the **Friedman** test on
piece-level condition means, followed by **Wilcoxon** signed-rank post-hoc
(Bonferroni-corrected).

> **Note:** emotion-label analyses here (and the Lift section) use the **raw
> participant emotion tags**, *not* the VA-derived labels — the derived labels come
> from VA position (`Neutral` is assigned by proximity to the VA centre, within
> `NEUTRAL_RADIUS`, independent of condition), so using them to test emotion–condition
> associations would be circular since VA itself varies by condition.
>
> Raw tags are reported **as-is** (8-tag table, saved to `participant_raw_tag_counts.csv`)
> and then bucketed into the 5 circumplex classes with `TAG_TO_QUADRANT` so the thesis
> speaks one vocabulary. The bucketing is a fixed conceptual grouping (Tension is always
> `HA_LV`) and is independent of each excerpt's measured VA — it adds no circularity.


In [ ]:
# ── §4.3a Rating statistics (Friedman · Wilcoxon) ────────────────────
# NEEDS : CONDITIONS, df_agg, df_long, EMOTION_LABELS, EXCLUDED_PARTICIPANTS, N_PARTICIPANTS_RAW,
#          OUTPUT_DIR, RAW_EMOTION_TAGS, TAG_TO_QUADRANT
# MAKES : COND_ORDER, FRIEDMAN_RES (measure → chi2 / p / W / n), kendall_w, PARTICIPANT_DATA,
#          rank_biserial, stars · fig_va_by_condition.png, participant_posthoc_tests.csv
#          participant_raw_tag_counts.csv, participant_stats_by_condition.csv
# FEEDS : §4.3c (PARTICIPANT_DATA, COND_ORDER), the Part 4 conclusion, §11.1 (FRIEDMAN_RES)
# ═══════════════════════════════════════════════════════════════════════════════
# §4.3a — Listeners heard it: rating statistics (participants only, no classifiers; moved here from the
# former §1.4 in Phase C2, so the manipulation check reads performer → listener in one place)
# ═══════════════════════════════════════════════════════════════════════════════
from scipy.stats import friedmanchisquare, wilcoxon, rankdata

# ── Data source: df_long, already screened in §1.1 (screening v3, §0.3) ───────
PARTICIPANT_DATA = df_long
print(f'ℹ️  Participant data = df_long, screened in §1.1: {len(EXCLUDED_PARTICIPANTS)} of '
      f'{N_PARTICIPANTS_RAW} excluded ({", ".join(EXCLUDED_PARTICIPANTS)}) → '
      f'{PARTICIPANT_DATA["participant_id"].nunique()} participants, {len(PARTICIPANT_DATA)} ratings.')

COND_ORDER = ['MEC', 'EXP', 'EXG']

# ── 1a. Mean & SD of valence / arousal per condition (response-level) ─────────
rows = []
for c in COND_ORDER:
    sub = PARTICIPANT_DATA[PARTICIPANT_DATA['condition'] == c]
    rows.append({
        'condition'     : c,
        'condition_name': CONDITIONS[c],
        'n_responses'   : len(sub),
        'valence_mean'  : sub['valence'].mean(),
        'valence_sd'    : sub['valence'].std(),
        'arousal_mean'  : sub['arousal'].mean(),
        'arousal_sd'    : sub['arousal'].std(),
    })
df_cond_stats = pd.DataFrame(rows)
print('\n── Valence / Arousal by condition (participant responses) ──')
print(df_cond_stats.to_string(index=False))

# ── 1b. Tag frequency per condition (RAW participant tags) ───────────────────
# The genuine 8-tag counts are computed and SAVED FIRST (never destroyed), then
# bucketed into the 5 circumplex classes via TAG_TO_QUADRANT so the whole thesis
# speaks ONE vocabulary. Bucketing does NOT reintroduce circularity: tag→quadrant
# is a fixed conceptual grouping (Tension is always HA_LV), independent of each
# excerpt's measured VA position.
raw_freq = {c: Counter() for c in COND_ORDER}
cls_freq = {c: Counter() for c in COND_ORDER}
ntag     = {c: 0 for c in COND_ORDER}
ncls     = {c: 0 for c in COND_ORDER}
for _, r in PARTICIPANT_DATA.iterrows():
    tags = r['emotion_tags'] if isinstance(r['emotion_tags'], list) else []
    for t in tags:
        raw_freq[r['condition']][t] += 1
        ntag[r['condition']]        += 1
    # set() ⇒ two tags from the same quadrant (e.g. Power + Joyful) count once
    for q in {TAG_TO_QUADRANT[t] for t in tags if t in TAG_TO_QUADRANT}:
        cls_freq[r['condition']][q] += 1
        ncls[r['condition']]        += 1

df_raw_tag_freq = pd.DataFrame({c: {e: raw_freq[c].get(e, 0) for e in RAW_EMOTION_TAGS}
                                for c in COND_ORDER}).reindex(RAW_EMOTION_TAGS)
df_raw_tag_freq.to_csv(os.path.join(OUTPUT_DIR, 'participant_raw_tag_counts.csv'))
print('\n── RAW emotion-tag counts per condition (original 8-tag vocabulary, preserved) ──')
print(df_raw_tag_freq.to_string())

df_emo_freq = pd.DataFrame({c: {e: cls_freq[c].get(e, 0) for e in EMOTION_LABELS}
                            for c in COND_ORDER}).reindex(EMOTION_LABELS)
print('\n── Circumplex-class counts per condition (raw tags bucketed via TAG_TO_QUADRANT) ──')
print(df_emo_freq.to_string())
denom = pd.Series({c: ncls[c] for c in COND_ORDER})
df_emo_freq_pct = df_emo_freq.divide(denom, axis=1) * 100
print('\n── Class share per condition (% of all class votes cast in that condition) ──')
print(df_emo_freq_pct.round(1).to_string())

# ── 1c. Friedman (repeated-measures) on PIECE-LEVEL condition means ───────────
# Same 20 pieces appear in all three conditions → within-subject (piece) design.
piece_val = df_agg.pivot(index='piece', columns='condition', values='valence_mean')[COND_ORDER].dropna()
piece_aro = df_agg.pivot(index='piece', columns='condition', values='arousal_mean')[COND_ORDER].dropna()

def kendall_w(chi2, n, k):
    return chi2 / (n * (k - 1))                      # Friedman effect size

def rank_biserial(x, y):
    d = np.asarray(x, float) - np.asarray(y, float)
    d = d[d != 0]
    if len(d) == 0: return 0.0
    r = rankdata(np.abs(d))
    Rpos, Rneg = r[d > 0].sum(), r[d < 0].sum()
    return (Rpos - Rneg) / (Rpos + Rneg)             # matched-pairs rank-biserial

def stars(p):
    return '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else 'n.s.'

posthoc_rows, FRIEDMAN_RES = [], {}
for name, mat in [('Valence', piece_val), ('Arousal', piece_aro)]:
    n, k = mat.shape
    chi2, p = friedmanchisquare(*[mat[c].values for c in COND_ORDER])
    W = kendall_w(chi2, n, k)
    FRIEDMAN_RES[name] = {'chi2': float(chi2), 'p': float(p), 'W': float(W), 'n': int(n)}
    print(f'\n── Friedman — {name} (piece-level, N={n} pieces, k={k}) ──')
    print(f'   chi2={chi2:.3f}  p={p:.4f} {stars(p)}   Kendall W={W:.3f}')
    if p < .05:
        print('   Post-hoc Wilcoxon signed-rank (Bonferroni x3):')
        for a, b in [('MEC', 'EXP'), ('EXP', 'EXG'), ('MEC', 'EXG')]:
            stat, pw = wilcoxon(mat[a].values, mat[b].values)
            p_bonf = min(pw * 3, 1.0)
            rb = rank_biserial(mat[a].values, mat[b].values)
            print(f'     {a} vs {b}: W={stat:.1f}  p={pw:.4f}  p_bonf={p_bonf:.4f} '
                  f'{stars(p_bonf)}  rank-biserial={rb:+.3f}')
            posthoc_rows.append({'measure': name, 'pair': f'{a}-{b}', 'wilcoxon_W': stat,
                                 'p_raw': pw, 'p_bonferroni': p_bonf, 'rank_biserial': rb})
    else:
        print('   Friedman not significant → post-hoc skipped.')

# ── 1d. Grouped bar chart: mean valence & arousal by condition (±SD) ──────────
fig, ax = plt.subplots(figsize=(7, 5))
xpos, w = np.arange(len(COND_ORDER)), 0.38
ax.bar(xpos - w/2, df_cond_stats['valence_mean'], w, yerr=df_cond_stats['valence_sd'],
       capsize=4, label='Valence', color='#2a6dd9', alpha=0.85)
ax.bar(xpos + w/2, df_cond_stats['arousal_mean'], w, yerr=df_cond_stats['arousal_sd'],
       capsize=4, label='Arousal', color='#d9762a', alpha=0.85)
ax.set_xticks(xpos); ax.set_xticklabels([CONDITIONS[c] for c in COND_ORDER])
ax.set_ylabel('Rating (0–6)'); ax.set_ylim(0, 6)
ax.set_title('Mean Valence & Arousal by Condition (±SD)')
ax.legend(); ax.grid(axis='y', alpha=0.15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_va_by_condition.png'), dpi=150, bbox_inches='tight'); plt.show()

# ── Save tables ───────────────────────────────────────────────────────────────
df_cond_stats.to_csv(os.path.join(OUTPUT_DIR, 'participant_stats_by_condition.csv'), index=False)
if posthoc_rows:
    pd.DataFrame(posthoc_rows).to_csv(os.path.join(OUTPUT_DIR, 'participant_posthoc_tests.csv'), index=False)
print('\n✅ Saved: participant_stats_by_condition.csv, participant_raw_tag_counts.csv, '
      'fig_va_by_condition.png'
      + (', participant_posthoc_tests.csv' if posthoc_rows else ''))

## Part 4 · Manipulation check — §4.3b Page's trend test on the ratings (valence, arousal)

The two rating rows of the §4.2 test — same Page statistic, same BH family (q from §4.2). The figure
shows the signed trend statistic z of every technique (§4.2) and both ratings side by side: what the
performer escalated against what the audience escalated in response.


In [ ]:
# ── §4.3b Listeners — Page's trend test on valence and arousal ───────
# NEEDS : df_dose, DOSE_FAMILY_N, EMO_VARS (§4.2), OUTPUT_DIR
# MAKES : DOSE_EMO (var → row of df_dose) · fig_dose_response.png
# FEEDS : the Part 4 conclusion, §11.1
# ═══════════════════════════════════════════════════════════════════════════════
# §4.3b · Did the AUDIENCE escalate? — the rating rows of the §4.2 Page family (same BH family)
# ═══════════════════════════════════════════════════════════════════════════════
_emo = df_dose[df_dose.kind == 'emotion']
print('═' * 100)
print(f"§4.3b · PAGE'S TREND TEST on the ratings — MEC < EXP < EXG (q from the ONE §4.2 family of "
      f'{DOSE_FAMILY_N} tests)')
print('═' * 100)
print(f'  {"rating":<26}{"direction":<11}{"z":>8}{"p":>10}{"q":>10}  {"sig(q)":<7}{"%monotone":>11}'
      f'{"mean EXG−MEC":>14}')
for t in _emo.itertuples():
    print(f'  {t.label:<26}{t.direction:<11}{t.z:>+8.2f}{t.p:>10.4f}{t.q:>10.4f}  {t.stars_q:<7}'
          f'{t.pct_monotone*100:>10.0f}%{t.mean_MEC_EXG:>+14.3f}')
DOSE_EMO = {t.var: t for t in _emo.itertuples()}
for v in EMO_VARS:
    if v in DOSE_EMO:
        r = DOSE_EMO[v]
        verdict = ('rises' if r.direction == 'increases' else 'falls') + \
                  (' monotonically' if r.q < 0.05 else ' — but n.s. after BH')
        print(f'  → {r.label}: {verdict} with expressivity (z={r.z:+.2f}, p={r.p:.4f}, q={r.q:.4f}, '
              f'EXG−MEC={r.mean_MEC_EXG:+.3f})')

# ── Figure: signed z per variable (techniques + ratings), coloured by kind, faded when q ≥ .05 ──
from matplotlib.colors import to_rgba
from matplotlib.patches import Patch
dsort = df_dose.sort_values('z')
kcol  = {'technique': '#2a6dd9', 'emotion': '#c0392b'}
bar_colors = [to_rgba(kcol[k], 1.0 if q < 0.05 else 0.30) for k, q in zip(dsort['kind'], dsort['q'])]
fig, ax = plt.subplots(figsize=(9, max(4.5, 0.34 * len(dsort))))
ax.barh(range(len(dsort)), dsort['z'].values, color=bar_colors, edgecolor='#333', linewidth=0.5)
ax.axvline(0, color='#333', lw=1)
ax.set_yticks(range(len(dsort)))
ax.set_yticklabels([f'{l} ⚑' if f else l for l, f in zip(dsort['label'], dsort['flag'])], fontsize=8)
ax.set_xlabel("Page's L trend statistic (z)    ←  decreases with expressivity  |  increases  →")
ax.set_title(f"§4.2 / §4.3b  Dose-response across ordered expressivity (MEC < EXP < EXG)\n"
             f"Page's trend test, N={int(df_dose['n_pieces'].max())} complete pieces · solid = BH q < .05 "
             f'(one family of {DOSE_FAMILY_N}), faded = n.s. · ⚑ = flagged', fontsize=11)
ax.legend(handles=[Patch(facecolor=kcol['technique'], label='technique (performer, §4.2)'),
                   Patch(facecolor=kcol['emotion'],   label='rating (audience, §4.3b)')],
          fontsize=9, loc='lower right')
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_dose_response.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ §4.3b — fig_dose_response.png (dose_response.csv was written in §4.2)')


## Part 4 · Manipulation check — §4.3c Lift analysis · class–condition binding · class share per condition
Quantifies how strongly each emotion is bound to each playing condition:

$$\text{lift}(e, c) = \frac{P(e \mid c)}{P(e)}$$

where each probability is the **fraction of excerpts carrying emotion label $e$**.
An excerpt "carries" $e$ if at least a threshold fraction of its raters tagged $e$
(primary threshold **0.30**, with a sensitivity sweep over 0.20 / 0.30 / 0.40 and a
threshold-free mean-vote-share robustness check).

- `lift = 1` → emotion independent of condition
- `lift > 1` → emotion **concentrated** in that condition
- `lift < 1` → emotion **suppressed**

> Uses **raw participant tags** (see the note in §4.3a) — VA-derived labels are avoided to
> prevent circularity (the derived labels, including `Neutral` assigned by proximity
> to the VA centre within `NEUTRAL_RADIUS`, come from VA position, which itself varies
> by condition).
>
> Tags are bucketed into the 5 circumplex classes via `TAG_TO_QUADRANT`; the raw 8-tag
> lift is computed and saved first (`lift_matrix_raw_tags.csv`).


In [ ]:
# ── §4.3c Lift analysis · class–condition binding ────────────────────
# NEEDS : COND_ORDER, CONDITIONS, df_expr, EMOTION_LABELS, OUTPUT_DIR, PARTICIPANT_DATA, QUADRANT_DISPLAY
#          RAW_EMOTION_TAGS, TAG_TO_QUADRANT
# MAKES : _as_classes, _as_raw, _cond_of, build_incidence, CLASS_SHARE_BY_COND, lift_from_incidence,
#          PRIMARY_THRESH, SENSITIVITY_THRESHOLDS · fig_lift_heatmap.png, lift_matrix.csv,
#          lift_matrix_meanshare.csv, lift_matrix_raw_tags.csv, lift_threshold_sensitivity.csv,
#          class_share_by_condition.csv
# FEEDS : the Part 4 conclusion (descriptive)
# ═══════════════════════════════════════════════════════════════════════════════
# §4.3c — Lift Analysis (class–condition binding, participants only)
# ═══════════════════════════════════════════════════════════════════════════════
from matplotlib.colors import TwoSlopeNorm
# Uses PARTICIPANT_DATA, COND_ORDER from §4.3a. RAW tags — NOT VA-derived labels.
#
# Raw tags are bucketed into the 5 circumplex classes via TAG_TO_QUADRANT so the
# whole thesis speaks ONE vocabulary. This does NOT reintroduce circularity:
# tag→quadrant is a fixed CONCEPTUAL grouping (Tension is always HA_LV), independent
# of each excerpt's measured VA position — so this cell stays tag-based and non-circular.
# The raw 8-tag lift is computed and saved first so the human-tag data survives.

def _cond_of(eid):
    return next((c for c in COND_ORDER if eid.endswith('_' + c)), 'UNKNOWN')

def _as_classes(tags):
    """raw tags → set of circumplex classes (set ⇒ no double counting when a rater
    picks two tags from the same quadrant, e.g. Power + Joyful Activation)."""
    return {TAG_TO_QUADRANT[t] for t in (tags if isinstance(tags, list) else [])
            if t in TAG_TO_QUADRANT}

def _as_raw(tags):
    return set(tags) if isinstance(tags, list) else set()

def build_incidence(data, thresh, vocab, mapper):
    """Binary (excerpt × vocab): excerpt carries v if >= thresh of its raters voted v."""
    inc = {}
    for eid, g in data.groupby('excerpt_id'):
        nres   = len(g)
        counts = Counter(v for tags in g['emotion_tags'] for v in mapper(tags))
        inc[eid] = {v: int(nres > 0 and counts.get(v, 0) / nres >= thresh) for v in vocab}
    df = pd.DataFrame(inc).T.reindex(columns=vocab)
    df['condition'] = [_cond_of(eid) for eid in df.index]
    return df

def lift_from_incidence(incidence, vocab):
    """lift[v, c] = P(v|c) / P(v overall); probs = fraction of excerpts carrying v."""
    overall = incidence[vocab].mean(axis=0).replace(0, np.nan)          # P(v)
    lift = pd.DataFrame(index=vocab, columns=COND_ORDER, dtype=float)
    for c in COND_ORDER:
        sub = incidence[incidence['condition'] == c][vocab]
        lift[c] = sub.mean(axis=0) / overall                            # P(v|c) / P(v)
    return lift

PRIMARY_THRESH         = 0.30
SENSITIVITY_THRESHOLDS = [0.20, 0.30, 0.40]

# ── (0) RAW 8-tag lift — preserved before any bucketing ──────────────────────
df_lift_raw = lift_from_incidence(
    build_incidence(PARTICIPANT_DATA, PRIMARY_THRESH, RAW_EMOTION_TAGS, _as_raw),
    RAW_EMOTION_TAGS)
df_lift_raw.to_csv(os.path.join(OUTPUT_DIR, 'lift_matrix_raw_tags.csv'))
print(f'── RAW 8-tag lift @ threshold={PRIMARY_THRESH:.2f} (preserved for reference) ──')
print(df_lift_raw.round(2).to_string())

# ── (1) Class-level lift — the reported analysis ─────────────────────────────
lift_by_thresh = {t: lift_from_incidence(
                        build_incidence(PARTICIPANT_DATA, t, EMOTION_LABELS, _as_classes),
                        EMOTION_LABELS)
                  for t in SENSITIVITY_THRESHOLDS}
df_lift = lift_by_thresh[PRIMARY_THRESH]

print(f'\n── Lift matrix (class × condition) @ threshold={PRIMARY_THRESH:.2f} ──')
print('   lift=1 independent · >1 concentrated · <1 suppressed · NaN = class never crosses threshold')
print(df_lift.round(2).to_string())
df_lift.to_csv(os.path.join(OUTPUT_DIR, 'lift_matrix.csv'))

# ── Heatmap (primary threshold), diverging around lift=1 ──────────────────────
M = df_lift.values.astype(float)
finite = M[np.isfinite(M)]
vmax = max(1.6, float(np.nanmax(finite))) if finite.size else 1.6
vmin = min(0.4, float(np.nanmin(finite))) if finite.size else 0.4
norm = TwoSlopeNorm(vmin=vmin, vcenter=1.0, vmax=vmax)
fig, ax = plt.subplots(figsize=(7.5, 5))
im = ax.imshow(np.ma.masked_invalid(M), cmap='RdBu_r', norm=norm, aspect='auto')
ax.set_xticks(range(len(COND_ORDER))); ax.set_xticklabels([CONDITIONS[c] for c in COND_ORDER])
ax.set_yticks(range(len(EMOTION_LABELS)))
ax.set_yticklabels([QUADRANT_DISPLAY[e] for e in EMOTION_LABELS])
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        v = M[i, j]
        ax.text(j, i, '—' if not np.isfinite(v) else f'{v:.2f}', ha='center', va='center',
                fontsize=9, color='white' if (np.isfinite(v) and (v >= vmax*0.85 or v <= vmin*1.15)) else 'black')
ax.set_title(f'Class–Condition Lift (raw tags → quadrant, threshold={PRIMARY_THRESH:.2f})')
fig.colorbar(im, ax=ax, label='lift = P(class | condition) / P(class)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_lift_heatmap.png'), dpi=150, bbox_inches='tight'); plt.show()

# ── Interpretation: top-2 classes by lift per condition (primary threshold) ───
print(f'\n── Top-2 classes by lift per condition (threshold={PRIMARY_THRESH:.2f}) ──')
for c in COND_ORDER:
    s = df_lift[c].dropna().sort_values(ascending=False)
    print(f'   {CONDITIONS[c]:<12}: ' + ', '.join(f'{e} ({v:.2f})' for e, v in s.head(2).items()))

# ── Sensitivity: is the TOP class per condition stable across thresholds? ─────
sens_rows = []
for c in COND_ORDER:
    row, tops = {'condition': CONDITIONS[c]}, []
    for t in SENSITIVITY_THRESHOLDS:
        s = lift_by_thresh[t][c].dropna().sort_values(ascending=False)
        top_e = s.index[0] if len(s) else '—'
        row[f'top@{t:.2f}'] = top_e if not len(s) else f'{top_e} ({s.iloc[0]:.2f})'
        tops.append(top_e)
    row['stable'] = 'yes' if len(set(tops)) == 1 else 'no'
    sens_rows.append(row)
df_sens = pd.DataFrame(sens_rows)
print('\n── Threshold sensitivity of top-class-per-condition ──')
print(df_sens.to_string(index=False))
df_sens.to_csv(os.path.join(OUTPUT_DIR, 'lift_threshold_sensitivity.csv'), index=False)

# ── Robustness: threshold-free lift using mean vote-share per excerpt ─────────
share_rows = {}
for eid, g in PARTICIPANT_DATA.groupby('excerpt_id'):
    nres   = len(g)
    counts = Counter(v for tags in g['emotion_tags'] for v in _as_classes(tags))
    share_rows[eid] = {e: (counts.get(e, 0) / nres if nres else 0.0) for e in EMOTION_LABELS}
share = pd.DataFrame(share_rows).T.reindex(columns=EMOTION_LABELS)
share['condition'] = [_cond_of(eid) for eid in share.index]
overall_share = share[EMOTION_LABELS].mean(axis=0).replace(0, np.nan)
lift_share = pd.DataFrame(index=EMOTION_LABELS, columns=COND_ORDER, dtype=float)
for c in COND_ORDER:
    lift_share[c] = share[share['condition'] == c][EMOTION_LABELS].mean(axis=0) / overall_share
print('\n── Robustness check — threshold-free lift (mean vote-share) ──')
print(lift_share.round(2).to_string())
lift_share.to_csv(os.path.join(OUTPUT_DIR, 'lift_matrix_meanshare.csv'))

# ── Class share per condition — the GROUND-TRUTH classes (VA-derived, §1.2) ─────────────────────
# Descriptive only (no test): the VA-derived labels are a function of the ratings, which themselves vary
# with the condition, so they are not used to TEST class–condition binding (the tag-based lift above is).
# It shows what the classifier of Part 6 is asked to predict in each condition.
_gt = pd.DataFrame({'condition': df_expr['condition'].to_numpy(),
                    'class': [l[0] if len(l) else 'none' for l in df_expr['emotion_labels']]})
_ct = pd.crosstab(_gt['class'], _gt['condition']).reindex(index=EMOTION_LABELS, columns=COND_ORDER).fillna(0)
CLASS_SHARE_BY_COND = (100 * _ct / _ct.sum(axis=0)).round(1)
print('\n── Ground-truth class share per condition (% of the 20 excerpts of that condition; VA-derived, §1.2) ──')
print(pd.concat({'n': _ct.astype(int), '%': CLASS_SHARE_BY_COND}, axis=1).to_string())
CLASS_SHARE_BY_COND.to_csv(os.path.join(OUTPUT_DIR, 'class_share_by_condition.csv'))

print('\n✅ Saved: lift_matrix.csv, lift_matrix_raw_tags.csv, fig_lift_heatmap.png, '
      'lift_threshold_sensitivity.csv, lift_matrix_meanshare.csv, class_share_by_condition.csv')


## Part 4 · Manipulation check — Conclusion: the manipulation check

**Reading (filled from the values printed by the cell below — no number is typed here):** the performer
**escalated the techniques** from MEC to EXP to EXG (§4.2: the techniques with BH q < .05), the listeners'
**arousal changed** with the manipulation (§4.3a Friedman, §4.3b Page), and their **valence did not**.
This is the premise of everything that follows: a model can only learn a technique → emotion link the
manipulation actually created, and the audience moved on arousal, not on valence. If the printed values
contradict this reading, the cell says so.


In [ ]:
# ── Part 4 · Conclusion — printed from the variables of §4.2 / §4.3 (nothing typed) ──
# NEEDS : df_dose, DOSE_EMO, DOSE_FAMILY_N (§4.2 / §4.3b), FRIEDMAN_RES (§4.3a)
# MAKES : MANIP_CHECK (dict — read by §11.1)
# FEEDS : §11.1 (claims 1 and 2)
_tech = df_dose[df_dose.kind == 'technique']
_tsig = _tech[_tech.q < 0.05]
_aro, _val = DOSE_EMO.get('arousal_mean'), DOSE_EMO.get('valence_mean')
MANIP_CHECK = {
    'n_techniques': int(len(_tech)),
    'n_techniques_q05': int(len(_tsig)),
    'n_techniques_q05_unflagged': int((_tsig['flag'] == '').sum()),
    'arousal_page_z': float(_aro.z) if _aro is not None else np.nan,
    'arousal_page_q': float(_aro.q) if _aro is not None else np.nan,
    'valence_page_z': float(_val.z) if _val is not None else np.nan,
    'valence_page_q': float(_val.q) if _val is not None else np.nan,
    'arousal_friedman_p': FRIEDMAN_RES.get('Arousal', {}).get('p', np.nan),
    'valence_friedman_p': FRIEDMAN_RES.get('Valence', {}).get('p', np.nan),
}
_m = MANIP_CHECK
print('📌 PART 4 · MANIPULATION CHECK')
print(f'   performers: {_m["n_techniques_q05"]} of {_m["n_techniques"]} techniques escalate monotonically with '
      f'BH q < .05 ({_m["n_techniques_q05_unflagged"]} of them not flagged ⚑; family of {DOSE_FAMILY_N} tests)')
print(f'   listeners — arousal: Page z = {_m["arousal_page_z"]:+.2f}, q = {_m["arousal_page_q"]:.4f} · '
      f'Friedman p = {_m["arousal_friedman_p"]:.4f}')
print(f'   listeners — valence: Page z = {_m["valence_page_z"]:+.2f}, q = {_m["valence_page_q"]:.4f} · '
      f'Friedman p = {_m["valence_friedman_p"]:.4f}')
_ok = (_m['n_techniques_q05'] > 0, _m['arousal_page_q'] < 0.05, not (_m['valence_page_q'] < 0.05))
print('   reading "techniques escalated · arousal changed · valence did not": '
      + ('✅ consistent with the printed values' if all(_ok) else
         '⚠️ NOT fully consistent — ' + ', '.join(t for t, ok in zip(
             ('no technique escalates (q < .05)', 'arousal does not change (q ≥ .05)',
              'valence DOES change (q < .05)'), _ok) if not ok)))


## Part 5 · Evaluation yardstick — §5.1 Evaluation protocol & CV engine
**Question of this part:** before any prediction is scored — what does *chance* look like for the  
deployed pipeline, and how is every number cross-validated?  
**Hands to the next part:** the CV engine (`get_cv`, `cv_predict_default`, `nested_ridge_predict`, …),
the Dummy floors (§5.2) and the empirical null of the deployed classifier (§5.3, `EMP_NULL_F1`,
`EMP_NULL_PRIMARY`) — so that Part 6 can read every F1 against its null the moment it is printed.

**One protocol, one engine.** Every cross-validated number from here to Part 10 comes from the helpers
defined in this cell — no later cell builds its own CV splitter.

**Two MERT representations — always named next to a number.** *MERT-PCA25* (the §2.1b PCA-95 block,
25 components, `X_A`) feeds every **classification** head; *MERT-full 768* (`X_MERT_FULL` = `MERT_REG`)
feeds every **regression** head (`MERT_REG_REPR = 'full'`). Why: tier `4A-mert-pca` (§6.4) showed that
PCA-95 keeps the high-variance **piece-identity** directions and discards the arousal-relevant variance
(Phase A run, `piece`: R² arousal −0.25 with PCA vs +0.45 full).

| | |
|---|---|
| **Primary protocol** `piece` | leave-one-**piece**-out (`LeaveOneGroupOut` on `PIECE_GROUPS`) — generalisation to *unseen material* |
| **Secondary** `loo` | leave-one-excerpt-out — **optimistic**: the held-out excerpt's two sibling renditions stay in training |
| Why | 95% (valence) / 91% (arousal) of the between-excerpt variance is **between pieces**; the condition explains 1% / 4%. LOOCV therefore mostly measures piece recognition. Both are always printed — `piece` first — and their gap is reported. |
| Default classifier | **Nested model selection** (`NestedSelect`, Phase B): inside every outer training fold an inner `GroupKFold(5)` over the training pieces picks one of 13 configs (NB · SVM-Linear balanced · LogReg balanced, C ∈ 10⁻³…10²) by pooled inner macro-F1, ties → NB / smallest C; the winner is refitted on the whole training fold. Choosing on the reported CV would be selection bias — nesting makes the score an unbiased estimate of the *select-then-fit procedure*. GaussianNB, the Phase A pre-declared default, fell below chance under `piece` and stays as a reported row (§6.1 keeps 8 classifiers as a sensitivity table only) |
| Regression | `nested_ridge_predict` — α chosen by a piece-grouped inner `GroupKFold(5)` inside every outer training fold (the old `RidgeCV` GCV leaked sibling renditions into its own inner loop) |
| Block weighting | optional `block_sizes` (column counts of the tool blocks) → `StandardScaler → BlockScaler → Ridge`: block *b* is scaled by w_b = √(P / (B·p_b)), so every tool block carries the same standardised variance P/B and the total stays P (the α grid keeps its meaning) — group ridge with α_b ∝ B·p_b / P. The weights depend only on the column counts, never on data or labels. Used for the multi-block MERT-full regression heads when `REG_BLOCK_WEIGHTING = 'equal'` (§0.3); B = 1 is exactly the unweighted head |
| Metrics | macro-F1 + chance floors (mean, SD, P95 over 200 re-seeded Dummy repeats); r **and** out-of-sample R² (baseline = training-fold mean) **and** RMSE |

**Guard.** For GaussianNB the new manual engine must reproduce the old
`cross_val_predict(Pipeline([StandardScaler, SafeMultiOutputClassifier(GaussianNB())]))` element-wise on
`X_A` under both protocols — proof that it is a faithful refactor, not a different model.

**Guard (B-NMS).** The nested selection runs its outer folds in parallel (joblib, one BLAS thread per
fold) and memoises byte-identical calls; the cell asserts that the parallel and the serial run give
**identical** predictions, chosen configs and inner scores on `X_B`, and prints how often each
config was chosen.

**Guard (block weighting).** On `X_C_REG` (arousal, `piece`) the cell asserts that a single block
reproduces the unweighted head exactly, that the weights keep Σ p_b·w_b² = P with an equal share per
block, that `BlockScaler` is stateless, and that the weighted closed-form `NestedRidgeOperator` equals the
weighted `nested_ridge_predict` (same α per fold, |Δŷ| ≤ 1e-8). It also prints the class prevalences of
`Y` and the analytic chance F1 derived from them (nothing hardcoded).

In [ ]:
# ── §5.1 Evaluation protocol & CV engine ────────────────────────────
# NEEDS : bh_fdr (§0.3), df_expr, DTW_MODEL_COLS, eids, EMOTION_LABELS, ESSENTIA_FORWARD, get_piece,
#          MERT_REG, N_DUMMY_REPEATS, REG_BLOCK_WEIGHTING, SEED, SELECT_CONFIGS, X_A, X_B, X_C, X_C_REG, Y
# MAKES : PIECE_GROUPS, PRIMARY_PROTOCOL, PROTOCOLS, PROTOCOL_LABEL, get_cv, cv_splits,
#          SafeMultiOutputClassifier, NB_CLF_NAME, make_nb_clf, DEFAULT_CLF_NAME,
#          NMS_N_JOBS, cv_predict_nested_select, cv_predict_default, nms_counts_str,
#          freedman_lane_targets, split_half_ceiling, cv_predict_multilabel, RIDGE_ALPHAS,
#          block_weights, BlockScaler, select_ridge_alpha, nested_ridge_predict,
#          NestedRidgeOperator, oos_baseline, oos_r2, rmse, within_piece_permutation,
#          warn_negative_loo_r, NEG_LOO_R_NOTE, DUMMY_STRATEGIES, dummy_predict,
#          chance_floors, floors_from_summary, DUMMY_REPS, DUMMY_SUMMARY, DUMMY_FIRST_PRED,
#          CHANCE_FLOOR_MEAN, CHANCE_FLOOR_P95, CHANCE_STRONGEST
# FEEDS : every cross-validated number in Parts 5–10 (§5.2–§5.3, §6.1–§6.9, §7.1–§7.5, §8.1,
#          §9.2, §10.1–§10.3) — no other cell builds its own CV splitter
# ═══════════════════════════════════════════════════════════════════════════════
# §5.1 · ONE evaluation protocol, ONE CV engine — every later cell calls these helpers
# ═══════════════════════════════════════════════════════════════════════════════
# >>> HELPERS:cv_engine
import hashlib
import itertools
import os
import warnings
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits
from sklearn.exceptions import ConvergenceWarning
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone as sk_clone
from sklearn.model_selection import LeaveOneOut, LeaveOneGroupOut, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.naive_bayes import GaussianNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, hamming_loss

# ── Protocols ─────────────────────────────────────────────────────────────
# 95% (valence) / 91% (arousal) of the between-excerpt variance is BETWEEN PIECES, the
# condition explains 1% / 4% (audit, Oct 2026). LOOCV keeps a held-out excerpt's two
# sibling renditions in training, so it mostly measures piece recognition. The PRIMARY
# protocol is therefore leave-one-PIECE-out; LOOCV is always reported next to it as the
# optimistic secondary number, and the gap between them is itself a result.
PRIMARY_PROTOCOL = 'piece'
PROTOCOLS        = ('piece', 'loo')            # printing / saving order: primary first
PROTOCOL_LABEL   = {
    'piece' : 'PRIMARY — unseen material (leave-one-piece-out)',
    'loo'   : "secondary — optimistic, same piece's other renditions stay in training (LOOCV)",
    'piece5': 'GroupKFold(5) over pieces (§7.3b probe only)',
}
INNER_SPLITS         = 5      # inner GroupKFold over the TRAINING rows' pieces
INNER_MIN_POS        = 5      # A5 edge case: fewer positives in the training fold → no tuning
INNER_MIN_POS_PIECES = 2      # … or positives in fewer than 2 training pieces → no tuning
RIDGE_ALPHAS         = np.logspace(-2, 4, 13)  # nested ridge grid (ascending = weaker → stronger)
DUMMY_STRATEGIES     = ('most_frequent', 'stratified', 'uniform')
DUMMY_SEED_STRIDE    = 100_000   # base seed of repeat r = seed + r·stride (> 1000·n_folds + n_labels)


def get_cv(protocol):
    """→ (cv, groups). The ONLY place a CV splitter is built.
    'piece' = LeaveOneGroupOut on PIECE_GROUPS (primary) · 'loo' = LeaveOneOut (secondary)
    · 'piece5' = GroupKFold(5) on PIECE_GROUPS (used only by the §7.3b MERT probe)."""
    if protocol == 'piece':
        return LeaveOneGroupOut(), PIECE_GROUPS
    if protocol == 'loo':
        return LeaveOneOut(), None
    if protocol == 'piece5':
        return GroupKFold(n_splits=5), PIECE_GROUPS
    raise ValueError(f'unknown protocol {protocol!r} — use "piece", "loo" or "piece5"')


def cv_splits(protocol, n):
    """Materialised outer (train, test) index pairs for `n` rows aligned to PIECE_GROUPS."""
    cv, g = get_cv(protocol)
    if g is not None and len(g) != n:
        raise ValueError(f'cv_splits: {n} rows but PIECE_GROUPS has {len(g)} — rows misaligned')
    return list(cv.split(np.zeros((n, 1)), groups=g))


# bh_fdr (Benjamini–Hochberg) now lives in §0.3 (Phase C2): the Part 4 manipulation check (§4.2)
# needs it before this engine runs. Same function, same behaviour; a stand-alone test of this
# HELPERS block must import / define it alongside.


# ── SafeMultiOutputClassifier (moved here from §6.1 — the §5.1 guard needs it) ──
# LOOCV with N=60 and 5 rare circumplex classes means some label columns will be
# all-zeros in a given training fold (e.g. a class present in only 1-2 samples).
# sklearn's SVC/SVM raises ValueError when it sees only 1 class.
# This wrapper silently predicts the majority class (always 0) for such columns
# instead of crashing — scientifically equivalent to "no positive evidence found".
class SafeMultiOutputClassifier(BaseEstimator, ClassifierMixin):
    """MultiOutputClassifier robust to single-class label columns in LOOCV folds."""
    def __init__(self, estimator):
        self.estimator = estimator

    def fit(self, X, Y):
        self.n_outputs_ = Y.shape[1]
        self.estimators_ = []
        self.majority_    = []
        for j in range(self.n_outputs_):
            y_col = Y[:, j]
            maj   = int(y_col.mean() >= 0.5)
            self.majority_.append(maj)
            if len(np.unique(y_col)) < 2:
                self.estimators_.append(None)   # degenerate — predict constant
            else:
                est = sk_clone(self.estimator)
                est.fit(X, y_col)
                self.estimators_.append(est)
        return self

    def predict(self, X):
        preds = []
        for j, est in enumerate(self.estimators_):
            if est is None:
                preds.append(np.full(len(X), self.majority_[j], dtype=int))
            else:
                preds.append(est.predict(X).astype(int))
        return np.column_stack(preds)


# ── NaiveBayes — the Phase A pre-declared default (kept as a reported row, B-NMS) ──
# Phase A locked GaussianNB (sklearn defaults, empirical priors per label) in advance as the
# single default classifier. Under the primary 'piece' protocol it fell BELOW the chance
# floor on every feature set (2026-10-09 run), so it is no longer the default; it stays as
# NMS config #1 and as the explicit row "NaiveBayes (pre-declared Phase A default — failed
# under piece)" in §6.6 / §11.1. Its known downsides remain stated: violated independence
# (correlated blocks double-count evidence), no shrinkage beyond var_smoothing, Gaussian fit
# of zero-inflated features, over-confident predict_proba (never interpreted here).
NB_CLF_NAME = 'NaiveBayes'


def make_nb_clf():
    return GaussianNB()


# ── B-NMS · Nested model selection — the default classifier from Phase B on ──────
# WHY NESTED: picking the classifier on the same CV that then reports its score is
# selection bias — the maximum of 13 noisy CV estimates is optimistic by construction,
# and at N=60 the spread between configs is of the same size as the effects we report.
# Nesting moves the choice INSIDE every outer training fold (inner GroupKFold over the
# training pieces only), so the outer score is an unbiased estimate of the whole
# "select-then-fit" PROCEDURE — not of any single config. The config chosen can differ
# between outer folds; that is expected and is printed (frequency per run).
# Per outer fold: for each of the 13 SELECT_CONFIGS (§0.3) predict the inner folds
# (binary relevance, scaler fitted inside every inner split, a single-class training label
# predicts that constant = SafeMultiOutput semantics), score the POOLED inner out-of-fold
# macro-F1 over the labels, pick the best — ties → the earliest config in SELECT_CONFIGS
# (NB, then smallest C, SVM before LogReg) — refit it on the whole outer training fold and
# predict the outer test rows.
# Runtime: a content-hash memo returns byte-identical calls (same X, Y, protocol, configs,
# PIECE_GROUPS) without refitting, and the outer folds run in parallel (joblib/loky). Every
# fold runs under threadpool_limits(1), so parallel and serial results are bit-identical —
# asserted below on X_B.
DEFAULT_CLF_NAME = 'NestedSelect'
NMS_N_JOBS       = max(1, os.cpu_count() or 1)
NMS_TIE_TOL      = 1e-12        # inner macro-F1 within this of the best counts as a tie
_NMS_MEMO        = {}


def _macro_f1_np(Y, P):
    """Macro-F1 over label columns (== f1_score(average='macro', zero_division=0))."""
    return float(np.mean([_f1_binary(Y[:, j], P[:, j]) for j in range(Y.shape[1])]))


def _fit_predict_labels(make_est, A, Ytr, B, conv):
    """Binary relevance on already-scaled blocks: fresh make_est() per label, a single-class
    training label predicts that constant. conv = [count] of ConvergenceWarnings seen."""
    out = np.zeros((B.shape[0], Ytr.shape[1]), dtype=int)
    for j in range(Ytr.shape[1]):
        y = Ytr[:, j].astype(int)
        if np.unique(y).size < 2:
            out[:, j] = int(y[0])
            continue
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter('always', ConvergenceWarning)
            out[:, j] = np.asarray(make_est().fit(A, y).predict(B)).astype(int)
        conv[0] += sum(issubclass(x.category, ConvergenceWarning) for x in w)
        for x in w:                                   # anything else is re-raised visibly
            if not issubclass(x.category, ConvergenceWarning):
                warnings.warn_explicit(x.message, x.category, x.filename, x.lineno)
    return out


def _nms_fold(X, Y, tr, te, g_tr, factories, n_inner):
    """One outer fold of the nested selection → (test predictions, chosen index, inner
    scores, #ConvergenceWarnings). Pure function of its arguments (joblib worker)."""
    with threadpool_limits(limits=1):
        conv = [0]
        Xtr, Ytr = X[tr], Y[tr]
        oof = np.zeros((len(factories),) + Ytr.shape, dtype=int)
        for itr, iva in GroupKFold(n_splits=n_inner).split(Xtr, groups=g_tr):
            sc = StandardScaler().fit(Xtr[itr])
            A, B = sc.transform(Xtr[itr]), sc.transform(Xtr[iva])
            for c, fac in enumerate(factories):
                oof[c][iva] = _fit_predict_labels(fac, A, Ytr[itr], B, conv)
        scores = np.array([_macro_f1_np(Ytr, oof[c]) for c in range(len(factories))])
        k = int(np.flatnonzero(scores >= scores.max() - NMS_TIE_TOL)[0])   # earliest = simplest
        sc = StandardScaler().fit(Xtr)
        pred = _fit_predict_labels(factories[k], sc.transform(Xtr), Ytr, sc.transform(X[te]), conv)
    return pred, k, scores, conv[0]


def _nms_key(X, Y, protocol, configs):
    h = hashlib.sha1()
    for a in (X, Y, PIECE_GROUPS.astype(str)):
        a = np.ascontiguousarray(a)
        h.update(repr((a.shape, a.dtype.str)).encode()); h.update(a.tobytes())
    h.update(repr((protocol, INNER_SPLITS, NMS_TIE_TOL,
                   [(name, repr(fac)) for name, fac in configs])).encode())
    return h.hexdigest()


def cv_predict_nested_select(X, Y, protocol, configs=None, n_jobs=None, use_memo=True,
                             label=None, verbose=True, return_info=False):
    """Nested model selection (B-NMS). Outer loop = cv_splits(protocol); inside every outer
    training fold an inner GroupKFold(INNER_SPLITS) over the TRAINING pieces chooses one of
    `configs` (default SELECT_CONFIGS) by pooled inner out-of-fold macro-F1 (ties → list
    order); the winner is refitted on the whole outer training fold. → Yp (n × L int),
    plus, with return_info, {'chosen': config name per outer fold, 'counts', 'inner_scores'
    (folds × configs), 'n_conv_warnings', 'memo_hit'}."""
    configs = list(SELECT_CONFIGS if configs is None else configs)
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y).astype(int)
    if Y.ndim != 2 or X.shape[0] != Y.shape[0]:
        raise ValueError(f'cv_predict_nested_select: X {X.shape} / Y {Y.shape} misaligned')
    if not np.isfinite(X).all():
        raise ValueError('cv_predict_nested_select: non-finite X')
    key = _nms_key(X, Y, protocol, configs) if use_memo else None
    hit = key is not None and key in _NMS_MEMO
    if hit:
        Yp, info = _NMS_MEMO[key]
        Yp, info = Yp.copy(), dict(info, memo_hit=True)
    else:
        splits = cv_splits(protocol, X.shape[0])
        for tr, te in splits:
            if protocol == 'piece' and set(PIECE_GROUPS[te]) & set(PIECE_GROUPS[tr]):
                raise AssertionError('cv_predict_nested_select: test piece present in training rows')
        facs = [fac for _, fac in configs]
        jobs = [(X, Y, tr, te, PIECE_GROUPS[tr], facs, INNER_SPLITS) for tr, te in splits]
        nj = NMS_N_JOBS if n_jobs is None else int(n_jobs)
        if nj == 1:
            res = [_nms_fold(*a) for a in jobs]
        else:
            res = Parallel(n_jobs=nj, backend='loky')(delayed(_nms_fold)(*a) for a in jobs)
        Yp = np.zeros(Y.shape, dtype=int)
        for (tr, te), (pred, _, _, _) in zip(splits, res):
            Yp[te] = pred
        chosen = [configs[k][0] for _, k, _, _ in res]
        info = {'chosen': chosen,
                'counts': {nm: chosen.count(nm) for nm, _ in configs if nm in chosen},
                'inner_scores': np.vstack([s for _, _, s, _ in res]),
                'n_conv_warnings': int(sum(c for _, _, _, c in res)), 'memo_hit': False}
        if key is not None:
            _NMS_MEMO[key] = (Yp.copy(), info)
    if verbose:
        print(f'     NestedSelect[{protocol}]{" " + label if label else ""}: '
              + nms_counts_str(info) + (' (memo)' if hit else '')
              + (f' · {info["n_conv_warnings"]} LogReg ConvergenceWarnings'
                 if info['n_conv_warnings'] else ''))
    return (Yp, info) if return_info else Yp


def nms_counts_str(info):
    """'SVM-Linear C=0.1×12 · NB×5 · …' — configs by decreasing frequency (ties: list order)."""
    c = info['counts']
    return ' · '.join(f'{k}×{v}' for k, v in sorted(c.items(), key=lambda kv: -kv[1]))


def cv_predict_default(X, Y, protocol, **kw):
    """The default classifier of every single-classifier analysis = nested selection."""
    return cv_predict_nested_select(X, Y, protocol, **kw)


def _f1_binary(y, p):
    """Binary F1 of the positive class (== f1_score(..., zero_division=0)), numpy-fast."""
    y = np.asarray(y).astype(int); p = np.asarray(p).astype(int)
    tp = int(np.sum((y == 1) & (p == 1)))
    den = 2 * tp + int(np.sum((y == 0) & (p == 1))) + int(np.sum((y == 1) & (p == 0)))
    return 0.0 if den == 0 else 2.0 * tp / den


def _grid_combos(tune_grid):
    keys = list(tune_grid)
    for k in keys:
        v = np.asarray(tune_grid[k], dtype=float)
        if k == 'C' and np.any(np.diff(v) <= 0):
            raise ValueError('tune_grid["C"] must be strictly ascending '
                             '(first = strongest regularisation, which wins ties)')
    return [dict(zip(keys, vals)) for vals in itertools.product(*(list(tune_grid[k]) for k in keys))]


def cv_predict_multilabel(make_est, X, Y, protocol, tune_grid=None, inner_scoring='f1',
                          fallback_params=None, return_info=False):
    """Cross-validated 0/1 predictions, binary relevance (one estimator per label).

    Manual outer loop over get_cv(protocol). Per outer training fold: StandardScaler is
    fitted on the training rows only; a label that is single-class in the training fold
    predicts that constant (exactly SafeMultiOutputClassifier's semantics); otherwise a
    FRESH make_est() is fitted on that label.

    tune_grid (A5): hyper-parameters are chosen PER LABEL INSIDE the outer training fold
    by an inner GroupKFold(INNER_SPLITS) over the training rows' pieces (scaler refitted
    inside every inner split). Score = binary F1 of that label on the POOLED inner
    out-of-fold predictions; ties → the first combo = strongest regularisation (grid must
    be ordered that way). If the training fold has < INNER_MIN_POS positives for the label,
    or its positives span < INNER_MIN_POS_PIECES pieces, tuning is SKIPPED and
    fallback_params (default C=1.0) is used — counted in info['n_skipped'].

    Returns Yp (n × L int, aligned to Y); with return_info=True also an info dict."""
    if inner_scoring != 'f1':
        raise ValueError('only inner_scoring="f1" (binary F1 of the label) is implemented')
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y)
    if Y.ndim != 2 or X.shape[0] != Y.shape[0]:
        raise ValueError(f'cv_predict_multilabel: X {X.shape} / Y {Y.shape} misaligned')
    n, L = Y.shape
    Yp   = np.zeros((n, L), dtype=int)
    info = {'n_fit': 0, 'n_constant': 0, 'n_tuned': 0, 'n_skipped': 0, 'chosen': []}
    combos = _grid_combos(tune_grid) if tune_grid is not None else None
    if combos is not None:
        fb = (dict(fallback_params) if fallback_params is not None
              else {k: 1.0 for k in tune_grid})
    for fold, (tr, te) in enumerate(cv_splits(protocol, n)):
        sc  = StandardScaler().fit(X[tr])
        Xtr, Xte = sc.transform(X[tr]), sc.transform(X[te])
        inner = None
        if combos is not None:
            g_tr  = PIECE_GROUPS[tr]
            inner = []
            for itr, iva in GroupKFold(n_splits=INNER_SPLITS).split(X[tr], groups=g_tr):
                isc = StandardScaler().fit(X[tr][itr])
                inner.append((itr, iva, isc.transform(X[tr][itr]), isc.transform(X[tr][iva])))
        for j in range(L):
            ytr = Y[tr, j].astype(int)
            if np.unique(ytr).size < 2:                # degenerate fold → the constant
                Yp[te, j] = int(ytr[0])
                info['n_constant'] += 1
                continue
            est = make_est()
            if combos is not None:
                pos = ytr == 1
                if pos.sum() < INNER_MIN_POS or np.unique(g_tr[pos]).size < INNER_MIN_POS_PIECES:
                    params = dict(fb)
                    info['n_skipped'] += 1
                else:
                    oof = np.zeros((len(combos), len(ytr)), dtype=int)
                    for itr, iva, A, B in inner:
                        yi = ytr[itr]
                        if np.unique(yi).size < 2:
                            oof[:, iva] = int(yi[0])
                            continue
                        for c, prm in enumerate(combos):
                            e = make_est()
                            e.set_params(**prm)
                            oof[c, iva] = np.asarray(e.fit(A, yi).predict(B)).astype(int)
                    scores = [_f1_binary(ytr, oof[c]) for c in range(len(combos))]
                    params = dict(combos[int(np.argmax(scores))])   # first max = strongest reg.
                    info['n_tuned'] += 1
                est.set_params(**params)
                info['chosen'].append((fold, j, params))
            est.fit(Xtr, ytr)
            info['n_fit'] += 1
            Yp[te, j] = np.asarray(est.predict(Xte)).astype(int)
    return (Yp, info) if return_info else Yp


# ── Block weighting of multi-block regression heads (equal a-priori share per tool block) ──
# After StandardScaler every column carries unit variance, so a block of p_b columns carries
# p_b units and ONE ridge α lets the 768 MERT-full columns dominate a ~10-column block. Scaling
# block b by w_b = sqrt(P / (B·p_b)) (P = Σ p_b, B = number of blocks) gives every block the same
# standardised variance P/B while the total stays Σ p_b·w_b² = P, so the α grid keeps its
# meaning. Ridge on the weighted columns = group ridge with α_b = α / w_b² ∝ B·p_b / P. The
# weights are a function of the column COUNTS only — never of the data or the labels — and are
# applied inside the fold, after the scaler (leakage-free by construction). B = 1 → w = 1.
def block_weights(sizes):
    """Per-block weights w_b = sqrt(P / (B·p_b)) for column counts `sizes` (p_b ≥ 1)."""
    p = np.asarray(sizes, dtype=float)
    if p.ndim != 1 or p.size == 0 or np.any(p < 1) or np.any(p != np.round(p)):
        raise ValueError(f'block_weights: sizes must be positive integer column counts, got {sizes!r}')
    return np.sqrt(p.sum() / (p.size * p))


def _block_col_weights(block_sizes, n_cols):
    """Per-column weight vector for `block_sizes` (must sum to n_cols); None when there is
    nothing to weight (no block structure given, or a single block — w = 1 exactly)."""
    if block_sizes is None:
        return None
    sizes = [int(s) for s in block_sizes]
    if sum(sizes) != int(n_cols):
        raise ValueError(f'block sizes {sizes} sum to {sum(sizes)}, but X has {n_cols} columns')
    if len(sizes) == 1:
        return None
    return np.repeat(block_weights(sizes), sizes)


class BlockScaler(BaseEstimator, TransformerMixin):
    """Fixed per-block column weights (block_weights), applied AFTER StandardScaler inside a
    pipeline. Stateless: fit() only checks the column count — nothing is estimated from X or y."""

    def __init__(self, sizes):
        self.sizes = sizes

    def fit(self, X, y=None):
        w = _block_col_weights(self.sizes, np.asarray(X).shape[1])
        self.weights_ = np.ones(np.asarray(X).shape[1]) if w is None else w
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        if X.shape[1] != self.weights_.size:
            raise ValueError(f'BlockScaler: X has {X.shape[1]} columns, fitted for {self.weights_.size}')
        return X * self.weights_[None, :]


# ── Nested ridge: α chosen by piece-grouped inner CV (replaces every RidgeCV) ──
def _ridge_pipe(alpha, block_sizes=None):
    """StandardScaler → [BlockScaler] → Ridge(α); the BlockScaler step only when there is more
    than one block (B = 1 → exactly the unweighted pipeline)."""
    if block_sizes is None or len(block_sizes) == 1:
        return Pipeline([('sc', StandardScaler()), ('r', Ridge(alpha=alpha))])
    return Pipeline([('sc', StandardScaler()), ('bw', BlockScaler(list(block_sizes))),
                     ('r', Ridge(alpha=alpha))])


def _pick_alpha_idx(sse):
    """Index of the α with minimal pooled inner SSE; ties (within 1e-9 relative) → the
    LARGEST α, i.e. the strongest regularisation (grid must be ascending)."""
    sse = np.asarray(sse, dtype=float)
    return int(np.flatnonzero(sse <= sse.min() * (1.0 + 1e-9) + 1e-15).max())


def select_ridge_alpha(X, y, groups, alphas=RIDGE_ALPHAS, make_pipe=None, block_sizes=None):
    """α by inner GroupKFold(INNER_SPLITS) over `groups` (pieces), minimising the POOLED
    out-of-fold squared error. make_pipe(α) → an unfitted pipeline (default
    StandardScaler → [BlockScaler(block_sizes)] → Ridge(α)); the scaler is always refitted
    inside each inner split. block_sizes (column counts per tool block) applies the fixed
    equal-share block weights after the scaler; it cannot be combined with make_pipe."""
    X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
    alphas = np.asarray(alphas, dtype=float)
    if np.any(np.diff(alphas) <= 0):
        raise ValueError('alphas must be strictly ascending')
    if make_pipe is not None and block_sizes is not None:
        raise ValueError('select_ridge_alpha: pass block_sizes OR a custom make_pipe, not both')
    w = _block_col_weights(block_sizes, X.shape[1])
    sse = np.zeros(len(alphas))
    for itr, iva in GroupKFold(n_splits=INNER_SPLITS).split(X, y, groups=groups):
        if make_pipe is None:                      # same maths, scaler fitted once per split
            sc = StandardScaler().fit(X[itr])
            A, B = sc.transform(X[itr]), sc.transform(X[iva])
            if w is not None:                      # fixed block weights, after the scaler
                A, B = A * w[None, :], B * w[None, :]
            for k, a in enumerate(alphas):
                yp = Ridge(alpha=a).fit(A, y[itr]).predict(B)
                sse[k] += float(np.sum((y[iva] - yp) ** 2))
        else:
            for k, a in enumerate(alphas):
                yp = make_pipe(a).fit(X[itr], y[itr]).predict(X[iva])
                sse[k] += float(np.sum((y[iva] - yp) ** 2))
    return float(alphas[_pick_alpha_idx(sse)])


def nested_ridge_predict(X, y, protocol, alphas=RIDGE_ALPHAS, make_pipe=None, block_sizes=None):
    """Outer loop = get_cv(protocol). Inside every outer TRAINING fold α is chosen by
    select_ridge_alpha on the training rows' pieces only; StandardScaler → Ridge(α) is
    then refitted on the whole training fold and predicts the test rows. No test row
    (and, under 'piece', no test-piece row) ever enters the inner selection.
    block_sizes (optional) = column counts of the tool blocks of X: the fixed equal-share
    block weights are applied after the scaler in every fit (B = 1 → identical to None).
    → (yhat aligned to y, chosen α per outer fold)."""
    X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
    if make_pipe is not None and block_sizes is not None:
        raise ValueError('nested_ridge_predict: pass block_sizes OR a custom make_pipe, not both')
    _block_col_weights(block_sizes, X.shape[1])                  # validates the sizes
    yhat, chosen = np.full(len(y), np.nan), []
    pipe_of = make_pipe if make_pipe is not None else (lambda a: _ridge_pipe(a, block_sizes))
    for tr, te in cv_splits(protocol, len(y)):
        a = select_ridge_alpha(X[tr], y[tr], PIECE_GROUPS[tr], alphas, make_pipe, block_sizes)
        yhat[te] = pipe_of(a).fit(X[tr], y[tr]).predict(X[te])
        chosen.append(a)
    if not np.isfinite(yhat).all():
        raise RuntimeError('nested_ridge_predict: non-finite predictions')
    return yhat, np.asarray(chosen)


class NestedRidgeOperator:
    """Closed-form twin of nested_ridge_predict (default StandardScaler → Ridge only).

    For a FIXED X, protocol and α grid, every ridge fit in the nested procedure is a
    linear smoother of y: ŷ_B = ȳ_A + B·V·diag(s/(s²+α))·Uᵀ·(y_A − ȳ_A) with A = U·S·Vᵀ the
    standardised training block. All SVDs are computed ONCE in __init__; __call__(y) then
    reproduces nested_ridge_predict(X, y, protocol) — same inner splits, same pooled-SSE
    α rule, same tie-break — with matrix products only. Built for permutation nulls where
    X never changes; wherever it is used, its equivalence to the sklearn path is ASSERTED
    on the observed targets first. block_sizes: as in nested_ridge_predict — the standardised
    blocks A and B are multiplied by the fixed block weights before the SVD."""

    def __init__(self, X, protocol, alphas=RIDGE_ALPHAS, block_sizes=None):
        X = np.asarray(X, dtype=float)
        self.alphas = np.asarray(alphas, dtype=float)
        if np.any(np.diff(self.alphas) <= 0):
            raise ValueError('alphas must be strictly ascending')
        w = _block_col_weights(block_sizes, X.shape[1])
        self.outer = []
        for tr, te in cv_splits(protocol, X.shape[0]):
            inner = []
            for itr, iva in GroupKFold(n_splits=INNER_SPLITS).split(X[tr], groups=PIECE_GROUPS[tr]):
                inner.append((tr[itr], tr[iva], self._smoother(X[tr[itr]], X[tr[iva]], w)))
            self.outer.append((tr, te, inner, self._smoother(X[tr], X[te], w)))

    @staticmethod
    def _smoother(A_raw, B_raw, w=None):
        sc = StandardScaler().fit(A_raw)
        A, B = sc.transform(A_raw), sc.transform(B_raw)
        if w is not None:                                            # fixed block weights
            A, B = A * w[None, :], B * w[None, :]
        U, s, Vt = np.linalg.svd(A, full_matrices=False)
        return U, s, B @ Vt.T

    def _predict_all(self, sm, y_a):
        U, s, BV = sm
        ym = float(y_a.mean())
        uty = U.T @ (y_a - ym)                                       # (k,)
        D = s[None, :] / (s[None, :] ** 2 + self.alphas[:, None])    # (n_alpha, k)
        return ym + (D * uty[None, :]) @ BV.T                        # (n_alpha, n_B)

    def __call__(self, y):
        y = np.asarray(y, dtype=float)
        yhat, chosen = np.full(len(y), np.nan), []
        for tr, te, inner, sm_out in self.outer:
            sse = np.zeros(len(self.alphas))
            for itr, iva, sm in inner:
                sse += np.sum((y[iva][None, :] - self._predict_all(sm, y[itr])) ** 2, axis=1)
            k = _pick_alpha_idx(sse)
            yhat[te] = self._predict_all(sm_out, y[tr])[k]
            chosen.append(float(self.alphas[k]))
        return yhat, np.asarray(chosen)


# ── Regression metrics ─────────────────────────────────────────────────────
def oos_baseline(y, protocol):
    """Per row: the TRAINING-fold mean of y in the outer fold that holds the row out."""
    y = np.asarray(y, dtype=float)
    b = np.full(len(y), np.nan)
    for tr, te in cv_splits(protocol, len(y)):
        b[te] = y[tr].mean()
    return b


def oos_r2(y, yhat, protocol=None, baseline=None, idx=None):
    """Out-of-sample R² = 1 − Σ(y−ŷ)² / Σ(y−b)², b = the training-fold mean of each outer
    fold (oos_baseline). Pass `baseline` to override (e.g. 0.0 for centred targets, §8.1)
    or to reuse a precomputed one; `idx` restricts the sums (bootstrap)."""
    y, yhat = np.asarray(y, dtype=float), np.asarray(yhat, dtype=float)
    if baseline is None:
        if protocol is None:
            raise ValueError('oos_r2 needs a protocol (or an explicit baseline)')
        baseline = oos_baseline(y, protocol)
    b = np.broadcast_to(np.asarray(baseline, dtype=float), y.shape)
    i = slice(None) if idx is None else idx
    den = float(np.sum((y[i] - b[i]) ** 2))
    if den <= 1e-12:
        return float('nan')
    return float(1.0 - np.sum((y[i] - yhat[i]) ** 2) / den)


def rmse(y, yhat):
    y, yhat = np.asarray(y, dtype=float), np.asarray(yhat, dtype=float)
    return float(np.sqrt(np.mean((y - yhat) ** 2)))


NEG_LOO_R_NOTE = ('⚠️ negative LOOCV r with weak predictors is a known artefact (the held-out '
                  'point shifts the training mean the opposite way) — read R²')


def warn_negative_loo_r(table, r_cols=('r_valence', 'r_arousal'), protocol_col='protocol'):
    """Print NEG_LOO_R_NOTE once if any r printed under 'loo' in `table` is negative."""
    if table is None or len(table) == 0 or protocol_col not in table:
        return False
    sub  = table[table[protocol_col] == 'loo']
    cols = [c for c in r_cols if c in sub]
    if cols and bool((sub[cols].to_numpy(dtype=float) < 0).any()):
        print('  ' + NEG_LOO_R_NOTE)
        return True
    return False


def within_piece_permutation(groups, rng):
    """Index permutation that shuffles rows ONLY inside each piece (never across pieces):
    y[perm] re-assigns a piece's targets to that piece's own renditions."""
    groups = np.asarray(groups)
    perm = np.arange(len(groups))
    for g in np.unique(groups):
        idx = np.flatnonzero(groups == g)
        perm[idx] = rng.permutation(idx)
    return perm


# ── Chance floors (A1) — DummyClassifier seeded per fold AND per label ────
# The pre-audit code re-created DummyClassifier(random_state=42) in every fold, so every
# fold drew the SAME first random number and 'stratified'/'uniform' predicted the same
# constant (0) everywhere → F1-macro = 0.000. Now: random_state = base_seed +
# 1000·fold + label, repeated over n_repeats base seeds ('most_frequent' is deterministic
# → 1 repeat). Degenerate (single-class) training folds predict that constant, mirroring
# SafeMultiOutputClassifier. DummyClassifier reads random_state at PREDICT time, so each
# (fold, label) dummy is fitted once and re-seeded per repeat — identical predictions to a
# fresh DummyClassifier(random_state=seed).fit(...) per repeat, at a fraction of the cost.
def _fit_dummies(Y, protocol, strategy):
    Y = np.asarray(Y)
    n, L = Y.shape
    Xd = np.zeros((n, 1))
    splits = cv_splits(protocol, n)
    if 1000 * len(splits) + L > DUMMY_SEED_STRIDE:
        raise ValueError('DUMMY_SEED_STRIDE too small for this many folds/labels')
    fitted = []
    for fold, (tr, te) in enumerate(splits):
        for j in range(L):
            col = Y[tr, j]
            est = (int(col[0]) if np.unique(col).size < 2
                   else DummyClassifier(strategy=strategy).fit(Xd[tr], col))
            fitted.append((fold, te, j, est))
    return fitted


def dummy_predict(Y, protocol, strategy, base_seed, fitted=None):
    """One repeat of a Dummy floor: random_state = base_seed + 1000·fold + label."""
    Y = np.asarray(Y)
    fitted = _fit_dummies(Y, protocol, strategy) if fitted is None else fitted
    Yp = np.zeros(Y.shape, dtype=int)
    Xd = np.zeros((Y.shape[0], 1))
    for fold, te, j, est in fitted:
        if isinstance(est, int):
            Yp[te, j] = est
        else:
            est.set_params(random_state=int(base_seed + 1000 * fold + j))
            Yp[te, j] = np.asarray(est.predict(Xd[te])).astype(int)
    return Yp


def _varies_across_folds(Yp, protocol):
    """True if, for at least one label, the per-fold prediction pattern is not identical
    in every outer fold (the property the pre-audit floors lacked)."""
    splits = cv_splits(protocol, Yp.shape[0])
    return any(len({tuple(Yp[te, j]) for _, te in splits}) > 1 for j in range(Yp.shape[1]))


def chance_floors(Y, n_repeats, seed, protocols=PROTOCOLS):
    """→ (per-repeat rows, summary per (protocol, strategy), first-repeat predictions)."""
    Y = np.asarray(Y)
    rows, first = [], {}
    for pr in protocols:
        for s in DUMMY_STRATEGIES:
            fitted = _fit_dummies(Y, pr, s)
            for r in range(1 if s == 'most_frequent' else int(n_repeats)):
                Yp = dummy_predict(Y, pr, s, seed + DUMMY_SEED_STRIDE * r, fitted=fitted)
                if r == 0:
                    first[(pr, s)] = Yp
                rows.append(dict(protocol=pr, strategy=s, repeat=r,
                                 f1_macro=float(f1_score(Y, Yp, average='macro', zero_division=0)),
                                 f1_micro=float(f1_score(Y, Yp, average='micro', zero_division=0)),
                                 hamming=float(hamming_loss(Y, Yp))))
    reps = pd.DataFrame(rows)
    summ = (reps.groupby(['protocol', 'strategy'], sort=False)
                .agg(n_repeats=('repeat', 'size'),
                     f1_macro_mean=('f1_macro', 'mean'),
                     f1_macro_sd=('f1_macro', lambda v: float(np.std(v, ddof=1)) if len(v) > 1 else 0.0),
                     f1_macro_p95=('f1_macro', lambda v: float(np.percentile(v, 95))),
                     f1_micro_mean=('f1_micro', 'mean'),
                     hamming_mean=('hamming', 'mean'))
                .reset_index())
    return reps, summ, first


def floors_from_summary(summ):
    """→ (mean of the strongest-by-mean strategy, MAX P95 across strategies (the strict
    'beats chance' line, D3), name of the strongest strategy) — each keyed by protocol."""
    mean, p95, strongest = {}, {}, {}
    for pr, sub in summ.groupby('protocol', sort=False):
        best = sub.loc[sub['f1_macro_mean'].idxmax()]
        mean[pr], strongest[pr] = float(best['f1_macro_mean']), str(best['strategy'])
        p95[pr] = float(sub['f1_macro_p95'].max())
    return mean, p95, strongest
# ── B-INC · Freedman–Lane permuted targets (reduced model = OLS on Z) ─────────
def freedman_lane_targets(y, Z, perms):
    """Freedman–Lane null targets: fit the REDUCED model y ~ Z by OLS on all rows (lstsq,
    minimum-norm — rank-deficient Z such as a centred one-hot is fine; no intercept is
    added, so centre y and Z first), then y*_b = ŷ_red + e[perm_b] for every permutation.
    Use within-piece permutations (within_piece_permutation) so the exchangeability unit
    stays the piece. → (n_perm × n) array."""
    y = np.asarray(y, dtype=float)
    Z = np.asarray(Z, dtype=float).reshape(len(y), -1)
    fit = Z @ np.linalg.lstsq(Z, y, rcond=None)[0]
    res = y - fit
    return np.vstack([fit + res[np.asarray(p)] for p in perms])


# ── B-NC · Split-half noise ceiling over RATERS (Spearman–Brown) ──────────────
def split_half_ceiling(raters, items, values, item_order, n_splits, seed, item_groups=None):
    """Randomly split the RATERS into two halves `n_splits` times (seeded); per half, the
    mean rating of every item in `item_order`; optionally centre each half's means within
    `item_groups` (pieces); Pearson r between the halves across items; Spearman–Brown
    r_full = 2r / (1 + r). A split that leaves an item without a rating in either half is
    REDRAWN (counted). NaN values are dropped first.
    → {'r_half': (n_splits,), 'r_full': (n_splits,), 'n_redraws': int}."""
    raters, items = np.asarray(raters), np.asarray(items)
    values = np.asarray(values, dtype=float)
    ok = np.isfinite(values)
    raters, items, values = raters[ok], items[ok], values[ok]
    item_order = list(item_order)
    pos = {e: i for i, e in enumerate(item_order)}
    if not set(items) <= set(pos):
        raise ValueError('split_half_ceiling: ratings for items outside item_order')
    ii = np.array([pos[e] for e in items])
    ur, ri = np.unique(raters, return_inverse=True)
    n_items = len(item_order)
    g = None if item_groups is None else np.asarray(item_groups)
    rng = np.random.default_rng(seed)
    r_half, n_redraws = [], 0
    while len(r_half) < n_splits:
        half = np.zeros(len(ur), dtype=bool)
        half[rng.permutation(len(ur))[: len(ur) // 2]] = True
        h = half[ri]
        means = []
        for m in (h, ~h):
            cnt = np.bincount(ii[m], minlength=n_items)
            if (cnt == 0).any():
                means = None
                break
            means.append(np.bincount(ii[m], weights=values[m], minlength=n_items) / cnt)
        if means is None:
            n_redraws += 1
            if n_redraws > 100 * n_splits:
                raise RuntimeError('split_half_ceiling: cannot find splits covering every item')
            continue
        if g is not None:
            means = [mu - pd.Series(mu).groupby(g).transform('mean').to_numpy() for mu in means]
        r_half.append(float(np.corrcoef(means[0], means[1])[0, 1]))
    r_half = np.asarray(r_half)
    return {'r_half': r_half, 'r_full': 2 * r_half / (1 + r_half), 'n_redraws': int(n_redraws)}
# <<< HELPERS:cv_engine


# ── Wiring: piece groups aligned row-for-row with eids / Y / every X_* ─────
assert list(df_expr['excerpt_id']) == list(eids), \
    '§5.1: df_expr row order must equal eids — every X_*, Y and PIECE_GROUPS pair positionally'
PIECE_GROUPS = df_expr['piece'].to_numpy()
assert len(PIECE_GROUPS) == Y.shape[0] == X_A.shape[0] == X_C.shape[0], \
    '§5.1: PIECE_GROUPS / Y / X_* row counts differ'
assert all(get_piece(e) == p for e, p in zip(eids, PIECE_GROUPS)), \
    '§5.1: PIECE_GROUPS disagrees with the piece parsed from the excerpt id'
_pc = pd.Series(PIECE_GROUPS).value_counts()
print(f'§5.1 · PIECE_GROUPS: {len(_pc)} pieces, renditions per piece = {sorted(set(_pc))}')
if (_pc != 3).any():
    print(f'⚠️⚠️  pieces without exactly 3 renditions: {list(_pc[_pc != 3].index)}')
for _pr in PROTOCOLS:
    print(f'   {_pr:<6} → {len(cv_splits(_pr, len(PIECE_GROUPS))):>3} outer folds · {PROTOCOL_LABEL[_pr]}')
print(f'   default classifier : {DEFAULT_CLF_NAME} — nested selection over {len(SELECT_CONFIGS)} configs '
      f'(inner GroupKFold({INNER_SPLITS}) on the training pieces, pooled macro-F1, ties → list order)')
print(f'                        configs: {", ".join(n for n, _ in SELECT_CONFIGS)}')
print(f'   os.cpu_count() = {os.cpu_count()} → NestedSelect outer folds on NMS_N_JOBS = {NMS_N_JOBS} '
      'worker(s) (joblib/loky, threadpool_limits(1) per fold)')
print(f'   nested ridge grid  : {len(RIDGE_ALPHAS)} α in [{RIDGE_ALPHAS[0]:g}, {RIDGE_ALPHAS[-1]:g}], '
      f'inner GroupKFold({INNER_SPLITS}) on the training pieces')

# ── Guard: the manual engine is a FAITHFUL refactor of the old pipeline ────
for _pr in PROTOCOLS:
    _cv, _g = get_cv(_pr)
    _old = cross_val_predict(
        Pipeline([('sc', StandardScaler()), ('clf', SafeMultiOutputClassifier(GaussianNB()))]),
        X_A, Y, cv=_cv, groups=_g, n_jobs=1)
    _new = cv_predict_multilabel(make_nb_clf, X_A, Y, _pr)
    assert np.array_equal(np.asarray(_old, dtype=int), _new), \
        f'§5.1: cv_predict_multilabel(GaussianNB) differs from the old cross_val_predict under {_pr!r}'
print('✅ §5.1 equivalence: cv_predict_multilabel(GaussianNB) == old cross_val_predict('
      'SafeMultiOutput(GaussianNB)) on X_A, element-wise, under both protocols')

# ── Guard (B-NMS): parallel == serial, exactly — on the small CREPE matrix X_B ──
_ser, _ser_i = cv_predict_nested_select(X_B, Y, PRIMARY_PROTOCOL, n_jobs=1, use_memo=False,
                                        verbose=False, return_info=True)
_par, _par_i = cv_predict_nested_select(X_B, Y, PRIMARY_PROTOCOL, n_jobs=max(2, NMS_N_JOBS),
                                        use_memo=False, verbose=False, return_info=True)
assert np.array_equal(_ser, _par) and _ser_i['chosen'] == _par_i['chosen']     and np.array_equal(_ser_i['inner_scores'], _par_i['inner_scores']),     '§5.1: NestedSelect parallel predictions differ from the serial run on X_B'
print(f'✅ §5.1 NestedSelect: parallel ({max(2, NMS_N_JOBS)} workers) == serial on X_B '
      f'({PRIMARY_PROTOCOL}) — predictions, chosen configs and inner scores identical')
print(f'   X_B · {PRIMARY_PROTOCOL}: chosen per outer fold → {nms_counts_str(_ser_i)}')

# ── Guard (block weighting): B = 1 is the unweighted head exactly · the weights keep the total
#    standardised variance P and give every block P/B · BlockScaler is stateless · the
#    closed-form operator reproduces the sklearn path on a WEIGHTED multi-block head
#    (X_C_REG = MERT_REG + CREPE + Essentia + DTW, arousal, PRIMARY protocol) ──────────────
_bw_sizes = [int(MERT_REG.shape[1]), int(X_B.shape[1]), len(ESSENTIA_FORWARD), len(DTW_MODEL_COLS)]
_X_bw = np.asarray(X_C_REG, dtype=float)
assert sum(_bw_sizes) == _X_bw.shape[1], \
    f'§5.1: block sizes {_bw_sizes} do not add up to X_C_REG ({_X_bw.shape[1]} columns)'
_w_bw = block_weights(_bw_sizes)
assert np.isclose(float(np.sum(np.asarray(_bw_sizes) * _w_bw ** 2)), float(sum(_bw_sizes))), \
    '§5.1: block weights change the total standardised variance'
assert np.allclose(np.asarray(_bw_sizes) * _w_bw ** 2, sum(_bw_sizes) / len(_bw_sizes)), \
    '§5.1: block weights do not give every block an equal share'
assert np.array_equal(block_weights([_X_bw.shape[1]]), np.ones(1)), '§5.1: B = 1 must give w = 1'
assert np.array_equal(BlockScaler(_bw_sizes).fit(_X_bw[:10]).weights_,
                      BlockScaler(_bw_sizes).fit(3.0 * _X_bw[::-1]).weights_), \
    '§5.1: BlockScaler weights depend on the data it was fitted on'
_y_bw = df_expr['arousal_mean'].to_numpy(dtype=float)
_h0, _a0 = nested_ridge_predict(_X_bw, _y_bw, PRIMARY_PROTOCOL)
_h1, _a1 = nested_ridge_predict(_X_bw, _y_bw, PRIMARY_PROTOCOL, block_sizes=[_X_bw.shape[1]])
assert np.array_equal(_h0, _h1) and np.array_equal(_a0, _a1), \
    '§5.1: block_sizes with a single block must reproduce the unweighted head exactly'
_hw, _aw = nested_ridge_predict(_X_bw, _y_bw, PRIMARY_PROTOCOL, block_sizes=_bw_sizes)
_hw2, _aw2 = NestedRidgeOperator(_X_bw, PRIMARY_PROTOCOL, block_sizes=_bw_sizes)(_y_bw)
assert np.array_equal(_aw, _aw2) and np.allclose(_hw, _hw2, rtol=0, atol=1e-8), \
    (f'§5.1: weighted NestedRidgeOperator disagrees with nested_ridge_predict on X_C_REG '
     f'(max |Δŷ| = {np.max(np.abs(_hw - _hw2)):.2e})')
print(f'✅ §5.1 block weighting: B = 1 identical to unweighted · Σ p_b·w_b² = P · closed-form operator == '
      f'sklearn path on weighted X_C_REG ({PRIMARY_PROTOCOL}, arousal)')
print('   equal-share weights on X_C_REG: '
      + ' · '.join(f'{n} p={p} w={w:.3f}' for n, p, w in zip(('MERT-full', 'CREPE', 'Essentia', 'DTW'),
                                                            _bw_sizes, _w_bw))
      + f'   (deployed: REG_BLOCK_WEIGHTING = {REG_BLOCK_WEIGHTING!r})')

# ── Chance floors (A1): computed ONCE here, read by §6.3 (ABL_FLOOR) and §5.2 ──
DUMMY_REPS, DUMMY_SUMMARY, DUMMY_FIRST_PRED = chance_floors(Y, N_DUMMY_REPEATS, SEED)
for _pr in PROTOCOLS:
    for _s in ('stratified', 'uniform'):
        assert _varies_across_folds(DUMMY_FIRST_PRED[(_pr, _s)], _pr), \
            (f'§5.1: Dummy {_s!r} predicts the same pattern in every {_pr!r} fold — '
             'the per-fold/per-label seeding is not reaching DummyClassifier')
print(f'✅ Dummy floors re-seeded per fold × label ({N_DUMMY_REPEATS} repeats): stratified / uniform '
      'predictions now vary across folds under both protocols')
CHANCE_FLOOR_MEAN, CHANCE_FLOOR_P95, CHANCE_STRONGEST = floors_from_summary(DUMMY_SUMMARY)
_u = DUMMY_SUMMARY.query('protocol == "loo" and strategy == "uniform"')['f1_macro_mean'].iloc[0]
_s = DUMMY_SUMMARY.query('protocol == "loo" and strategy == "stratified"')['f1_macro_mean'].iloc[0]
# Analytic expectations from the ACTUAL class prevalences π_j of Y (nothing hardcoded): a uniform
# coin per label has precision π, recall ½ → F1 = π / (π + ½); 'stratified' has F1 ≈ π.
_prev = np.asarray(Y, dtype=float).mean(axis=0)
_u_an = float(np.mean(_prev / (_prev + 0.5)))
_s_an = float(np.mean(_prev))
print('   class prevalences (from Y): ' + ' · '.join(f'{c} {int(n)}' for c, n in
                                                   zip(EMOTION_LABELS, np.asarray(Y).sum(axis=0))))
# expected at N = 116: HA_HV 11 · HA_LV 20 · LA_HV 5 · LA_LV 17 · Neutral 7
if not (0.22 <= _u <= 0.32):
    print(f'⚠️  LOOCV uniform floor mean = {_u:.3f}, outside the expected ~[0.22, 0.32] '
          f'(analytic ≈ {_u_an:.3f} for the prevalences above)')
print(f'   LOOCV uniform mean = {_u:.3f} (analytic ≈ {_u_an:.3f}) · stratified mean = {_s:.3f} '
      f'(analytic ≈ {_s_an:.3f})')
for _pr in PROTOCOLS:
    print(f'   {_pr:<6} chance floor: mean {CHANCE_FLOOR_MEAN[_pr]:.4f} ({CHANCE_STRONGEST[_pr]}) · '
          f'strict P95 (max over strategies) {CHANCE_FLOOR_P95[_pr]:.4f}')
print('   (full floor table in §5.2)')

## Part 5 · Evaluation yardstick — §5.2 Dummy floors

The **random / majority floors** — `DummyClassifier` under the same CV, re-seeded per fold and label,
200 repeats (computed once in §5.1) — as a distribution: mean, SD and P95. They give a score a meaningful
zero point, but they ignore the **selection procedure** of the deployed classifier, so they are only the
first half of the yardstick; §5.3 is the second. The comparison of these floors with the default
classifier per feature set (and the Phase A NaiveBayes default) is in §6.6, after the classifiers have run.

**Why F1-macro is the headline metric.** F1-macro averages the five per-class F1 scores with equal
weight, so a total failure on a rare class (LA_HV, Neutral) drags it down just as much as a failure on a
frequent one. F1-micro and subset accuracy pool over samples and are dominated by the frequent labels
(HA_LV, LA_LV) — they can look healthy while the rare classes collapse.


In [ ]:
# ── §5.2 DummyClassifier baseline floors ───────────────────────────
# NEEDS : CHANCE_FLOOR_MEAN, CHANCE_FLOOR_P95, CHANCE_STRONGEST, DUMMY_SUMMARY, N_DUMMY_REPEATS, OUTPUT_DIR,
#          PROTOCOLS, PROTOCOL_LABEL (§5.1)
# MAKES : BASELINES, df_dummy_floors, strongest, strongest_f1, strongest_p95 · metrics_dummy_floors.csv
# FEEDS : §5.3 (printed next to the empirical null), §6.6 (floors vs the default per feature set, the
#          baselines figure), §11.1
# ═══════════════════════════════════════════════════════════════════════════════
# §5.2 · Baseline floors — DummyClassifier under the SAME CV (ignores the features)
# ═══════════════════════════════════════════════════════════════════════════════
# The floors are computed ONCE in §5.1 (shared with §6.3): each Dummy is seeded per fold AND per label and
# the random strategies are repeated N_DUMMY_REPEATS times, so the floor is a distribution (mean, SD, P95),
# not a single draw. The pre-audit version reused one seed in every fold, collapsed every random floor to
# F1-macro = 0 and claimed the headline was "entirely learned signal" — that claim is withdrawn.
# Phase C2: this cell prints the floors only. The comparison with the default classifier per feature set
# (metrics_baselines.csv and its figure) is in §6.6, after the classifiers have run; and every ✓ printed
# from §5.3 on reads the EMPIRICAL null (§5.3), which is the right chance level for the deployed pipeline.
BASELINES = list(DUMMY_SUMMARY['strategy'].unique())
df_dummy_floors = DUMMY_SUMMARY[['protocol', 'strategy', 'n_repeats', 'f1_macro_mean', 'f1_macro_sd',
                                 'f1_macro_p95', 'f1_micro_mean', 'hamming_mean']].copy()
strongest, strongest_f1, strongest_p95 = dict(CHANCE_STRONGEST), dict(CHANCE_FLOOR_MEAN), dict(CHANCE_FLOOR_P95)
for pr in PROTOCOLS:
    print(f'\n📊 DUMMY FLOORS — {PROTOCOL_LABEL[pr]}')
    print(f'   (random floors: mean / SD / P95 over {N_DUMMY_REPEATS} re-seeded repeats; most_frequent is '
          'deterministic → 1 repeat)')
    print(df_dummy_floors[df_dummy_floors.protocol == pr].drop(columns='protocol').round(4).to_string(index=False))
    print(f'   strongest random floor: {strongest[pr]} (mean F1-macro {strongest_f1[pr]:.4f}) · strict P95 '
          f'(max over strategies) {strongest_p95[pr]:.4f}')
print('\n⚠️  The Dummy floors ignore the features AND the selection procedure. For the deployed nested-selection')
print('    classifier they are too LOW (B3 run: 25 noise columns scored piece F1 0.354 > the Dummy P95 0.323) —')
print('    §5.3 builds the empirical null of the deployed pipeline; every ✓ from §5.3 on reads that null.')
df_dummy_floors.to_csv(os.path.join(OUTPUT_DIR, 'metrics_dummy_floors.csv'), index=False)
print('✅ §5.2 — metrics_dummy_floors.csv (floors only; the per-set comparison is metrics_baselines.csv, §6.6)')


## Part 5 · Evaluation yardstick — §5.3 Empirical chance null of the deployed classification pipeline — the draws

**Why.** The Dummy floors (§5.2) ignore the features *and the selection procedure*. The B3 run showed that
this is not an adequate null for the deployed classifier: **25 i.i.d. N(0, 1) columns through NestedSelect
scored piece F1 = 0.354**, above the Dummy strict P95 of 0.323 (uniform mean 0.272), and at the fixed
C = 0.001 of the validation curve (§7.1) the noise reached **0.468**. Mechanism: at a tiny C the balanced
linear SVC is a class-mean-difference (prototype) classifier whose threshold depends on the training-label
composition of each leave-one-piece-out fold, and the labels cluster within pieces — so "predicting" from
noise inherits the fold-wise label balance. Regression is unaffected (noise R² ≈ 0, §7.1). **This section
runs before any classifier is scored (Phase C2)**, so every F1 of Part 6 is read against it the moment it
is printed; the full read-out table is §6.9.

**(A) The map — 100 draws, two widths** (Phase C1, unchanged): each draw is an i.i.d. standard-normal
matrix (60 × d, seed `NULL_SEED` + draw index) through `cv_predict_nested_select` on the same `Y`, the same
§5.1 folds, the same 13 `SELECT_CONFIGS`, inner `GroupKFold(5)` and tie rule; score = the pooled test
F1-macro exactly as the headline. `NULL_N_DRAWS = 100`, `NULL_DIMS = (25, 10)`, `NULL_PROTOCOLS = ('piece',)`,
`RUN_NULL_LOO = False`. Cached in `null_f1_draws.csv` (settings hash). **Phase C3:** the map is still drawn and
cached, because draws 0–99 of the primary d = 10 / 25 reuse its draws, but it is **no longer the reference of
any reported F1** (see "How it is read").

**(B) The exact-width primary null (Phase C2, C2 obligation 1; per-width targets since Phase C3).** A
**pre-declared primary family**: the headline sets **A–D**, **Interpretable (27)** (`INTERP_PRED_COLS`), **DTW**
(reference-free timing) and **CREPE + DTW**. Each gets a null at its **exact column count** (widths 7, 10, 17,
25, 27, 35, 52). **Draw targets (Phase C3, §0.3):** `NULL_N_DRAWS_PRIMARY = {7: 1000, 17: 1000}`, every other
width `NULL_N_DRAWS_PRIMARY_DEFAULT = 600` — the C2 run stopped at its budget with 600 draws per width (561 at
d = 52), and the DTW set passed BH with p = 4/601 = 0.0067 against a BH limit of 0.0071, so one draw decided it;
d = 17 is CREPE + DTW and Essentia + DTW. The targets are **not** in the settings hash. Same seeding scheme (draws 0–99 of a width that is also a map width are the map draws and are
reused), cached in `null_f1_draws_primary.csv`. The empirical p of every primary set is BH-corrected
**within the family** in §6.9. The multiclass model (C2 obligation 4) is **on hold** and not in the family.

**Resuming from the Kaggle input (Phase C3).** Before any cache is read, the cell copies `null_f1_draws.csv`
and `null_f1_draws_primary.csv` from `NULL_CACHE_INPUT_DIR`
(`/kaggle/input/datasets/george031218732003/violin-thesis-null-cache`, the C2 caches) into `OUTPUT_DIR` — the
input is read-only and the cell appends to the cache — unless an `OUTPUT_DIR` copy already exists. It then
prints the rows per (protocol, d) and how many match the current settings hash, with a loud warning if none do
(the cache was invalidated).

**Runtime and resumption.** ≈ 7.3 s per `piece` draw in the C2 run. From the C2 cache the C3 targets need
≈ 839 new draws (d = 7: 400, d = 17: 400, d = 52: 39) ≈ 1.7 h. The cell draws round-robin (50 draws per width
per round, each width up to its own target), checkpoints the CSV after every 50 draws and stops at
`NULL_PRIMARY_BUDGET_MIN = 240` min (a safety cap); a re-run with the CSV kept in `OUTPUT_DIR` resumes where it
stopped. An incomplete null is printed loudly (`NULL_PRIMARY_COMPLETE = False`) and every p carries its `n_draws`.

**How it is read — one rule everywhere (Phase C3).** `emp_null_ref(f1, protocol, d)` is the only reader (every
cell that marks ✓ or reports an empirical-null p calls it; `emp_null_dim` / `emp_null_p` are thin wrappers):
(1) a primary null exists at exactly d → **"exact-width null d = …"**; (2) else the **smallest primary width ≥ d**
→ **"null borrowed from d = … (conservative, next larger width)"** — the null P95 rises with d; (3) else (d > 52,
e.g. MERT-full 768) → **"null borrowed from d = 52 (extrapolated)"**. It returns the p, the null P95, the width,
the number of draws and the label; `emp_null_mark(…)` prints **✓** = F1 above that null's P95, **·** = not
above, **–** = no null drawn (e.g. `loo`). One-sided p = (1 + #draws ≥ observed) / (1 + #draws).


In [ ]:
# ── §5.3 Empirical chance null of the deployed classification pipeline — the draws ──
# NEEDS : CHANCE_FLOOR_MEAN, CHANCE_FLOOR_P95, CHANCE_STRONGEST, cv_predict_nested_select, DEFAULT_CLF_NAME,
#          DTW_MODEL_COLS, f1_score, hashlib, INNER_SPLITS, INTERP_PRED_COLS (§4.1), nms_counts_str,
#          NMS_TIE_TOL, NULL_CACHE_INPUT_DIR, NULL_N_DRAWS_PRIMARY, NULL_N_DRAWS_PRIMARY_DEFAULT (§0.3),
#          OLD_CACHE_NAMES, OUTPUT_DIR, PIECE_GROUPS, PROTOCOLS,
#          SELECT_C_GRID, SELECT_CONFIGS, X_A, X_B, X_C, X_D, Y — read, never rebound
# MAKES : NULL_N_DRAWS, NULL_DIMS, NULL_SEED, NULL_PROTOCOLS, RUN_NULL_LOO, NULL_CACHE, EMP_NULL_F1, df_null_draws
#          (the 100-draw MAP — drawn and cached, its draws seed d = 10 / 25 of the primary null; NOT a reference
#          since Phase C3) · PRIMARY_NULL_FAMILY, PRIMARY_NULL_ALIASES, PRIMARY_NULL_WIDTHS, null_primary_target,
#          NULL_PRIMARY_CACHE, NULL_PRIMARY_BUDGET_MIN, NULL_PRIMARY_CHUNK, df_null_primary, EMP_NULL_PRIMARY,
#          NULL_PRIMARY_COMPLETE, emp_null_ref, emp_null_mark, emp_null_dim, emp_null_p (the exact-width PRIMARY
#          null + the ONE shared reader, Phase C3 rule) · _null_* · null_f1_draws.csv, null_f1_draws_primary.csv
# FEEDS : every classification F1 printed from here on (§6.1 ✓, §6.3 ✓, §6.4 ✓, §6.9 read-out with BH in the
#          primary family, §7.1 / §7.2 read-outs, §10.2 alone_emp_p, §11.1); the CHANCE_FLOOR_* names of §5.1
#          are NOT rebound
# ═══════════════════════════════════════════════════════════════════════════════
# §5.3 · What does the DEPLOYED NestedSelect pipeline score on pure noise? (moved up in Phase C2 so the null
# exists before the first F1 is printed; the read-out of every reported F1 is §6.9)
# ═══════════════════════════════════════════════════════════════════════════════
# Every draw goes through cv_predict_nested_select exactly as ablate() calls it (§5.1 folds, SELECT_CONFIGS,
# inner GroupKFold, tie rule, StandardScaler inside every fold); only the feature matrix is noise.
import time as _null_time


def _null_f1(Yt, Yp):
    """Pooled macro-F1 exactly as the headline (== §6.3 _sc_f1, which is defined later)."""
    return float(f1_score(Yt, Yp, average='macro', zero_division=0))


_null_t0 = _null_time.time()
# ══ (0) Phase C3 · resume from the Kaggle input (no redraws) ══════════════════════════════════════════════
# Kaggle inputs are read-only and this cell APPENDS to its caches, so the C2 caches are COPIED into OUTPUT_DIR
# (never read in place) before any cache is read; an OUTPUT_DIR copy that already exists is never overwritten.
import shutil as _null_shutil
for _null_fn in ('null_f1_draws.csv', 'null_f1_draws_primary.csv'):
    _null_src, _null_dst = os.path.join(NULL_CACHE_INPUT_DIR, _null_fn), os.path.join(OUTPUT_DIR, _null_fn)
    if os.path.exists(_null_dst):
        print(f'📂 {_null_fn}: already in OUTPUT_DIR — kept (the input copy is not used)')
    elif os.path.exists(_null_src):
        _null_shutil.copy2(_null_src, _null_dst)
        print(f'📥 {_null_fn}: copied {NULL_CACHE_INPUT_DIR} → OUTPUT_DIR')
    else:
        print(f'ℹ️  {_null_fn}: neither in {NULL_CACHE_INPUT_DIR} nor in OUTPUT_DIR — it will be drawn from scratch')

# ══ (A) the 100-draw MAP — two widths; drawn and cached because draws 0–99 of the primary d = 10 / 25 REUSE
#    them, but since Phase C3 it is no longer the reference of any reported F1 (rule C, emp_null_ref below) ══
NULL_N_DRAWS   = 100
NULL_DIMS      = (25, 10)          # 25 = MERT-PCA25 / interpretable layer · 10 = CREPE / Essentia (DTW 7 → 10)
NULL_SEED      = 0                 # draw i uses np.random.default_rng(NULL_SEED + i)
NULL_PROTOCOLS = ('piece',)
RUN_NULL_LOO   = False             # True → also draw under 'loo' (≈ 3× the cost per dimension)
NULL_CACHE     = os.path.join(OUTPUT_DIR, 'null_f1_draws.csv')
assert os.path.basename(NULL_CACHE) not in OLD_CACHE_NAMES, '§5.3: retired cache name'
_null_prs = tuple(NULL_PROTOCOLS) + tuple(p for p in ('loo',) if RUN_NULL_LOO and p not in NULL_PROTOCOLS)
assert _null_prs and set(_null_prs) <= set(PROTOCOLS), f'§5.3: protocols must be a subset of {PROTOCOLS}'
_null_n = int(np.shape(Y)[0])
_null_Yi = np.asarray(Y).astype(int)
_null_cmin = [n for n, f in SELECT_CONFIGS if np.isclose(float(f.keywords.get('C', np.nan)), float(np.min(SELECT_C_GRID)))]


def _null_X(d, i):
    """Draw i of dimension d: an i.i.d. standard-normal (n × d) matrix, seed NULL_SEED + i."""
    return np.random.default_rng(NULL_SEED + int(i)).standard_normal((_null_n, int(d)))


def _null_sha(a):
    a = np.ascontiguousarray(a)
    return hashlib.sha1(repr((a.shape, a.dtype.str)).encode() + a.tobytes()).hexdigest()


def _null_draw_row(pr, d, i, sig):
    """One null draw through the deployed NestedSelect → one row of the draws table."""
    Yp, info = cv_predict_nested_select(_null_X(d, i), _null_Yi, pr, use_memo=False, verbose=False,
                                        return_info=True)
    return dict(protocol=pr, d=int(d), draw=int(i), seed=int(NULL_SEED + i), f1_macro=_null_f1(_null_Yi, Yp),
                n_folds=len(info['chosen']), n_folds_smallest_C=int(sum(c in _null_cmin for c in info['chosen'])),
                configs=nms_counts_str(info), settings_sha1=sig)


_null_settings = dict(n_draws=int(NULL_N_DRAWS), dims=[int(d) for d in NULL_DIMS], seed=int(NULL_SEED),
                      protocols=list(_null_prs), n=_null_n, y_sha1=_null_sha(_null_Yi),
                      groups_sha1=_null_sha(np.asarray(PIECE_GROUPS).astype(str)),
                      configs=[(n, repr(f)) for n, f in SELECT_CONFIGS], inner_splits=int(INNER_SPLITS),
                      tie_tol=float(NMS_TIE_TOL), generator='numpy default_rng(seed + draw).standard_normal')
_null_sig = hashlib.sha1(repr(sorted(_null_settings.items())).encode()).hexdigest()
_null_expected = {(p, int(d), int(i)) for p in _null_prs for d in NULL_DIMS for i in range(NULL_N_DRAWS)}
# The PRIMARY-null settings hash (moved up from (B) in Phase C3, unchanged, so that both caches are checked before
# anything is read or drawn). n_draws = 1000 is the value the C2 cache rows were hashed with and is FROZEN here:
# since Phase C3 the per-width draw targets (§0.3 NULL_N_DRAWS_PRIMARY / _DEFAULT) do NOT enter the hash, so a
# change of a target can never invalidate the cached draws (each row is keyed by protocol, d and draw).
_NULL_PRIMARY_HASH_N_DRAWS = 1000
_nps = dict(n_draws=int(_NULL_PRIMARY_HASH_N_DRAWS), seed=int(NULL_SEED), protocols=list(NULL_PROTOCOLS), n=_null_n,
            y_sha1=_null_settings['y_sha1'], groups_sha1=_null_settings['groups_sha1'],
            configs=_null_settings['configs'], inner_splits=int(INNER_SPLITS), tie_tol=float(NMS_TIE_TOL),
            generator=_null_settings['generator'])
_np_sig = hashlib.sha1(repr(sorted(_nps.items())).encode()).hexdigest()   # widths NOT in the hash: rows are keyed

# Phase C3 · cache check: rows per (protocol, d) and how many match the CURRENT settings hash
print('🔑 §5.3 cache check — rows per (protocol, d) · rows whose settings_sha1 matches the current settings')
for _null_fn, _null_sg in (('null_f1_draws.csv', _null_sig), ('null_f1_draws_primary.csv', _np_sig)):
    _null_fp = os.path.join(OUTPUT_DIR, _null_fn)
    if not os.path.exists(_null_fp):
        print(f'   {_null_fn}: not present')
        continue
    _null_cc = pd.read_csv(_null_fp)
    _null_cc['_match'] = (_null_cc['settings_sha1'] == _null_sg) if 'settings_sha1' in _null_cc.columns else False
    _null_gc = _null_cc.groupby(['protocol', 'd'])['_match'].agg(['size', 'sum'])
    print(f'   {_null_fn}: {len(_null_cc)} rows, {int(_null_cc["_match"].sum())} match · '
          + ', '.join(f'{_p} d={int(_d)}: {int(_g["sum"])}/{int(_g["size"])}' for (_p, _d), _g in _null_gc.iterrows()))
    if len(_null_cc) and int(_null_cc['_match'].sum()) == 0:
        print(f'   {"⚠️" * 10}\n   ⚠️⚠️  {_null_fn}: 0 of {len(_null_cc)} rows match the current settings hash — THE CACHE WAS '
              f'INVALIDATED (Y, piece groups, SELECT_CONFIGS, inner splits, tie rule, seed, protocols or generator '
              f'changed). Every draw would be recomputed. Stop the run and check before spending the GPU time.\n'
              f'   {"⚠️" * 10}')

_null_loaded = False
if os.path.exists(NULL_CACHE):
    _null_nc = pd.read_csv(NULL_CACHE)
    _null_loaded = ('settings_sha1' in _null_nc.columns and set(_null_nc['settings_sha1']) == {_null_sig}
                    and len(_null_nc) == len(_null_expected)
                    and set(zip(_null_nc['protocol'], _null_nc['d'].astype(int), _null_nc['draw'].astype(int))) == _null_expected)
    print(f'📂 {os.path.basename(NULL_CACHE)} found — settings {"MATCH → loaded (no refit)" if _null_loaded else "DIFFER → recomputing"}')
if _null_loaded:
    df_null_draws = _null_nc
else:
    print(f'🔄 Empirical null (map) — {NULL_N_DRAWS} draws × d ∈ {list(NULL_DIMS)} × {list(_null_prs)} through '
          f'{DEFAULT_CLF_NAME} (≈ 8 s per piece draw on 4 CPUs in the C1 run)...')
    _null_nrows = []
    for _null_pr in _null_prs:
        for _null_d in NULL_DIMS:
            _null_t = _null_time.time()
            for _null_i in range(NULL_N_DRAWS):
                _null_nrows.append(_null_draw_row(_null_pr, _null_d, _null_i, _null_sig))
            print(f'   {_null_pr:<5} d={_null_d:<3} {NULL_N_DRAWS} draws in {(_null_time.time() - _null_t) / 60:.1f} min')
    df_null_draws = pd.DataFrame(_null_nrows)
    df_null_draws.to_csv(NULL_CACHE, index=False)
    print(f'✅ Saved: {NULL_CACHE}')
# (the guard "draw 0 at d = 25 IS the §7.1 random control" runs in §7.1, where that control is built)

EMP_NULL_F1 = {}
for (_null_pr, _null_d), _null_sub in df_null_draws.groupby(['protocol', 'd']):
    _null_v = _null_sub.sort_values('draw')['f1_macro'].to_numpy(dtype=float)
    EMP_NULL_F1[(str(_null_pr), int(_null_d))] = {'mean': float(_null_v.mean()), 'p95': float(np.percentile(_null_v, 95)),
                                        'p99': float(np.percentile(_null_v, 99)), 'draws': _null_v}
# (Phase C3: the map helpers emp_null_dim / emp_null_p are redefined in (B) on the ONE rule of emp_null_ref)


# ══ (B) the EXACT-WIDTH PRIMARY null (Phase C2, C2 obligation 1) ══════════════════════════════════════
# PRE-DECLARED primary family (fixed before any C2 result is seen): the four headline sets A–D, the
# predictive interpretable layer, DTW (reference-free timing) and CREPE + DTW. Each gets a null at its
# EXACT column count, null_primary_target(d) draws (Phase C3: 1000 at d = 7 / 17, 600 elsewhere — §0.3), same
# seeding scheme as the map (draw i = seed NULL_SEED + i, so draws 0..99 of a width that is also in NULL_DIMS are
# byte-identical to the map draws and are reused).
# The multiclass model of C2 obligation 4 is ON HOLD and therefore not in the family. BH within the
# family is applied in §6.9, once the observed F1 exist.
PRIMARY_NULL_FAMILY = [   # (name, source of the observed F1, key in that source, exact width)
    ('A: MERT',                         '§6.1 headline', 'A: MERT',                 int(np.shape(X_A)[1])),
    ('B: CREPE',                        '§6.1 headline', 'B: CREPE',                int(np.shape(X_B)[1])),
    ('C: Combined',                     '§6.1 headline', 'C: Combined',             int(np.shape(X_C)[1])),
    ('D: MERT+Essentia',                '§6.1 headline', 'D: MERT+Essentia',        int(np.shape(X_D)[1])),
    (f'Interpretable ({len(INTERP_PRED_COLS)})', '§6.3 lattice', 'CREPE + Essentia + DTW', len(INTERP_PRED_COLS)),
    ('DTW',                             '§6.3 lattice', 'DTW',                      len(DTW_MODEL_COLS)),
    ('CREPE + DTW',                     '§6.3 lattice', 'CREPE + DTW',              int(np.shape(X_B)[1]) + len(DTW_MODEL_COLS)),
]
PRIMARY_NULL_WIDTHS = sorted({w for *_, w in PRIMARY_NULL_FAMILY})


def null_primary_target(w):
    """Phase C3: the declared number of primary-null draws at width w (§0.3 per-width targets; NOT in the hash)."""
    return int(NULL_N_DRAWS_PRIMARY.get(int(w), NULL_N_DRAWS_PRIMARY_DEFAULT))


# The same matrices under their §6.3 lattice names (identical width and content). Since Phase C3 the reference
# rule is width-based (emp_null_ref), so the aliases are kept for the record only — the rule does not need them.
PRIMARY_NULL_ALIASES = {'MERT': 'A: MERT', 'CREPE': 'B: CREPE', 'MERT + CREPE + Essentia + DTW': 'C: Combined',
                        'MERT + Essentia': 'D: MERT+Essentia',
                        'CREPE + Essentia + DTW': f'Interpretable ({len(INTERP_PRED_COLS)})'}
NULL_PRIMARY_CACHE  = os.path.join(OUTPUT_DIR, 'null_f1_draws_primary.csv')
NULL_PRIMARY_CHUNK  = 50       # draws per width per round (round-robin → balanced widths if the budget stops it)
NULL_PRIMARY_BUDGET_MIN = 240  # wall-clock budget of THIS cell's new draws (min); None = no limit. When it is
                               # reached the draws so far are saved and the next run RESUMES from the cache.
                               # (Phase C3: a safety cap — the C3 targets need ≈ 839 new draws ≈ 1.7 h)
assert os.path.basename(NULL_PRIMARY_CACHE) not in OLD_CACHE_NAMES, '§5.3: retired cache name'
# (the settings hash _nps / _np_sig is built above, before the cache check — Phase C3)

_np_rows = {}
if os.path.exists(NULL_PRIMARY_CACHE):
    _np_c = pd.read_csv(NULL_PRIMARY_CACHE)
    _np_ok = _np_c[_np_c['settings_sha1'] == _np_sig] if 'settings_sha1' in _np_c.columns else _np_c.iloc[0:0]
    _np_rows = {(r['protocol'], int(r['d']), int(r['draw'])): dict(r) for r in _np_ok.to_dict('records')}
    print(f'📂 {os.path.basename(NULL_PRIMARY_CACHE)} found — {len(_np_rows)} rows with matching settings reused '
          f'({len(_np_c) - len(_np_ok)} stale rows ignored)')
# reuse the map draws (same seed scheme → identical matrices → identical NestedSelect results)
_np_reused = 0
for _r in df_null_draws.to_dict('records'):
    _k = (_r['protocol'], int(_r['d']), int(_r['draw']))
    if _r['protocol'] in NULL_PROTOCOLS and int(_r['d']) in PRIMARY_NULL_WIDTHS and _k not in _np_rows \
            and int(_r['draw']) < null_primary_target(_r['d']):
        _np_rows[_k] = dict(_r, settings_sha1=_np_sig, source='map (reused)')
        _np_reused += 1
# round-robin in chunks of NULL_PRIMARY_CHUNK draws over the widths, each width up to its own target
_np_todo = [(p, w, i) for i0 in range(0, max(null_primary_target(w) for w in PRIMARY_NULL_WIDTHS), NULL_PRIMARY_CHUNK)
            for p in NULL_PROTOCOLS for w in PRIMARY_NULL_WIDTHS
            for i in range(i0, min(i0 + NULL_PRIMARY_CHUNK, null_primary_target(w)))
            if (p, w, i) not in _np_rows]
print(f'🔄 Exact-width PRIMARY null — {list(NULL_PROTOCOLS)} · target draws per width: '
      + ', '.join(f'd={w}: {null_primary_target(w)}' for w in PRIMARY_NULL_WIDTHS)
      + f'\n   {len(_np_rows)} available ({_np_reused} reused from the map), {len(_np_todo)} to draw ('
      + ', '.join(f'd={w}: {sum(t[1] == w for t in _np_todo)}' for w in PRIMARY_NULL_WIDTHS)
      + f') ≈ {len(_np_todo) * 7.3 / 3600:.1f} h at ≈ 7.3 s per piece draw (C2 run); budget '
      f'{"none" if NULL_PRIMARY_BUDGET_MIN is None else f"{NULL_PRIMARY_BUDGET_MIN} min"}')


def _np_save():
    pd.DataFrame(list(_np_rows.values())).sort_values(['protocol', 'd', 'draw']).to_csv(NULL_PRIMARY_CACHE, index=False)


_np_t = _null_time.time()
_np_stopped = False
for _j, (_p, _w, _i) in enumerate(_np_todo):
    if NULL_PRIMARY_BUDGET_MIN is not None and (_null_time.time() - _np_t) / 60 > NULL_PRIMARY_BUDGET_MIN:
        _np_stopped = True
        break
    _np_rows[(_p, _w, _i)] = dict(_null_draw_row(_p, _w, _i, _np_sig), source='primary')
    if (_j + 1) % NULL_PRIMARY_CHUNK == 0:
        _np_save()
        print(f'   … {_j + 1}/{len(_np_todo)} new draws ({(_null_time.time() - _np_t) / 60:.0f} min) — checkpoint saved')
if _np_todo:
    _np_save()
    print(f'✅ Saved: {NULL_PRIMARY_CACHE}')
df_null_primary = pd.DataFrame(list(_np_rows.values())).sort_values(['protocol', 'd', 'draw']).reset_index(drop=True)

EMP_NULL_PRIMARY = {}
for (_p, _w), _sub in df_null_primary.groupby(['protocol', 'd']):
    if int(_w) not in PRIMARY_NULL_WIDTHS:
        continue
    _sub = _sub[_sub['draw'].astype(int) < null_primary_target(_w)]      # never more than the declared draws
    _v = _sub.sort_values('draw')['f1_macro'].to_numpy(dtype=float)
    EMP_NULL_PRIMARY[(str(_p), int(_w))] = {'mean': float(_v.mean()), 'p95': float(np.percentile(_v, 95)),
                                            'p99': float(np.percentile(_v, 99)), 'draws': _v, 'n_draws': int(len(_v)),
                                            'target': null_primary_target(_w)}
NULL_PRIMARY_COMPLETE = all(EMP_NULL_PRIMARY.get((p, w), {}).get('n_draws', 0) >= null_primary_target(w)
                            for p in NULL_PROTOCOLS for w in PRIMARY_NULL_WIDTHS)
print('   final draws per width: ' + ', '.join(
    f'd={w}: {EMP_NULL_PRIMARY.get((p, w), {}).get("n_draws", 0)}/{null_primary_target(w)}'
    for p in NULL_PROTOCOLS for w in PRIMARY_NULL_WIDTHS))
if not NULL_PRIMARY_COMPLETE:
    print(f'⚠️⚠️  PRIMARY NULL INCOMPLETE{" (time budget reached)" if _np_stopped else ""}: '
          + ', '.join(f'd={w}: {EMP_NULL_PRIMARY.get((p, w), {}).get("n_draws", 0)}/{null_primary_target(w)}'
                      for p in NULL_PROTOCOLS for w in PRIMARY_NULL_WIDTHS)
          + f'\n      Every primary p below is computed on the draws AVAILABLE (n_draws is printed with it). Re-run '
            f'this cell with {os.path.basename(NULL_PRIMARY_CACHE)} kept in OUTPUT_DIR — it resumes where it stopped.')


def emp_null_dim(n_features):
    """Phase C3 (rule C): the PRIMARY-null width an observed block width d is read against —
    (1) d itself if it is a primary width · (2) else the smallest primary width ≥ d (the null P95 rises with d,
    so the next larger width is the conservative choice) · (3) else (d > max) the largest primary width."""
    n, ws = int(n_features), [int(w) for w in PRIMARY_NULL_WIDTHS]
    if n in ws:
        return n
    up = [w for w in ws if w >= n]
    return min(up) if up else max(ws)


def emp_null_ref(f1, protocol, n_features, set_name=None):
    """The empirical null an observed F1 is read against → dict(p, p95, null_d, n_draws, kind, label).
    Phase C3 — ONE rule for every cell, by width only (set_name is accepted for call compatibility and ignored):
      1. a primary null exists at exactly d            → kind 'exact',        'exact-width null d = …'
      2. else the smallest primary width ≥ d           → kind 'borrowed',     'null borrowed from d = … (conservative,
                                                                               next larger width)'
      3. else (d above the widest, e.g. MERT-full 768) → kind 'extrapolated', 'null borrowed from d = 52 (extrapolated)'
    The 100-draw map (EMP_NULL_F1) is never used as a reference. p = (1 + #draws ≥ f1) / (1 + #draws); NaN where
    the null was not drawn (e.g. 'loo' with RUN_NULL_LOO = False)."""
    nan = float('nan')
    n = int(n_features)
    d = emp_null_dim(n)
    e = EMP_NULL_PRIMARY.get((protocol, d))
    if e is None:
        return dict(p=nan, p95=nan, null_d=d, n_draws=0, kind='not drawn', label=f'no null under {protocol!r}')
    if d == n:
        kind, label = 'exact', f'exact-width null d = {d} ({e["n_draws"]} draws)'
    elif d > n:
        kind, label = 'borrowed', (f'null borrowed from d = {d} (conservative, next larger width; observed d = {n}; '
                                   f'{e["n_draws"]} draws)')
    else:
        kind, label = 'extrapolated', (f'null borrowed from d = {d} (extrapolated; observed d = {n}; '
                                       f'{e["n_draws"]} draws)')
    v = e['draws']
    p = float((1 + np.sum(v >= float(f1) - 1e-12)) / (1 + len(v))) if np.isfinite(f1) else nan
    return dict(p=p, p95=float(e['p95']), null_d=int(d), n_draws=int(len(v)), kind=kind, label=label)


def emp_null_p(f1, protocol, n_features):
    """The empirical p of emp_null_ref (Phase C3 rule) — kept as a short name for the callers."""
    return emp_null_ref(f1, protocol, n_features)['p']


def emp_null_mark(f1, protocol, n_features, set_name=None):
    """'✓' = F1 above the P95 of its empirical null (§5.3) · '·' = not above · '–' = no null drawn."""
    r = emp_null_ref(f1, protocol, n_features, set_name)
    if not np.isfinite(r['p95']) or not np.isfinite(f1):
        return '–'
    return '✓' if f1 > r['p95'] else '·'


print(f'\n📊 §5.3 · EMPIRICAL NULL of {DEFAULT_CLF_NAME} (noise features) next to the Dummy floors (§5.1 / §5.2)')
_null_tab = []
for (_kind, _src) in (('map', EMP_NULL_F1), ('primary (exact width)', EMP_NULL_PRIMARY)):
    for (_null_pr, _null_d), _null_e in sorted(_src.items()):
        _dfs = df_null_draws if _kind == 'map' else df_null_primary
        _null_s = _dfs[(_dfs.protocol == _null_pr) & (_dfs.d == _null_d)]
        _null_tab.append(dict(null=_kind, protocol=_null_pr, d=_null_d, n_draws=len(_null_e['draws']),
                              dummy_mean=CHANCE_FLOOR_MEAN[_null_pr], dummy_strongest=CHANCE_STRONGEST[_null_pr],
                              dummy_strict_p95=CHANCE_FLOOR_P95[_null_pr], emp_mean=_null_e['mean'],
                              emp_p95=_null_e['p95'], emp_p99=_null_e['p99'], emp_max=float(_null_e['draws'].max()),
                              share_above_dummy_p95=float(np.mean(_null_e['draws'] > CHANCE_FLOOR_P95[_null_pr])),
                              mean_folds_smallest_C=float(_null_s['n_folds_smallest_C'].mean()),
                              n_folds=int(_null_s['n_folds'].iloc[0])))
_null_tab = pd.DataFrame(_null_tab)
print(_null_tab.round(4).to_string(index=False))
print(f'   share_above_dummy_p95 = share of noise draws whose pooled F1 exceeds the Dummy strict P95 · '
      f'mean_folds_smallest_C = outer folds (of n_folds) whose NestedSelect choice was C = {np.min(SELECT_C_GRID):g}')
print('   PRE-DECLARED primary family (exact-width null; BH within the family in §6.9): '
      + '; '.join(f'{nm} (d = {w}, from {src})' for nm, src, _, w in PRIMARY_NULL_FAMILY))
print(f'   ONE reference rule for every reported F1 (Phase C3, emp_null_ref): (1) the primary null at exactly its '
      f'width d ("exact-width null d = …") · (2) else the smallest primary width ≥ d ("null borrowed from d = … '
      f'(conservative, next larger width)") · (3) else d > {max(PRIMARY_NULL_WIDTHS)} → d = {max(PRIMARY_NULL_WIDTHS)} '
      f'("… (extrapolated)"). Primary widths: {PRIMARY_NULL_WIDTHS}.')
print('   The "map" rows above are still drawn and cached (draws 0–99 of the primary d = 10 / 25 reuse them) but are '
      'NOT a reference for any result.')
print('   From here on every printed classification ✓ = F1 above the P95 of that empirical null (– = no null drawn).')
print(f'\n⏱  §5.3 runtime {(_null_time.time() - _null_t0) / 60:.1f} min · ✅ §5.3 — null_f1_draws.csv, '
      'null_f1_draws_primary.csv (EMP_NULL_F1, EMP_NULL_PRIMARY, emp_null_ref / emp_null_mark)')


## Part 6 · Prediction — §6.1 Multi-label classification — sensitivity table (both protocols)
**Question of this part:** can the acoustic features predict the audience response on **unseen pieces**,  
read against the yardstick of Part 5?
**Hands to the next part:** `all_recs` / `DEFAULT_RECS`, `REG_RES`, the lattice (`ABL_ROWS`, `ABL_PRED`),
`PRED_QUADS` and the null read-out `df_emp_null_readout` — Part 7 asks *why* MERT behaves as it does here.

Targets are the **5 circumplex classes** (`HA_HV` / `HA_LV` / `LA_HV` / `LA_LV` / `Neutral`).

**This is a sensitivity table, not a selection.** Eight classifiers × four feature sets × both
protocols show how much the numbers depend on the learner. Every downstream single-classifier
number uses the default — **nested model selection** (`DEFAULT_CLF_NAME` = `NestedSelect`, §5.1,
the ★default row) — never the best fixed row here. NaiveBayes, the Phase A pre-declared default,
stays in the table (it fell below chance under `piece` in the 2026-10-09 run). Class balancing (A5): SVM-Linear / SVM-RBF / LogReg use `class_weight='balanced'` with `C`
tuned **per label inside each training fold** by a piece-grouped inner CV; RandomForest / ExtraTrees
are balanced (untuned); XGBoost gets `scale_pos_weight = n_neg/n_pos` from the training fold; KNN and
NaiveBayes are unchanged.

**How to read the four sets (not a leaderboard).** A (MERT-PCA25, the opaque reference) and B (CREPE, named
pitch techniques) are two tool categories, C = every tool, D = MERT + Essentia. The comparisons feed the
thesis questions — how much of the audience signal the interpretable route recovers (cost of
explainability, §10.1) and where the tools complement each other (§6.3 lattice, §7.5) — under the
PRIMARY leave-one-piece-out protocol. Which set "wins" is not itself a finding at N = 60.

**Chance (Phase C2).** Only the deployed **NestedSelect** rows carry a ✓ / · mark: ✓ = F1 above the P95 of
the **empirical null of the deployed pipeline** (§5.3 — the exact-width primary null for A–D). The null is
built for the deployed nested-selection pipeline only, so the eight fixed classifiers are shown without a
mark. `A: MERT` means MERT-PCA25 throughout Part 6 (classification); the regression heads use MERT-full 768.

In [ ]:
# ── §6.1 Multi-label classification — sensitivity table (both protocols) ──
# NEEDS : cv_predict_default, cv_predict_multilabel, DEFAULT_CLF_NAME, emp_null_mark, emp_null_ref (§5.3),
#          make_nb_clf, NB_CLF_NAME, nms_counts_str, OUTPUT_DIR, PRIMARY_PROTOCOL, PROTOCOLS, PROTOCOL_LABEL,
#          SEED, X_A, X_B, X_C, X_D, Y
# MAKES : all_recs, BalancedXGB, DEFAULT_RECS, DEFAULT_SET, df_inner_skips, FEATURE_SETS,
#          make_clfs, run_set, TUNE_GRID_C · ablation_results.csv (the A–D × classifier table)
# FEEDS : §6.4 (TUNE_GRID_C), §6.6, §6.7, §6.9, §11.1
# SafeMultiOutputClassifier now lives in §5.1 (the engine's equivalence guard needs it);
# cv_predict_multilabel reproduces its single-class-fold semantics exactly.
# B-NMS: the DEFAULT is the nested selection (§5.1, ★default row). The 8 fixed classifiers
# below remain a SENSITIVITY table — NaiveBayes in it is the Phase A pre-declared default,
# which failed under 'piece' (kept visible, never re-picked).

TUNE_GRID_C = {'C': np.logspace(-3, 2, 6)}     # ascending = strongest regularisation first (wins ties)


class BalancedXGB:
    """XGBClassifier with scale_pos_weight = n_neg / n_pos computed from the y it is FITTED on —
    i.e. per label on each training fold, never on the test rows."""
    def __init__(self, random_state=42):
        self.random_state = random_state

    def fit(self, X, y):
        y = np.asarray(y).astype(int)
        n_pos = int(y.sum())
        self.scale_pos_weight_ = (len(y) - n_pos) / max(n_pos, 1)
        self.model_ = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, subsample=0.8,
                                    colsample_bytree=0.8, eval_metric='logloss',
                                    random_state=self.random_state, n_jobs=-1,
                                    scale_pos_weight=self.scale_pos_weight_)
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        return np.asarray(self.model_.predict(X)).astype(int)


def make_clfs():
    """name → (factory, tune_grid). Factories return a FRESH unfitted estimator. NOT used for
    selection — the default classifier is the nested selection of §5.1."""
    return {
        'SVM-Linear'    : (lambda: SVC(kernel='linear', class_weight='balanced'), TUNE_GRID_C),
        'SVM-RBF'       : (lambda: SVC(kernel='rbf', class_weight='balanced'), TUNE_GRID_C),
        'LogReg'        : (lambda: LogisticRegression(max_iter=2000, class_weight='balanced'), TUNE_GRID_C),
        'RandomForest'  : (lambda: RandomForestClassifier(n_estimators=200, max_depth=6, class_weight='balanced',
                                                          random_state=SEED, n_jobs=-1), None),
        'ExtraTrees'    : (lambda: ExtraTreesClassifier(n_estimators=200, max_depth=6, class_weight='balanced',
                                                        random_state=SEED, n_jobs=-1), None),
        'KNN'           : (lambda: KNeighborsClassifier(n_neighbors=5, metric='cosine'), None),
        NB_CLF_NAME     : (make_nb_clf, None),          # Phase A pre-declared default (failed under piece)
        'XGBoost'       : (lambda: BalancedXGB(random_state=SEED), None),
    }


def run_set(X_feat, Y_tgt, set_name):
    recs = []
    print(f'\n── {set_name}  X={X_feat.shape} ──')
    for name, (factory, grid) in make_clfs().items():
        line = []
        for pr in PROTOCOLS:
            Yp, info = cv_predict_multilabel(factory, X_feat, Y_tgt, pr, tune_grid=grid,
                                             return_info=True)
            hl  = hamming_loss(Y_tgt, Yp)
            f1m = f1_score(Y_tgt, Yp, average='macro', zero_division=0)
            recs.append({'feature_set': set_name, 'classifier': name, 'protocol': pr,
                         'default': name == DEFAULT_CLF_NAME, 'tuned': grid is not None,
                         'hamming_loss': hl, 'f1_macro': f1m,
                         'f1_micro': f1_score(Y_tgt, Yp, average='micro', zero_division=0),
                         'subset_acc': (Y_tgt == Yp).all(axis=1).mean(), 'Y_pred': Yp,
                         'n_tuned': info['n_tuned'], 'n_tune_skipped': info['n_skipped']})
            line.append(f'{pr}: F1m={f1m:.3f} HL={hl:.3f}')
        skip = (f'   [inner-CV skipped {info["n_skipped"]}/{info["n_tuned"] + info["n_skipped"]} '
                f'label×fold under loo]' if grid is not None else '')
        print(f'  {name:<22} ' + '  |  '.join(line) + skip)
    # ── the DEFAULT: nested selection over SELECT_CONFIGS (choice made inside each outer fold) ──
    line = []
    for pr in PROTOCOLS:
        Yp, info = cv_predict_default(X_feat, Y_tgt, pr, label=set_name, return_info=True)
        hl  = hamming_loss(Y_tgt, Yp)
        f1m = f1_score(Y_tgt, Yp, average='macro', zero_division=0)
        _en = emp_null_ref(f1m, pr, X_feat.shape[1], set_name)        # §5.3 — exact width for A–D
        recs.append({'feature_set': set_name, 'classifier': DEFAULT_CLF_NAME, 'protocol': pr,
                     'default': True, 'tuned': False, 'hamming_loss': hl, 'f1_macro': f1m,
                     'emp_null_p': _en['p'], 'emp_null_p95': _en['p95'], 'emp_null_label': _en['label'],
                     'f1_micro': f1_score(Y_tgt, Yp, average='micro', zero_division=0),
                     'subset_acc': (Y_tgt == Yp).all(axis=1).mean(), 'Y_pred': Yp,
                     'n_tuned': 0, 'n_tune_skipped': 0, 'selected': nms_counts_str(info),
                     'chosen': info['chosen']})
        line.append(f'{pr}: F1m={f1m:.3f}{emp_null_mark(f1m, pr, X_feat.shape[1], set_name)} HL={hl:.3f}')
    print(f'  {DEFAULT_CLF_NAME + " ★default":<22} ' + '  |  '.join(line))
    return recs


print('✓ / · on the NestedSelect rows = F1 above / not above the P95 of the empirical null of the DEPLOYED '
      'pipeline (§5.3, exact width for A–D; – = no null drawn under that protocol). The null is defined for the '
      'deployed nested selection only — the 8 fixed classifiers carry no mark.')
all_recs  = run_set(X_A, Y, 'A: MERT')
all_recs += run_set(X_B, Y, 'B: CREPE')
all_recs += run_set(X_C, Y, 'C: Combined')
all_recs += run_set(X_D, Y, 'D: MERT+Essentia')      # isolates Essentia's contribution
FEATURE_SETS = ['A: MERT', 'B: CREPE', 'C: Combined', 'D: MERT+Essentia']

# The default classifier's records — what every later single-classifier analysis reads.
# Per-class / per-condition / α analyses use it on X_C, the a-priori "all tools" set.
DEFAULT_RECS = {(r['feature_set'], r['protocol']): r for r in all_recs
                if r['classifier'] == DEFAULT_CLF_NAME}
DEFAULT_SET  = 'C: Combined'

_s41 = pd.DataFrame([{k: v for k, v in r.items() if k not in ('Y_pred', 'chosen')} for r in all_recs])
for pr in PROTOCOLS:
    print(f'\n📊 §6.1 F1-macro — {PROTOCOL_LABEL[pr]}   ({DEFAULT_CLF_NAME} = default; the other '
          'rows are a sensitivity table — NOT used for selection)')
    print(_s41.query('protocol == @pr').pivot(index='classifier', columns='feature_set',
                                              values='f1_macro')[FEATURE_SETS].round(4).to_string())
print('\n📊 §6.1 the DEPLOYED default vs its empirical null (§5.3; exact-width null for A–D)')
_fs_width = dict(zip(FEATURE_SETS, (int(np.shape(_x)[1]) for _x in (X_A, X_B, X_C, X_D))))
for pr in PROTOCOLS:
    for fs in FEATURE_SETS:
        r = DEFAULT_RECS[(fs, pr)]
        _mk = emp_null_mark(r['f1_macro'], pr, _fs_width[fs], fs)
        print(f'   {pr:<6} {fs:<20} d={_fs_width[fs]:<3} F1 {r["f1_macro"]:.4f}  null P95 {r["emp_null_p95"]:.4f}  '
              f'p {r["emp_null_p"]:.4f}  {_mk}  [{r["emp_null_label"]}]')
print('   (the BH-corrected p within the pre-declared primary family is §6.9)')
_gap = (_s41.query('protocol == "piece"').set_index(['classifier', 'feature_set'])['f1_macro']
        - _s41.query('protocol == "loo"').set_index(['classifier', 'feature_set'])['f1_macro'])
print('\n⚠️  OPTIMISM GAP (piece − loo), F1-macro:')
print(_gap.unstack('feature_set')[FEATURE_SETS].round(4).to_string())

# A5 edge case: how often inner-CV tuning was skipped (< 5 training positives, or positives
# in < 2 training pieces) and C=1.0 (still balanced) was used instead.
df_inner_skips = (_s41[_s41['tuned']].groupby(['classifier', 'protocol'])[['n_tuned', 'n_tune_skipped']]
                  .sum().assign(pct_skipped=lambda d: 100 * d.n_tune_skipped
                                / (d.n_tuned + d.n_tune_skipped).clip(lower=1)))
print('\n── Inner-CV tuning skipped (label × outer fold, all 4 sets) → C=1.0, still balanced ──')
print(df_inner_skips.round(1).to_string())
# The A–D × classifier table (the former §4.4 wrote it; its Wilcoxon over 8 classifiers was removed in
# Phase C2 — n = 8 classifiers are not independent units and the minimum attainable p is 0.0078).
pd.DataFrame([{'Feature Set': r['feature_set'], 'Classifier': r['classifier'], 'Protocol': r['protocol'],
               'Default': r['classifier'] == DEFAULT_CLF_NAME, 'Hamming ↓': r['hamming_loss'],
               'F1-macro ↑': r['f1_macro'], 'F1-micro ↑': r['f1_micro'], 'Subset Acc ↑': r['subset_acc'],
               'emp_null_p': r.get('emp_null_p', np.nan)} for r in all_recs]
             ).to_csv(os.path.join(OUTPUT_DIR, 'ablation_results.csv'), index=False)
print(f'\n✅ Experiments complete ({len(FEATURE_SETS)} sets × ({len(make_clfs())} classifiers + '
      f'{DEFAULT_CLF_NAME}) × '
      f'{len(PROTOCOLS)} protocols) · default for every later analysis: {DEFAULT_CLF_NAME}')


## Part 6 · Prediction — §6.2 Valence / arousal regression

**Deployed (headline):** the nested piece-grouped ridge of §5.1 (α chosen inside every outer training fold)
on **MERT-full 768** and on **Interpretable (27)** (`INTERP_PRED_COLS`, §4.1). Scored by Pearson r,
out-of-sample R² (baseline = the training-fold mean) and RMSE, both protocols, `piece` first.

**Why regression uses MERT-full, classification MERT-PCA25.** Tier `4A-mert-pca` (§6.4) showed that the
PCA-95 block keeps the high-variance **piece-identity** directions and discards the arousal-relevant
variance (Phase A, `piece`: R² arousal −0.25 with PCA vs +0.45 with all 768 dims), so every regression head
sees MERT at full dimensionality (`MERT_REG_REPR = 'full'`).

**Sensitivity (not deployed):** the fixed Ridge (α = 10), SVR and MLP on MERT-PCA25 — the pre-Phase-A
models, kept only to show how much the numbers depend on the learner. Nothing downstream reads them any
more (§6.8 and Fig 5 now use the deployed nested ridge).


In [ ]:
# ── §6.2 Valence / arousal regression ───────────────────────────────
# NEEDS : df_expr, get_cv, INTERP_PRED_COLS, nested_ridge_predict, oos_r2, OUTPUT_DIR, PRIMARY_PROTOCOL,
#          PROTOCOLS, PROTOCOL_LABEL, rmse, warn_negative_loo_r, X_A, X_INTERP (§4.1), X_MERT_FULL
# MAKES : conditions, REG, REG_INTERP, REG_MERTFULL, REG_DEPLOYED, REG_RES (protocol → model →
#          predictions/metrics), REG_ROLE, reg_res (= REG_RES[PRIMARY_PROTOCOL]), y_arousal, y_valence
#          · regression_results.csv
# FEEDS : §6.7 (conditions), §6.8 (deployed nested-ridge predictions → quadrants, Fig 5), §11.1
# ═══════════════════════════════════════════════════════════════════════════════
# §6.2 · Valence / arousal regression — the DEPLOYED head first, the fixed models as a sensitivity table
# ═══════════════════════════════════════════════════════════════════════════════
# DEPLOYED (headline): nested piece-grouped ridge (§5.1 — α chosen inside every outer training fold) on
#   · MERT-full 768 (X_MERT_FULL = MERT_REG, MERT_REG_REPR = 'full') — the representation every regression
#     head uses (tier 4A-mert-pca, §6.4: PCA-95 keeps the piece-identity directions and discards the
#     arousal-relevant variance);
#   · Interpretable (27) (X_INTERP = INTERP_PRED_COLS, §4.1) — the same nested ridge.
# SENSITIVITY (not deployed): fixed Ridge α = 10, SVR and MLP on MERT-PCA25 (X_A) — the pre-Phase-A models,
#   kept to show how much the numbers depend on the learner; nothing downstream reads them any more.
REG_MERTFULL = 'Ridge-nested · MERT-full'
REG_INTERP   = f'Ridge-nested · Interpretable ({len(INTERP_PRED_COLS)})'
REG_DEPLOYED = [REG_MERTFULL, REG_INTERP]
conditions  = df_expr['condition'].values
y_valence   = df_expr['valence_mean'].values
y_arousal   = df_expr['arousal_mean'].values
REG = {'Ridge':Ridge(alpha=10.0),'SVR':SVR(kernel='rbf',C=1.0,epsilon=0.3),
       'MLP':MLPRegressor(hidden_layer_sizes=(256,64),alpha=0.01,max_iter=1000,random_state=42)}
REG_ROLE = {REG_MERTFULL: f'deployed · nested ridge · MERT-full {X_MERT_FULL.shape[1]}',
            REG_INTERP: f'deployed · nested ridge · Interpretable ({X_INTERP.shape[1]})',
            **{nm: f'sensitivity (not deployed) · fixed {nm} · MERT-PCA{X_A.shape[1]}' for nm in REG}}
REG_RES, _reg_rows = {pr: {} for pr in PROTOCOLS}, []


def _reg_record(pr, nm, vp, ap):
    vr, _ = pearsonr(y_valence, vp); ar, _ = pearsonr(y_arousal, ap)
    vr2, ar2 = oos_r2(y_valence, vp, pr), oos_r2(y_arousal, ap, pr)
    vrmse, armse = rmse(y_valence, vp), rmse(y_arousal, ap)
    REG_RES[pr][nm] = {'vp': vp, 'ap': ap, 'vr': vr, 'ar': ar, 'vr2': vr2, 'ar2': ar2,
                       'vrmse': vrmse, 'armse': armse}
    _reg_rows.append(dict(protocol=pr, model=nm, role=REG_ROLE[nm], r_valence=vr, r_arousal=ar, r2_val=vr2,
                          r2_aro=ar2, rmse_val=vrmse, rmse_aro=armse))
    print(f'  {nm:<40} {vr:>7.3f} {ar:>7.3f} {vr2:>+8.3f} {ar2:>+8.3f} {vrmse:>9.3f} {armse:>9.3f}')


for pr in PROTOCOLS:
    cv, g = get_cv(pr)
    print(f'\n── {PROTOCOL_LABEL[pr]} ──')
    print(f"  {'Model':<40} {'Val r':>7} {'Aro r':>7} {'Val R²':>8} {'Aro R²':>8} {'Val RMSE':>9} {'Aro RMSE':>9}")
    print('  ' + '-' * 86)
    print('  DEPLOYED — nested piece-grouped ridge (α inside every training fold)')
    for nm, X in ((REG_MERTFULL, X_MERT_FULL), (REG_INTERP, X_INTERP)):
        vp, _ = nested_ridge_predict(X, y_valence, pr)
        ap, _ = nested_ridge_predict(X, y_arousal, pr)
        _reg_record(pr, nm, vp, ap)
    print(f'  SENSITIVITY (not deployed) — fixed models on MERT-PCA{X_A.shape[1]}')
    for nm, reg in REG.items():
        vp=cross_val_predict(Pipeline([('sc',StandardScaler()),('r',sk_clone(reg))]),X_A,y_valence,cv=cv,groups=g,n_jobs=-1)
        ap=cross_val_predict(Pipeline([('sc',StandardScaler()),('r',sk_clone(reg))]),X_A,y_arousal,cv=cv,groups=g,n_jobs=-1)
        _reg_record(pr, nm, vp, ap)
df_reg = pd.DataFrame(_reg_rows)
warn_negative_loo_r(df_reg)
print('\n⚠️  OPTIMISM GAP (piece − loo):')
for nm in REG_DEPLOYED + list(REG):
    p, l = REG_RES['piece'][nm], REG_RES['loo'][nm]
    print(f'   {nm:<40} r_val {p["vr"] - l["vr"]:+.3f}  r_aro {p["ar"] - l["ar"]:+.3f}  '
          f'R²_val {p["vr2"] - l["vr2"]:+.3f}  R²_aro {p["ar2"] - l["ar2"]:+.3f}')
reg_res = REG_RES[PRIMARY_PROTOCOL]          # back-compat alias — the PRIMARY protocol
df_reg.to_csv(os.path.join(OUTPUT_DIR, 'regression_results.csv'), index=False)
print(f'✅ §6.2 — regression_results.csv (role column: deployed / sensitivity; reg_res = the PRIMARY protocol)')


## Part 6 · Prediction — §6.3 Systematic ablation · engine + block lattice

§6.1 above compares four **hand-picked** feature sets; §7.5 asks what each tool **adds on top of
MERT**. Neither can answer the symmetric question — *what does each tool contribute when it is
removed* — because MERT is present in every block of both. This section scores the **complete
lattice** of the four tool blocks (2⁴−1 = **15 subsets**), so the unique and marginal contribution
of every tool, MERT included, becomes readable.

**One protocol, fixed for the whole study** (identical to §10.1 / §7.5, so every number in Parts 6–10
is comparable by construction): the default classifier — **nested model selection** over 13
configs (§5.1), chosen inside every outer training fold — for class F1-macro, the **nested
piece-grouped ridge** (`nested_ridge_predict`, α chosen inside every training fold) for
valence/arousal — scored by Pearson *r*, **out-of-sample R²** and RMSE — and `StandardScaler` always
**inside** the CV fold. **B-MERTFULL:** wherever MERT is in a subset, the regression heads see it at
full dimensionality (`ABL_BLOCKS_REG`), the classification heads keep the PCA block; the lattice
rebuild is asserted for both (`X_C/X_D/X_F` and `X_C_REG/X_D_REG/X_F_REG`). Every row records which
configs the nested selection chose (`selected`, `n_configs_selected`). No feature
selection happens here — each block enters exactly as its §2.x gate forwarded it. This cell reads
`Y` only to **score**, never to choose features, so the unsupervised-forwarding invariant is intact.

**Block weighting of the multi-block MERT-full heads (`REG_BLOCK_WEIGHTING`, §0.3).** *Problem.* In the
Phase B run (N = 118, `piece`) every regression set that contained MERT-full reached R²_arousal ≈ 0.45–0.46
whatever else it contained — the `MERT + X` rows equalled MERT alone — while Essentia + DTW alone reached
0.67. After `StandardScaler` each column carries one unit of variance, so the 768 MERT columns carry 768
units against ~10 per interpretable block, and a single ridge α lets MERT dominate. *Fix.* Block *b*
(p_b columns, P = Σ p_b, B blocks) is multiplied by **w_b = √(P / (B·p_b))** after the scaler: every tool
block then carries the same standardised variance P/B, the total stays P (so `RIDGE_ALPHAS` keeps its
meaning), and ridge on the weighted columns is a group ridge with α_b ∝ B·p_b / P — an equal *a-priori*
share of the penalty per tool. *Why it is leakage-free.* The weights are a function of the **column counts
only** — never of the data, the labels or the fold — and are applied inside every fold, after the scaler
(`StandardScaler → BlockScaler → Ridge`). *Scope.* Only regression heads that concatenate MERT-full with
≥ 1 other tool block (MERT / CREPE / Essentia / DTW are the blocks); single-block heads, the PCA-MERT
matrices, every classification head and the §9.2 read-out are unchanged. Rows record `block_weighting`
(`equal` / `none` / `n/a`) and the number of outer folds whose α hit the grid edge (printed, never
silently widened). Tier 2's unique R² therefore uses the deployed weighting; tier `4E-block-weighting`
(§6.4) compares weighted vs unweighted directly. The rebuild asserts compare the **raw** matrices.

**Two CV protocols, both reported.**

| Protocol | What is held out | Reading |
|---|---|---|
| `piece` | all three renditions of one piece (`LeaveOneGroupOut`) | **PRIMARY** — honest generalisation to **unseen material** |
| `loo` | one excerpt | secondary, **optimistic** — holding out `Bach_EXP` leaves `Bach_MEC` and `Bach_EXG` in training, and features are strongly piece-dependent |

The gap between them is itself a result. Both are reported everywhere (`piece` first); neither is
quietly dropped. The chance floor is the §5.1 Dummy distribution: `marginal_f1` subtracts its mean and
`beats_chance` compares against its strict P95 (kept as a column). **Phase C2:** the printed ✓ (and
`beats_emp_null`, `emp_null_p`) read the **empirical null of the deployed pipeline** (§5.3) — the exact-width
null of the set's own width (Phase C3 rule, §5.3: e.g. A–D under their lattice names, CREPE + Essentia + DTW =
Interpretable (27), DTW, CREPE + DTW, Essentia, Essentia + DTW), else the next larger primary width ("borrowed",
conservative), above 52 "extrapolated"; `–` = no null under that protocol. Tier
`1b-tempogram-sensitivity` scores *DTW ref-free − tempogram* and *CREPE + Essentia + DTW − tempogram* against
their parents (paired permutation p, BH within the tier) — the tempogram features failed the manual audit.

**Statistics.** Every score carries a **bootstrap 95% CI over the 60 excerpts**; every Δ against a
tier reference carries a **paired permutation p** (predictions are swapped per excerpt and the
metric recomputed from scratch — valid for macro-F1, Pearson *r* and R² alike, which a sign-flip
test on per-sample errors would not be); p-values (F1, r and R² for both dimensions) are **BH-FDR
corrected within each tier × protocol**.

> **How to read the lattice.** At N=60 an F1 difference below ~0.02 is noise. This is a
> **descriptive attribution map, not model selection** — picking the winning subset *because* it won
> here and then reporting its score would be selection on the same CV.

In [ ]:
# ── §6.3 Systematic ablation · engine (Tier 0) + block lattice (Tier 1/2) ──
# NEEDS : bh_fdr, CHANCE_FLOOR_MEAN, CHANCE_FLOOR_P95, CREPE_FORWARD, cv_predict_default, cv_predict_multilabel,
#          DEFAULT_CLF_NAME, df_dtw_fwd, df_ess_fwd, df_expr, DTW_MODEL_COLS, eids, EMP_NULL_PRIMARY, emp_null_mark,
#          emp_null_ref (§5.3), ESSENTIA_FORWARD, get_cv, INTERP_PRED_COLS, MERT_REG, MERT_REG_REPR,
#          TEMPOGRAM_UNRELIABLE, X_INTERP (§4.1),
#          nested_ridge_predict, nms_counts_str, oos_baseline, oos_r2, OUTPUT_DIR, PIECE_GROUPS,
#          PROTOCOLS, PROTOCOL_LABEL, REG_BLOCK_WEIGHTING, RIDGE_ALPHAS, rmse, SEED,
#          warn_negative_loo_r, X_A, X_B, X_C, X_C_REG, X_D, X_D_REG, X_F, X_F_REG, Y
# MAKES : ABL_ALPHAS, ABL_BLOCKS, ABL_BLOCKS_REG, ABL_FLOOR, ABL_FLOOR_P95, ABL_GROUPS, ABL_N_BOOT,
#          ABL_N_PERM,
#          ABL_ORDER, ABL_PRED, ABL_ROWS, ABL_SEED, ABL_Y_ARO, ABL_Y_VAL, ablate, abl_frame,
#          abl_alpha_edge_report, abl_vs_ref, boot_ci, df_abl_blocks, df_abl_unique, perm_p,
#          _abl_cv, _abl_mat, _abl_reg_kw, _abl_stack, _abl_stack_reg, _cv_cls, _cv_reg, _sc_f1,
#          _sc_r, _sc_r2 · ablation_blocks.csv, ablation_blocks_vs_full.csv,
#          ablation_unique_marginal.csv, 2 figures · df_abl_tempogram, df_abl_tempogram_deltas,
#          TG_DROP · ablation_tempogram_sensitivity.csv
# FEEDS : §6.4, §6.5, §6.9 (lattice F1 read-out), §7.1, §7.2, §7.5 (MERT + X rows read from ABL_PRED),
#          §8.1 (_abl_mat), §10.2, §11.1 (tempogram sensitivity)
# ═══════════════════════════════════════════════════════════════════════════════
# §6.3 · Systematic ablation — ONE engine, ONE protocol, every tier reads it
# ═══════════════════════════════════════════════════════════════════════════════
# Why this exists: §6.1 compares four hand-picked sets and §7.5 only ever ADDS to
# MERT, so neither can show a tool being unnecessary — MERT is in every block there.
# Phase C2: the printed ✓ of every row = F1 above the P95 of its EMPIRICAL null (§5.3 — exact width for the
# primary sets, else the nearest map width, "borrowed"); beats_chance (Dummy strict P95) is kept as a column.
# The full lattice below removes that asymmetry.
#
# PROTOCOL (Phase B): classifier = the default NESTED SELECTION over SELECT_CONFIGS (§5.1,
# B-NMS — chosen inside every outer training fold, never on the reported CV); regression =
# nested piece-grouped ridge (§5.1) scored by r, out-of-sample R² and RMSE. B-MERTFULL: the
# regression heads see MERT at full dimensionality (MERT_REG = X_MERT_FULL) wherever the MERT
# block is in the set; the classification heads keep the PCA-forwarded X_A. Both CV
# protocols, `piece` (PRIMARY) first.
# BLOCK WEIGHTING (REG_BLOCK_WEIGHTING, §0.3): a regression head that concatenates MERT-full
# with ≥ 1 other tool block gets the §5.1 equal-share block weights (StandardScaler →
# BlockScaler → Ridge, inside every fold; weights from the column counts only). Single-block
# heads, the classification heads and the PCA-MERT matrices are never weighted. Every row
# records `block_weighting` ('equal' / 'none' / 'n/a') and how many outer folds chose an α at
# the edge of the grid.
#
# LEAKAGE NOTE: this cell reads Y to SCORE, never to SELECT. Every block enters
# exactly as §2.1b-§2.4b forwarded it (unsupervised), and StandardScaler stays
# inside each CV fold. Nothing here changes what the models are given.
from itertools import combinations as _abl_combos

ABL_SEED   = SEED
ABL_N_BOOT = 2000        # bootstrap resamples over the 60 excerpts → 95% CI per score
ABL_N_PERM = 2000        # paired permutations → p for each Δ vs the tier reference
ABL_ALPHAS = RIDGE_ALPHAS    # nested ridge grid (§5.1) — α chosen by piece-grouped inner CV

# Row order is positional throughout: Y[i] ↔ eids[i] ↔ df_expr.iloc[i].
assert list(df_expr['excerpt_id']) == list(eids), \
    '§6.3: df_expr row order must match `eids` — the ablation pairs them positionally'
ABL_GROUPS = PIECE_GROUPS    # alias — §5.1's PIECE_GROUPS is the single source of truth
ABL_Y_VAL  = df_expr['valence_mean'].to_numpy(dtype=float)
ABL_Y_ARO  = df_expr['arousal_mean'].to_numpy(dtype=float)
print(f'§6.3 · piece-wise protocol groups: {len(np.unique(ABL_GROUPS))} pieces / '
      f'{len(eids)} excerpts · classifier = {DEFAULT_CLF_NAME} · regression = nested ridge')


# ── Tier 0 · the shared machinery ──────────────────────────────────────────
def _abl_mat(df):
    """Row-align a tabular block to `eids`; coerce to a finite float matrix."""
    D = df.reindex(eids).apply(pd.to_numeric, errors='coerce')
    return D.fillna(D.mean(numeric_only=True)).to_numpy(dtype=float)


def _abl_cv(protocol):
    """Back-compat shim: the splitter now comes from §5.1's get_cv (single source of truth).
    'loo' → leave-one-EXCERPT-out (optimistic), 'piece' → leave-one-PIECE-out (primary)."""
    return get_cv(protocol)


def _cv_cls(X, Ym, protocol, make_clf=None, tune_grid=None, label=None):
    """→ (Yp, info). Default = the §5.1 nested selection (info has 'chosen'/'counts');
    tier 4D passes its own fixed classifier (info has 'n_skipped')."""
    if make_clf is None:
        return cv_predict_default(X, Ym, protocol, label=label, return_info=True)
    return cv_predict_multilabel(make_clf, X, Ym, protocol, tune_grid=tune_grid, return_info=True)


def _cv_reg(X, y, protocol, block_sizes=None):
    """→ (yhat, α per outer fold) from the nested piece-grouped ridge (replaces RidgeCV);
    block_sizes → the §5.1 equal-share block weights inside every fold."""
    return nested_ridge_predict(X, y, protocol, alphas=ABL_ALPHAS, block_sizes=block_sizes)


def _n_alpha_edge(alphas_per_fold):
    """Outer folds whose nested α sits at either end of the grid (lowest or highest α)."""
    a = np.asarray(alphas_per_fold, dtype=float)
    return int(np.sum(np.isclose(a, ABL_ALPHAS[0]) | np.isclose(a, ABL_ALPHAS[-1])))


def _sc_f1(Yt, Yp, idx=None):
    i = slice(None) if idx is None else idx
    return float(f1_score(Yt[i], Yp[i], average='macro', zero_division=0))


def _sc_r(yt, yp, idx=None):
    i = slice(None) if idx is None else idx
    a, b = np.asarray(yt, dtype=float)[i], np.asarray(yp, dtype=float)[i]
    if a.size < 3 or np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return float('nan')                      # degenerate resample — dropped, not zeroed
    return float(pearsonr(a, b)[0])


def _sc_r2(yt, yp, base, idx=None):
    """Out-of-sample R² against the precomputed training-fold-mean baseline (§5.1 oos_r2)."""
    return oos_r2(yt, yp, baseline=base, idx=idx)


def boot_ci(score_of_idx, n, rng, n_boot=None):
    """Percentile 95% CI from a bootstrap over the EXCERPTS (the sampling unit).

    NOTE: in a resample where a rare class has neither true nor predicted positives,
    macro-F1 scores that class 0 (zero_division=0), so these CIs are mildly
    conservative on the low side for the rarest classes. Stated, not hidden."""
    n_boot = ABL_N_BOOT if n_boot is None else n_boot
    v = [score_of_idx(rng.integers(0, n, n)) for _ in range(n_boot)]
    v = np.asarray([x for x in v if np.isfinite(x)], dtype=float)
    if v.size == 0:
        return (float('nan'), float('nan'))
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))


def perm_p(pred_a, pred_b, score_of_pred, rng, n_perm=None):
    """Two-sided PAIRED permutation on per-excerpt predictions.

    Per excerpt, randomly swap which model owns the prediction, then recompute the
    metric gap FROM SCRATCH. Valid for any metric (macro-F1, Pearson r, R²) precisely
    because the metric is never decomposed into per-sample terms — which is the
    assumption a sign-flip test on per-sample errors would silently make."""
    n_perm = ABL_N_PERM if n_perm is None else n_perm
    a, b   = np.asarray(pred_a), np.asarray(pred_b)
    obs    = score_of_pred(a) - score_of_pred(b)
    if not np.isfinite(obs):
        return float('nan')
    n, hit = a.shape[0], 0
    for _ in range(n_perm):
        sw = rng.random(n) < 0.5
        m  = sw[:, None] if a.ndim == 2 else sw
        d  = score_of_pred(np.where(m, b, a)) - score_of_pred(np.where(m, a, b))
        if np.isfinite(d) and abs(d) >= abs(obs) - 1e-12:
            hit += 1
    return (hit + 1) / (n_perm + 1)


# bh_fdr lives in §0.3 (Phase C2 — the Part 4 manipulation check needs it first) — same function.

ABL_ROWS, ABL_PRED = [], {}

# ── Chance floor under the SAME protocol — computed once in §5.1 (A1) ──────
# ABL_FLOOR     = mean of the strongest-by-mean Dummy strategy (→ marginal_f1)
# ABL_FLOOR_P95 = the STRICT 'beats chance' line: max P95 across the three strategies
ABL_FLOOR     = dict(CHANCE_FLOOR_MEAN)
ABL_FLOOR_P95 = dict(CHANCE_FLOOR_P95)
for _pr in PROTOCOLS:
    print(f'   chance floor ({_pr}): mean {ABL_FLOOR[_pr]:.4f} · strict P95 {ABL_FLOOR_P95[_pr]:.4f}')


def ablate(X, name, tier, Ym=None, y_val=None, y_aro=None, protocols=PROTOCOLS, note='',
           verbose=True, make_clf=None, tune_grid=None, clf_name=None, regression=True,
           X_reg=None, reg_block_sizes=None, block_weighting=None):
    """Score ONE feature matrix under the fixed protocol and record it.

    Classifier = the §5.1 default (nested selection, B-NMS) unless make_clf is given (tier
    4D only). Regression = nested piece-grouped ridge for valence and arousal on X_reg
    (default: X itself; pass the MERT_REG-based matrix when MERT is in the set, B-MERTFULL),
    scored by Pearson r, out-of-sample R² (baseline = training-fold mean) and RMSE;
    regression=False skips the ridge head (a classification-only tier whose rows all share
    one X). reg_block_sizes (column counts of X_reg's tool blocks) applies the §5.1
    equal-share block weights inside every fold; block_weighting is the label recorded in the
    row — default 'equal' when reg_block_sizes is given, else 'n/a' (pass 'none' for an
    in-scope multi-block MERT-full head that is deliberately fitted unweighted; _abl_reg_kw
    builds both arguments for a lattice subset).
    Per-excerpt predictions are cached in ABL_PRED so every Δ can afterwards be
    tested PAIRED and every CI bootstrapped WITHOUT refitting a single model."""
    X   = np.asarray(X, dtype=float)
    Xr  = X         if X_reg is None else np.asarray(X_reg, dtype=float)
    if not regression:
        reg_block_sizes, block_weighting = None, 'n/a'
    if block_weighting is None:
        block_weighting = 'equal' if reg_block_sizes is not None else 'n/a'
    if block_weighting not in ('equal', 'none', 'n/a'):
        raise ValueError(f'{tier}/{name}: block_weighting must be equal / none / n/a')
    if (block_weighting == 'equal') != (reg_block_sizes is not None):
        raise ValueError(f'{tier}/{name}: block_weighting={block_weighting!r} but '
                         f'reg_block_sizes={reg_block_sizes!r}')
    Ym  = Y         if Ym    is None else np.asarray(Ym)
    y_v = ABL_Y_VAL if y_val is None else np.asarray(y_val, dtype=float)
    y_a = ABL_Y_ARO if y_aro is None else np.asarray(y_aro, dtype=float)
    if X.shape[0] != Ym.shape[0] or X.shape[0] != len(y_v) or Xr.shape[0] != X.shape[0]:
        raise ValueError(f'{tier}/{name}: row mismatch — X={X.shape[0]}, X_reg={Xr.shape[0]}, '
                         f'Y={Ym.shape[0]}, y={len(y_v)}')
    clf_name   = clf_name or (DEFAULT_CLF_NAME if make_clf is None else 'custom')
    deployed_y = np.array_equal(Ym, Y)           # chance floors were computed for the deployed Y
    nan        = float('nan')

    out = {}
    for pr in protocols:
        rng = np.random.default_rng(ABL_SEED)          # reproducible per (set, protocol)
        Yp, info = _cv_cls(X, Ym, pr, make_clf, tune_grid, label=f'{tier} · {name}')
        n  = X.shape[0]
        f1 = _sc_f1(Ym, Yp)
        f1_lo, f1_hi = boot_ci(lambda i: _sc_f1(Ym, Yp, i), n, rng)
        if regression:
            (vp, al_v), (ap, al_a) = (_cv_reg(Xr, y_v, pr, reg_block_sizes),
                                      _cv_reg(Xr, y_a, pr, reg_block_sizes))
            bv, ba = oos_baseline(y_v, pr), oos_baseline(y_a, pr)
            rv, ra = _sc_r(y_v, vp), _sc_r(y_a, ap)
            rv_lo, rv_hi = boot_ci(lambda i: _sc_r(y_v, vp, i), n, rng)
            ra_lo, ra_hi = boot_ci(lambda i: _sc_r(y_a, ap, i), n, rng)
            r2v, r2a = _sc_r2(y_v, vp, bv), _sc_r2(y_a, ap, ba)
            mv, ma   = rmse(y_v, vp), rmse(y_a, ap)
            av, aa   = float(np.median(al_v)), float(np.median(al_a))
            ev, ea   = _n_alpha_edge(al_v), _n_alpha_edge(al_a)
            n_outer  = int(len(al_v))
        else:
            vp = ap = bv = ba = None
            rv = ra = rv_lo = rv_hi = ra_lo = ra_hi = r2v = r2a = mv = ma = av = aa = nan
            ev = ea = n_outer = nan
        ABL_PRED[(tier, name, pr)] = dict(cls=Yp, val=vp, aro=ap, Yt=Ym, yv=y_v, ya=y_a,
                                          bv=bv, ba=ba)
        # §5.3 empirical null — defined for the DEPLOYED path only (NestedSelect on the deployed Y)
        _en_ok = deployed_y and make_clf is None
        _en = emp_null_ref(f1, pr, X.shape[1], name)
        row = dict(tier=tier, set_name=name, protocol=pr, classifier=clf_name,
                   n_features=int(X.shape[1]),
                   n_features_reg=(int(Xr.shape[1]) if regression else nan),
                   f1_macro=f1, f1_lo=f1_lo, f1_hi=f1_hi,
                   beats_chance=(bool(f1 > ABL_FLOOR_P95[pr]) if deployed_y else nan),
                   emp_null_p=(_en['p'] if _en_ok else nan),
                   emp_null_p95=(_en['p95'] if _en_ok else nan),
                   emp_null_d=(_en['null_d'] if _en_ok else nan),
                   emp_null_kind=(_en['kind'] if _en_ok else
                                  'n/a (other Y)' if not deployed_y else 'n/a (not the deployed classifier)'),
                   beats_emp_null=((bool(f1 > _en['p95']) if np.isfinite(_en['p95']) else nan)
                                   if _en_ok else nan),
                   r_valence=rv, rv_lo=rv_lo, rv_hi=rv_hi, r2_val=r2v, rmse_val=mv,
                   r_arousal=ra, ra_lo=ra_lo, ra_hi=ra_hi, r2_aro=r2a, rmse_aro=ma,
                   alpha_val_median=av, alpha_aro_median=aa,
                   block_weighting=block_weighting,
                   n_alpha_edge_val=ev, n_alpha_edge_aro=ea, n_outer_folds=n_outer,
                   n_tune_skipped=int(info.get('n_skipped', 0)),
                   selected=(nms_counts_str(info) if 'counts' in info else clf_name),
                   n_configs_selected=(len(info['counts']) if 'counts' in info else 1),
                   note=note)
        ABL_ROWS.append(row)
        out[pr] = row
    if verbose:
        parts = []
        for pr in protocols:
            r  = out[pr]
            bc = (emp_null_mark(r['f1_macro'], pr, X.shape[1], name)
                  if (deployed_y and make_clf is None) else ' ')
            s  = f'{pr}: F1={r["f1_macro"]:.3f}{bc}'
            if regression:
                s += f' R²v={r["r2_val"]:+.2f} R²a={r["r2_aro"]:+.2f}'
            parts.append(s)
        dd = (f'{X.shape[1]:>4}' if (not regression or Xr.shape[1] == X.shape[1])
              else f'{X.shape[1]}/{Xr.shape[1]}')
        bw = {'equal': ' [w]', 'none': ' [unw]', 'n/a': ''}[block_weighting]
        print(f'  {name:<46} d={dd:>8}  ' + ' | '.join(parts) + bw)
    return out


def abl_frame():
    return pd.DataFrame(ABL_ROWS)


_ABL_PCOLS = ['p_f1', 'p_r_valence', 'p_r_arousal', 'p_r2_val', 'p_r2_aro']


def abl_vs_ref(tier, ref_name, protocols=PROTOCOLS, ref_map=None, pcols=None):
    """Δ vs the tier's reference set + paired permutation p + BH-FDR.

    ref_map (optional): {set_name: its own reference set_name} — lets ONE tier compare
    several matrices, each against its own reference (tier 4D-balancing), while keeping a
    single BH family per (tier, protocol). Reference rows themselves are skipped.
    The FDR family is deliberately "every p-value of this tier under this protocol"
    (F1, r_val, r_aro, R²_val, R²_aro) — the honest denominator for a lattice that makes
    many comparisons. pcols (optional) restricts the tested metrics to a subset of
    _ABL_PCOLS — e.g. the regression-only tier 4E, whose classification head is identical by
    construction; untested metrics get p = q = NaN and stay out of the BH family.
    When two runs do NOT share a ground truth (e.g. the §6.5 radius
    sweep, where Y itself changes) they are not paired on the same target, so a paired
    permutation would be meaningless: Δ is still reported, p is NaN, and `paired` says so."""
    rows, nan = [], float('nan')
    pcols = list(_ABL_PCOLS if pcols is None else pcols)
    if not set(pcols) <= set(_ABL_PCOLS):
        raise ValueError(f'abl_vs_ref: pcols must be a subset of {_ABL_PCOLS}, got {pcols}')
    refs = set(ref_map.values()) if ref_map else {ref_name}
    for pr in protocols:
        rng = np.random.default_rng(ABL_SEED + 1)
        for (t, nm, p), cur in ABL_PRED.items():
            if t != tier or p != pr or nm in refs:
                continue
            rn = ref_map.get(nm) if ref_map else ref_name
            if rn is None:
                continue
            ref = ABL_PRED.get((tier, rn, pr))
            if ref is None:
                raise KeyError(f'{tier}: reference {rn!r} was never ablated under {pr!r}')
            reg    = cur['val'] is not None and ref['val'] is not None
            paired = (np.array_equal(cur['Yt'], ref['Yt'])
                      and np.allclose(cur['yv'], ref['yv'])
                      and np.allclose(cur['ya'], ref['ya']))
            d_f1 = _sc_f1(cur['Yt'], cur['cls']) - _sc_f1(ref['Yt'], ref['cls'])
            if reg:
                d_rv  = _sc_r(cur['yv'], cur['val']) - _sc_r(ref['yv'], ref['val'])
                d_ra  = _sc_r(cur['ya'], cur['aro']) - _sc_r(ref['ya'], ref['aro'])
                d_r2v = (_sc_r2(cur['yv'], cur['val'], cur['bv'])
                         - _sc_r2(ref['yv'], ref['val'], ref['bv']))
                d_r2a = (_sc_r2(cur['ya'], cur['aro'], cur['ba'])
                         - _sc_r2(ref['ya'], ref['aro'], ref['ba']))
            else:
                d_rv = d_ra = d_r2v = d_r2a = nan
            p_f1 = p_rv = p_ra = p_r2v = p_r2a = nan
            if paired:
                if 'p_f1' in pcols:
                    p_f1 = perm_p(cur['cls'], ref['cls'], lambda P: _sc_f1(ref['Yt'], P), rng)
                if reg:
                    if 'p_r_valence' in pcols:
                        p_rv  = perm_p(cur['val'], ref['val'], lambda P: _sc_r(ref['yv'], P), rng)
                    if 'p_r_arousal' in pcols:
                        p_ra  = perm_p(cur['aro'], ref['aro'], lambda P: _sc_r(ref['ya'], P), rng)
                    if 'p_r2_val' in pcols:
                        p_r2v = perm_p(cur['val'], ref['val'],
                                       lambda P: _sc_r2(ref['yv'], P, ref['bv']), rng)
                    if 'p_r2_aro' in pcols:
                        p_r2a = perm_p(cur['aro'], ref['aro'],
                                       lambda P: _sc_r2(ref['ya'], P, ref['ba']), rng)
            rows.append(dict(tier=tier, set_name=nm, protocol=pr, ref=rn, paired=paired,
                             d_f1=d_f1, p_f1=p_f1, d_r_valence=d_rv, p_r_valence=p_rv,
                             d_r_arousal=d_ra, p_r_arousal=p_ra,
                             d_r2_val=d_r2v, p_r2_val=p_r2v, d_r2_aro=d_r2a, p_r2_aro=p_r2a))
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    for _pr, sub in df.groupby('protocol'):
        k    = len(sub)
        allp = np.concatenate([sub[c].to_numpy(dtype=float) for c in pcols])
        q    = bh_fdr(allp)
        for i, c in enumerate(pcols):
            df.loc[sub.index, 'q' + c[1:]] = q[i * k:(i + 1) * k]
    for c in _ABL_PCOLS:                       # untested metrics: q column present, all NaN
        if 'q' + c[1:] not in df.columns:
            df['q' + c[1:]] = nan
    return df


# ── Tier 1 · the complete block lattice ────────────────────────────────────
ABL_ORDER  = ['MERT', 'CREPE', 'Essentia', 'DTW']
ABL_BLOCKS = {
    'MERT'    : np.asarray(X_A, dtype=float),   # §2.1b PCA-forwarded
    'CREPE'   : np.asarray(X_B, dtype=float),   # §2.2b forwarded
    'Essentia': _abl_mat(df_ess_fwd),           # §2.3b ESSENTIA_FORWARD
    'DTW'     : _abl_mat(df_dtw_fwd),           # §2.4b DTW_MODEL_COLS
}


# B-MERTFULL: the REGRESSION heads see MERT at full dimensionality (MERT_REG, §2.5);
# every other block is identical to ABL_BLOCKS.
ABL_BLOCKS_REG = dict(ABL_BLOCKS, MERT=np.asarray(MERT_REG, dtype=float))


def _abl_stack(names):
    return np.hstack([ABL_BLOCKS[b] for b in names])


def _abl_stack_reg(names):
    return np.hstack([ABL_BLOCKS_REG[b] for b in names])


def _abl_reg_kw(names, weighting=None):
    """ablate() kwargs for the REGRESSION head of a lattice subset `names`: the column counts
    of its tool blocks and the weighting label. In scope = MERT-full (MERT_REG_REPR='full')
    concatenated with ≥ 1 other tool block; there the deployed REG_BLOCK_WEIGHTING applies
    ('equal' → block weights, 'none' → unweighted but recorded as an in-scope head). Out of
    scope (single block, PCA-MERT, no MERT) → no weighting, recorded 'n/a'. `weighting`
    overrides the deployed setting (tier 4E)."""
    names = list(names)
    mode = REG_BLOCK_WEIGHTING if weighting is None else weighting
    if mode not in ('equal', 'none'):
        raise ValueError(f'_abl_reg_kw: weighting must be equal / none, got {mode!r}')
    in_scope = MERT_REG_REPR == 'full' and 'MERT' in names and len(names) >= 2
    if not in_scope:
        return dict(reg_block_sizes=None, block_weighting='n/a')
    if mode == 'none':
        return dict(reg_block_sizes=None, block_weighting='none')
    return dict(reg_block_sizes=[int(ABL_BLOCKS_REG[b].shape[1]) for b in names],
                block_weighting='equal')


def abl_alpha_edge_report(df, label):
    """Print how many outer folds chose an α at the edge of RIDGE_ALPHAS in the WEIGHTED
    heads of `df` (ablate rows); warn — the grid is NOT widened silently."""
    w = df[df['block_weighting'] == 'equal'] if 'block_weighting' in df else df.iloc[0:0]
    if not len(w):
        print(f'   α grid edge ({label}): no block-weighted head in this table')
        return 0
    n_edge = int(w['n_alpha_edge_val'].sum() + w['n_alpha_edge_aro'].sum())
    n_all  = int(2 * w['n_outer_folds'].sum())
    print(f'   α grid edge ({label}, weighted heads): {n_edge} of {n_all} outer-fold α choices at '
          f'{ABL_ALPHAS[0]:g} or {ABL_ALPHAS[-1]:g}')
    if n_edge:
        _e = w[(w['n_alpha_edge_val'] + w['n_alpha_edge_aro']) > 0]
        for r in _e.itertuples():
            print(f'     ⚠️  {r.protocol:<5} {r.set_name:<46} val {int(r.n_alpha_edge_val)}/{int(r.n_outer_folds)} · '
                  f'aro {int(r.n_alpha_edge_aro)}/{int(r.n_outer_folds)}  (median α val {r.alpha_val_median:g}, '
                  f'aro {r.alpha_aro_median:g})')
        print('     ⚠️  the RIDGE_ALPHAS grid is NOT changed automatically — widen it in §5.1 only by decision')
    return n_edge


# The lattice must REPRODUCE the §2.5 matrices exactly, or it is measuring something
# other than what the pipeline deploys.
for _nm, _built, _ref in [('X_C', _abl_stack(ABL_ORDER),            X_C),
                          ('X_D', _abl_stack(['MERT', 'Essentia']), X_D),
                          ('X_F', _abl_stack(['MERT', 'DTW']),      X_F)]:
    assert np.allclose(_built, np.asarray(_ref, dtype=float), atol=1e-4, rtol=1e-4), \
        f'§6.3: lattice rebuild of {_nm} differs from §2.5 — the blocks are out of sync'
print('✅ lattice reproduces §2.5 X_C / X_D / X_F exactly (classification matrices)')
for _nm, _built, _ref in [('X_C_REG', _abl_stack_reg(ABL_ORDER),            X_C_REG),
                          ('X_D_REG', _abl_stack_reg(['MERT', 'Essentia']), X_D_REG),
                          ('X_F_REG', _abl_stack_reg(['MERT', 'DTW']),      X_F_REG)]:
    assert _built.shape == np.shape(_ref) and np.allclose(_built, np.asarray(_ref, dtype=float),
                                                         atol=1e-4, rtol=1e-4), \
        f'§6.3: lattice rebuild of {_nm} differs from §2.5 — the regression blocks are out of sync'
print(f'✅ lattice reproduces §2.5 X_C_REG / X_D_REG / X_F_REG exactly (regression matrices, '
      f'MERT_REG_REPR={MERT_REG_REPR!r}, MERT block {ABL_BLOCKS_REG["MERT"].shape[1]}-d)')
# (the asserts above compare the RAW stacked matrices — the block weights are applied later,
#  inside every fold, by the ridge pipeline, so they cannot affect the rebuild)
print(f'   regression block weighting: REG_BLOCK_WEIGHTING={REG_BLOCK_WEIGHTING!r} on every MERT-full '
      f'subset with ≥ 2 blocks ([w] = weighted, [unw] = in scope but unweighted)\n')

ABL_FULL = ' + '.join(ABL_ORDER)
print(f'📊 TIER 1 · block lattice — all {2 ** len(ABL_ORDER) - 1} subsets of '
      f'{{{", ".join(ABL_ORDER)}}}   (✓ = F1 above the P95 of its empirical null, §5.3; – = no null '
      'drawn under that protocol)')
print(f'   "MERT" in a set name = MERT-PCA{ABL_BLOCKS["MERT"].shape[1]} in the classification head (F1) and '
      f'MERT-full {ABL_BLOCKS_REG["MERT"].shape[1]} in the regression heads (r, R², RMSE) · emp_null_kind (Phase C3 '
      'rule, §5.3): exact = the primary null of the same width; borrowed = the smallest primary width ≥ the set '
      '(conservative); extrapolated = wider than every primary width; emp_null_d prints the width used)')
for _k in range(1, len(ABL_ORDER) + 1):
    for _combo in _abl_combos(ABL_ORDER, _k):
        ablate(_abl_stack(list(_combo)), ' + '.join(_combo), tier='1-blocks',
               X_reg=_abl_stack_reg(list(_combo)), **_abl_reg_kw(_combo))

df_abl_blocks = abl_frame().query('tier == "1-blocks"').reset_index(drop=True)
_delta_blocks = abl_vs_ref('1-blocks', ABL_FULL)

_T1_COLS = ['set_name', 'n_features', 'n_features_reg', 'block_weighting', 'f1_macro', 'emp_null_kind',
            'emp_null_d', 'emp_null_p', 'beats_emp_null',
            'r_valence', 'r2_val', 'r_arousal', 'r2_aro', 'rmse_aro', 'n_configs_selected']
for _pr in PROTOCOLS:
    print(f'\n── {PROTOCOL_LABEL[_pr]} ──')
    print(df_abl_blocks.query('protocol == @_pr').sort_values('f1_macro', ascending=False)[_T1_COLS]
          .round(4).to_string(index=False))
warn_negative_loo_r(df_abl_blocks)
abl_alpha_edge_report(df_abl_blocks, 'Tier 1')

_loo_full  = df_abl_blocks.query('protocol == "loo"   and set_name == @ABL_FULL').iloc[0]
_pw_full   = df_abl_blocks.query('protocol == "piece" and set_name == @ABL_FULL').iloc[0]
print(f'\n⚠️  OPTIMISM GAP on the full set: F1 piece={_pw_full["f1_macro"]:.4f} vs '
      f'loo={_loo_full["f1_macro"]:.4f} ({_pw_full["f1_macro"] - _loo_full["f1_macro"]:+.4f}); '
      f'R²_arousal {_loo_full["r2_aro"]:+.3f} (loo) → {_pw_full["r2_aro"]:+.3f} (piece).')
print('    LOOCV keeps the same piece\'s other two renditions in training. Quote the '
      'piece-wise\n    number whenever the claim is about NEW material.')


# ── Tier 2 · unique vs marginal contribution (derived — no extra model fits) ─
def _abl_get(nm, pr, tier='1-blocks'):
    return next(r for r in ABL_ROWS
                if r['tier'] == tier and r['set_name'] == nm and r['protocol'] == pr)


urows = []
for pr in PROTOCOLS:
    for b in ABL_ORDER:
        drop_name = ' + '.join([x for x in ABL_ORDER if x != b])
        r_full, r_drop, r_alone = _abl_get(ABL_FULL, pr), _abl_get(drop_name, pr), _abl_get(b, pr)
        dr = _delta_blocks.query('protocol == @pr and set_name == @drop_name')
        urows.append(dict(
            protocol=pr, block=b, n_features=r_alone['n_features'],
            # UNIQUE = what is lost when this block alone is removed from the full set
            unique_f1        = r_full['f1_macro']  - r_drop['f1_macro'],
            unique_r_valence = r_full['r_valence'] - r_drop['r_valence'],
            unique_r_arousal = r_full['r_arousal'] - r_drop['r_arousal'],
            unique_r2_val    = r_full['r2_val']    - r_drop['r2_val'],
            unique_r2_aro    = r_full['r2_aro']    - r_drop['r2_aro'],
            p_unique_f1      = float(dr['p_f1'].iloc[0]) if len(dr) else float('nan'),
            q_unique_f1      = float(dr['q_f1'].iloc[0]) if len(dr) else float('nan'),
            # MARGINAL = what this block delivers on its own, above the chance floor (mean)
            alone_f1         = r_alone['f1_macro'],
            marginal_f1      = r_alone['f1_macro'] - ABL_FLOOR[pr],
            alone_beats_chance = r_alone['beats_chance'],
            alone_beats_emp_null = r_alone['beats_emp_null'],
            alone_r_valence  = r_alone['r_valence'],
            alone_r_arousal  = r_alone['r_arousal'],
            alone_r2_val     = r_alone['r2_val'],
            alone_r2_aro     = r_alone['r2_aro'],
        ))
df_abl_unique = pd.DataFrame(urows)

# The unique R² (full − full∖block) is read off the Tier-1 rows, so it carries the DEPLOYED block
# weighting: every multi-block MERT-full head above was fitted with REG_BLOCK_WEIGHTING. The
# "alone" rows are single-block heads (never weighted).
print('\n📊 TIER 2 · UNIQUE (full − full∖block) vs MARGINAL (block alone − chance-floor mean)')
print(f'   regression columns use the deployed block weighting (REG_BLOCK_WEIGHTING={REG_BLOCK_WEIGHTING!r})')
for pr in PROTOCOLS:
    print(f'\n── {PROTOCOL_LABEL[pr]} ──')
    print(df_abl_unique.query('protocol == @pr')[
        ['block', 'n_features', 'unique_f1', 'q_unique_f1', 'unique_r2_aro',
         'alone_f1', 'marginal_f1', 'alone_beats_emp_null', 'alone_r2_aro']].round(4).to_string(index=False))

print('\n  Reading key — the four cells this table separates:')
print('   · unique > 0 & marginal > 0 : the tool carries signal AND no other tool replaces it')
print('   · unique ≈ 0 & marginal > 0 : the tool has signal, but it is REDUNDANT with the rest')
print('   · unique > 0 & marginal ≈ 0 : only useful in combination (a suppressor/interaction)')
print('   · unique ≈ 0 & marginal ≈ 0 : this tool adds nothing measurable at N=60')
print('  Pair with the §7.3b probe: low probe R² + unique > 0 = genuinely complementary.')
print(f'\n  ⚠️  At N={len(eids)} an |ΔF1| below ~0.02 is noise. Read q_unique_f1 (BH-FDR '
      'within tier),\n      and treat this lattice as a descriptive map, NOT as model selection.')

# ── The predictive interpretable layer of §4.1 IS the lattice set "CREPE + Essentia + DTW" (values) ──
_lat_interp_cols = list(CREPE_FORWARD) + list(ESSENTIA_FORWARD) + list(DTW_MODEL_COLS)
_lat_interp = _abl_stack(['CREPE', 'Essentia', 'DTW'])
assert _lat_interp.shape[1] == len(_lat_interp_cols) == len(INTERP_PRED_COLS), '§6.3: interpretable width mismatch'
assert np.allclose(_lat_interp[:, [_lat_interp_cols.index(c) for c in INTERP_PRED_COLS]], X_INTERP,
                   atol=1e-4, rtol=1e-4), '§6.3: X_INTERP (§4.1) ≠ the lattice set CREPE + Essentia + DTW'
print(f'\n✅ X_INTERP (§4.1, {X_INTERP.shape[1]} columns) == the lattice set "CREPE + Essentia + DTW" (up to the '
      '§2.5 float32 rounding of the CREPE block) — "Interpretable (27)" in §6.9 / §11.1 reads that lattice row')

# ── Tempogram sensitivity (Phase C2, C2 obligation 3): the timing block without the three tempogram features
#    that failed the manual audit (TEMPOGRAM_UNRELIABLE, §0.3). Each row is compared with its PARENT set
#    (re-scored inside this tier → memo hit / identical ridge), paired permutation p, BH within the tier. ──
TG_DROP = [c for c in DTW_MODEL_COLS if c in TEMPOGRAM_UNRELIABLE]
_tg_keep = [i for i, c in enumerate(DTW_MODEL_COLS) if c not in TG_DROP]
_TG_TIER = '1b-tempogram-sensitivity'
print(f'\n📊 TIER 1b · tempogram sensitivity — the timing block without {TG_DROP}')
if TG_DROP and _tg_keep:
    _tg_dtw = ABL_BLOCKS['DTW'][:, _tg_keep]
    _tg_sets = [('DTW', ABL_BLOCKS['DTW'], f'DTW ref-free − tempogram ({_tg_dtw.shape[1]}d)', _tg_dtw),
                ('CREPE + Essentia + DTW', _lat_interp,
                 f'CREPE + Essentia + DTW − tempogram ({ABL_BLOCKS["CREPE"].shape[1] + ABL_BLOCKS["Essentia"].shape[1] + _tg_dtw.shape[1]}d)',
                 np.hstack([ABL_BLOCKS['CREPE'], ABL_BLOCKS['Essentia'], _tg_dtw]))]
    _tg_ref = {}
    for _par, _Xp, _nm, _Xs in _tg_sets:
        ablate(_Xp, _par, tier=_TG_TIER, note='parent (re-scored inside the tier)')
        ablate(_Xs, _nm, tier=_TG_TIER, note=f'parent {_par!r} without {TG_DROP}')
        _tg_ref[_nm] = _par
    df_abl_tempogram_deltas = abl_vs_ref(_TG_TIER, None, ref_map=_tg_ref)
    df_abl_tempogram = abl_frame().query('tier == @_TG_TIER').reset_index(drop=True)
    for _pr in PROTOCOLS:
        for _par in ('DTW', 'CREPE + Essentia + DTW'):     # the parents must reproduce their Tier-1 rows
            assert np.array_equal(ABL_PRED[(_TG_TIER, _par, _pr)]['cls'], ABL_PRED[('1-blocks', _par, _pr)]['cls']), \
                f'§6.3: tempogram parent {_par!r} ({_pr}) differs from its Tier-1 row'
        print(f'   [{PROTOCOL_LABEL[_pr]}]')
        print(df_abl_tempogram_deltas.query('protocol == @_pr')[
            ['set_name', 'ref', 'd_f1', 'p_f1', 'q_f1', 'd_r2_val', 'p_r2_val', 'd_r2_aro', 'p_r2_aro', 'q_r2_aro']]
            .round(4).to_string(index=False))
    df_abl_tempogram_deltas.to_csv(os.path.join(OUTPUT_DIR, 'ablation_tempogram_sensitivity.csv'), index=False)
    print('   Δ = without the tempogram − parent; p = paired permutation (§6.3), q = BH within the tier.')
else:
    df_abl_tempogram = df_abl_tempogram_deltas = pd.DataFrame()
    print('   ➖ skipped: no tempogram column among DTW_MODEL_COLS, or nothing would be left')

df_abl_blocks.to_csv(os.path.join(OUTPUT_DIR, 'ablation_blocks.csv'), index=False)
_delta_blocks.to_csv(os.path.join(OUTPUT_DIR, 'ablation_blocks_vs_full.csv'), index=False)
df_abl_unique.to_csv(os.path.join(OUTPUT_DIR, 'ablation_unique_marginal.csv'), index=False)

# ── Figure 1 · the lattice, both protocols ─────────────────────────────────
_ord = (df_abl_blocks.query('protocol == "piece"')
        .assign(k=lambda d: d.set_name.str.count(r'\+') + 1)
        .sort_values(['k', 'f1_macro'], ascending=[True, False])['set_name'].tolist())
fig, ax = plt.subplots(figsize=(13, 5.5))
xs = np.arange(len(_ord))
for off, pr, col in [(-0.2, 'piece', '#e07b39'), (0.2, 'loo', '#2a6dd9')]:
    sub  = df_abl_blocks.query('protocol == @pr').set_index('set_name').reindex(_ord)
    ax.bar(xs + off, sub['f1_macro'], 0.38, label=f'{pr}', color=col,
           edgecolor='#333', linewidth=0.5)
    # clipped at 0: a percentile CI can sit fractionally the wrong side of the point
    # estimate, and matplotlib rejects a negative yerr outright.
    ax.errorbar(xs + off, sub['f1_macro'],
                yerr=[np.clip(sub['f1_macro'] - sub['f1_lo'], 0, None),
                      np.clip(sub['f1_hi'] - sub['f1_macro'], 0, None)],
                fmt='none', ecolor='#333', elinewidth=0.8, capsize=2)
ax.axhline(ABL_FLOOR['piece'], color='#b00020', ls='--', lw=1.1,
           label=f'chance mean (piece) = {ABL_FLOOR["piece"]:.3f}')
ax.axhline(ABL_FLOOR_P95['piece'], color='#b00020', ls=':', lw=1.1,
           label=f'Dummy chance P95 (piece) = {ABL_FLOOR_P95["piece"]:.3f}')
for _d, _ls in ((10, '-.'), (25, (0, (5, 1)))):    # Phase C3: the exact-width PRIMARY null, not the map
    if ('piece', _d) in EMP_NULL_PRIMARY:
        ax.axhline(EMP_NULL_PRIMARY[('piece', _d)]['p95'], color='#6a1b9a', ls=_ls, lw=1.1,
                   label=f'empirical-null P95 (piece, exact-width d = {_d}, '
                         f'{EMP_NULL_PRIMARY[("piece", _d)]["n_draws"]} draws) = {EMP_NULL_PRIMARY[("piece", _d)]["p95"]:.3f}')
ax.set_xticks(xs)
ax.set_xticklabels(_ord, rotation=32, ha='right', fontsize=7.5)
ax.set_ylabel('F1-macro ↑')
ax.set_title('§6.3 Tier 1 — block lattice: every subset of {MERT, CREPE, Essentia, DTW}\n'
             f'{DEFAULT_CLF_NAME} (nested selection) · bars = leave-one-piece-out (primary) vs LOOCV · '
             'whiskers = bootstrap 95% CI', fontsize=11)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_ablation_lattice.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2 · unique vs marginal, per block ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, pr in zip(axes, PROTOCOLS):
    sub = df_abl_unique.query('protocol == @pr').set_index('block').reindex(ABL_ORDER)
    xb  = np.arange(len(ABL_ORDER))
    ax.bar(xb - 0.2, sub['unique_f1'],   0.38, label='unique (full − full∖block)',
           color='#8e44ad', edgecolor='#333', linewidth=0.5)
    ax.bar(xb + 0.2, sub['marginal_f1'], 0.38, label='marginal (alone − floor mean)',
           color='#2a9d2a', edgecolor='#333', linewidth=0.5)
    for xi, (u, q) in enumerate(zip(sub['unique_f1'], sub['q_unique_f1'])):
        if np.isfinite(q) and q < 0.05:
            ax.text(xi - 0.2, u, '*', ha='center',
                    va='bottom' if u >= 0 else 'top', fontsize=13, color='#b00020')
    ax.axhline(0, color='#333', lw=0.8)
    ax.axhspan(-0.02, 0.02, color='#999', alpha=0.15, zorder=0)
    ax.set_xticks(xb)
    ax.set_xticklabels(ABL_ORDER)
    ax.set_title(f'protocol: {pr}' + (' (primary)' if pr == 'piece' else ' (optimistic)'))
    ax.grid(axis='y', alpha=0.25)
axes[0].set_ylabel('ΔF1-macro')
axes[0].legend(fontsize=8, loc='upper left')
fig.suptitle('§6.3 Tier 2 — unique vs marginal contribution per tool block\n'
             'grey band = ±0.02 noise floor at N=60 · * = BH-FDR q<.05 on the unique gap',
             fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_ablation_unique_marginal.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ §6.3 — ablation_blocks.csv, ablation_blocks_vs_full.csv, '
      'ablation_unique_marginal.csv, ablation_tempogram_sensitivity.csv,\n   fig_ablation_lattice.png, '
      'fig_ablation_unique_marginal.png')

## Part 6 · Prediction — §6.4 Configuration ablation (feature side)

Tier 1 asks *which tools*; this asks *whether the knobs inside them were load-bearing choices*.
Every setting here is currently a **hand-picked constant** somewhere in Part 2, and each one silently
changes what the models receive:

| Sub-tier | Knob | Where it lives |
|---|---|---|
| `4A-mert-pca` | `MERT_PCA_VAR` — 0.90 / **0.95 (deployed)** / 0.99 / full 768 | §2.1b |
| `4A-essentia-curation` | the hand-picked `ESSENTIA_CANDIDATES` vs gating the **whole** extracted block (diagnostic columns excluded) | §2.3b |
| `4A-dtw-family` | reference-free (**deployed**) / DTW-deviation / both — i.e. `DTW_DEV_IN_MODELS` | §2.4b |
| `4D-balancing` | class balancing on `X_A`, `X_C`, interpretable: **nested selection (deployed, reference)** / NB empirical priors (Phase A default) / NB balanced priors / SVM-Linear C=1 unbalanced (pre-audit) / SVM-Linear balanced + inner-CV C — classification only; descriptive | §5.1 / §6.1 (A5) |
| `4B-gate` | `corr_max` of the shared forwarding gate — 0.80 / 0.90 / 0.95 / 1.0 (= prune off) | §0.3 gate |
| `4E-block-weighting` | `REG_BLOCK_WEIGHTING` — every multi-block MERT-full regression head (the 7 lattice subsets with MERT + ≥ 1 other block, i.e. incl. `X_C_REG` / `X_D_REG` / `X_F_REG`) fitted **unweighted** and with **equal-share block weights**; Δ = weighted − unweighted, paired permutation on the same targets, BH within (tier, protocol) over r / R² only (the classification head is identical by construction); α grid-edge hits printed; descriptive — the deployed setting is the config value | §0.3 / §5.1 |

**Why `4E` (block weighting).** In the Phase B run (N = 118, `piece`) every regression set containing
MERT-full scored R²_arousal ≈ 0.45–0.46 regardless of the other blocks, while Essentia + DTW alone reached
0.67: after `StandardScaler` the 768 MERT columns carry 768 units of variance against ~10 per interpretable
block, so one ridge α lets MERT dominate. The fix scales block *b* by w_b = √(P / (B·p_b)) after the
scaler (equal variance share P/B per block, total P, group ridge with α_b ∝ B·p_b / P). It is leakage-free
because the weights depend on the column counts only and are applied inside every fold. The deployed
member of every 4E pair is asserted to reproduce its Tier-1 lattice row exactly.

**A note the `4B-gate` sweep exists to surface:** §2.2b calls the gate with a hard-coded
`corr_max=0.95` for CREPE, while §2.3b defines `CORR_MAX = 0.90` and uses it for Essentia **and**
DTW. The asymmetry is deliberate — the comment in §2.2b argues a named technique should only be
dropped when it is a true duplicate — but `CORR_MAX` is defined in a cell that runs *after* §2.2b, so
CREPE could not have used it in any case. The sweep turns that from an accident of cell order into a
measured choice; the `gate deployed` row is the current mixed setting.

**Removed in Phase B:** `4A-crepe-portamento` (raw count vs per second) — the B9 portamento features
are rates / shares by construction.

**Cost note.** Everything here is free (re-gating and re-assembly only). The knobs that need
re-extraction from audio — CREPE `step_ms`, Essentia frame sizes — and the knobs that
are now cheap from the cached CREPE tracks but not yet approved (voicing threshold, the B8/B9
descriptor constants), plus the tempogram window / tempo prior, are **not** run here and stay listed
as pending. **Per-layer MERT is no longer pending (Phase C1):** §2.1c caches the time-mean of all 13
hidden states once, and §7.2 runs the *diagnostic* layer study (tier `6-mert-layers`); the deployed
`MERT_LAYERS = [5, 6, 7]` is not changed by it.

In [ ]:
# ── §6.4 Configuration ablation (Tier 4A feature-side + 4B shared gate + 4D balancing
#    + 4E block weighting) ──
# NEEDS : ablate, abl_alpha_edge_report, abl_frame, abl_vs_ref, ABL_BLOCKS, ABL_ORDER, ABL_PRED,
#          CORR_MAX, CREPE_CANDIDATES,
#          DEFAULT_CLF_NAME, df_crepe, df_crepe_fwd, df_dtw, df_dtw_fwd, df_ess_fwd,
#          df_essentia, df_mert, DTW_CANDIDATES, DTW_DEV_IN_MODELS, DTW_FORWARD_DEV,
#          DTW_FORWARD_REFFREE, DTW_REFFREE, eids, ESSENTIA_CANDIDATES, ESSENTIA_DIAGNOSTIC,
#          make_nb_clf, MERT_REG_REPR, NB_CLF_NAME, OUTPUT_DIR, PROTOCOLS, PROTOCOL_LABEL,
#          REG_BLOCK_WEIGHTING, STD_MIN, TUNE_GRID_C,
#          unsup_forward_select, warn_negative_loo_r, X_A, X_C, X_MERT_FULL, Y, ABL_FULL, _abl_combos,
#          _abl_mat, _abl_reg_kw, _abl_stack, _abl_stack_reg, _sc_f1
# MAKES : ABL_GATE_AUDIT, df_abl_balancing, df_abl_bw, df_abl_config, _abl_gate_mix,
#          _BAL_CLFS, _BAL_SETS, _BW_REF · ablation_config.csv, ablation_config_deltas.csv, 1 figure
# FEEDS : §10.2 (master consolidation)
# ═══════════════════════════════════════════════════════════════════════════════
# §6.4 · Configuration ablation — were the Part-2 constants load-bearing choices?
# ═══════════════════════════════════════════════════════════════════════════════
# Tier 1 asked WHICH TOOLS. This asks whether the hand-picked knobs INSIDE them
# matter. Everything here is free (re-gate + re-assemble). Knobs that would need
# re-extraction from audio are listed at the end as explicitly pending, not silently
# skipped.
from sklearn.decomposition import PCA

# The gate writes an audit CSV named after its `name=` argument. Sweeping it would
# OVERWRITE the §2.3b / §2.4b audit trail, so every sweep call writes into a
# subfolder instead. The deployed audit files stay untouched.
ABL_GATE_AUDIT = os.path.join(OUTPUT_DIR, 'ablation_gate_audit')
os.makedirs(ABL_GATE_AUDIT, exist_ok=True)

_cfg_tiers = []

# ── 4A · MERT dimensionality (MERT_PCA_VAR) ────────────────────────────────
# The deployed 0.95 set is X_A itself — reused verbatim rather than recomputed, so
# the reference row IS the matrix Part 6 fitted. Unchanged in Phase B: here BOTH heads see
# each representation as listed (no X_reg) — this tier is the evidence for B-MERTFULL.
print('📊 4A · MERT dimensionality — how much of the 768-dim space is needed?')
_M = df_mert.reindex(eids).to_numpy(dtype=float)
ablate(X_A, f'MERT PCA 95% (deployed, {X_A.shape[1]}d)', tier='4A-mert-pca')
for _v in (0.90, 0.99):
    _Z = PCA(n_components=_v, svd_solver='full').fit_transform(_M)
    ablate(_Z, f'MERT PCA {_v:.0%} ({_Z.shape[1]}d)', tier='4A-mert-pca')
ablate(X_MERT_FULL, 'MERT full 768d (no reduction)', tier='4A-mert-pca',
       note='768 dims at N=60 — heavily over-parameterised, read with care')
_cfg_tiers.append(('4A-mert-pca', f'MERT PCA 95% (deployed, {X_A.shape[1]}d)'))

# ── 4A · Essentia curation (11 hand-picked candidates vs the whole block) ──
print(f'\n📊 4A · Essentia curation — is the a-priori {len(ESSENTIA_CANDIDATES)}-candidate '
      'shortlist doing work?')
# the whole block = every extracted column EXCEPT the diagnostic-only ones (B11 raw flux)
_ess_whole = df_essentia.drop(columns=ESSENTIA_DIAGNOSTIC)
_ess_full_fwd = unsup_forward_select(_ess_whole.reindex(eids), corr_max=CORR_MAX,
                                     name='ablation_essentia_fullblock', eps=STD_MIN,
                                     out_dir=ABL_GATE_AUDIT, verbose=False)
_ref_ess = f'Essentia curated (deployed, {df_ess_fwd.shape[1]}d)'
ablate(_abl_mat(df_ess_fwd), _ref_ess, tier='4A-essentia-curation')
ablate(_abl_mat(df_essentia[_ess_full_fwd]),
       f'Essentia whole block, gated ({len(_ess_full_fwd)}d)', tier='4A-essentia-curation',
       note=(f'gate over all {_ess_whole.shape[1]} extracted non-diagnostic columns instead of '
             f'the {len(ESSENTIA_CANDIDATES)} curated'))
assert not set(_ess_full_fwd) & set(ESSENTIA_DIAGNOSTIC), '§6.4: a diagnostic column was gated in'
_cfg_tiers.append(('4A-essentia-curation', _ref_ess))

# ── 4A · DTW family (the DTW_DEV_IN_MODELS switch, measured) ───────────────
print('\n📊 4A · DTW family — reference-free vs within-piece deviation')
_ref_dtw = f'DTW ref-free (deployed, {len(DTW_FORWARD_REFFREE)}d)'
ablate(_abl_mat(df_dtw[DTW_FORWARD_REFFREE]), _ref_dtw, tier='4A-dtw-family')
if DTW_FORWARD_DEV:
    ablate(_abl_mat(df_dtw[DTW_FORWARD_DEV]),
           f'DTW deviation only ({len(DTW_FORWARD_DEV)}d)', tier='4A-dtw-family',
           note='0/1 at MEC BY CONSTRUCTION — partly encodes the condition')
    ablate(_abl_mat(df_dtw[DTW_FORWARD_REFFREE + DTW_FORWARD_DEV]),
           f'DTW both families ({len(DTW_FORWARD_REFFREE) + len(DTW_FORWARD_DEV)}d)',
           tier='4A-dtw-family', note='what DTW_DEV_IN_MODELS=True would deploy')
    print('   ⚠️  A HIGH score for the deviation family is NOT a virtue: its MEC rows are the')
    print('       self-alignment identity (§2.4), so it partly reads off the condition label.')
    print('       Only the ref-free family is an honest predictor of unseen material.')
else:
    print('   ➖ no DTW-deviation column survived the §2.4b gate — family comparison skipped')
_cfg_tiers.append(('4A-dtw-family', _ref_dtw))

# ── 4A · CREPE portamento normalisation — REMOVED in Phase B (B9) ─────────
# The Phase A tier compared the raw portamento COUNT with count / second. The B9 detector
# reports crepe_portamento_rate (glides per second of voiced time) and _share (glides / all
# transitions), i.e. it is length-normalised by construction, so the comparison is obsolete.
print('\nℹ️  4A-crepe-portamento removed (B9: the portamento features are rates/shares by construction)')

# ── 4B · the shared forwarding gate (corr_max) ─────────────────────────────
print('\n📊 4B · shared gate corr_max — how much redundancy pruning is right?')


def _abl_gate_mix(cm):
    """Re-gate the three TABULAR blocks at one corr_max and stack them.

    MERT is excluded: its gate is PCA (§2.1b), not a correlation prune, and is swept
    separately above. DTW is filtered by DTW_DEV_IN_MODELS exactly as §2.4b does. Every
    block is gated over its a-priori CANDIDATE list, so diagnostic-only columns
    (CREPE_DIAGNOSTIC, ESSENTIA_DIAGNOSTIC, TIMING_DIAGNOSTIC) can never enter."""
    cr = unsup_forward_select(df_crepe.reindex(eids), corr_max=cm,
                              name=f'ablation_crepe_c{cm:.2f}', eps=STD_MIN,
                              out_dir=ABL_GATE_AUDIT, order=CREPE_CANDIDATES, verbose=False)
    es = unsup_forward_select(df_essentia.reindex(eids), corr_max=cm,
                              name=f'ablation_essentia_c{cm:.2f}', eps=STD_MIN,
                              out_dir=ABL_GATE_AUDIT, order=ESSENTIA_CANDIDATES, verbose=False)
    dt = unsup_forward_select(df_dtw.reindex(eids), corr_max=cm,
                              name=f'ablation_dtw_c{cm:.2f}', eps=STD_MIN,
                              out_dir=ABL_GATE_AUDIT, order=DTW_CANDIDATES, verbose=False)
    dt = [c for c in dt if DTW_DEV_IN_MODELS or c in DTW_REFFREE]
    X  = np.hstack([_abl_mat(df_crepe[cr]), _abl_mat(df_essentia[es]), _abl_mat(df_dtw[dt])])
    return X, (len(cr), len(es), len(dt))


_ref_gate = (f'gate deployed — CREPE .95 / Ess {CORR_MAX:g} / DTW {CORR_MAX:g} '
             f'({df_crepe_fwd.shape[1]}+{df_ess_fwd.shape[1]}+{df_dtw_fwd.shape[1]}d)')
ablate(np.hstack([_abl_mat(df_crepe_fwd), _abl_mat(df_ess_fwd), _abl_mat(df_dtw_fwd)]),
       _ref_gate, tier='4B-gate',
       note='the CURRENT mixed setting — CREPE is hard-coded to 0.95 in §2.2b')
for _cm in (0.80, 0.90, 0.95, 1.00):
    _Xg, _n = _abl_gate_mix(_cm)
    ablate(_Xg, f'gate corr_max={_cm:.2f} uniform ({_n[0]}+{_n[1]}+{_n[2]}d)', tier='4B-gate',
           note='no redundancy prune' if _cm >= 1.0 else '')
_cfg_tiers.append(('4B-gate', _ref_gate))

# ── 4D · class balancing (A5) — classification only ─────────────────────────
# How does class balancing behave? Five classifiers on three matrices, each compared
# against the DEPLOYED default — the nested selection (B-NMS) — on the SAME matrix; one BH
# family per protocol across the whole tier. The ridge head is identical across these
# rows (same X), so it is not refitted (regression=False → r/R² left NaN).
# History: in the Phase A run this tier showed the pre-declared NB default below chance
# under 'piece'; the approved response was NOT to adopt the winning row (a post-hoc pick
# on the same CV) but to move the choice INSIDE every outer fold (nested selection, §5.1).
print('\n📊 4D · class balancing — does balancing help? (reference = nested selection on the same X)')
_BAL_SETS = [('X_A', np.asarray(X_A, dtype=float)),
             ('X_C', np.asarray(X_C, dtype=float)),
             # "interpretable" = CREPE-fwd + Essentia-fwd + DTW model cols = INTERP_PRED_COLS of §4.1
             # (the predictive layer) — identical to the 4B 'gate deployed' matrix and to the
             # lattice row 'CREPE + Essentia + DTW'
             ('Interp', np.hstack([_abl_mat(df_crepe_fwd), _abl_mat(df_ess_fwd),
                                   _abl_mat(df_dtw_fwd)]))]
_BAL_CLFS = [('Nested selection (deployed)', None, None),          # make_clf=None → §5.1 default
             ('NB (Phase A default, empirical priors)', make_nb_clf, None),
             ('NB balanced priors', lambda: GaussianNB(priors=[0.5, 0.5]), None),
             ('SVM-Linear C=1 unbalanced (pre-audit)', lambda: SVC(kernel='linear', C=1.0), None),
             ('SVM-Linear balanced + inner-CV C',
              lambda: SVC(kernel='linear', class_weight='balanced'), TUNE_GRID_C)]
_BAL_REF = {}
for _sn, _Xb in _BAL_SETS:
    for _cn, _mk, _tg in _BAL_CLFS:
        _nm = f'{_sn} · {_cn}'
        ablate(_Xb, _nm, tier='4D-balancing', make_clf=_mk, tune_grid=_tg, clf_name=_cn,
               regression=False, note='classification only — ridge head identical across rows')
        _BAL_REF[_nm] = f'{_sn} · {_BAL_CLFS[0][0]}'
# Consistency guard: the nested selection on X_A must equal the Tier-1 'MERT' row exactly
# (same procedure, same array, same protocol). X_C is only allclose to the lattice stack
# (§2.5 rounds every block to float32, the lattice keeps Essentia/DTW at float64), so that
# pair is reported, not asserted.
for _pr in PROTOCOLS:
    assert np.array_equal(ABL_PRED[('4D-balancing', f'X_A · {_BAL_CLFS[0][0]}', _pr)]['cls'],
                          ABL_PRED[('1-blocks', 'MERT', _pr)]['cls']), \
        f'§6.4: 4D nested selection on X_A differs from the lattice row MERT ({_pr})'
    _dxc = (_sc_f1(Y, ABL_PRED[('4D-balancing', f'X_C · {_BAL_CLFS[0][0]}', _pr)]['cls'])
            - _sc_f1(Y, ABL_PRED[('1-blocks', ABL_FULL, _pr)]['cls']))
    print(f'   {_pr}: 4D X_C nested selection vs lattice full set ΔF1 = {_dxc:+.2e} '
          '(float32 X_C vs float64 lattice blocks)')
print('   ✅ 4D nested selection on X_A reproduces the Tier-1 lattice predictions exactly')
_bal_deltas = abl_vs_ref('4D-balancing', None, ref_map=_BAL_REF)
df_abl_balancing = abl_frame().query('tier == "4D-balancing"').reset_index(drop=True)

# ── 4E · block weighting of the multi-block MERT-full regression heads ─────────
# Every lattice subset that concatenates MERT-full with ≥ 1 other tool block, fitted twice:
# UNWEIGHTED (the Phase B heads) and with the §5.1 equal-share block weights. The reference of
# each pair is its unweighted twin, so Δ = weighted − unweighted (the effect of the weighting).
# Classification is identical by construction (same X → NestedSelect memo hit), so only the
# four regression metrics are tested: paired permutation on the SAME targets, BH within
# (tier, protocol). Descriptive — the deployed setting is REG_BLOCK_WEIGHTING (§0.3).
print('\n📊 4E · block weighting — multi-block MERT-full regression heads, weighted vs unweighted')
_BW_REF, _bw_deltas = {}, pd.DataFrame()
if MERT_REG_REPR != 'full':
    print(f"   ➖ skipped: MERT_REG_REPR={MERT_REG_REPR!r} — the weighting applies to MERT-full heads only")
else:
    _bw_sets = [list(_c) for _k in range(2, len(ABL_ORDER) + 1)
                for _c in _abl_combos(ABL_ORDER, _k) if 'MERT' in _c]
    for _combo in _bw_sets:
        _base = ' + '.join(_combo)
        for _wt, _lab in (('none', 'unweighted'), ('equal', 'weighted')):
            ablate(_abl_stack(_combo), f'{_base} · {_lab}', tier='4E-block-weighting',
                   X_reg=_abl_stack_reg(_combo), **_abl_reg_kw(_combo, weighting=_wt),
                   note=('deployed' if _wt == REG_BLOCK_WEIGHTING else 'alternative')
                        + ' · classification head identical across the pair')
        _BW_REF[f'{_base} · weighted'] = f'{_base} · unweighted'
    _bw_deltas = abl_vs_ref('4E-block-weighting', None, ref_map=_BW_REF,
                            pcols=['p_r_valence', 'p_r_arousal', 'p_r2_val', 'p_r2_aro'])
    # the deployed member of every pair must reproduce the Tier-1 lattice row exactly
    for _combo in _bw_sets:
        _base = ' + '.join(_combo)
        _dep  = f'{_base} · {"weighted" if REG_BLOCK_WEIGHTING == "equal" else "unweighted"}'
        for _pr in PROTOCOLS:
            for _k in ('val', 'aro'):
                assert np.array_equal(ABL_PRED[('4E-block-weighting', _dep, _pr)][_k],
                                      ABL_PRED[('1-blocks', _base, _pr)][_k]), \
                    f'§6.4: 4E deployed head {_dep!r} differs from the lattice row ({_pr}, {_k})'
    print('   ✅ the deployed member of every 4E pair reproduces its Tier-1 lattice row exactly')
df_abl_bw = abl_frame().query('tier == "4E-block-weighting"').reset_index(drop=True)

# ── Deltas + BH-FDR within each sub-tier, then one table ───────────────────
# A sub-tier can legitimately hold only its reference (e.g. the DTW family split when
# no deviation column survived the gate) — abl_vs_ref then returns an empty frame.
_dlist = [d for d in (abl_vs_ref(t, r) for t, r in _cfg_tiers) if len(d)]
if len(_bal_deltas):
    _dlist.append(_bal_deltas)
if len(_bw_deltas):
    _dlist.append(_bw_deltas)
_cfg_deltas = (pd.concat(_dlist, ignore_index=True) if _dlist else
               pd.DataFrame(columns=['tier', 'set_name', 'protocol', 'd_f1', 'q_f1',
                                     'd_r_arousal', 'q_r_arousal', 'd_r2_aro', 'q_r2_aro']))
df_abl_config = (abl_frame()
                 .query('tier.str.startswith("4A") or tier.str.startswith("4B") '
                        'or tier.str.startswith("4D") or tier.str.startswith("4E")', engine='python')
                 .reset_index(drop=True))

print('\n📊 §6.4 CONFIGURATION ABLATION — piece-wise (PRIMARY) then LOOCV, '
      'Δ vs each sub-tier reference')
for _t, _r in _cfg_tiers:
    print(f'\n── {_t}   (reference: {_r}) ──')
    for _pr in PROTOCOLS:
        print(f'   [{PROTOCOL_LABEL[_pr]}]')
        _tab = df_abl_config.query('tier == @_t and protocol == @_pr')[
            ['set_name', 'n_features', 'f1_macro', 'beats_emp_null', 'r_valence', 'r2_val',
             'r_arousal', 'r2_aro']]
        print(_tab.round(4).to_string(index=False))
    warn_negative_loo_r(df_abl_config.query('tier == @_t'))
    for _pr in PROTOCOLS:
        _d = (_cfg_deltas.query('tier == @_t and protocol == @_pr')
              if len(_cfg_deltas) else _cfg_deltas)
        if len(_d):
            print(f'   Δ vs reference ({_pr}):')
            print(_d[['set_name', 'd_f1', 'q_f1', 'd_r2_val', 'q_r2_val', 'd_r2_aro', 'q_r2_aro']]
                  .round(4).to_string(index=False))

print('\n── 4D-balancing   (reference per matrix: nested selection, deployed) ──')
for _pr in PROTOCOLS:
    print(f'   [{PROTOCOL_LABEL[_pr]}]')
    print(df_abl_balancing.query('protocol == @_pr')[
        ['set_name', 'f1_macro', 'f1_lo', 'f1_hi', 'beats_emp_null', 'n_tune_skipped', 'selected']]
        .round(4).to_string(index=False))
    _d = _bal_deltas.query('protocol == @_pr') if len(_bal_deltas) else _bal_deltas
    if len(_d):
        print(f'   ΔF1 vs nested selection ({_pr}) — BH-FDR over the whole tier:')
        print(_d[['set_name', 'd_f1', 'p_f1', 'q_f1']].round(4).to_string(index=False))
print(f'   ⚠️  Descriptive only — the default classifier stays {DEFAULT_CLF_NAME}. Switching to the')
print('       best FIXED row of this table after seeing it would be a post-hoc selection on the same CV;')
print(f'       {NB_CLF_NAME} is kept as the Phase A pre-declared default that failed under piece.')

if len(df_abl_bw):
    print(f'\n── 4E-block-weighting   (reference per set: its unweighted twin; deployed = '
          f'{REG_BLOCK_WEIGHTING!r}) ──')
    for _pr in PROTOCOLS:
        print(f'   [{PROTOCOL_LABEL[_pr]}]')
        print(df_abl_bw.query('protocol == @_pr')[
            ['set_name', 'n_features_reg', 'block_weighting', 'r_valence', 'r2_val', 'r_arousal',
             'r2_aro', 'alpha_val_median', 'alpha_aro_median', 'n_alpha_edge_val', 'n_alpha_edge_aro']]
            .round(4).to_string(index=False))
        _d = _bw_deltas.query('protocol == @_pr') if len(_bw_deltas) else _bw_deltas
        if len(_d):
            print(f'   Δ weighted − unweighted ({_pr}) — paired permutation, BH-FDR over the tier '
                  '(regression metrics only):')
            print(_d[['set_name', 'd_r_valence', 'q_r_valence', 'd_r2_val', 'q_r2_val',
                      'd_r_arousal', 'q_r_arousal', 'd_r2_aro', 'q_r2_aro']]
                  .round(4).to_string(index=False))
    warn_negative_loo_r(df_abl_bw)
    abl_alpha_edge_report(df_abl_bw, 'tier 4E')
    print('   ⚠️  Descriptive — the deployed weighting is the §0.3 config value, not the winner of this table.')

df_abl_config.to_csv(os.path.join(OUTPUT_DIR, 'ablation_config.csv'), index=False)
_cfg_deltas.to_csv(os.path.join(OUTPUT_DIR, 'ablation_config_deltas.csv'), index=False)

# ── Figure · one panel per sub-tier, F1 under both protocols ───────────────
_fig_tiers = _cfg_tiers + [('4D-balancing', 'nested selection per matrix')]
fig, axes = plt.subplots(1, len(_fig_tiers),
                         figsize=(4.4 * len(_fig_tiers), 4.6), squeeze=False)
for ax, (_t, _r) in zip(axes[0], _fig_tiers):
    sub  = df_abl_config.query('tier == @_t')
    names = sub.query('protocol == "piece"')['set_name'].tolist()
    xs    = np.arange(len(names))
    for off, pr, col in [(-0.2, 'piece', '#e07b39'), (0.2, 'loo', '#2a6dd9')]:
        v = sub.query('protocol == @pr').set_index('set_name').reindex(names)['f1_macro']
        ax.bar(xs + off, v, 0.38, label=pr, color=col, edgecolor='#333', linewidth=0.5)
    ax.set_xticks(xs)
    ax.set_xticklabels([n.split('(')[0].strip() for n in names], rotation=28,
                       ha='right', fontsize=7)
    ax.set_title(_t, fontsize=9)
    ax.grid(axis='y', alpha=0.25)
axes[0][0].set_ylabel('F1-macro ↑')
axes[0][0].legend(fontsize=8)
fig.suptitle('§6.4 Configuration ablation — the Part-2 constants, measured '
             '(reference = the deployed setting in each panel)', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_ablation_config.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n⏭  STILL PENDING (needs re-extraction from audio, not runnable from cached features):')
print('   · CREPE  : step_ms 5/10/20 (re-extraction). Voicing threshold 0.3/0.5/0.7 and the B2/B8b/B9b')
print('              descriptor constants (PLATEAU_*, VIB_MIN_NOTE_MS, VIB_FIT_R2_MIN, VIB_MIN_EXTENT, GLIDE_*)')
print('              are cheap from crepe_tracks_v2.npz — not run as an ablation tier (§2.2 prints report-only')
print('              sensitivities and the MEC calibration grids). The B2 re-articulation detector is DISABLED')
print('              since B3 (REART_ENABLED = False); §2.2 reports it (REART_DIP_DB 3/6/9 on rms10_v3.npz)')
print('   · Essentia: ESS_FRAME / ESS_HOP / ESSENTIA_LOUD_SR (the B12 DR gate is swept in §2.3)')
print('   · timing : DTW_TG_WIN (151), the tempo prior centre (100 BPM) and its 40-200 BPM range')
print('   · MERT   : per-layer — NO LONGER PENDING (Phase C1): §2.1c caches the time-mean of all 13 hidden')
print('              states (emb_mert_layers_v3.npz) and §7.2 runs the DIAGNOSTIC layer study (tier')
print('              6-mert-layers); decision recorded in Phase C2: MERT_LAYERS stays [5, 6, 7]')
print('\n✅ §6.4 — ablation_config.csv (incl. 4D-balancing), ablation_config_deltas.csv, '
      'fig_ablation_config.png')

## Part 6 · Prediction — §6.5 Target-side ablation · Neutral radius · normalized vs raw

The tiers above vary the **features**. This one varies the **ground truth itself** — the two places
where a constant chosen by hand decides what the classifier is even asked to predict.

**`NEUTRAL_RADIUS` — 0.5 / 0.75 / 1.0.** The deployed 0.75 was chosen geometrically and never tuned
(a standing open issue). It sets how much of the VA plane collapses into `Neutral`, so it directly
controls class balance: a larger radius manufactures a bigger, easier `Neutral` class while starving
the four quadrants. Macro-F1 is therefore reported **twice** — with and without `Neutral` — because
the with-Neutral number can rise for the wrong reason.

**Normalized vs raw ground truth.** §1.1b made per-rater-normalized means the ground truth. The
validation in §1.2b already shows how closely the two agree (raw↔norm r and the number of excerpts
whose class / quadrant changes are printed there for the current panel); this rebuilds
`Y` and the regression targets from the **raw** means and re-scores, turning that agreement from a
diagnostic into a line of the ablation table. The `X_C_REG` regression rows use the deployed block
weighting (`REG_BLOCK_WEIGHTING`, §6.3); the MERT-alone row is a single block and is never weighted.

> **Why no p-values in this section.** Both sweeps change `Y` (and the raw sweep changes the
> regression targets too). Two runs with different ground truths are not paired on the same target,
> so a paired permutation test would be meaningless — `ablate` records the scores, the comparison is
> reported as **descriptive**, and `abl_vs_ref` would correctly refuse to pair them.

In [ ]:
# ── §6.5 Target-side ablation (Tier 4C) — Neutral radius · normalized vs raw ──
# NEEDS : ablate, ABL_ORDER, ABL_PRED, df_expr, eids, EMOTION_LABELS, MERT_REG, NEUTRAL_RADIUS,
#          OUTPUT_DIR, PROTOCOLS, PROTOCOL_LABEL, va_to_quadrant, VA_MID, warn_negative_loo_r, X_A,
#          X_C, X_C_REG, Y, _abl_reg_kw
# MAKES : ABL_RADII, df_abl_radius, df_abl_gt, NEUTRAL_IDX, _f1_ex_neutral, _labels_at,
#          _Y_at · ablation_neutral_radius.csv, ablation_ground_truth.csv, 1 figure
# FEEDS : §10.2 (master consolidation), open issue "Neutral radius un-tuned"
# ═══════════════════════════════════════════════════════════════════════════════
# §6.5 · Target-side ablation — the constants that decide WHAT is predicted
# ═══════════════════════════════════════════════════════════════════════════════
ABL_RADII   = (0.5, 0.75, 1.0)
NEUTRAL_IDX = list(EMOTION_LABELS).index('Neutral')


def _labels_at(vals, aros, radius):
    """Re-derive the 5-class labels at an arbitrary Neutral radius.

    Mirrors §1.2 `derive_labels` exactly — same hypot distance from (VA_MID, VA_MID),
    same quadrant rule via va_to_quadrant. The assert below proves the mirror is
    faithful; if §1.2 ever changes, this cell fails loudly instead of drifting."""
    out = []
    for v, a in zip(np.asarray(vals, dtype=float), np.asarray(aros, dtype=float)):
        out.append(['Neutral'] if np.hypot(v - VA_MID, a - VA_MID) <= radius
                   else [va_to_quadrant(v, a)])
    return out


def _Y_at(vals, aros, radius):
    return MultiLabelBinarizer(classes=EMOTION_LABELS).fit_transform(
        _labels_at(vals, aros, radius))


assert np.array_equal(_Y_at(df_expr['valence_mean'], df_expr['arousal_mean'],
                            NEUTRAL_RADIUS), Y), \
    '§6.5: label re-derivation does not reproduce the §1.3 Y — the sweep would be invalid'
print(f'✅ label re-derivation reproduces §1.3 Y exactly at the deployed '
      f'NEUTRAL_RADIUS={NEUTRAL_RADIUS}')


def _f1_ex_neutral(Yt, Yp):
    """Macro-F1 over the four QUADRANTS only. A wider radius inflates the Neutral
    class and can lift plain macro-F1 for the wrong reason — this is the control."""
    per  = f1_score(Yt, Yp, average=None, zero_division=0)
    keep = [j for j in range(Yt.shape[1]) if j != NEUTRAL_IDX]
    return float(np.mean(np.asarray(per)[keep]))


# ── Sweep: 3 radii × 2 representative feature sets ─────────────────────────
# X_A (MERT alone) and X_C (every tool) bracket the pipeline, so a radius effect
# that shows up in only one of them is visible rather than averaged away.
# (name, classification matrix, regression matrix — B-MERTFULL: MERT full-dim for the ridge,
#  regression kwargs — the deployed block weighting of the multi-block X_C_REG head, §6.3)
_SETS_4C = [('MERT (X_A)', np.asarray(X_A, dtype=float), np.asarray(MERT_REG, dtype=float),
             _abl_reg_kw(['MERT'])),
            ('Combined (X_C)', np.asarray(X_C, dtype=float), np.asarray(X_C_REG, dtype=float),
             _abl_reg_kw(ABL_ORDER))]

print(f'\n📊 4C · NEUTRAL_RADIUS sweep {ABL_RADII} — deployed = {NEUTRAL_RADIUS}')
rad_rows = []
for _r in ABL_RADII:
    _Yr    = _Y_at(df_expr['valence_mean'], df_expr['arousal_mean'], _r)
    _n_neu = int(_Yr[:, NEUTRAL_IDX].sum())
    _sup   = _Yr.sum(axis=0)
    _flip  = int((_Yr != Y).any(axis=1).sum())
    print(f'\n── radius {_r}  →  Neutral {_n_neu}/{len(_Yr)} excerpts · '
          f'{int((_sup > 0).sum())}/{len(EMOTION_LABELS)} classes populated · '
          f'{_flip}/{len(Y)} excerpts relabelled vs the deployed radius')
    print('   support: ' + '  '.join(f'{c}={int(n)}' for c, n in zip(EMOTION_LABELS, _sup)))
    for _sname, _X, _Xr, _rkw in _SETS_4C:
        _tag = f'r={_r} · {_sname}'
        ablate(_X, _tag, tier='4C-neutral-radius', Ym=_Yr, X_reg=_Xr, **_rkw,
               note=f'Neutral={_n_neu}/60, relabelled={_flip}')
        for _pr in PROTOCOLS:
            _pred = ABL_PRED[('4C-neutral-radius', _tag, _pr)]
            rad_rows.append(dict(
                radius=_r, feature_set=_sname, protocol=_pr, deployed=(_r == NEUTRAL_RADIUS),
                n_neutral=_n_neu, n_classes_present=int((_sup > 0).sum()), n_relabelled=_flip,
                f1_macro=_sc_f1(_Yr, _pred['cls']),
                f1_macro_ex_neutral=_f1_ex_neutral(_Yr, _pred['cls'])))

df_abl_radius = pd.DataFrame(rad_rows)
print('\n📊 NEUTRAL_RADIUS — macro-F1 WITH and WITHOUT the Neutral class')
for _pr in PROTOCOLS:
    print(f'\n── {PROTOCOL_LABEL[_pr]} ──')
    print(df_abl_radius.query('protocol == @_pr')[
        ['radius', 'feature_set', 'deployed', 'n_neutral', 'n_classes_present',
         'f1_macro', 'f1_macro_ex_neutral']].round(4).to_string(index=False))
print('\n  Read the two columns TOGETHER: a radius that lifts `f1_macro` while flattening')
print('  `f1_macro_ex_neutral` has bought its gain by manufacturing an easy Neutral class,')
print('  not by predicting emotion better. Report the sensitivity, and keep the deployed')
print(f'  radius {NEUTRAL_RADIUS} unless a different one wins on BOTH columns.')

# ── Ground truth: normalized (deployed) vs raw means ───────────────────────
print('\n📊 4C · ground truth — per-rater NORMALIZED (§1.1b, deployed) vs RAW means')
_Y_raw  = MultiLabelBinarizer(classes=EMOTION_LABELS).fit_transform(
    df_expr['emotion_labels_raw'])
_n_flip = int((_Y_raw != Y).any(axis=1).sum())
print(f'   excerpts whose CLASS changes between raw and normalized: {_n_flip}/{len(Y)}')
ablate(np.asarray(X_C, dtype=float), 'ground truth: NORMALIZED (deployed)',
       tier='4C-ground-truth', X_reg=X_C_REG, **_abl_reg_kw(ABL_ORDER))
ablate(np.asarray(X_C, dtype=float), 'ground truth: RAW means', tier='4C-ground-truth',
       X_reg=X_C_REG, **_abl_reg_kw(ABL_ORDER),
       Ym=_Y_raw, y_val=df_expr['valence_mean_raw'], y_aro=df_expr['arousal_mean_raw'],
       note=f'{_n_flip}/60 excerpts carry a different class under raw means')
df_abl_gt = abl_frame().query('tier == "4C-ground-truth"').reset_index(drop=True)
for _pr in PROTOCOLS:
    print(f'   [{PROTOCOL_LABEL[_pr]}]')
    print(df_abl_gt.query('protocol == @_pr')[
        ['set_name', 'f1_macro', 'r_valence', 'r2_val', 'rmse_val', 'r_arousal', 'r2_aro', 'rmse_aro']]
        .round(4).to_string(index=False))
warn_negative_loo_r(df_abl_gt)
print('   (R² of each row is against ITS OWN targets — normalized vs raw means — so the two rows')
print('    are not paired on the same target; the comparison is descriptive, no p-value.)')
print('\n  Normalization is a DEFENSIBILITY step (§1.2b): the expected result here is')
print('  "no material change", which is what makes the findings robust to rater scale-use')
print('  rather than an artifact of it. A large gap would be the surprise, not the goal.')

df_abl_radius.to_csv(os.path.join(OUTPUT_DIR, 'ablation_neutral_radius.csv'), index=False)
df_abl_gt.to_csv(os.path.join(OUTPUT_DIR, 'ablation_ground_truth.csv'), index=False)

# ── Figure · radius sensitivity, with vs without Neutral ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for ax, _pr in zip(axes, PROTOCOLS):
    sub = df_abl_radius.query('protocol == @_pr')
    for _sname, _mk in zip([s for s, *_ in _SETS_4C], ('o', 's')):
        s2 = sub.query('feature_set == @_sname').sort_values('radius')
        ax.plot(s2['radius'], s2['f1_macro'], marker=_mk, color='#2a6dd9',
                label=f'{_sname} — macro-F1')
        ax.plot(s2['radius'], s2['f1_macro_ex_neutral'], marker=_mk, ls='--', color='#e07b39',
                label=f'{_sname} — macro-F1 ex-Neutral')
    ax.axvline(NEUTRAL_RADIUS, color='#b00020', ls=':', lw=1.2,
               label=f'deployed = {NEUTRAL_RADIUS}')
    ax.set_xlabel('NEUTRAL_RADIUS')
    ax.set_title(f'protocol: {_pr}')
    ax.grid(alpha=0.25)
axes[0].set_ylabel('F1-macro ↑')
axes[0].legend(fontsize=7, loc='best')
fig.suptitle('§6.5 Neutral-radius sensitivity — solid = all 5 classes, '
             'dashed = the 4 quadrants only', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_ablation_neutral_radius.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ §6.5 — ablation_neutral_radius.csv, ablation_ground_truth.csv, '
      'fig_ablation_neutral_radius.png')

## Part 6 · Prediction — §6.6 Per-class metrics · chance-corrected agreement α · floors vs the default

Under the severe class imbalance of §3.1 (some classes carry only ~5 positive excerpts out of 60) a
headline average can hide the fact that a rare label is *never predicted positive*. One per-class table
for the **default classifier** (nested model selection, §5.1) on the a-priori "all tools" set `X_C`, both
protocols: **precision, recall, F1, support** and the **chance-corrected agreement α** (Krippendorff's α
between `Y` and the predictions, per class — merged here from the former §4.8). No classifier is retrained
— every number is recomputed from the predictions stored in `all_recs['Y_pred']`; no best-of selection
(pre-audit this cell picked the "headline" as the best of 32 set × classifier cells on the same CV).

> **What α is not.** The chance-corrected agreement α is a *model-evaluation* metric (prediction vs
> ground truth), **not inter-rater reliability** — it says nothing about how much the participants
> agreed with each other (that is the excerpt-QC script, outside the notebook).

The second cell sets the §5.2 Dummy floors next to the default classifier per feature set and the Phase A
pre-declared default (NaiveBayes, failed under `piece`) — `metrics_baselines.csv` — now with the empirical
null of §5.3 as well; the figures follow.


In [ ]:
# ── §6.6 Per-class precision / recall / F1 / chance-corrected agreement α ──
# NEEDS : DEFAULT_CLF_NAME, DEFAULT_RECS, DEFAULT_SET, mlb, OUTPUT_DIR, PROTOCOLS, PROTOCOL_LABEL, Y
# MAKES : df_perlabel (incl. alpha), kripp_alpha, labels, order · metrics_per_label.csv
# FEEDS : the §6.6 figures below
# ═══════════════════════════════════════════════════════════════════════════════
# §6.6 · Per-label metrics — the DEFAULT classifier on X_C, both protocols (no best-of)
# ═══════════════════════════════════════════════════════════════════════════════
# Pre-audit this cell picked the "headline" = the best of 32 (set × classifier) cells on the
# same LOOCV — a selection. Now: the default classifier (§5.1 — since Phase B the NESTED
# selection, whose config choice is made inside every outer fold) on the a-priori "all
# tools" set X_C, read straight from §6.1's predictions (nothing is retrained).
# Phase C2: the former §4.8 (Krippendorff's α per class, prediction vs ground truth) is merged in as the
# column "chance-corrected agreement α" — a model-evaluation metric, NOT inter-rater reliability.
from sklearn.metrics import precision_recall_fscore_support


def kripp_alpha(mat):
    """Krippendorff's α (nominal) for a (raters × items) matrix — here the 2 'raters' are the ground
    truth and the prediction of one class column (unchanged from the former §4.8)."""
    nr,ni=mat.shape; tp=dp=0
    for j in range(ni):
        v=mat[:,j]; v=v[~np.isnan(v.astype(float))]; n=len(v)
        if n<2: continue
        for a in range(n):
            for b in range(a+1,n):
                tp+=1; dp+=(v[a]!=v[b])
    Do=dp/tp if tp else 0
    av=mat.flatten(); av=av[~np.isnan(av.astype(float))]
    _,c=np.unique(av.astype(int),return_counts=True); p=c/c.sum()
    De=1-np.sum(p**2)
    return 1-Do/De if De else 1.0


labels      = list(mlb.classes_)                       # column order of Y / Y_pred
support_all = Y.sum(axis=0).astype(int)                # positives per label (ground truth)
order       = list(np.argsort(support_all))            # ascending support (rare → frequent)

rows = []
for pr in PROTOCOLS:
    r   = DEFAULT_RECS[(DEFAULT_SET, pr)]
    tag = f'{DEFAULT_SET} / {DEFAULT_CLF_NAME}'
    p, rc, f1, sup = precision_recall_fscore_support(Y, r['Y_pred'], average=None, zero_division=0)
    alpha = [kripp_alpha(np.array([Y[:, j].astype(float), r['Y_pred'][:, j].astype(float)]))
             for j in range(len(labels))]
    print('\n' + '─' * 92)
    print(f'{tag}   ·   {PROTOCOL_LABEL[pr]}   (F1-macro={r["f1_macro"]:.4f})')
    print('─' * 92)
    print(f'  {"class":<20}{"prec":>7}{"recall":>8}{"f1":>7}{"support":>9}{"chance-corrected α":>21}')
    for j in order:
        flag = '   ← recall=0 (collapsed)' if rc[j] == 0 else ''
        print(f'  {labels[j]:<20}{p[j]:>7.3f}{rc[j]:>8.3f}{f1[j]:>7.3f}{int(sup[j]):>9}{alpha[j]:>21.4f}{flag}')
        rows.append({'protocol': pr, 'model': tag, 'class': labels[j],
                     'precision': float(p[j]), 'recall': float(rc[j]),
                     'f1': float(f1[j]), 'support': int(sup[j]), 'alpha_chance_corrected': float(alpha[j])})
    collapsed = [labels[j] for j in range(len(labels)) if rc[j] == 0]
    print(f'  collapsed classes (recall=0, never predicted positive): '
          f'{collapsed if collapsed else "none"}')
print('\n  chance-corrected agreement α = Krippendorff\'s α between Y and the predictions of that class — a '
      'model-evaluation\n  agreement metric, NOT inter-rater reliability (participant agreement is the excerpt-QC '
      'script, outside the notebook).')

df_perlabel = pd.DataFrame(rows)
df_perlabel.to_csv(os.path.join(OUTPUT_DIR, 'metrics_per_label.csv'), index=False)
print(f'\n✅ §6.6 per-label metrics → metrics_per_label.csv  '
      f'({len(df_perlabel)} rows, default classifier on {DEFAULT_SET}, both protocols — no classifier retrained)')


In [ ]:
# ── §6.6b Floors vs the default classifier per feature set ─────────
# NEEDS : all_recs, CHANCE_FLOOR_P95, DEFAULT_CLF_NAME, DEFAULT_RECS, DEFAULT_SET, DUMMY_SUMMARY, emp_null_mark,
#          FEATURE_SETS, N_DUMMY_REPEATS, NB_CLF_NAME, OUTPUT_DIR, PROTOCOLS, PROTOCOL_LABEL, strongest,
#          strongest_f1, strongest_p95 (§5.2), X_C
# MAKES : df_base, NB_PHASEA_TAG, NB_RECS · metrics_baselines.csv
# FEEDS : the §6.6 figures below, §11.1
# ═══════════════════════════════════════════════════════════════════════════════
# §6.6b · The §5.2 Dummy floors next to the default classifier (and the Phase A default) per feature set
# ═══════════════════════════════════════════════════════════════════════════════
# (moved here from the former §4.5b in Phase C2 — the floors themselves are printed in §5.2, before any
# classifier runs). beats_chance = above the Dummy strict P95 (kept for continuity); beats_emp_null = the
# deployed NestedSelect F1 above the P95 of its EMPIRICAL null (§5.3) — the relevant chance level. The
# empirical null is defined for the deployed pipeline only, so the NaiveBayes rows carry none.
# B-NMS: the Phase A pre-declared default stays VISIBLE next to the nested selection.
NB_PHASEA_TAG = f'{NB_CLF_NAME} (pre-declared Phase A default — failed under piece)'
NB_RECS = {(r['feature_set'], r['protocol']): r for r in all_recs if r['classifier'] == NB_CLF_NAME}

summ = []
for pr in PROTOCOLS:
    for _, b in DUMMY_SUMMARY[DUMMY_SUMMARY.protocol == pr].iterrows():
        summ.append({'protocol': pr, 'model': f'Baseline · {b.strategy}', 'kind': 'baseline',
                     'n_repeats': int(b.n_repeats), 'f1_macro': b.f1_macro_mean,
                     'f1_sd': b.f1_macro_sd, 'f1_p95': b.f1_macro_p95,
                     'f1_micro': b.f1_micro_mean, 'hamming': b.hamming_mean, 'beats_chance': np.nan,
                     'emp_null_p95': np.nan, 'emp_null_p': np.nan, 'beats_emp_null': np.nan})
    for fs in FEATURE_SETS:
        r = DEFAULT_RECS[(fs, pr)]
        summ.append({'protocol': pr, 'model': f'{fs} / {DEFAULT_CLF_NAME}', 'kind': 'default',
                     'n_repeats': 1, 'f1_macro': r['f1_macro'], 'f1_sd': np.nan, 'f1_p95': np.nan,
                     'f1_micro': r['f1_micro'], 'hamming': r['hamming_loss'],
                     'beats_chance': bool(r['f1_macro'] > CHANCE_FLOOR_P95[pr]),
                     'emp_null_p95': r['emp_null_p95'], 'emp_null_p': r['emp_null_p'],
                     'beats_emp_null': (bool(r['f1_macro'] > r['emp_null_p95'])
                                        if np.isfinite(r['emp_null_p95']) else np.nan)})
    for fs in FEATURE_SETS:
        r = NB_RECS[(fs, pr)]
        summ.append({'protocol': pr, 'model': f'{fs} / {NB_PHASEA_TAG}', 'kind': 'phaseA-default',
                     'n_repeats': 1, 'f1_macro': r['f1_macro'], 'f1_sd': np.nan, 'f1_p95': np.nan,
                     'f1_micro': r['f1_micro'], 'hamming': r['hamming_loss'],
                     'beats_chance': bool(r['f1_macro'] > CHANCE_FLOOR_P95[pr]),
                     'emp_null_p95': np.nan, 'emp_null_p': np.nan, 'beats_emp_null': np.nan})
df_base = pd.DataFrame(summ)[['protocol', 'model', 'kind', 'n_repeats', 'f1_macro', 'f1_sd', 'f1_p95',
                              'f1_micro', 'hamming', 'beats_chance', 'emp_null_p95', 'emp_null_p',
                              'beats_emp_null']]

for pr in PROTOCOLS:
    print(f'\n📊 BASELINE FLOORS vs the DEFAULT classifier ({DEFAULT_CLF_NAME}) per feature set, and '
          f'the Phase A default ({NB_CLF_NAME}) — {PROTOCOL_LABEL[pr]}')
    print(f'   (random floors: mean / SD / P95 over {N_DUMMY_REPEATS} re-seeded repeats; beats_chance = above '
          'the Dummy strict P95; beats_emp_null = above the empirical-null P95 of the deployed pipeline, §5.3)')
    print(df_base[df_base.protocol == pr].drop(columns='protocol').round(4).to_string(index=False))
    r = DEFAULT_RECS[(DEFAULT_SET, pr)]
    _mk = emp_null_mark(r['f1_macro'], pr, int(np.shape(X_C)[1]), DEFAULT_SET)
    print(f'\n   Default on {DEFAULT_SET}: F1-macro={r["f1_macro"]:.4f} → {r["f1_macro"] - strongest_f1[pr]:+.4f} vs '
          f'the Dummy mean ({strongest[pr]}), {r["f1_macro"] - strongest_p95[pr]:+.4f} vs the Dummy strict P95 · '
          f'empirical null: {_mk} (P95 {r["emp_null_p95"]:.4f}, p {r["emp_null_p"]:.4f}) under {pr}.')

df_base.to_csv(os.path.join(OUTPUT_DIR, 'metrics_baselines.csv'), index=False)
print('\n✅ §6.6b baselines → metrics_baselines.csv  (Dummy floors from §5.1 / §5.2, empirical null from §5.3)')


In [ ]:
# ── §6.6 Figures — per-label bars + floors vs the default ──────────
# NEEDS : DEFAULT_CLF_NAME, DEFAULT_SET, df_base (§6.6b), df_perlabel, labels, order (§6.6), NB_CLF_NAME,
#          OUTPUT_DIR, PRIMARY_PROTOCOL, strongest, strongest_f1, strongest_p95 (§5.2)
# MAKES : fig_metrics_per_label.png, fig_metrics_baselines.png
# FEEDS : — (terminal: figures only)
# ═══════════════════════════════════════════════════════════════════════════════
# §6.6 · Figures — per-label bars (default on X_C, PRIMARY protocol) + F1-macro floors vs default
# ═══════════════════════════════════════════════════════════════════════════════
_pr58    = PRIMARY_PROTOCOL
head_tag = f'{DEFAULT_SET} / {DEFAULT_CLF_NAME}'
ordered  = [labels[j] for j in order]                     # ascending support
dfh      = (df_perlabel[(df_perlabel['model'] == head_tag) & (df_perlabel['protocol'] == _pr58)]
            .set_index('class').loc[ordered])

# ── Fig 1: precision / recall / F1 per class for the default classifier ───────
fig, ax = plt.subplots(figsize=(12, 5.5))
x = np.arange(len(ordered)); w = 0.26
ax.bar(x - w, dfh['precision'], w, label='Precision', color='#4a90d9')
ax.bar(x,     dfh['recall'],    w, label='Recall',    color='#e0a000')
ax.bar(x + w, dfh['f1'],        w, label='F1',        color='#2a9d2a')
for j, emo in enumerate(ordered):
    if dfh.loc[emo, 'recall'] == 0:      # `emo` is a circumplex class code
        ax.annotate('collapsed\n(recall=0)', (x[j], 0.02), ha='center', va='bottom',
                    fontsize=7, color='#b00020', rotation=0)
ax.set_xticks(x)
ax.set_xticklabels([f'{e}\n(n={int(dfh.loc[e, "support"])})' for e in ordered],
                   rotation=30, ha='right', fontsize=8)
ax.set_ylim(0, 1.0); ax.set_ylabel('Score')
ax.set_title(f'§6.6  Per-class metrics — default classifier: {head_tag} · {_pr58} (primary)\n'
             f'(classes ordered by support; rare classes collapse to zero recall)')
ax.legend(loc='upper left', fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_metrics_per_label.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Fig 2: F1-macro — random floors (mean ± SD, P95) vs the default per feature set ─
fig, ax = plt.subplots(figsize=(10, 5.5))
plot_df = df_base[df_base.protocol == _pr58].reset_index(drop=True)
colors  = [{'baseline': '#bdbdbd', 'phaseA-default': '#d9a26a'}.get(k, '#2a6dd9')
           for k in plot_df['kind']]          # orange = NB, the Phase A default (failed under piece)
bars = ax.bar(plot_df['model'], plot_df['f1_macro'], color=colors, edgecolor='#333', linewidth=0.6,
              yerr=plot_df['f1_sd'].fillna(0), capsize=3)
for b, v in zip(bars, plot_df['f1_macro']):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.005, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
# Phase C2: the empirical-null P95 of each deployed row (§5.3, exact width for A–D) as a purple tick
_en_x = [i for i, k in enumerate(plot_df['kind']) if k == 'default' and np.isfinite(plot_df.loc[i, 'emp_null_p95'])]
if _en_x:
    ax.scatter(_en_x, plot_df.loc[_en_x, 'emp_null_p95'], marker='_', s=900, color='#6a1b9a', zorder=4,
               label='empirical-null P95 of that set (§5.3)')
ax.axhline(strongest_f1[_pr58], color='#b00020', ls='--', lw=1.3,
           label=f'Dummy chance mean ({strongest[_pr58]}, {strongest_f1[_pr58]:.3f})')
ax.axhline(strongest_p95[_pr58], color='#b00020', ls=':', lw=1.3,
           label=f'Dummy strict P95 (max over strategies, {strongest_p95[_pr58]:.3f})')
ax.set_ylabel('F1-macro ↑')
ax.set_ylim(0, max(plot_df['f1_macro'].max() * 1.18, plot_df['emp_null_p95'].max(skipna=True) * 1.1,
                  strongest_p95[_pr58] * 1.3, 0.1))
ax.set_title(f'§6.6  F1-macro — random/majority floors (grey) vs the default {DEFAULT_CLF_NAME} (blue)\n'
             f'and the Phase A default {NB_CLF_NAME} (orange) — {_pr58}, primary')
ax.set_xticks(range(len(plot_df)))
ax.set_xticklabels(plot_df['model'], rotation=25, ha='right', fontsize=8)
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_metrics_baselines.png'), dpi=150, bbox_inches='tight')
plt.show()

print('✅ §6.6 figures → fig_metrics_per_label.png , fig_metrics_baselines.png')


## Part 6 · Prediction — §6.7 Per-condition F1 (MEC / EXP / EXG)


In [ ]:
# ── §6.7 Per-condition F1 ───────────────────────────────────────────
# NEEDS : CONDITIONS, conditions, DEFAULT_CLF_NAME, DEFAULT_RECS, DEFAULT_SET, PROTOCOLS,
#          PROTOCOL_LABEL, Y
# MAKES : default_rec (protocol → §6.1 record of the default classifier on X_C)
# FEEDS : — (printed only)
# No best-of selection (pre-audit: "best overall" over 32 cells). The default classifier on
# the a-priori "all tools" set X_C, both protocols.
default_rec = {pr: DEFAULT_RECS[(DEFAULT_SET, pr)] for pr in PROTOCOLS}
print(f'Default classifier: {DEFAULT_SET} / {DEFAULT_CLF_NAME}')
for pr in PROTOCOLS:
    print(f'\n── {PROTOCOL_LABEL[pr]}  (F1-macro overall = {default_rec[pr]["f1_macro"]:.4f}) ──')
    for cond in ['MEC','EXP','EXG']:
        mask=conditions==cond
        if mask.sum()<2: continue
        f1=f1_score(Y[mask],default_rec[pr]['Y_pred'][mask],average='macro',zero_division=0)
        print(f'  {cond} ({CONDITIONS[cond]}): {f1:.4f}')


## Part 6 · Prediction — §6.8 Quadrant accuracy from the predicted valence / arousal

The circumplex quadrant of each excerpt's **predicted** (valence, arousal) against its true quadrant
(split at `VA_MID`). Phase C2: the predictions are the **deployed nested ridge** of §6.2 — one row for
**MERT-full 768**, one for **Interpretable (27)** — no longer the fixed Ridge α = 10 on MERT-PCA25. Two
chance references: the **majority-quadrant share** of the ground truth, and a **permutation null** (the
predicted quadrants shuffled across the 60 excerpts, 1000×, seeded — it keeps how often each quadrant is
predicted and breaks which excerpt gets which). Fig 5 plots the same predictions for all 60 excerpts (MEC
included); Fig 6 is the ground-truth circumplex.


In [ ]:
# ── §6.8 Quadrant accuracy from the predicted valence / arousal ─────
# NEEDS : df_expr, OUTPUT_DIR, PRIMARY_PROTOCOL, PROTOCOLS, PROTOCOL_LABEL, QUADRANT_DISPLAY, REG_INTERP,
#          REG_MERTFULL, REG_RES (§6.2, deployed nested ridge), SEED, va_to_quadrant
# MAKES : df_quad_acc, PRED_QUADS (protocol → quadrants, MERT-full — the headline), PRED_QUADS_INTERP,
#          pred_quads (= PRIMARY), QUAD_N_PERM, true_quads · quadrant_accuracy.csv, fig_quadrant_cm.png
# FEEDS : Fig 5 below, §11.1 (all_predictions.csv)
# ═══════════════════════════════════════════════════════════════════════════════
# §6.8 · Do the predicted (valence, arousal) land in the right circumplex quadrant?
# ═══════════════════════════════════════════════════════════════════════════════
# Phase C2: computed from the DEPLOYED nested-ridge predictions of §6.2 (MERT-full 768 and Interpretable
# (27)) — no longer from the fixed Ridge α = 10 on MERT-PCA25. Chance references: (a) the majority-quadrant
# share of the ground truth; (b) a permutation null — the predicted quadrants shuffled across the 60
# excerpts QUAD_N_PERM times (seeded), which keeps the predicted quadrant frequencies and breaks the
# excerpt ↔ prediction link; one-sided p = (1 + #perm ≥ observed) / (1 + QUAD_N_PERM).
QUAD_N_PERM = 1000
true_quads  = df_expr['quadrant'].values
_qmaj       = pd.Series(true_quads).value_counts(normalize=True)
_QSRC       = {'MERT-full 768': REG_MERTFULL, f'Interpretable ({REG_INTERP.split("(")[-1]}': REG_INTERP}
PRED_QUADS, PRED_QUADS_INTERP, _qrows = {}, {}, []
for pr in PROTOCOLS:
    _rng = np.random.default_rng(SEED + 708)
    for _lab, _key in _QSRC.items():
        _pv, _pa = REG_RES[pr][_key]['vp'], REG_RES[pr][_key]['ap']
        _pq = np.array([va_to_quadrant(v, a) for v, a in zip(_pv, _pa)])
        (PRED_QUADS if _key == REG_MERTFULL else PRED_QUADS_INTERP)[pr] = _pq
        _acc = accuracy_score(true_quads, _pq)
        _null = np.array([accuracy_score(true_quads, _rng.permutation(_pq)) for _ in range(QUAD_N_PERM)])
        _qrows.append(dict(protocol=pr, representation=_lab, model=_key, accuracy=float(_acc),
                           majority_share=float(_qmaj.iloc[0]), majority_quadrant=str(_qmaj.index[0]),
                           perm_null_mean=float(_null.mean()), perm_null_p95=float(np.percentile(_null, 95)),
                           perm_p=float((1 + np.sum(_null >= _acc - 1e-12)) / (1 + QUAD_N_PERM))))
df_quad_acc = pd.DataFrame(_qrows)
for pr in PROTOCOLS:
    print(f'\n── Quadrant accuracy (deployed nested ridge VA → circumplex) — {PROTOCOL_LABEL[pr]} ──')
    print(df_quad_acc.query('protocol == @pr')[['representation', 'accuracy', 'majority_quadrant', 'majority_share',
                                                 'perm_null_mean', 'perm_null_p95', 'perm_p']]
          .round(4).to_string(index=False))
for _lab in _QSRC:
    _a = df_quad_acc.set_index(['protocol', 'representation'])['accuracy']
    print(f'⚠️  OPTIMISM GAP (piece − loo), {_lab}: {_a[("piece", _lab)] - _a[("loo", _lab)]:+.4f}')
print(f'   chance references: majority quadrant {_qmaj.index[0]} = {_qmaj.iloc[0]:.3f} of the 60 excerpts · '
      f'permutation null = predicted quadrants shuffled across excerpts ({QUAD_N_PERM}×, seed SEED + 708)')
df_quad_acc.to_csv(os.path.join(OUTPUT_DIR, 'quadrant_accuracy.csv'), index=False)
pred_quads = PRED_QUADS[PRIMARY_PROTOCOL]
qa = float(accuracy_score(true_quads, pred_quads))

ql = ['HA_HV','HA_LV','LA_HV','LA_LV']
qn = [f'{q}\n{QUADRANT_DISPLAY[q]}'.replace(' / ', ' /\n') for q in ql]
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, (_lab, _key) in zip(axes, _QSRC.items()):
    _pq = (PRED_QUADS if _key == REG_MERTFULL else PRED_QUADS_INTERP)[PRIMARY_PROTOCOL]
    _r = df_quad_acc.query('protocol == @PRIMARY_PROTOCOL and model == @_key').iloc[0]
    sns.heatmap(confusion_matrix(true_quads, _pq, labels=ql), ax=ax, xticklabels=qn, yticklabels=qn, annot=True,
                fmt='d', cmap='Blues', linewidths=0.4, linecolor='#ddd')
    ax.set_title(f'{_lab} · nested ridge · {PRIMARY_PROTOCOL} (primary)\naccuracy {_r.accuracy:.3f} · '
                 f'majority {_r.majority_share:.3f} · perm p {_r.perm_p:.3g}', fontsize=10)
    ax.set_xlabel('predicted quadrant'); ax.set_ylabel('true quadrant'); ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'fig_quadrant_cm.png'),dpi=150,bbox_inches='tight')
plt.show()
print('✅ §6.8 — quadrant_accuracy.csv, fig_quadrant_cm.png')


In [ ]:
# ── Fig 5: VA scatter — the deployed nested ridge (MERT-full 768 and Interpretable (27)) ──
# NEEDS : CONDITIONS, conditions, oos_r2, OUTPUT_DIR, PRIMARY_PROTOCOL, PROTOCOL_LABEL, REG_INTERP, REG_MERTFULL,
#          REG_RES, rmse, y_arousal, y_valence
# MAKES : fig_va_scatter.png
# FEEDS : — (terminal: figure only)
# Phase C2: the same source as §6.8 (the deployed nested-ridge predictions of §6.2), all 60 excerpts — MEC is
# in the colour map now (the old figure plotted EXP / EXG only) — and every title is computed from the
# plotted data. The local metrics are named _*_f5 so they never shadow the §5.1 rmse() helper.
cc = {'MEC': '#999999', 'EXP': '#2a6dd9', 'EXG': '#2a9d2a'}
_f5_src = [('MERT-full 768', REG_MERTFULL), (f'Interpretable ({REG_INTERP.split("(")[-1]}', REG_INTERP)]
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
fig.suptitle(f'VA regression — deployed nested ridge · {PROTOCOL_LABEL[PRIMARY_PROTOCOL]}', fontsize=13)
for _row, (_lab, _key) in enumerate(_f5_src):
    _rr = REG_RES[PRIMARY_PROTOCOL][_key]
    for ax, (tname, yt, yp) in zip(axes[_row], [('Valence', y_valence, _rr['vp']), ('Arousal', y_arousal, _rr['ap'])]):
        for cond, col in cc.items():
            m = conditions == cond
            ax.scatter(yt[m], yp[m], color=col, alpha=0.8, s=55, label=f'{CONDITIONS[cond]} (n={int(m.sum())})')
        ax.plot([0, 6], [0, 6], '--', color='#ccc', lw=1)
        _r_f5 = pearsonr(yt, yp)[0]; _rmse_f5 = rmse(yt, yp); _r2_f5 = oos_r2(yt, yp, PRIMARY_PROTOCOL)
        ax.set_title(f'{_lab} — {tname}  (n={len(yt)})\nr={_r_f5:.3f}  R²={_r2_f5:+.3f}  RMSE={_rmse_f5:.3f}',
                     fontsize=10)
        ax.set_xlabel(f'Participant {tname} (mean)'); ax.set_ylabel(f'Predicted {tname}')
        ax.set_xlim(0, 6); ax.set_ylim(0, 6); ax.legend(fontsize=8); ax.grid(alpha=0.15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'fig_va_scatter.png'),dpi=150,bbox_inches='tight'); plt.show()


In [ ]:
# ── Fig 6: Circumplex scatter (ground truth, all 60 excerpts) ─────────────────
# NEEDS : CONDITIONS, df_all, OUTPUT_DIR, VA_MID · MAKES : fig_circumplex.png · FEEDS : —
fig,ax=plt.subplots(figsize=(8,7))
cp={'MEC':'#aaa','EXP':'#2a6dd9','EXG':'#2a9d2a'}
# ── Connect the three interpretations (MEC→EXP→EXG) of each piece ─────────────
cond_order=['MEC','EXP','EXG']
for piece in df_all['piece'].unique():
    sub=df_all[df_all['piece']==piece].set_index('condition')
    if not all(c in sub.index for c in cond_order): continue
    xs=[sub.loc[c,'valence_mean'] for c in cond_order]
    ys=[sub.loc[c,'arousal_mean'] for c in cond_order]
    ax.plot(xs,ys,'-',color='#c8c8c8',lw=0.9,alpha=0.6,zorder=1)
for cond,col in cp.items():
    m=df_all['condition']==cond
    ax.scatter(df_all.loc[m,'valence_mean'],df_all.loc[m,'arousal_mean'],
               color=col,alpha=0.8,s=70,label=CONDITIONS[cond],zorder=3)
ax.axhline(VA_MID,color='#ccc',lw=1); ax.axvline(VA_MID,color='#ccc',lw=1)
# quadrant region labels = the classification CLASSES (x=valence, y=arousal)
for x,y,lbl in [(4.5,4.5,'HA_HV'),(1.5,4.5,'HA_LV'),
                   (4.5,1.5,'LA_HV'),(1.5,1.5,'LA_LV')]:
    ax.text(x,y,lbl,ha='center',va='center',fontsize=9,color='#999',style='italic')
ax.set_xlim(0,6); ax.set_ylim(0,6)
ax.set_xlabel('Valence (participant mean)',fontsize=11)
ax.set_ylabel('Arousal (participant mean)',fontsize=11)
ax.set_title('Circumplex Map — All 60 Excerpts',fontsize=12)
ax.legend(fontsize=10); ax.grid(alpha=0.15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'fig_circumplex.png'),dpi=150,bbox_inches='tight'); plt.show()

## Part 6 · Prediction — §6.9 Every classification F1 read against the empirical null

The read-out half of the former §4.5c (the draws are §5.3). Every classification F1 reported in Part 6 —
the §6.1 headline A–D and every §6.3 lattice and tempogram-sensitivity row — against the Dummy strict P95
and against its **empirical null** (`emp_null_readout.csv`). Part 7 appends the §7.1 blocks and the §7.2
layer candidates to the same table.

**Primary family (pre-declared, §5.3):** A–D, Interpretable (27), DTW, CREPE + DTW — each against its
**exact-width** null (Phase C3: 1000 draws at d = 7 / 17, 600 at the other widths; `null_n_draws` and
`null_target` are printed next to every p), one-sided empirical p, **BH-FDR within the family** → `q_primary`;
"beats the empirical null" for a primary set = q < 0.05 (`emp_null_primary.csv`). **Every other set** is
descriptive (p < 0.05 uncorrected) and read by the same Phase C3 rule (§5.3): the primary null of its own width
if one exists (e.g. Essentia + DTW, d = 17), else the next larger primary width (*"null borrowed from d = …
(conservative, next larger width)"*), else d = 52 (*extrapolated*). The 100-draw map is not a reference.
Under `loo` no null is drawn (`RUN_NULL_LOO = False`) → n/a.


In [ ]:
# ── §6.9 Every classification F1 read against the empirical null ────
# NEEDS : ABL_ROWS (§6.3), bh_fdr, CHANCE_FLOOR_P95, DEFAULT_RECS, emp_null_ref, EMP_NULL_PRIMARY, FEATURE_SETS,
#          null_primary_target, NULL_PRIMARY_COMPLETE, OUTPUT_DIR, PRIMARY_NULL_FAMILY,
#          PRIMARY_PROTOCOL, PROTOCOLS, PROTOCOL_LABEL, RUN_NULL_LOO (§5.3), X_A, X_B, X_C, X_D
# MAKES : df_emp_null_readout, df_emp_null_primary, emp_readout_rows · emp_null_readout.csv,
#          emp_null_primary.csv
# FEEDS : §7.1 / §7.2 (append their own rows and re-export emp_null_readout.csv), §11.1 (claim 5)
# ═══════════════════════════════════════════════════════════════════════════════
# §6.9 · Which classification results survive the chance level of the deployed pipeline?
# ═══════════════════════════════════════════════════════════════════════════════
# (the read-out half of the former §4.5c — the draws are §5.3). Every reported F1 of Part 6 — the §6.1
# headline A–D and every §6.3 lattice / tempogram row — is read against (i) the Dummy strict P95 and (ii) its
# empirical null by the ONE Phase C3 rule of emp_null_ref (§5.3): the primary null at exactly its width, else the
# smallest primary width ≥ it ("null borrowed from d = … (conservative, next larger width)"), else d = 52
# (extrapolated). The 100-draw map is not a reference. The §7.1 blocks and §7.2 layer candidates append their rows
# in Part 7.
# PRIMARY FAMILY: one empirical p per set (piece), BH-FDR WITHIN the family → q; "beats the empirical null" =
# q < 0.05. Everything else is descriptive: p < 0.05, no correction.


def emp_readout_rows(source, set_name, protocol, n_features, f1, note=''):
    """One read-out row: the observed F1 against the Dummy strict P95 and its empirical null (§5.3)."""
    e = emp_null_ref(f1, protocol, n_features, set_name)
    return dict(source=source, set_name=set_name, protocol=protocol, n_features=int(n_features),
                f1_macro=float(f1), note=note, null_kind=e['kind'], null_label=e['label'], null_d=e['null_d'],
                null_n_draws=e['n_draws'], null_extrapolated=bool(e['kind'] == 'extrapolated'),
                dummy_strict_p95=CHANCE_FLOOR_P95[protocol],
                beats_dummy_p95=bool(f1 > CHANCE_FLOOR_P95[protocol]),
                emp_null_p95=e['p95'], above_emp_p95=(bool(f1 > e['p95']) if np.isfinite(e['p95']) else np.nan),
                emp_p=e['p'], beats_emp_null=(bool(e['p'] < 0.05) if np.isfinite(e['p']) else np.nan))


_rd = []
_fs_X = {'A: MERT': X_A, 'B: CREPE': X_B, 'C: Combined': X_C, 'D: MERT+Essentia': X_D}
assert list(_fs_X) == list(FEATURE_SETS), f'§6.9: FEATURE_SETS changed: {FEATURE_SETS}'
for _fs, _Xfs in _fs_X.items():
    for _pr in PROTOCOLS:
        _rd.append(emp_readout_rows('§6.1 headline', _fs, _pr, np.shape(_Xfs)[1],
                                    DEFAULT_RECS[(_fs, _pr)]['f1_macro']))
for _r in ABL_ROWS:
    if _r['tier'] in ('1-blocks', '1b-tempogram-sensitivity'):
        if _r['tier'] == '1b-tempogram-sensitivity' and 'tempogram' not in _r['set_name']:
            continue                                    # the re-scored parents are already the lattice rows
        _rd.append(emp_readout_rows('§6.3 lattice' if _r['tier'] == '1-blocks' else '§6.3 tempogram sensitivity',
                                    _r['set_name'], _r['protocol'], _r['n_features'], _r['f1_macro'],
                                    note='= X_F (§2.5)' if _r['set_name'] == 'MERT + DTW' else ''))
df_emp_null_readout = pd.DataFrame(_rd)

# ── The PRE-DECLARED primary family — exact-width null, BH within the family ──
_prow = []
for _nm, _src, _key, _w in PRIMARY_NULL_FAMILY:
    for _pr in PROTOCOLS:
        _hit = df_emp_null_readout[(df_emp_null_readout.source == _src) & (df_emp_null_readout.set_name == _key)
                                   & (df_emp_null_readout.protocol == _pr)]
        assert len(_hit) == 1 and int(_hit['n_features'].iloc[0]) == _w, \
            f'§6.9: primary set {_nm!r} has no unique {_src} row {_key!r} of width {_w} ({_pr})'
        _h = _hit.iloc[0]
        _prow.append(dict(primary_set=_nm, source=_src, row=_key, protocol=_pr, d=_w, f1_macro=float(_h.f1_macro),
                          null_kind=_h.null_kind, null_n_draws=int(_h.null_n_draws), null_target=null_primary_target(_w),
                          emp_null_p95=float(_h.emp_null_p95),
                          emp_p=float(_h.emp_p)))
df_emp_null_primary = pd.DataFrame(_prow)
df_emp_null_primary['q_primary'] = np.nan
for _pr, _sub in df_emp_null_primary.groupby('protocol'):
    df_emp_null_primary.loc[_sub.index, 'q_primary'] = bh_fdr(_sub['emp_p'].to_numpy(dtype=float))
df_emp_null_primary['beats_emp_null_bh'] = [(bool(q < 0.05) if np.isfinite(q) else np.nan)
                                            for q in df_emp_null_primary['q_primary']]
df_emp_null_readout['primary_set'] = ''
df_emp_null_readout['q_primary'] = np.nan
for _r in df_emp_null_primary.itertuples():
    _i = df_emp_null_readout[(df_emp_null_readout.source == _r.source) & (df_emp_null_readout.set_name == _r.row)
                             & (df_emp_null_readout.protocol == _r.protocol)].index
    df_emp_null_readout.loc[_i, 'primary_set'] = _r.primary_set
    df_emp_null_readout.loc[_i, 'q_primary'] = _r.q_primary

df_emp_null_readout.to_csv(os.path.join(OUTPUT_DIR, 'emp_null_readout.csv'), index=False)
df_emp_null_primary.to_csv(os.path.join(OUTPUT_DIR, 'emp_null_primary.csv'), index=False)

print('📊 §6.9 · PRIMARY FAMILY (pre-declared) — exact-width empirical null, BH within the family')
if not NULL_PRIMARY_COMPLETE:
    print(f'   ⚠️⚠️  the primary null is INCOMPLETE (below its per-width target for some width — see §5.3); '
          'p and q use the draws available (null_n_draws)')
for _pr in PROTOCOLS:
    _s = df_emp_null_primary[df_emp_null_primary.protocol == _pr]
    print(f'\n── {PROTOCOL_LABEL[_pr]} ──')
    if not _s['emp_p'].notna().any():
        print(f'   (no empirical null drawn under {_pr!r} — RUN_NULL_LOO = {RUN_NULL_LOO}; p / q = n/a)')
    print(_s[['primary_set', 'd', 'f1_macro', 'null_n_draws', 'null_target', 'emp_null_p95', 'emp_p', 'q_primary',
              'beats_emp_null_bh']].round(4).to_string(index=False))
_pp = df_emp_null_primary[df_emp_null_primary.protocol == PRIMARY_PROTOCOL]
print(f'\n   {PRIMARY_PROTOCOL}: {int((_pp.q_primary < 0.05).sum())}/{len(_pp)} primary sets beat their exact-width '
      f'empirical null after BH (q < 0.05): '
      + (', '.join(_pp[_pp.q_primary < 0.05]['primary_set']) or 'none'))

_cols = ['source', 'set_name', 'n_features', 'f1_macro', 'dummy_strict_p95', 'beats_dummy_p95', 'null_label',
         'emp_null_p95', 'emp_p', 'beats_emp_null', 'primary_set', 'q_primary', 'note']
for _pr in PROTOCOLS:
    _sub = df_emp_null_readout[df_emp_null_readout.protocol == _pr]
    print(f'\n── {PROTOCOL_LABEL[_pr]} — every reported F1 of Part 6 vs the Dummy strict P95 and its empirical null ──')
    with pd.option_context('display.width', 250, 'display.max_colwidth', 60):
        print(_sub[[c for c in _cols if c != 'note' or _sub['note'].astype(bool).any()]].round(4).to_string(index=False))
_pd = df_emp_null_readout[df_emp_null_readout.protocol == PRIMARY_PROTOCOL]
_flip = _pd[_pd['beats_dummy_p95'] & ~(_pd['emp_p'] < 0.05)]
print(f'\n   {PRIMARY_PROTOCOL}: {int(_pd["beats_dummy_p95"].sum())}/{len(_pd)} reported F1 beat the Dummy strict P95, '
      f'{int((_pd["emp_p"] < 0.05).sum())}/{len(_pd)} beat their empirical null (p < 0.05, uncorrected) · beat the '
      f'Dummy P95 but NOT the empirical null: '
      + (', '.join(f'{s} [{src}]' for s, src in zip(_flip['set_name'], _flip['source'])) or 'none'))
print('   null_label (Phase C3 rule): "exact-width null d = …" = the primary null of the same width · "null borrowed '
      'from d = … (conservative, next larger width)" = the smallest primary width ≥ the set · "… (extrapolated)" = '
      'wider than every primary width (d = 52 used) · n draws in brackets')
print('✅ §6.9 — emp_null_readout.csv (Part 6 rows; §7.1 / §7.2 append theirs), emp_null_primary.csv')


## Part 7 · MERT diagnostics — what the embedding encodes — §7.1 MERT generalisation · train vs test · validation curve · nearest-neighbour identity
**Question of this part:** Part 6 showed how MERT scores on unseen pieces — *what does the embedding  
actually encode*: the piece, the condition, the named techniques, the rating shift — and what does each
interpretable tool add on top of it?
**Hands to the next part:** `MG_VERDICT` (§7.1), the layer study (§7.2), the probe R² per feature
(`PROBE_R2`, §7.3b), the embedding-shift table (§7.4) and the tool-complementarity table (§7.5); Part 8
then removes the piece altogether.

**Why (Phase B3).** In the full B2 run MERT collapsed when the piece was held out: classification F1
LOOCV → piece MERT 0.623 → 0.310 (−50 %, below the chance P95 0.323) against CREPE 0.500 → 0.333,
Essentia 0.480 → 0.356, DTW 0.570 → 0.542; adding MERT to an interpretable block lowered the piece F1 in
6 of 7 combinations and raised the LOOCV F1 in 7 of 7; the unique F1 of MERT (piece) was −0.122
(q = 0.034); NestedSelect picked C = 0.001 (a class-mean-difference (prototype) classifier whose threshold depends on the fold's label composition) in 12/20 piece folds and C = 10 in
43/60 LOOCV folds. MERT-PCA25 ridge had a negative out-of-sample R² under `piece` (valence −0.18,
arousal −0.25); MERT-full ridge transfers arousal (R² 0.45) but not valence (0.01). **No train score was
computed anywhere**, so two explanations could not be told apart: **capacity overfitting** (the model
memorises the training rows) and a **piece-identity shortcut** (LOOCV keeps the other two renditions of
the held-out piece in training, and MERT may simply recognise the piece). This section measures both.
It is descriptive: nothing here selects a model or a feature.

**B1 · train vs test, per block and protocol (`piece` first, then `loo`).** Blocks: MERT-PCA25 (the
deployed classification block `X_A`), MERT-full 768, CREPE, Essentia, DTW (ref-free) — exactly the
§6.3 lattice blocks —, the predictive interpretable layer (the 27 features of `INTERP_PRED_COLS`, §4.1 —
the Phase C2 construction; §10.3 asserts it is
the same matrix) and a **capacity control**: 25 i.i.d. standard-normal columns (seed 0) — the gap that
memorising pure noise gives at this dimension. *Classification:* the per-fold configuration NestedSelect
chose is read from §5.1's memo (`cv_predict_nested_select(…, return_info=True)` — the §6.3 rows only
store the chosen-config counts, not the per-fold order; for the blocks already run in §6.3 / §6.4 the
call is a memo hit and fits nothing new, the interpretable layer and the random control are run here);
it is refitted on each outer training fold with §5.1's own scaler + binary-relevance code and scored on
the fold's training rows (train F1, mean over folds) and on the test rows (test F1, pooled exactly as the
headline — asserted equal to the NestedSelect predictions and to the §6.3 / §6.4 rows). *Regression:*
the nested ridge (§5.1) per block; train R² = in-sample R² of the refitted fold model (mean over folds),
test = the out-of-sample R². Representation as in §6.3 / §6.4: MERT-PCA25 → the PCA block (as tier
`4A-mert-pca`), MERT-full → `X_MERT_FULL` (= `MERT_REG` when `MERT_REG_REPR = 'full'`); every block here
is a single tool block, so the deployed block weighting (multi-block MERT-full heads only) does not apply.

**B2 · validation curve under `piece`.** SVM-Linear (balanced, the `SELECT_CONFIGS` factory) at each
fixed C of the §0.3 / §5.1 grid (`SELECT_C_GRID`, 10⁻³ … 10², no inner selection), train and test
F1-macro vs C for MERT-PCA25, the interpretable layer and the random control (test predictions asserted
equal to `cv_predict_multilabel`). Reading: the NestedSelect choice for MERT under `piece` (C = 0.001 in
most folds) is a class-mean-difference (prototype) classifier whose threshold depends on the fold's label composition — the deployed MERT classifier **under-fits**, so its low piece F1
by itself is not proof of overfitting. Capacity overfitting shows up as: *for every C at which the
training F1 rises, the test F1 stays at chance.*

**B3 · nearest-neighbour identity test (label-free).** Piece and condition are design metadata, never
ratings. Spaces: MERT-full (cosine on the raw embedding), MERT-PCA25, CREPE, Essentia, DTW and the
interpretable layer (each z-scored over the 60 excerpts — no model is fitted, so this is not a leak).
For each space: the share of the 60 excerpts whose nearest neighbour (self excluded) is from the **same
piece** (chance 2/59 = 3.4 %) and from the **same condition** (chance 19/59 = 32.2 %), with permutation
p-values (10 000 shuffles of the label vectors — which keep 3 excerpts per piece and 20 per condition —
fixed seed), plus the silhouette score by piece and by condition.

**B4 · PRE-DECLARED READING RULES** (printed before the results; the cell prints which rule fires;
thresholds stay descriptive and are never tuned on the result):
- **CAPACITY OVERFITTING** — MERT-PCA25's `piece` train − test F1 gap exceeds both the interpretable
  layer's and the random control's by ≥ `MG_GAP_MARGIN` = 0.10, **and** on the validation curve every C at
  which MERT's train F1 has risen (≥ `MG_CURVE_RISE` = 0.05 above its value at the smallest C; at least
  one such C) keeps MERT's test F1 inside the chance band (≤ the strict P95 of `piece`; mean ≈ 0.272,
  P95 ≈ 0.323).
- **PIECE-IDENTITY SHORTCUT** — in a MERT space (MERT-full or MERT-PCA25) the same-piece NN share is
  ≥ `MG_NN_FOLD` = 3 × chance, its permutation p < 0.05 and it exceeds every interpretable space (CREPE,
  Essentia, DTW, interpretable layer), **while** the same-condition share is not significant (p ≥ 0.05).
- Both can fire. If neither fires, the cell says so.

Outputs in `OUTPUT_DIR/mert_diagnostics/`: `mert_train_test_gap.csv`, `mert_validation_curve.csv`,
`mert_nn_identity.csv`, `fig_mert_validation_curve.png`, `fig_mert_nn_identity.png`.


In [ ]:
# ── §7.1 MERT generalisation diagnostics — train vs test · validation curve · NN identity ──
# NEEDS : ABL_BLOCKS, ABL_PRED, ABL_ROWS, ABL_Y_ARO, ABL_Y_VAL, CHANCE_FLOOR_MEAN, CHANCE_FLOOR_P95,
#          cv_predict_default, cv_predict_multilabel, cv_splits, df_emp_null_readout, emp_readout_rows (§6.9),
#          df_expr, df_null_draws, NULL_DIMS, NULL_SEED, _null_X (§5.3), eids, X_INTERP (§4.1),
#          nested_ridge_predict, nms_counts_str, oos_r2, OUTPUT_DIR, PIECE_GROUPS, PRIMARY_PROTOCOL,
#          PROTOCOLS, PROTOCOL_LABEL, SEED, SELECT_C_GRID, SELECT_CONFIGS, threadpool_limits, X_A,
#          X_MERT_FULL, Y,
#          _fit_predict_labels, _ridge_pipe (§5.1 helpers — reused, never rebound), _sc_f1 (§6.3)
# MAKES : MG_DIR, MG_GAP_MARGIN, MG_CURVE_RISE, MG_NN_FOLD, MG_N_PERM, MG_RANDOM_SEED, MG_X_INTERP,
#          MG_X_RANDOM, MG_BLOCKS, df_mg_gap, df_mg_curve, df_mg_nn, MG_VERDICT, _mg_* ·
#          mert_diagnostics/mert_train_test_gap.csv, mert_validation_curve.csv, mert_nn_identity.csv,
#          fig_mert_validation_curve.png, fig_mert_nn_identity.png
# FEEDS : the report (MERT generalisation reading); §7.2 (MG_X_INTERP, df_mg_nn); §10.3 (asserts
#          MG_X_INTERP == the §10.2 matrix); emp_null_readout.csv (the §7.1 rows are appended here)
# ═══════════════════════════════════════════════════════════════════════════════
# §7.1 · Capacity overfitting or piece-identity shortcut? (Phase B3, descriptive)
# ═══════════════════════════════════════════════════════════════════════════════
# Reads Y only to SCORE; the NN identity test reads design metadata (piece, condition) only — never a
# rating. Every CV split comes from §5.1 (cv_splits / cv_predict_default / cv_predict_multilabel /
# nested_ridge_predict); StandardScaler is fitted inside every training fold. Nothing is selected here.
from sklearn.metrics import pairwise_distances, silhouette_score

MG_DIR = os.path.join(OUTPUT_DIR, 'mert_diagnostics')
os.makedirs(MG_DIR, exist_ok=True)
MG_RANDOM_SEED = 0          # the capacity control: i.i.d. N(0, 1) columns
MG_GAP_MARGIN  = 0.10       # "clearly exceeds": MERT's piece train − test F1 gap ≥ the other gap + 0.10
MG_CURVE_RISE  = 0.05       # a C "raises the train F1": train F1 ≥ train F1 at the smallest C + 0.05
MG_NN_FOLD     = 3.0        # "far above chance": same-piece NN share ≥ 3 × chance (2/59)
MG_N_PERM      = 10_000     # label shuffles of the NN identity test
_mg_pr0        = PRIMARY_PROTOCOL

# ── B4 · the reading rules, declared BEFORE any result is computed ─────────────
print('§7.1 · PRE-DECLARED READING RULES (descriptive thresholds, never tuned on the result)')
print(f'   CAPACITY OVERFITTING fires iff (a) MERT-PCA{X_A.shape[1]}\'s {_mg_pr0} train − test F1 gap ≥ the '
      f'interpretable layer\'s gap + {MG_GAP_MARGIN:g} AND ≥ the random control\'s gap + {MG_GAP_MARGIN:g}, and')
print(f'      (b) on the {_mg_pr0} validation curve every C at which MERT\'s train F1 has risen (≥ {MG_CURVE_RISE:g} '
      f'above its value at the smallest C; ≥ 1 such C) keeps MERT\'s test F1 ≤ the strict chance P95 '
      f'({CHANCE_FLOOR_P95[_mg_pr0]:.3f}; mean {CHANCE_FLOOR_MEAN[_mg_pr0]:.3f})')
print(f'   PIECE-IDENTITY SHORTCUT fires iff in a MERT space the same-piece NN share is ≥ {MG_NN_FOLD:g} × chance '
      '(2/59), its permutation p < 0.05 and it exceeds every interpretable space, while the same-condition '
      'share is not significant (p ≥ 0.05)')
print('   Both can fire; if neither fires the cell says so.\n')

# ── The blocks ─────────────────────────────────────────────────────────────────
# Interpretable layer = the PREDICTIVE layer of §4.1 (INTERP_PRED_COLS, Phase C2 — no DTW-deviation), i.e.
# X_INTERP; §10.3 asserts it is byte-identical to §10.2's _X_fam(_interp_cols).
MG_X_INTERP = np.array(X_INTERP, dtype=float, copy=True)
MG_X_RANDOM = np.random.default_rng(MG_RANDOM_SEED).standard_normal((len(eids), int(X_A.shape[1])))
_mg_pca = f'MERT-PCA{X_A.shape[1]}'
_mg_ref_pca = ('4A-mert-pca', f'MERT PCA 95% (deployed, {X_A.shape[1]}d)')
_mg_ref_full = ('4A-mert-pca', 'MERT full 768d (no reduction)')
# (name, classification matrix, regression matrix, §6.3/§6.4 row of the classifier, row of the ridge)
MG_BLOCKS = [
    (_mg_pca, X_A, X_A, ('1-blocks', 'MERT'), _mg_ref_pca),
    (f'MERT-full {X_MERT_FULL.shape[1]}', X_MERT_FULL, X_MERT_FULL, _mg_ref_full, _mg_ref_full),
    ('CREPE', ABL_BLOCKS['CREPE'], ABL_BLOCKS['CREPE'], ('1-blocks', 'CREPE'), ('1-blocks', 'CREPE')),
    ('Essentia', ABL_BLOCKS['Essentia'], ABL_BLOCKS['Essentia'], ('1-blocks', 'Essentia'), ('1-blocks', 'Essentia')),
    ('DTW (ref-free)', ABL_BLOCKS['DTW'], ABL_BLOCKS['DTW'], ('1-blocks', 'DTW'), ('1-blocks', 'DTW')),
    (f'Interpretable ({MG_X_INTERP.shape[1]})', MG_X_INTERP, MG_X_INTERP, None, None),
    (f'Random {MG_X_RANDOM.shape[1]} (control)', MG_X_RANDOM, MG_X_RANDOM, None, None),
]
_mg_interp_name = MG_BLOCKS[5][0]
_mg_rand_name = MG_BLOCKS[6][0]
_MG_FAC = dict(SELECT_CONFIGS)
# configs at the smallest C of the grid (a class-mean-difference (prototype) classifier whose threshold depends on the fold's label composition), read off the factories themselves
_mg_cmin = [n for n, f in SELECT_CONFIGS if np.isclose(f.keywords.get('C', np.nan), float(np.min(SELECT_C_GRID)))]
assert _mg_cmin, '§7.1: no SELECT_CONFIGS entry at the smallest C'


def _mg_cls(X, protocol, label):
    """NestedSelect (§5.1) → per-fold chosen config (memo hit when the block already ran) → refit on
    every outer training fold with §5.1's scaler + binary-relevance code (one BLAS thread, as in the
    nested selection) → train F1 (mean over folds) and test predictions (asserted = NestedSelect)."""
    X = np.asarray(X, dtype=float)
    Yi = np.asarray(Y).astype(int)
    Yp, info = cv_predict_default(X, Yi, protocol, label=label, return_info=True)
    splits = cv_splits(protocol, X.shape[0])
    assert len(info['chosen']) == len(splits), f'§7.1 {label}: chosen configs ≠ outer folds'
    Yte, tr_f1, conv = np.zeros_like(Yi), [], [0]
    with threadpool_limits(limits=1):
        for (tr, te), cname in zip(splits, info['chosen']):
            sc = StandardScaler().fit(X[tr])
            A, B = sc.transform(X[tr]), sc.transform(X[te])
            tr_f1.append(_sc_f1(Yi[tr], _fit_predict_labels(_MG_FAC[cname], A, Yi[tr], A, conv)))
            Yte[te] = _fit_predict_labels(_MG_FAC[cname], A, Yi[tr], B, conv)
    assert np.array_equal(Yte, Yp), f'§7.1 {label} ({protocol}): refitted configs do not reproduce NestedSelect'
    return dict(train=float(np.mean(tr_f1)), test=_sc_f1(Yi, Yp), Yp=Yp, chosen=list(info['chosen']),
                counts=nms_counts_str(info))


def _mg_reg(X, y, protocol):
    """Nested ridge (§5.1) → test = out-of-sample R²; train = in-sample R² of the refitted fold model
    (the α that fold chose), mean over the outer folds."""
    X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
    yhat, alphas = nested_ridge_predict(X, y, protocol)
    tr_r2 = []
    for (tr, te), a in zip(cv_splits(protocol, len(y)), alphas):
        m = _ridge_pipe(a).fit(X[tr], y[tr])
        assert np.allclose(m.predict(X[te]), yhat[te], rtol=0, atol=1e-9), '§7.1: ridge refit ≠ nested ridge'
        p = m.predict(X[tr])
        tr_r2.append(1.0 - float(np.sum((y[tr] - p) ** 2)) / float(np.sum((y[tr] - y[tr].mean()) ** 2)))
    return dict(train=float(np.mean(tr_r2)), test=oos_r2(y, yhat, protocol), yhat=yhat)


# ── B1 · train vs test, per block and protocol ─────────────────────────────────
print(f'📊 §7.1 B1 · train vs test — {len(MG_BLOCKS)} blocks × {len(PROTOCOLS)} protocols ({", ".join(PROTOCOLS)})')
_mg_rows, _mg_cls_res = [], {}
for _pr in PROTOCOLS:
    for _nm, _Xc, _Xr, _rc, _rr in MG_BLOCKS:
        _c = _mg_cls(_Xc, _pr, f'§7.1 · {_nm}')
        _mg_cls_res[(_nm, _pr)] = _c
        _v, _a = _mg_reg(_Xr, ABL_Y_VAL, _pr), _mg_reg(_Xr, ABL_Y_ARO, _pr)
        _ok = ''
        if _rc is not None:                    # the same block, already scored in §6.3 / §6.4
            _ref = ABL_PRED[(_rc[0], _rc[1], _pr)]
            _row = next(r for r in ABL_ROWS if r['tier'] == _rc[0] and r['set_name'] == _rc[1]
                        and r['protocol'] == _pr)
            assert np.array_equal(_c['Yp'], _ref['cls']) and abs(_c['test'] - _row['f1_macro']) < 1e-12, \
                f'§7.1: {_nm} ({_pr}) test F1 {_c["test"]:.6f} ≠ the {_rc} row {_row["f1_macro"]:.6f}'
            _refr = ABL_PRED[(_rr[0], _rr[1], _pr)]
            assert (np.allclose(_v['yhat'], _refr['val'], rtol=0, atol=1e-9)
                    and np.allclose(_a['yhat'], _refr['aro'], rtol=0, atol=1e-9)), \
                f'§7.1: {_nm} ({_pr}) ridge predictions ≠ the {_rr} row'
            _ok = f'{_rc[0]} / {_rc[1]}'
        _mg_rows.append(dict(block=_nm, protocol=_pr, n_features=int(np.shape(_Xc)[1]),
                             train_f1=_c['train'], test_f1=_c['test'], gap_f1=_c['train'] - _c['test'],
                             n_folds=len(_c['chosen']),
                             n_folds_smallest_C=int(sum(ch in _mg_cmin for ch in _c['chosen'])),
                             chosen=_c['counts'],
                             train_r2_val=_v['train'], test_r2_val=_v['test'], gap_r2_val=_v['train'] - _v['test'],
                             train_r2_aro=_a['train'], test_r2_aro=_a['test'], gap_r2_aro=_a['train'] - _a['test'],
                             reproduces=_ok))
df_mg_gap = pd.DataFrame(_mg_rows)
df_mg_gap.to_csv(os.path.join(MG_DIR, 'mert_train_test_gap.csv'), index=False)
print('✅ pooled test F1 and ridge predictions reproduce the §6.3 / §6.4 rows for every block that exists '
      'there (MERT-PCA, MERT-full, CREPE, Essentia, DTW); the refitted per-fold configs reproduce NestedSelect')
for _pr in PROTOCOLS:
    print(f'\n── {PROTOCOL_LABEL[_pr]} ──')
    print(df_mg_gap.query('protocol == @_pr')[
        ['block', 'n_features', 'train_f1', 'test_f1', 'gap_f1', 'n_folds_smallest_C', 'n_folds',
         'train_r2_val', 'test_r2_val', 'train_r2_aro', 'test_r2_aro']].round(3).to_string(index=False))
    print(f'   chance ({_pr}): mean {CHANCE_FLOOR_MEAN[_pr]:.3f} · strict P95 {CHANCE_FLOOR_P95[_pr]:.3f} · '
          f'n_folds_smallest_C = outer folds whose NestedSelect choice was C = {min(SELECT_C_GRID):g} '
          "(a class-mean-difference (prototype) classifier whose threshold depends on the fold's label composition)")
    for _r in df_mg_gap.query('protocol == @_pr').itertuples():
        print(f'   {_r.block:<22} chosen: {_r.chosen}')

# ── B2 · validation curve under the primary protocol (fixed C, no inner selection) ─
_mg_svm = [(n, f) for n, f in SELECT_CONFIGS if n.startswith('SVM-Linear')]
assert len(_mg_svm) == len(SELECT_C_GRID), '§7.1: one SVM-Linear config per C of SELECT_C_GRID expected'
_mg_curve_sets = [(_mg_pca, X_A), (_mg_interp_name, MG_X_INTERP), (_mg_rand_name, MG_X_RANDOM)]
_mg_crow = []
print(f'\n📊 §7.1 B2 · validation curve ({_mg_pr0}) — SVM-Linear balanced at fixed C ∈ '
      f'{[float(f.keywords["C"]) for _, f in _mg_svm]}')
for _nm, _X in _mg_curve_sets:
    _X = np.asarray(_X, dtype=float)
    _Yi = np.asarray(Y).astype(int)
    for _cn, _fac in _mg_svm:
        _Yte, _trf, _conv = np.zeros_like(_Yi), [], [0]
        with threadpool_limits(limits=1):
            for _tr, _te in cv_splits(_mg_pr0, _X.shape[0]):
                _sc = StandardScaler().fit(_X[_tr])
                _A, _B = _sc.transform(_X[_tr]), _sc.transform(_X[_te])
                _trf.append(_sc_f1(_Yi[_tr], _fit_predict_labels(_fac, _A, _Yi[_tr], _A, _conv)))
                _Yte[_te] = _fit_predict_labels(_fac, _A, _Yi[_tr], _B, _conv)
        assert np.array_equal(_Yte, cv_predict_multilabel(_fac, _X, _Yi, _mg_pr0)), \
            f'§7.1: validation-curve test predictions ≠ cv_predict_multilabel ({_nm}, {_cn})'
        _mg_crow.append(dict(block=_nm, protocol=_mg_pr0, config=_cn, C=float(_fac.keywords['C']),
                             train_f1=float(np.mean(_trf)), test_f1=_sc_f1(_Yi, _Yte)))
df_mg_curve = pd.DataFrame(_mg_crow)
df_mg_curve.to_csv(os.path.join(MG_DIR, 'mert_validation_curve.csv'), index=False)
print(df_mg_curve.pivot(index='C', columns='block', values='train_f1').round(3)
      .add_prefix('train · ').join(df_mg_curve.pivot(index='C', columns='block', values='test_f1').round(3)
                                   .add_prefix('test · ')).to_string())

fig, ax = plt.subplots(figsize=(9, 5))
for (_nm, _), _col in zip(_mg_curve_sets, ('#8e44ad', '#2a9d2a', '#888888')):
    _s = df_mg_curve[df_mg_curve.block == _nm].sort_values('C')
    ax.plot(_s.C, _s.train_f1, color=_col, ls='--', marker='o', label=f'{_nm} — train')
    ax.plot(_s.C, _s.test_f1, color=_col, ls='-', marker='s', label=f'{_nm} — test ({_mg_pr0})')
ax.axhspan(CHANCE_FLOOR_MEAN[_mg_pr0], CHANCE_FLOOR_P95[_mg_pr0], color='#b00020', alpha=0.10, lw=0)
ax.axhline(CHANCE_FLOOR_MEAN[_mg_pr0], color='#b00020', ls='--', lw=1.0,
           label=f'chance mean ({_mg_pr0}) = {CHANCE_FLOOR_MEAN[_mg_pr0]:.3f}')
ax.axhline(CHANCE_FLOOR_P95[_mg_pr0], color='#b00020', ls=':', lw=1.0,
           label=f'chance P95 ({_mg_pr0}) = {CHANCE_FLOOR_P95[_mg_pr0]:.3f}')
ax.set_xscale('log')
ax.set_xlabel('C (SVM-Linear, class_weight balanced, fixed — no inner selection)')
ax.set_ylabel('F1-macro')
ax.set_title(f'§7.1 B2 — validation curve ({_mg_pr0}): train (dashed) vs test (solid)\n'
             'capacity overfitting = train rises while test stays in the chance band', fontsize=10)
ax.legend(fontsize=7, loc='upper left')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(MG_DIR, 'fig_mert_validation_curve.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── B3 · nearest-neighbour identity test (label-free: piece / condition are design metadata) ──
def _mg_z(X):
    """z-score over the 60 excerpts (no model fitted); constant columns dropped."""
    X = np.asarray(X, dtype=float)
    sd = X.std(axis=0)
    keep = sd > 1e-12
    return (X[:, keep] - X[:, keep].mean(axis=0)) / sd[keep]


_mg_piece = np.asarray(PIECE_GROUPS)
_mg_cond = df_expr['condition'].to_numpy()
assert list(df_expr['excerpt_id']) == list(eids), '§7.1: df_expr rows must follow eids'
_mg_spaces = [('MERT-full (cosine, raw)', np.asarray(X_MERT_FULL, dtype=float), 'cosine', True),
              (f'{_mg_pca} (z)', _mg_z(X_A), 'euclidean', True),
              ('CREPE (z)', _mg_z(ABL_BLOCKS['CREPE']), 'euclidean', False),
              ('Essentia (z)', _mg_z(ABL_BLOCKS['Essentia']), 'euclidean', False),
              ('DTW ref-free (z)', _mg_z(ABL_BLOCKS['DTW']), 'euclidean', False),
              (f'Interpretable {MG_X_INTERP.shape[1]} (z)', _mg_z(MG_X_INTERP), 'euclidean', False)]
_mg_n = len(eids)
_mg_perm = np.argsort(np.random.default_rng(SEED).random((MG_N_PERM, _mg_n)), axis=1)   # label shuffles
_mg_chance = {'piece': 2.0 / (_mg_n - 1), 'condition': 19.0 / (_mg_n - 1)}
_mg_nrow = []
for _nm, _X, _metric, _is_mert in _mg_spaces:
    _Dm = pairwise_distances(_X, metric=_metric)
    np.fill_diagonal(_Dm, np.inf)
    _nn = np.argmin(_Dm, axis=1)
    _rec = dict(space=_nm, metric=_metric, is_mert=_is_mert, n_dims=int(_X.shape[1]))
    for _lab, _g in (('piece', _mg_piece), ('condition', _mg_cond)):
        _codes = pd.factorize(_g)[0]
        _obs = float(np.mean(_codes[_nn] == _codes))
        _L = _codes[_mg_perm]                                    # (n_perm, n) permuted label vectors
        _null = np.mean(_L[:, _nn] == _L, axis=1)
        _rec.update({f'same_{_lab}_share': _obs, f'same_{_lab}_chance': _mg_chance[_lab],
                     f'same_{_lab}_null_mean': float(_null.mean()),
                     f'p_same_{_lab}': float((1 + np.sum(_null >= _obs - 1e-12)) / (1 + MG_N_PERM)),
                     f'silhouette_{_lab}': float(silhouette_score(_X, _codes, metric=_metric))})
    _mg_nrow.append(_rec)
df_mg_nn = pd.DataFrame(_mg_nrow)
df_mg_nn.to_csv(os.path.join(MG_DIR, 'mert_nn_identity.csv'), index=False)
print(f'\n📊 §7.1 B3 · nearest-neighbour identity (self excluded; chance same-piece {100 * _mg_chance["piece"]:.1f} %, '
      f'same-condition {100 * _mg_chance["condition"]:.1f} %; {MG_N_PERM} label shuffles, seed {SEED})')
print(df_mg_nn[['space', 'n_dims', 'same_piece_share', 'p_same_piece', 'same_condition_share',
                'p_same_condition', 'silhouette_piece', 'silhouette_condition']].round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
_xs = np.arange(len(df_mg_nn))
_cols = ['#8e44ad' if m else '#2a9d2a' for m in df_mg_nn.is_mert]
for _ax, _lab in zip(axes, ('piece', 'condition')):
    _ax.bar(_xs, 100 * df_mg_nn[f'same_{_lab}_share'], color=_cols, edgecolor='#333', linewidth=0.5)
    _ax.axhline(100 * _mg_chance[_lab], color='#b00020', ls='--', lw=1.1,
                label=f'chance {100 * _mg_chance[_lab]:.1f} %')
    for _x, (_v, _p) in enumerate(zip(df_mg_nn[f'same_{_lab}_share'], df_mg_nn[f'p_same_{_lab}'])):
        _ax.text(_x, 100 * _v + 0.8, f'p={_p:.3g}\nsil={df_mg_nn[f"silhouette_{_lab}"].iloc[_x]:+.2f}',
                 ha='center', va='bottom', fontsize=6.5)
    _ax.set_xticks(_xs)
    _ax.set_xticklabels(df_mg_nn.space, rotation=28, ha='right', fontsize=7.5)
    _ax.set_ylabel(f'nearest neighbour from the same {_lab} (%)')
    _ax.set_title(f'same {_lab}', fontsize=10)
    _ax.legend(fontsize=8)
    _ax.grid(axis='y', alpha=0.25)
fig.suptitle('§7.1 B3 — nearest-neighbour identity (purple = MERT spaces, green = interpretable) · '
             'label-free: piece / condition are design metadata', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(MG_DIR, 'fig_mert_nn_identity.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── B4 · which pre-declared rule fires ─────────────────────────────────────────
_g = df_mg_gap.query('protocol == @_mg_pr0').set_index('block')['gap_f1']
_cap_a = bool(_g[_mg_pca] >= _g[_mg_interp_name] + MG_GAP_MARGIN and _g[_mg_pca] >= _g[_mg_rand_name] + MG_GAP_MARGIN)
_cv = df_mg_curve[df_mg_curve.block == _mg_pca].sort_values('C')
_rise = _cv[_cv.train_f1 >= float(_cv.train_f1.iloc[0]) + MG_CURVE_RISE]
_cap_b = bool(len(_rise) > 0 and (_rise.test_f1 <= CHANCE_FLOOR_P95[_mg_pr0]).all())
_interp_best = float(df_mg_nn.loc[~df_mg_nn.is_mert, 'same_piece_share'].max())
_sc_rows = df_mg_nn[df_mg_nn.is_mert
                    & (df_mg_nn.same_piece_share >= MG_NN_FOLD * _mg_chance['piece'])
                    & (df_mg_nn.p_same_piece < 0.05)
                    & (df_mg_nn.same_piece_share > _interp_best)
                    & (df_mg_nn.p_same_condition >= 0.05)]
MG_VERDICT = dict(capacity_overfitting=bool(_cap_a and _cap_b), capacity_gap_part=_cap_a,
                  capacity_curve_part=_cap_b, n_rising_C=int(len(_rise)),
                  piece_identity_shortcut=bool(len(_sc_rows) > 0), shortcut_spaces=list(_sc_rows.space))
print(f'\n🔎 §7.1 B4 · rule evaluation ({_mg_pr0}):')
print(f'   train − test F1 gap: {_mg_pca} {_g[_mg_pca]:+.3f} · interpretable {_g[_mg_interp_name]:+.3f} · '
      f'random control {_g[_mg_rand_name]:+.3f} → part (a) {"✓" if _cap_a else "✗"}')
print(f'   validation curve: {len(_rise)} C value(s) with a risen train F1; their test F1 = '
      f'{[round(float(x), 3) for x in _rise.test_f1]} vs P95 {CHANCE_FLOOR_P95[_mg_pr0]:.3f} → part (b) '
      f'{"✓" if _cap_b else "✗"}')
print(f'   CAPACITY OVERFITTING: {"FIRES" if MG_VERDICT["capacity_overfitting"] else "does not fire"}')
print(f'   same-piece NN share: ' + ' · '.join(f'{r.space} {100 * r.same_piece_share:.1f} % (p {r.p_same_piece:.3g})'
                                              for r in df_mg_nn.itertuples()))
print(f'   PIECE-IDENTITY SHORTCUT: {"FIRES in " + ", ".join(MG_VERDICT["shortcut_spaces"]) if MG_VERDICT["piece_identity_shortcut"] else "does not fire"}')
if not (MG_VERDICT['capacity_overfitting'] or MG_VERDICT['piece_identity_shortcut']):
    print('   → NEITHER rule fires: the MERT collapse under piece is not explained by either mechanism at the '
          'pre-declared thresholds.')
elif MG_VERDICT['capacity_overfitting'] and MG_VERDICT['piece_identity_shortcut']:
    print('   → BOTH rules fire.')
# ── Empirical null (§5.3): the guard that draw 0 at d = 25 IS this random control, and the §7.1 rows of the read-out ──
_null_d0 = int(MG_X_RANDOM.shape[1])
if NULL_SEED == MG_RANDOM_SEED and _null_d0 in NULL_DIMS:
    assert np.array_equal(_null_X(_null_d0, 0), MG_X_RANDOM), '§7.1: §5.3 draw 0 ≠ the random control matrix'
    for _null_pr in sorted(set(df_null_draws['protocol'])):
        _null_g = df_mg_gap[(df_mg_gap['block'] == _mg_rand_name) & (df_mg_gap['protocol'] == _null_pr)]
        _null_f0 = df_null_draws[(df_null_draws.protocol == _null_pr) & (df_null_draws.d == _null_d0)
                                 & (df_null_draws.draw == 0)]
        assert len(_null_g) == 1 and len(_null_f0) == 1 and \
            abs(float(_null_f0['f1_macro'].iloc[0]) - float(_null_g['test_f1'].iloc[0])) < 1e-12, \
            f'§7.1: §5.3 draw 0 (d = {_null_d0}, {_null_pr}) does not reproduce the random control F1'
    print(f'✅ §5.3 draw 0 (d = {_null_d0}) reproduces the §7.1 random control F1')
_mg_rd = [emp_readout_rows('§7.1 block', _r.block, _r.protocol, int(_r.n_features), float(_r.test_f1),
                           note='random control' if _r.block == _mg_rand_name else '')
          for _r in df_mg_gap.itertuples()]
df_emp_null_readout = pd.concat([df_emp_null_readout[df_emp_null_readout.source != '§7.1 block'],
                                 pd.DataFrame(_mg_rd)], ignore_index=True)
df_emp_null_readout.to_csv(os.path.join(OUTPUT_DIR, 'emp_null_readout.csv'), index=False)
print(f'\n📊 §7.1 blocks vs their empirical null ({PRIMARY_PROTOCOL}; §5.3)')
print(df_emp_null_readout[(df_emp_null_readout.source == '§7.1 block')
                          & (df_emp_null_readout.protocol == PRIMARY_PROTOCOL)][
    ['set_name', 'n_features', 'f1_macro', 'null_label', 'emp_null_p95', 'emp_p', 'beats_emp_null', 'note']]
      .round(4).to_string(index=False))

print(f'\n✅ §7.1 — {MG_DIR}/mert_train_test_gap.csv, mert_validation_curve.csv, mert_nn_identity.csv, '
      'fig_mert_validation_curve.png, fig_mert_nn_identity.png · emp_null_readout.csv (§7.1 rows appended)')


## Part 7 · MERT diagnostics — what the embedding encodes — §7.2 MERT layer study (Phase C1) · piece identity · condition decoding · prediction profile

**DIAGNOSTIC — the deployed `MERT_LAYERS` is not changed by this cell.** **Decision (Phase C2, C2 obligation
6): `MERT_LAYERS` stays `[5, 6, 7]`; this study is diagnostic** (C1 run: the piece-identity shortcut is present
at every depth, no candidate beats the empirical null, the nested choice is unstable, and the suggested W09-11
was not adopted). Nothing here feeds `emb_mert`, `X_A`–`X_F`, `MERT_REG` or any later cell's model input; the
suggested candidate is printed as "SUGGESTED — not applied".

**Why.** The supervisors asked which MERT layers to use. §7.1 (B3 run) found a **piece-identity shortcut**
in the deployed embedding (nearest neighbour from the same piece: MERT-full 100 %, MERT-PCA25 90 %,
interpretable 57 %, chance 3.4 %; F1 LOOCV 0.62 → piece 0.31). Lower layers of an SSL music model are
usually more acoustic, upper layers more abstract, so piece identity and the encoding of *how* a passage
is played may peak at different depths. This cell measures that, layer by layer, on the §2.1c cache.

**Candidates (25), each a 768-d time-mean embedding per excerpt:** `L00` … `L12` (single hidden states;
`L00` = the transformer input), `W00-02` … `W10-12` (mean of 3 consecutive hidden states — `W05-07` is the
deployed embedding; it is asserted equal to `emb_mert` and then *is* `emb_mert`, so the reproduction checks
test the pipeline path, not GPU float noise) and `ALL` (mean of the 13). References in every table: the
interpretable layer (the 27 features of `INTERP_PRED_COLS`, §4.1 — `MG_X_INTERP` of §7.1) and `W05-07`,
marked *deployed*.

**STAGE A — no ratings** (only the piece and condition design metadata, never `Y`; cosine on the raw
embedding — the interpretable reference is z-scored over the 60 excerpts first, as in §7.1, then the same
cosine is used):
- **A1 · piece identity** — the share of excerpts whose nearest neighbour (self excluded) is from the same
  piece (chance 2/59), permutation p with the §7.1 scheme and seed (`MG_N_PERM` = 10 000 shuffles of the
  piece labels with seed `SEED`, which keep 3 per piece), silhouette by piece. Descriptive (no BH). For
  `W05-07` it is asserted equal to the §7.1 `MERT-full (cosine, raw)` row.
- **A2 · condition decoding across pieces** — *does the layer encode HOW it was played in a way that
  generalises to an unseen piece?* Each excerpt is **piece-centred** (minus the mean of its piece's 3
  renditions — label-free: it uses only that piece's own features, never a rating or a condition label);
  on the §5.1 LOPO folds a **parameter-free nearest-centroid classifier** (cosine) builds the MEC / EXP /
  EXG centroids from the 19 training pieces and assigns each of the 3 test renditions to its nearest
  centroid. Accuracy over the 60 (chance 1/3); permutation p from 1000 permutations of the condition labels
  **within** each piece (§5.1 `within_piece_permutation`, seed `SEED`).
- **A3 · within-piece ordering** — the share of the 20 pieces whose cosine d(MEC, EXG) is strictly the
  largest of the three within-piece distances (chance 1/3), same 1000 within-piece permutations.
- **BH-FDR across the 25 candidates**, separately for A2 and A3.

**PRE-DECLARED SUGGESTION RULE** (printed before the results):
1. among the candidates with A2 q < 0.05, take the one with the lowest A1 same-piece share;
2. if other passing candidates are within 5 percentage points of it, prefer a 3-layer window among them
   (ties inside a step: lowest A1 share, then lowest piece silhouette, then candidate order);
3. if none passes: *"no layer encodes condition beyond chance — keep W05-07"*.

**STAGE B — prediction profile (descriptive; NOT used to choose)** — `RUN_MERT_LAYER_STAGE_B`,
`MERT_LAYER_PROTOCOLS = ('piece', 'loo')`. Each candidate's MERT block is built exactly as the deployed
pipeline builds it: *classification* = §2.1b replicated (PCA with `MERT_PCA_VAR` = 0.95, `svd_solver='full'`,
fitted once on the 60 excerpts in `emb_mert` order — transductive and label-free, as deployed — float32 rows
as `bfv()` stores them); *regression* = the full 768 dims through the deployed `MERT_REG_REPR` path and the
nested ridge. Every candidate goes through `ablate()` (§6.3) in a new tier `6-mert-layers` with its own
reference rows (`W05-07` deployed + the interpretable layer); Δ vs `W05-07` by `abl_vs_ref` (paired
permutation, BH within tier × protocol). **HARD CHECK:** the `W05-07` rows must reproduce the §6.3 `MERT`
lattice row exactly (predictions, F1, r, R², both protocols) — asserted. Reported per candidate: piece F1,
LOOCV F1, the gap LOOCV − piece, R² valence / arousal under both protocols, the NestedSelect configuration
counts; the **empirical-null p of the piece F1 and its BH q across the 25** are added by the next cell
(§7.2, continued — the null itself is drawn in **§5.3**, before Part 6; the
null is computed there, after the Dummy floors), which re-exports `mert_layer_study.csv` and the figure.
"Beats `W05-07` beyond the empirical null" (pre-declared): F1 — ΔF1 > 0 with q < 0.05 (paired, tier BH)
**and** the candidate's empirical-null q < 0.05 (next cell); R² arousal — ΔR² > 0 with q < 0.05 (the B3 random
control showed that the regression scale has a valid zero: noise R² ≈ 0).

**NESTED LAYER SELECTION — how stable is a "best layer"?** (`RUN_MERT_LAYER_NESTED`, `piece` only). Outer =
the §5.1 piece folds; inner = LOPO over the 19 training pieces (the §5.1 `get_cv('piece')` splitter applied to
the training rows). The inner loop picks the candidate with **fixed estimators** (no configuration search,
for cost): classification = the deployed `SVM-Linear C=1` config (balanced) on a per-candidate PCA 95 % fitted
on the inner training rows only (scaler inside); regression (arousal and valence separately) =
`StandardScaler → RidgeCV` over the §5.1 α grid on the 768-d input (RidgeCV's own efficient leave-one-out
picks α *inside* the inner training rows). The candidate with the best pooled inner score (macro-F1 / SSE;
ties → candidate order) is refitted on the outer training fold and scores the outer test piece. Reported: the
nested outer score (F1, R² arousal, R² valence), the same fixed estimator always on `W05-07`, the chosen
candidate per outer fold and the frequency table of the choices. **The nested scores use fixed estimators —
a selection-stability diagnostic, not comparable with the headline numbers.** Cached in
`mert_layer_nested.csv` (+ `mert_layer_nested_pred.csv`), reused when the settings hash matches.

Outputs in `OUTPUT_DIR/mert_layers/`: `mert_layer_study.csv` (one row per candidate + the interpretable
reference, every Stage A / B metric), `mert_layer_nested.csv`, `mert_layer_nested_pred.csv`,
`fig_mert_layer_profile.png`; tier `6-mert-layers` joins `ablation_master.csv`.
**Runtime** (T4, 4 CPUs): Stage A ≈ 1–2 min · Stage B ≈ 35–45 min with both protocols (≈ 10–15 min with
`MERT_LAYER_PROTOCOLS = ('piece',)`) · nested ≈ 3–10 min (cached afterwards).

In [ ]:
# ── §7.2 MERT layer study (Phase C1) — DIAGNOSTIC · piece identity · condition decoding · prediction profile ──
# NEEDS : ablate, abl_frame, abl_vs_ref, ABL_PRED, ABL_ROWS, ABL_Y_ARO, ABL_Y_VAL, bh_fdr, CHANCE_FLOOR_MEAN,
#          CHANCE_FLOOR_P95, cv_splits, DEFAULT_CLF_NAME, delayed, df_expr, df_mg_nn, eids, emb_mert,
#          emb_mert_layers, EMOTION_LABELS, f1_score, get_cv, hashlib, MERT_LAYERS, MERT_N_HIDDEN, MERT_PCA_VAR,
#          MERT_REG, MERT_REG_REPR, MG_N_PERM, MG_X_INTERP, NMS_N_JOBS, oos_r2, OUTPUT_DIR, Parallel,
#          PIECE_GROUPS, Pipeline, plt, PRIMARY_PROTOCOL, PROTOCOLS, PROTOCOL_LABEL, RIDGE_ALPHAS, SEED,
#          SELECT_CONFIGS, StandardScaler, threadpool_limits, within_piece_permutation, X_A, X_MERT_FULL, Y,
#          _fit_predict_labels (§5.1), _mg_z (§7.1), _sc_f1 (§6.3) — every §5.1 / HELPERS / §6.3 / §7.1
#          name is read, never rebound
# MAKES : ML_DIR, RUN_MERT_LAYER_STAGE_B, RUN_MERT_LAYER_NESTED, MERT_LAYER_PROTOCOLS, ML_N_PERM_COND,
#          ML_SUGGEST_TOL, ML_TIER, ML_DEPLOYED, ML_CANDIDATES, ML_CAND_NAMES, ML_SET_NAME, ML_INTERP_LABEL,
#          ML_INTERP_NAME, ML_NESTED_CLF, ML_NESTED_SCORES, ML_SUGGESTED, df_mert_layer_a, df_mert_layer_b,
#          df_mert_layer_deltas, df_mert_layer_nested, df_mert_layer_nested_freq, df_mert_layer_study,
#          ml_plot_profile, _ml_* · mert_layers/mert_layer_study.csv, mert_layer_nested.csv,
#          mert_layer_nested_pred.csv, fig_mert_layer_profile.png · tier 6-mert-layers in ABL_ROWS
#          (ablation_master.csv re-exported)
# FEEDS : §7.2 (continued: empirical-null p + BH q of the candidates' piece F1 → re-exports mert_layer_study.csv and
#          the figure with the empirical P95), §10.2 / §10.3 (ablation_master.csv carries tier 6-mert-layers),
#          the author's MANUAL decision on MERT_LAYERS — nothing here changes it
# ═══════════════════════════════════════════════════════════════════════════════
# §7.2 · Which MERT layer? — DIAGNOSTIC ONLY (the deployed MERT_LAYERS is not changed here)
# ═══════════════════════════════════════════════════════════════════════════════
# Stage A reads design metadata only (piece, condition) — never a rating. Stage B / the nested selection read
# Y only to SCORE. Every outer split comes from §5.1 (cv_splits, get_cv); every fitted transform (PCA in the
# nested loop, StandardScaler) is fitted inside its training fold, except the Stage B PCA, which replicates
# the deployed §2.1b PCA (fitted once on the 60 excerpts, label-free) on purpose.
import time as _ml_time
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
from sklearn.metrics import pairwise_distances, silhouette_score

_ml_t0 = _ml_time.time()
ML_DIR = os.path.join(OUTPUT_DIR, 'mert_layers')
os.makedirs(ML_DIR, exist_ok=True)

# ── Flags / constants ───────────────────────────────────────────────────────────
RUN_MERT_LAYER_STAGE_B = True               # prediction profile through ablate() (tier 6-mert-layers)
RUN_MERT_LAYER_NESTED  = True               # nested layer selection — stability diagnostic, piece only
MERT_LAYER_PROTOCOLS   = ('piece', 'loo')   # Stage B protocols (primary first); ('piece',) for a quick run
ML_N_PERM_COND = 1000                       # within-piece condition permutations (A2, A3)
ML_SUGGEST_TOL = 0.05                       # suggestion rule, step 2: "within 5 percentage points"
ML_TIER        = '6-mert-layers'
if not (MERT_LAYER_PROTOCOLS and set(MERT_LAYER_PROTOCOLS) <= set(PROTOCOLS)
        and MERT_LAYER_PROTOCOLS[0] == PRIMARY_PROTOCOL):
    raise ValueError(f'§7.2: MERT_LAYER_PROTOCOLS must be a subset of {PROTOCOLS} starting with '
                     f'{PRIMARY_PROTOCOL!r}, got {MERT_LAYER_PROTOCOLS!r}')
print(f'§7.2 · MERT LAYER STUDY — **DIAGNOSTIC**: the deployed MERT_LAYERS = {list(MERT_LAYERS)} is NOT changed '
      'by this cell; the author decides by hand after reading it.')

# ── The 25 candidates (each a 768-d time-mean embedding per excerpt) ────────────────
_ml_dep_layers = [int(l) for l in MERT_LAYERS]
assert len(_ml_dep_layers) == 3 and _ml_dep_layers == list(range(_ml_dep_layers[0], _ml_dep_layers[0] + 3)), \
    '§7.2: the deployed MERT_LAYERS must be 3 consecutive hidden states to be one of the W candidates'
ML_DEPLOYED = f'W{_ml_dep_layers[0]:02d}-{_ml_dep_layers[-1]:02d}'
ML_CANDIDATES = {}
for _ml_l in range(MERT_N_HIDDEN):
    ML_CANDIDATES[f'L{_ml_l:02d}'] = dict(layers=(_ml_l,), kind='single layer', plot_x=float(_ml_l))
for _ml_s in range(MERT_N_HIDDEN - 2):
    ML_CANDIDATES[f'W{_ml_s:02d}-{_ml_s + 2:02d}'] = dict(layers=(_ml_s, _ml_s + 1, _ml_s + 2), kind='3-layer window',
                                                    plot_x=float(_ml_s + 1))         # marker at the centre layer
ML_CANDIDATES['ALL'] = dict(layers=tuple(range(MERT_N_HIDDEN)), kind=f'mean of all {MERT_N_HIDDEN}',
                            plot_x=float(MERT_N_HIDDEN))                          # drawn right of L12
ML_CAND_NAMES = list(ML_CANDIDATES)
assert len(ML_CAND_NAMES) == 25 and ML_DEPLOYED in ML_CANDIDATES, '§7.2: 25 candidates incl. the deployed window'
ML_INTERP_LABEL = f'Interpretable ({MG_X_INTERP.shape[1]})'

assert set(emb_mert_layers) == set(emb_mert), '§7.2: the §2.1c cache and emb_mert cover different excerpts'
_ML_EMB = {nm: {e: np.mean([emb_mert_layers[e][l] for l in c['layers']], axis=0) for e in emb_mert}
           for nm, c in ML_CANDIDATES.items()}                                    # emb_mert key order, float32
_ml_dmax = max(float(np.max(np.abs(_ML_EMB[ML_DEPLOYED][e] - np.asarray(emb_mert[e])))) for e in emb_mert)
assert all(np.allclose(_ML_EMB[ML_DEPLOYED][e], emb_mert[e], rtol=1e-3, atol=1e-4) for e in emb_mert), \
    f'§7.2: {ML_DEPLOYED} (mean of the cached hidden states {_ml_dep_layers}) ≠ emb_mert (max |diff| {_ml_dmax:.2e})'
_ML_EMB[ML_DEPLOYED] = {e: emb_mert[e] for e in emb_mert}    # the deployed vectors themselves (see the markdown)
print(f'   {ML_DEPLOYED} == emb_mert (max |diff| of the layer mean {_ml_dmax:.2e}) → the deployed vectors are used as '
      f'{ML_DEPLOYED}, so the reproduction checks test the pipeline path, not GPU float noise')
_ML_FULL = {nm: np.vstack([np.asarray(emb[e]) for e in eids]).astype(float) for nm, emb in _ML_EMB.items()}
assert np.array_equal(_ML_FULL[ML_DEPLOYED], np.asarray(X_MERT_FULL, dtype=float)), \
    f'§7.2: the {ML_DEPLOYED} full matrix differs from X_MERT_FULL'

# ── Design metadata (never a rating) ──────────────────────────────────────────────
assert list(df_expr['excerpt_id']) == list(eids), '§7.2: df_expr rows must follow eids'
_ml_n = len(eids)
_ml_piece = np.asarray(PIECE_GROUPS)
_ml_piece_codes = pd.factorize(_ml_piece)[0]                       # = the §7.1 coding
_ML_COND_CODE = {'MEC': 0, 'EXP': 1, 'EXG': 2}
assert df_expr['condition'].isin(list(_ML_COND_CODE)).all(), '§7.2: unknown condition code'
_ml_cond = df_expr['condition'].map(_ML_COND_CODE).to_numpy(dtype=int)
_ml_piece_rows = {p: np.flatnonzero(_ml_piece == p) for p in np.unique(_ml_piece)}
assert all(sorted(_ml_cond[r]) == [0, 1, 2] for r in _ml_piece_rows.values()), \
    '§7.2: every piece must have exactly one MEC, one EXP and one EXG rendition'
_ml_splits = cv_splits('piece', _ml_n)                             # §5.1 LOPO folds
_ml_perm_nn = np.argsort(np.random.default_rng(SEED).random((MG_N_PERM, _ml_n)), axis=1)   # = §7.1 A1 scheme
_ml_rng = np.random.default_rng(SEED)
_ml_perm_wp = [within_piece_permutation(_ml_piece, _ml_rng) for _ in range(ML_N_PERM_COND)]


def _ml_center(X):
    """Piece-centre: subtract, for each piece, the mean of ITS OWN 3 renditions. Label-free — it uses only
    that piece's features (never a rating or a condition label); a held-out piece is centred with its own
    renditions only, so nothing crosses a fold."""
    X = np.asarray(X, dtype=float)
    out = X.copy()
    for rows in _ml_piece_rows.values():
        out[rows] = X[rows] - X[rows].mean(axis=0)
    return out


def _ml_unit(X):
    return X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)


def _ml_ncc_prep(Xc):
    """Per §5.1 LOPO fold: the training rows and the unit-normalised test rows (fixed across permutations)."""
    return [(tr, te, Xc[tr], _ml_unit(Xc[te])) for tr, te in _ml_splits]


def _ml_ncc_acc(prep, codes):
    """A2 — parameter-free nearest-centroid classifier (cosine): MEC / EXP / EXG centroids from the training
    pieces, each test rendition assigned to the nearest centroid (ties → MEC, EXP, EXG order) → accuracy."""
    hit = 0
    for tr, te, Xtr, Ute in prep:
        M = np.eye(3)[codes[tr]]                                    # one-hot (n_train × 3)
        C = (M.T @ Xtr) / M.sum(axis=0)[:, None]                    # condition centroids (3 × d)
        hit += int(np.sum(np.argmax(Ute @ _ml_unit(C).T, axis=1) == codes[te]))
    return hit / _ml_n


def _ml_order_share(D, codes):
    """A3 — share of pieces whose cosine d(MEC, EXG) is STRICTLY the largest of the 3 within-piece distances."""
    k = 0
    for rows in _ml_piece_rows.values():
        r = {int(codes[i]): i for i in rows}
        d_me, d_mg, d_eg = D[r[0], r[1]], D[r[0], r[2]], D[r[1], r[2]]
        k += int(d_mg > d_me and d_mg > d_eg)
    return k / len(_ml_piece_rows)


def _ml_stage_a(X):
    """A1 / A2 / A3 for one space (cosine throughout)."""
    X = np.asarray(X, dtype=float)
    D = pairwise_distances(X, metric='cosine')
    Dn = D.copy()
    np.fill_diagonal(Dn, np.inf)
    nn = np.argmin(Dn, axis=1)
    a1 = float(np.mean(_ml_piece_codes[nn] == _ml_piece_codes))
    L = _ml_piece_codes[_ml_perm_nn]
    a1_null = np.mean(L[:, nn] == L, axis=1)
    prep = _ml_ncc_prep(_ml_center(X))
    a2 = _ml_ncc_acc(prep, _ml_cond)
    a2_null = np.array([_ml_ncc_acc(prep, _ml_cond[p]) for p in _ml_perm_wp])
    a3 = _ml_order_share(D, _ml_cond)
    a3_null = np.array([_ml_order_share(D, _ml_cond[p]) for p in _ml_perm_wp])
    return dict(a1_same_piece_share=a1, a1_p=float((1 + np.sum(a1_null >= a1 - 1e-12)) / (1 + MG_N_PERM)),
                a1_silhouette_piece=float(silhouette_score(X, _ml_piece_codes, metric='cosine')),
                a2_acc=a2, a2_null_mean=float(a2_null.mean()),
                a2_p=float((1 + np.sum(a2_null >= a2 - 1e-12)) / (1 + ML_N_PERM_COND)),
                a3_share=a3, a3_null_mean=float(a3_null.mean()),
                a3_p=float((1 + np.sum(a3_null >= a3 - 1e-12)) / (1 + ML_N_PERM_COND)))


# ── PRE-DECLARED SUGGESTION RULE — printed BEFORE any result ─────────────────────
print('\nPRE-DECLARED SUGGESTION RULE (descriptive; the suggested candidate is printed, never applied):')
print('   1. among the candidates with A2 q < 0.05 (BH over the 25), take the one with the lowest A1 same-piece share;')
print(f'   2. if other passing candidates are within {100 * ML_SUGGEST_TOL:.0f} percentage points of that share, prefer a '
      '3-layer window among them (ties: lowest A1 share, then lowest piece silhouette, then candidate order);')
print(f'   3. if none passes: "no layer encodes condition beyond chance — keep {ML_DEPLOYED}".')

# ── STAGE A — no ratings ────────────────────────────────────────────────────────────
print(f'\n📊 STAGE A · A1 piece identity ({MG_N_PERM} label shuffles, seed {SEED}, chance {100 * 2 / (_ml_n - 1):.1f} %) · '
      f'A2 condition decoding (piece-centred NCC, LOPO, chance 33.3 %) · A3 MEC–EXG largest (chance 33.3 %) — '
      f'{ML_N_PERM_COND} within-piece permutations, seed {SEED}')
_ml_ta = _ml_time.time()
_ml_arows = []
for _ml_i, _ml_nm in enumerate(ML_CAND_NAMES):
    _ml_c = ML_CANDIDATES[_ml_nm]
    _ml_arows.append(dict(candidate=_ml_nm, order=_ml_i, kind=_ml_c['kind'], layers=' '.join(map(str, _ml_c['layers'])),
                          plot_x=_ml_c['plot_x'], is_reference=False, deployed=(_ml_nm == ML_DEPLOYED),
                          **_ml_stage_a(_ML_FULL[_ml_nm])))
_ml_arows.append(dict(candidate=ML_INTERP_LABEL, order=len(ML_CAND_NAMES), kind='reference (z-scored, cosine)',
                      layers='', plot_x=np.nan, is_reference=True, deployed=False, **_ml_stage_a(_mg_z(MG_X_INTERP))))
df_mert_layer_a = pd.DataFrame(_ml_arows)
_ml_isc = ~df_mert_layer_a['is_reference']
for _ml_k in ('a2', 'a3'):                              # BH across the 25 candidates, separately for A2 and A3
    df_mert_layer_a[f'{_ml_k}_q'] = np.nan
    df_mert_layer_a.loc[_ml_isc, f'{_ml_k}_q'] = bh_fdr(df_mert_layer_a.loc[_ml_isc, f'{_ml_k}_p'].to_numpy(dtype=float))
# guard: the deployed A1 must equal the §7.1 MERT-full (cosine, raw) row (same space, metric, shuffles)
_ml_e = df_mg_nn[df_mg_nn['space'] == 'MERT-full (cosine, raw)']
_ml_d = df_mert_layer_a[df_mert_layer_a['candidate'] == ML_DEPLOYED].iloc[0]
assert len(_ml_e) == 1 and abs(_ml_d['a1_same_piece_share'] - float(_ml_e['same_piece_share'].iloc[0])) < 1e-12 \
    and abs(_ml_d['a1_p'] - float(_ml_e['p_same_piece'].iloc[0])) < 1e-12 \
    and abs(_ml_d['a1_silhouette_piece'] - float(_ml_e['silhouette_piece'].iloc[0])) < 1e-12, \
    f'§7.2: {ML_DEPLOYED} A1 does not reproduce the §7.1 MERT-full (cosine, raw) row'
print(f'✅ {ML_DEPLOYED} A1 reproduces §7.1 (MERT-full, cosine, raw) exactly · Stage A {_ml_time.time() - _ml_ta:.0f} s')
print(df_mert_layer_a[['candidate', 'kind', 'a1_same_piece_share', 'a1_p', 'a1_silhouette_piece', 'a2_acc', 'a2_p',
                       'a2_q', 'a3_share', 'a3_p', 'a3_q']].round(4).to_string(index=False))
_ml_e2 = df_mg_nn[df_mg_nn['space'].str.startswith('Interpretable')]
if len(_ml_e2):
    print(f'   (interpretable reference: cosine on the z-scored layer here; §7.1 reports the euclidean same-piece share '
          f'{100 * float(_ml_e2["same_piece_share"].iloc[0]):.1f} %)')

# ── The pre-declared suggestion ───────────────────────────────────────────────────
_ml_pass = df_mert_layer_a[_ml_isc & (df_mert_layer_a['a2_q'] < 0.05)]
if not len(_ml_pass):
    ML_SUGGESTED = None
    print(f'\n🔎 SUGGESTION: no layer encodes condition beyond chance — keep {ML_DEPLOYED}')
else:
    _ml_best = float(_ml_pass['a1_same_piece_share'].min())
    _ml_tolset = _ml_pass[_ml_pass['a1_same_piece_share'] <= _ml_best + ML_SUGGEST_TOL + 1e-12]
    _ml_win = _ml_tolset[_ml_tolset['kind'] == '3-layer window']
    _ml_pool = _ml_win if len(_ml_win) else _ml_tolset[_ml_tolset['a1_same_piece_share'] <= _ml_best + 1e-12]
    _ml_pick = _ml_pool.sort_values(['a1_same_piece_share', 'a1_silhouette_piece', 'order']).iloc[0]
    ML_SUGGESTED = str(_ml_pick['candidate'])
    print(f'\n🔎 SUGGESTED — not applied: {ML_SUGGESTED} ({_ml_pick["kind"]}; A1 same-piece '
          f'{100 * _ml_pick["a1_same_piece_share"]:.1f} %, A2 acc {_ml_pick["a2_acc"]:.3f} q {_ml_pick["a2_q"]:.3g}) — '
          f'{len(_ml_pass)} of 25 candidates pass A2 q < 0.05; lowest passing A1 {100 * _ml_best:.1f} %; '
          f'{len(_ml_tolset)} within {100 * ML_SUGGEST_TOL:.0f} pp ({len(_ml_win)} of them 3-layer windows). '
          f'The deployed MERT_LAYERS stays {list(MERT_LAYERS)} until the author decides.')

# ── STAGE B — prediction profile through the §6.3 machinery (descriptive; NOT used to choose) ──
ML_SET_NAME = {nm: f'MERT {nm}' + (' (deployed)' if nm == ML_DEPLOYED else '') for nm in ML_CAND_NAMES}
ML_INTERP_NAME = f'{ML_INTERP_LABEL} — reference'


def _ml_pca_block(emb):
    """§2.1b REPLICATED (nothing rebound): PCA(MERT_PCA_VAR, svd_solver='full') fitted once on the 60 excerpts
    in emb_mert order (transductive, label-free — exactly as deployed), components stored per excerpt as float32
    rows like bfv(), stacked in eids order → (classification matrix, number of components)."""
    D = pd.DataFrame(emb).T
    p = PCA(n_components=MERT_PCA_VAR, svd_solver='full').fit(D.values)
    Z = p.transform(D.values)
    fwd = dict(zip(D.index, Z))
    return np.array([np.array(fwd[e], dtype=np.float32).flatten() for e in eids]).astype(float), int(Z.shape[1])


df_mert_layer_b, df_mert_layer_deltas, _ml_npc = pd.DataFrame(), pd.DataFrame(), {}
_ml_tb = _ml_time.time()
if not RUN_MERT_LAYER_STAGE_B:
    print('\nℹ️  STAGE B skipped (RUN_MERT_LAYER_STAGE_B = False)')
else:
    print(f'\n📊 STAGE B · prediction profile — tier {ML_TIER}, protocols {", ".join(MERT_LAYER_PROTOCOLS)} '
          f'(classification = §2.1b PCA {MERT_PCA_VAR:.0%} replicated → {DEFAULT_CLF_NAME}; regression = '
          f'MERT_REG_REPR={MERT_REG_REPR!r} → nested ridge)')
    for _ml_nm in ML_CAND_NAMES:
        _ml_Xc, _ml_npc[_ml_nm] = _ml_pca_block(_ML_EMB[_ml_nm])
        _ml_Xr = _ML_FULL[_ml_nm] if MERT_REG_REPR == 'full' else _ml_Xc
        if _ml_nm == ML_DEPLOYED:
            _ml_dx = float(np.max(np.abs(_ml_Xc - np.asarray(X_A, dtype=float)))) if _ml_Xc.shape == np.shape(X_A) else np.inf
            assert np.isfinite(_ml_dx) and _ml_dx <= 1e-5, \
                f'§7.2: the replicated §2.1b PCA block of {ML_DEPLOYED} differs from X_A (max |diff| {_ml_dx:.2e})'
            assert (np.array_equal(_ml_Xr, np.asarray(MERT_REG, dtype=float)) if MERT_REG_REPR == 'full'
                    else np.allclose(_ml_Xr, np.asarray(MERT_REG, dtype=float), rtol=0, atol=1e-5)), \
                f'§7.2: the {ML_DEPLOYED} regression block differs from MERT_REG'
            print(f'   {ML_DEPLOYED}: replicated PCA block vs X_A max |diff| = {_ml_dx:.1e} '
                  f'({"bit-identical → NestedSelect memo hit" if _ml_dx == 0 else "NOT bit-identical — refitted"}); '
                  'regression block == MERT_REG')
        ablate(_ml_Xc, ML_SET_NAME[_ml_nm], tier=ML_TIER, X_reg=_ml_Xr, protocols=MERT_LAYER_PROTOCOLS,
               note=(f'{ML_CANDIDATES[_ml_nm]["kind"]} {ML_CANDIDATES[_ml_nm]["layers"]} · PCA {_ml_npc[_ml_nm]}d (cls) / '
                     f'{_ml_Xr.shape[1]}d (reg) · diagnostic'))
    ablate(MG_X_INTERP, ML_INTERP_NAME, tier=ML_TIER, protocols=MERT_LAYER_PROTOCOLS,
           note=f'reference: the {MG_X_INTERP.shape[1]}-feature predictive interpretable layer (§4.1 / §7.1 MG_X_INTERP)')

    # HARD CHECK — W05-07 must reproduce the §6.3 'MERT' lattice row exactly (predictions, F1, r, R², RMSE)
    for _ml_pr in MERT_LAYER_PROTOCOLS:
        _ml_a, _ml_b = ABL_PRED[(ML_TIER, ML_SET_NAME[ML_DEPLOYED], _ml_pr)], ABL_PRED[('1-blocks', 'MERT', _ml_pr)]
        _ml_ra = next(r for r in ABL_ROWS if r['tier'] == ML_TIER and r['set_name'] == ML_SET_NAME[ML_DEPLOYED]
                   and r['protocol'] == _ml_pr)
        _ml_rb = next(r for r in ABL_ROWS if r['tier'] == '1-blocks' and r['set_name'] == 'MERT' and r['protocol'] == _ml_pr)
        assert (np.array_equal(_ml_a['cls'], _ml_b['cls']) and np.array_equal(_ml_a['val'], _ml_b['val'])
                and np.array_equal(_ml_a['aro'], _ml_b['aro'])), \
            f'§7.2 HARD CHECK: {ML_DEPLOYED} predictions differ from the §6.3 MERT row ({_ml_pr})'
        for _ml_k in ('f1_macro', 'r_valence', 'r2_val', 'rmse_val', 'r_arousal', 'r2_aro', 'rmse_aro'):
            assert abs(float(_ml_ra[_ml_k]) - float(_ml_rb[_ml_k])) < 1e-12, \
                f'§7.2 HARD CHECK: {ML_DEPLOYED} {_ml_k} {_ml_ra[_ml_k]:.6f} ≠ §6.3 MERT {_ml_rb[_ml_k]:.6f} ({_ml_pr})'
    print(f'✅ HARD CHECK — {ML_DEPLOYED} reproduces the §6.3 MERT lattice row exactly '
          f'(predictions, F1, r, R², RMSE under {", ".join(MERT_LAYER_PROTOCOLS)})')

    df_mert_layer_deltas = abl_vs_ref(ML_TIER, None, protocols=MERT_LAYER_PROTOCOLS,
                                      ref_map={ML_SET_NAME[nm]: ML_SET_NAME[ML_DEPLOYED]
                                               for nm in ML_CAND_NAMES if nm != ML_DEPLOYED})
    _ml_inv = {v: k for k, v in ML_SET_NAME.items()}
    _ml_inv[ML_INTERP_NAME] = ML_INTERP_LABEL
    _ml_bt = abl_frame().query('tier == @ML_TIER')
    _ml_brows = []
    for _ml_sn, _ml_sub in _ml_bt.groupby('set_name', sort=False):
        _ml_r = dict(candidate=_ml_inv[_ml_sn], n_pca=_ml_npc.get(_ml_inv[_ml_sn], np.nan))
        for _ml_x in _ml_sub.itertuples():
            _ml_r.update({f'f1_{_ml_x.protocol}': _ml_x.f1_macro, f'r2_val_{_ml_x.protocol}': _ml_x.r2_val,
                       f'r2_aro_{_ml_x.protocol}': _ml_x.r2_aro, f'r_val_{_ml_x.protocol}': _ml_x.r_valence,
                       f'r_aro_{_ml_x.protocol}': _ml_x.r_arousal, f'configs_{_ml_x.protocol}': _ml_x.selected})
        _ml_brows.append(_ml_r)
    df_mert_layer_b = pd.DataFrame(_ml_brows)
    if {'f1_piece', 'f1_loo'} <= set(df_mert_layer_b.columns):
        df_mert_layer_b['gap_f1_loo_minus_piece'] = df_mert_layer_b['f1_loo'] - df_mert_layer_b['f1_piece']
    if len(df_mert_layer_deltas):
        for _ml_pr, _ml_dsub in df_mert_layer_deltas.groupby('protocol'):
            _ml_dd = _ml_dsub.assign(candidate=_ml_dsub['set_name'].map(_ml_inv))[
                ['candidate', 'd_f1', 'q_f1', 'd_r2_aro', 'q_r2_aro', 'd_r2_val', 'q_r2_val']]
            _ml_dd.columns = ['candidate'] + [f'{c}_vs_dep_{_ml_pr}' for c in _ml_dd.columns[1:]]
            df_mert_layer_b = df_mert_layer_b.merge(_ml_dd, on='candidate', how='left')
    for _ml_pr in MERT_LAYER_PROTOCOLS:
        print(f'\n── {PROTOCOL_LABEL[_ml_pr]} — Δ vs {ML_DEPLOYED} (paired permutation, BH within tier × protocol) ──')
        _ml_cols = ['candidate', 'n_pca', f'f1_{_ml_pr}', f'd_f1_vs_dep_{_ml_pr}', f'q_f1_vs_dep_{_ml_pr}', f'r2_aro_{_ml_pr}',
                 f'd_r2_aro_vs_dep_{_ml_pr}', f'q_r2_aro_vs_dep_{_ml_pr}', f'r2_val_{_ml_pr}', f'q_r2_val_vs_dep_{_ml_pr}']
        print(df_mert_layer_b[[c for c in _ml_cols if c in df_mert_layer_b.columns]].round(4).to_string(index=False))
    if 'gap_f1_loo_minus_piece' in df_mert_layer_b.columns:
        print('\n   optimism gap F1 (LOOCV − piece):')
        print(df_mert_layer_b[['candidate', 'f1_piece', 'f1_loo', 'gap_f1_loo_minus_piece']].round(4)
              .to_string(index=False))
    print(f'\n   NestedSelect configuration counts ({PRIMARY_PROTOCOL}):')
    for _ml_r in df_mert_layer_b.itertuples():
        print(f'   {_ml_r.candidate:<18} {getattr(_ml_r, "configs_" + PRIMARY_PROTOCOL)}')
    print(f'   chance ({PRIMARY_PROTOCOL}): Dummy mean {CHANCE_FLOOR_MEAN[PRIMARY_PROTOCOL]:.3f} · strict P95 '
          f'{CHANCE_FLOOR_P95[PRIMARY_PROTOCOL]:.3f} — the empirical-null p / q of every candidate are added in the next cell')
    print(f'   Stage B {(_ml_time.time() - _ml_tb) / 60:.1f} min')

# ── NESTED LAYER SELECTION — how stable is a "best layer"? (piece only, FIXED estimators) ──
ML_NESTED_CLF = next((n for n, f in SELECT_CONFIGS if n.startswith('SVM-Linear')
                      and np.isclose(float(f.keywords.get('C', np.nan)), 1.0)), None)
assert ML_NESTED_CLF is not None, '§7.2: no SVM-Linear C=1 entry in SELECT_CONFIGS'
_ml_fac_nested = dict(SELECT_CONFIGS)[ML_NESTED_CLF]
_ML_NESTED_CSV = os.path.join(ML_DIR, 'mert_layer_nested.csv')
_ML_NESTED_PRED_CSV = os.path.join(ML_DIR, 'mert_layer_nested_pred.csv')
_ML_TARGETS = [('cls', 'class F1'), ('aro', 'R² arousal'), ('val', 'R² valence')]


def _ml_nested_fold(Xs, Yi, yv, ya, tr, te, inner, fac, alphas, pca_var, dep):
    """One outer piece fold of the nested layer selection (pure function of its arguments — joblib worker).
    Xs = (K, n, 768) candidate stack; inner = the LOPO splits of the training rows (indices into tr).
    Classification = PCA(pca_var) fitted on the inner training rows → StandardScaler → `fac` per label
    (binary relevance, §5.1 _fit_predict_labels); regression = StandardScaler → RidgeCV(alphas) on the 768 dims.
    Candidate choice = best pooled inner macro-F1 / smallest pooled inner SSE, ties → candidate order."""
    with threadpool_limits(limits=1):
        conv = [0]

        def _cls(Xa, Ya, Xb):
            p = PCA(n_components=pca_var, svd_solver='full').fit(Xa)
            A, B = p.transform(Xa), p.transform(Xb)
            sc = StandardScaler().fit(A)
            return _fit_predict_labels(fac, sc.transform(A), Ya, sc.transform(B), conv)

        def _reg(Xa, y_, Xb):
            return Pipeline([('sc', StandardScaler()), ('r', RidgeCV(alphas=alphas))]).fit(Xa, y_).predict(Xb)

        K, Ytr = Xs.shape[0], Yi[tr]
        yt = {'val': yv[tr], 'aro': ya[tr]}
        f1_in, sse = np.zeros(K), {'val': np.zeros(K), 'aro': np.zeros(K)}
        for k in range(K):
            Xk = Xs[k][tr]
            oof = np.zeros_like(Ytr)
            for itr, iva in inner:
                oof[iva] = _cls(Xk[itr], Ytr[itr], Xk[iva])
                for t in ('val', 'aro'):
                    sse[t][k] += float(np.sum((yt[t][iva] - _reg(Xk[itr], yt[t][itr], Xk[iva])) ** 2))
            f1_in[k] = float(f1_score(Ytr, oof, average='macro', zero_division=0))
        kc = int(np.flatnonzero(f1_in >= f1_in.max() - 1e-12)[0])
        kr = {t: int(np.flatnonzero(sse[t] <= sse[t].min() * (1.0 + 1e-9) + 1e-15)[0]) for t in sse}
        sst = {t: float(np.sum((yt[t] - yt[t].mean()) ** 2)) for t in yt}
        out = dict(k_cls=kc, k_val=kr['val'], k_aro=kr['aro'], inner_cls=f1_in,
                   inner_val=1.0 - sse['val'] / sst['val'], inner_aro=1.0 - sse['aro'] / sst['aro'],
                   cls_nested=_cls(Xs[kc][tr], Ytr, Xs[kc][te]), cls_fixed=_cls(Xs[dep][tr], Ytr, Xs[dep][te]))
        for t, y_ in (('val', yv), ('aro', ya)):
            out[f'{t}_nested'] = _reg(Xs[kr[t]][tr], y_[tr], Xs[kr[t]][te])
            out[f'{t}_fixed'] = _reg(Xs[dep][tr], y_[tr], Xs[dep][te])
        out['n_conv'] = conv[0]
    return out


df_mert_layer_nested, df_mert_layer_nested_freq, ML_NESTED_SCORES = pd.DataFrame(), pd.DataFrame(), {}
_ml_tn = _ml_time.time()
if not RUN_MERT_LAYER_NESTED:
    print('\nℹ️  NESTED layer selection skipped (RUN_MERT_LAYER_NESTED = False)')
else:
    _ml_lopo, _ = get_cv('piece')                 # the §5.1 LOPO splitter, applied to each outer training fold
    _ml_jobs = []
    for _ml_tr, _ml_te in _ml_splits:
        _ml_inner = list(_ml_lopo.split(np.zeros((len(_ml_tr), 1)), groups=PIECE_GROUPS[_ml_tr]))
        assert len(_ml_inner) == len(np.unique(PIECE_GROUPS[_ml_tr])) and not set(PIECE_GROUPS[_ml_te]) & set(PIECE_GROUPS[_ml_tr]), \
            '§7.2: inner LOPO must cover every training piece and never the test piece'
        _ml_jobs.append((_ml_tr, _ml_te, _ml_inner))
    _ML_STACK = np.stack([_ML_FULL[nm] for nm in ML_CAND_NAMES])
    _ml_dep = ML_CAND_NAMES.index(ML_DEPLOYED)
    _ml_Yi = np.asarray(Y).astype(int)
    _ml_h = hashlib.sha1()
    for _ml_arr in (_ML_STACK, _ml_Yi, ABL_Y_VAL, ABL_Y_ARO, np.asarray(PIECE_GROUPS).astype(str),
                 np.asarray(RIDGE_ALPHAS, dtype=float)):
        _ml_arr = np.ascontiguousarray(_ml_arr)
        _ml_h.update(repr((_ml_arr.shape, _ml_arr.dtype.str)).encode())
        _ml_h.update(_ml_arr.tobytes())
    _ml_h.update(repr((ML_CAND_NAMES, ML_DEPLOYED, ML_NESTED_CLF, repr(_ml_fac_nested), float(MERT_PCA_VAR),
                       'StandardScaler+RidgeCV(gcv)', 'inner LOPO', 'C1-v1')).encode())
    _ml_sig = _ml_h.hexdigest()
    _ml_lab = list(EMOTION_LABELS)
    _ml_cached = False
    if os.path.exists(_ML_NESTED_CSV) and os.path.exists(_ML_NESTED_PRED_CSV):
        _ml_f, _ml_p = pd.read_csv(_ML_NESTED_CSV), pd.read_csv(_ML_NESTED_PRED_CSV)
        _ml_cached = ('settings_sha1' in _ml_f.columns and 'settings_sha1' in _ml_p.columns
                      and set(_ml_f['settings_sha1']) == {_ml_sig} and set(_ml_p['settings_sha1']) == {_ml_sig}
                      and len(_ml_f) == len(_ML_TARGETS) * len(_ml_splits) and list(_ml_p['excerpt_id']) == list(eids))
        print(f'\n📂 mert_layer_nested.csv found — settings {"MATCH → loaded (no refit)" if _ml_cached else "DIFFER → recomputing"}')
    if _ml_cached:
        df_mert_layer_nested = _ml_f
        _ml_pred = _ml_p
    else:
        print(f'\n🔄 NESTED layer selection — {len(_ml_jobs)} outer piece folds × {len(ML_CAND_NAMES)} candidates × '
              f'inner LOPO ({ML_NESTED_CLF} on PCA {MERT_PCA_VAR:.0%} · StandardScaler → RidgeCV), '
              f'{NMS_N_JOBS} worker(s)...')
        _ml_res = Parallel(n_jobs=NMS_N_JOBS, backend='loky')(
            delayed(_ml_nested_fold)(_ML_STACK, _ml_Yi, ABL_Y_VAL, ABL_Y_ARO, _ml_tr, _ml_te, _ml_inner, _ml_fac_nested,
                                     np.asarray(RIDGE_ALPHAS, dtype=float), MERT_PCA_VAR, _ml_dep)
            for _ml_tr, _ml_te, _ml_inner in _ml_jobs)
        _ml_frows, _ml_P = [], {k: np.zeros((_ml_n, len(_ml_lab)), dtype=int) for k in ('cls_nested', 'cls_fixed')}
        _ml_P.update({k: np.full(_ml_n, np.nan) for k in ('val_nested', 'val_fixed', 'aro_nested', 'aro_fixed')})
        _ml_fold_of = np.full(_ml_n, -1)
        for _ml_i, ((_ml_tr, _ml_te, _), _ml_r) in enumerate(zip(_ml_jobs, _ml_res)):
            _ml_fold_of[_ml_te] = _ml_i
            for _ml_k in _ml_P:
                _ml_P[_ml_k][_ml_te] = _ml_r[_ml_k]
            for _ml_t, _ml_tl in _ML_TARGETS:
                _ml_kk = _ml_r[f'k_{_ml_t}']
                _ml_row = dict(fold=_ml_i, test_piece=str(PIECE_GROUPS[_ml_te][0]), target=_ml_tl,
                            chosen=ML_CAND_NAMES[_ml_kk], chosen_is_deployed=(_ml_kk == _ml_dep),
                            inner_chosen=float(_ml_r[f'inner_{_ml_t}'][_ml_kk]), inner_deployed=float(_ml_r[f'inner_{_ml_t}'][_ml_dep]))
                _ml_row.update({f'inner_{nm}': float(v) for nm, v in zip(ML_CAND_NAMES, _ml_r[f'inner_{_ml_t}'])})
                _ml_row['settings_sha1'] = _ml_sig
                _ml_frows.append(_ml_row)
        df_mert_layer_nested = pd.DataFrame(_ml_frows)
        _ml_pred = pd.DataFrame({'excerpt_id': eids, 'piece': PIECE_GROUPS, 'fold': _ml_fold_of})
        for _ml_k in ('cls_nested', 'cls_fixed'):
            for _ml_j, _ml_lb in enumerate(_ml_lab):
                _ml_pred[f'{_ml_k}_{_ml_lb}'] = _ml_P[_ml_k][:, _ml_j]
        for _ml_k in ('val_nested', 'val_fixed', 'aro_nested', 'aro_fixed'):
            _ml_pred[_ml_k] = _ml_P[_ml_k]
        _ml_pred['settings_sha1'] = _ml_sig
        assert (_ml_fold_of >= 0).all() and np.isfinite(_ml_pred[['val_nested', 'aro_nested']].to_numpy()).all()
        df_mert_layer_nested.to_csv(_ML_NESTED_CSV, index=False)
        _ml_pred.to_csv(_ML_NESTED_PRED_CSV, index=False)
        print(f'   {sum(r["n_conv"] for r in _ml_res)} ConvergenceWarnings · computed in {(_ml_time.time() - _ml_tn) / 60:.1f} min '
              f'→ {os.path.basename(_ML_NESTED_CSV)}, {os.path.basename(_ML_NESTED_PRED_CSV)}')
    _ml_cn = _ml_pred[[f'cls_nested_{l}' for l in _ml_lab]].to_numpy(dtype=int)
    _ml_cf = _ml_pred[[f'cls_fixed_{l}' for l in _ml_lab]].to_numpy(dtype=int)
    ML_NESTED_SCORES = {
        'nested (layer chosen per fold)': dict(
            f1=_sc_f1(_ml_Yi, _ml_cn), r2_aro=oos_r2(ABL_Y_ARO, _ml_pred['aro_nested'].to_numpy(float), 'piece'),
            r2_val=oos_r2(ABL_Y_VAL, _ml_pred['val_nested'].to_numpy(float), 'piece')),
        f'fixed {ML_DEPLOYED} (same estimators)': dict(
            f1=_sc_f1(_ml_Yi, _ml_cf), r2_aro=oos_r2(ABL_Y_ARO, _ml_pred['aro_fixed'].to_numpy(float), 'piece'),
            r2_val=oos_r2(ABL_Y_VAL, _ml_pred['val_fixed'].to_numpy(float), 'piece'))}
    print(f'\n📊 NESTED layer selection (piece; FIXED estimators — {ML_NESTED_CLF} on PCA {MERT_PCA_VAR:.0%} / '
          'StandardScaler → RidgeCV — a selection-stability diagnostic, NOT comparable with the headline)')
    print(pd.DataFrame(ML_NESTED_SCORES).T.round(4).to_string())
    df_mert_layer_nested_freq = (df_mert_layer_nested.groupby(['target', 'chosen']).size()
                                 .unstack('target', fill_value=0)
                                 .reindex(index=ML_CAND_NAMES, columns=[t for _, t in _ML_TARGETS], fill_value=0))
    print('\n   choice frequency over the outer folds (candidates never chosen omitted):')
    print(df_mert_layer_nested_freq[df_mert_layer_nested_freq.sum(axis=1) > 0].to_string())
    for _, _ml_tl in _ML_TARGETS:
        _ml_s = df_mert_layer_nested[df_mert_layer_nested['target'] == _ml_tl].sort_values('fold')
        print(f'   {_ml_tl:<11} per fold: ' + ' · '.join(f'{p}→{c}' for p, c in zip(_ml_s['test_piece'], _ml_s['chosen'])))
    print(f'   nested {(_ml_time.time() - _ml_tn) / 60:.1f} min')

# ── One table: Stage A + Stage B + Δ vs deployed + nested choice counts ────────────
df_mert_layer_study = df_mert_layer_a.copy()
if len(df_mert_layer_b):
    df_mert_layer_study = df_mert_layer_study.merge(df_mert_layer_b, on='candidate', how='left')
if len(df_mert_layer_nested_freq):
    _ml_nf = df_mert_layer_nested_freq.copy()
    _ml_nf.columns = [f'n_nested_chosen_{c.replace("²", "2").replace(" ", "_")}' for c in _ml_nf.columns]
    df_mert_layer_study = df_mert_layer_study.merge(_ml_nf, left_on='candidate', right_index=True, how='left')
df_mert_layer_study.to_csv(os.path.join(ML_DIR, 'mert_layer_study.csv'), index=False)


def ml_plot_profile(df, emp_p95=None, emp_label='', path=None):
    """fig_mert_layer_profile.png — x = hidden state 0–12 (ALL drawn at 13); single layers = lines, 3-layer
    windows = open squares at their centre layer, ALL = star; horizontal references per metric: the
    interpretable layer (dashed) and the deployed window (dotted); grey = chance; the F1 panel also draws the
    Dummy chance mean / strict P95 (piece) and, when given, the empirical-null P95 (§5.3)."""
    from matplotlib.lines import Line2D
    path = path or os.path.join(ML_DIR, 'fig_mert_layer_profile.png')
    cand, ref = df[~df['is_reference']], df[df['is_reference']].iloc[0]
    dep = cand[cand['deployed']].iloc[0]
    sing = cand[cand['kind'] == 'single layer'].sort_values('plot_x')
    win = cand[cand['kind'] == '3-layer window'].sort_values('plot_x')
    alr = cand[cand['candidate'] == 'ALL']
    has_b = 'f1_piece' in df.columns and df['f1_piece'].notna().any()
    fig, axes = plt.subplots(1, 2, figsize=(16, 5.8))
    panels = [(axes[0], 'Stage A — no ratings (design metadata only)',
               [('a1_same_piece_share', 'A1 same-piece NN share', '#8e44ad', 2.0 / (_ml_n - 1)),
                ('a2_acc', 'A2 condition decoding (piece-centred NCC)', '#2a9d2a', 1.0 / 3),
                ('a3_share', 'A3 d(MEC, EXG) largest', '#e07b39', 1.0 / 3)]),
              (axes[1], f'Stage B — prediction ({PRIMARY_PROTOCOL})',
               [('f1_piece', 'F1-macro (NestedSelect)', '#2a6dd9', None),
                ('r2_aro_piece', 'R² arousal (nested ridge)', '#b00020', None),
                ('r2_val_piece', 'R² valence (nested ridge)', '#16a085', None)])]
    for ax, title, mets in panels:
        ax.set_title(title, fontsize=10)
        ax.set_xticks(list(range(MERT_N_HIDDEN)) + [MERT_N_HIDDEN])
        ax.set_xticklabels([str(i) for i in range(MERT_N_HIDDEN)] + ['ALL'])
        ax.set_xlabel('MERT hidden state (0 = transformer input; windows at their centre layer)')
        ax.grid(alpha=0.25)
        ax.axvspan(dep['plot_x'] - 1, dep['plot_x'] + 1, color='#ffd54f', alpha=0.18, lw=0)
        if ax is axes[1] and not has_b:
            ax.text(0.5, 0.5, 'Stage B not run', ha='center', va='center', transform=ax.transAxes)
            continue
        handles = []
        for col, lab, c, chance in mets:
            ax.plot(sing['plot_x'], sing[col], '-o', color=c, ms=4, lw=1.4)
            ax.plot(win['plot_x'], win[col], 's', color=c, ms=8, mfc='none', mew=1.5)
            ax.plot(alr['plot_x'], alr[col], '*', color=c, ms=13)
            if np.isfinite(ref[col]):
                ax.axhline(ref[col], color=c, ls='--', lw=1.0)
            ax.axhline(dep[col], color=c, ls=':', lw=1.2)
            if chance is not None:
                ax.axhline(chance, color='#9e9e9e', ls='-', lw=0.8)
            handles.append(Line2D([], [], color=c, marker='o', ms=4, label=lab))
        if ax is axes[1]:
            ax.axhline(0, color='#333', lw=0.6)
            ax.axhline(CHANCE_FLOOR_MEAN[PRIMARY_PROTOCOL], color='#6d4c41', ls='--', lw=0.9)
            ax.axhline(CHANCE_FLOOR_P95[PRIMARY_PROTOCOL], color='#6d4c41', ls=':', lw=1.1)
            handles += [Line2D([], [], color='#6d4c41', ls='--', label=f'Dummy chance mean F1 {CHANCE_FLOOR_MEAN[PRIMARY_PROTOCOL]:.3f}'),
                        Line2D([], [], color='#6d4c41', ls=':', label=f'Dummy strict P95 F1 {CHANCE_FLOOR_P95[PRIMARY_PROTOCOL]:.3f}')]
            if emp_p95 is not None and np.isfinite(emp_p95):
                ax.axhline(emp_p95, color='#6a1b9a', ls='-.', lw=1.3)
                handles.append(Line2D([], [], color='#6a1b9a', ls='-.', label=f'empirical-null P95 F1 {emp_p95:.3f} {emp_label}'))
        else:
            handles.append(Line2D([], [], color='#9e9e9e', label='chance (A1 2/59 · A2 / A3 1/3)'))
        handles += [Line2D([], [], color='#555', marker='s', mfc='none', ls='none', label='3-layer window'),
                    Line2D([], [], color='#555', marker='*', ms=11, ls='none', label='ALL (mean of 13)'),
                    Line2D([], [], color='#555', ls='--', label=f'{ML_INTERP_LABEL} (reference)'),
                    Line2D([], [], color='#555', ls=':', label=f'{ML_DEPLOYED} (deployed)')]
        ax.legend(handles=handles, fontsize=7, loc='best')
    fig.suptitle('§7.2 MERT layer study — DIAGNOSTIC (the deployed MERT_LAYERS is not changed) · '
                 f'shaded = the deployed window {ML_DEPLOYED}', fontsize=11)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()


# (the profile figure is drawn by the next cell, after the empirical-null columns are merged — Phase C2)

# ── One-line answers ────────────────────────────────────────────────────────────────
_ml_mc = df_mert_layer_study[~df_mert_layer_study['is_reference']]
_ml_mi = df_mert_layer_study[df_mert_layer_study['is_reference']].iloc[0]
_ml_md = _ml_mc[_ml_mc['deployed']].iloc[0]
_ml_lo = _ml_mc.sort_values(['a1_same_piece_share', 'a1_silhouette_piece', 'order']).iloc[0]
_ml_h2 = _ml_mc.sort_values(['a2_acc', 'a2_q', 'order'], ascending=[False, True, True]).iloc[0]
_ml_h3 = _ml_mc.sort_values(['a3_share', 'a3_q', 'order'], ascending=[False, True, True]).iloc[0]
print('\n🔎 §7.2 SUMMARY (DIAGNOSTIC — nothing is applied)')
print(f'   · piece identity is LOWEST at {_ml_lo.candidate}: same-piece NN {100 * _ml_lo.a1_same_piece_share:.1f} % '
      f'(silhouette {_ml_lo.a1_silhouette_piece:+.3f}) vs deployed {ML_DEPLOYED} {100 * _ml_md.a1_same_piece_share:.1f} % '
      f'and the interpretable layer {100 * _ml_mi.a1_same_piece_share:.1f} % (chance {100 * 2 / (_ml_n - 1):.1f} %)')
print(f'   · condition is encoded BEST at {_ml_h2.candidate} (A2 acc {_ml_h2.a2_acc:.3f}, q {_ml_h2.a2_q:.3g}; '
      f'{int((_ml_mc.a2_q < 0.05).sum())}/25 with q < 0.05; deployed {_ml_md.a2_acc:.3f}, interpretable {_ml_mi.a2_acc:.3f}) · '
      f'A3 best {_ml_h3.candidate} {_ml_h3.a3_share:.2f} (q {_ml_h3.a3_q:.3g}; deployed {_ml_md.a3_share:.2f})')
if RUN_MERT_LAYER_STAGE_B and 'd_f1_vs_dep_piece' in df_mert_layer_study.columns:
    _ml_bf = _ml_mc[(_ml_mc['d_f1_vs_dep_piece'] > 0) & (_ml_mc['q_f1_vs_dep_piece'] < 0.05)]['candidate'].tolist()
    _ml_br = _ml_mc[(_ml_mc['d_r2_aro_vs_dep_piece'] > 0) & (_ml_mc['q_r2_aro_vs_dep_piece'] < 0.05)]['candidate'].tolist()
    print(f'   · beats {ML_DEPLOYED} on piece F1 (ΔF1 > 0, paired q < 0.05): {", ".join(_ml_bf) or "none"} — the '
          'empirical-null half of the rule is checked in the next cell; on piece R² arousal (ΔR² > 0, paired q < 0.05): '
          f'{", ".join(_ml_br) or "none"}')
else:
    print('   · Stage B not run — no prediction comparison')
if len(df_mert_layer_nested_freq):
    _ml_parts = []
    for _ml_tl in df_mert_layer_nested_freq.columns:
        _ml_v = df_mert_layer_nested_freq[_ml_tl]
        _ml_parts.append(f'{_ml_tl}: {_ml_v.idxmax()} {int(_ml_v.max())}/{int(_ml_v.sum())} folds ({int((_ml_v > 0).sum())} distinct)')
    print('   · nested choice stability (piece, fixed estimators): ' + ' · '.join(_ml_parts))
else:
    print('   · nested layer selection not run')
print(f'   · SUGGESTED — not applied: {ML_SUGGESTED or f"none (keep {ML_DEPLOYED})"}; MERT_LAYERS stays {list(MERT_LAYERS)} '
      '(decision recorded in Phase C2 — the study is diagnostic)')

if RUN_MERT_LAYER_STAGE_B:
    abl_frame().to_csv(os.path.join(OUTPUT_DIR, 'ablation_master.csv'), index=False)
    print(f'\n📦 ablation_master.csv re-exported with tier {ML_TIER} (§10.2 / §10.3 re-export it again with every tier)')
print(f'⏱  §7.2 runtime {(_ml_time.time() - _ml_t0) / 60:.1f} min')
print(f'✅ §7.2 — {ML_DIR}/mert_layer_study.csv, mert_layer_nested.csv, mert_layer_nested_pred.csv, '
      'fig_mert_layer_profile.png')

In [ ]:
# ── §7.2 (continued) MERT layer candidates vs the empirical null ────
# NEEDS : ABL_ROWS, bh_fdr, df_emp_null_readout, emp_readout_rows (§6.9), emp_null_ref (§5.3),
#          df_mert_layer_study, ML_CAND_NAMES, ML_DEPLOYED, ML_DIR, ML_SET_NAME, ML_TIER, ml_plot_profile,
#          MERT_LAYER_PROTOCOLS, RUN_MERT_LAYER_STAGE_B (§7.2), OUTPUT_DIR, PRIMARY_PROTOCOL, X_A
# MAKES : df_mert_layer_study_null · mert_layers/mert_layer_study.csv (+ empirical-null columns),
#          fig_mert_layer_profile.png, emp_null_readout.csv (§7.2 rows appended)
# FEEDS : §11.1 (layer study), the thesis' MERT reading
# ═══════════════════════════════════════════════════════════════════════════════
# (moved here from the second half of the former §4.5c: the empirical null now exists before Part 6 (§5.3),
#  so the layer candidates are read against it right after the layer study)
_ml_rd = [emp_readout_rows('§7.2 layer', _r['set_name'], _r['protocol'], int(_r['n_features']), float(_r['f1_macro']))
          for _r in ABL_ROWS if _r['tier'] == ML_TIER]
df_emp_null_readout = pd.concat([df_emp_null_readout[df_emp_null_readout.source != '§7.2 layer'],
                                 pd.DataFrame(_ml_rd)], ignore_index=True)
df_emp_null_readout.to_csv(os.path.join(OUTPUT_DIR, 'emp_null_readout.csv'), index=False)

df_mert_layer_study_null = df_mert_layer_study.copy()
if RUN_MERT_LAYER_STAGE_B and PRIMARY_PROTOCOL in MERT_LAYER_PROTOCOLS:
    _null_pp = df_emp_null_readout[df_emp_null_readout.protocol == PRIMARY_PROTOCOL]
    _null_ly = _null_pp[_null_pp.source == '§7.2 layer'].set_index('set_name')
    _null_lp = np.array([float(_null_ly.loc[ML_SET_NAME[nm], 'emp_p']) for nm in ML_CAND_NAMES])
    _null_lq = bh_fdr(_null_lp)
    _null_lmap = pd.DataFrame({'candidate': ML_CAND_NAMES,
                          f'emp_null_d_{PRIMARY_PROTOCOL}': [int(_null_ly.loc[ML_SET_NAME[nm], 'null_d']) for nm in ML_CAND_NAMES],
                          f'emp_null_p_{PRIMARY_PROTOCOL}': _null_lp, f'emp_null_q_{PRIMARY_PROTOCOL}': _null_lq})
    df_mert_layer_study_null = df_mert_layer_study_null.merge(_null_lmap, on='candidate', how='left')
    _null_qi = f'emp_null_q_{PRIMARY_PROTOCOL}'
    df_mert_layer_study_null[f'beats_emp_null_{PRIMARY_PROTOCOL}'] = df_mert_layer_study_null[_null_qi] < 0.05
    df_mert_layer_study_null.loc[df_mert_layer_study_null['is_reference'], f'beats_emp_null_{PRIMARY_PROTOCOL}'] = np.nan
    df_mert_layer_study_null.to_csv(os.path.join(ML_DIR, 'mert_layer_study.csv'), index=False)
    _null_lc = df_mert_layer_study_null[~df_mert_layer_study_null['is_reference']]
    print(f'📊 §7.2 candidates — piece F1 vs the empirical null (BH across the 25; Phase C3 rule: the primary null '
          'of the same PCA width, else the next larger primary width — "borrowed", conservative)')
    print(_null_lc[['candidate', 'n_pca', f'f1_{PRIMARY_PROTOCOL}', f'emp_null_d_{PRIMARY_PROTOCOL}',
               f'emp_null_p_{PRIMARY_PROTOCOL}', _null_qi, f'd_f1_vs_dep_{PRIMARY_PROTOCOL}',
               f'q_f1_vs_dep_{PRIMARY_PROTOCOL}']].round(4).to_string(index=False))
    _null_win = _null_lc[(_null_lc[f'd_f1_vs_dep_{PRIMARY_PROTOCOL}'] > 0) & (_null_lc[f'q_f1_vs_dep_{PRIMARY_PROTOCOL}'] < 0.05)
               & (_null_lc[_null_qi] < 0.05)]['candidate'].tolist()
    _null_dep_p = float(_null_lc.loc[_null_lc['candidate'] == ML_DEPLOYED, f'emp_null_p_{PRIMARY_PROTOCOL}'].iloc[0])
    print(f'🔎 §7.2: beats {ML_DEPLOYED} on piece F1 BEYOND the empirical null (ΔF1 > 0 with paired q < 0.05 AND '
          f'empirical-null q < 0.05): {", ".join(_null_win) or "none"} · {int((_null_lc[_null_qi] < 0.05).sum())}/25 '
          f'candidates beat the empirical null at all ({ML_DEPLOYED}: p {_null_dep_p:.3g})')
    _null_ek = emp_null_ref(np.nan, PRIMARY_PROTOCOL, int(np.shape(X_A)[1]))   # the deployed MERT-PCA width (Phase C3 rule)
    ml_plot_profile(df_mert_layer_study_null, emp_p95=_null_ek['p95'],
                    emp_label=f'({_null_ek["label"]})')
    print(f'   → {ML_DIR}/mert_layer_study.csv and fig_mert_layer_profile.png written with the empirical null')
else:
    ml_plot_profile(df_mert_layer_study)
    print('ℹ️  §7.2 Stage B not run under the primary protocol — no layer candidates to read against the null')
print('✅ §7.2 (continued) — mert_layer_study.csv, fig_mert_layer_profile.png, emp_null_readout.csv (§7.2 rows appended)')


## Part 7 · MERT diagnostics — what the embedding encodes — §7.3 MERT ↔ interpretable-feature correspondence · linear probe
**Hands on:** `df_probe` and `PROBE_R2` (the per-feature probe R², read by the §7.5 CREPE-unique rule) — the
tool-taxonomy evidence for the write-up.

MERT's 768 embedding dimensions are **anonymous** — individually uninterpretable — whereas
the 10 CREPE descriptors and the forwarded timing / DTW features are **named, physical** violin
techniques (vibrato, portamento, F0 statistics, tempo, IOIs…). This section asks the MERT side of
Note 2: *which anonymous MERT dimensions behave like the named techniques?* (the §7.3b probe also
covers the forwarded Essentia features). We answer it two complementary ways — a digested
per-technique correlation map (which raw dims track each technique) and a cross-validated
linear probe (can MERT actually **predict** each technique). Purely descriptive: no
classifier, label, or feature matrix is altered.

In [ ]:
# ── §7.3 MERT-dim ↔ feature correlation map ─────────────────────────
# NEEDS : CREPE_CANDIDATES, CREPE_LABELS, df_crepe, df_dtw, DTW_FEAT_LABELS, DTW_FORWARD, eids,
#          OUTPUT_DIR, X_MERT_FULL
# MAKES : CORR_COLS, CORR_SHORT, CREPE_COLS, CREPE_SHORT · fig_mert_crepe_topcorr.png
#          mert_crepe_corr_full.csv, mert_crepe_top_dims.csv
# FEEDS : — (terminal: writes files / figures only)
# ═══════════════════════════════════════════════════════════════════════════════
# §7.3 · Direct correlation map: which MERT dims track each named technique
# (N=60 → correlations are NOISY; exploratory, not confirmatory)
# ═══════════════════════════════════════════════════════════════════════════════
CREPE_COLS  = list(CREPE_CANDIDATES)            # the Phase B descriptors (§2.2), no diagnostic
CREPE_SHORT = dict(CREPE_LABELS)

# Timing joins pitch here: the question "does the SSL representation encode rubato?"
# is the same question as "does it encode vibrato?", asked of the DTW block (§2.4b).
CORR_COLS  = list(CREPE_COLS) + list(DTW_FORWARD)
CORR_SHORT = {**CREPE_SHORT, **DTW_FEAT_LABELS}

# ── Align the named features to X_A row order (eids) + defensive NaN impute ────
Ccr = pd.concat([df_crepe[CREPE_COLS], df_dtw[DTW_FORWARD]], axis=1).reindex(eids)[CORR_COLS]
Ccr = Ccr.apply(pd.to_numeric, errors='coerce')
Ccr = Ccr.fillna(Ccr.mean(numeric_only=True))

# ── Mask constant / all-zero columns (correlation undefined) ──────────────────
sd = Ccr.std(axis=0, ddof=0)
const_cols = [c for c in CORR_COLS if (sd[c] <= 1e-12) or pd.isna(sd[c])]
valid_cols = [c for c in CORR_COLS if c not in const_cols]
if const_cols:
    print(f'⚠️  Constant/all-zero columns masked: {[CORR_SHORT[c] for c in const_cols]}')

# ── Standardize both sides; Pearson r = z-scored dot product / N ──────────────
# Correspondence is a property of the REPRESENTATION → full 768-dim space,
# not the PCA-forwarded model input X_A (§2.1b).
Xz = StandardScaler().fit_transform(X_MERT_FULL)               # (60, 768)
N  = Xz.shape[0]
dim_names = [f'mert_{d:03d}' for d in range(X_MERT_FULL.shape[1])]
df_corr_full = pd.DataFrame(np.nan, index=dim_names, columns=[CORR_SHORT[c] for c in CORR_COLS])

if valid_cols:
    Cz = StandardScaler().fit_transform(Ccr[valid_cols].values)   # (60, k)
    corr_valid = (Xz.T @ Cz) / N                                  # (768, k) Pearson r
    for k, c in enumerate(valid_cols):
        df_corr_full[CORR_SHORT[c]] = corr_valid[:, k]
df_corr_full.to_csv(os.path.join(OUTPUT_DIR, 'mert_crepe_corr_full.csv'))

# ── (a) top-5 MERT dims per technique  ·  (b) strongest |r| per technique ─────
top_rows, strongest = [], []
for k, c in enumerate(valid_cols):
    r = corr_valid[:, k]
    for rank, d in enumerate(np.argsort(-np.abs(r))[:5], 1):
        top_rows.append({'feature': CORR_SHORT[c], 'rank': rank,
                         'mert_dim': dim_names[d], 'r': round(float(r[d]), 4)})
    db = int(np.argmax(np.abs(r)))
    strongest.append({'feature': CORR_SHORT[c],
                      'block': 'DTW' if c in DTW_FORWARD else 'CREPE',
                      'max_abs_r': round(float(abs(r[db])), 4),
                      'at_dim': dim_names[db], 'signed_r': round(float(r[db]), 4)})
df_top = pd.DataFrame(top_rows)
df_top.to_csv(os.path.join(OUTPUT_DIR, 'mert_crepe_top_dims.csv'), index=False)
df_strong = pd.DataFrame(strongest).sort_values('max_abs_r', ascending=False)

print('\n── Top-5 MERT dims per named technique (signed r, exploratory) ──')
for c in valid_cols:
    sub = df_top[df_top.feature == CORR_SHORT[c]]
    print(f'  {CORR_SHORT[c]:<28} ' + ', '.join(f'{t.mert_dim}({t.r:+.2f})' for t in sub.itertuples()))
print('\n── Strongest single-dim |r| per technique (represented at all?) ──')
print(df_strong.to_string(index=False))

# ── Digested heatmap: union of per-feature top-5 dims × all named techniques ──
if valid_cols:
    union = sorted(set(df_top['mert_dim']),
                   key=lambda d: -np.nanmax(np.abs(df_corr_full.loc[d].values.astype(float))))
    H = df_corr_full.loc[union, [CORR_SHORT[c] for c in CORR_COLS]]
    fig, ax = plt.subplots(figsize=(max(8, 0.62 * len(CORR_COLS)), max(4, 0.32 * len(union))))
    sns.heatmap(H, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, center=0,
                mask=H.isna(), cbar_kws={'label': 'Pearson r'}, ax=ax)
    ax.set_title('MERT dims most correlated with CREPE (pitch) + DTW (timing) techniques\n'
                 '(union of per-feature top-5)')
    ax.set_xlabel('named technique'); ax.set_ylabel('MERT dimension')
    fig.text(0.5, -0.01, 'N=60: correlations are noisy — exploratory, not confirmatory.',
             ha='center', fontsize=8, style='italic')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'fig_mert_crepe_topcorr.png'), dpi=150, bbox_inches='tight')
    plt.show()
print('\n✅ Saved: mert_crepe_top_dims.csv, mert_crepe_corr_full.csv, fig_mert_crepe_topcorr.png')


In [ ]:
# ── §7.3b Linear probe · can MERT predict each feature ──────────────
# NEEDS : CREPE_CANDIDATES, CREPE_LABELS, df_crepe, df_dtw, df_essentia, DTW_FEAT_LABELS,
#          DTW_FORWARD, eids,
#          ESSENTIA_FEAT_LABELS, ESSENTIA_FORWARD, nested_ridge_predict, OUTPUT_DIR,
#          PROTOCOL_LABEL, X_MERT_FULL
# MAKES : ALPHAS, CREPE_COLS, CREPE_SHORT, df_probe, N_COMPONENTS, PROBE_BLOCK, PROBE_COLS,
#          PROBE_LABEL, PROBE_PROTOCOL, PROBE_R2 (column → probe R²), _probe_pipe · fig_mert_crepe_probe_r2.png,
#          mert_crepe_probe_r2.csv
# FEEDS : §7.5 (PROBE_R2 → the CREPE-unique rule, CREPE_UNIQUE_R2_MAX)
# ═══════════════════════════════════════════════════════════════════════════════
# §7.3b · Linear probe: can MERT PREDICT each named interpretable feature?
# Headline test — CV R² of Ridge(MERT → feature). High R² ⇒ MERT already encodes it.
# Targets = the 10 Phase B CREPE descriptors + every forwarded Essentia and DTW feature.
# REPORTING ONLY: nothing is auto-dropped on the basis of this R².
# Uses the FULL 768-dim space (X_MERT_FULL): the question is what the MERT
# REPRESENTATION encodes, not what the PCA-forwarded model input retains.
# ═══════════════════════════════════════════════════════════════════════════════
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

CREPE_COLS  = list(CREPE_CANDIDATES)            # the Phase B descriptors (§2.2), no diagnostic
CREPE_SHORT = dict(CREPE_LABELS)

# ── Probe targets: CREPE ∪ forwarded Essentia ∪ forwarded DTW ────────────────
# Timing is asked exactly the same question as pitch and dynamics: does the SSL
# representation already encode rubato, or is DTW measuring something MERT cannot see?
PROBE_LABEL = {**CREPE_SHORT, **ESSENTIA_FEAT_LABELS, **DTW_FEAT_LABELS}
PROBE_BLOCK = {**{c: 'CREPE' for c in CREPE_COLS},
               **{c: 'Essentia' for c in ESSENTIA_FORWARD},
               **{c: 'DTW' for c in DTW_FORWARD}}
PROBE_COLS  = list(CREPE_COLS) + list(ESSENTIA_FORWARD) + list(DTW_FORWARD)

# ── Chosen regularization: StandardScaler → PCA(20) → Ridge (α nested) ───────
# Why PCA: 768 MERT dims >> N=60, so raw Ridge overfits badly. PCA compresses to
# 20 orthogonal components (fit INSIDE each CV fold → no leakage) before a Ridge.
# A7 (Phase A): the outer CV is GroupKFold(5) on the PIECES (get_cv('piece5'), §5.1) — no
# piece is ever split between training and test — and α is chosen INSIDE each outer
# training fold by an inner GroupKFold on the training pieces (nested_ridge_predict). The
# pre-audit code picked α by the outer CV score itself, which is optimistic. The grid now
# reaches 1e4: α=1000 (the old edge) won for 13/29 targets in the 2026-08-16 run.
N_COMPONENTS   = 20
ALPHAS         = [1.0, 10.0, 100.0, 1000.0, 10000.0]
PROBE_PROTOCOL = 'piece5'


def _probe_pipe(a):
    return Pipeline([('sc', StandardScaler()),
                     ('pca', PCA(n_components=N_COMPONENTS, random_state=42)),
                     ('rdg', Ridge(alpha=a, random_state=42))])


print(f'Probe: StandardScaler → PCA({N_COMPONENTS}) → Ridge (α ∈ {ALPHAS}, chosen by inner '
      f'piece-grouped CV), outer {PROTOCOL_LABEL[PROBE_PROTOCOL]}.')
print('R² here is across pieces; a feature that only encodes the piece (e.g. F0 mean = register) '
      'will now score lower than before.')
print(f'Rationale: 768 dims >> N={X_MERT_FULL.shape[0]} → PCA+Ridge guards against overfitting.')
print(f'Targets: {len(CREPE_COLS)} CREPE + {len(ESSENTIA_FORWARD)} Essentia-forward '
      f'+ {len(DTW_FORWARD)} DTW-forward\n')

Tgt = pd.concat([df_crepe, df_essentia[ESSENTIA_FORWARD],
                 df_dtw[DTW_FORWARD]], axis=1).reindex(eids)[PROBE_COLS]
Tgt = Tgt.apply(pd.to_numeric, errors='coerce')
Tgt = Tgt.fillna(Tgt.mean(numeric_only=True))

probe = []
for c in PROBE_COLS:
    y = Tgt[c].values.astype(float)
    if np.nanstd(y) <= 1e-12:                       # constant target → R² undefined
        probe.append({'column': c, 'feature': PROBE_LABEL[c], 'block': PROBE_BLOCK[c], 'alpha_median': np.nan,
                      'alpha_folds': '', 'n_components': N_COMPONENTS, 'cv_r2': np.nan})
        print(f'  [{PROBE_BLOCK[c]:<8}] {PROBE_LABEL[c]:<30} constant target — skipped'); continue
    yp, _al = nested_ridge_predict(X_MERT_FULL, y, PROBE_PROTOCOL, alphas=ALPHAS,
                                   make_pipe=_probe_pipe)
    r2 = r2_score(y, yp)
    _edge = int(np.sum(_al >= ALPHAS[-1]))
    probe.append({'column': c, 'feature': PROBE_LABEL[c], 'block': PROBE_BLOCK[c],
                  'alpha_median': float(np.median(_al)), 'alpha_folds': ' '.join(f'{a:g}' for a in _al),
                  'alpha_at_grid_edge': _edge, 'n_components': N_COMPONENTS,
                  'cv_r2': round(float(r2), 4)})
    print(f'  [{PROBE_BLOCK[c]:<8}] {PROBE_LABEL[c]:<30} α/fold={" ".join(f"{a:g}" for a in _al):<24} '
          f'CV R²={r2:+.3f}' + (f'   (α at grid edge in {_edge}/5 folds)' if _edge else ''))

df_probe = pd.DataFrame(probe).sort_values('cv_r2', ascending=False, na_position='last')
df_probe.to_csv(os.path.join(OUTPUT_DIR, 'mert_crepe_probe_r2.csv'), index=False)
# Phase C2: the per-feature probe R² by column name — §7.5 selects the CREPE-unique features from it
PROBE_R2 = {r['column']: (float(r['cv_r2']) if pd.notna(r['cv_r2']) else np.nan) for r in probe}

# ── Bar chart of CV R² per feature, coloured by block ────────────────────────
dfp   = df_probe.dropna(subset=['cv_r2'])
bcol  = {'CREPE': '#2a6dd9', 'Essentia': '#d98a2a', 'DTW': '#2a9d2a'}
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(dfp['feature'], dfp['cv_r2'],
       color=[bcol[b] for b in dfp['block']],
       edgecolor=['#333' if v > 0 else '#c44' for v in dfp['cv_r2']], linewidth=0.8)
ax.axhline(0, color='#333', lw=1)
ax.axhline(0.2, color='#888', ls=':', lw=1)
ax.set_ylabel('Cross-validated R²  (out-of-fold)')
ax.set_title('Can MERT linearly predict each interpretable feature?  (Ridge probe, '
             'GroupKFold(5) over pieces, nested α)\n'
             'blue = CREPE · orange = Essentia-fwd · green = DTW-fwd · '
             'dotted = R²=0.2 "well represented"')
ax.set_xticks(range(len(dfp))); ax.set_xticklabels(dfp['feature'], rotation=40, ha='right')
ax.grid(axis='y', alpha=0.15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_mert_crepe_probe_r2.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Note 2 answer: rank features by how strongly MERT encodes them ───────────
print('\n── Note 2 (MERT side) — features ranked by MERT representation (probe R²) ──')
for t in df_probe.itertuples():
    if pd.isna(t.cv_r2):
        verdict, r2s = 'n/a (constant)', 'nan'
    else:
        verdict = ('well-represented' if t.cv_r2 >= 0.2 else
                   'weakly represented' if t.cv_r2 > 0 else 'NOT linearly represented')
        r2s = f'{t.cv_r2:+.3f}'
    print(f'  [{t.block:<8}] {t.feature:<30} R²={r2s:<8} {verdict}')
good = [t.feature for t in df_probe.itertuples() if (not pd.isna(t.cv_r2)) and t.cv_r2 >= 0.2]
bad  = [t.feature for t in df_probe.itertuples() if (not pd.isna(t.cv_r2)) and t.cv_r2 <= 0]
print(f'\n➡️  MERT linearly encodes : {good if good else "none strongly"}')
print(f'➡️  MERT does NOT recover  : {bad if bad else "none fully absent"}')

# ── Essentia-specific read-out for the supervisor (reporting only, no auto-drop) ─
ess = df_probe[df_probe['block'] == 'Essentia'].dropna(subset=['cv_r2'])
if len(ess):
    redundant   = list(ess[ess.cv_r2 >= 0.2]['feature'])
    complement  = list(ess[ess.cv_r2 <  0.2]['feature'])
    print('\n── Essentia forwarding vs MERT (REPORTING ONLY — nothing auto-dropped) ──')
    print(f'  already encoded by MERT (R²≥0.2, redundant) : {redundant if redundant else "none"}')
    print(f'  genuinely complementary (R²<0.2)            : {complement if complement else "none"}')
    print('  ⚠️  Low probe R² means MERT lacks the information — it does NOT by itself')
    print('      prove the feature is emotion-relevant. That is what §7.5 tests.')

# ── DTW-specific read-out: does the SSL representation encode TIMING? ────────────
dtw_p = df_probe[df_probe['block'] == 'DTW'].dropna(subset=['cv_r2'])
if len(dtw_p):
    dtw_red = list(dtw_p[dtw_p.cv_r2 >= 0.2]['feature'])
    dtw_cmp = list(dtw_p[dtw_p.cv_r2 <  0.2]['feature'])
    print('\n── DTW / timing vs MERT (REPORTING ONLY — nothing auto-dropped) ──')
    print(f'  MERT already encodes (R²≥0.2)               : {dtw_red if dtw_red else "none"}')
    print(f'  genuinely complementary (R²<0.2)            : {dtw_cmp if dtw_cmp else "none"}')
    print('  This is the tool-taxonomy answer for the χρονικές παραλλαγές axis: whether a')
    print('  frozen SSL embedding recovers rubato, or whether an explicit alignment/timing')
    print('  algorithm is still required. §7.5 (MERT + DTW) tests whether it PAYS OFF for emotion.')
_n_edge = int(sum(r.get('alpha_at_grid_edge', 0) or 0 for r in probe))
if _n_edge:
    print(f'⚠️  α hit the grid edge ({ALPHAS[-1]:g}) in {_n_edge} outer folds across all targets')
print('✅ Saved: mert_crepe_probe_r2.csv, fig_mert_crepe_probe_r2.png  |  §7.3b complete')


## Part 7 · MERT diagnostics — what the embedding encodes — §7.4 Embedding shift vs rating shift

**Unique to the within-piece design.** How far does the MERT embedding (full 768 dims) move between two
renditions of the same piece, and does that distance track how far the **ratings** move? One measure
(Phase C2): Pearson r between the L2 distance and the **signed** ΔValence / ΔArousal for MEC → EXP,
EXP → EXG and MEC → EXG — 6 tests, BH-FDR → `q` (the former |Δ| version is removed). Fig 8 has one panel
per transition, each title with the signed r, p and q.

Two **exploratory** correlations of the MEC → EXG distance, outside the BH family and labelled as such: with
the **MEC-level arousal** of the piece (ceiling hypothesis) and with **ΔLUFS MEC → EXG** (level / timbre-change
hypothesis). Whether the embedding moves *most* between MEC and EXG is answered without any rating by
**§7.2 A3** (share of pieces whose d(MEC, EXG) is the largest of the three within-piece distances).


In [ ]:
# ── §7.4 Embedding shift vs rating shift ────────────────────────────
# NEEDS : bh_fdr (§0.3), df_all, df_essentia, emb_mert, OUTPUT_DIR
# MAKES : df_sh, df_sh_tests, df_sh_explore, SHIFT_TRANS, SHIFT_PIECE_LABEL · analysis_d_shifts.csv,
#          analysis_d_shift_tests.csv
# FEEDS : Fig 8 below, §11.1 (all outputs)
# ═══════════════════════════════════════════════════════════════════════════════
# §7.4 · Does the size of the MERT embedding shift between two renditions track the SIGNED rating shift?
# ═══════════════════════════════════════════════════════════════════════════════
# NOTE: uses emb_mert — the FULL 768-dim embedding, NOT emb_mert_fwd. Distance here is a representational-
# shift measure, not a model input, so it must use the full space; a PCA-forwarded subset would understate it.
# Phase C2: ONE measure — Pearson r between the L2 distance and the SIGNED ΔValence / ΔArousal (the |Δ|
# version is removed); BH-FDR over the 6 tests (3 transitions × 2 targets) → q. Two EXPLORATORY correlations
# of the MEC→EXG distance (not in the BH family, clearly labelled): with the MEC-level arousal of the piece
# (a ceiling hypothesis: a piece that is already intense at MEC has less room to move) and with ΔLUFS MEC→EXG
# (a level / timbre-change hypothesis: the embedding moves with loudness, not with emotion).
_LUFS = 'essentia_loudness_integrated_lufs_mean'
SHIFT_TRANS = [('MEC → EXP', 'ME', 'MEC', 'EXP'), ('EXP → EXG', 'EX', 'EXP', 'EXG'), ('MEC → EXG', 'MX', 'MEC', 'EXG')]
shifts = []
for piece in df_all['piece'].unique():
    sub  = df_all[df_all['piece']==piece]
    ids  = {r['condition']:r['excerpt_id'] for _,r in sub.iterrows()}
    if not all(c in ids for c in ['MEC','EXP','EXG']): continue
    if not all(ids[c] in emb_mert for c in ['MEC','EXP','EXG']): continue
    em   = {c: emb_mert[ids[c]] for c in ['MEC','EXP','EXG']}
    lb   = {c: sub[sub['condition']==c].iloc[0] for c in ['MEC','EXP','EXG']}
    rec  = {'piece': piece, 'aro_MEC': float(lb['MEC']['arousal_mean']),
            'dlufs_MX': (float(df_essentia.loc[ids['EXG'], _LUFS]) - float(df_essentia.loc[ids['MEC'], _LUFS])
                         if _LUFS in df_essentia.columns else np.nan)}
    for _, k, a, b in SHIFT_TRANS:
        rec[f'dist_{k}'] = float(np.linalg.norm(em[b] - em[a]))
        rec[f'dval_{k}'] = float(lb[b]['valence_mean'] - lb[a]['valence_mean'])
        rec[f'daro_{k}'] = float(lb[b]['arousal_mean'] - lb[a]['arousal_mean'])
    shifts.append(rec)
df_sh = pd.DataFrame(shifts)


# Phase C3 · short UNIQUE piece labels for Fig 8 (built once): the first 10 characters, extended only where two
# names would collide (e.g. the five Shostakovich pieces), up to the full name.
def _short_unique_labels(names, n0=10):
    names = [str(s) for s in names]
    out = {}
    for s in names:
        n = min(n0, len(s))
        while n < len(s) and any(o != s and o[:n] == s[:n] for o in names):
            n += 1
        out[s] = s[:n]
    assert len(set(out.values())) == len(out), f'§7.4: piece labels not unique: {out}'
    return out


SHIFT_PIECE_LABEL = _short_unique_labels(df_sh['piece'])

_t = []
for step, k, _, _ in SHIFT_TRANS:
    for dim, col in (('ΔValence', f'dval_{k}'), ('ΔArousal', f'daro_{k}')):
        x, y = df_sh[f'dist_{k}'].to_numpy(dtype=float), df_sh[col].to_numpy(dtype=float)
        r, p = pearsonr(x, y) if len(x) >= 3 else (np.nan, np.nan)
        _t.append(dict(transition=step, key=k, target=dim, n_pieces=len(x), r=float(r), p=float(p)))
df_sh_tests = pd.DataFrame(_t)
df_sh_tests['q'] = bh_fdr(df_sh_tests['p'].to_numpy(dtype=float))
_sig = lambda v: '***' if v < .001 else '**' if v < .01 else '*' if v < .05 else 'n.s.'
print('§7.4 — MERT embedding distance (L2, full 768) vs the SIGNED rating shift · BH over the 6 tests')
print(f"{'Transition':<12} {'Target':<10} {'r':>8} {'p':>9} {'q':>9}  sig(q)")
print('-' * 58)
for t in df_sh_tests.itertuples():
    print(f'{t.transition:<12} {t.target:<10} {t.r:>+8.3f} {t.p:>9.4f} {t.q:>9.4f}  {_sig(t.q)}')

# ── Exploratory (NOT in the BH family): what does the MEC → EXG distance track? ──
_e = []
for lab, col, hyp in (('arousal at MEC', 'aro_MEC', 'ceiling: little room to move when MEC is already intense'),
                      ('ΔLUFS MEC→EXG', 'dlufs_MX', 'level / timbre change: the embedding moves with loudness')):
    x, y = df_sh['dist_MX'].to_numpy(dtype=float), df_sh[col].to_numpy(dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    r, p = pearsonr(x[ok], y[ok]) if ok.sum() >= 3 else (np.nan, np.nan)
    _e.append(dict(analysis='EXPLORATORY', x='dist_MX (MEC→EXG)', y=lab, hypothesis=hyp, n_pieces=int(ok.sum()),
                   r=float(r), p=float(p)))
df_sh_explore = pd.DataFrame(_e)
print('\n🔍 EXPLORATORY (not in the BH family; hypotheses stated, nothing tested confirmatorily):')
for t in df_sh_explore.itertuples():
    print(f'   r(dist MEC→EXG, {t.y:<15}) = {t.r:+.3f} (p {t.p:.4f}, n = {t.n_pieces}) — {t.hypothesis}')
print('   ⇢ "does the embedding move most between MEC and EXG?" is answered WITHOUT ratings by §7.2 A3 '
      '(share of pieces whose d(MEC, EXG) is the largest of the three within-piece distances).')
df_sh.to_csv(os.path.join(OUTPUT_DIR,'analysis_d_shifts.csv'),index=False)
pd.concat([df_sh_tests.assign(analysis='BH family (6)'), df_sh_explore], ignore_index=True).to_csv(
    os.path.join(OUTPUT_DIR, 'analysis_d_shift_tests.csv'), index=False)
print('✅ §7.4 — analysis_d_shifts.csv, analysis_d_shift_tests.csv')


In [ ]:
# ── Fig 8: Embedding distance vs the signed rating shift — three transitions ──
# NEEDS : df_sh, df_sh_tests, OUTPUT_DIR, SHIFT_PIECE_LABEL, SHIFT_TRANS (§7.4) · MAKES : fig_embedding_shift.png · FEEDS : —
_cols = {'MEC → EXP': '#2a6dd9', 'EXP → EXG': '#2a9d2a', 'MEC → EXG': '#8e44ad'}
fig, axes = plt.subplots(1, 3, figsize=(18, 5.6))
fig.suptitle('§7.4 — MERT embedding distance (L2, full 768) vs the SIGNED rating shift · '
             'filled = ΔArousal, open = ΔValence · q = BH over the 6 tests', fontsize=12)
for ax, (step, k, _, _) in zip(axes, SHIFT_TRANS):
    col = _cols[step]
    x = df_sh[f'dist_{k}'].to_numpy(dtype=float)
    ttl = [step]
    for tgt, ycol, face in (('ΔArousal', f'daro_{k}', col), ('ΔValence', f'dval_{k}', 'none')):
        y = df_sh[ycol].to_numpy(dtype=float)
        ax.scatter(x, y, facecolors=face, edgecolors=col, alpha=0.85, s=60, zorder=3, label=tgt)
        if len(x) >= 3:
            m_, b_ = np.polyfit(x, y, 1)
            xs = np.linspace(x.min(), x.max(), 100)
            ax.plot(xs, m_ * xs + b_, '--' if tgt == 'ΔArousal' else ':', color=col, alpha=0.6, lw=1.4)
        _r = df_sh_tests[(df_sh_tests.key == k) & (df_sh_tests.target == tgt)].iloc[0]
        ttl.append(f'{tgt}: r={_r.r:+.3f}, p={_r.p:.4f}, q={_r.q:.4f}')
    for _, row in df_sh.iterrows():
        ax.annotate(SHIFT_PIECE_LABEL[row['piece']], (row[f'dist_{k}'], row[f'daro_{k}']), textcoords='offset points',
                    xytext=(5, 3), fontsize=6.5, color='#666')
    ax.set_title('\n'.join(ttl), fontsize=10)
    ax.axhline(0, color='#ccc', lw=1); ax.set_xlabel('L2 distance in MERT embedding space')
    ax.set_ylabel('rating shift (participant mean, signed)'); ax.grid(alpha=0.15); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'fig_embedding_shift.png'),dpi=150,bbox_inches='tight'); plt.show()


## Part 7 · MERT diagnostics — what the embedding encodes — §7.5 What each tool adds on top of MERT

**Starting from MERT, does adding a named-technique tool move the scores?** Rewritten in Phase C2:

- **No refit of existing rows.** MERT, MERT + CREPE, MERT + Essentia and MERT + DTW are §6.3 lattice rows; their
  per-excerpt predictions are read from `ABL_PRED` (classification: NestedSelect on **MERT-PCA25** + X;
  regression: nested ridge on **MERT-full 768** + X with the deployed equal-share block weighting).
- **ΔF1, ΔR² valence, ΔR² arousal vs MERT**, each with the lattice's **paired permutation p** and **BH-FDR
  within §7.5** (per protocol, every row × the three metrics). The q value is the verdict — the former
  "ADDS / HURTS" ±0.005 threshold is gone.
- **CREPE-unique by a rule** (replaces the hard-coded `CREPE_UNIQUE2` pair): the CREPE features whose §7.3b
  MERT-probe R² is below `CREPE_UNIQUE_R2_MAX` = 0.2 (§0.3). The cell prints the list (asserted non-empty)
  and each feature's number of declared median substitutions (§2.2 — vibrato rate is substituted for every
  excerpt without a vibrato window). **"MERT + CREPE-unique" is the only new fit** (same engine, same block
  weighting, tier `7.5-mert-plus`), with its permutation p vs MERT.

The removal view (what is lost when a tool is taken *out* of the full set) is the §6.3 Tier 2 table; since
§7.5 now reads the lattice rows directly, the former §6.2b cross-check is no longer needed and was removed.


In [ ]:
# ── §7.5 What each tool adds on top of MERT ─────────────────────────
# NEEDS : ablate, ABL_BLOCKS, ABL_BLOCKS_REG, ABL_PRED, ABL_ROWS, ABL_SEED, bh_fdr, CREPE_FORWARD, CREPE_LABELS,
#          CREPE_UNIQUE_R2_MAX (§0.3), MERT_REG_REPR, OUTPUT_DIR, perm_p, PRIMARY_PROTOCOL, PROBE_R2 (§7.3b),
#          PROTOCOLS, PROTOCOL_LABEL, REG_BLOCK_WEIGHTING, VIB_N_SUBST_NONOTES, VIB_N_SUBST_NOVIB (§2.2),
#          _sc_f1, _sc_r2 (§6.3)
# MAKES : BLOCK_F1 (protocol → row → F1), block_f1 (= PRIMARY), CREPE_UNIQUE, CREPE_SUBST_COUNTS, df_block,
#          T75_TIER · featsel_crepe_block.csv, fig_featsel_crepe_block.png (tier 7.5-mert-plus in ABL_ROWS)
# FEEDS : §11.1 (all outputs); ablation_master.csv (re-exported by §10.2 / §10.3)
# ═══════════════════════════════════════════════════════════════════════════════
# §7.5 · Starting from MERT, does adding a named-technique tool move the scores? (rewritten in Phase C2)
# ═══════════════════════════════════════════════════════════════════════════════
# The rows MERT, MERT + CREPE, MERT + Essentia, MERT + DTW already exist in the §6.3 lattice: they are NOT
# refitted here — their per-excerpt predictions are read from ABL_PRED (classification: NestedSelect on
# MERT-PCA25 + X; regression: nested ridge on MERT-full 768 + X with the deployed block weighting). ΔF1, ΔR²
# valence and ΔR² arousal vs MERT, each with the lattice's paired permutation p (perm_p, §6.3) and BH-FDR within
# §7.5 (per protocol, over every row × the three metrics). The q value is the verdict.
# The ONLY new fit: "MERT + CREPE-unique" — CREPE features that MERT does not encode, chosen by a RULE (no longer
# the hard-coded CREPE_UNIQUE2 pair): probe R² (§7.3b) < CREPE_UNIQUE_R2_MAX. Same engine (ablate), same block
# weighting, tier '7.5-mert-plus'.
T75_TIER = '7.5-mert-plus'

# ── The CREPE-unique rule ──
_cr_cols = list(CREPE_FORWARD)                      # column order of the lattice CREPE block (= X_B)
CREPE_UNIQUE = [c for c in _cr_cols if np.isfinite(PROBE_R2.get(c, np.nan)) and PROBE_R2[c] < CREPE_UNIQUE_R2_MAX]
print(f'§7.5 · CREPE-unique = CREPE features whose MERT-probe R² (§7.3b) < CREPE_UNIQUE_R2_MAX = {CREPE_UNIQUE_R2_MAX:g}:')
CREPE_SUBST_COUNTS = {c: 0 for c in _cr_cols}       # declared median substitutions of §2.2
CREPE_SUBST_COUNTS['crepe_vibrato_rate_hz'] = int(VIB_N_SUBST_NOVIB) + int(VIB_N_SUBST_NONOTES)
for _c in ('crepe_vibrato_extent_cents', 'crepe_vibrato_coverage'):
    if _c in CREPE_SUBST_COUNTS:
        CREPE_SUBST_COUNTS[_c] = int(VIB_N_SUBST_NONOTES)
for _c in _cr_cols:
    print(f'   {"✓" if _c in CREPE_UNIQUE else " "} {CREPE_LABELS.get(_c, _c):<22} probe R² {PROBE_R2.get(_c, np.nan):+.3f}'
          f' · median-substituted values: {CREPE_SUBST_COUNTS[_c]}/60')
assert CREPE_UNIQUE, (f'§7.5: no CREPE feature has a MERT-probe R² below {CREPE_UNIQUE_R2_MAX} — the CREPE-unique '
                      'row is undefined (rule from §0.3)')
print(f'   → CREPE-unique ({len(CREPE_UNIQUE)}): {[CREPE_LABELS.get(c, c) for c in CREPE_UNIQUE]}  ⚠️ a feature with '
      'many median-substituted values carries little information of its own')

# ── The one new fit: MERT + CREPE-unique ──
_idx = [_cr_cols.index(c) for c in CREPE_UNIQUE]
_Cu = ABL_BLOCKS['CREPE'][:, _idx]
_name_cu = f'MERT + CREPE-unique ({len(CREPE_UNIQUE)})'
if MERT_REG_REPR == 'full' and REG_BLOCK_WEIGHTING == 'equal':
    _kw = dict(reg_block_sizes=[int(ABL_BLOCKS_REG['MERT'].shape[1]), len(CREPE_UNIQUE)], block_weighting='equal')
elif MERT_REG_REPR == 'full':
    _kw = dict(reg_block_sizes=None, block_weighting='none')
else:
    _kw = dict(reg_block_sizes=None, block_weighting='n/a')
print(f'\n📊 new fit (tier {T75_TIER}, same engine and block weighting as §6.3):')
ablate(np.hstack([ABL_BLOCKS['MERT'], _Cu]), _name_cu, tier=T75_TIER,
       X_reg=np.hstack([ABL_BLOCKS_REG['MERT'], _Cu]), note='MERT + the CREPE features MERT does not encode', **_kw)

# ── Δ vs MERT, paired permutation, BH within §7.5 ──
_ROWS = [('MERT + CREPE', '1-blocks'), ('MERT + Essentia', '1-blocks'), ('MERT + DTW', '1-blocks'),
         (_name_cu, T75_TIER)]
_rows = []
for pr in PROTOCOLS:
    rng = np.random.default_rng(ABL_SEED + 75)
    ref = ABL_PRED[('1-blocks', 'MERT', pr)]
    f_ref = _sc_f1(ref['Yt'], ref['cls'])
    _rows.append(dict(protocol=pr, row='MERT (reference)', n_features=int(ABL_BLOCKS['MERT'].shape[1]),
                      f1_macro=f_ref, r2_val=_sc_r2(ref['yv'], ref['val'], ref['bv']),
                      r2_aro=_sc_r2(ref['ya'], ref['aro'], ref['ba'])))
    for nm, tier in _ROWS:
        cur = ABL_PRED[(tier, nm, pr)]
        assert np.array_equal(cur['Yt'], ref['Yt']) and np.allclose(cur['yv'], ref['yv']), '§7.5: unpaired targets'
        _nf = next(r['n_features'] for r in ABL_ROWS if r['tier'] == tier and r['set_name'] == nm and r['protocol'] == pr)
        _rows.append(dict(
            protocol=pr, row=nm, n_features=int(_nf),
            f1_macro=_sc_f1(cur['Yt'], cur['cls']), r2_val=_sc_r2(cur['yv'], cur['val'], cur['bv']),
            r2_aro=_sc_r2(cur['ya'], cur['aro'], cur['ba']),
            d_f1=_sc_f1(cur['Yt'], cur['cls']) - f_ref,
            p_f1=perm_p(cur['cls'], ref['cls'], lambda P: _sc_f1(ref['Yt'], P), rng),
            d_r2_val=_sc_r2(cur['yv'], cur['val'], cur['bv']) - _sc_r2(ref['yv'], ref['val'], ref['bv']),
            p_r2_val=perm_p(cur['val'], ref['val'], lambda P: _sc_r2(ref['yv'], P, ref['bv']), rng),
            d_r2_aro=_sc_r2(cur['ya'], cur['aro'], cur['ba']) - _sc_r2(ref['ya'], ref['aro'], ref['ba']),
            p_r2_aro=perm_p(cur['aro'], ref['aro'], lambda P: _sc_r2(ref['ya'], P, ref['ba']), rng),
            source='new fit (tier 7.5-mert-plus)' if tier == T75_TIER else 'read from the §6.3 lattice (no refit)'))
df_block = pd.DataFrame(_rows)
for c in ('q_f1', 'q_r2_val', 'q_r2_aro'):
    df_block[c] = np.nan
for pr, sub in df_block[df_block['row'] != 'MERT (reference)'].groupby('protocol'):
    _pc = ['p_f1', 'p_r2_val', 'p_r2_aro']
    _q = bh_fdr(np.concatenate([sub[c].to_numpy(dtype=float) for c in _pc]))
    for i, c in enumerate(_pc):
        df_block.loc[sub.index, 'q' + c[1:]] = _q[i * len(sub):(i + 1) * len(sub)]
BLOCK_F1 = {pr: dict(zip(df_block.query('protocol == @pr')['row'], df_block.query('protocol == @pr')['f1_macro']))
            for pr in PROTOCOLS}
block_f1 = BLOCK_F1[PRIMARY_PROTOCOL]     # back-compat alias — the PRIMARY protocol

for pr in PROTOCOLS:
    print(f'\n── {PROTOCOL_LABEL[pr]} — Δ vs MERT (classification: MERT-PCA25 + X · regression: MERT-full 768 + X, '
          'block-weighted) ──')
    print(df_block.query('protocol == @pr')[['row', 'n_features', 'f1_macro', 'd_f1', 'p_f1', 'q_f1', 'r2_val',
                                             'd_r2_val', 'p_r2_val', 'q_r2_val', 'r2_aro', 'd_r2_aro', 'p_r2_aro',
                                             'q_r2_aro']].round(4).to_string(index=False))
print('\n  q = BH-FDR within §7.5 (per protocol, every row × F1 / R²v / R²a) — the q value is the verdict; there is no')
print('  ±0.005 "adds / hurts" threshold any more. Pair with §7.3b: low probe R² + a positive Δ with q < .05 = a tool')
print('  that is genuinely complementary to MERT.')
df_block.to_csv(os.path.join(OUTPUT_DIR, 'featsel_crepe_block.csv'), index=False)

# ── Figure: Δ vs MERT per row, both protocols, three metrics ──
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=False)
_rl = [nm for nm, _ in _ROWS]
for ax, (m, lab) in zip(axes, (('d_f1', 'ΔF1-macro (MERT-PCA25 + X)'), ('d_r2_val', 'ΔR² valence (MERT-full 768 + X)'),
                               ('d_r2_aro', 'ΔR² arousal (MERT-full 768 + X)'))):
    for off, pr, col in [(-0.2, 'piece', '#e07b39'), (0.2, 'loo', '#2a6dd9')]:
        sub = df_block.query('protocol == @pr').set_index('row').reindex(_rl)
        bars = ax.bar(np.arange(len(_rl)) + off, sub[m], 0.38, color=col, edgecolor='#333', linewidth=0.5,
                      label=pr + (' (primary)' if pr == 'piece' else ' (optimistic)'))
        for b, v, q in zip(bars, sub[m], sub['q_' + m[2:]]):
            if np.isfinite(q) and q < 0.05:
                ax.text(b.get_x() + b.get_width() / 2, v, '*', ha='center', va='bottom' if v >= 0 else 'top',
                        fontsize=13, color='#b00020')
    ax.axhline(0, color='#333', lw=0.8)
    ax.set_xticks(np.arange(len(_rl))); ax.set_xticklabels(_rl, rotation=20, ha='right', fontsize=8)
    ax.set_title(lab, fontsize=10); ax.grid(axis='y', alpha=0.25)
axes[0].legend(fontsize=8)
fig.suptitle('§7.5  What each tool adds on top of MERT — Δ vs MERT · * = BH q < .05 within §7.5', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_featsel_crepe_block.png'), dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ §7.5 — featsel_crepe_block.csv, fig_featsel_crepe_block.png (tier 7.5-mert-plus joins ablation_master.csv)')


## Part 8 · Within-piece prediction — §8.1 Within-piece prediction (does technique CHANGE predict rating CHANGE?)
**Question of this part:** once the piece is removed altogether, does the *change* in the measured techniques  
predict the *change* in the ratings — beyond what the condition label alone explains?
**Hands to the next part:** `df_within_piece` (incremental ΔR², Freedman–Lane q) and `WP_CEILING` — Part 9 then
asks *which* technique.

The analysis closest to the thesis question. Everything above predicts *absolute* ratings, and 95% /
91% of their variance is between pieces — so a model can score well by recognising the piece. Here
the piece is **removed**: targets are `valence_mean` / `arousal_mean` **minus their piece mean**
over the three renditions, and every feature is **piece-centred** the same way. What is left is the
within-piece variation the MEC → EXP → EXG manipulation creates.

| Feature set | Content |
|---|---|
| **Condition-only** (reference) | one-hot MEC/EXP/EXG, centred — what the condition label alone explains |
| Interpretable (27) | `INTERP_PRED_COLS` (§4.1): CREPE + Essentia-fwd + DTW **reference-free** timing (never the DTW-deviation family, which encodes the condition by construction) |
| MERT-PCA25 | `X_A` |
| MERT-full 768 | `X_MERT_FULL` |
| Interpretable + MERT-PCA25 | both |

**Model:** `nested_ridge_predict`, **leave-one-piece-out only** — LOOCV is invalid here, because
centring couples the three renditions of a piece (a held-out excerpt's centred value is determined by
its two siblings in training). **Metrics:** r, out-of-sample R² against the centred mean (baseline
= 0), RMSE. **Significance:** a *within-piece* permutation — the targets are shuffled across the
three renditions **inside each piece** (never across pieces) and the whole nested model is refitted,
`WP_N_PERM` times; one-sided p for R² and r. The refits use `NestedRidgeOperator`, the closed-form
twin of `nested_ridge_predict`, after asserting it reproduces the sklearn path on the observed targets.

> **Reading key.** A feature set is only interesting if it **beats Condition-only**. Otherwise it is
> just re-encoding the manipulation — telling MEC from EXG — rather than tracking *how much* the
> audience's rating moved within a piece. Classification is not done within-piece.

**B-INC — does a feature set add beyond the condition label?** For every set and target the full
model *Condition + Features* (centred one-hot ⊕ centred features) is compared with *Condition-only*,
both nested ridge under `piece`: ΔR² = R²(Condition + Features) − R²(Condition-only). Null =
**Freedman–Lane**: the reduced model (OLS on the centred one-hot, all rows) is fitted, its residuals
are permuted **within piece** (the same `WP_PERMS`), added back to its fitted values, and both
models are refitted with the closed-form operators (equivalence asserted on the observed target).
One-sided p, BH over the 8 (set × target) tests → `r2_cond_plus`, `d_r2_incremental`,
`p_incremental`, `q_incremental`.

**B-NC — explainable-variance ceiling.** `NC_N_SPLITS` = 200 seeded random splits of the **raters**
into halves; per half the excerpt means of `valence_norm` / `arousal_norm`, centred within piece;
Pearson r between halves → Spearman–Brown 2r/(1+r). Mean and 2.5–97.5 % per target, printed next to
the R² values with `R² / ceiling`; the uncentred (between-excerpt) ceiling is given for context.
Splits that leave an excerpt without a rating in one half are redrawn and counted.

In [ ]:
# ── §8.1 Within-piece prediction (technique CHANGE → rating CHANGE) ──
# NEEDS : bh_fdr, df_expr, df_interp, df_long, DTW_FORWARD_DEV, eids, freedman_lane_targets,
#          INTERP_PRED_COLS (§4.1), NC_N_SPLITS, nested_ridge_predict,
#          NestedRidgeOperator, oos_r2, OUTPUT_DIR, PIECE_GROUPS, rmse, SEED, split_half_ceiling,
#          within_piece_permutation, WP_N_PERM, X_A, X_MERT_FULL, _abl_mat
# MAKES : df_within_piece, df_wp_ceiling, WP_CEILING, WP_PERMS, WP_PROTOCOL, WP_SETS,
#          WP_TARGETS, _piece_center · within_piece.csv, within_piece_noise_ceiling.csv
# FEEDS : the report's within-piece claim (does technique change predict rating change?); §11.1 (claim 8)
# ═══════════════════════════════════════════════════════════════════════════════
# §8.1 · Within-piece prediction — the piece is removed, only the manipulation is left
# ═══════════════════════════════════════════════════════════════════════════════
WP_PROTOCOL = 'piece'
assert WP_PROTOCOL == 'piece', \
    '§8.1: LOOCV is invalid within-piece — centring couples the three renditions of a piece'
assert list(df_expr['excerpt_id']) == list(eids), '§8.1: df_expr must be row-aligned to eids'
_wp_conds = df_expr.groupby('piece')['condition'].apply(lambda s: tuple(sorted(s)))
assert all(c == ('EXG', 'EXP', 'MEC') for c in _wp_conds), \
    f'§8.1: every piece needs exactly one MEC / EXP / EXG rendition — {dict(_wp_conds)}'


def _piece_center(M, groups=PIECE_GROUPS):
    """Subtract each piece's mean over its three renditions (column-wise)."""
    M = np.asarray(M, dtype=float)
    out = M.copy()
    for g in np.unique(groups):
        i = groups == g
        out[i] = M[i] - M[i].mean(axis=0)
    return out


# ── Feature sets (all piece-centred) ────────────────────────────────────────
_wp_cond = np.column_stack([(df_expr['condition'].to_numpy() == c).astype(float)
                            for c in ('MEC', 'EXP', 'EXG')])
# Phase C2: the interpretable set is the PREDICTIVE layer of §4.1 (INTERP_PRED_COLS = CREPE + Essentia-forward +
# DTW reference-free timing — never the DTW-deviation family, which encodes the condition by construction).
assert not set(INTERP_PRED_COLS) & set(DTW_FORWARD_DEV), '§8.1: a DTW-deviation column is in INTERP_PRED_COLS'
_wp_interp = _abl_mat(df_interp[INTERP_PRED_COLS])
_WP_REF = 'Condition-only (one-hot MEC/EXP/EXG)'
WP_SETS = {
    _WP_REF                                                      : _wp_cond,
    f'Interpretable ({_wp_interp.shape[1]}: CREPE+Ess+DTW ref-free)': _wp_interp,
    f'MERT-PCA{X_A.shape[1]} (X_A)'                                : np.asarray(X_A, dtype=float),
    f'MERT-full {X_MERT_FULL.shape[1]}'                             : np.asarray(X_MERT_FULL, dtype=float),
    f'Interpretable + MERT-PCA ({_wp_interp.shape[1] + X_A.shape[1]}d)':
        np.hstack([_wp_interp, np.asarray(X_A, dtype=float)]),
}
WP_SETS = {k: _piece_center(v) for k, v in WP_SETS.items()}
WP_TARGETS = {t: _piece_center(df_expr[f'{t}_mean'].to_numpy(dtype=float)[:, None]).ravel()
              for t in ('valence', 'arousal')}
for _t, _yc in WP_TARGETS.items():
    _sums = pd.Series(_yc).groupby(PIECE_GROUPS).sum().abs().max()
    assert _sums < 1e-9, f'§8.1: centred {_t} does not sum to 0 within a piece'
    print(f'§8.1 · centred {_t}: within-piece SD = {_yc.std():.3f} (scale 0–6)')

# The SAME within-piece permutations for every set and target (comparable p-values).
_wp_rng  = np.random.default_rng(SEED + 409)
WP_PERMS = [within_piece_permutation(PIECE_GROUPS, _wp_rng) for _ in range(WP_N_PERM)]


def _wp_r(y, yh):
    return float(pearsonr(y, yh)[0]) if np.std(yh) > 1e-12 and np.std(y) > 1e-12 else float('nan')


def _wp_p(null, obs):
    """One-sided permutation p (H1: better than chance), (1 + #{null ≥ obs}) / (1 + N)."""
    if not np.isfinite(obs):
        return float('nan')
    null = np.nan_to_num(np.asarray(null, dtype=float), nan=-np.inf)
    return float((1 + np.sum(null >= obs - 1e-12)) / (1 + len(null)))


_wp_rows = []
print(f'\n🔄 §8.1 · nested ridge ({WP_PROTOCOL}) + {WP_N_PERM} within-piece permutations per set × target')
for _sn, _Xc in WP_SETS.items():
    _op = NestedRidgeOperator(_Xc, WP_PROTOCOL)
    for _t, _yc in WP_TARGETS.items():
        _yh, _al = nested_ridge_predict(_Xc, _yc, WP_PROTOCOL)
        _yh2, _al2 = _op(_yc)
        assert np.array_equal(_al, _al2) and np.allclose(_yh, _yh2, rtol=0, atol=1e-8), \
            (f'§8.1: closed-form NestedRidgeOperator disagrees with nested_ridge_predict on '
             f'{_sn}/{_t} (max |Δŷ| = {np.max(np.abs(_yh - _yh2)):.2e}) — permutation null invalid')
        _r, _r2, _e = _wp_r(_yc, _yh), oos_r2(_yc, _yh, baseline=0.0), rmse(_yc, _yh)
        # training rows are whole pieces → their centred mean is 0, so the OOS baseline is 0 too
        assert np.isclose(_r2, oos_r2(_yc, _yh, WP_PROTOCOL), atol=1e-8), \
            '§8.1: baseline-0 R² differs from the training-fold-mean R²'
        _nr, _nr2 = [], []
        for _perm in WP_PERMS:
            _yp = _yc[_perm]
            _yhp, _ = _op(_yp)
            _nr.append(_wp_r(_yp, _yhp))
            _nr2.append(oos_r2(_yp, _yhp, baseline=0.0))
        _wp_rows.append(dict(protocol=WP_PROTOCOL, feature_set=_sn, n_features=int(_Xc.shape[1]),
                             target=_t, r=_r, r2=_r2, rmse=_e,
                             p_r=_wp_p(_nr, _r), p_r2=_wp_p(_nr2, _r2),
                             null_r2_p95=float(np.nanpercentile(_nr2, 95)),
                             alpha_median=float(np.median(_al)), n_perm=len(WP_PERMS)))
print('✅ closed-form permutation refits match nested_ridge_predict on every observed target '
      '(identical α per fold, |Δŷ| ≤ 1e-8)')

# ── B-INC · do the features add BEYOND the condition label? (Freedman–Lane) ──
# Full model = centred condition one-hot ⊕ centred features; reduced = Condition-only. Both
# are nested ridge (piece). Null: the reduced model is fitted by OLS on the centred one-hot
# (all rows), its residuals are permuted WITHIN PIECE (the same WP_PERMS) and added back to
# its fitted values; both models are refitted with the closed-form operators (equivalence
# asserted on the observed target first) and ΔR² is recomputed. One-sided p, BH over the
# (set × target) incremental tests of this cell.
_wp_Z   = WP_SETS[_WP_REF]                                   # centred one-hot (rank 2)
_op_ref = NestedRidgeOperator(_wp_Z, WP_PROTOCOL)
_inc    = {}
print(f'\n🔄 §8.1 B-INC · Condition + Features vs Condition-only — Freedman–Lane, '
      f'{len(WP_PERMS)} within-piece permutations')
for _sn, _Xc in WP_SETS.items():
    if _sn == _WP_REF:
        continue
    _Xcp = np.hstack([_wp_Z, _Xc])
    _opc = NestedRidgeOperator(_Xcp, WP_PROTOCOL)
    for _t, _yc in WP_TARGETS.items():
        _yh, _al = nested_ridge_predict(_Xcp, _yc, WP_PROTOCOL)
        _yh2, _al2 = _opc(_yc)
        assert np.array_equal(_al, _al2) and np.allclose(_yh, _yh2, rtol=0, atol=1e-8), \
            (f'§8.1 B-INC: closed-form operator disagrees with nested_ridge_predict on '
             f'Condition + {_sn}/{_t} (max |Δŷ| = {np.max(np.abs(_yh - _yh2)):.2e})')
        _r2_cp  = oos_r2(_yc, _yh, baseline=0.0)
        _r2_ref = oos_r2(_yc, _op_ref(_yc)[0], baseline=0.0)
        _obs    = _r2_cp - _r2_ref
        _null   = []
        for _ys in freedman_lane_targets(_yc, _wp_Z, WP_PERMS):
            _null.append(oos_r2(_ys, _opc(_ys)[0], baseline=0.0)
                         - oos_r2(_ys, _op_ref(_ys)[0], baseline=0.0))
        _inc[(_sn, _t)] = dict(r2_cond_plus=_r2_cp, d_r2_incremental=_obs,
                               p_incremental=_wp_p(_null, _obs),
                               null_d_r2_p95=float(np.nanpercentile(_null, 95)))
print('✅ B-INC closed-form refits match nested_ridge_predict on every observed Condition + '
      'Features target')
_inc_keys = list(_inc)
_inc_q    = bh_fdr([_inc[k]['p_incremental'] for k in _inc_keys])
for _k, _q in zip(_inc_keys, _inc_q):
    _inc[_k]['q_incremental'] = float(_q)
print(f'   BH-FDR across the {len(_inc_keys)} incremental tests (sets × targets) of this cell')

# ── B-NC · noise ceiling: split-half reliability over RATERS (Spearman–Brown) ──
_nc_long = df_long[['participant_id', 'excerpt_id', 'valence_norm', 'arousal_norm']]
assert set(_nc_long['excerpt_id']) == set(eids), \
    '§8.1 B-NC: df_long excerpts differ from the 60 modelled excerpts'
WP_CEILING = {}
for _t in WP_TARGETS:
    _full = _nc_long.groupby('excerpt_id')[f'{_t}_norm'].mean().reindex(eids).to_numpy(dtype=float)
    assert np.allclose(_full, df_expr[f'{_t}_mean'].to_numpy(dtype=float), atol=1e-9), \
        f'§8.1 B-NC: df_long {_t}_norm means do not reproduce df_expr {_t}_mean'
    for _kind, _grp in (('within-piece (centred)', PIECE_GROUPS), ('between-excerpt (uncentred)', None)):
        _o = split_half_ceiling(_nc_long['participant_id'].to_numpy(), _nc_long['excerpt_id'].to_numpy(),
                                _nc_long[f'{_t}_norm'].to_numpy(dtype=float), eids, NC_N_SPLITS,
                                seed=SEED + 410, item_groups=_grp)
        WP_CEILING[(_t, _kind)] = dict(target=_t, kind=_kind, n_splits=NC_N_SPLITS,
                                       r_half_mean=float(_o['r_half'].mean()),
                                       ceiling_mean=float(_o['r_full'].mean()),
                                       ceiling_lo=float(np.percentile(_o['r_full'], 2.5)),
                                       ceiling_hi=float(np.percentile(_o['r_full'], 97.5)),
                                       n_redraws=_o['n_redraws'])
df_wp_ceiling = pd.DataFrame(list(WP_CEILING.values()))
print(f'\n📏 §8.1 B-NC · explainable-variance ceiling — {NC_N_SPLITS} random rater split-halves '
      f'(seeded), Spearman–Brown 2r/(1+r)')
print(df_wp_ceiling.round(4).to_string(index=False))
_n_rd = int(df_wp_ceiling['n_redraws'].sum())
print(f'   redrawn splits (an excerpt without a rating in one half): {_n_rd}')

df_within_piece = pd.DataFrame(_wp_rows)
_ref_r2 = df_within_piece[df_within_piece.feature_set == _WP_REF].set_index('target')['r2']
df_within_piece['d_r2_vs_condition'] = df_within_piece['r2'] - df_within_piece['target'].map(_ref_r2)
df_within_piece['beats_condition'] = np.where(df_within_piece.feature_set == _WP_REF, np.nan,
                                              df_within_piece['d_r2_vs_condition'] > 0)
for _c in ('r2_cond_plus', 'd_r2_incremental', 'p_incremental', 'q_incremental', 'null_d_r2_p95'):
    df_within_piece[_c] = [_inc.get((_s, _t), {}).get(_c, np.nan)
                           for _s, _t in zip(df_within_piece['feature_set'], df_within_piece['target'])]
_ceil = {_t: WP_CEILING[(_t, 'within-piece (centred)')]['ceiling_mean'] for _t in WP_TARGETS}
df_within_piece['nc_ceiling']      = df_within_piece['target'].map(_ceil)
df_within_piece['r2_over_ceiling'] = df_within_piece['r2'] / df_within_piece['nc_ceiling']
df_within_piece['r2_cond_plus_over_ceiling'] = (df_within_piece['r2_cond_plus']
                                                / df_within_piece['nc_ceiling'])

print('\n📊 §8.1 WITHIN-PIECE PREDICTION — piece-centred targets & features, leave-one-piece-out')
print('   R² baseline = 0 (the centred mean) · p = one-sided within-piece permutation')
for _t in WP_TARGETS:
    print(f'\n── Δ{_t} within piece ──')
    _c = WP_CEILING[(_t, 'within-piece (centred)')]
    _u = WP_CEILING[(_t, 'between-excerpt (uncentred)')]
    print(f'   explainable-variance ceiling (within-piece): {_c["ceiling_mean"]:.3f} '
          f'[{_c["ceiling_lo"]:.3f}, {_c["ceiling_hi"]:.3f}]  ·  between-excerpt (context): '
          f'{_u["ceiling_mean"]:.3f} [{_u["ceiling_lo"]:.3f}, {_u["ceiling_hi"]:.3f}]')
    print(df_within_piece.query('target == @_t')[
        ['feature_set', 'n_features', 'r', 'p_r', 'r2', 'p_r2', 'null_r2_p95', 'rmse',
         'd_r2_vs_condition', 'r2_cond_plus', 'd_r2_incremental', 'p_incremental',
         'q_incremental', 'r2_over_ceiling']].round(4).to_string(index=False))
print('\n  Reading key: the Condition-only row (top) is what the MEC/EXP/EXG label alone explains.')
print('  A feature set is only interesting if it adds BEYOND Condition-only. d_r2_vs_condition')
print('  compares two separate models (descriptive); the TEST is B-INC: d_r2_incremental =')
print('  R²(Condition + Features) − R²(Condition-only), Freedman–Lane within-piece p, BH q.')
print('  r2_over_ceiling = R² / the split-half explainable-variance ceiling of the centred target.')

df_within_piece.to_csv(os.path.join(OUTPUT_DIR, 'within_piece.csv'), index=False)
df_wp_ceiling.to_csv(os.path.join(OUTPUT_DIR, 'within_piece_noise_ceiling.csv'), index=False)
print('\n✅ §8.1 — within_piece.csv, within_piece_noise_ceiling.csv')


## Part 9 · Which techniques — §9.1 Condition-delta · technique change vs rating change
**Question of this part:** *which technique* moves *which* rating or class — the thesis question, answered along  
three paths (condition deltas §9.1, the signed linear read-out §9.2, the nonlinear SHAP cross-check §9.3).
**Hands to the next part:** `df_delta_tests`, `df_readout_reg` / `df_readout_cls` + `INTERP_CV_R` / `INTERP_CV_F1`
(the interpretable baseline of §10.1), `df_shap_importance`.

For each of the 20 pieces compute the **change in every interpretable feature** — the **descriptive**
layer of §4.1 (`INTERP_DESC_COLS`: CREPE pitch, the forwarded Essentia dynamics / timbre / tonal block, the
reference-free timing and the DTW-deviation features) — between MEC→EXP, EXP→EXG and MEC→EXG, then correlate
with ΔValence / ΔArousal from listener ratings. One BH-FDR family over every test; stars from q. **⚑** marks the
DTW-deviation rows (0/1 at MEC by construction) and the tempogram features that failed the manual audit.
Fig TE1 has one panel per transition (MEC → EXG added in Phase C2) and prints the raw p < .05 count, the
count expected by chance and the q < .05 count.

*Directly answers: "When the performer added more vibrato — or more dynamic range —
did listeners rate it as more arousing?"*


In [ ]:
# ── §9.1 Condition-delta · technique vs rating change ───────────────
# NEEDS : bh_fdr, df_all, df_interp, feat_keys, feat_short, INTERP_FLAG_REASON (§4.1 — the DESCRIPTIVE layer,
#          INTERP_DESC_COLS), OUTPUT_DIR
# MAKES : corr_matrix, df_delta, df_delta_tests, p_EX, p_ME, p_MG, q_EX, q_ME, q_MG, r_EX, r_ME, r_MG, sig
#          · analysis_condition_deltas.csv, analysis_condition_delta_tests.csv
# FEEDS : Fig TE1, §11.1 (claim 9)
# ═══════════════════════════════════════════════════════════════════════════════
# Path B — Condition-Delta Analysis (DESCRIPTIVE — the interpretable layer was built in §4.1)
# ═══════════════════════════════════════════════════════════════════════════════
# Reads the DESCRIPTIVE layer (feat_keys = INTERP_DESC_COLS): the DTW-deviation rows are kept and flagged ⚑
# (their MEC→EXP and MEC→EXG deltas are the raw deviation itself, 0/1 at MEC by construction), as are the
# tempogram features that failed the manual audit. One BH-FDR family over every test of this cell.
_dev_in = [c for c in feat_keys if INTERP_FLAG_REASON.get(c, '').startswith('DTW-deviation')]
if _dev_in:
    print(f'⚑ {len(_dev_in)} DTW-DEVIATION features {_dev_in} — 0/1 at MEC by construction: their MEC→EXP and '
          'MEC→EXG deltas are the\n  raw deviation itself, not a difference of two independent measurements.')
_flag = {c: (' ⚑' if c in INTERP_FLAG_REASON else '') for c in feat_keys}

# ── Build piece-level delta dataframe ─────────────────────────────────────────
piece_deltas = []
for piece in df_all['piece'].unique():
    sub  = df_all[df_all['piece'] == piece]
    ids  = {r['condition']: r['excerpt_id'] for _, r in sub.iterrows()}
    if not all(c in ids for c in ['MEC','EXP','EXG']): continue
    feats = {c: (df_interp.loc[ids[c]] if ids[c] in df_interp.index else None)
             for c in ['MEC','EXP','EXG']}
    if any(v is None for v in feats.values()): continue
    va = {c: {'val': sub[sub['condition']==c].iloc[0]['valence_mean'],
               'aro': sub[sub['condition']==c].iloc[0]['arousal_mean']}
          for c in ['MEC','EXP','EXG']}
    rec = {'piece': piece}
    for fk in feat_keys:
        rec[f'dME_{fk}'] = float(feats['EXP'][fk]) - float(feats['MEC'][fk])
        rec[f'dEX_{fk}'] = float(feats['EXG'][fk]) - float(feats['EXP'][fk])
        rec[f'dMG_{fk}'] = float(feats['EXG'][fk]) - float(feats['MEC'][fk])
    rec.update({
        'dME_val': va['EXP']['val'] - va['MEC']['val'],
        'dME_aro': va['EXP']['aro'] - va['MEC']['aro'],
        'dEX_val': va['EXG']['val'] - va['EXP']['val'],
        'dEX_aro': va['EXG']['aro'] - va['EXP']['aro'],
        'dMG_val': va['EXG']['val'] - va['MEC']['val'],
        'dMG_aro': va['EXG']['aro'] - va['MEC']['aro'],
    })
    piece_deltas.append(rec)

df_delta = pd.DataFrame(piece_deltas)
print(f'Piece-level delta matrix: {df_delta.shape}  ({len(df_delta)} pieces)')

# ── Correlation matrices ───────────────────────────────────────────────────────
def corr_matrix(df, feat_prefix, target_cols, fkeys):
    r_mat = np.zeros((len(fkeys), len(target_cols)))
    p_mat = np.ones((len(fkeys),  len(target_cols)))
    for i, fk in enumerate(fkeys):
        x = df[f'{feat_prefix}{fk}'].values
        for j, tc in enumerate(target_cols):
            y     = df[tc].values
            valid = ~(np.isnan(x) | np.isnan(y))
            if valid.sum() >= 4:
                r, p = pearsonr(x[valid], y[valid])
                r_mat[i, j] = r; p_mat[i, j] = p
    return r_mat, p_mat

r_ME, p_ME = corr_matrix(df_delta, 'dME_', ['dME_val','dME_aro'], feat_keys)
r_EX, p_EX = corr_matrix(df_delta, 'dEX_', ['dEX_val','dEX_aro'], feat_keys)
r_MG, p_MG = corr_matrix(df_delta, 'dMG_', ['dMG_val','dMG_aro'], feat_keys)

def sig(p): return '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else 'n.s.'

# ── Benjamini–Hochberg over ONE family = every test printed in this cell (A14) ──
# features × {ΔValence, ΔArousal} × 3 transitions. Significance stars come from q, not p.
_TRANS = [('MEC→EXP', r_ME, p_ME, 'MEC → EXP  (Expressive minus Mechanical)'),
          ('EXP→EXG', r_EX, p_EX, 'EXP → EXG  (Exaggerated minus Expressive)'),
          ('MEC→EXG', r_MG, p_MG, 'MEC → EXG  (Exaggerated minus Mechanical)')]
_tests = [dict(transition=_tn, feature=fk, label=feat_short[i], target=tgt,
               r=float(_rm[i, j]), p=float(_pm[i, j]))
          for _tn, _rm, _pm, _ in _TRANS for i, fk in enumerate(feat_keys)
          for j, tgt in enumerate(['ΔValence', 'ΔArousal'])]
df_delta_tests = pd.DataFrame(_tests)
df_delta_tests['q'] = bh_fdr(df_delta_tests['p'].to_numpy(dtype=float))
df_delta_tests['flag'] = [_flag[f].strip() for f in df_delta_tests['feature']]
df_delta_tests['stars_q'] = [sig(q) if np.isfinite(q) else 'n/a' for q in df_delta_tests['q']]
_qmat = {_tn: df_delta_tests[df_delta_tests.transition == _tn]['q'].to_numpy().reshape(len(feat_keys), 2)
         for _tn, *_ in _TRANS}
q_ME, q_EX, q_MG = _qmat['MEC→EXP'], _qmat['EXP→EXG'], _qmat['MEC→EXG']

for (_tn, r_mat, p_mat, title), q_mat in zip(_TRANS, (q_ME, q_EX, q_MG)):
    print(f'\n{title}')
    print(f'  {"Technique":<28} {"ΔValence r (p, q)":<32} {"ΔArousal r (p, q)"}')
    print('  ' + '-'*88)
    for i, fs in enumerate(feat_short):
        rv, pv, _qv = r_mat[i,0], p_mat[i,0], q_mat[i,0]
        ra, pa, _qa = r_mat[i,1], p_mat[i,1], q_mat[i,1]
        print(f'  {fs + _flag[feat_keys[i]]:<28} r={rv:+.3f} p={pv:.3f} q={_qv:.3f} {sig(_qv):<6} '
              f'r={ra:+.3f} p={pa:.3f} q={_qa:.3f} {sig(_qa)}')

_n_tests = int(np.isfinite(df_delta_tests['p']).sum())
_n_raw   = int((df_delta_tests['p'] < .05).sum())
_n_q     = int((df_delta_tests['q'] < .05).sum())
print(f'\n── Multiple testing (one BH-FDR family = all {_n_tests} tests above: '
      f'{len(feat_keys)} features × 2 targets × {len(_TRANS)} transitions) ──')
print(f'   raw p < .05 : {_n_raw}   vs  {_n_tests * 0.05:.1f} expected under the global null '
      f'({_n_tests} × 0.05)')
print(f'   q < .05     : {_n_q}   ← the stars above (and in Fig TE1) are q-based · ⚑ = flagged '
      '(DTW-deviation / tempogram)')

df_delta.to_csv(os.path.join(OUTPUT_DIR,'analysis_condition_deltas.csv'), index=False)
df_delta_tests.to_csv(os.path.join(OUTPUT_DIR,'analysis_condition_delta_tests.csv'), index=False)
print(f'\n✅ Saved: analysis_condition_deltas.csv (per-piece deltas, unchanged), '
      f'analysis_condition_delta_tests.csv (r, p, q per test)')


In [ ]:
# ── Fig TE1: Condition-Delta Correlation Heatmap — three transitions ──────────
# NEEDS : df_delta, df_delta_tests, feat_keys, feat_short, INTERP_FLAG_REASON, OUTPUT_DIR, q_EX, q_ME, q_MG,
#          r_EX, r_ME, r_MG (§9.1) · MAKES : fig_delta_correlation.png · FEEDS : —
# Phase C2 (Patch 1 / TE1): the MEC → EXG panel is added; ⚑ marks the flagged rows (DTW-deviation — 0/1 at MEC
# by construction — and the tempogram features); BH over ALL §9.1 tests (one family); the subtitle prints the raw
# p < .05 count, the count expected by chance and the q < .05 count.
_n_tests = int(np.isfinite(df_delta_tests['p']).sum())
_n_raw   = int((df_delta_tests['p'] < .05).sum())
_n_q     = int((df_delta_tests['q'] < .05).sum())
_ylab    = [f'{s} ⚑' if k in INTERP_FLAG_REASON else s for k, s in zip(feat_keys, feat_short)]
fig, axes = plt.subplots(1, 3, figsize=(19, max(7, 0.42 * len(feat_short))))
fig.suptitle(
    'Path B — Condition-Delta: Technique Changes vs Listener Ratings\n'
    f'N={len(df_delta)} pieces · BH-FDR over all {_n_tests} §9.1 tests · raw p < .05: {_n_raw} '
    f'(expected by chance {_n_tests * 0.05:.1f}) · q < .05: {_n_q} · * q<.05 ** q<.01 *** q<.001 · '
    '⚑ = DTW-deviation (0/1 at MEC) or tempogram (failed the audit)',
    fontsize=12
)
for ax, (r_mat, q_mat, title) in zip(axes, [
    (r_ME, q_ME, 'MEC → EXP\n(Expressive minus Mechanical)'),
    (r_EX, q_EX, 'EXP → EXG\n(Exaggerated minus Expressive)'),
    (r_MG, q_MG, 'MEC → EXG\n(Exaggerated minus Mechanical)'),
]):
    im = ax.imshow(r_mat, vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['ΔValence', 'ΔArousal'], fontsize=11)
    ax.set_yticks(range(len(_ylab))); ax.set_yticklabels(_ylab, fontsize=8)
    ax.set_title(title, fontsize=11, pad=10)
    for i in range(r_mat.shape[0]):
        for jj in range(r_mat.shape[1]):
            q = q_mat[i, jj]; rv = r_mat[i, jj]
            s = '***' if q < .001 else '**' if q < .01 else '*' if q < .05 else ''
            ax.text(jj, i, f'{rv:+.2f}{s}', ha='center', va='center', fontsize=8.5,
                    fontweight='bold' if s else 'normal',
                    color='white' if abs(rv) > 0.6 else 'black')
    plt.colorbar(im, ax=ax, label='Pearson r', shrink=0.8)
plt.tight_layout()
fpath = os.path.join(OUTPUT_DIR, 'fig_delta_correlation.png')
plt.savefig(fpath, dpi=150, bbox_inches='tight'); plt.show()
print(f'✅ Fig TE1 saved → {fpath}')


In [ ]:
# ── Fig 7: Low-level features by condition ────────────────────────────────────
# NEEDS : CREPE_CANDIDATES, CREPE_LABELS, df_all, df_crepe, OUTPUT_DIR · MAKES : fig_lowlevel_conditions.png
df_ll = df_crepe[CREPE_CANDIDATES].copy()
df_ll['condition']=df_all.set_index('excerpt_id').reindex(df_ll.index)['condition']
_units = {'crepe_vibrato_rate_hz': ' (Hz)', 'crepe_vibrato_extent_cents': ' (± cents)',
          'crepe_portamento_rate': ' (glides / voiced s)', 'crepe_portamento_mean_ext_cents': ' (cents)',
          'crepe_f0_range_cents': ' (cents, P95−P5)', 'crepe_f0_mean_hz': ' (Hz)'}
feats_plot=[(c, CREPE_LABELS[c] + _units.get(c, '')) for c in CREPE_CANDIDATES]
fig,axes=plt.subplots(2,5,figsize=(20,8))
fig.suptitle('CREPE Pitch Features by Condition (Validation Mode)',fontsize=13)
bpal={'MEC':'#aaa','EXP':'#2a6dd9','EXG':'#2a9d2a'}
for ax,(fc,fl) in zip(axes.flatten(),feats_plot):
    if fc not in df_ll.columns: ax.set_visible(False); continue
    data=[df_ll.loc[df_ll['condition']==c,fc].dropna().values for c in ['MEC','EXP','EXG']]
    bp=ax.boxplot(data,positions=[0,1,2],patch_artist=True,widths=0.5,
                  medianprops=dict(color='white',lw=2))
    for patch,col in zip(bp['boxes'],[bpal['MEC'],bpal['EXP'],bpal['EXG']]):
        patch.set_facecolor(col); patch.set_alpha(0.75)
    ax.set_xticks([0,1,2]); ax.set_xticklabels(['MEC','EXP','EXG'])
    ax.set_title(fl,fontsize=9); ax.grid(axis='y',alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'fig_lowlevel_conditions.png'),dpi=150,bbox_inches='tight'); plt.show()

## Part 9 · Which techniques — §9.2 Interpretable read-out: which technique pushes which emotion, and how much

A **transparent linear model** over the **predictive** interpretable layer of §4.1 (`INTERP_PRED_COLS`, 27
features: CREPE + Essentia + reference-free timing — no DTW-deviation) — our proxy for a **mid-level
perceptual layer** — reported as **signed standardized coefficients**. **⚑** marks the tempogram features
that failed the manual audit; when one ranks first, the printout also names the top non-flagged feature.

This is the mid-level route familiar from *Aljanaki & Soleymani (2018)* (mid-level perceptual
features as an interpretable bottleneck for music emotion) and *Chowdhury et al. (2019)*
(mid-level features as the explanatory layer between audio and emotion). We do not use their
annotated mid-level dataset; we substitute our own **domain-specific measured** violin
features (vibrato, portamento, F0 statistics, loudness, timbre, tonal) as that layer.

**Pairs with §9.3.** SHAP gives *magnitude* — how much a technique moves the needle.
This gives *signed direction* and a readable equation: `arousal ≈ intercept + β₁·technique₁ + …`

> **On the unsupervised invariant.** The §2.1b/§2.2b/§2.3b forwarding gates are unsupervised
> because they *select features that feed other models*. §9.2 is different: it is a standalone
> **supervised interpretive** model whose output is a coefficient table, not a feature set
> passed downstream. It therefore uses the labels — correctly and by design. Honesty is kept
> by reporting its **predictive** score under both CV protocols (leave-one-piece-out primary,
> LOOCV secondary; nested piece-grouped ridge), separately from the all-data coefficients.

**Hands to §10.1:** its cross-validated scores (`INTERP_CV_R[protocol]`,
`INTERP_CV_F1[protocol]`) are the interpretable-only baseline for the *Cost of Explainability*.

In [ ]:
# ── §9.2 Interpretable linear read-out ──────────────────────────────
# NEEDS : cv_predict_multilabel, df_expr, df_interp, eids, flag_label, INTERP_FLAG_REASON, INTERP_PRED_COLS,
#          top_feature_note (§4.1), mlb, nested_ridge_predict, oos_r2, OUTPUT_DIR, PIECE_GROUPS, PROTOCOLS,
#          PROTOCOL_LABEL, rmse, select_ridge_alpha, warn_negative_loo_r, Y
# MAKES : df_readout_reg (betas × {valence,arousal}), df_readout_cls (betas × 5 classes),
#          INTERP_CV_R (protocol → dim → {r, r2, rmse, alpha_median}), INTERP_CV_F1 (protocol → float),
#          interp_cols, READOUT_ORDER_CLS (class → columns by |β|), READOUT_ORDER_REG (dim → columns by |β|),
#          interp_readout_coefficients.csv, interp_readout_cv.csv, 2 figures
# FEEDS : §9.3 (SHAP vs β agreement), §10.1 Cost of Explainability (reuses INTERP_CV_R / INTERP_CV_F1)
# ═══════════════════════════════════════════════════════════════════════════════
# §9.2 · Transparent linear read-out — interpretable techniques → emotion
# ═══════════════════════════════════════════════════════════════════════════════
# SUPERVISED BY DESIGN. Unlike the §2.x forwarding gates (which select features that
# feed other models and must therefore stay label-free), this cell IS the model: its
# product is a signed coefficient table, nothing here is passed to a downstream
# learner. Two separate fits, never mixed:
#   (a) coefficients  — fit on all 60 excerpts, descriptive/interpretive only; α chosen by
#       the SAME piece-grouped rule as the engine (GroupKFold(5) over the 60 rows' pieces)
#   (b) r / R² / F1   — cross-validated under BOTH protocols (§5.1 engine), the only numbers
#       that may be read as predictive

# ── 1. Interpretable block — the PREDICTIVE layer of §4.1 (INTERP_PRED_COLS), NAMED features only ──
# Phase C2: no DTW-deviation feature (descriptive only); ⚑ marks the tempogram features in every label.
interp_cols = list(INTERP_PRED_COLS)
interp_lbls = [flag_label(c) for c in interp_cols]
print(f'ℹ️  Interpretable read-out on INTERP_PRED_COLS ({len(interp_cols)} features, §4.1) · ⚑ = '
      + ', '.join(f'{flag_label(c)}' for c in interp_cols if c in INTERP_FLAG_REASON))

# CHECK — the interpretable layer must contain no embedding dimension
_leaked = [c for c in interp_cols if 'mert' in str(c).lower()]
assert not _leaked, f'§9.2 must stay interpretable — MERT columns present: {_leaked}'

Xi = df_interp.loc[eids, interp_cols].apply(pd.to_numeric, errors='coerce')
if int(Xi.isna().sum().sum()):
    print(f'⚠️  {int(Xi.isna().sum().sum())} NaN cells in the interpretable block → column-mean imputed')
    Xi = Xi.fillna(Xi.mean(numeric_only=True))
X_interp = Xi.values.astype(float)

# CHECK — rows aligned to eids and to Y, same order, length 60
assert list(Xi.index) == list(eids),        '§9.2: df_interp rows are not in eids order'
assert X_interp.shape[0] == Y.shape[0],     '§9.2: X_interp / Y row-count mismatch'
print(f'Interpretable block: {X_interp.shape[0]} excerpts × {X_interp.shape[1]} named features '
      f'(rows aligned to eids and Y ✅)')

_ex = df_expr.set_index('excerpt_id')
y_val = _ex.loc[eids, 'valence_mean'].values.astype(float)     # 0–6 scale, midpoint 3.0
y_aro = _ex.loc[eids, 'arousal_mean'].values.astype(float)

# ── 2. REGRESSION read-out — the readable equation ───────────────────────────
INTERP_CV_R, reg_betas, _cv_rows = {pr: {} for pr in PROTOCOLS}, {}, []
READOUT_ORDER_REG, READOUT_ORDER_CLS = {}, {}
print('\n' + '═' * 78)
print('§9.2 (a) REGRESSION READ-OUT — standardized betas (V–A points per 1 SD of feature)')
print('═' * 78)
for dim, y in [('valence', y_val), ('arousal', y_aro)]:
    # (a) coefficients — all 60, descriptive. α by piece-grouped inner CV (no RidgeCV/GCV).
    _a_all = select_ridge_alpha(X_interp, y, PIECE_GROUPS)
    pipe = Pipeline([('sc', StandardScaler()), ('ridge', Ridge(alpha=_a_all))]).fit(X_interp, y)
    beta = pipe.named_steps['ridge'].coef_.ravel()
    b0   = float(pipe.named_steps['ridge'].intercept_)
    reg_betas[dim] = beta
    top = np.argsort(-np.abs(beta))[:5]
    eq  = '  '.join(f'{beta[i]:+.3f}·{interp_lbls[i]}' for i in top)
    READOUT_ORDER_REG[dim] = [interp_cols[i] for i in np.argsort(-np.abs(beta))]
    print(f'\n{dim} ≈ {b0:+.2f}  {eq}   …   (all-data α={_a_all:.3g}, piece-grouped)')
    print(f'   top feature by |β|: {top_feature_note(READOUT_ORDER_REG[dim])}')

    # (b) predictive score — nested ridge, both protocols (PRIMARY first)
    for pr in PROTOCOLS:
        yp, _al = nested_ridge_predict(X_interp, y, pr)
        r   = float(pearsonr(y, yp)[0])
        INTERP_CV_R[pr][dim] = {'r': r, 'r2': oos_r2(y, yp, pr), 'rmse': rmse(y, yp),
                                'alpha_median': float(np.median(_al))}
        _cv_rows.append(dict(protocol=pr, target=dim, **INTERP_CV_R[pr][dim]))
        print(f'   {pr:<6} r={r:+.3f}   R²={INTERP_CV_R[pr][dim]["r2"]:+.3f}   '
              f'RMSE={INTERP_CV_R[pr][dim]["rmse"]:.3f}   (median nested α={np.median(_al):.3g})')

df_readout_reg = pd.DataFrame(reg_betas, index=interp_lbls)[['valence', 'arousal']]

# ── 3. CLASSIFICATION read-out — one-vs-rest per circumplex class ────────────
print('\n' + '═' * 78)
print('§9.2 (b) CLASSIFICATION READ-OUT — signed betas + cross-validated F1 per circumplex class')
print('═' * 78)

def _clf_est():
    return LogisticRegression(penalty='l2', C=1.0, max_iter=1000, class_weight='balanced')

def _clf_pipe():
    return Pipeline([('sc', StandardScaler()), ('lr', _clf_est())])

cls_betas = {}
for j, cname in enumerate(mlb.classes_):
    yj = Y[:, j].astype(int)
    npos = int(yj.sum())
    if npos < 1 or npos == len(yj):
        print(f'  {cname:<9} — skipped (degenerate column, {npos}/{len(yj)} positive)')
        cls_betas[cname] = np.full(X_interp.shape[1], np.nan)
        continue
    cls_betas[cname] = _clf_pipe().fit(X_interp, yj).named_steps['lr'].coef_.ravel()

# Cross-validated per-class F1 through the §5.1 engine (single-class training folds predict
# the constant, as SafeMultiOutputClassifier) — both protocols.
INTERP_CV_F1, cls_f1 = {}, {}
for pr in PROTOCOLS:
    _Yp = cv_predict_multilabel(_clf_est, X_interp, Y, pr)
    _f1 = f1_score(Y, _Yp, average=None, zero_division=0)
    cls_f1[pr] = {c: float(_f1[j]) for j, c in enumerate(mlb.classes_)}
    INTERP_CV_F1[pr] = float(np.nanmean(list(cls_f1[pr].values())))
for j, cname in enumerate(mlb.classes_):
    top = np.argsort(-np.abs(np.nan_to_num(cls_betas[cname])))[:3]
    drivers = ', '.join(f'{cls_betas[cname][i]:+.2f}·{interp_lbls[i]}' for i in top)
    READOUT_ORDER_CLS[cname] = [interp_cols[i] for i in np.argsort(-np.abs(np.nan_to_num(cls_betas[cname])))]
    print(f'  {cname:<9} n={int(Y[:, j].sum()):<3} F1 piece={cls_f1["piece"][cname]:.3f} '
          f'loo={cls_f1["loo"][cname]:.3f}   top: {drivers}')
    if READOUT_ORDER_CLS[cname] and READOUT_ORDER_CLS[cname][0] in INTERP_FLAG_REASON:
        print(f'            ⚑ top feature is flagged → {top_feature_note(READOUT_ORDER_CLS[cname])}')

df_readout_cls = pd.DataFrame(cls_betas, index=interp_lbls)[list(mlb.classes_)]
for pr in PROTOCOLS:
    _cv_rows.append(dict(protocol=pr, target='class F1-macro (mean per-class F1)', f1=INTERP_CV_F1[pr]))
    print(f'\n→ [{PROTOCOL_LABEL[pr]}]')
    print(f'   INTERP_CV_F1 (macro mean of per-class F1) = {INTERP_CV_F1[pr]:.4f}')
    print(f'   INTERP_CV_R  = ' + ', '.join(f'{d}: r={v["r"]:+.3f} R²={v["r2"]:+.3f}'
                                         for d, v in INTERP_CV_R[pr].items()))
df_interp_cv = pd.DataFrame(_cv_rows)
warn_negative_loo_r(df_interp_cv.rename(columns={'r': 'r_cv'}), r_cols=('r_cv',))

# ── 4. Figures — diverging, centred at 0 ─────────────────────────────────────
_ord = np.argsort(-np.abs(df_readout_reg['arousal'].values))       # sort by |beta| on arousal
reg_sorted = df_readout_reg.iloc[_ord]
vmax_r = float(np.nanmax(np.abs(df_readout_reg.values))) or 1.0

fig, ax = plt.subplots(figsize=(7.5, max(4, 0.34 * len(reg_sorted))))
sns.heatmap(reg_sorted, annot=True, fmt='+.3f', cmap='RdBu_r', center=0,
            vmin=-vmax_r, vmax=vmax_r, linewidths=0.4, linecolor='#eee',
            cbar_kws={'label': 'standardized β (V–A points per 1 SD)'}, ax=ax)
ax.set_title('§9.2  Interpretable read-out — technique → V/A dimension\n'
             '(signed standardized betas, all 60 excerpts; sorted by |β| on arousal)',
             fontsize=11)
ax.set_ylabel('interpretable feature (CREPE pitch + Essentia dynamics/timbre/tonal\n+ DTW timing/rubato)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_readout_regression.png'), dpi=150, bbox_inches='tight')
plt.show()

vmax_c = float(np.nanmax(np.abs(df_readout_cls.values))) or 1.0
fig, ax = plt.subplots(figsize=(9, max(4, 0.34 * len(df_readout_cls))))
sns.heatmap(df_readout_cls.iloc[_ord], annot=True, fmt='+.2f', annot_kws={'fontsize': 7},
            cmap='RdBu_r', center=0, vmin=-vmax_c, vmax=vmax_c,
            linewidths=0.4, linecolor='#eee',
            cbar_kws={'label': 'standardized β (log-odds per 1 SD)'}, ax=ax)
ax.set_title('§9.2  Interpretable read-out — technique → circumplex class\n'
             '(one-vs-rest logistic betas, all 60 excerpts)', fontsize=11)
ax.set_ylabel('interpretable feature')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_readout_classes.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── 5. Persist ───────────────────────────────────────────────────────────────
pd.concat([df_readout_reg, df_readout_cls], axis=1).to_csv(
    os.path.join(OUTPUT_DIR, 'interp_readout_coefficients.csv'))
df_interp_cv.to_csv(os.path.join(OUTPUT_DIR, 'interp_readout_cv.csv'), index=False)

print('\n── Caveats ──')
print('  · Coefficients are fitted on all 60 excerpts — an interpretive table, NOT a')
print('    generalization claim. Only the cross-validated r / R² / F1 above are predictive.')
print('  · The forwarded blocks are correlation-pruned, but residual collinearity can make')
print('    individual signs unstable among correlated features — read the table at the level')
print('    of feature GROUPS (pitch / dynamics / timbre / tonal), not single coefficients.')
print('  · Betas are standardized: units are per 1 SD of the feature, so they are comparable')
print('    across features but not interpretable in the raw physical units.')
print('\n✅ §9.2 — interp_readout_coefficients.csv, interp_readout_cv.csv, fig_readout_regression.png, '
      'fig_readout_classes.png')


## Part 9 · Which techniques — §9.3 SHAP technique importance per circumplex class (nonlinear, in-sample cross-check)
**A nonlinear, in-sample cross-check (RandomForest fitted on all 60, no CV score).** For each of the
**5 circumplex classes** fit a **RandomForest** on the **predictive** interpretable layer of §4.1
(`INTERP_PRED_COLS`, 27 features: CREPE pitch + the forwarded Essentia dynamics / timbre / tonal + the
reference-free timing features; no DTW-deviation). **SHAP TreeExplainer** quantifies how much each technique
drives each class prediction. ⚑ marks the tempogram features. The cell ends with an **agreement table**: the
top feature per class by SHAP vs by the §9.2 one-vs-rest betas, and the number of classes where they agree.

*Mean |SHAP| = how much that technique "moves the needle" for that class.* (Fitted in-sample — a
descriptive view; whether to cross-validate it is still an open, not-yet-approved item.)

In [ ]:
# ── §9.3 SHAP technique importance per class ────────────────────────
# NEEDS : flag_label, INTERP_FLAG_REASON, INTERP_PRED_COLS, top_feature_note, X_INTERP (§4.1), mlb, OUTPUT_DIR,
#          READOUT_ORDER_CLS (§9.2), Y
# MAKES : df_shap_importance, df_shap_vs_beta, SHAP_COLS, SHAP_LABELS, shap_vals_all, X_interp_sc
#          · shap_technique_importance.csv, shap_vs_readout_agreement.csv
# FEEDS : Fig TE2 / TE3 below
# ═══════════════════════════════════════════════════════════════════════════════
# Path A — SHAP Feature Importance: a NONLINEAR, IN-SAMPLE cross-check
# (RandomForest fitted on all 60 excerpts, no CV score — a descriptive view, never a predictive claim)
# ═══════════════════════════════════════════════════════════════════════════════

# X_INTERP (§4.1) = the PREDICTIVE interpretable layer INTERP_PRED_COLS: CREPE (10) + Essentia-forward +
# DTW reference-free timing — no DTW-deviation feature. ⚑ marks the tempogram features in every label.
SHAP_COLS   = list(INTERP_PRED_COLS)
SHAP_LABELS = [flag_label(c) for c in SHAP_COLS]
assert X_INTERP.shape[1] == len(SHAP_COLS), '§9.3: X_INTERP and INTERP_PRED_COLS disagree'
scaler_shap  = StandardScaler()
X_interp_sc  = scaler_shap.fit_transform(X_INTERP)

# (5 classes × n_interpretable_features) — mean absolute SHAP value
shap_mean_abs = np.zeros((len(mlb.classes_), len(SHAP_COLS)))
shap_vals_all = {}   # class → (N, n_feat) raw SHAP for the beeswarm

print(f'SHAP (nonlinear, in-sample cross-check — RF fitted on all 60, no CV score)  |  {len(X_INTERP)} samples  |  '
      f'{X_INTERP.shape[1]} interpretable features  '
      f'|  {len(mlb.classes_)} classes')
print('─' * 60)

for j, emotion in enumerate(mlb.classes_):
    y_col = Y[:, j]
    if len(np.unique(y_col)) < 2:
        print(f'  {emotion:<24} — skipped (single class)')
        continue
    rf = RandomForestClassifier(n_estimators=400, max_depth=6,
                                class_weight='balanced', random_state=42, n_jobs=-1)
    rf.fit(X_interp_sc, y_col)
    explainer = shap.TreeExplainer(rf)
    sv        = explainer.shap_values(X_interp_sc)
    #error fix
    # Extract positive class SHAP values safely across different SHAP versions
    if isinstance(sv, list):
        sv1 = sv[1]                  # Older SHAP versions
    elif len(sv.shape) == 3:
        sv1 = sv[:, :, 1]            # Newer SHAP versions: (samples, features, classes)
    else:
        sv1 = sv                     # Fallback
    shap_vals_all[emotion]  = sv1
    shap_mean_abs[j]        = np.abs(sv1).mean(axis=0)
    top_i = np.argmax(shap_mean_abs[j])
    print(f'  {emotion:<24}  top: {SHAP_LABELS[top_i]:<28} mean|SHAP|={shap_mean_abs[j,top_i]:.4f}'
          + (f'   ⚑ → {top_feature_note([SHAP_COLS[i] for i in np.argsort(-shap_mean_abs[j])])}'
             if SHAP_COLS[top_i] in INTERP_FLAG_REASON else ''))

df_shap_importance = pd.DataFrame(
    shap_mean_abs,
    index   = list(mlb.classes_),
    columns = SHAP_LABELS
)
df_shap_importance.to_csv(os.path.join(OUTPUT_DIR,'shap_technique_importance.csv'))
print(f'\n✅ Saved: shap_technique_importance.csv')
print('\nFull matrix (mean |SHAP|):')
print(df_shap_importance.round(4).to_string())

# ── Agreement: the top feature per class by SHAP (nonlinear, in-sample) vs by the §9.2 betas (linear) ──
_agree = []
for j, emotion in enumerate(mlb.classes_):
    if emotion not in shap_vals_all or not READOUT_ORDER_CLS.get(emotion):
        continue
    _sh_order = [SHAP_COLS[i] for i in np.argsort(-shap_mean_abs[j])]
    _b_order  = READOUT_ORDER_CLS[emotion]
    _agree.append(dict(cls=emotion, top_shap=flag_label(_sh_order[0]), top_beta=flag_label(_b_order[0]),
                       agree=bool(_sh_order[0] == _b_order[0]),
                       top_shap_unflagged=next((flag_label(c) for c in _sh_order if c not in INTERP_FLAG_REASON), ''),
                       top_beta_unflagged=next((flag_label(c) for c in _b_order if c not in INTERP_FLAG_REASON), ''),
                       shap_top_in_beta_top3=bool(_sh_order[0] in _b_order[:3])))
df_shap_vs_beta = pd.DataFrame(_agree)
print('\n── Top feature per class: SHAP (RF, in-sample) vs the §9.2 one-vs-rest betas (linear) ──')
print(df_shap_vs_beta.to_string(index=False))
print(f'   classes where the top feature agrees: {int(df_shap_vs_beta["agree"].sum())}/{len(df_shap_vs_beta)} · '
      f'SHAP top within the β top-3: {int(df_shap_vs_beta["shap_top_in_beta_top3"].sum())}/{len(df_shap_vs_beta)}')
df_shap_vs_beta.to_csv(os.path.join(OUTPUT_DIR, 'shap_vs_readout_agreement.csv'), index=False)

In [ ]:
# ── Fig TE2: SHAP Technique Importance Heatmap ───────────────────────────────
fig, ax = plt.subplots(figsize=(max(13, 0.95 * df_shap_importance.shape[1]), 5))
sns.heatmap(
    df_shap_importance, annot=True, fmt='.3f', annot_kws={'fontsize': 7}, cmap='YlOrRd',
    linewidths=0.5, linecolor='#ddd', ax=ax,
    cbar_kws={'label':'Mean |SHAP value|', 'shrink':0.7}
)
ax.tick_params(axis='x', rotation=40, labelsize=8)
plt.setp(ax.get_xticklabels(), ha='right')
ax.set_xlabel('Performance technique (CREPE pitch + Essentia dynamics/timbre/tonal + reference-free timing; '
              '⚑ = tempogram, failed the audit)', fontsize=10)
ax.set_ylabel('Perceived Class (circumplex quadrant)', fontsize=11)
ax.set_title(
    'Path A — Technique Importance per Circumplex Class  (SHAP · RF 400 trees · fitted in-sample on all 60, no CV)\n'
    'Higher = that technique contributes more to predicting that class',
    fontsize=12
)
plt.tight_layout()
fpath = os.path.join(OUTPUT_DIR,'fig_shap_importance.png')
plt.savefig(fpath, dpi=150, bbox_inches='tight'); plt.show()
print(f'✅ Fig TE2 saved → {fpath}')

In [ ]:
# ── Fig TE3: Ranked bar per class + beeswarm for top class ───────────────────
# Renamed from COND_COLORS: that name is the MEC/EXP/EXG palette in §3.1 and this
# dict shadowed it. Keys are now the 5 circumplex CLASSES, not the old 8 emotions.
CLASS_COLORS = {
    'HA_HV':'#e06c26','HA_LV':'#c0392b','LA_HV':'#57a64a',
    'LA_LV':'#2980b9','Neutral':'#7f8c8d'
}
emotions_active = [e for e in mlb.classes_ if e in shap_vals_all]
ncols = min(3, max(1, len(emotions_active))); nrows = (len(emotions_active) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5*nrows))
axes = np.atleast_1d(axes).flatten()
for idx, emotion in enumerate(emotions_active):
    ax    = axes[idx]
    vals  = df_shap_importance.loc[emotion].values
    order = np.argsort(vals)[::-1]
    colors = ['#c0392b' if i==order[0] else '#85c1e9' for i in range(len(vals))]
    ax.barh([SHAP_LABELS[i] for i in order[::-1]], vals[order[::-1]],
            color=[colors[i] for i in order[::-1]])
    ax.set_title(f'{emotion}\n{QUADRANT_DISPLAY[emotion]}', fontsize=10,
                 color=CLASS_COLORS.get(emotion,'#333'), fontweight='bold')
    ax.set_xlabel('Mean |SHAP|', fontsize=8)
    ax.tick_params(axis='y', labelsize=7); ax.tick_params(axis='x', labelsize=7)
for idx in range(len(emotions_active), len(axes)):
    axes[idx].set_visible(False)
fig.suptitle(
    'Path A — Which Technique Drives Each Circumplex Class? (SHAP Ranking)\n'
    'Red bar = most influential technique (in-sample RF; ⚑ = tempogram, failed the audit — never quoted alone)',
    fontsize=13, y=1.01
)
plt.tight_layout()
fpath = os.path.join(OUTPUT_DIR,'fig_shap_ranked.png')
plt.savefig(fpath, dpi=150, bbox_inches='tight'); plt.show()
print(f'✅ Fig TE3 saved → {fpath}')

# ── Beeswarm for the class where SHAP is most decisive ───────────────────────
key_em = max(shap_vals_all, key=lambda e: df_shap_importance.loc[e].max())
print(f'\nBeeswarm → {key_em} ({QUADRANT_DISPLAY[key_em]})')
shap.summary_plot(shap_vals_all[key_em], features=X_interp_sc,
                  feature_names=SHAP_LABELS, plot_type='beeswarm',
                  show=False, max_display=len(SHAP_LABELS))
plt.title(f'SHAP Beeswarm — {key_em} ({QUADRANT_DISPLAY[key_em]})', fontsize=12)
fpath_bee = os.path.join(OUTPUT_DIR, f'fig_shap_beeswarm_{key_em.replace(" ","_")}.png')
plt.savefig(fpath_bee, dpi=150, bbox_inches='tight'); plt.show()
print(f'✅ Beeswarm saved → {fpath_bee}')


## Part 10 · Cost of explainability — §10.1 Cost of explainability
**Question of this part:** what does predicting through the named techniques cost against the MERT black box,  
and where inside the interpretable layer is the predictive work done?
**Hands to the next part:** `COE` / `df_coe_gaps`, `df_abl_subfamily` / `df_abl_commonality`, the overlap verdicts
of §10.3 and `ablation_master.csv` — the inputs of the §11.1 claims table.

**What do we lose by predicting emotion through the interpretable technique layer instead of
the MERT black box?**

Every feature set is scored under the **same protocol** — the nested piece-grouped ridge for the
valence / arousal regression (r, out-of-sample R², RMSE), the default classifier
(nested model selection, §5.1) for class F1 — under **both** CV protocols (leave-one-piece-out primary), so
the only thing that varies is the representation.

A **small gap** means the measured violin techniques carry almost all of the emotional signal,
which is the core justification for the interpretable route (cf. *Chowdhury et al. 2019*): the
explanation comes nearly for free. A **large gap** is equally informative — it is the price of
explainability, stated in the same units as the headline result. A **negative** gap means the
interpretable route *outperforms* the representation (on unseen pieces, under `piece`).

The headline black box for V/A is `max(MERT-PCA, MERT-full)` — a best-of over two black boxes,
i.e. **conservative** (it can only inflate the cost); the two separate gaps are printed and saved
next to it (`coe_gaps.csv`).

Part 7 asked *what MERT encodes* (§7.1–§7.4) and *what each tool adds on top of it* (§7.5); §10.1 asks
*what the interpretable layer costs*. **Phase C2:** the interpretable side is `INTERP_PRED_COLS` (§4.1) —
no DTW-deviation feature on either side, so the CoE is condition-independent (this closes the former
`DTW_DEV_IN_INTERP` question). Labels: **MERT-PCA25** (classification head), **MERT-full 768** (regression).


In [ ]:
# ── §10.1 Cost of Explainability ─────────────────────────────────────
# NEEDS : cv_predict_default, DEFAULT_CLF_NAME, df_expr, df_interp, eids, INTERP_CV_R (§9.2),
#          DTW_FORWARD_DEV, INTERP_PRED_COLS (§4.1), mlb, nested_ridge_predict, oos_r2, OUTPUT_DIR,
#          PRIMARY_PROTOCOL, PROTOCOLS, PROTOCOL_LABEL, rmse, warn_negative_loo_r, X_A,
#          X_MERT_FULL, Y
# MAKES : COE (protocol → headline gaps), CoE_arousal, CoE_f1, CoE_valence (= PRIMARY headline),
#          df_coe (protocol × feature_set × {r, R², RMSE, F1}), df_coe_gaps · coe.csv,
#          coe_gaps.csv, 1 figure
# FEEDS : the report's headline claim; §10.2 tie-back; §11.1 (claim 7)
# ═══════════════════════════════════════════════════════════════════════════════
# §10.1 · Cost of Explainability — interpretable layer vs the MERT black box
# ═══════════════════════════════════════════════════════════════════════════════

# ── 1. Self-contained inputs (only INTERP_CV_R is borrowed from §9.2) ───────
interpretable_cols = list(INTERP_PRED_COLS)           # Phase C2: the PREDICTIVE layer (no DTW-deviation)
assert 'mert' not in ''.join(map(str, interpretable_cols)).lower(), \
    '§10.1: the interpretable set must contain no embedding dimension'

_Xi = df_interp.loc[eids, interpretable_cols].apply(pd.to_numeric, errors='coerce')
_Xi = _Xi.fillna(_Xi.mean(numeric_only=True))
X_interp = _Xi.to_numpy(dtype=float)

# Phase C2: the interpretable side is INTERP_PRED_COLS (§4.1) — the DTW-deviation family (0/1 at MEC by
# construction, it encodes the condition) is OUT of both sides, so the CoE is strictly condition-independent.
assert not set(interpretable_cols) & set(DTW_FORWARD_DEV), '§10.1: a DTW-deviation column in the CoE'
print(f'§10.1 interpretable side: Interpretable ({len(interpretable_cols)}) = INTERP_PRED_COLS (no DTW-deviation)')

_ex   = df_expr.set_index('excerpt_id')
y_val = _ex.loc[eids, 'valence_mean'].to_numpy(dtype=float)
y_aro = _ex.loc[eids, 'arousal_mean'].to_numpy(dtype=float)
assert X_interp.shape[0] == Y.shape[0] == len(eids), '§10.1: row-count mismatch'

# ── 2. ONE protocol per task, applied identically to every feature set ─────
def score_reg(X, y, protocol):
    """Nested piece-grouped ridge (§5.1) → (Pearson r, out-of-sample R², RMSE)."""
    yp, _ = nested_ridge_predict(X, y, protocol)
    return float(pearsonr(y, yp)[0]), oos_r2(y, yp, protocol), rmse(y, yp)

def score_clf(X, Ym, protocol, label=None):
    """The default classifier (§5.1 — nested selection, B-NMS) → macro F1."""
    Yp = cv_predict_default(X, Ym, protocol, label=label)
    return float(f1_score(Ym, Yp, average='macro', zero_division=0))

# ── 3. Feature sets — identical scoring functions for all three ────────────
_INTERP = f'Interpretable ({X_interp.shape[1]}: CREPE+Essentia+DTW ref-free)'
_PCA    = f'MERT-PCA{X_A.shape[1]} (deployed classification block)'
_FULL   = f'MERT-full {X_MERT_FULL.shape[1]} (deployed regression block)'
SETS = {_INTERP: X_interp, _PCA: X_A, _FULL: X_MERT_FULL}
coe_rows = []
print(f'📊 §10.1 · COST OF EXPLAINABILITY — nested ridge for V/A, {DEFAULT_CLF_NAME} for class F1')
for pr in PROTOCOLS:
    print(f'\n── {PROTOCOL_LABEL[pr]} ──')
    for name, X in SETS.items():
        rv, r2v, mv = score_reg(X, y_val, pr)
        ra, r2a, ma = score_reg(X, y_aro, pr)
        f1m         = score_clf(X, Y, pr, label=f'§10.1 · {name}')
        coe_rows.append({'protocol': pr, 'feature_set': name, 'n_features': int(X.shape[1]),
                         'r_valence': rv, 'r2_val': r2v, 'rmse_val': mv,
                         'r_arousal': ra, 'r2_aro': r2a, 'rmse_aro': ma, 'f1_macro': f1m})
        print(f'  {name:<46} r_val={rv:+.3f} R²_val={r2v:+.3f}  r_aro={ra:+.3f} R²_aro={r2a:+.3f}  '
              f'F1-macro={f1m:.4f}')
df_coe = pd.DataFrame(coe_rows)
warn_negative_loo_r(df_coe)
print('\n  ⚠️  At 768 dims with N=60 the full-MERT CLASSIFIER is heavily over-parameterised —')
print('      treat its F1 with care. The deployed comparison is Interpretable vs MERT-PCA.')

# ── 4. Protocol cross-check against §9.2 — REGRESSION ONLY.
#       §9.2's class read-out used logistic regression, this cell uses the default (NestedSelect), so
#       the F1 numbers are not comparable and are deliberately NOT cross-checked. Both cells
#       call the same nested_ridge_predict on the same matrix → the r must match exactly. ──
for pr in PROTOCOLS:
    _r_here = float(df_coe.query('protocol == @pr and feature_set == @_INTERP')['r_arousal'].iloc[0])
    assert abs(_r_here - INTERP_CV_R[pr]['arousal']['r']) < 1e-9, (
        f'§10.1 protocol mismatch with §9.2 ({pr}): r_arousal {_r_here:+.4f} vs '
        f'{INTERP_CV_R[pr]["arousal"]["r"]:+.4f} — both should be the nested ridge')
print('  ✅ cross-check: interpretable r_arousal matches §9.2 exactly under both protocols')

# ── 5. The gaps ───────────────────────────────────────────────────────────
# HEADLINE (unchanged definition): for V/A the black box is max(MERT-PCA, MERT-full) — a
# BEST-OF over two black boxes, i.e. CONSERVATIVE (it can only inflate the cost); for class
# F1 the black box is MERT-PCA (the full-768 classifier is over-parameterised). The two
# separate gaps (vs MERT-PCA, vs MERT-full) are reported next to it and saved.
_GAP_METRICS = ['r_valence', 'r_arousal', 'r2_val', 'r2_aro', 'f1_macro']
COE, gap_rows = {}, []
for pr in PROTOCOLS:
    sub = df_coe[df_coe.protocol == pr].set_index('feature_set')
    i_, p_, f_ = sub.loc[_INTERP], sub.loc[_PCA], sub.loc[_FULL]
    COE[pr] = {m: (float(p_[m] - i_[m]) if m == 'f1_macro' else float(max(p_[m], f_[m]) - i_[m]))
               for m in _GAP_METRICS}
    gap_rows.append({'protocol': pr, 'comparison': 'HEADLINE: max(MERT-PCA, MERT-full) − interp '
                     '(F1: MERT-PCA − interp)', **COE[pr]})
    gap_rows.append({'protocol': pr, 'comparison': 'MERT-PCA − interpretable',
                     **{m: float(p_[m] - i_[m]) for m in _GAP_METRICS}})
    gap_rows.append({'protocol': pr, 'comparison': 'MERT-full − interpretable',
                     **{m: float(f_[m] - i_[m]) for m in _GAP_METRICS}})
df_coe_gaps = pd.DataFrame(gap_rows)
CoE_valence = COE[PRIMARY_PROTOCOL]['r_valence']
CoE_arousal = COE[PRIMARY_PROTOCOL]['r_arousal']
CoE_f1      = COE[PRIMARY_PROTOCOL]['f1_macro']

def _verdict(gap, what, protocol):
    where = ('on unseen pieces' if protocol == 'piece'
             else 'on held-out renditions (LOOCV, optimistic)')
    if not np.isfinite(gap):
        return f'  → {what}: n/a'
    if gap < -1e-12:
        return (f'  → {what}: {where} the interpretable route OUTPERFORMS the representation '
                f'by {-gap:.3f}.')
    if abs(gap) <= 1e-12:
        return f'  → {what}: {where} the interpretable layer MATCHES the black box.'
    if gap < 0.05:
        return (f'  → {what}: recovered to within {gap:+.3f} of the black box — the measured '
                f'techniques carry ~all of this signal.')
    if gap < 0.15:
        return f'  → {what}: a moderate cost of {gap:+.3f}; the technique layer covers most but not all.'
    return f'  → {what}: a substantial cost of {gap:+.3f}; much of this signal is not in the named features.'

for pr in PROTOCOLS:
    print('\n' + '═' * 92)
    print(f'COST OF EXPLAINABILITY — {PROTOCOL_LABEL[pr]}')
    print('(black box − interpretable; positive = the interpretable route loses, NEGATIVE = it wins)')
    print('═' * 92)
    print(df_coe_gaps[df_coe_gaps.protocol == pr].drop(columns='protocol').round(3).to_string(index=False))
    print()
    print(_verdict(COE[pr]['r_arousal'], 'AROUSAL (r)', pr))
    print(_verdict(COE[pr]['r_valence'], 'VALENCE (r)', pr))
    print(_verdict(COE[pr]['r2_aro'],    'AROUSAL (R²)', pr))
    print(_verdict(COE[pr]['r2_val'],    'VALENCE (R²)', pr))
    print(_verdict(COE[pr]['f1_macro'],  'CLASS F1', pr))
print(f'\n  Headline (PRIMARY, {PRIMARY_PROTOCOL}): ΔR valence {CoE_valence:+.3f} · ΔR arousal '
      f'{CoE_arousal:+.3f} · ΔF1 {CoE_f1:+.3f}')
print('  Note: BOTH routes are expected to predict VALENCE less well than AROUSAL — a standard')
print('  MER finding, and Part 4 (§4.3) already showed this manipulation moves arousal, not valence. A')
print('  large valence gap therefore says less about the interpretable layer than a large')
print('  arousal gap would.')

# ── 6. Figure — one panel per protocol ────────────────────────────────────
METRICS = [('r_valence', 'Valence r'), ('r_arousal', 'Arousal r'),
           ('r2_val', 'Valence R²'), ('r2_aro', 'Arousal R²'), ('f1_macro', 'Class F1-macro')]
setcol = ['#2a9d2a', '#8e44ad', '#888888']
x, w   = np.arange(len(METRICS)), 0.26
fig, axes = plt.subplots(1, len(PROTOCOLS), figsize=(8.5 * len(PROTOCOLS), 5.5), sharey=True)
for ax, pr in zip(np.atleast_1d(axes), PROTOCOLS):
    sub = df_coe[df_coe.protocol == pr].set_index('feature_set')
    for i, (name, col) in enumerate(zip(SETS, setcol)):
        vals = [float(sub.loc[name, m]) for m, _ in METRICS]
        bars = ax.bar(x + (i - 1) * w, vals, w, label=name, color=col, edgecolor='#333', linewidth=0.5)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.008, f'{v:.2f}',
                    ha='center', va='bottom', fontsize=6.5)
    for j, (m, _) in enumerate(METRICS):
        lo  = float(sub.loc[_INTERP, m])
        hi  = lo + COE[pr][m]
        ax.annotate('', xy=(x[j] + 0.34, hi), xytext=(x[j] + 0.34, lo),
                    arrowprops=dict(arrowstyle='<->', color='#b00020', lw=1.3))
        ax.text(x[j] + 0.38, (hi + lo) / 2, f'CoE\n{COE[pr][m]:+.3f}',
                fontsize=7, color='#b00020', va='center')
    ax.axhline(0, color='#333', lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels([lbl for _, lbl in METRICS], fontsize=9)
    ax.set_title(f'{pr}' + (' — PRIMARY (unseen pieces)' if pr == 'piece' else ' — optimistic (LOOCV)'),
                 fontsize=10)
    ax.grid(axis='y', alpha=0.25)
_vals = df_coe[[m for m, _ in METRICS]].to_numpy(dtype=float)
np.atleast_1d(axes)[0].set_ylim(min(0.0, float(np.nanmin(_vals))) - 0.05,
                                max(1.0, float(np.nanmax(_vals)) * 1.25))
np.atleast_1d(axes)[0].set_ylabel('score ↑')
np.atleast_1d(axes)[0].legend(fontsize=8, loc='upper left')
fig.suptitle('§10.1  Cost of Explainability — interpretable technique layer vs the MERT black box\n'
             f'nested piece-grouped ridge for V/A · {DEFAULT_CLF_NAME} for class F1 · '
             'headline black box = max(MERT-PCA, MERT-full) (F1: MERT-PCA)', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_coe.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── 7. Persist ────────────────────────────────────────────────────────────
df_coe.to_csv(os.path.join(OUTPUT_DIR, 'coe.csv'), index=False)
df_coe_gaps.to_csv(os.path.join(OUTPUT_DIR, 'coe_gaps.csv'), index=False)
print('\n✅ §10.1 — coe.csv, coe_gaps.csv, fig_coe.png')


## Part 10 · Cost of explainability — §10.2 Sub-family ablation · decomposing the cost of explainability

§10.1 gives the cost of explainability as **one number per task**: what the whole interpretable layer
loses against the MERT black box. It cannot say *where inside the interpretable layer* the predictive
work is done. This section partitions that layer into the expressive families named in the brief —

`Pitch / F0` · `Vibrato` · `Portamento` · `Dynamics` · `Timbre` · `Tonal` · `Timing (ref-free)`

(**Phase C2:** the layer is the **predictive** one of §4.1, `INTERP_PRED_COLS` — 27 features. The former
`Timing (DTW-dev)` family and its "− DTW-dev" removal row no longer apply: the DTW-deviation features are
0/1 at MEC by construction and are descriptive only; the cell prints one line saying so.)

— and scores each family **removed** (leave-one-family-out, against the full interpretable set) and
**alone**. The CoE stops being a single verdict and becomes a decomposition: *this much of the
explainable signal lives in dynamics, this much in timing, this much nowhere measurable.*

This is also the third and independent line of evidence for the attribution claims, alongside
the dose-response (§4.2) and the read-out / SHAP (§9.2 / §9.3) — and the only **predictive** one. A family that a
correlational analysis flags can still turn out to carry no out-of-sample signal, and vice versa.

> ### ⚠️ What a null result for a family does and does not license
> A family scoring Δ≈0 means **its descriptors carry no predictable signal**, which is not
> automatically the claim *the technique does not matter*. For `Vibrato` the Phase A descriptors were
> known to be broken (rate = argmax inside the same [4,8] Hz band the signal was bandpassed to, so it
> could never report "no vibrato"; features over concatenated non-contiguous voiced frames; depth
> conflating extent with coverage). **Phase B (B8) repaired them**: per-note analysis on
> time-contiguous segments, rate searched in [3, 10] Hz with an explicit presence test, and extent
> (`crepe_vibrato_extent_cents`) separated from coverage (`crepe_vibrato_coverage`). The manual audit
> of 2026-11-05 then showed that B8's FFT presence test rejected clear vibrato (at 0.5–2 s the Hann
> main lobe is wider than its ±0.5 Hz peak window) and that the boundary rule merged notes and slides,
> so **Phase B2 (B8b) replaced it**: B2 plateau notes ≥ 500 ms, 500 ms windows (hop 250 ms), a
> least-squares 4–8 Hz sinusoid fit per window (vibrato iff R² ≥ 0.50 and half-extent ≥ 3 c),
> coverage = vibrato-window time / analysed-note time.
>
> A vibrato null is therefore a **musical result at this N** (60 excerpts), with the remaining
> caveats printed by the cell: CREPE's 10 ms resolution, only notes ≥ `VIB_MIN_NOTE_MS` (500 ms)
> analysed (Phase B3 also rejects windows whose best rate is on the 4 / 8 Hz grid edge), and the
> declared median substitutions of §2.2.

**Phase B3 additions.** (1) The F1 panel of the figure draws the **chance mean** (`CHANCE_FLOOR_MEAN`)
and the **strict P95** (`CHANCE_FLOOR_P95`) of the primary protocol — an "alone" bar starts at 0, but
chance is ≈ 0.27. (2) Removal losses whose BH-FDR q (within the tier) is < 0.05 are marked `*`.
(3) A **commonality summary** per protocol for F1 (taken above the chance mean), R² arousal and R²
valence: Σ alone, Σ removal loss, the full score and *shared ≈ full − Σ loss*. In the full B2 run the full
layer gave R²a 0.627 with Σ alone ≈ 1.55 and Σ loss ≈ 0.23 — most of the arousal signal is carried by
several families at once. Out-of-sample scores are not exactly additive (every subset gets its own
nested selection / ridge α), so the split is approximate; **§10.3** tests the overlap directly.
(4) **Pre-declared timing reading.** The B2 re-articulation splits followed the vibrato and inflated the
note-based timing features; with them disabled (B3), the cell prints `Timing (ref-free)` alone F1
(`piece`) next to Phase B (0.417) and B2 (0.542): near 0.54 → the timing signal is real; back near 0.42 →
the B2 value was the vibrato artefact. Record which; quote only the B3 value in the thesis.

**Phase C1 additions.** The B3 run showed that the Dummy floor is too low for the deployed NestedSelect
path (25 noise columns scored piece F1 0.354 > the Dummy strict P95 0.323; §5.3). The F1 panel therefore
also draws the **empirical-null P95** (§5.3, `piece`; Phase C2: the **exact-width** primary null of
Interpretable (27), plus — Phase C3 — the primary-null P95 of each width the families alone are read against),
and the decomposition printout gains `alone_emp_p`, the empirical-null p of every family-alone F1 (Phase C3
rule: the primary null of the family's width, else the next larger primary width, conservative; the label is in
`alone_emp_null`; NaN under a protocol whose null was not drawn), plus the
empirical-null p of the full layer. An "alone" F1 above the Dummy P95 but not below the empirical-null
P95 is not evidence that the family carries signal.

In [ ]:
# ── §10.2 Sub-family ablation of the interpretable layer (Tier 3) + master table ──
# NEEDS : ablate, abl_frame, abl_vs_ref, ABL_ORDER, ABL_ROWS, CHANCE_FLOOR_MEAN, CHANCE_FLOOR_P95,
#          CREPE_STEP_MS, df_interp, DTW_DEV, DTW_FORWARD_DEV, DTW_REFFREE, eids, emp_null_ref (§5.3), INTERP_FEAT_LABELS, INTERP_PRED_COLS (§4.1), OUTPUT_DIR, PRIMARY_PROTOCOL, PROTOCOLS, PROTOCOL_LABEL,
#          VIB_MIN_NOTE_MS, VIB_N_SUBST_NONOTES, VIB_N_SUBST_NOVIB, warn_negative_loo_r
# MAKES : df_abl_family, df_abl_subfamily, df_abl_commonality, df_ablation, INTERP_FAMILIES,
#          _interp_cols, _interp_family, _REF_INTERP, _X_fam · ablation_subfamily.csv,
#          ablation_subfamily_decomposition.csv, ablation_subfamily_deltas.csv,
#          ablation_subfamily_commonality.csv, ablation_master.csv, 1 figure
# FEEDS : the report's cost-of-explainability decomposition; §10.3 (family overlap: INTERP_FAMILIES,
#          _interp_cols, _REF_INTERP, _X_fam, df_abl_subfamily; re-exports ablation_master.csv)
# ═══════════════════════════════════════════════════════════════════════════════
# §10.2 · Where inside the interpretable layer does the predictive work happen?
# ═══════════════════════════════════════════════════════════════════════════════
# §10.1 reports ONE cost-of-explainability number per task. This decomposes it by
# expressive family: each family removed (leave-one-family-out) and each family alone,
# under the SAME protocol as every other tier.

def _interp_family(col):
    """Map an interpretable column to its expressive family. Explicit rules, no
    catch-all: an unmapped column raises below rather than landing in a silent bucket."""
    if col in DTW_REFFREE:                                    return 'Timing (ref-free)'
    if col in DTW_DEV:                                        return 'Timing (DTW-dev)'
    if 'vibrato' in col:                                      return 'Vibrato'
    if 'portamento' in col:                                   return 'Portamento'
    if col.startswith('crepe_'):                              return 'Pitch / F0'
    if 'loudness' in col or 'dynamic_range' in col:           return 'Dynamics'
    if any(k in col for k in ('spectral_', 'dissonance', 'log_attack')):
        return 'Timbre'
    if 'key_strength' in col or 'hpcp' in col:                return 'Tonal'
    return None


# Phase C2: the PREDICTIVE layer of §4.1 (INTERP_PRED_COLS) — the DTW-deviation family is descriptive only
_interp_cols = list(INTERP_PRED_COLS)
assert _interp_cols, '§10.2: the interpretable frame is empty'
assert not set(_interp_cols) & set(DTW_FORWARD_DEV), '§10.2: a DTW-deviation column in the predictive layer'
print('ℹ️  §10.2: the former "Timing (DTW-dev)" family (and its "− DTW-dev" removal row) no longer applies — the '
      'DTW-deviation features are 0/1 at MEC by construction, so since Phase C2 they are descriptive only and '
      'never enter a predictive analysis (§4.1).')
assert not any('mert' in c.lower() for c in _interp_cols), \
    '§10.2: the interpretable set must contain no embedding dimension'

_unmapped = [c for c in _interp_cols if _interp_family(c) is None]
if _unmapped:
    raise KeyError(f'§10.2: no family rule for {_unmapped} — add one rather than letting '
                   'these columns fall into an unnamed bucket')

INTERP_FAMILIES = {}
for _c in _interp_cols:
    INTERP_FAMILIES.setdefault(_interp_family(_c), []).append(_c)

print(f'§10.2 · interpretable layer = {len(_interp_cols)} features in '
      f'{len(INTERP_FAMILIES)} families')
for _f, _cs in INTERP_FAMILIES.items():
    print(f'   {_f:<20} {len(_cs)}  ({", ".join(INTERP_FEAT_LABELS[c] for c in _cs)})')


def _X_fam(cols):
    """Row-align the interpretable frame to `eids` — same construction as §10.1."""
    D = df_interp.loc[eids, list(cols)].apply(pd.to_numeric, errors='coerce')
    return D.fillna(D.mean(numeric_only=True)).to_numpy(dtype=float)


_REF_INTERP = f'ALL interpretable ({len(_interp_cols)}d)'
print(f'\n📊 TIER 3 · leave-one-family-out + family-alone   (both protocols, piece first)')
ablate(_X_fam(_interp_cols), _REF_INTERP, tier='3-subfamily')
for _fam, _cols in INTERP_FAMILIES.items():
    _rest = [c for c in _interp_cols if c not in _cols]
    if _rest:
        ablate(_X_fam(_rest), f'− {_fam} ({len(_rest)}d)', tier='3-subfamily',
               note=f'leave-one-family-out: {_fam} removed')
    ablate(_X_fam(_cols), f'only {_fam} ({len(_cols)}d)', tier='3-subfamily',
           note=f'family alone: {_fam}')

_fam_deltas   = abl_vs_ref('3-subfamily', _REF_INTERP)
df_abl_family = abl_frame().query('tier == "3-subfamily"').reset_index(drop=True)

# ── Assemble the decomposition table ───────────────────────────────────────
frows = []
for _pr in PROTOCOLS:
    _full = next(r for r in ABL_ROWS if r['tier'] == '3-subfamily'
                 and r['set_name'] == _REF_INTERP and r['protocol'] == _pr)
    for _fam, _cols in INTERP_FAMILIES.items():
        _rest_name  = f'− {_fam} ({len([c for c in _interp_cols if c not in _cols])}d)'
        _alone_name = f'only {_fam} ({len(_cols)}d)'
        _drop = next((r for r in ABL_ROWS if r['tier'] == '3-subfamily'
                      and r['set_name'] == _rest_name and r['protocol'] == _pr), None)
        _alone = next(r for r in ABL_ROWS if r['tier'] == '3-subfamily'
                      and r['set_name'] == _alone_name and r['protocol'] == _pr)
        _d = _fam_deltas.query('protocol == @_pr and set_name == @_rest_name')
        frows.append(dict(
            protocol=_pr, family=_fam, n_features=len(_cols),
            # cost of REMOVING this family from the full interpretable layer
            loss_f1        = (_full['f1_macro']  - _drop['f1_macro'])  if _drop else np.nan,
            loss_r_valence = (_full['r_valence'] - _drop['r_valence']) if _drop else np.nan,
            loss_r_arousal = (_full['r_arousal'] - _drop['r_arousal']) if _drop else np.nan,
            loss_r2_val    = (_full['r2_val']    - _drop['r2_val'])    if _drop else np.nan,
            loss_r2_aro    = (_full['r2_aro']    - _drop['r2_aro'])    if _drop else np.nan,
            q_loss_f1      = float(_d['q_f1'].iloc[0]) if len(_d) else np.nan,
            q_loss_r_arousal = float(_d['q_r_arousal'].iloc[0]) if len(_d) else np.nan,
            q_loss_r2_aro  = float(_d['q_r2_aro'].iloc[0]) if len(_d) else np.nan,
            q_loss_r2_val  = float(_d['q_r2_val'].iloc[0]) if len(_d) else np.nan,
            # what the family delivers on its own
            alone_f1=_alone['f1_macro'], alone_r_valence=_alone['r_valence'],
            alone_r_arousal=_alone['r_arousal'], alone_r2_val=_alone['r2_val'],
            alone_r2_aro=_alone['r2_aro']))
df_abl_subfamily = pd.DataFrame(frows)
# The empirical-null p of every family-alone F1 — Phase C3: the ONE rule of emp_null_ref (§5.3: the primary null of
# the same width, else the next larger primary width, conservative); the label names the width and its draws.
# NaN = that protocol's null was not drawn (RUN_NULL_LOO = False → 'loo' is n/a)
_alone_en = [emp_null_ref(r.alone_f1, r.protocol, r.n_features) for r in df_abl_subfamily.itertuples()]
df_abl_subfamily['alone_emp_p'] = [e['p'] for e in _alone_en]
df_abl_subfamily['alone_emp_null'] = [e['label'] for e in _alone_en]

print('\n📊 §10.2 — LOSS when a family is removed vs what it achieves ALONE')
for _pr in PROTOCOLS:
    print(f'\n── {PROTOCOL_LABEL[_pr]} ──')
    print(df_abl_subfamily.query('protocol == @_pr')[
        ['family', 'n_features', 'loss_f1', 'q_loss_f1', 'loss_r_arousal', 'loss_r2_aro',
         'q_loss_r2_aro', 'alone_f1', 'alone_emp_p', 'alone_r_arousal', 'alone_r2_aro', 'alone_r2_val']]
        .round(4).to_string(index=False))
    print('   alone_emp_p null used: ' + ' · '.join(
        f'{r.family}: {r.alone_emp_null}' for r in df_abl_subfamily[df_abl_subfamily.protocol == _pr].itertuples()))
    _full_f1 = next(r for r in ABL_ROWS if r['tier'] == '3-subfamily' and r['set_name'] == _REF_INTERP
                    and r['protocol'] == _pr)['f1_macro']
    print(f'   alone_emp_p = empirical-null p of the family-alone F1 (§5.3, Phase C3 rule: exact width, else the next '
          f'larger primary width — "borrowed", conservative; NaN = not drawn under {_pr!r}) · full layer F1 {_full_f1:.3f}: '
          f'empirical-null p {emp_null_ref(_full_f1, _pr, len(_interp_cols), f"Interpretable ({len(_interp_cols)})")["p"]:.3g} '
          f'({emp_null_ref(_full_f1, _pr, len(_interp_cols), f"Interpretable ({len(_interp_cols)})")["label"]})')
warn_negative_loo_r(df_abl_family)

# ── Commonality summary (B3): how much of the full score is SHARED between families ──────────
# Σ alone counts a signal once per family that carries it; Σ removal loss counts only what each family
# carries ALONE (what no other family can replace). shared ≈ full − Σ loss is the part the families
# carry jointly / redundantly. F1 is taken ABOVE the chance mean (an "alone" bar starts at chance, not
# at 0). ⚠️ Out-of-sample scores are NOT exactly additive (each subset gets its own nested selection /
# ridge α, and F1 / R² are not sums over features), so "shared" is an approximate, descriptive split —
# §10.3 tests the overlap directly (pair removal, probes, correlations).
_crow = []
for _pr in PROTOCOLS:
    _full = next(r for r in ABL_ROWS if r['tier'] == '3-subfamily'
                 and r['set_name'] == _REF_INTERP and r['protocol'] == _pr)
    _sub = df_abl_subfamily.query('protocol == @_pr')
    for _lab, _fcol, _acol, _lcol, _base in (('F1 − chance mean', 'f1_macro', 'alone_f1', 'loss_f1',
                                               CHANCE_FLOOR_MEAN[_pr]),
                                              ('R² arousal', 'r2_aro', 'alone_r2_aro', 'loss_r2_aro', 0.0),
                                              ('R² valence', 'r2_val', 'alone_r2_val', 'loss_r2_val', 0.0)):
        _sum_alone = float((_sub[_acol] - _base).sum())
        _sum_loss = float(_sub[_lcol].sum(skipna=True))
        _fs = float(_full[_fcol]) - _base
        _crow.append(dict(protocol=_pr, metric=_lab, full=_fs, sum_alone=_sum_alone, sum_loss=_sum_loss,
                          shared=_fs - _sum_loss, n_families=int(len(_sub))))
df_abl_commonality = pd.DataFrame(_crow)
df_abl_commonality.to_csv(os.path.join(OUTPUT_DIR, 'ablation_subfamily_commonality.csv'), index=False)
print('\n📊 §10.2 — commonality summary (Σ alone · Σ removal loss · full · shared ≈ full − Σ loss)')
for _pr in PROTOCOLS:
    print(f'── {PROTOCOL_LABEL[_pr]} ──')
    print(df_abl_commonality.query('protocol == @_pr')[['metric', 'full', 'sum_alone', 'sum_loss', 'shared']]
          .round(3).to_string(index=False))
print('   ⚠️  out-of-sample scores are not exactly additive — "shared" is an approximate, descriptive split;')
print('       Σ alone ≫ full and Σ loss ≪ full = most of the signal is carried by several families at once')
print('       (redundancy) — or the removal losses are noise / dilution / a different NestedSelect choice per')
print('       subset. §10.3 separates these with pair removal, cross-family probes and correlations.')

# ── Tie back to §10.1's single cost-of-explainability number ────────────────
if 'CoE_arousal' in globals():
    print(f'\n  §10.1 reported the cost of explainability ({PRIMARY_PROTOCOL}, primary) as '
          f'ΔR arousal = {CoE_arousal:+.3f}, ΔR valence = {CoE_valence:+.3f}.')
    print('  The `loss_r_arousal` column above says which families that single number is '
          'made of:\n  a family with a large loss is where the explainable arousal signal '
          'actually lives.')

# ── The validity caveat, printed where the number is read ──────────────────
if 'Vibrato' in INTERP_FAMILIES:
    _vib = df_abl_subfamily.query('family == "Vibrato" and protocol == @PRIMARY_PROTOCOL').iloc[0]
    print(f'\n  ⚠️  VIBRATO ({PRIMARY_PROTOCOL}) — loss_f1={_vib["loss_f1"]:+.4f}, '
          f'alone_f1={_vib["alone_f1"]:.4f}.')
    print('      The descriptor was REPAIRED in Phase B (B8) and again in Phase B2 (B8b, after the manual')
    print('      audit): B2 plateau notes, 500 ms windows with a least-squares 4–8 Hz sinusoid fit and an')
    print('      explicit R² + half-extent test per window (so "no vibrato" is representable), extent')
    print('      split from coverage (= vibrato-window time / analysed-note time). A null here is')
    print(f'      therefore a MUSICAL result — at this N (= {len(eids)} excerpts) — with the remaining caveats:')
    print(f'      · CREPE resolves pitch at {CREPE_STEP_MS} ms steps (fs = {1000 / CREPE_STEP_MS:.0f} Hz), so'
          ' fast/shallow vibrato is under-resolved;')
    print(f'      · only notes ≥ {VIB_MIN_NOTE_MS} ms are analysed — vibrato on shorter notes is invisible;')
    print(f'      · declared median substitutions (§2.2): {VIB_N_SUBST_NONOTES} excerpt(s) without any '
          f'note ≥ {VIB_MIN_NOTE_MS} ms,')
    print(f'        {VIB_N_SUBST_NOVIB} excerpt(s) with notes but no vibrato window (rate only).')

if 'Timing (ref-free)' in INTERP_FAMILIES:
    # PRE-DECLARED reading (Phase B3, written before the B3 run): the B2 re-articulation splits followed
    # the vibrato and inflated the note-based timing features with expressiveness. Compare the B3 value
    # with the Phase B (0.417) and B2 (0.542) runs: near 0.54 → the timing signal is real; back near 0.42
    # → the B2 value was the vibrato artefact. Printed, never asserted; quote only the B3 value.
    _tim = df_abl_subfamily.query('family == "Timing (ref-free)" and protocol == @PRIMARY_PROTOCOL').iloc[0]
    _tv = float(_tim['alone_f1'])
    _near = 'B2 (0.542) — the timing signal is REAL' if abs(_tv - 0.542) < abs(_tv - 0.417) else \
            'Phase B (0.417) — the B2 value was the VIBRATO ARTEFACT of the re-articulation splits'
    print(f'\n  🔎 PRE-DECLARED timing reading ({PRIMARY_PROTOCOL}): Timing (ref-free) alone F1 = {_tv:.3f} '
          f'(Phase B 0.417 · B2 0.542; removal loss {float(_tim["loss_f1"]):+.3f}) → closer to {_near}.')
    print('      Record which in docs/thesis.md; quote only this (B3) value in the thesis.')


df_abl_family.to_csv(os.path.join(OUTPUT_DIR, 'ablation_subfamily.csv'), index=False)
df_abl_subfamily.to_csv(os.path.join(OUTPUT_DIR, 'ablation_subfamily_decomposition.csv'),
                        index=False)
_fam_deltas.to_csv(os.path.join(OUTPUT_DIR, 'ablation_subfamily_deltas.csv'), index=False)

# ── Figure · loss-when-removed vs alone, per family ────────────────────────
_fams = list(INTERP_FAMILIES)
# Phase C2: the empirical-null P95 of the full layer = its EXACT-width primary null (§5.3, Interpretable (27))
_full_pr = next(r for r in ABL_ROWS if r['tier'] == '3-subfamily' and r['set_name'] == _REF_INTERP
                and r['protocol'] == PRIMARY_PROTOCOL)
_emp_full = emp_null_ref(_full_pr['f1_macro'], PRIMARY_PROTOCOL, len(_interp_cols), f'Interpretable ({len(_interp_cols)})')
if not np.isfinite(_emp_full['p95']):
    print(f'ℹ️  no empirical null drawn under {PRIMARY_PROTOCOL!r} (§5.3) — line not drawn')
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5), sharey=False)
for ax, (_col_loss, _col_alone, _col_q, _lbl) in zip(
        axes, [('loss_f1', 'alone_f1', 'q_loss_f1', 'class F1-macro'),
               ('loss_r2_aro', 'alone_r2_aro', 'q_loss_r2_aro', 'arousal (out-of-sample R²)')]):
    sub = df_abl_subfamily.query('protocol == @PRIMARY_PROTOCOL').set_index('family').reindex(_fams)
    xs  = np.arange(len(_fams))
    ax.bar(xs - 0.2, sub[_col_loss], 0.38, label='loss when REMOVED (* = BH q < .05)',
           color='#b00020', edgecolor='#333', linewidth=0.5)
    ax.bar(xs + 0.2, sub[_col_alone], 0.38, label='score ALONE',
           color='#2a9d2a', edgecolor='#333', linewidth=0.5)
    # B3: mark the removal losses that survive BH-FDR within the tier
    for _xi, (_lv, _qv) in enumerate(zip(sub[_col_loss], sub[_col_q])):
        if np.isfinite(_qv) and _qv < 0.05:
            ax.text(_xi - 0.2, _lv, '*', ha='center', va='bottom' if _lv >= 0 else 'top',
                    fontsize=14, color='#b00020')
    if _col_alone == 'alone_f1':
        # B3: an "alone" F1 bar starts at 0, but chance is not 0 — draw the chance mean and strict P95
        ax.axhline(CHANCE_FLOOR_MEAN[PRIMARY_PROTOCOL], color='#555', ls='--', lw=1.0,
                   label=f'chance mean = {CHANCE_FLOOR_MEAN[PRIMARY_PROTOCOL]:.3f}')
        ax.axhline(CHANCE_FLOOR_P95[PRIMARY_PROTOCOL], color='#555', ls=':', lw=1.0,
                   label=f'chance P95 (strict) = {CHANCE_FLOOR_P95[PRIMARY_PROTOCOL]:.3f}')
        if np.isfinite(_emp_full['p95']):
            ax.axhline(_emp_full['p95'], color='#6a1b9a', ls='-.', lw=1.2,
                       label=f'empirical-null P95 of the full layer ({_emp_full["label"]}) = {_emp_full["p95"]:.3f}')
        # Phase C3: the null P95 the families ALONE are read against (one line per null width used)
        _alone_ref = {}
        for _nf in sorted(set(sub['n_features'].dropna().astype(int))):
            _e = emp_null_ref(np.nan, PRIMARY_PROTOCOL, _nf)
            if np.isfinite(_e['p95']):
                _alone_ref.setdefault(_e['null_d'], (_e['p95'], _e['n_draws']))
        for _d, (_p95, _nd) in sorted(_alone_ref.items()):
            ax.axhline(_p95, color='#6a1b9a', ls=':', lw=0.9, alpha=0.6,
                       label=f'null P95 d = {_d} ({_nd} draws; the families alone) = {_p95:.3f}')
    ax.axhline(0, color='#333', lw=0.8)
    ax.set_xticks(xs)
    ax.set_xticklabels(_fams, rotation=28, ha='right', fontsize=8)
    ax.set_title(_lbl, fontsize=10)
    ax.grid(axis='y', alpha=0.25)
axes[0].set_ylabel('score / Δscore')
axes[0].legend(fontsize=8)
fig.suptitle(f'§10.2 Sub-family ablation — decomposing the cost of explainability '
             f'({PRIMARY_PROTOCOL}, primary)\n'
             'a family with a large REMOVAL loss is where the explainable signal lives',
             fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_ablation_subfamily.png'), dpi=150, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════════════════════════════
# Master consolidation — every tier, one file
# ═══════════════════════════════════════════════════════════════════════════
df_ablation = abl_frame()
df_ablation.to_csv(os.path.join(OUTPUT_DIR, 'ablation_master.csv'), index=False)
print(f'\n📦 MASTER ABLATION TABLE — {len(df_ablation)} scored rows across '
      f'{df_ablation.tier.nunique()} tiers')
print(df_ablation.groupby(['tier', 'protocol']).size().rename('rows').to_string())
print('\n✅ §10.2 — ablation_subfamily.csv, ablation_subfamily_decomposition.csv,')
print('   ablation_subfamily_deltas.csv, ablation_subfamily_commonality.csv, ablation_master.csv,')
print('   fig_ablation_subfamily.png  (§10.3 adds tier 3-subfamily-pairs and re-exports ablation_master.csv)')

## Part 10 · Cost of explainability — §10.3 Family overlap diagnostics · pair removal · cross-family probes · correlations

**Phase C2:** the layer is the predictive one of §4.1 (`INTERP_PRED_COLS`, 27 features, 7 families — no
DTW-deviation), so the pair tier has 21 pairs; test (1) of the triangulation rule now requires I > `OV_I_TOL`
(0.02), matching the reading band (C2 obligation 5b).

**Why (Phase B3).** In the full B2 run (`piece`) the full interpretable layer (30 features, 8 families)
gave F1 0.449 and R² arousal 0.627, but the family-alone arousal R² summed to ≈ 1.55 and the removal
losses to ≈ 0.23: most of the arousal signal is **shared** between families. "Small removal loss + high
alone score" can mean several different things — **redundancy** (another family carries the same
signal), **noise** (only 2 of 24 removal losses had q < 0.05), **dilution** (Timing alone 0.542 beat all 30
features at 0.449) or simply **a different NestedSelect choice per subset**. This section separates them
with three independent lines of evidence and one pre-declared triangulation rule. Descriptive only — no
subset is selected.

**C1 · pair removal** (tier `3-subfamily-pairs`, the existing `ablate` / `abl_vs_ref` machinery; the
interpretable reference is re-scored inside the tier so q is BH-FDR **within the tier**). Each of the
family pairs is removed together; with loss(X) = full − score without X (loss(A), loss(B) from the §10.2
tier of the same run) the **interaction** is I(A, B) = loss(A ∪ B) − loss(A) − loss(B), for F1, R² arousal
and R² valence: **I > 0** → A and B substitute for each other (each one's loss is small because the other
covers it); **I ≈ 0** (|I| < 0.02, the §6.3 noise band) → each contributes something of its own;
**I < 0** → they work only together. I is interpreted only when the pair's own loss has q < 0.05 (else
"n.s."). Protocol: `piece` always; `RUN_PAIR_ABLATION_LOO = False` (set True to add LOOCV — `ablate` already
takes a `protocols` argument, so no engine change was needed). The tier's runtime is printed and its rows
reach `ablation_master.csv` (re-exported here).

**C2 · probe per family (label-free).** Out-of-sample R² of predicting each feature of family A from
(i) all features outside A ("rest") and (ii) each family B alone (8 × 8 A ← B matrix; the diagonal uses
A's other features), with the §5.1 nested ridge under `piece` (closed-form `NestedRidgeOperator`, asserted
equal to `nested_ridge_predict`); the predictor scaler is fitted inside every fold, and z-scoring the
target would not change R² (ridge with intercept, SSE-based α choice). Two versions: **RAW**, and
**WITHIN-PIECE** — every feature minus its piece mean over the piece's three renditions. The
within-piece version is label-free: it uses the held-out piece's own **feature** mean (audio-derived
design structure), never a rating.

**C3 · correlation heatmaps.** Spearman ρ, feature × feature (27 × 27), ordered by family with family outlines, RAW and
WITHIN-PIECE, plus an 8 × 8 family summary of the mean |ρ| (within a family: the mean over its
off-diagonal pairs). Raw correlations are dominated by the **repertoire** (register, the tempo of the
score); within-piece correlations show how the **techniques co-vary from MEC to EXG**.

**C4 · PRE-DECLARED TRIANGULATION RULE** (printed before the results): families A and B are claimed to
overlap **only if at least 2 of 3** agree — (1) I(A, B) > `OV_I_TOL` (0.02) with the pair's loss q < 0.05 (any of F1 /
R²a / R²v); (2) within-piece probe R²(A ← B) or R²(B ← A) ≥ 0.30; (3) within-piece family mean |ρ| ≥ 0.40.
The cell lists the pairs that pass and the pairs that pass only one test.

**C5 · outputs** (`OUTPUT_DIR`): `overlap_pairs.csv` (I, q, interpretation, the three tests),
`overlap_probe.csv` (+ `overlap_probe_family.csv`), `overlap_corr.csv` (+ `overlap_corr_family.csv`),
`fig_overlap_pairs_interaction.png`, `fig_overlap_probe.png`, `fig_overlap_corr.png`.


In [ ]:
# ── §10.3 Family overlap diagnostics — pair removal · cross-family probes · correlations ──
# NEEDS : ablate, abl_frame, abl_vs_ref, ABL_ROWS, df_abl_subfamily, eids, INTERP_FAMILIES,
#          INTERP_FEAT_LABELS, MG_X_INTERP (§7.1 — the 27-column predictive layer), nested_ridge_predict, NestedRidgeOperator, oos_r2,
#          OUTPUT_DIR, PIECE_GROUPS, PRIMARY_PROTOCOL, PROTOCOLS, PROTOCOL_LABEL, _interp_cols,
#          _REF_INTERP, _X_fam (§10.2)
# MAKES : RUN_PAIR_ABLATION_LOO, OV_I_TOL, OV_PROBE_MIN, OV_RHO_MIN, df_ov_pairs, df_ov_probe,
#          df_ov_probe_fam, df_ov_corr, df_ov_corr_fam, df_ablation (re-exported with the pair tier),
#          _ov_* · overlap_pairs.csv, overlap_probe.csv, overlap_probe_family.csv, overlap_corr.csv,
#          overlap_corr_family.csv, ablation_master.csv (re-export), 3 figures
# FEEDS : the report's family-overlap reading (Part 10); ablation_master.csv
# ═══════════════════════════════════════════════════════════════════════════════
# §10.3 · Do the expressive families overlap? (Phase B3, descriptive)
# ═══════════════════════════════════════════════════════════════════════════════
# C1 reads Y to SCORE (through ablate, the §6.3 engine); C2 / C3 are label-free (features only — the
# within-piece version subtracts each piece's own FEATURE mean, never a rating). Every CV split comes
# from §5.1 (ablate → cv_predict_default / nested_ridge_predict; NestedRidgeOperator). Nothing is selected.
import time as _ov_time
from itertools import combinations as _ov_combos

RUN_PAIR_ABLATION_LOO = False   # True → the pair tier is also scored under LOOCV (≈ 3× the runtime)
OV_I_TOL     = 0.02             # |I| below this = "≈ 0" (the §6.3 noise band at N = 60)
OV_PROBE_MIN = 0.30             # triangulation test 2: within-piece probe R² threshold
OV_RHO_MIN   = 0.40             # triangulation test 3: within-piece family mean |ρ| threshold

# The interpretable matrix of §7.1 must be the §10.2 matrix (same construction, byte for byte)
assert np.array_equal(MG_X_INTERP, _X_fam(_interp_cols)), \
    '§10.3: the §7.1 interpretable layer differs from the §10.2 _X_fam(_interp_cols) matrix'
_ov_fams = list(INTERP_FAMILIES)
_ov_pairs = list(_ov_combos(_ov_fams, 2))
_ov_prot = tuple(p for p in PROTOCOLS if p == PRIMARY_PROTOCOL or RUN_PAIR_ABLATION_LOO)

print('§10.3 · PRE-DECLARED TRIANGULATION RULE — families A and B overlap only if ≥ 2 of 3 agree:')
print(f'   (1) I(A,B) = loss(A∪B) − loss(A) − loss(B) > OV_I_TOL = {OV_I_TOL:g} with the pair\'s own loss q < 0.05 '
      '(F1, R²a or R²v) — Phase C2: the same band as the reading "substitutes (I > 0.02)"')
print(f'   (2) within-piece probe R²(A←B) or R²(B←A) ≥ {OV_PROBE_MIN:g}')
print(f'   (3) within-piece family mean |ρ| ≥ {OV_RHO_MIN:g}')
print(f'   (I is read only when the pair loss has q < 0.05; |I| < {OV_I_TOL:g} = "≈ 0": each contributes its own; '
      'I > 0: substitutes; I < 0: only together)\n')

# ── C1 · pair removal (tier 3-subfamily-pairs) ─────────────────────────────────
_ov_t0 = _ov_time.perf_counter()
print(f'📊 TIER 3-subfamily-pairs · {len(_ov_pairs)} family pairs removed together from the '
      f'{len(_interp_cols)}-feature layer · protocols {_ov_prot}')
ablate(_X_fam(_interp_cols), _REF_INTERP, tier='3-subfamily-pairs', protocols=_ov_prot,
       note='reference (re-scored inside the tier so q is BH-FDR within the tier)')
_ov_pname = {}
for _A, _B in _ov_pairs:
    _rest = [c for c in _interp_cols if c not in INTERP_FAMILIES[_A] and c not in INTERP_FAMILIES[_B]]
    if not _rest:
        print(f'   ➖ {_A} + {_B}: nothing left — skipped')
        continue
    _nm = f'− {_A} − {_B} ({len(_rest)}d)'
    ablate(_X_fam(_rest), _nm, tier='3-subfamily-pairs', protocols=_ov_prot,
           note=f'pair removal: {_A} + {_B}')
    _ov_pname[(_A, _B)] = _nm
_ov_pd = abl_vs_ref('3-subfamily-pairs', _REF_INTERP, protocols=_ov_prot)
_ov_runtime = _ov_time.perf_counter() - _ov_t0
print(f'   ⏱  tier 3-subfamily-pairs: {len(_ov_pname) + 1} sets × {len(_ov_prot)} protocol(s) in '
      f'{_ov_runtime / 60:.1f} min')


def _ov_row(tier, name, pr):
    return next(r for r in ABL_ROWS if r['tier'] == tier and r['set_name'] == name and r['protocol'] == pr)


_OV_MET = (('f1', 'f1_macro', 'loss_f1', 'q_f1'), ('r2_aro', 'r2_aro', 'loss_r2_aro', 'q_r2_aro'),
           ('r2_val', 'r2_val', 'loss_r2_val', 'q_r2_val'))
_ov_prow = []
for _pr in _ov_prot:
    _full = _ov_row('3-subfamily-pairs', _REF_INTERP, _pr)
    _full56 = _ov_row('3-subfamily', _REF_INTERP, _pr)
    assert (abs(_full['f1_macro'] - _full56['f1_macro']) < 1e-12
            and np.allclose([_full['r2_aro'], _full['r2_val']], [_full56['r2_aro'], _full56['r2_val']],
                            rtol=0, atol=1e-9)), \
        f'§10.3: the pair-tier reference ({_pr}) does not reproduce the §10.2 full interpretable row'
    _sf = df_abl_subfamily.query('protocol == @_pr').set_index('family')
    for (_A, _B), _nm in _ov_pname.items():
        _drop = _ov_row('3-subfamily-pairs', _nm, _pr)
        _d = _ov_pd.query('protocol == @_pr and set_name == @_nm')
        _rec = dict(protocol=_pr, family_a=_A, family_b=_B, set_name=_nm,
                    n_features_left=int(_drop['n_features']))
        for _k, _sc, _lc, _qc in _OV_MET:
            _lab = float(_full[_sc]) - float(_drop[_sc])
            _I = _lab - float(_sf.loc[_A, _lc]) - float(_sf.loc[_B, _lc])
            _q = float(_d[_qc].iloc[0]) if len(_d) else np.nan
            _rec.update({f'loss_pair_{_k}': _lab, f'loss_a_{_k}': float(_sf.loc[_A, _lc]),
                         f'loss_b_{_k}': float(_sf.loc[_B, _lc]), f'I_{_k}': _I, f'q_pair_{_k}': _q,
                         f'reading_{_k}': ('n.s.' if not (np.isfinite(_q) and _q < 0.05) else
                                           'substitutes (I > 0)' if _I > OV_I_TOL else
                                           'only together (I < 0)' if _I < -OV_I_TOL else
                                           'each its own (I ≈ 0)')})
        _ov_prow.append(_rec)
df_ov_pairs = pd.DataFrame(_ov_prow)
for _pr in _ov_prot:
    print(f'\n── {PROTOCOL_LABEL[_pr]} — interaction I(A,B) per target (read only where the pair loss q < 0.05) ──')
    with pd.option_context('display.width', 250):
        print(df_ov_pairs.query('protocol == @_pr')[
            ['family_a', 'family_b', 'loss_pair_f1', 'I_f1', 'q_pair_f1', 'reading_f1',
             'loss_pair_r2_aro', 'I_r2_aro', 'q_pair_r2_aro', 'reading_r2_aro',
             'I_r2_val', 'q_pair_r2_val', 'reading_r2_val']].round(3).to_string(index=False))

# heatmaps of I (primary protocol), one per target; n.s. cells greyed
_ov_n = len(_ov_fams)
fig, axes = plt.subplots(1, 3, figsize=(20, 6.4))
for _ax, (_k, _title) in zip(axes, (('f1', 'class F1-macro'), ('r2_aro', 'R² arousal'), ('r2_val', 'R² valence'))):
    _M = np.full((_ov_n, _ov_n), np.nan)
    _S = np.zeros((_ov_n, _ov_n), dtype=bool)
    for _r in df_ov_pairs.query('protocol == @PRIMARY_PROTOCOL').itertuples():
        _i, _j = _ov_fams.index(_r.family_a), _ov_fams.index(_r.family_b)
        _M[_i, _j] = _M[_j, _i] = getattr(_r, f'I_{_k}')
        _S[_i, _j] = _S[_j, _i] = getattr(_r, f'reading_{_k}') != 'n.s.'
    _lim = float(np.nanmax(np.abs(_M))) if np.isfinite(_M).any() else 1.0
    _im = _ax.imshow(np.where(_S, _M, np.nan), cmap='RdBu_r', vmin=-_lim, vmax=_lim)
    _ax.imshow(np.where(~_S & np.isfinite(_M), 1.0, np.nan), cmap='Greys', vmin=0, vmax=4, alpha=0.6)
    for _i in range(_ov_n):
        for _j in range(_ov_n):
            if np.isfinite(_M[_i, _j]):
                _ax.text(_j, _i, f'{_M[_i, _j]:+.2f}' + ('' if _S[_i, _j] else '\nn.s.'), ha='center',
                         va='center', fontsize=6.5)
    _ax.set_xticks(range(_ov_n)); _ax.set_xticklabels(_ov_fams, rotation=40, ha='right', fontsize=7.5)
    _ax.set_yticks(range(_ov_n)); _ax.set_yticklabels(_ov_fams, fontsize=7.5)
    _ax.set_title(f'I(A,B) — {_title}', fontsize=10)
    plt.colorbar(_im, ax=_ax, fraction=0.046, pad=0.04)
fig.suptitle(f'§10.3 C1 — pair-removal interaction I(A,B) = loss(A∪B) − loss(A) − loss(B) ({PRIMARY_PROTOCOL}) · '
             'red = substitutes · blue = only together · grey = pair loss n.s. (q ≥ .05)', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_overlap_pairs_interaction.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── C2 · cross-family probes (label-free; RAW and WITHIN-PIECE) ───────────────
_ov_cols = [c for f in _ov_fams for c in INTERP_FAMILIES[f]]          # ordered by family
_ov_fam_of = {c: f for f in _ov_fams for c in INTERP_FAMILIES[f]}
_ov_X = pd.DataFrame(_X_fam(_interp_cols), index=eids, columns=_interp_cols)[_ov_cols]
_ov_versions = {'raw': _ov_X,
                'within-piece': _ov_X - _ov_X.groupby(np.asarray(PIECE_GROUPS)).transform('mean')}
_ov_rows, _ov_checked = [], set()
for _ver, _F in _ov_versions.items():
    _ops = {}

    def _ov_op(cols, _F=_F, _ops=_ops):
        k = tuple(cols)
        if k not in _ops:
            _ops[k] = NestedRidgeOperator(_F[list(cols)].to_numpy(dtype=float), PRIMARY_PROTOCOL)
        return _ops[k]

    for _c in _ov_cols:
        _A = _ov_fam_of[_c]
        _y = _F[_c].to_numpy(dtype=float)
        _preds = [('rest (all other families)', [x for x in _ov_cols if _ov_fam_of[x] != _A])]
        _preds += [(_B, [x for x in INTERP_FAMILIES[_B] if x != _c]) for _B in _ov_fams]
        for _pn, _pc in _preds:
            if not _pc or np.std(_y) < 1e-12:
                _r2 = np.nan
            else:
                _yh, _al = _ov_op(_pc)(_y)
                if _ver not in _ov_checked:      # closed form == the sklearn nested ridge, once per version
                    _yh2, _al2 = nested_ridge_predict(_F[_pc].to_numpy(dtype=float), _y, PRIMARY_PROTOCOL)
                    assert np.array_equal(_al, _al2) and np.allclose(_yh, _yh2, rtol=0, atol=1e-8), \
                        f'§10.3: NestedRidgeOperator ≠ nested_ridge_predict ({_ver}, {_c} ← {_pn})'
                    _ov_checked.add(_ver)
                _r2 = oos_r2(_y, _yh, PRIMARY_PROTOCOL)
            _ov_rows.append(dict(version=_ver, target=_c, target_label=INTERP_FEAT_LABELS.get(_c, _c),
                                 target_family=_A, predictor=_pn, n_predictors=len(_pc), r2=_r2))
df_ov_probe = pd.DataFrame(_ov_rows)
df_ov_probe_fam = (df_ov_probe.groupby(['version', 'target_family', 'predictor'], sort=False)['r2']
                   .mean().reset_index())
df_ov_probe.to_csv(os.path.join(OUTPUT_DIR, 'overlap_probe.csv'), index=False)
df_ov_probe_fam.to_csv(os.path.join(OUTPUT_DIR, 'overlap_probe_family.csv'), index=False)
print(f'\n✅ C2 · NestedRidgeOperator == nested_ridge_predict asserted once per version ({PRIMARY_PROTOCOL})')
_ov_pcols = ['rest (all other families)'] + _ov_fams
_ov_PM = {}
for _ver in _ov_versions:
    _ov_PM[_ver] = (df_ov_probe_fam.query('version == @_ver')
                    .pivot(index='target_family', columns='predictor', values='r2')
                    .reindex(index=_ov_fams, columns=_ov_pcols))
    print(f'\n📊 C2 · probe R² (out-of-sample, {PRIMARY_PROTOCOL}) — {_ver.upper()} — rows = family A predicted '
          '(mean over its features), columns = predictor (diagonal = A\'s other features)')
    with pd.option_context('display.width', 250):
        print(_ov_PM[_ver].round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(19, 6.6))
for _ax, _ver in zip(axes, _ov_versions):
    _M = _ov_PM[_ver].to_numpy(dtype=float)
    _im = _ax.imshow(_M, cmap='viridis', vmin=min(0.0, float(np.nanmin(_M)) if np.isfinite(_M).any() else 0.0),
                     vmax=1.0)
    for _i in range(_M.shape[0]):
        for _j in range(_M.shape[1]):
            if np.isfinite(_M[_i, _j]):
                _ax.text(_j, _i, f'{_M[_i, _j]:.2f}', ha='center', va='center', fontsize=6.5,
                         color='white' if _M[_i, _j] < 0.5 else 'black')
    _ax.set_xticks(range(len(_ov_pcols))); _ax.set_xticklabels(_ov_pcols, rotation=40, ha='right', fontsize=7.5)
    _ax.set_yticks(range(len(_ov_fams))); _ax.set_yticklabels(_ov_fams, fontsize=7.5)
    _ax.set_title(f'{_ver}: R²(A ← predictor), mean over A\'s features', fontsize=10)
    plt.colorbar(_im, ax=_ax, fraction=0.046, pad=0.04)
fig.suptitle(f'§10.3 C2 — cross-family probes (nested ridge, {PRIMARY_PROTOCOL}, label-free) · within-piece = '
             'features minus their piece mean', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_overlap_probe.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── C3 · Spearman correlations (RAW and WITHIN-PIECE), ordered by family ───────
_ov_R, _ov_crow, _ov_frow = {}, [], []
for _ver, _F in _ov_versions.items():
    _R = _F.corr(method='spearman')
    _ov_R[_ver] = _R
    for _i, _a in enumerate(_ov_cols):
        for _b in _ov_cols[_i + 1:]:
            _ov_crow.append(dict(version=_ver, feature_a=_a, feature_b=_b, family_a=_ov_fam_of[_a],
                                 family_b=_ov_fam_of[_b], rho=float(_R.loc[_a, _b])))
    for _A in _ov_fams:
        for _B in _ov_fams:
            _ca, _cb = INTERP_FAMILIES[_A], INTERP_FAMILIES[_B]
            if _A == _B:
                _v = [abs(_R.loc[_ca[i], _ca[j]]) for i in range(len(_ca)) for j in range(i + 1, len(_ca))]
            else:
                _v = [abs(_R.loc[x, z]) for x in _ca for z in _cb]
            _v = [x for x in _v if np.isfinite(x)]
            _ov_frow.append(dict(version=_ver, family_a=_A, family_b=_B,
                                 mean_abs_rho=float(np.mean(_v)) if _v else np.nan))
df_ov_corr = pd.DataFrame(_ov_crow)
df_ov_corr_fam = pd.DataFrame(_ov_frow)
df_ov_corr.to_csv(os.path.join(OUTPUT_DIR, 'overlap_corr.csv'), index=False)
df_ov_corr_fam.to_csv(os.path.join(OUTPUT_DIR, 'overlap_corr_family.csv'), index=False)
_ov_CF = {v: df_ov_corr_fam.query('version == @v').pivot(index='family_a', columns='family_b',
                                                         values='mean_abs_rho').reindex(index=_ov_fams,
                                                                                         columns=_ov_fams)
          for v in _ov_versions}
for _ver in _ov_versions:
    print(f'\n📊 C3 · family mean |Spearman ρ| — {_ver.upper()} (diagonal = within-family off-diagonal pairs)')
    with pd.option_context('display.width', 250):
        print(_ov_CF[_ver].round(2).to_string())
print('   raw correlations are dominated by the repertoire (register, the tempo of the score); within-piece '
      'correlations show how the techniques co-vary from MEC to EXG.')

fig, axes = plt.subplots(2, 2, figsize=(20, 19), gridspec_kw={'height_ratios': [2.3, 1.0]})
_ov_edges = np.cumsum([0] + [len(INTERP_FAMILIES[f]) for f in _ov_fams])
_ov_lbl = [INTERP_FEAT_LABELS.get(c, c) for c in _ov_cols]
for _ax, _ver in zip(axes[0], _ov_versions):
    _im = _ax.imshow(_ov_R[_ver].to_numpy(dtype=float), cmap='RdBu_r', vmin=-1, vmax=1)
    for _k in range(len(_ov_fams)):
        _ax.add_patch(plt.Rectangle((_ov_edges[_k] - 0.5, _ov_edges[_k] - 0.5),
                                    _ov_edges[_k + 1] - _ov_edges[_k], _ov_edges[_k + 1] - _ov_edges[_k],
                                    fill=False, edgecolor='black', lw=1.6))
    _ax.set_xticks(range(len(_ov_cols))); _ax.set_xticklabels(_ov_lbl, rotation=90, fontsize=6.5)
    _ax.set_yticks(range(len(_ov_cols))); _ax.set_yticklabels(_ov_lbl, fontsize=6.5)
    _ax.set_title(f'Spearman ρ — {_ver} (boxes = families)', fontsize=10)
    plt.colorbar(_im, ax=_ax, fraction=0.046, pad=0.04)
for _ax, _ver in zip(axes[1], _ov_versions):
    _M = _ov_CF[_ver].to_numpy(dtype=float)
    _im = _ax.imshow(_M, cmap='magma_r', vmin=0, vmax=1)
    for _i in range(_ov_n):
        for _j in range(_ov_n):
            if np.isfinite(_M[_i, _j]):
                _ax.text(_j, _i, f'{_M[_i, _j]:.2f}', ha='center', va='center', fontsize=7,
                         color='white' if _M[_i, _j] > 0.5 else 'black')
    _ax.set_xticks(range(_ov_n)); _ax.set_xticklabels(_ov_fams, rotation=40, ha='right', fontsize=7.5)
    _ax.set_yticks(range(_ov_n)); _ax.set_yticklabels(_ov_fams, fontsize=7.5)
    _ax.set_title(f'family mean |ρ| — {_ver}', fontsize=10)
    plt.colorbar(_im, ax=_ax, fraction=0.046, pad=0.04)
fig.suptitle('§10.3 C3 — feature correlations ordered by family · raw = repertoire-dominated · '
             'within-piece = how techniques co-vary from MEC to EXG', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_overlap_corr.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── C4 · triangulation (pre-declared rule, primary protocol) ───────────────────
_ov_wpP, _ov_wpC = _ov_PM['within-piece'], _ov_CF['within-piece']
_ov_t = []
for _r in df_ov_pairs.query('protocol == @PRIMARY_PROTOCOL').itertuples():
    _A, _B = _r.family_a, _r.family_b
    _t1 = any(np.isfinite(getattr(_r, f'q_pair_{k}')) and getattr(_r, f'q_pair_{k}') < 0.05
              and getattr(_r, f'I_{k}') > OV_I_TOL for k, *_ in _OV_MET)
    _p = np.nanmax([_ov_wpP.loc[_A, _B], _ov_wpP.loc[_B, _A]]) \
        if np.isfinite([_ov_wpP.loc[_A, _B], _ov_wpP.loc[_B, _A]]).any() else np.nan
    _t2 = bool(np.isfinite(_p) and _p >= OV_PROBE_MIN)
    _rho = float(_ov_wpC.loc[_A, _B])
    _t3 = bool(np.isfinite(_rho) and _rho >= OV_RHO_MIN)
    _ov_t.append(dict(family_a=_A, family_b=_B, test1_interaction=bool(_t1), test2_probe=_t2,
                      wp_probe_r2_max=_p, test3_corr=_t3, wp_mean_abs_rho=_rho,
                      n_tests_passed=int(_t1) + int(_t2) + int(_t3),
                      overlap_claimed=bool(int(_t1) + int(_t2) + int(_t3) >= 2)))
_ov_tri = pd.DataFrame(_ov_t)
df_ov_pairs = df_ov_pairs.merge(_ov_tri.assign(protocol=PRIMARY_PROTOCOL), how='left',
                                on=['protocol', 'family_a', 'family_b'])
df_ov_pairs.to_csv(os.path.join(OUTPUT_DIR, 'overlap_pairs.csv'), index=False)
print(f'\n🔎 C4 · TRIANGULATION ({PRIMARY_PROTOCOL}; ≥ 2 of 3 tests → overlap claimed)')
_ov_yes = _ov_tri[_ov_tri.overlap_claimed]
_ov_one = _ov_tri[_ov_tri.n_tests_passed == 1]
print(f'   pairs that OVERLAP ({len(_ov_yes)}): '
      + ('; '.join(f'{r.family_a} ↔ {r.family_b} ({"1" if r.test1_interaction else ""}'
                   f'{"2" if r.test2_probe else ""}{"3" if r.test3_corr else ""})' for r in _ov_yes.itertuples())
         if len(_ov_yes) else 'none'))
print(f'   pairs passing ONLY ONE test ({len(_ov_one)}): '
      + ('; '.join(f'{r.family_a} ↔ {r.family_b} (test '
                   f'{"1" if r.test1_interaction else "2" if r.test2_probe else "3"})' for r in _ov_one.itertuples())
         if len(_ov_one) else 'none'))
with pd.option_context('display.width', 250):
    print(_ov_tri[['family_a', 'family_b', 'test1_interaction', 'test2_probe', 'wp_probe_r2_max', 'test3_corr',
                   'wp_mean_abs_rho', 'n_tests_passed', 'overlap_claimed']].round(3).to_string(index=False))

# ── C5 · the pair tier joins the master ablation table ─────────────────────────
df_ablation = abl_frame()
df_ablation.to_csv(os.path.join(OUTPUT_DIR, 'ablation_master.csv'), index=False)
print(f'\n📦 MASTER ABLATION TABLE re-exported — {len(df_ablation)} scored rows across '
      f'{df_ablation.tier.nunique()} tiers (incl. 3-subfamily-pairs)')
print(f'✅ §10.3 — overlap_pairs.csv, overlap_probe.csv, overlap_probe_family.csv, overlap_corr.csv, '
      'overlap_corr_family.csv, fig_overlap_pairs_interaction.png, fig_overlap_probe.png, fig_overlap_corr.png, '
      'ablation_master.csv')


## Part 11 · Synthesis — §11.1 Claims table + save all outputs
**Question of this part:** what is the single defensible summary of everything above?  
**Hands on:** `claims.csv` (`df_claims` — every claim with its value read from a variable, its comparator and a
verdict ✔ / ◐ / ✘ from a rule stated next to it), `final_summary.csv`, `all_predictions.csv` and the
output-file reference below.



In [ ]:
# ── §11.1 Claims table + save all outputs ───────────────────────────
# NEEDS : all_recs, CHANCE_FLOOR_MEAN, CHANCE_FLOOR_P95, COE (§10.1), DEFAULT_CLF_NAME, DEFAULT_RECS, DEFAULT_SET,
#          df_abl_subfamily (§10.2), df_abl_tempogram_deltas (§6.3), df_delta_tests (§9.1), df_emp_null_primary,
#          df_emp_null_readout (§6.9 / §7.1 / §7.2), df_expr, df_mert_layer_a (§7.2), df_mg_nn, MG_NN_FOLD,
#          MG_VERDICT (§7.1), df_sh (§7.4), df_within_piece (§8.1), FEATURE_SETS, INTERP_PRED_COLS, MANIP_CHECK
#          (Part 4), mlb, NB_CLF_NAME, NB_RECS, OUTPUT_DIR, PRED_QUADS, PRED_QUADS_INTERP (§6.8), PRIMARY_PROTOCOL,
#          PROTOCOLS, REG_INTERP, REG_MERTFULL, REG_RES (§6.2)
# MAKES : df_claims, df_final · claims.csv, final_summary.csv, all_predictions.csv, analysis_d_shifts.csv
# FEEDS : — (terminal: writes files only)
# ═══════════════════════════════════════════════════════════════════════════════
# §11.1 · The claims table — every value read from a variable, every verdict from a stated rule
# ═══════════════════════════════════════════════════════════════════════════════
# Verdicts: ✔ = supported · ◐ = partly / with a stated caveat · ✘ = not supported. Each rule is written in the
# comment next to its row and was fixed before the C2 run (except claim 10, revised post hoc in Phase C3 — see its
# comment); nothing is typed in by hand.
_pr = PRIMARY_PROTOCOL
_claims = []


def _claim(claim, section, value, comparator, verdict, rule):
    _claims.append(dict(claim=claim, section=section, value=value, comparator=comparator, verdict=verdict, rule=rule))


# 1 · RULE: ✔ ≥ 3 techniques with q < .05 that are NOT flagged ⚑ · ◐ 1–2 unflagged, or only flagged ones · ✘ none
_m = MANIP_CHECK
_v = ('✔' if _m['n_techniques_q05_unflagged'] >= 3 else
      '◐' if _m['n_techniques_q05'] >= 1 else '✘')
_claim('Performers escalated the techniques (MEC < EXP < EXG)', '§4.2',
       f'{_m["n_techniques_q05"]} of {_m["n_techniques"]} techniques q < .05 ({_m["n_techniques_q05_unflagged"]} unflagged)',
       "Page's trend test, BH over the techniques + 2 ratings", _v,
       '✔ ≥ 3 unflagged techniques q < .05 · ◐ 1–2 or only flagged · ✘ none')

# 2 · RULE: ✔ arousal Page q < .05 AND valence Page q ≥ .05 · ◐ arousal q < .05 but valence also q < .05 · ✘ arousal q ≥ .05
_aq, _vq = _m['arousal_page_q'], _m['valence_page_q']
_v = '✘' if not (_aq < 0.05) else ('✔' if not (_vq < 0.05) else '◐')
_claim("Listeners' arousal changed, valence did not", '§4.3',
       f'arousal z {_m["arousal_page_z"]:+.2f} (q {_aq:.4f}) · valence z {_m["valence_page_z"]:+.2f} (q {_vq:.4f}) · '
       f'Friedman p arousal {_m["arousal_friedman_p"]:.4f} / valence {_m["valence_friedman_p"]:.4f}',
       "Page's trend test on the ratings (same BH family as §4.2)", _v,
       '✔ arousal q < .05 and valence q ≥ .05 · ◐ both q < .05 · ✘ arousal q ≥ .05')

# 3 / 4 · RULE: ✔ out-of-sample R² (piece) ≥ 0.10 · ◐ 0 < R² < 0.10 · ✘ R² ≤ 0 (0 = the training-fold-mean baseline)
for _dim, _k, _n in (('arousal', 'ar2', 3), ('valence', 'vr2', 4)):
    _r2 = float(REG_RES[_pr][REG_INTERP][_k])
    _claim(('Interpretable features predict arousal in unseen pieces' if _dim == 'arousal' else
            '… and valence in unseen pieces'), '§6.2',
           f'R² {_dim} = {_r2:+.3f} ({REG_INTERP}, {_pr})', 'out-of-sample R² vs the training-fold mean (0)',
           '✔' if _r2 >= 0.10 else '◐' if _r2 > 0 else '✘', '✔ R² ≥ 0.10 · ◐ 0 < R² < 0.10 · ✘ R² ≤ 0')

# 5 · RULE: ✔ ≥ 4 of the 7 pre-declared primary sets beat their exact-width null after BH (q < .05) · ◐ 1–3 · ✘ none
_pp = df_emp_null_primary[df_emp_null_primary.protocol == _pr]
_nq = int((_pp['q_primary'] < 0.05).sum())
_claim('Emotion-class classification generalises to unseen pieces', '§6.9',
       f'{_nq} of {len(_pp)} primary sets q < .05: ' + (', '.join(_pp[_pp.q_primary < 0.05]['primary_set']) or 'none')
       + ' (null draws per width: ' + ', '.join(f'd={int(_d)}: {int(_n)}' for _d, _n in
                                                 _pp.drop_duplicates('d').sort_values('d')[['d', 'null_n_draws']].to_numpy())
       + ')',
       'exact-width empirical null of the deployed pipeline, BH within the primary family',
       '✔' if _nq >= 4 else '◐' if _nq >= 1 else '✘', '✔ ≥ 4 of 7 primary sets q < .05 · ◐ 1–3 · ✘ none')

# 6 · RULE: ✔ the pre-declared §7.1 piece-identity rule fired AND every §7.2 candidate's same-piece NN share
#          ≥ MG_NN_FOLD × chance · ◐ only one of the two · ✘ neither
_nn = df_mg_nn.set_index('space')
_mf = float(_nn.loc[[s for s in _nn.index if s.startswith('MERT-full')][0], 'same_piece_share'])
_ch = float(_nn['same_piece_chance'].iloc[0])
_a1 = df_mert_layer_a.loc[~df_mert_layer_a['is_reference'].astype(bool), 'a1_same_piece_share']
_a1_ok = bool(len(_a1) and float(_a1.min()) >= MG_NN_FOLD * _ch)
_n_ok = int(bool(MG_VERDICT['piece_identity_shortcut'])) + int(_a1_ok)
_claim('MERT encodes piece identity', '§7.1 · §7.2',
       f'MERT-full same-piece NN {100 * _mf:.1f} % (chance {100 * _ch:.1f} %) · §7.2 A1 range '
       + (f'{100 * float(_a1.min()):.1f}–{100 * float(_a1.max()):.1f} %' if len(_a1) else 'n/a')
       + f' · §7.1 rule {"fired" if MG_VERDICT["piece_identity_shortcut"] else "did not fire"}',
       f'same-piece nearest neighbour vs chance (≥ {MG_NN_FOLD:g} × chance)',
       '✔' if _n_ok == 2 else '◐' if _n_ok == 1 else '✘',
       '✔ §7.1 rule fired AND every layer ≥ 3 × chance · ◐ one of the two · ✘ neither')

# 7 · RULE: ✔ all four regression gaps (black box − interpretable: r_val, r_aro, R²_val, R²_aro) < 0 under piece ·
#          ◐ some < 0 · ✘ none (negative = the interpretable layer wins)
_g = COE[_pr]
_neg = [k for k in ('r_valence', 'r_arousal', 'r2_val', 'r2_aro') if _g[k] < 0]
_claim('Interpretable beats MERT on unseen pieces (cost of explainability)', '§10.1',
       ' · '.join(f'Δ{k} {_g[k]:+.3f}' for k in ('r_valence', 'r_arousal', 'r2_val', 'r2_aro', 'f1_macro')),
       'max(MERT-PCA25, MERT-full 768) − Interpretable (F1: MERT-PCA25 − Interpretable); negative = interpretable wins',
       '✔' if len(_neg) == 4 else '◐' if _neg else '✘', '✔ all 4 regression gaps < 0 · ◐ some · ✘ none')

# 8 · RULE: ✔ Interpretable's incremental ΔR² (Condition + Features vs Condition-only) q < .05 for BOTH targets ·
#          ◐ one target · ✘ neither
_wi = df_within_piece[df_within_piece['feature_set'].astype(str).str.startswith('Interpretable (')]
_wv = {t: _wi[_wi['target'] == t].iloc[0] for t in ('valence', 'arousal') if (_wi['target'] == t).any()}
_nq8 = sum(bool(r['q_incremental'] < 0.05) for r in _wv.values())
_claim('Technique change predicts rating change within a piece (beyond the condition)', '§8.1',
       ' · '.join(f'{t}: ΔR² {r["d_r2_incremental"]:+.3f} (q {r["q_incremental"]:.3f})' for t, r in _wv.items()),
       'Freedman–Lane incremental test vs Condition-only, BH over the §8.1 tests',
       '✔' if _nq8 == 2 else '◐' if _nq8 == 1 else '✘', '✔ q < .05 for both targets · ◐ one · ✘ neither')

# 9 · RULE: ✔ ≥ 1 unflagged technique-delta test with q < .05 · ◐ only flagged ones · ✘ none
_dq = df_delta_tests[df_delta_tests['q'] < 0.05]
_dq_clean = _dq[_dq['flag'] == '']
_claim('A single technique drives a single emotion change', '§9.1',
       f'{len(_dq)} of {int(np.isfinite(df_delta_tests["p"]).sum())} tests q < .05 ({len(_dq_clean)} unflagged; '
       f'raw p < .05: {int((df_delta_tests["p"] < .05).sum())}, expected by chance '
       f'{0.05 * int(np.isfinite(df_delta_tests["p"]).sum()):.1f})',
       'condition-delta correlations, BH over all §9.1 tests',
       '✔' if len(_dq_clean) else '◐' if len(_dq) else '✘', '✔ ≥ 1 unflagged q < .05 · ◐ only flagged · ✘ none')

# 10 · RULE — revised in Phase C3 after the C2 run showed the arousal dependence on the tempogram — stated as a
#      post-hoc revision (the C2 rule checked only that the timing F1 survives removing the tempogram; §6.3 showed
#      R² arousal 0.32 → 0.09 for "DTW ref-free − tempogram (4d)", p .016, q .08):
#      ✔ Timing (ref-free) has the highest family-alone F1 (piece) AND beats its null (Phase C3 rule, alone_emp_p < .05)
#        AND the timing set without the tempogram (§6.3 tier 1b) keeps its F1 above its null (empirical p < .05, same
#        rule) AND loses < 0.10 of R² arousal (ΔR²a = without − with > −0.10)
#      ◐ highest family-alone F1, but one of the other conditions fails · ✘ not the highest
_sf = df_abl_subfamily[df_abl_subfamily.protocol == _pr].sort_values('alone_f1', ascending=False)
_top = _sf.iloc[0]
_tim = _sf[_sf.family == 'Timing (ref-free)']
_tg = (df_abl_tempogram_deltas.query('protocol == @_pr and ref == "DTW"') if len(df_abl_tempogram_deltas)
       else pd.DataFrame())
_tg_rd = (df_emp_null_readout[(df_emp_null_readout.source == '§6.3 tempogram sensitivity')
                              & (df_emp_null_readout.set_name == str(_tg['set_name'].iloc[0]))
                              & (df_emp_null_readout.protocol == _pr)] if len(_tg) else pd.DataFrame())
_tg_f1_ok = bool(len(_tg_rd) and float(_tg_rd['emp_p'].iloc[0]) < 0.05)
_tg_dr2a = float(_tg['d_r2_aro'].iloc[0]) if len(_tg) else float('nan')
_tg_r2_ok = bool(np.isfinite(_tg_dr2a) and _tg_dr2a > -0.10)
_t_first = bool(len(_tim) and _top['family'] == 'Timing (ref-free)')
_t_null = bool(len(_tim) and float(_tim['alone_emp_p'].iloc[0]) < 0.05)
_claim('Timing is the most informative family', '§10.2 · §6.3',
       f'top family alone: {_top["family"]} F1 {float(_top["alone_f1"]):.3f}'
       + (f' · Timing (ref-free) F1 {float(_tim["alone_f1"].iloc[0]):.3f}, null p {float(_tim["alone_emp_p"].iloc[0]):.3g}'
          f' ({_tim["alone_emp_null"].iloc[0]})' if len(_tim) else '')
       + ((f' · {_tg["set_name"].iloc[0]}: '
           + (f'F1 {float(_tg_rd["f1_macro"].iloc[0]):.3f}, null p {float(_tg_rd["emp_p"].iloc[0]):.3g} '
              f'({_tg_rd["null_label"].iloc[0]}), ' if len(_tg_rd) else 'F1 vs null n/a, ')
           + f'ΔF1 {float(_tg["d_f1"].iloc[0]):+.3f} (q {float(_tg["q_f1"].iloc[0]):.3f}), '
           f'ΔR²a {_tg_dr2a:+.3f} (p {float(_tg["p_r2_aro"].iloc[0]):.3f}, q {float(_tg["q_r2_aro"].iloc[0]):.3f})')
          if len(_tg) else ' · tempogram sensitivity n/a'),
       'family-alone F1 (piece) vs the other families and the empirical null; the timing set without the tempogram '
       '(§6.3): F1 vs its null and the loss of R² arousal',
       ('✔' if (_t_first and _t_null and _tg_f1_ok and _tg_r2_ok) else '◐' if _t_first else '✘'),
       '✔ highest alone F1 AND beats its null AND without the tempogram F1 beats its null AND ΔR²a > −0.10 · '
       '◐ highest, another condition fails · ✘ not highest (rule revised post hoc in Phase C3)')

df_claims = pd.DataFrame(_claims)
df_claims.insert(0, 'n', range(1, len(df_claims) + 1))
print('📌 §11.1 CLAIMS TABLE  (piece = PRIMARY · value read from variables · verdict from the rule in the last column)')
with pd.option_context('display.width', 260, 'display.max_colwidth', 120):
    print(df_claims[['n', 'claim', 'section', 'verdict', 'value']].to_string(index=False))
df_claims.to_csv(os.path.join(OUTPUT_DIR, 'claims.csv'), index=False)

# ── Final summary table — the DEFAULT classifier, never a best-of (kept from the former §7.1) ────────────
rows=[]
for sn in FEATURE_SETS:          # A / B / C / D
    row = {'Feature Set': sn}
    for pr in PROTOCOLS:
        f1d = DEFAULT_RECS[(sn, pr)]['f1_macro']
        row[f'{DEFAULT_CLF_NAME} F1-macro ({pr})'] = round(f1d, 4)
        row[f'{DEFAULT_CLF_NAME} configs ({pr})'] = DEFAULT_RECS[(sn, pr)]['selected']
        row[f'{NB_CLF_NAME} (pre-declared Phase A default — failed under piece) F1-macro ({pr})'] = \
            round(NB_RECS[(sn, pr)]['f1_macro'], 4)
        row[f'Chance mean ({pr})']   = round(CHANCE_FLOOR_MEAN[pr], 4)
        row[f'Chance P95 ({pr})']    = round(CHANCE_FLOOR_P95[pr], 4)
        row[f'Beats chance P95 ({pr})'] = bool(f1d > CHANCE_FLOOR_P95[pr])
        _en = df_emp_null_primary[(df_emp_null_primary['primary_set'] == sn) & (df_emp_null_primary['protocol'] == pr)]
        assert len(_en) == 1 and abs(float(_en['f1_macro'].iloc[0]) - f1d) < 1e-12, \
            f'§11.1: the §6.9 primary read-out has no unique, matching row for {sn} ({pr})'
        _ep, _eq = float(_en['emp_p'].iloc[0]), float(_en['q_primary'].iloc[0])
        row[f'Empirical-null p, exact width ({pr})'] = round(_ep, 4) if np.isfinite(_ep) else np.nan
        row[f'Empirical-null q, primary BH ({pr})'] = round(_eq, 4) if np.isfinite(_eq) else np.nan
        row[f'Beats empirical null ({pr})'] = bool(_eq < 0.05) if np.isfinite(_eq) else 'n/a (not drawn)'
        row[f'Mean F1 sensitivity clfs ({pr})'] = round(float(np.mean(
            [r['f1_macro'] for r in all_recs if r['feature_set'] == sn and r['protocol'] == pr
             and r['classifier'] != DEFAULT_CLF_NAME])), 4)
    rows.append(row)
df_final=pd.DataFrame(rows)
print('\n📊 FINAL SUMMARY  (default classifier, both protocols — piece = PRIMARY)'); print('='*70)
print(df_final.to_string(index=False))
print('   Beats empirical null = q < 0.05 against the EXACT-WIDTH null of the deployed pipeline, BH within the '
      'pre-declared primary family (§5.3 / §6.9); "n/a (not drawn)" = no null under that protocol')
df_final.to_csv(os.path.join(OUTPUT_DIR,'final_summary.csv'),index=False)

# ── Save predictions — the default classifier on X_C and the DEPLOYED nested ridge, per protocol ──
for pr in PROTOCOLS:
    df_expr[f'default_pred_classes_{pr}'] = [list(mlb.classes_[np.where(r)[0]])
                                             for r in DEFAULT_RECS[(DEFAULT_SET, pr)]['Y_pred']]
    df_expr[f'pred_valence_{pr}']  = REG_RES[pr][REG_MERTFULL]['vp']        # MERT-full 768, nested ridge
    df_expr[f'pred_arousal_{pr}']  = REG_RES[pr][REG_MERTFULL]['ap']
    df_expr[f'pred_quadrant_{pr}'] = PRED_QUADS[pr]
    df_expr[f'pred_valence_interp_{pr}']  = REG_RES[pr][REG_INTERP]['vp']   # Interpretable (27), nested ridge
    df_expr[f'pred_arousal_interp_{pr}']  = REG_RES[pr][REG_INTERP]['ap']
    df_expr[f'pred_quadrant_interp_{pr}'] = PRED_QUADS_INTERP[pr]
df_expr.to_csv(os.path.join(OUTPUT_DIR,'all_predictions.csv'),index=False)
df_sh.to_csv(os.path.join(OUTPUT_DIR,'analysis_d_shifts.csv'),index=False)
print(f'\n✅ claims.csv, final_summary.csv, all_predictions.csv · all outputs in: {OUTPUT_DIR}')


---
## 📋 Output File Reference (all in `OUTPUT_DIR`; listed in notebook order — Phase C2)
| File | Contents |
|---|---|
| **Part 2 — Feature extraction (caches)** | |
| `emb_mert_v2.npz` | MERT 768-dim embeddings (all 60 excerpts, cached) — v2 = after the pos-conv weight restore (§2.1); the pre-fix `emb_mert.npz` is never read |
| `emb_mert_layers_v3.npz` | Phase C1 (§2.1c): per excerpt the time-mean of all 13 MERT hidden states (13 × 768, float32; label-free); read only by the diagnostic §7.2 — the deployed `emb_mert_v2.npz` is unchanged |
| `crepe_tracks_v2.npz` | CREPE tracks per excerpt (`time, f0_hz, confidence, voiced`) — extracted once (§2.2) |
| `rms10_v3.npz` | Per-excerpt 10 ms RMS envelope (dB) on the CREPE grid — since Phase B3 it only feeds the report-only evaluation of the (disabled) re-articulation detector (§2.2) |
| `feat_crepe_v4.csv` | Phase B3 CREPE descriptors (10) + the diagnostic `crepe_vibrato_n_notes`. `feat_crepe_v2.csv` / `_v3.csv` are retired (`OLD_CACHE_NAMES`, asserted in §2.5) |
| `essentia_features_v2.csv` / `.npz` | Essentia block (normalised flux, gated dynamic range, …) (§2.3) |
| `rms_frames_v2.npz` | Per-excerpt frame RMS dB (the B12 silence-gate sweep) |
| `recording_level_check.csv` | §2.3c recording-level sanity check (peak, clipping, LUFS robust z, noise floor) — report only |
| `dtw_features_v4.csv` | (v4 = note onsets from the B3 segmentation; v2 / v3 retired) FULL timing block: 7 reference-free `timing_*` + 1 diagnostic + 5 DTW-deviation `dtw_*` features |
| `mert_forwarding.csv` · `crepe_forwarding.csv` · `essentia_forwarding.csv` · `dtw_forwarding.csv` | §2.1b / §2.2b / §2.3b / §2.4b controlled (unsupervised) forwarding audits |
| **Part 3 — Tool validation / EDA** | |
| `fig_eda_label_counts.png` · `fig_eda_label_cooccurrence.png` · `eda_label_cooccurrence.csv` | §3.1 class counts / co-occurrence |
| `fig_eda_n_responses.png` · `eda_quadrant_counts.csv` | §3.2 rater coverage / quadrant occupancy |
| `fig_eda_crepe_hist.png` · `fig_eda_crepe_corr.png` · `eda_crepe_summary.csv` | §3.3 CREPE sanity |
| `eda_dtw_summary.csv` · `fig_eda_dtw_hist.png` · `fig_eda_dtw_by_condition.png` | §3.3b DTW timing sanity |
| `fig_eda_pca_scree.png` · `eda_pca_variance.csv` | §3.4 MERT PCA scree |
| **Part 4 — Manipulation check** | |
| `dose_response.csv` | §4.2 / §4.3b Page's trend test per technique (INTERP_DESC_COLS) and per rating: z, p, **q (one BH family)**, ⚑ flags |
| `fig_dose_response.png` | §4.3b signed trend z, techniques and ratings (solid = q < .05) |
| `participant_stats_by_condition.csv` · `participant_posthoc_tests.csv` · `participant_raw_tag_counts.csv` · `fig_va_by_condition.png` | §4.3a rating statistics (Friedman, Wilcoxon post-hoc) |
| `lift_matrix.csv` · `lift_matrix_raw_tags.csv` · `lift_threshold_sensitivity.csv` · `lift_matrix_meanshare.csv` · `fig_lift_heatmap.png` | §4.3c tag-based lift (class–condition binding) |
| `class_share_by_condition.csv` | §4.3c ground-truth class share per condition (descriptive) |
| **Part 5 — Evaluation yardstick** | |
| `metrics_dummy_floors.csv` | §5.2 Dummy floors (mean / SD / P95 over 200 re-seeded repeats) per protocol and strategy |
| `null_f1_draws.csv` | §5.3 (A) the 100-draw map: pooled F1 of NestedSelect on i.i.d. N(0, 1) features per (protocol, d ∈ {25, 10}, draw) + chosen configs; settings hash → cache; seeds the primary d = 10 / 25, not a reference since Phase C3; copied from the Kaggle input `violin-thesis-null-cache` if present |
| `null_f1_draws_primary.csv` | §5.3 (B) the **exact-width primary null** (Phase C3: 1000 draws at d = 7 / 17, 600 at the other widths of the pre-declared primary family); settings hash per row (the draw targets are not in it) → cache, **resumable**; copied from the Kaggle input `violin-thesis-null-cache` if present |
| **Part 6 — Prediction** | |
| `ablation_results.csv` | §6.1 4-set × (8-classifier sensitivity table + NestedSelect default), both protocols, + the empirical-null p of the default rows |
| `regression_results.csv` | §6.2 deployed nested ridge (MERT-full 768, Interpretable (27)) + the sensitivity rows (fixed Ridge / SVR / MLP on MERT-PCA25): r, R², RMSE per protocol, `role` column |
| `ablation_blocks.csv` · `ablation_blocks_vs_full.csv` · `ablation_unique_marginal.csv` · `fig_ablation_lattice.png` · `fig_ablation_unique_marginal.png` | §6.3 block lattice (rows carry `emp_null_p`, `beats_emp_null`) |
| `ablation_tempogram_sensitivity.csv` | §6.3 tier `1b-tempogram-sensitivity`: the timing block without the 3 tempogram features vs its parent (Δ, paired p, BH q) |
| `ablation_config.csv` · `ablation_config_deltas.csv` · `fig_ablation_config.png` | §6.4 tiers 4A / 4B / 4D / 4E |
| `ablation_neutral_radius.csv` · `ablation_ground_truth.csv` · `fig_ablation_neutral_radius.png` | §6.5 target-side ablation |
| `metrics_per_label.csv` · `fig_metrics_per_label.png` | §6.6 per-class precision / recall / F1 / support + chance-corrected agreement α (default on X_C) |
| `metrics_baselines.csv` · `fig_metrics_baselines.png` | §6.6 Dummy floors vs the default per feature set (+ the empirical-null P95 / p) and the Phase A NaiveBayes default |
| `quadrant_accuracy.csv` · `fig_quadrant_cm.png` | §6.8 quadrant accuracy from the deployed nested ridge (MERT-full 768, Interpretable (27)) + majority share + permutation null |
| `fig_va_scatter.png` · `fig_circumplex.png` | §6.8 Fig 5 (deployed nested ridge, all 60 excerpts) · Fig 6 ground-truth circumplex |
| `emp_null_readout.csv` | §6.9 every reported classification F1 (§6.1 headline, §6.3 lattice + tempogram rows; §7.1 blocks and §7.2 layers appended there) vs the Dummy strict P95 and its empirical null (Phase C3 rule: exact width / "borrowed" from the next larger primary width / "extrapolated" label, p, primary BH q) |
| `emp_null_primary.csv` | §6.9 the pre-declared primary family: F1, exact-width null P95, p, **BH q within the family** |
| **Part 7 — MERT diagnostics** | |
| `mert_diagnostics/mert_train_test_gap.csv` · `mert_validation_curve.csv` · `mert_nn_identity.csv` · `fig_mert_validation_curve.png` · `fig_mert_nn_identity.png` | §7.1 train vs test, validation curve, nearest-neighbour identity |
| `mert_layers/mert_layer_study.csv` · `mert_layer_nested.csv` · `mert_layer_nested_pred.csv` · `fig_mert_layer_profile.png` | §7.2 layer study (DIAGNOSTIC; `MERT_LAYERS` stays [5, 6, 7]) incl. the empirical-null p / BH q of every candidate |
| `mert_crepe_top_dims.csv` · `mert_crepe_corr_full.csv` · `fig_mert_crepe_topcorr.png` | §7.3 MERT-dim ↔ technique correlation map |
| `mert_crepe_probe_r2.csv` · `fig_mert_crepe_probe_r2.png` | §7.3b linear probe (GroupKFold(5) over pieces, α nested) — `PROBE_R2` |
| `analysis_d_shifts.csv` · `analysis_d_shift_tests.csv` · `fig_embedding_shift.png` | §7.4 embedding distance vs the signed ΔV / ΔA (6 tests, BH q) + 2 exploratory correlations; Fig 8 (3 panels) |
| `featsel_crepe_block.csv` · `fig_featsel_crepe_block.png` | §7.5 Δ vs MERT for MERT + CREPE / Essentia / DTW (read from the lattice) and MERT + CREPE-unique (new fit): ΔF1, ΔR²v, ΔR²a, paired p, BH q |
| **Part 8 — Within-piece prediction** | |
| `within_piece.csv` · `within_piece_noise_ceiling.csv` | §8.1 r, R², RMSE, within-piece permutation p; B-INC incremental ΔR², p, q; split-half ceilings |
| **Part 9 — Which techniques** | |
| `analysis_condition_deltas.csv` · `analysis_condition_delta_tests.csv` · `fig_delta_correlation.png` · `fig_lowlevel_conditions.png` | §9.1 condition deltas (r, p, BH q, ⚑) — Fig TE1 with 3 panels |
| `interp_readout_coefficients.csv` · `interp_readout_cv.csv` · `fig_readout_regression.png` · `fig_readout_classes.png` | §9.2 interpretable read-out (27 features) |
| `shap_technique_importance.csv` · `shap_vs_readout_agreement.csv` · `fig_shap_importance.png` · `fig_shap_ranked.png` · `fig_shap_beeswarm_*.png` | §9.3 SHAP (in-sample cross-check) + top-feature agreement with the §9.2 betas |
| **Part 10 — Cost of explainability** | |
| `coe.csv` · `coe_gaps.csv` · `fig_coe.png` | §10.1 cost of explainability (headline max(MERT-PCA25, MERT-full 768) and the two separate gaps) |
| `ablation_subfamily.csv` · `ablation_subfamily_decomposition.csv` · `ablation_subfamily_deltas.csv` · `ablation_subfamily_commonality.csv` · `fig_ablation_subfamily.png` | §10.2 sub-family ablation (27-feature layer) |
| `overlap_pairs.csv` · `overlap_probe.csv` · `overlap_probe_family.csv` · `overlap_corr.csv` · `overlap_corr_family.csv` · 3 figures | §10.3 family overlap (pair removal, probes, correlations, triangulation) |
| `ablation_master.csv` | every ablation tier (§6.3–§6.5, §7.2 `6-mert-layers`, §7.5 `7.5-mert-plus`, §10.2 `3-subfamily`, §10.3 `3-subfamily-pairs`) in one table — re-exported by §10.2 / §10.3 |
| **Part 11 — Synthesis** | |
| `claims.csv` | §11.1 claims table: claim, section, value (from variables), comparator, verdict ✔ / ◐ / ✘, rule |
| `final_summary.csv` | NestedSelect F1 (piece, loo) + configs, Dummy mean / P95, exact-width empirical-null p and primary BH q, NaiveBayes (Phase A default), mean F1 of the sensitivity classifiers |
| `all_predictions.csv` | Per-excerpt GT + default-classifier and deployed nested-ridge predictions (MERT-full 768 and Interpretable (27)) and quadrants per protocol |
